In [71]:
### Making preliminary functions and loading packages
cell1_start_time = time(); start_load_pkgs = time()
    using Pkg;  
#    Pkg.add("Plots"); using Plots;  
    Pkg.add("JSON3"); using JSON3;    
    Pkg.add("Dates"); using Dates;
    Pkg.add("JLD2"); using JLD2
    Pkg.add("HypothesisTests"); using HypothesisTests
    Pkg.add("DataFrames"); using DataFrames
    Pkg.add("CSV"); using CSV
    Pkg.add("XLSX"); using XLSX
    Pkg.add("FASTX"); using FASTX
#    Pkg.add("Combinatorics"); using Combinatorics
    Pkg.add("StatsBase"); using StatsBase
    using Statistics
date_now = Dates.format(now(), "yyyy_mm_dd__IMMp"); println(date_now)
println(pwd()); cd("/Users/ryhisner"); println(pwd())
#    Pkg.add("LaTeXStrings");  using LaTeXStrings
#    Pkg.add("PyPlot"); using PyPlot
#    Pkg.add("PlotlyJS"); using PlotlyJS
#    Pkg.add("PGFPlotsX"); using PGFPlotsX
#    Pkg.add("UnicodePlots"); using UnicodePlots
#    Pkg.add("InspectDR"); using InspectDR
#    Pkg.add("GLMakie"); using GLMakie
#    Pkg.add("CairoMakie"); using CairoMakie
#    Pkg.add("WGLMakie"); using WGLMakie
#    Pkg.add("GMT"); using GMT
#####################################################################################################################################
### Turns time in seconds to hours, minutes, & seconds
function seconds_to_hrs_min_sec(t)
    hours = 0
    minutes = 0
    seconds = 0
    hours = t÷3600
    minutes = (t%3600)÷60
    if t > 0.0001
        seconds = (t%3600)%60
    end
    hours_int = Int(hours)
    minutes_int = Int(minutes)
    minutes_str = split(string(minutes_int), ".")[1]
    hours_fin = split(string(hours_int), ".")[1]
    minutes_fin = ""
    minutes_fin = lpad(minutes_str, 2, "0")
    seconds_rd = round(digits=2, seconds)
    seconds_1 = string(split(string(seconds_rd), ".")[1])
    seconds_2 = string(split(string(seconds_rd), ".")[2])
    seconds_left = lpad(seconds_1, 2, "0")
    seconds_right = rpad(seconds_2, 2, "0")
    seconds_fin = "$(seconds_left).$(seconds_right)"
    final_time = "$hours_int:$minutes_fin:$seconds_fin"
    final_time2 = "$hours_int hr, $minutes_int min, $seconds_fin sec"
    return final_time, final_time2
end
#####################################################################################################################################
load_pkgs_runtime = time() - start_load_pkgs
load_pkgs_hms1, load_pkgs_hms2 = seconds_to_hrs_min_sec(load_pkgs_runtime)
println("Time to Load Packages = ", load_pkgs_hms1); println("Time to Load Packages = ", load_pkgs_hms2)
#####################################################################################################################################
function add_leading_zero(int_str::String)
    int_str2 = int_str
    if length(int_str) == 1 && int_str ≠ "0"
        int_str2 = "0"*int_str
    end
    return int_str2
end     
######################################################################################################################################
### Adds zero to truncated digit so all numbers have same # of digits & line up nicely E.g. if number = 9.1 & digits_rd = 3, it returns 9.100
function add_zeros_to_rounded(number::Float64, digits_rd::Int)
    if number ≥ 1
        num_final = 0.0
        num_whole = number*10^digits_rd
        num_whole_str = split(string(num_whole), ".")[1]
        len = length(num_whole_str)
        if len ≤ 2
            num_final = "."*"0"^(digits_rd-len)*num_whole_str
        else
            num_final = num_whole_str[1:end-digits_rd]*"."*num_whole_str[end-digits_rd+1:end]
        end
    return num_final
    end
end
#####################################################################################################################################
## Makes all digits go to 2 decimal places so they line up nicely
function number_to_string_to_two_decimal_places(number::Float64, decimals::Int)  
    num_str = string(number)
    before_dec = split(num_str, ".")[1]
    before_dec_str = string(before_dec)
    after_dec = split(num_str, ".")[2]
    after_dec_str = ""
    if length(after_dec) ≥ decimals
        after_dec_str = after_dec[1:decimals]
    else
        after_dec_len = length(after_dec)
        zero_array = Vector{String}()
        missing_zeros_to_fill = decimals - after_dec_len
        for i in 1:missing_zeros_to_fill
            push!(zero_array, "0")
        end
        zeros_str = join(zero_array)
        after_dec_str = after_dec[1:after_dec_len]*zeros_str
    end
    final_num_string = before_dec_str*"."*after_dec_str
    return final_num_string
end
#####################################################################################################################################
hydrophobic_index_dict = Dict{String, Int}("L"=>97, "I"=>99, "F"=>100, "W"=>97, "V"=>76, "M"=>74, "C"=>49, "Y"=>63, "A"=>41, "T"=>13, "E"=>-31, "H"=>8, "G"=>0, "S"=>-5, "Q"=>-10, "D"=>-55, "R"=>-14, "K"=>-23, "N"=>-28, "P"=>5, "*"=>0)
#####################################################################################################################################
wuhan_ref_seq = "..."
ref_seq = wuhan_ref_seq
######################################################################################################################################
clade_set_complete = Set(["recombinant", "19A", "19B", "20A", "20B", "20C", "20D", "20E", "20F", "20G", "20H", "20I", "20J", "21A", "21B", "21C", "21D", "21E", "21F", "21G", "21H", "21I", "21J", "21K", "21L", "21M", "22A", "22B", "22C", "22D", "22E", "22F", "23A", "23B", "23C", "23D", "23E", "23F", "23G", "23H", "23I", "24A", "24B", "24C", "24D", "24E", "24F", "24G", "24H", "24I", "25A", "25B", "25C", "25D", "25E", "25F", "25G", "25H", "25I"])
clade_arr_complete = ["recombinant", "19A", "19B", "20A", "20B", "20C", "20D", "20E", "20F", "20G", "20H", "20I", "20J", "21A", "21B", "21C", "21D", "21E", "21F", "21G", "21H", "21I", "21J", "21K", "21L", "21M", "22A", "22B", "22C", "22D", "22E", "22F", "23A", "23B", "23C", "23D", "23E", "23F", "23G", "23H", "23I", "24A", "24B", "24C", "24D", "24E", "24F", "24G", "24H", "24I", "25A", "25B", "25C", "25D", "25E", "25F", "25G", "25H", "25I"]
clade_to_pango = Dict("recombinant"=>"recombinant", "19A"=>"B", "19B"=>"A", "20A"=>"B.1", "20B"=>"B.1.1", "20C"=>"B.1", "20D"=>"B.1.1.1", "20E"=>"B.1.177", "20F"=>"D.2", "20G"=>"B.1.2", "20H"=>"B.1.351", "20I"=>"B.1.1.7", "20J"=>"P.1", "21A"=>"B.1.617.2", "21B"=>"B.1.617.1", "21C"=>"B.1.427", "21D"=>"B.1.525", "21E"=>"P.3", "21F"=>"B.1.526", "21G"=>"C.37", "21H"=>"B.1.621", "21I"=>"B.1.617.2", "21J"=>"B.1.617.2", "21K"=>"BA.1", "21L"=>"BA.2", "21M"=>"BA.3", "22A"=>"BA.4", "22B"=>"BA.5", "22C"=>"BA.2.12.1", "22D"=>"BA.2.75", "22E"=>"BQ.1", "22F"=>"XBB", "23A"=>"XBB.1.5", "23B"=>"XBB.1.16", "23C"=>"CH.1.1", "23D"=>"XBB.1.9", "23E"=>"XBB.2.3", "23F"=>"EG.5.1", "23G"=>"XBB.1.5.70", "23H"=>"HK.3", "23I"=>"BA.2.86", "24A"=>"JN.1", "24B"=>"JN.1.11.1", "24C"=>"KP.3", "24D"=>"XDV.1", "24E"=>"KP.3.1.1", "24F"=>"XEC", "24G"=>"KP.2.3", "24H"=>"LF.7", "24I"=>"MV.1", "25A"=>"LP.8.1", "25B"=>"NB.1.8.1", "25C"=>"XFG", "25D"=>"MC.10.2.1", "25E"=>"PY.1", "25F"=>"NW.1.2", "25G"=>"XFC", "25H"=>"XFJ", "25I"=>"BA.3.2")
######################################################################################################################################
EPI_ISL(n) = split(n, "|")[2]
country(n) = split(n, "/")[2]
sequence_date(n) = split(n, "|")[3]
seq_lab(n) = split(n, "/")[3]
US_state(n) = split(split(n, "/")[3], "-")[1]
mut_gene(n) = split(string(n), ":")[1]
######################################################################################################################################
AAsub_gene(n) = aa_gene_comprehensive_dict[n]
AAsub_gene_num(n) = [aa_gene_comprehensive_dict[n], aa_pos_comprehensive_dict[n]]
mut_num_pos_only(n) = aa_pos_comprehensive_dict[n]
AAsub_gene_num_pos_only(n) = [aa_gene_comprehensive_dict[n], aa_pos_comprehensive_dict[n]]
mut_gene_Dict = Dict{String, Int}("ORF1a"=>1, "ORF1b"=>2, "S"=>3, "E"=>4, "M"=>5, "N"=>6, "ORF3a"=>7, "ORF6"=>8, "ORF7a"=>9, "ORF7b"=>10, "ORF8"=>11, "ORF9b"=>12)
#################################
AA_gene_sortKey_2(n) = (mut_gene_Dict[aa_gene_comprehensive_dict[n]], aa_pos_comprehensive_dict[n])
AA_gene_sortKey(n) = (mut_gene_Dict[aa_gene_comprehensive_dict[n[1]]], aa_pos_comprehensive_dict[n[1]])
AA_ct_sortKey1(n) = (1000÷mut_gene_Dict[aa_gene_comprehensive_dict[n[1]]], aa_pos_comprehensive_dict[n[1]])
AA_ct_sortKey2(n) = (n[2], AA_ct_sortKey1(n))
AA_gene_pos_sortKey(n) = (mut_gene_Dict[aa_gene_comprehensive_dict[n[1]]], aa_pos_comprehensive_dict[n[1]])
AA_gene_pos_sortKey_2(n) = (mut_gene_Dict[aa_gene_comprehensive_dict[n]], aa_pos_comprehensive_dict[n])
AA_ct_pos_sortKey1(n) = (1000÷mut_gene_Dict[aa_gene_comprehensive_dict[n[1]]], aa_pos_comprehensive_dict[n[1]])
AA_ct_pos_sortKey2(n) = (n[2], AA_ct_pos_sortKey1(n))
######################################################################################################################################
function AA_order_key(mut)
    gene = aa_gene_comprehensive_dict[mut]
    AApos = aa_pos_comprehensive_dict[mut]
    gene_pos = gene_print_order[gene]
    return (gene_pos, AApos)
end
#####################################################################################################################################
function pango_variant_sort_key(pango::String)
    dotparts = split(pango, ".")
    k1 = string(dotparts[1])
    k2 = 0
    k3 = 0
    k4 = 0
    if length(dotparts) ≥ 2
        k2 = parse(Int, string(dotparts[2]))
    end
    if length(dotparts) ≥ 3
        k3 = parse(Int, string(dotparts[3]))
    end
    if length(dotparts) ≥ 4
        k4 = parse(Int, string(dotparts[4]))
    end
    return (k1, k2, k3, k4)
end
######################################################################################################################################
gene_print_order = Dict{String, Int}("S"=>1, "N"=>2, "E"=>3, "M"=>4, "ORF3a"=>5, "ORF6"=>6, "ORF7a"=>7, "ORF7b"=>8, "ORF8"=>9, "ORF9b"=>10, "ORF1a"=>12, "ORF1b"=>13)
######################################################################################################################################
function pango_minus_X_fx(pango::String, minus::Int)
    unaliased = pango_to_pango_unaliased_v2[pango]
    dot_ct = count(".", unaliased)
    println(dot_ct)
    if dot_ct ≥ minus
        dotsplits = split(unaliased, ".")
        minus_X_unaliased = join(dotsplits[1:dot_ct+1-minus], ".")
        minus_X_pango = pango_unaliased_to_pango[minus_X_unaliased]
        println("$(pango), $(unaliased), minus-$(minus) = $(minus_X_unaliased)")
        return minus_X_pango
    else
        return pango
    end
end
######################################################################################################################################
######################################################################################################################################
function read_fasta(filepath::String)
    reader = FASTX.FASTA.Reader(open(filepath, "r"))
    fasta_in = [record for record in reader]
    close(reader)
    return[String(FASTX.FASTX.description(rec)) for rec in fasta_in],
    [uppercase(String(FASTX.FASTA.sequence(rec))) for rec in fasta_in]
end
##########################################################################################################################################################################
##########################################################################################################################################################################
wuhan_ref_seq = "..."
refAA_ORF1a = "MESLVPGFNEKTHVQLSLPVLQVRDVLVRGFGDSVEEVLSEARQHLKDGTCGLVEVEKGVLPQLEQPYVFIKRSDARTAPHGHVMVELVAELEGIQYGRSGETLGVLVPHVGEIPVAYRKVLLRKNGNKGAGGHSYGADLKSFDLGDELGTDPYEDFQENWNTKHSSGVTRELMRELNGGAYTRYVDNNFCGPDGYPLECIKDLLARAGKASCTLSEQLDFIDTKRGVYCCREHEHEIAWYTERSEKSYELQTPFEIKLAKKFDTFNGECPNFVFPLNSIIKTIQPRVEKKKLDGFMGRIRSVYPVASPNECNQMCLSTLMKCDHCGETSWQTGDFVKATCEFCGTENLTKEGATTCGYLPQNAVVKIYCPACHNSEVGPEHSLAEYHNESGLKTILRKGGRTIAFGGCVFSYVGCHNKCAYWVPRASANIGCNHTGVVGEGSEGLNDNLLEILQKEKVNINIVGDFKLNEEIAIILASFSASTSAFVETVKGLDYKAFKQIVESCGNFKVTKGKAKKGAWNIGEQKSILSPLYAFASEAARVVRSIFSRTLETAQNSVRVLQKAAITILDGISQYSLRLIDAMMFTSDLATNNLVVMAYITGGVVQLTSQWLTNIFGTVYEKLKPVLDWLEEKFKEGVEFLRDGWEIVKFISTCACEIVGGQIVTCAKEIKESVQTFFKLVNKFLALCADSIIIGGAKLKALNLGETFVTHSKGLYRKCVKSREETGLLMPLKAPKEIIFLEGETLPTEVLTEEVVLKTGDLQPLEQPTSEAVEAPLVGTPVCINGLMLLEIKDTEKYCALAPNMMVTNNTFTLKGGAPTKVTFGDDTVIEVQGYKSVNITFELDERIDKVLNEKCSAYTVELGTEVNEFACVVADAVIKTLQPVSELLTPLGIDLDEWSMATYYLFDESGEFKLASHMYCSFYPPDEDEEEGDCEEEEFEPSTQYEYGTEDDYQGKPLEFGATSAALQPEEEQEEDWLDDDSQQTVGQQDGSEDNQTTTIQTIVEVQPQLEMELTPVVQTIEVNSFSGYLKLTDNVYIKNADIVEEAKKVKPTVVVNAANVYLKHGGGVAGALNKATNNAMQVESDDYIATNGPLKVGGSCVLSGHNLAKHCLHVVGPNVNKGEDIQLLKSAYENFNQHEVLLAPLLSAGIFGADPIHSLRVCVDTVRTNVYLAVFDKNLYDKLVSSFLEMKSEKQVEQKIAEIPKEEVKPFITESKPSVEQRKQDDKKIKACVEEVTTTLEETKFLTENLLLYIDINGNLHPDSATLVSDIDITFLKKDAPYIVGDVVQEGVLTAVVIPTKKAGGTTEMLAKALRKVPTDNYITTYPGQGLNGYTVEEAKTVLKKCKSAFYILPSIISNEKQEILGTVSWNLREMLAHAEETRKLMPVCVETKAIVSTIQRKYKGIKIQEGVVDYGARFYFYTSKTTVASLINTLNDLNETLVTMPLGYVTHGLNLEEAARYMRSLKVPATVSVSSPDAVTAYNGYLTSSSKTPEEHFIETISLAGSYKDWSYSGQSTQLGIEFLKRGDKSVYYTSNPTTFHLDGEVITFDNLKTLLSLREVRTIKVFTTVDNINLHTQVVDMSMTYGQQFGPTYLDGADVTKIKPHNSHEGKTFYVLPNDDTLRVEAFEYYHTTDPSFLGRYMSALNHTKKWKYPQVNGLTSIKWADNNCYLATALLTLQQIELKFNPPALQDAYYRARAGEAANFCALILAYCNKTVGELGDVRETMSYLFQHANLDSCKRVLNVVCKTCGQQQTTLKGVEAVMYMGTLSYEQFKKGVQIPCTCGKQATKYLVQQESPFVMMSAPPAQYELKHGTFTCASEYTGNYQCGHYKHITSKETLYCIDGALLTKSSEYKGPITDVFYKENSYTTTIKPVTYKLDGVVCTEIDPKLDNYYKKDNSYFTEQPIDLVPNQPYPNASFDNFKFVCDNIKFADDLNQLTGYKKPASRELKVTFFPDLNGDVVAIDYKHYTPSFKKGAKLLHKPIVWHVNNATNKATYKPNTWCIRCLWSTKPVETSNSFDVLKSEDAQGMDNLACEDLKPVSEEVVENPTIQKDVLECNVKTTEVVGDIILKPANNSLKITEEVGHTDLMAAYVDNSSLTIKKPNELSRVLGLKTLATHGLAAVNSVPWDTIANYAKPFLNKVVSTTTNIVTRCLNRVCTNYMPYFFTLLLQLCTFTRSTNSRIKASMPTTIAKNTVKSVGKFCLEASFNYLKSPNFSKLINIIIWFLLLSVCLGSLIYSTAALGVLMSNLGMPSYCTGYREGYLNSTNVTIATYCTGSIPCSVCLSGLDSLDTYPSLETIQITISSFKWDLTAFGLVAEWFLAYILFTRFFYVLGLAAIMQLFFSYFAVHFISNSWLMWLIINLVQMAPISAMVRMYIFFASFYYVWKSYVHVVDGCNSSTCMMCYKRNRATRVECTTIVNGVRRSFYVYANGGKGFCKLHNWNCVNCDTFCAGSTFISDEVARDLSLQFKRPINPTDQSSYIVDSVTVKNGSIHLYFDKAGQKTYERHSLSHFVNLDNLRANNTKGSLPINVIVFDGKSKCEESSAKSASVYYSQLMCQPILLLDQALVSDVGDSAEVAVKMFDAYVNTFSSTFNVPMEKLKTLVATAEAELAKNVSLDNVLSTFISAARQGFVDSDVETKDVVECLKLSHQSDIEVTGDSCNNYMLTYNKVENMTPRDLGACIDCSARHINAQVAKSHNIALIWNVKDFMSLSEQLRKQIRSAAKKNNLPFKLTCATTRQVVNVVTTKIALKGGKIVNNWLKQLIKVTLVFLFVAAIFYLITPVHVMSKHTDFSSEIIGYKAIDGGVTRDIASTDTCFANKHADFDTWFSQRGGSYTNDKACPLIAAVITREVGFVVPGLPGTILRTTNGDFLHFLPRVFSAVGNICYTPSKLIEYTDFATSACVLAAECTIFKDASGKPVPYCYDTNVLEGSVAYESLRPDTRYVLMDGSIIQFPNTYLEGSVRVVTTFDSEYCRHGTCERSEAGVCVSTSGRWVLNNDYYRSLPGVFCGVDAVNLLTNMFTPLIQPIGALDISASIVAGGIVAIVVTCLAYYFMRFRRAFGEYSHVVAFNTLLFLMSFTVLCLTPVYSFLPGVYSVIYLYLTFYLTNDVSFLAHIQWMVMFTPLVPFWITIAYIICISTKHFYWFFSNYLKRRVVFNGVSFSTFEEAALCTFLLNKEMYLKLRSDVLLPLTQYNRYLALYNKYKYFSGAMDTTSYREAACCHLAKALNDFSNSGSDVLYQPPQTSITSAVLQSGFRKMAFPSGKVEGCMVQVTCGTTTLNGLWLDDVVYCPRHVICTSEDMLNPNYEDLLIRKSNHNFLVQAGNVQLRVIGHSMQNCVLKLKVDTANPKTPKYKFVRIQPGQTFSVLACYNGSPSGVYQCAMRPNFTIKGSFLNGSCGSVGFNIDYDCVSFCYMHHMELPTGVHAGTDLEGNFYGPFVDRQTAQAAGTDTTITVNVLAWLYAAVINGDRWFLNRFTTTLNDFNLVAMKYNYEPLTQDHVDILGPLSAQTGIAVLDMCASLKELLQNGMNGRTILGSALLEDEFTPFDVVRQCSGVTFQSAVKRTIKGTHHWLLLTILTSLLVLVQSTQWSLFFFLYENAFLPFAMGIIAMSAFAMMFVKHKHAFLCLFLLPSLATVAYFNMVYMPASWVMRIMTWLDMVDTSLSGFKLKDCVMYASAVVLLILMTARTVYDDGARRVWTLMNVLTLVYKVYYGNALDQAISMWALIISVTSNYSGVVTTVMFLARGIVFMCVEYCPIFFITGNTLQCIMLVYCFLGYFCTCYFGLFCLLNRYFRLTLGVYDYLVSTQEFRYMNSQGLLPPKNSIDAFKLNIKLLGVGGKPCIKVATVQSKMSDVKCTSVVLLSVLQQLRVESSSKLWAQCVQLHNDILLAKDTTEAFEKMVSLLSVLLSMQGAVDINKLCEEMLDNRATLQAIASEFSSLPSYAAFATAQEAYEQAVANGDSEVVLKKLKKSLNVAKSEFDRDAAMQRKLEKMADQAMTQMYKQARSEDKRAKVTSAMQTMLFTMLRKLDNDALNNIINNARDGCVPLNIIPLTTAAKLMVVIPDYNTYKNTCDGTTFTYASALWEIQQVVDADSKIVQLSEISMDNSPNLAWPLIVTALRANSAVKLQNNELSPVALRQMSCAAGTTQTACTDDNALAYYNTTKGGRFVLALLSDLQDLKWARFPKSDGTGTIYTELEPPCRFVTDTPKGPKVKYLYFIKGLNNLNRGMVLGSLAATVRLQAGNATEVPANSTVLSFCAFAVDAAKAYKDYLASGGQPITNCVKMLCTHTGTGQAITVTPEANMDQESFGGASCCLYCRCHIDHPNPKGFCDLKGKYVQIPTTCANDPVGFTLKNTVCTVCGMWKGYGCSCDQLREPMLQSADAQSFLNGFAV"
refAA_ORF1b = "NRVCGVSAARLTPCGTGTSTDVVYRAFDIYNDKVAGFAKFLKTNCCRFQEKDEDDNLIDSYFVVKRHTFSNYQHEETIYNLLKDCPAVAKHDFFKFRIDGDMVPHISRQRLTKYTMADLVYALRHFDEGNCDTLKEILVTYNCCDDDYFNKKDWYDFVENPDILRVYANLGERVRQALLKTVQFCDAMRNAGIVGVLTLDNQDLNGNWYDFGDFIQTTPGSGVPVVDSYYSLLMPILTLTRALTAESHVDTDLTKPYIKWDLLKYDFTEERLKLFDRYFKYWDQTYHPNCVNCLDDRCILHCANFNVLFSTVFPPTSFGPLVRKIFVDGVPFVVSTGYHFRELGVVHNQDVNLHSSRLSFKELLVYAADPAMHAASGNLLLDKRTTCFSVAALTNNVAFQTVKPGNFNKDFYDFAVSKGFFKEGSSVELKHFFFAQDGNAAISDYDYYRYNLPTMCDIRQLLFVVEVVDKYFDCYDGGCINANQVIVNNLDKSAGFPFNKWGKARLYYDSMSYEDQDALFAYTKRNVIPTITQMNLKYAISAKNRARTVAGVSICSTMTNRQFHQKLLKSIAATRGATVVIGTSKFYGGWHNMLKTVYSDVENPHLMGWDYPKCDRAMPNMLRIMASLVLARKHTTCCSLSHRFYRLANECAQVLSEMVMCGGSLYVKPGGTSSGDATTAYANSVFNICQAVTANVNALLSTDGNKIADKYVRNLQHRLYECLYRNRDVDTDFVNEFYAYLRKHFSMMILSDDAVVCFNSTYASQGLVASIKNFKSVLYYQNNVFMSEAKCWTETDLTKGPHEFCSQHTMLVKQGDDYVYLPYPDPSRILGAGCFVDDIVKTDGTLMIERFVSLAIDAYPLTKHPNQEYADVFHLYLQYIRKLHDELTGHMLDMYSVMLTNDNTSRYWEPEFYEAMYTPHTVLQAVGACVLCNSQTSLRCGACIRRPFLCCKCCYDHVISTSHKLVLSVNPYVCNAPGCDVTDVTQLYLGGMSYYCKSHKPPISFPLCANGQVFGLYKNTCVGSDNVTDFNAIATCDWTNAGDYILANTCTERLKLFAAETLKATEETFKLSYGIATVREVLSDRELHLSWEVGKPRPPLNRNYVFTGYRVTKNSKVQIGEYTFEKGDYGDAVVYRGTTTYKLNVGDYFVLTSHTVMPLSAPTLVPQEHYVRITGLYPTLNISDEFSSNVANYQKVGMQKYSTLQGPPGTGKSHFAIGLALYYPSARIVYTACSHAAVDALCEKALKYLPIDKCSRIIPARARVECFDKFKVNSTLEQYVFCTVNALPETTADIVVFDEISMATNYDLSVVNARLRAKHYVYIGDPAQLPAPRTLLTKGTLEPEYFNSVCRLMKTIGPDMFLGTCRRCPAEIVDTVSALVYDNKLKAHKDKSAQCFKMFYKGVITHDVSSAINRPQIGVVREFLTRNPAWRKAVFISPYNSQNAVASKILGLPTQTVDSSQGSEYDYVIFTQTTETAHSCNVNRFNVAITRAKVGILCIMSDRDLYDKLQFTSLEIPRRNVATLQAENVTGLFKDCSKVITGLHPTQAPTHLSVDTKFKTEGLCVDIPGIPKDMTYRRLISMMGFKMNYQVNGYPNMFITREEAIRHVRAWIGFDVEGCHATREAVGTNLPLQLGFSTGVNLVAVPTGYVDTPNNTDFSRVSAKPPPGDQFKHLIPLMYKGLPWNVVRIKIVQMLSDTLKNLSDRVVFVLWAHGFELTSMKYFVKIGPERTCCLCDRRATCFSTASDTYACWHHSIGFDYVYNPFMIDVQQWGFTGNLQSNHDLYCQVHGNAHVASCDAIMTRCLAVHECFVKRVDWTIEYPIIGDELKINAACRKVQHMVVKAALLADKFPVLHDIGNPKAIKCVPQADVEWKFYDAQPCSDKAYKIEELFYSYATHSDKFTDGVCLFWNCNVDRYPANSIVCRFDTRVLSNLNLPGCDGGSLYVNKHAFHTPAFDKSAFVNLKQLPFFYYSDSPCESHGKQVVSDIDYVPLKSATCITRCNLGGAVCRHHANEYRLYLDAYNMMISAGFSLWVYKQFDTYNLWNTFTRLQSLENVAFNVVNKGHFDGQQGEVPVSIINNTVYTKVDGVDVELFENKTTLPVNVAFELWAKRNIKPVPEVKILNNLGVDIAANTVIWDYKRDAPAHISTIGVCSMTDIAKKPTETICAPLTVFFDGRVDGQVDLFRNARNGVLITEGSVKGLQPSVGPKQASLNGVTLIGEAVKTQFNYYKKVDGVVQQLPETYFTQSRNLQEFKPRSQMEIDFLELAMDEFIERYKLEGYAFEHIVYGDFSHSQLGGLHLLIGLAKRFKESPFELEDFIPMDSTVKNYFITDAQTGSSKCVCSVIDLLLDDFVEIIKSQDLSVVSKVVKVTIDYTEISFMLWCKDGHVETFYPKLQSSQAWQPGVAMPNLYKMQRMLLEKCDLQNYGDSATLPKGIMMNVAKYTQLCQYLNTLTLAVPYNMRVIHFGAGSDKGVAPGTAVLRQWLPTGTLLVDSDLNDFVSDADSTLIGDCATVHTANKWDLIISDMYDPKTKNVTKENDSKEGFFTYICGFIQQKLALGGSVAIKITEHSWNADLYKLMGHFAWWTAFVTNVNASSSEAFLIGCNYLGKPREQIDGYVMHANYIFWRNTNPIQLSSYSLFDMSKFPLKLRGTAVMSLKEGQINDMILSLLSKGRLIIRENNRVVISSDVLVNN*"
refAA_S = "MFVFLVLLPLVSSQCVNLTTRTQLPPAYTNSFTRGVYYPDKVFRSSVLHSTQDLFLPFFSNVTWFHAIHVSGTNGTKRFDNPVLPFNDGVYFASTEKSNIIRGWIFGTTLDSKTQSLLIVNNATNVVIKVCEFQFCNDPFLGVYYHKNNKSWMESEFRVYSSANNCTFEYVSQPFLMDLEGKQGNFKNLREFVFKNIDGYFKIYSKHTPINLVRDLPQGFSALEPLVDLPIGINITRFQTLLALHRSYLTPGDSSSGWTAGAAAYYVGYLQPRTFLLKYNENGTITDAVDCALDPLSETKCTLKSFTVEKGIYQTSNFRVQPTESIVRFPNITNLCPFGEVFNATRFASVYAWNRKRISNCVADYSVLYNSASFSTFKCYGVSPTKLNDLCFTNVYADSFVIRGDEVRQIAPGQTGKIADYNYKLPDDFTGCVIAWNSNNLDSKVGGNYNYLYRLFRKSNLKPFERDISTEIYQAGSTPCNGVEGFNCYFPLQSYGFQPTNGVGYQPYRVVVLSFELLHAPATVCGPKKSTNLVKNKCVNFNFNGLTGTGVLTESNKKFLPFQQFGRDIADTTDAVRDPQTLEILDITPCSFGGVSVITPGTNTSNQVAVLYQDVNCTEVPVAIHADQLTPTWRVYSTGSNVFQTRAGCLIGAEHVNNSYECDIPIGAGICASYQTQTNSPRRARSVASQSIIAYTMSLGAENSVAYSNNSIAIPTNFTISVTTEILPVSMTKTSVDCTMYICGDSTECSNLLLQYGSFCTQLNRALTGIAVEQDKNTQEVFAQVKQIYKTPPIKDFGGFNFSQILPDPSKPSKRSFIEDLLFNKVTLADAGFIKQYGDCLGDIAARDLICAQKFNGLTVLPPLLTDEMIAQYTSALLAGTITSGWTFGAGAALQIPFAMQMAYRFNGIGVTQNVLYENQKLIANQFNSAIGKIQDSLSSTASALGKLQDVVNQNAQALNTLVKQLSSNFGAISSVLNDILSRLDKVEAEVQIDRLITGRLQSLQTYVTQQLIRAAEIRASANLAATKMSECVLGQSKRVDFCGKGYHLMSFPQSAPHGVVFLHVTYVPAQEKNFTTAPAICHDGKAHFPREGVFVSNGTHWFVTQRNFYEPQIITTDNTFVSGNCDVVIGIVNNTVYDPLQPELDSFKEELDKYFKNHTSPDVDLGDISGINASVVNIQKEIDRLNEVAKNLNESLIDLQELGKYEQYIKWPWYIWLGFIAGLIAIVMVTIMLCCMTSCCSCLKGCCSCGSCCKFDEDDSEPVLKGVKLHYT*"
refAA_ORF3a = "MDLFMRIFTIGTVTLKQGEIKDATPSDFVRATATIPIQASLPFGWLIVGVALLAVFQSASKIITLKKRWQLALSKGVHFVCNLLLLFVTVYSHLLLVAAGLEAPFLYLYALVYFLQSINFVRIIMRLWLCWKCRSKNPLLYDANYFLCWHTNCYDYCIPYNSVTSSIVITSGDGTTSPISEHDYQIGGYTEKWESGVKDCVVLHSYFTSDYYQLYSTQLSTDTGVEHVTFFIYNKIVDEPEEHVQIHTIDGSSGVVNPVMEPIYDEPTTTTSVPL*"
refAA_E = "MYSFVSEETGTLIVNSVLLFLAFVVFLLVTLAILTALRLCAYCCNIVNVSLVKPSFYVYSRVKNLNSSRVPDLLV*"
refAA_M = "MADSNGTITVEELKKLLEQWNLVIGFLFLTWICLLQFAYANRNRFLYIIKLIFLWLLWPVTLACFVLAAVYRINWITGGIAIAMACLVGLMWLSYFIASFRLFARTRSMWSFNPETNILLNVPLHGTILTRPLLESELVIGAVILRGHLRIAGHHLGRCDIKDLPKEITVATSRTLSYYKLGASQRVAGDSGFAAYSRYRIGNYKLNTDHSSSSDNIALLVQ*"
refAA_ORF6 = "MFHLVDFQVTIAEILLIIMRTFKVSIWNLDYIINLIIKNLSKSLTENKYSQLDEEQPMEID*"
refAA_ORF7a = "MKIILFLALITLATCELYHYQECVRGTTVLLKEPCSSGTYEGNSPFHPLADNKFALTCFSTQFAFACPDGVKHVYQLRARSVSPKLFIRQEEVQELYSPIFLIVAAIVFITLCFTLKRKTE*"
refAA_ORF7b = "MIELSLIDFYLCFLAFLLFLVLIMLIIFWFSLELQDHNETCHA*"
refAA_ORF8 = "MKFLVFLGIITTVAAFHQECSLQSCTQHQPYVVDDPCPIHFYSKWYIRVGARKSAPLIELCVDEAGSKSPIQYIDIGNYTVSCLPFTINCQEPKLGSLVVRCSFYEDFLEYHDVRVVLDFI*"
refAA_N = "MSDNGPQNQRNAPRITFGGPSDSTGSNQNGERSGARSKQRRPQGLPNNTASWFTALTQHGKEDLKFPRGQGVPINTNSSPDDQIGYYRRATRRIRGGDGKMKDLSPRWYFYYLGTGPEAGLPYGANKDGIIWVATEGALNTPKDHIGTRNPANNAAIVLQLPQGTTLPKGFYAEGSRGGSQASSRSSSRSRNSSRNSTPGSSRGTSPARMAGNGGDAALALLLLDRLNQLESKMSGKGQQQQGQTVTKKSAAEASKKPRQKRTATKAYNVTQAFGRRGPEQTQGNFGDQELIRQGTDYKHWPQIAQFAPSASAFFGMSRIGMEVTPSGTWLTYTGAIKLDDKDPNFKDQVILLNKHIDAYKTFPPTEPKKDKKKKADETQALPQRQKKQQTVTLLPAADLDDFSKQLQQSMSSADSTQA*"
refAA_ORF9b = "MDPKISEMHPALRLVDPQIQLAVTRMENAVGRDQNNVGPKVYPIILRLGSPLSLNMARKTLNSLEDKAFQLTPIAVQMTKLATTEELPDEFVVVTVK*"
######################################################################################################################################
gene_array = ["ORF1a", "ORF1b", "S", "ORF3a", "E", "M", "ORF6", "ORF7a", "ORF7b", "ORF8", "N", "ORF9b"]
gene_order_dict = Dict{String, Int}("ORF1a"=>1, "ORF1b"=>2, "S"=>3, "ORF3a"=>4, "E"=>4, "M"=>6, "ORF6"=>7, "ORF7a"=>8, "ORF7b"=>9, "ORF8"=>10, "N"=>11, "ORF9b"=>12)
gene_AA_sortKey(m) = (gene_order_dict[string(split(m, ":")[1])], aa_pos_comprehensive_dict[m])
gene_AApos_sortKey(m) = (gene_order_dict[string(split(m, ":")[1])], aa_pos_comprehensive_dict[m])
######################################################################################################################################
gene_AA_dict = Dict{String, String}()
gene_AA_dict["ORF1a"] = "MESLVPGFNEKTHVQLSLPVLQVRDVLVRGFGDSVEEVLSEARQHLKDGTCGLVEVEKGVLPQLEQPYVFIKRSDARTAPHGHVMVELVAELEGIQYGRSGETLGVLVPHVGEIPVAYRKVLLRKNGNKGAGGHSYGADLKSFDLGDELGTDPYEDFQENWNTKHSSGVTRELMRELNGGAYTRYVDNNFCGPDGYPLECIKDLLARAGKASCTLSEQLDFIDTKRGVYCCREHEHEIAWYTERSEKSYELQTPFEIKLAKKFDTFNGECPNFVFPLNSIIKTIQPRVEKKKLDGFMGRIRSVYPVASPNECNQMCLSTLMKCDHCGETSWQTGDFVKATCEFCGTENLTKEGATTCGYLPQNAVVKIYCPACHNSEVGPEHSLAEYHNESGLKTILRKGGRTIAFGGCVFSYVGCHNKCAYWVPRASANIGCNHTGVVGEGSEGLNDNLLEILQKEKVNINIVGDFKLNEEIAIILASFSASTSAFVETVKGLDYKAFKQIVESCGNFKVTKGKAKKGAWNIGEQKSILSPLYAFASEAARVVRSIFSRTLETAQNSVRVLQKAAITILDGISQYSLRLIDAMMFTSDLATNNLVVMAYITGGVVQLTSQWLTNIFGTVYEKLKPVLDWLEEKFKEGVEFLRDGWEIVKFISTCACEIVGGQIVTCAKEIKESVQTFFKLVNKFLALCADSIIIGGAKLKALNLGETFVTHSKGLYRKCVKSREETGLLMPLKAPKEIIFLEGETLPTEVLTEEVVLKTGDLQPLEQPTSEAVEAPLVGTPVCINGLMLLEIKDTEKYCALAPNMMVTNNTFTLKGGAPTKVTFGDDTVIEVQGYKSVNITFELDERIDKVLNEKCSAYTVELGTEVNEFACVVADAVIKTLQPVSELLTPLGIDLDEWSMATYYLFDESGEFKLASHMYCSFYPPDEDEEEGDCEEEEFEPSTQYEYGTEDDYQGKPLEFGATSAALQPEEEQEEDWLDDDSQQTVGQQDGSEDNQTTTIQTIVEVQPQLEMELTPVVQTIEVNSFSGYLKLTDNVYIKNADIVEEAKKVKPTVVVNAANVYLKHGGGVAGALNKATNNAMQVESDDYIATNGPLKVGGSCVLSGHNLAKHCLHVVGPNVNKGEDIQLLKSAYENFNQHEVLLAPLLSAGIFGADPIHSLRVCVDTVRTNVYLAVFDKNLYDKLVSSFLEMKSEKQVEQKIAEIPKEEVKPFITESKPSVEQRKQDDKKIKACVEEVTTTLEETKFLTENLLLYIDINGNLHPDSATLVSDIDITFLKKDAPYIVGDVVQEGVLTAVVIPTKKAGGTTEMLAKALRKVPTDNYITTYPGQGLNGYTVEEAKTVLKKCKSAFYILPSIISNEKQEILGTVSWNLREMLAHAEETRKLMPVCVETKAIVSTIQRKYKGIKIQEGVVDYGARFYFYTSKTTVASLINTLNDLNETLVTMPLGYVTHGLNLEEAARYMRSLKVPATVSVSSPDAVTAYNGYLTSSSKTPEEHFIETISLAGSYKDWSYSGQSTQLGIEFLKRGDKSVYYTSNPTTFHLDGEVITFDNLKTLLSLREVRTIKVFTTVDNINLHTQVVDMSMTYGQQFGPTYLDGADVTKIKPHNSHEGKTFYVLPNDDTLRVEAFEYYHTTDPSFLGRYMSALNHTKKWKYPQVNGLTSIKWADNNCYLATALLTLQQIELKFNPPALQDAYYRARAGEAANFCALILAYCNKTVGELGDVRETMSYLFQHANLDSCKRVLNVVCKTCGQQQTTLKGVEAVMYMGTLSYEQFKKGVQIPCTCGKQATKYLVQQESPFVMMSAPPAQYELKHGTFTCASEYTGNYQCGHYKHITSKETLYCIDGALLTKSSEYKGPITDVFYKENSYTTTIKPVTYKLDGVVCTEIDPKLDNYYKKDNSYFTEQPIDLVPNQPYPNASFDNFKFVCDNIKFADDLNQLTGYKKPASRELKVTFFPDLNGDVVAIDYKHYTPSFKKGAKLLHKPIVWHVNNATNKATYKPNTWCIRCLWSTKPVETSNSFDVLKSEDAQGMDNLACEDLKPVSEEVVENPTIQKDVLECNVKTTEVVGDIILKPANNSLKITEEVGHTDLMAAYVDNSSLTIKKPNELSRVLGLKTLATHGLAAVNSVPWDTIANYAKPFLNKVVSTTTNIVTRCLNRVCTNYMPYFFTLLLQLCTFTRSTNSRIKASMPTTIAKNTVKSVGKFCLEASFNYLKSPNFSKLINIIIWFLLLSVCLGSLIYSTAALGVLMSNLGMPSYCTGYREGYLNSTNVTIATYCTGSIPCSVCLSGLDSLDTYPSLETIQITISSFKWDLTAFGLVAEWFLAYILFTRFFYVLGLAAIMQLFFSYFAVHFISNSWLMWLIINLVQMAPISAMVRMYIFFASFYYVWKSYVHVVDGCNSSTCMMCYKRNRATRVECTTIVNGVRRSFYVYANGGKGFCKLHNWNCVNCDTFCAGSTFISDEVARDLSLQFKRPINPTDQSSYIVDSVTVKNGSIHLYFDKAGQKTYERHSLSHFVNLDNLRANNTKGSLPINVIVFDGKSKCEESSAKSASVYYSQLMCQPILLLDQALVSDVGDSAEVAVKMFDAYVNTFSSTFNVPMEKLKTLVATAEAELAKNVSLDNVLSTFISAARQGFVDSDVETKDVVECLKLSHQSDIEVTGDSCNNYMLTYNKVENMTPRDLGACIDCSARHINAQVAKSHNIALIWNVKDFMSLSEQLRKQIRSAAKKNNLPFKLTCATTRQVVNVVTTKIALKGGKIVNNWLKQLIKVTLVFLFVAAIFYLITPVHVMSKHTDFSSEIIGYKAIDGGVTRDIASTDTCFANKHADFDTWFSQRGGSYTNDKACPLIAAVITREVGFVVPGLPGTILRTTNGDFLHFLPRVFSAVGNICYTPSKLIEYTDFATSACVLAAECTIFKDASGKPVPYCYDTNVLEGSVAYESLRPDTRYVLMDGSIIQFPNTYLEGSVRVVTTFDSEYCRHGTCERSEAGVCVSTSGRWVLNNDYYRSLPGVFCGVDAVNLLTNMFTPLIQPIGALDISASIVAGGIVAIVVTCLAYYFMRFRRAFGEYSHVVAFNTLLFLMSFTVLCLTPVYSFLPGVYSVIYLYLTFYLTNDVSFLAHIQWMVMFTPLVPFWITIAYIICISTKHFYWFFSNYLKRRVVFNGVSFSTFEEAALCTFLLNKEMYLKLRSDVLLPLTQYNRYLALYNKYKYFSGAMDTTSYREAACCHLAKALNDFSNSGSDVLYQPPQTSITSAVLQSGFRKMAFPSGKVEGCMVQVTCGTTTLNGLWLDDVVYCPRHVICTSEDMLNPNYEDLLIRKSNHNFLVQAGNVQLRVIGHSMQNCVLKLKVDTANPKTPKYKFVRIQPGQTFSVLACYNGSPSGVYQCAMRPNFTIKGSFLNGSCGSVGFNIDYDCVSFCYMHHMELPTGVHAGTDLEGNFYGPFVDRQTAQAAGTDTTITVNVLAWLYAAVINGDRWFLNRFTTTLNDFNLVAMKYNYEPLTQDHVDILGPLSAQTGIAVLDMCASLKELLQNGMNGRTILGSALLEDEFTPFDVVRQCSGVTFQSAVKRTIKGTHHWLLLTILTSLLVLVQSTQWSLFFFLYENAFLPFAMGIIAMSAFAMMFVKHKHAFLCLFLLPSLATVAYFNMVYMPASWVMRIMTWLDMVDTSLSGFKLKDCVMYASAVVLLILMTARTVYDDGARRVWTLMNVLTLVYKVYYGNALDQAISMWALIISVTSNYSGVVTTVMFLARGIVFMCVEYCPIFFITGNTLQCIMLVYCFLGYFCTCYFGLFCLLNRYFRLTLGVYDYLVSTQEFRYMNSQGLLPPKNSIDAFKLNIKLLGVGGKPCIKVATVQSKMSDVKCTSVVLLSVLQQLRVESSSKLWAQCVQLHNDILLAKDTTEAFEKMVSLLSVLLSMQGAVDINKLCEEMLDNRATLQAIASEFSSLPSYAAFATAQEAYEQAVANGDSEVVLKKLKKSLNVAKSEFDRDAAMQRKLEKMADQAMTQMYKQARSEDKRAKVTSAMQTMLFTMLRKLDNDALNNIINNARDGCVPLNIIPLTTAAKLMVVIPDYNTYKNTCDGTTFTYASALWEIQQVVDADSKIVQLSEISMDNSPNLAWPLIVTALRANSAVKLQNNELSPVALRQMSCAAGTTQTACTDDNALAYYNTTKGGRFVLALLSDLQDLKWARFPKSDGTGTIYTELEPPCRFVTDTPKGPKVKYLYFIKGLNNLNRGMVLGSLAATVRLQAGNATEVPANSTVLSFCAFAVDAAKAYKDYLASGGQPITNCVKMLCTHTGTGQAITVTPEANMDQESFGGASCCLYCRCHIDHPNPKGFCDLKGKYVQIPTTCANDPVGFTLKNTVCTVCGMWKGYGCSCDQLREPMLQSADAQSFLNGFAV*"
gene_AA_dict["ORF1b"] = "RVCGVSAARLTPCGTGTSTDVVYRAFDIYNDKVAGFAKFLKTNCCRFQEKDEDDNLIDSYFVVKRHTFSNYQHEETIYNLLKDCPAVAKHDFFKFRIDGDMVPHISRQRLTKYTMADLVYALRHFDEGNCDTLKEILVTYNCCDDDYFNKKDWYDFVENPDILRVYANLGERVRQALLKTVQFCDAMRNAGIVGVLTLDNQDLNGNWYDFGDFIQTTPGSGVPVVDSYYSLLMPILTLTRALTAESHVDTDLTKPYIKWDLLKYDFTEERLKLFDRYFKYWDQTYHPNCVNCLDDRCILHCANFNVLFSTVFPPTSFGPLVRKIFVDGVPFVVSTGYHFRELGVVHNQDVNLHSSRLSFKELLVYAADPAMHAASGNLLLDKRTTCFSVAALTNNVAFQTVKPGNFNKDFYDFAVSKGFFKEGSSVELKHFFFAQDGNAAISDYDYYRYNLPTMCDIRQLLFVVEVVDKYFDCYDGGCINANQVIVNNLDKSAGFPFNKWGKARLYYDSMSYEDQDALFAYTKRNVIPTITQMNLKYAISAKNRARTVAGVSICSTMTNRQFHQKLLKSIAATRGATVVIGTSKFYGGWHNMLKTVYSDVENPHLMGWDYPKCDRAMPNMLRIMASLVLARKHTTCCSLSHRFYRLANECAQVLSEMVMCGGSLYVKPGGTSSGDATTAYANSVFNICQAVTANVNALLSTDGNKIADKYVRNLQHRLYECLYRNRDVDTDFVNEFYAYLRKHFSMMILSDDAVVCFNSTYASQGLVASIKNFKSVLYYQNNVFMSEAKCWTETDLTKGPHEFCSQHTMLVKQGDDYVYLPYPDPSRILGAGCFVDDIVKTDGTLMIERFVSLAIDAYPLTKHPNQEYADVFHLYLQYIRKLHDELTGHMLDMYSVMLTNDNTSRYWEPEFYEAMYTPHTVLQAVGACVLCNSQTSLRCGACIRRPFLCCKCCYDHVISTSHKLVLSVNPYVCNAPGCDVTDVTQLYLGGMSYYCKSHKPPISFPLCANGQVFGLYKNTCVGSDNVTDFNAIATCDWTNAGDYILANTCTERLKLFAAETLKATEETFKLSYGIATVREVLSDRELHLSWEVGKPRPPLNRNYVFTGYRVTKNSKVQIGEYTFEKGDYGDAVVYRGTTTYKLNVGDYFVLTSHTVMPLSAPTLVPQEHYVRITGLYPTLNISDEFSSNVANYQKVGMQKYSTLQGPPGTGKSHFAIGLALYYPSARIVYTACSHAAVDALCEKALKYLPIDKCSRIIPARARVECFDKFKVNSTLEQYVFCTVNALPETTADIVVFDEISMATNYDLSVVNARLRAKHYVYIGDPAQLPAPRTLLTKGTLEPEYFNSVCRLMKTIGPDMFLGTCRRCPAEIVDTVSALVYDNKLKAHKDKSAQCFKMFYKGVITHDVSSAINRPQIGVVREFLTRNPAWRKAVFISPYNSQNAVASKILGLPTQTVDSSQGSEYDYVIFTQTTETAHSCNVNRFNVAITRAKVGILCIMSDRDLYDKLQFTSLEIPRRNVATLQAENVTGLFKDCSKVITGLHPTQAPTHLSVDTKFKTEGLCVDIPGIPKDMTYRRLISMMGFKMNYQVNGYPNMFITREEAIRHVRAWIGFDVEGCHATREAVGTNLPLQLGFSTGVNLVAVPTGYVDTPNNTDFSRVSAKPPPGDQFKHLIPLMYKGLPWNVVRIKIVQMLSDTLKNLSDRVVFVLWAHGFELTSMKYFVKIGPERTCCLCDRRATCFSTASDTYACWHHSIGFDYVYNPFMIDVQQWGFTGNLQSNHDLYCQVHGNAHVASCDAIMTRCLAVHECFVKRVDWTIEYPIIGDELKINAACRKVQHMVVKAALLADKFPVLHDIGNPKAIKCVPQADVEWKFYDAQPCSDKAYKIEELFYSYATHSDKFTDGVCLFWNCNVDRYPANSIVCRFDTRVLSNLNLPGCDGGSLYVNKHAFHTPAFDKSAFVNLKQLPFFYYSDSPCESHGKQVVSDIDYVPLKSATCITRCNLGGAVCRHHANEYRLYLDAYNMMISAGFSLWVYKQFDTYNLWNTFTRLQSLENVAFNVVNKGHFDGQQGEVPVSIINNTVYTKVDGVDVELFENKTTLPVNVAFELWAKRNIKPVPEVKILNNLGVDIAANTVIWDYKRDAPAHISTIGVCSMTDIAKKPTETICAPLTVFFDGRVDGQVDLFRNARNGVLITEGSVKGLQPSVGPKQASLNGVTLIGEAVKTQFNYYKKVDGVVQQLPETYFTQSRNLQEFKPRSQMEIDFLELAMDEFIERYKLEGYAFEHIVYGDFSHSQLGGLHLLIGLAKRFKESPFELEDFIPMDSTVKNYFITDAQTGSSKCVCSVIDLLLDDFVEIIKSQDLSVVSKVVKVTIDYTEISFMLWCKDGHVETFYPKLQSSQAWQPGVAMPNLYKMQRMLLEKCDLQNYGDSATLPKGIMMNVAKYTQLCQYLNTLTLAVPYNMRVIHFGAGSDKGVAPGTAVLRQWLPTGTLLVDSDLNDFVSDADSTLIGDCATVHTANKWDLIISDMYDPKTKNVTKENDSKEGFFTYICGFIQQKLALGGSVAIKITEHSWNADLYKLMGHFAWWTAFVTNVNASSSEAFLIGCNYLGKPREQIDGYVMHANYIFWRNTNPIQLSSYSLFDMSKFPLKLRGTAVMSLKEGQINDMILSLLSKGRLIIRENNRVVISSDVLVNN*"
gene_AA_dict["S"] = "MFVFLVLLPLVSSQCVNLTTRTQLPPAYTNSFTRGVYYPDKVFRSSVLHSTQDLFLPFFSNVTWFHAIHVSGTNGTKRFDNPVLPFNDGVYFASTEKSNIIRGWIFGTTLDSKTQSLLIVNNATNVVIKVCEFQFCNDPFLGVYYHKNNKSWMESEFRVYSSANNCTFEYVSQPFLMDLEGKQGNFKNLREFVFKNIDGYFKIYSKHTPINLVRDLPQGFSALEPLVDLPIGINITRFQTLLALHRSYLTPGDSSSGWTAGAAAYYVGYLQPRTFLLKYNENGTITDAVDCALDPLSETKCTLKSFTVEKGIYQTSNFRVQPTESIVRFPNITNLCPFGEVFNATRFASVYAWNRKRISNCVADYSVLYNSASFSTFKCYGVSPTKLNDLCFTNVYADSFVIRGDEVRQIAPGQTGKIADYNYKLPDDFTGCVIAWNSNNLDSKVGGNYNYLYRLFRKSNLKPFERDISTEIYQAGSTPCNGVEGFNCYFPLQSYGFQPTNGVGYQPYRVVVLSFELLHAPATVCGPKKSTNLVKNKCVNFNFNGLTGTGVLTESNKKFLPFQQFGRDIADTTDAVRDPQTLEILDITPCSFGGVSVITPGTNTSNQVAVLYQDVNCTEVPVAIHADQLTPTWRVYSTGSNVFQTRAGCLIGAEHVNNSYECDIPIGAGICASYQTQTNSPRRARSVASQSIIAYTMSLGAENSVAYSNNSIAIPTNFTISVTTEILPVSMTKTSVDCTMYICGDSTECSNLLLQYGSFCTQLNRALTGIAVEQDKNTQEVFAQVKQIYKTPPIKDFGGFNFSQILPDPSKPSKRSFIEDLLFNKVTLADAGFIKQYGDCLGDIAARDLICAQKFNGLTVLPPLLTDEMIAQYTSALLAGTITSGWTFGAGAALQIPFAMQMAYRFNGIGVTQNVLYENQKLIANQFNSAIGKIQDSLSSTASALGKLQDVVNQNAQALNTLVKQLSSNFGAISSVLNDILSRLDKVEAEVQIDRLITGRLQSLQTYVTQQLIRAAEIRASANLAATKMSECVLGQSKRVDFCGKGYHLMSFPQSAPHGVVFLHVTYVPAQEKNFTTAPAICHDGKAHFPREGVFVSNGTHWFVTQRNFYEPQIITTDNTFVSGNCDVVIGIVNNTVYDPLQPELDSFKEELDKYFKNHTSPDVDLGDISGINASVVNIQKEIDRLNEVAKNLNESLIDLQELGKYEQYIKWPWYIWLGFIAGLIAIVMVTIMLCCMTSCCSCLKGCCSCGSCCKFDEDDSEPVLKGVKLHYT*"
gene_AA_dict["E"] = "MYSFVSEETGTLIVNSVLLFLAFVVFLLVTLAILTALRLCAYCCNIVNVSLVKPSFYVYSRVKNLNSSRVPDLLV*"
gene_AA_dict["M"] = "MADSNGTITVEELKKLLEQWNLVIGFLFLTWICLLQFAYANRNRFLYIIKLIFLWLLWPVTLACFVLAAVYRINWITGGIAIAMACLVGLMWLSYFIASFRLFARTRSMWSFNPETNILLNVPLHGTILTRPLLESELVIGAVILRGHLRIAGHHLGRCDIKDLPKEITVATSRTLSYYKLGASQRVAGDSGFAAYSRYRIGNYKLNTDHSSSSDNIALLVQ*"
gene_AA_dict["N"] = "MSDNGPQNQRNAPRITFGGPSDSTGSNQNGERSGARSKQRRPQGLPNNTASWFTALTQHGKEDLKFPRGQGVPINTNSSPDDQIGYYRRATRRIRGGDGKMKDLSPRWYFYYLGTGPEAGLPYGANKDGIIWVATEGALNTPKDHIGTRNPANNAAIVLQLPQGTTLPKGFYAEGSRGGSQASSRSSSRSRNSSRNSTPGSSRGTSPARMAGNGGDAALALLLLDRLNQLESKMSGKGQQQQGQTVTKKSAAEASKKPRQKRTATKAYNVTQAFGRRGPEQTQGNFGDQELIRQGTDYKHWPQIAQFAPSASAFFGMSRIGMEVTPSGTWLTYTGAIKLDDKDPNFKDQVILLNKHIDAYKTFPPTEPKKDKKKKADETQALPQRQKKQQTVTLLPAADLDDFSKQLQQSMSSADSTQA*"
gene_AA_dict["ORF3a"] = "MDLFMRIFTIGTVTLKQGEIKDATPSDFVRATATIPIQASLPFGWLIVGVALLAVFQSASKIITLKKRWQLALSKGVHFVCNLLLLFVTVYSHLLLVAAGLEAPFLYLYALVYFLQSINFVRIIMRLWLCWKCRSKNPLLYDANYFLCWHTNCYDYCIPYNSVTSSIVITSGDGTTSPISEHDYQIGGYTEKWESGVKDCVVLHSYFTSDYYQLYSTQLSTDTGVEHVTFFIYNKIVDEPEEHVQIHTIDGSSGVVNPVMEPIYDEPTTTTSVPL*"
gene_AA_dict["ORF6"] = "MFHLVDFQVTIAEILLIIMRTFKVSIWNLDYIINLIIKNLSKSLTENKYSQLDEEQPMEID*"
gene_AA_dict["ORF7a"] = "MKIILFLALITLATCELYHYQECVRGTTVLLKEPCSSGTYEGNSPFHPLADNKFALTCFSTQFAFACPDGVKHVYQLRARSVSPKLFIRQEEVQELYSPIFLIVAAIVFITLCFTLKRKTE*"
gene_AA_dict["ORF7b"] = "MIELSLIDFYLCFLAFLLFLVLIMLIIFWFSLELQDHNETCHA*"
gene_AA_dict["ORF8"] = "MKFLVFLGIITTVAAFHQECSLQSCTQHQPYVVDDPCPIHFYSKWYIRVGARKSAPLIELCVDEAGSKSPIQYIDIGNYTVSCLPFTINCQEPKLGSLVVRCSFYEDFLEYHDVRVVLDFI*"
gene_AA_dict["ORF9b"] = "MDPKISEMHPALRLVDPQIQLAVTRMENAVGRDQNNVGPKVYPIILRLGSPLSLNMARKTLNSLEDKAFQLTPIAVQMTKLATTEELPDEFVVVTVK*"
######################################################################################################################################
ORF_size_dict = Dict{String, Int}()
for (orf, aaseq) in gene_AA_dict
    orf_len = length(aaseq)
    ORF_size_dict[orf] = orf_len
end
###########################################################################################################################################################################
###########################################################################################################################################################################
AA_residues = Set(["A", "C", "D", "E", "F", "G", "H", "I", "K", "L", "M", "N", "P", "Q", "R", "S", "T", "V", "W", "Y", "-", "*", "X"])
aa_pos_comprehensive_dict = Dict{String, Int}()
aa_gene_comprehensive_dict = Dict{String, String}()
aa_gene_and_pos_comprehensive_dict = Dict{String, String}()
refAA_comprehensive_dict = Dict{String, String}()
qryAA_comprehensive_dict = Dict{String, String}()
global nonspike_muts = Set{String}()
for (orf, orf_len) in ORF_size_dict
    for aa1 in AA_residues
        for aa2 in AA_residues
            for i in 1:orf_len
                orf_mut = "$(orf):$(aa1)$(i)$(aa2)"
                orf_pos_mut = "$(orf):$(i)"
                aa_pos_comprehensive_dict[orf_mut] = i
                aa_pos_comprehensive_dict[orf_pos_mut] = i
                aa_gene_comprehensive_dict[orf_mut] = orf
                aa_gene_comprehensive_dict[orf_pos_mut] = orf
                aa_gene_and_pos_comprehensive_dict[orf_mut] = orf_pos_mut
                aa_gene_and_pos_comprehensive_dict[orf_pos_mut] = orf_pos_mut
                if orf ≠ "S"
                    push!(nonspike_muts, orf_mut)
                    push!(nonspike_muts, orf_pos_mut)
                end
                refAA_comprehensive_dict[orf_mut] = aa1
                qryAA_comprehensive_dict[orf_mut] = aa2
            end
        end
    end
end
aa_gene_comprehensive_dict["NTD_disulfide"] = "S"
aa_pos_comprehensive_dict["NTD_disulfide"] = 1
aa_gene_and_pos_comprehensive_dict["NTD_disulfide"] = "S:NTD_disulfide"
refAA_comprehensive_dict["NTD_disulfide"] = "NTDdisulfide"
qryAA_comprehensive_dict["NTD_disulfide"] = "NTD_disulfide"
######################################################## Below: 100% comprehensive ref_nuc & qry_nuc dicts #################################################################
ref_nuc_comprehensive_dict = Dict{String, String}()
qry_nuc_comprehensive_dict = Dict{String, String}()
nuc_mut_int_comprehensive_dict = Dict{String, Int}()
nuc_mut_int_string_comprehensive_dict = Dict{String, String}()
nuc_residues1 = ["T", "C", "A", "G"]
nuc_residues2 = ["T", "C", "A", "G", "Y", "R", "K", "W", "M", "S", "-", "N"]
for nres1 in nuc_residues1
    for nres2 in nuc_residues2
        for i in 1:30000
            mut = "$(nres1)$(i)$(nres2)"
            nucmpos = i
            ref_nuc_comprehensive_dict[mut] = nres1
            qry_nuc_comprehensive_dict[mut] = nres2
            nuc_mut_int_comprehensive_dict[mut] = i
            nuc_mut_int_string_comprehensive_dict[mut] = string(i)
        end
    end
end
##########################################################################################################################################################################
##########################################################################################################################################################################
NSP_AA_size = Dict{String, Int}("NSP1"=>180, "NSP2"=>638, "NSP3"=>1945, "NSP4"=>500, "NSP5"=>306, "NSP6"=>290, "NSP7"=>83, "NSP8"=>198, "NSP9"=>113, "NSP10"=>139, "NSP11"=>0, "NSP12"=>932, "NSP13"=>601, "NSP14"=>527, "NSP15"=>346, "NSP16"=>299)                                                                # "NSP12"=>BitSet(1:923), 
NSP_ranges = Dict{String, String}("NSP1"=>"ORF1a:1-180", "NSP2"=>"ORF1a:181-818", "NSP3"=>"ORF1a:819-2763", "NSP4"=>"ORF1a:2764-3263", "NSP5"=>"ORF1a:3264-3569", "NSP6"=>"ORF1a:3570-3859", "NSP7"=>"ORF1a:3860-3942", "NSP8"=>"ORF1a:3943-4140", "NSP9"=>"ORF1a:4141-4253", "NSP10"=>"ORF1a:4254-4392", "NSP11"=>"", "NSP12"=>"1a:4393-1b:923", "NSP13"=>"ORF1b:924-1524", "NSP14"=>"ORF1b:1525-2051", "NSP15"=>"ORF1b:2052-2397", "NSP16"=>"ORF1b:2398-2696", "S"=>"S:1-1274", "ORF3a"=>"ORF3a:1-276", "E"=>"E:1-76", "M"=>"M:1-223", "ORF6"=>"ORF6:1-62", "ORF7a"=>"ORF7a:1-122", "ORF7b"=>"ORF7b:1-44", "ORF8"=>"ORF8:1-122", "N"=>"N:1-420", "ORF9b"=>"ORF9b:1-98")
NSP_ranges_num_only = Dict{String, BitSet}("NSP1"=>BitSet(1:180), "NSP2"=>BitSet(181:818), "NSP3"=>BitSet(819:2763), "NSP4"=>BitSet(2764:3263), "NSP5"=>BitSet(3264:3569), "NSP6"=>BitSet(3570:3859), "NSP7"=>BitSet(3860:3942), "NSP8"=>BitSet(3943:4140), "NSP9"=>BitSet(4141:4253), "NSP10"=>BitSet(4254:4392), "NSP11"=>BitSet(), "NSP12"=>BitSet([4393:4401; 1:923]), "NSP13"=>BitSet(924:1524), "NSP14"=>BitSet(1525:2051), "NSP15"=>BitSet(2052:2397), "NSP16"=>BitSet(2398:2696), "S"=>BitSet(1:1274), "ORF3a"=>BitSet(1:276), "E"=>BitSet(1:76), "M"=>BitSet(1:223), "ORF6"=>BitSet(1:62), "ORF7a"=>BitSet(1:122), "ORF7b"=>BitSet(1:44), "ORF8"=>BitSet(1:122), "N"=>BitSet(1:420), "ORF9b"=>BitSet(1:98))
NSP_ranges1a = Dict{Int, BitSet}(1=>BitSet(1:180), 2=>BitSet(181:818), 3=>BitSet(819:2763), 4=>BitSet(2764:3263), 5=>BitSet(3264:3569), 6=>BitSet(3570:3859), 7=>BitSet(3860:3942), 8=>BitSet(3943:4140), 9=>BitSet(4141:4253), 10=>BitSet(4254:4392), 12=>BitSet(4393:4401))
NSP_ranges1b = Dict{Int, BitSet}(12=>BitSet(1:923), 13=>BitSet(924:1524), 14=>BitSet(1525:2051), 15=>BitSet(2052:2397), 16=>BitSet(2398:2696))
NSP1a_add = Dict{Int, Int}(1=>0, 2=>180, 3=>818, 4=>2763, 5=>3263, 6=>3569, 7=>3859, 8=>3942, 9=>4140, 10=>4253, 11=>0, 12=>4392)
NSP1b_add = Dict{Int, Int}(12=>-9, 13=>923, 14=>1524, 15=>2051, 16=>2397)
NSP1ab_add = Dict{Int, Int}(1=>0, 2=>180, 3=>818, 4=>2763, 5=>3263, 6=>3569, 7=>3859, 8=>3942, 9=>4140, 10=>4353, 11=>0, 12=>-9, 13=>923, 14=>1524, 15=>2051, 16=>2397)
######################################################################################################################################
NSP_muts_pos_dict = Dict{String, Int}()
NSP_muts_gene_dict = Dict{String, String}()
NSP_ref_AA_dict = Dict{String, String}()
NSP_qry_AA_dict = Dict{String, String}()
NSP_set = Set(["NSP1", "NSP2", "NSP3", "NSP4", "NSP5", "NSP6", "NSP7", "NSP8", "NSP9", "NSP10", "NSP12", "NSP13", "NSP14", "NSP15", "NSP16"])
for nsp in NSP_set
    nsp_len = NSP_AA_size[nsp]
    for aa1 in AA_residues
        for aa2 in AA_residues
            for i in 1:nsp_len
                nspmut = "$(nsp):$(aa1)$(i)$(aa2)"
                nsppos = "$(nsp):$(i)"
                NSP_muts_gene_dict[nspmut] = nsp
                NSP_muts_pos_dict[nspmut] = i
                NSP_muts_gene_dict[nsppos] = nsp
                NSP_muts_pos_dict[nsppos] = i
                NSP_ref_AA_dict[nspmut] = aa1
                NSP_qry_AA_dict[nspmut] = aa2  
            end
        end
    end
end
###########################################################################################################################################################################
function AAmut_to_AApos(m)
    gn = split(m, ":")[1]
    mt = split(m, ":")[2]
    pos = mt[2:end-1]
    AApos = gn*":"*pos
    return AApos
end
########################################################################################################################################################################
function mixed2nuc(mix_mut)
    mut = mix_mut
    qry_nuc = qry_nuc_comprehensive_dict[mut]
    ref_nuc = ref_nuc_comprehensive_dict[mut]
    nuc_int = nuc_mut_int_comprehensive_dict[mut]
    ref_n_pos = "$(ref_nuc)$(nuc_int)"
    if length(mix_mut) ≥ 4
        if qry_nuc == "T"
            mut = mix_mut
        elseif qry_nuc == "C"
            mut = mix_mut
        elseif qry_nuc == "A"
            mut = mix_mut
        elseif qry_nuc == "G"
            mut = mix_mut
        elseif qry_nuc == "N"
            mut = mix_mut
        end   
        if ref_nuc == "T"
            if qry_nuc == "Y"
                mut = ref_n_pos*"C"
            elseif qry_nuc == "W"
                mut = ref_n_pos*"A"
            elseif qry_nuc == "K"
                mut = ref_n_pos*"G"
            elseif qry_nuc == "M"
                mut = "$(ref_n_pos)C, $(ref_n_pos)A"
            elseif qry_nuc == "S"
                mut = "$(ref_n_pos)C, $(ref_n_pos)G"
            elseif qry_nuc == "R"
                mut = "$(ref_n_pos)A, $(ref_n_pos)G"
            end
        end
        if ref_nuc == "C"
            if qry_nuc == "Y"
                mut = ref_n_pos*"T"
            elseif qry_nuc == "M"
                mut = ref_n_pos*"A"
            elseif qry_nuc == "S"
                mut = ref_n_pos*"G"
            elseif qry_nuc == "R"
                mut = "$(ref_n_pos)A, $(ref_n_pos)G"
            elseif qry_nuc == "W"
                mut = "$(ref_n_pos)T, $(ref_n_pos)A"
            elseif qry_nuc == "K"
                mut = "$(ref_n_pos)T, $(ref_n_pos)G"
            end
        end
        if ref_nuc == "A"
            if qry_nuc == "R"
                mut = ref_n_pos*"G"
            elseif qry_nuc == "W"
                mut = ref_n_pos*"T"
            elseif qry_nuc == "M"
                mut = ref_n_pos*"C"
            elseif qry_nuc == "Y"
                mut = "$(ref_n_pos)T, $(ref_n_pos)C"
            elseif qry_nuc == "K"
                mut = "$(ref_n_pos)T, $(ref_n_pos)G"
            elseif qry_nuc == "S"
                mut = "$(ref_n_pos)C, $(ref_n_pos)G"
            end
        end
        if ref_nuc == "G"
            if qry_nuc == "R"
                mut = ref_n_pos*"A"
            elseif qry_nuc == "K"
                mut = ref_n_pos*"T"
            elseif qry_nuc == "S"
                mut = ref_n_pos*"C"
            elseif qry_nuc == "Y"
                mut = "$(ref_n_pos)T, $(ref_n_pos)C"
            elseif qry_nuc == "W"
                mut = "$(ref_n_pos)T, $(ref_n_pos)A"
            elseif qry_nuc == "M"
                mut = "$(ref_n_pos)C, $(ref_n_pos)A"
            end
        end
    end
    return mut
end
#####################################################################################################################################
####################### Making gene AA & nuc references for all designated variants #################################################
#####################################################################################################################################
gene_AA_pango_dict = Dict{String, Dict{String, String}}()
nuc_genome_pango_dict = Dict{String, String}()
pango_consensus_set = Set{String}()
headers1a, seqs1a = read_fasta("___pango_consensus_sequences/pango_consensus_AA_ORF1a_2025_06_25_NNL.fasta")
for i in 1:length(headers1a)
    pango = headers1a[i]
    push!(pango_consensus_set, pango)
end
for pango in pango_consensus_set
    gene_AA_pango_dict[pango] = Dict{String, String}()
end
################################################################################################
for gene in gene_array
    aa_file = "___pango_consensus_sequences/pango_consensus_AA_$(gene)_2025_06_25_NNL.fasta"
    headers, seqs = read_fasta(aa_file)
    for i in 1:length(headers)
        pango = headers[i]
        aa_seq = seqs[i]
        gene_AA_pango_dict[pango][gene] = aa_seq
    end
end
nuc_file = "___pango_consensus_sequences/pango_consensus_sequences_genome_nuc_2025_06_25_NNL.fasta"
nuc_headers, nuc_seqs = read_fasta(nuc_file)
for i in 1:length(nuc_headers)
    pango = nuc_headers[i]
    nuc_seq = nuc_seqs[i]
    if length(nuc_seq) ≥ 28000
        nuc_genome_pango_dict[pango] = nuc_seq
    end
end
seqs_in_nuc_genome_pango_dict = length(nuc_genome_pango_dict)
println("seqs_in_nuc_genome_pango_dict = $(seqs_in_nuc_genome_pango_dict)")
######################################################################################################################################
######################################################################################################################################
gene_order_dict = Dict{String, Int}("ORF1a"=>1, "ORF1b"=>2, "S"=>3, "ORF3a"=>4, "E"=>4, "M"=>6, "ORF6"=>7, "ORF7a"=>8, "ORF7b"=>9, "ORF8"=>10, "N"=>11, "ORF9b"=>12)
gene_AA_sortKey(m) = (gene_order_dict[aa_gene_comprehensive_dict[m]], aa_pos_comprehensive_dict[m])
gene_AApos_sortKey(m) = (gene_order_dict[aa_gene_comprehensive_dict[m]], aa_pos_comprehensive_dict[m])
######################################################################################################################################
gene_hydrophobic_dict = Dict{String, Float64}()
for (gene, aaseq) in gene_AA_dict
    gene_hydrophobe_sum = 0
    for aa in aaseq
        hydrophobe_score = hydrophobic_index_dict[string(aa)]
        gene_hydrophobe_sum += hydrophobe_score
    end
    gene_hydrophobe_score = gene_hydrophobe_sum/length(aaseq)
    gene_hydrophobic_dict[gene] = gene_hydrophobe_score
end 
######################################################################################################################################
AA_res_set = Set(["A", "C", "D", "E", "F", "G", "H", "I", "K", "L", "M", "N", "P", "Q", "R", "S", "T", "V", "W", "Y", "-"])
AA_res_set_noDel = Set(["A", "C", "D", "E", "F", "G", "H", "I", "K", "L", "M", "N", "P", "Q", "R", "S", "T", "V", "W", "Y"])
AA_res_pairs = Set{Tuple{String, String}}()
for res1 in AA_res_set_noDel
    for res2 in AA_res_set
        push!(AA_res_pairs, (res1, res2) )
    end
end
###########################################################################################################################################################################
AA_triplets = Dict{String, String}("TTT"=>"F", "TTC"=>"F", "TTA"=>"L", "TTG"=>"L", "TCT"=>"S", "TCC"=>"S", "TCA"=>"S", "TCG"=>"S", "TAT"=>"Y", "TAC"=>"Y", "TAA"=>"*", "TAG"=>"*", "TGT"=>"C", "TGC"=>"C", "TGA"=>"*", "TGG"=>"W", "CTT"=>"L", "CTC"=>"L", "CTA"=>"L", "CTG"=>"L", "CCT"=>"P", "CCC"=>"P", "CCA"=>"P", "CCG"=>"P", "CAT"=>"H", "CAC"=>"H", "CAA"=>"Q", "CAG"=>"Q", "CGT"=>"R", "CGC"=>"R", "CGA"=>"R", "CGG"=>"R", "ATT"=>"I", "ATC"=>"I", "ATA"=>"I", "ATG"=>"M", "ACT"=>"T", "ACC"=>"T", "ACA"=>"T", "ACG"=>"T", "AAT"=>"N", "AAC"=>"N", "AAA"=>"K", "AAG"=>"K", "AGT"=>"S", "AGC"=>"S", "AGA"=>"R", "AGG"=>"R", "GTT"=>"V", "GTC"=>"V", "GTA"=>"V", "GTG"=>"V", "GCT"=>"A", "GCC"=>"A", "GCA"=>"A", "GCG"=>"A", "GAT"=>"D", "GAC"=>"D", "GAA"=>"E", "GAG"=>"E", "GGT"=>"G", "GGC"=>"G", "GGA"=>"G", "GGG"=>"G", "TT-"=>"X", "TC-"=>"X", "TA-"=>"X", "TG-"=>"X", "T-T"=>"X", "T-C"=>"X", "T-A"=>"X", "T-G"=>"X", "T--"=>"X", "CT-"=>"X", "CC-"=>"X", "CA-"=>"X", "CG-"=>"X", "C-T"=>"X", "C-C"=>"X", "C-A"=>"X", "C-G"=>"X", "C--"=>"X", "AT-"=>"X", "AC-"=>"X", "AA-"=>"X", "AG-"=>"X", "A-T"=>"X", "A-C"=>"X", "A-A"=>"X", "A-G"=>"X", "A--"=>"X", "GT-"=>"X", "GC-"=>"X", "GA-"=>"X", "GG-"=>"X", "G-T"=>"X", "G-C"=>"X", "G-A"=>"X", "G-G"=>"X", "G--"=>"X", "-TT"=>"X", "-TC"=>"X", "-TA"=>"X", "-TG"=>"X", "-T-"=>"X", "-CT"=>"X", "-CC"=>"X", "-CA"=>"X", "-CG"=>"X", "-C-"=>"X", "-AT"=>"X", "-AC"=>"X", "-AA"=>"X", "-AG"=>"X", "-A-"=>"X", "-GT"=>"X", "-GC"=>"X", "-GA"=>"X", "-GG"=>"X", "-G-"=>"X", "--T"=>"X", "--C"=>"X", "--A"=>"X", "--G"=>"X", "---"=>"X", "NTT"=>"X", "TNT"=>"X", "TTN"=>"X", "NTC"=>"X", "TNC"=>"X", "TCN"=>"X", "NTA"=>"X", "TNA"=>"X", "TAN"=>"X", "NTG"=>"X", "TNG"=>"X", "TGN"=>"X", "NT-"=>"X", "TN-"=>"X", "T-N"=>"X", "NTN"=>"X", "TNN"=>"X", "NCT"=>"X", "CNT"=>"X", "CTN"=>"X", "NCC"=>"X", "CNC"=>"X", "CCN"=>"X", "NCA"=>"X", "CNA"=>"X", "CAN"=>"X", "NCG"=>"X", "CNG"=>"X", "CGN"=>"X", "NC-"=>"X", "CN-"=>"X", "C-N"=>"X", "NCN"=>"X", "CNN"=>"X", "NAT"=>"X", "ANT"=>"X", "ATN"=>"X", "NAC"=>"X", "ANC"=>"X", "ACN"=>"X", "NAA"=>"X", "ANA"=>"X", "AAN"=>"X", "NAG"=>"X", "ANG"=>"X", "AGN"=>"X", "NA-"=>"X", "AN-"=>"X", "A-N"=>"X", "NAN"=>"X", "ANN"=>"X", "NGT"=>"X", "GNT"=>"X", "GTN"=>"X", "NGC"=>"X", "GNC"=>"X", "GCN"=>"X", "NGA"=>"X", "GNA"=>"X", "GAN"=>"X", "NGG"=>"X", "GNG"=>"X", "GGN"=>"X", "NG-"=>"X", "GN-"=>"X", "G-N"=>"X", "NGN"=>"X", "GNN"=>"X", "N-T"=>"X", "-NT"=>"X", "-TN"=>"X", "N-C"=>"X", "-NC"=>"X", "-CN"=>"X", "N-A"=>"X", "-NA"=>"X", "-AN"=>"X", "N-G"=>"X", "-NG"=>"X", "-GN"=>"X", "N--"=>"X", "-N-"=>"X", "--N"=>"X", "N-N"=>"X", "-NN"=>"X", "NNT"=>"X", "NNC"=>"X", "NNA"=>"X", "NNG"=>"X", "NN-"=>"X", "NNN"=>"X")                 
AA_triplet_dels = Dict{String, String}("TTT"=>"F", "TTC"=>"F", "TTA"=>"L", "TTG"=>"L", "TCT"=>"S", "TCC"=>"S", "TCA"=>"S", "TCG"=>"S", "TAT"=>"Y", "TAC"=>"Y", "TAA"=>"*", "TAG"=>"*", "TGT"=>"C", "TGC"=>"C", "TGA"=>"*", "TGG"=>"W", "CTT"=>"L", "CTC"=>"L", "CTA"=>"L", "CTG"=>"L", "CCT"=>"P", "CCC"=>"P", "CCA"=>"P", "CCG"=>"P", "CAT"=>"H", "CAC"=>"H", "CAA"=>"Q", "CAG"=>"Q", "CGT"=>"R", "CGC"=>"R", "CGA"=>"R", "CGG"=>"R", "ATT"=>"I", "ATC"=>"I", "ATA"=>"I", "ATG"=>"M", "ACT"=>"T", "ACC"=>"T", "ACA"=>"T", "ACG"=>"T", "AAT"=>"N", "AAC"=>"N", "AAA"=>"K", "AAG"=>"K", "AGT"=>"S", "AGC"=>"S", "AGA"=>"R", "AGG"=>"R", "GTT"=>"V", "GTC"=>"V", "GTA"=>"V", "GTG"=>"V", "GCT"=>"A", "GCC"=>"A", "GCA"=>"A", "GCG"=>"A", "GAT"=>"D", "GAC"=>"D", "GAA"=>"E", "GAG"=>"E", "GGT"=>"G", "GGC"=>"G", "GGA"=>"G", "GGG"=>"G", "TT-"=>"X", "TC-"=>"X", "TA-"=>"X", "TG-"=>"X", "T-T"=>"X", "T-C"=>"X", "T-A"=>"X", "T-G"=>"X", "T--"=>"-", "CT-"=>"X", "CC-"=>"X", "CA-"=>"X", "CG-"=>"X", "C-T"=>"X", "C-C"=>"X", "C-A"=>"X", "C-G"=>"X", "C--"=>"-", "AT-"=>"X", "AC-"=>"X", "AA-"=>"X", "AG-"=>"X", "A-T"=>"X", "A-C"=>"X", "A-A"=>"X", "A-G"=>"X", "A--"=>"-", "GT-"=>"X", "GC-"=>"X", "GA-"=>"X", "GG-"=>"X", "G-T"=>"X", "G-C"=>"X", "G-A"=>"X", "G-G"=>"X", "G--"=>"-", "-TT"=>"X", "-TC"=>"X", "-TA"=>"X", "-TG"=>"X", "-T-"=>"X", "-CT"=>"X", "-CC"=>"X", "-CA"=>"X", "-CG"=>"X", "-C-"=>"X", "-AT"=>"X", "-AC"=>"X", "-AA"=>"X", "-AG"=>"X", "-A-"=>"X", "-GT"=>"X", "-GC"=>"X", "-GA"=>"X", "-GG"=>"X", "-G-"=>"X", "--T"=>"-", "--C"=>"-", "--A"=>"-", "--G"=>"-", "---"=>"-", "NTT"=>"X", "TNT"=>"X", "TTN"=>"X", "NTC"=>"X", "TNC"=>"X", "TCN"=>"X", "NTA"=>"X", "TNA"=>"X", "TAN"=>"X", "NTG"=>"X", "TNG"=>"X", "TGN"=>"X", "NT-"=>"X", "TN-"=>"X", "T-N"=>"X", "NTN"=>"X", "TNN"=>"X", "NCT"=>"X", "CNT"=>"X", "CTN"=>"X", "NCC"=>"X", "CNC"=>"X", "CCN"=>"X", "NCA"=>"X", "CNA"=>"X", "CAN"=>"X", "NCG"=>"X", "CNG"=>"X", "CGN"=>"X", "NC-"=>"X", "CN-"=>"X", "C-N"=>"X", "NCN"=>"X", "CNN"=>"X", "NAT"=>"X", "ANT"=>"X", "ATN"=>"X", "NAC"=>"X", "ANC"=>"X", "ACN"=>"X", "NAA"=>"X", "ANA"=>"X", "AAN"=>"X", "NAG"=>"X", "ANG"=>"X", "AGN"=>"X", "NA-"=>"X", "AN-"=>"X", "A-N"=>"X", "NAN"=>"X", "ANN"=>"X", "NGT"=>"X", "GNT"=>"X", "GTN"=>"X", "NGC"=>"X", "GNC"=>"X", "GCN"=>"X", "NGA"=>"X", "GNA"=>"X", "GAN"=>"X", "NGG"=>"X", "GNG"=>"X", "GGN"=>"X", "NG-"=>"X", "GN-"=>"X", "G-N"=>"X", "NGN"=>"X", "GNN"=>"X", "N-T"=>"X", "-NT"=>"X", "-TN"=>"X", "N-C"=>"X", "-NC"=>"X", "-CN"=>"X", "N-A"=>"X", "-NA"=>"X", "-AN"=>"X", "N-G"=>"X", "-NG"=>"X", "-GN"=>"X", "N--"=>"X", "-N-"=>"X", "--N"=>"X", "N-N"=>"X", "-NN"=>"X", "NNT"=>"X", "NNC"=>"X", "NNA"=>"X", "NNG"=>"X", "NN-"=>"X", "NNN"=>"X")                 
###########################################################################################################################################################################
nonleap_month_day_dict = Dict{Int, Int}(0=>0, 1=>31, 2=>28, 3=>31, 4=>30, 5=>31, 6=>30, 7=>31, 8=>31, 9=>30, 10=>31, 11=>30, 12=>31)
leap_month_day_dict = Dict{Int, Int}(0=>0, 1=>31, 2=>29, 3=>31, 4=>30, 5=>31, 6=>30, 7=>31, 8=>31, 9=>30, 10=>31, 11=>30, 12=>31)
index_to_tuple = Dict{Int, Tuple{Int, Int, Int}}()
tuple_to_index = Dict{Tuple{Int, Int, Int}, Int}()
index = 0
for year in 2020:2027
    for month in 1:12
        if year%4 == 0
            month_days = leap_month_day_dict[month]
            for day in 1:month_days
                index += 1
                index_to_tuple[index] = (year, month, day)
            end
        else
            month_days = nonleap_month_day_dict[month]
            for day in 1:month_days
                index += 1
                index_to_tuple[index] = (year, month, day)
            end
        end
    end
end
for (index, date) in index_to_tuple
    tuple_to_index[date] = index
end
for y in 2020:2027
    tuple_to_index[(y, 0, 0)] = y*1000000
    index_to_tuple[y*1000000] = (y, 0, 0)
    for m in 1:12
        tuple_to_index[(y, m, 0)] = y*1000000 + m*1000
        index_to_tuple[y*1000000 + m*1000] = (y, m, 0)
    end
end
tuple_to_index[(0, 0, 0)] = 1000000000
index_to_tuple[1000000000] = (0, 0, 0)
############################################################################################################################################################################
############################################################################################################################################################################
function mut_ct_by_date_range_all_UNIVERSAL(date1::Int, date2::Int, all_dict::Dict{Int, Dict{String, Int}})
    for i in 1:3000
        if !haskey(all_dict, i)
            all_dict[i] = Dict{String, Int}()
        end
    end
    date1_to_date2_ct = Dict{String, Int}()
    for i in date1:date2
        for (mut, count) in all_dict[i]
            date1_to_date2_ct[mut] = get!(date1_to_date2_ct, mut, 0) + count
        end
    end
    return date1_to_date2_ct
end
############################################################################################################################################################################
function nuc_mut_pos_only_ct_by_date_range_all_UNIVERSAL(date1::Int, date2::Int, all_dict::Dict{Int, Dict{Int, Int}})
    for i in 1:3000
        if !haskey(all_dict, i)
            all_dict[i] = Dict{Int, Int}()
        end
    end
    date1_to_date2_ct = Dict()
    for i in date1:date2
        for (mut, count) in all_dict[i]
            date1_to_date2_ct[mut] = get!(date1_to_date2_ct, mut, 0) + count
        end
    end
    return date1_to_date2_ct
end
###########################################################################################################################################################################
###########################################################################################################################################################################
function ORF1abMut_to_NSP(mut::String)
    NSPmut = ""
#    NSP_muts = Dict("NSP$(i)" => Dict{String, Int}() for i in 1:16 if i ≠ 11)
    gene = aa_gene_comprehensive_dict[mut]
    pos = aa_pos_comprehensive_dict[mut]
    refAA = refAA_comprehensive_dict[mut]
    qryAA = qryAA_comprehensive_dict[mut]
    if gene == "ORF1a"
        for (NSP, range) in NSP_ranges1a
            if pos in range
                NSPpos = pos - NSP1a_add[NSP]
                NSPmut = "NSP$(NSP):$(refAA)$(NSPpos)$(qryAA)"
            end
        end
    end
    if gene == "ORF1b"
        for (NSP, range) in NSP_ranges1b
            if pos in range
                NSPpos = pos - NSP1b_add[NSP]
                NSPmut = "NSP$(NSP):$(refAA)$(NSPpos)$(qryAA)"
            end
        end
    end
    return NSPmut
end
#####################################################################################################################################
function NSPmut_to_ORF1ab(NSPmut::String)
    ORFmut = ""
    ORFpos = ""
    NSP_num = parse(Int, split(NSPmut, ":")[1][4:end])
    NSP_pos = NSP_muts_pos_dict[NSPmut]
    refAA = NSP_ref_AA_dict[NSPmut]
    qryAA = NSP_qry_AA_dict[NSPmut]
    if NSPnum in [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
        ORF1_pos = NSP1a_add[NSP_num] + NSP_pos
        ORFmut = "ORF1a:$(refAA)$(ORF1_pos)$(qryAA)"
    end
    if NSPnum in [13, 14, 15, 16]
        ORF1_pos = NSP1b_add[NSP_num] + NSP_pos
        ORFmut = "ORF1b:$(refAA)$(ORF1_pos)$(qryAA)"
    end
    if NSP_num == 12
        if NSP_pos in [1:8...]
            ORF1_pos = NSP1a_add[NSP_num] + NSP_pos
            ORFmut = "ORF1a:$(refAA)$(ORF1_pos)$(qryAA)"
        end
        if NSP_pos in [10:932...]
            ORF1_pos = NSP1b_add[NSP_num] + NSP_pos
            ORFmut = "ORF1b:$(refAA)$(ORF1_pos)$(qryAA)"
        end
    end
    ORF1pos = parse(Int, ORFpos)
    return ORFmut
end
#####################################################################################################################################
function NSP_range_to_ORF1ab_range(NSP_range::String)
    NSP = split(NSP_range, ":")[1]
    if length(NSP) < 3
        return ""
    end
    if NSP[1:3] ≠ "NSP"
        return ""
    end
    NSP_int = parse(Int, NSP[4:end])
    range = split(NSP_range, ":")[2]
    NSPpos1_int = parse(Int, split(range, "-")[1])
    NSPpos2_int = parse(Int, split(range, "-")[2])
    ORF_int1 = 0
    ORF_int2 = 0
    ORF1_AorB_pos1 = ""
    ORF1_AorB_pos2 = ""
    if NSP_int in [1:10...]
        ORF_int1 = NSPpos1_int + NSP1a_add[NSP_int]
        ORF_int2 = NSPpos2_int + NSP1a_add[NSP_int]
        ORF1_AorB_pos1 = "ORF1a"
        ORF1_AorB_pos2 = ""
    end
    if NSP_int in [13:16...]
        ORF_int1 = NSPpos1_int + NSP1b_add[NSP_int]
        ORF_int2 = NSPpos2_int + NSP1b_add[NSP_int]
        ORF1_AorB_pos1 = "ORF1b"
        ORF1_AorB_pos2 = ""
    end
    if NSP_int == 12
        if NSPpos1_int in [1:9...]
            ORF_int1 = NSPpos1_int + NSP1a_add[NSP_int]
            ORF1_AorB_pos1 = "1a"
        else
            ORF_int1 = NSPpos1_int + NSP1b_add[NSP_int]
            ORF1_AorB_pos1 = "ORF1b"
        end
        if NSPpos2_int in [1:9...]
            ORF_int2 = NSPpos2_int + NSP1a_add[NSP_int]
            ORF1_AorB_pos2 = "1a"
        end
        if !(NSPpos2_int in [1:9...])
            ORF_int2 = NSPpos2_int + NSP1b_add[NSP_int]
            if ORF1_AorB_pos1 == "1a"
                ORF1_AorB_pos2 = "1b:"
            else
            ORF1_AorB_pos2 = ""
            end
        end
    end
    ORF_int1_str = string(ORF_int1)
    ORF_int2_str = string(ORF_int2)
    ORF_range = ORF1_AorB_pos1*":"*ORF_int1_str*"-"*ORF1_AorB_pos2*ORF_int2_str
    return ORF_range
end
#####################################################################################################################################
function ORF1ab_range_to_NSP_range(ab_range::String)
    ab = split(ab_range, ":")[1]
    range = split(ab_range, ":")[2]
    pos1 = split(range, "-")[1]
    pos2 = split(range, "-")[2]
    pos1_int = parse(Int, pos1)
    pos2_int = parse(Int, pos2)
    NSPint1 = 0
    NSPint2 = 0
    NSPpos1 = ""
    NSPpos2 = ""
    NSPrange = ""
    NSPrange_pt1 = ""
    top_ct = 0
    if ab == "ORF1a"
        ct = 0
        for (NSP, rng) in NSP_ranges1a
            if pos1_int in rng && pos2_int in rng
                NSPint1 = pos1_int - NSP1a_add[NSP]
                NSPint2 = pos2_int - NSP1a_add[NSP]
                NSPpos1 = string(NSPint1)
                NSPpos2 = string(NSPint2)
                NSPstr = string(NSP)
                NSPrange = "NSP"*NSPstr*":"*NSPpos1*"-"*NSPpos2
            end
            if pos1_int in rng && !(pos2_int in rng)
                ct += 1
                NSPint1 = pos1_int - NSP1a_add[NSP]
                NSPpos1 = string(NSPint1)
                NSPstr = string(NSP)
                NSPrange_pt1 = "NSP"*NSPstr*":"*NSPpos1*"-"
            end
            if ct > 0
                top_ct += 1
                if pos2_int in rng
                    NSPint2 = pos2_int - NSP1a_add[NSP]
                    NSPpos2 = string(NSPint2)
                    NSPstr = string(NSP)
                    NSPrange_pt2 = "NSP"*NSPstr*":"*NSPpos2
                    NSPrange = NSPrange_pt1*NSPrange_pt2
                end
            end
        end
    end
    if ab == "ORF1b"
        ct = 0
        for (NSP, rng) in NSP_ranges1b
            if pos1_int in rng && pos2_int in rng
                NSPint1 = pos1_int - NSP1b_add[NSP]
                NSPint2 = pos2_int - NSP1b_add[NSP]
                NSPpos1 = string(NSPint1)
                NSPpos2 = string(NSPint2)
                NSPstr = string(NSP)
                NSPrange = "NSP"*NSPstr*":"*NSPpos1*"-"*NSPpos2
            end
            if pos1_int in rng && !(pos2_int in rng)
                ct += 1
                NSPint1 = pos1_int - NSP1b_add[NSP]
                NSPpos1 = string(NSPint1)
                NSPstr = string(NSP)
                NSPrange_pt1 = "NSP"*NSPstr*":"*NSPpos1*"-"
            end
            if ct > 0
                top_ct += 1
                if pos2_int in rng
                    NSPint2 = pos2_int - NSP1b_add[NSP]
                    NSPpos2 = string(NSPint2)
                    NSPstr = string(NSP)
                    NSPrange_pt2 = "NSP"*NSPstr*":"*NSPpos2
                    NSPrange = NSPrange_pt1*NSPrange_pt2
                end
            end
        end
    end 
    return NSPrange
end
######################################################################################################################################
function multiepi_to_epis(multi)  
    epi_num_only_pre(n) = split(n, "_")[3]
    epi_num_only_first(n) = parse(Int, split(epi_num_only_pre(n), "-")[1])
    epi_num_only_last(n) = parse(Int, split(epi_num_only_pre(n), "-")[2])
    first = epi_num_only_first(multi)
    last = epi_num_only_last(multi)
    epi_arr = Vector{String}()
    for i in first:last
        i_str = string(i)
        epi = "EPI_ISL_"*i_str
        push!(epi_arr, epi)
    end
    return epi_arr
end
######################################################################################################################################
function stringlist_to_strings(txt::String)
    epi_num_only_pre(n) = split(n, "_")[3]
    function epi_sortkey(epi)
        epinum = epi_num_only_pre(epi)
        epi_key = (length(epinum), epinum)
        return epi_key
    end
    arr_of_strings1 = Vector{String}()
    arr_of_strings2 = Vector{String}()
    no_newlines = replace(txt, "\n" =>" ")
    for seq in split(no_newlines, ", ")
        if '-' in seq
            multis = multiepi_to_epis(seq)
            for mseq in multis
                push!(arr_of_strings2, mseq)
            end
        else 
            push!(arr_of_strings2, seq)
        end
    end
    sort_arr_of_strings2 = sort(collect(arr_of_strings2), by = x -> epi_sortkey(x))    
    return sort_arr_of_strings2
end
######################################################################################################################################
function stringlist_to_strings_set(txt::String)
    epi_num_only_pre(n) = split(n, "_")[3]
    function epi_sortkey(epi)
        epinum = epi_num_only_pre(epi)
        epi_key = (length(epinum), epinum)
        return epi_key
    end
    set_of_strings = Set{String}()
    no_newlines = replace(txt, "\n" =>" ")
    for seq in split(no_newlines, ", ")
        if '-' in seq
            multis = multiepi_to_epis(seq)
            for mseq in multis
                push!(set_of_strings, mseq)
            end
        else 
            push!(set_of_strings, seq)
        end
    end
    return set_of_strings
end  
######################################################################################################################################
function stringlist_to_set(txt::String)
    set_of_strings = Set{String}()
    no_newlines = replace(txt, "\n" =>" ")
    for seq in split(no_newlines, ", ")
        if '-' in seq
            multis = multiepi_to_epis(seq)
            for mseq in multis
                push!(set_of_strings, mseq)
            end
        else 
            push!(set_of_strings, seq)
        end
    end   
    return set_of_strings
end
#######################################################################################################################################
function list_to_strings(list::String)
    string_vec = string.(split(list, ", "))
    return string_vec
end
##################################################
function list_to_strings_set(list::String)
    string_vec = string.(split(list, ", "))
    string_set = Set{String}()
    for str in string_vec
        push!(string_set, str)
    end
    return string_set
end
#####################################################################################################################################
function cross_check_set(old_seq_set::Set{String}, new_seq_set::Set{String})
    new_set2 = Set{String}()
    old_set2 = Set{String}()
    old_seq_set_len = length(old_seq_set)
    new_seq_set_len = length(new_seq_set)
    difference = new_seq_set_len - old_seq_set_len
    println("Number of Sequences in Old List = $(old_seq_set_len)")
    println("Number of Sequences in New List = $(new_seq_set_len)")
    println("Difference between Old & New Sets = $(difference)")
    println()
    for seq in new_seq_set
        if !(seq in old_seq_set)
            push!(new_set2, seq)
        end
        if seq in old_seq_set
            push!(old_set2, seq)
        end
    end
    new_len = length(new_set2)
    old_len = length(old_set2)
    if difference == new_len
        println("The numbers check out: difference between old & new sets = Number of new seqs")
    else
        println("The numbers don't check out: difference between old & new sets not equal to Number of new seqs")
    end
    println()
    println("Number of New Sequences = $(new_len)")
    println("Number of Old Sequences = $(old_len)")
    new_set2_sort = sort(collect(new_set2), by = x -> (length(x), x))
    return new_set2, new_set2_sort
end
############################################################################################################################################################################
############################################################################################################################################################################
############################################################################################################################################################################
cell1_runtime = round(digits=1, time() - cell1_start_time)
c1runtime1, c1runtime2 = seconds_to_hrs_min_sec(cell1_runtime)
println("Cell1 Runtime v0 = $(cell1_runtime) seconds")
println("Cell1 Runtime v1 = $(c1runtime1)")
println("Cell1 Runtime v2 = $(c1runtime2)"); println()
######################################################################################################################################

   Resolving package versions...
  No Changes to `~/.julia/environments/v1.11/Project.toml`
  No Changes to `~/.julia/environments/v1.11/Manifest.toml`
   Resolving package versions...
  No Changes to `~/.julia/environments/v1.11/Project.toml`
  No Changes to `~/.julia/environments/v1.11/Manifest.toml`
   Resolving package versions...
  No Changes to `~/.julia/environments/v1.11/Project.toml`
  No Changes to `~/.julia/environments/v1.11/Manifest.toml`
   Resolving package versions...
  No Changes to `~/.julia/environments/v1.11/Project.toml`
  No Changes to `~/.julia/environments/v1.11/Manifest.toml`
   Resolving package versions...
  No Changes to `~/.julia/environments/v1.11/Project.toml`
  No Changes to `~/.julia/environments/v1.11/Manifest.toml`
   Resolving package versions...
  No Changes to `~/.julia/environments/v1.11/Project.toml`
  No Changes to `~/.julia/environments/v1.11/Manifest.toml`
   Resolving package versions...
  No Changes to `~/.julia/environments/v1.11/Project.to

2026_03_08__220PM
/Users/ryhisner
/Users/ryhisner
Time to Load Packages = 0:00:22.38
Time to Load Packages = 0 hr, 0 min, 22.38 sec
seqs_in_nuc_genome_pango_dict = 3586
Cell1 Runtime v0 = 63.7 seconds
Cell1 Runtime v1 = 0:01:03.70
Cell1 Runtime v2 = 0 hr, 1 min, 03.70 sec



In [99]:
### Fx: load_all_seq_dicts, 2026_03_08  |  Loads HQCS datasets  | many large dictionaries not needed & therefore commented out  
load_all_start = time();   date_now = Dates.format(now(), "yyyy_mm_dd__IMMp"); println(date_now)
function load_all_seq_dicts(date::String, fx_name::String, ndjson_name::String, max_AA_mut::Int, revs_thresh::Int, qc_max::Int)
    start_all = time()
    filename = "$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)"
############################################################################################################################################################################    
    global seq_ct_by_year_all = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_seq_ct_by_year.jld2", "seq_ct_by_year")
    global seq_ct_by_year_month_all = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_seq_ct_by_year_month.jld2", "seq_ct_by_year_month")
    global seq_ct_by_year_month_day_all = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_seq_ct_by_year_month_day.jld2", "seq_ct_by_year_month_day")
    global pango_date_index_ct = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_pango_date_index_ct.jld2", "pango_date_index_ct")
############################################################################################################################################################################
    global avg_private_AA_per_circ_seq = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_avg_private_AA_per_circ_seq.jld2", "avg_private_AA_per_circ_seq")
    global all_seq_ct_all = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_all_seq_ct.jld2", "all_seq_ct")
    global qualifying_seq_ct_all = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_qualifying_seq_ct.jld2", "qualifying_seq_ct")
    global nuc_muts_ct_all = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_nuc_muts_ct.jld2", "nuc_muts_ct")
    global nuc_muts_ct_no_dels_all = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_nuc_muts_ct_no_dels.jld2", "nuc_muts_ct_no_dels")
    global AA_muts_ct_all = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_AA_muts_ct.jld2", "AA_muts_ct")
    global AA_dels_ct_all = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_AA_dels_ct.jld2", "AA_dels_ct")
    global AA_muts_ct_no_dels_all = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_AA_muts_ct_no_dels.jld2", "AA_muts_ct_no_dels")
    global AA_muts_ct_pos_only_all = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_AA_muts_ct_pos_only.jld2", "AA_muts_ct_pos_only")
    global AA_muts_ct_pos_only_no_dels_all = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_AA_muts_ct_pos_only_no_dels.jld2", "AA_muts_ct_pos_only_no_dels")
    global gene_mut_density_all = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_gene_mut_density.jld2", "gene_mut_density")
    global domain_mut_density_all = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_domain_mut_density.jld2", "domain_mut_density")
    global nuc_muts_ct_sort_all = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_nuc_muts_ct_sort.jld2", "nuc_muts_ct_sort")
    global nuc_muts_ct_sort_by_seq_ct_all = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_nuc_muts_ct_sort_by_seq_ct.jld2", "nuc_muts_ct_sort_by_seq_ct")
    global nuc_muts_ct_no_dels_sort_all = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_nuc_muts_ct_no_dels_sort.jld2", "nuc_muts_ct_no_dels_sort")
    global nuc_muts_ct_no_dels_sort_by_seq_ct_all = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_nuc_muts_ct_no_dels_sort_by_seq_ct.jld2", "nuc_muts_ct_no_dels_sort_by_seq_ct")
    global AA_muts_ct_sort_all = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_AA_muts_ct_sort.jld2", "AA_muts_ct_sort")
    global AA_muts_ct_sort_by_seq_ct_all = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_AA_muts_ct_sort_by_seq_ct.jld2", "AA_muts_ct_sort_by_seq_ct")
    global AA_muts_ct_pos_only_sort_all = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_AA_muts_ct_pos_only_sort.jld2", "AA_muts_ct_pos_only_sort")
    global AA_muts_ct_pos_only_sort_by_seq_ct_all = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_AA_muts_ct_pos_only_sort_by_seq_ct.jld2", "AA_muts_ct_pos_only_sort_by_seq_ct")
    global AA_muts_ct_no_dels_sort_all = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_AA_muts_ct_no_dels_sort.jld2", "AA_muts_ct_no_dels_sort")
    global AA_muts_ct_no_dels_sort_by_seq_ct_all = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_AA_muts_ct_no_dels_sort_by_seq_ct.jld2", "AA_muts_ct_no_dels_sort_by_seq_ct")
    global AA_muts_ct_pos_only_no_dels_sort_all = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_AA_muts_ct_pos_only_no_dels_sort.jld2", "AA_muts_ct_pos_only_no_dels_sort")
    global AA_muts_ct_pos_only_no_dels_sort_by_seq_ct_all = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_AA_muts_ct_pos_only_no_dels_sort_by_seq_ct.jld2", "AA_muts_ct_pos_only_no_dels_sort_by_seq_ct")
    global gene_mut_density_sort_by_gene_all = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_gene_mut_density_sort_by_gene.jld2", "gene_mut_density_sort_by_gene")
    global gene_mut_density_sort_by_density_all = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_gene_mut_density_sort_by_density.jld2", "gene_mut_density_sort_by_density")
    global domain_mut_density_sort_by_gene_all = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_domain_mut_density_sort_by_gene.jld2", "domain_mut_density_sort_by_gene")
    global domain_mut_density_sort_by_density_all = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_domain_mut_density_sort_by_density.jld2", "domain_mut_density_sort_by_density")
    global AA_muts_seq_all = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_AA_muts_seq.jld2", "AA_muts_seq")
    global nuc_dels_ct_all = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_nuc_dels_ct.jld2", "nuc_dels_ct")
############################################################################################################################################################################
    global AA_no_dels_sub_count_all = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_AA_no_dels_sub_count.jld2", "AA_no_dels_sub_count")
    global total_AA_subs_all = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_total_AA_subs.jld2", "total_AA_subs")
#    global date_nuc_mut_ct_all = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_date_nuc_mut_ct.jld2", "date_nuc_mut_ct")
#    global date_nuc_mut_ct_no_dels_all = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_date_nuc_mut_ct_no_dels.jld2", "date_nuc_mut_ct_no_dels")
#    global date_AA_mut_ct_all = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_date_AA_mut_ct.jld2", "date_AA_mut_ct")
#    global date_AA_mut_ct_no_dels_all = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_date_AA_mut_ct_no_dels.jld2", "date_AA_mut_ct_no_dels")
#    global date_AA_mut_ct_pos_only_no_dels_all = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_date_AA_mut_ct_pos_only_no_dels.jld2", "date_AA_mut_ct_pos_only_no_dels")
#    global seq_date_index_all = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_seq_date_index.jld2", "seq_date_index")
    global pango_date_index_ct = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_pango_date_index_ct.jld2", "pango_date_index_ct")
    global clade_date_index_ct = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_clade_date_index_ct.jld2", "clade_date_index_ct")
############################################################################################################################################################################
######################################################## Below: semi-permanent, unt#il new ndjson made #####################################################################
############################################################################################################################################################################
## Changed mind about pango_date_index_ct--better to have high qc filters in place for those to weed out shitty misassigned seqs
#    global pango_date_index_ct = load("2025-11-12_all_private_muts_EPI_ISL_400000_20080000_v5_maxAAmut7_maxRevs2_qcMax10_pango_date_index_ct.jld2", "pango_date_index_ct")
#    global pango_date_index_ct = load("2025_08_25_all_private_muts_EPI_ISL_400000_20080000_maxAAmut90_maxRevs5_qcMax8000_pango_date_index_ct.jld2", "pango_date_index_ct")
#    global seq_date_index_all = load("dictionaries/2025-11-12_all_private_muts_EPI_ISL_400000_20080000_v5_maxAAmut90_maxRevs5_qcMax8000_seq_date_index.jld2", "seq_date_index")
#    global seq_date_tuple_all = load("dictionaries/2025-11-12_all_private_muts_EPI_ISL_400000_20080000_v5_maxAAmut90_maxRevs5_qcMax8000_seq_date_tuple.jld2", "seq_date_tuple")
#    global seq_pango_all = load("dictionaries/2025_08_25_all_private_muts_EPI_ISL_400000_20080000_maxAAmut90_maxRevs5_qcMax8000_seq_pango.jld2", "seq_pango")



    
#####################################################################################################################################
    global country_set = load("2026_01_07_all_private_muts_EPI_ISL_400001_20300000_maxAAmut90_maxRevs5_qcMax8000/2026_01_07_all_private_muts_EPI_ISL_400001_20300000_maxAAmut90_maxRevs5_qcMax8000_country_set.jld2", "country_set")
    global clade_set = load("2026_01_07_all_private_muts_EPI_ISL_400001_20300000_maxAAmut90_maxRevs5_qcMax8000/2026_01_07_all_private_muts_EPI_ISL_400001_20300000_maxAAmut90_maxRevs5_qcMax8000_clade_set.jld2", "clade_set")
    global pango_set = load("2026_01_07_all_private_muts_EPI_ISL_400001_20300000_maxAAmut90_maxRevs5_qcMax8000/2026_01_07_all_private_muts_EPI_ISL_400001_20300000_maxAAmut90_maxRevs5_qcMax8000_pango_set.jld2", "pango_set")
#    global seq_lab_set = load("2026_01_07_all_private_muts_EPI_ISL_400001_20300000_maxAAmut90_maxRevs5_qcMax8000/2026_01_07_all_private_muts_EPI_ISL_400001_20300000_maxAAmut90_maxRevs5_qcMax8000_seq_lab_set.jld2", "seq_lab_set")
#####################################################################################################################################
    
    
    
    
    #    date_temp = "2025-08-03"; max_AA_mut_temp = 5; revs_thresh_temp = 1; qc_max_temp = 5; ndjson_name_temp = "EPI_ISL_400000_20080000"
#    global country_set = load("dictionaries/$(date_temp)_$(fx_name)_$(ndjson_name_temp)_maxAAmut$(max_AA_mut_temp)_maxRevs$(revs_thresh_temp)_qcMax$(qc_max_temp)_country_set.jld2", "country_set")
#    global clade_set = load("dictionaries/$(date_temp)_$(fx_name)_$(ndjson_name_temp)_maxAAmut$(max_AA_mut_temp)_maxRevs$(revs_thresh_temp)_qcMax$(qc_max_temp)_clade_set.jld2", "clade_set")
#    global pango_set = load("dictionaries/$(date_temp)_$(fx_name)_$(ndjson_name_temp)_maxAAmut$(max_AA_mut_temp)_maxRevs$(revs_thresh_temp)_qcMax$(qc_max_temp)_pango_set.jld2", "pango_set")
#    global pango_unaliased_set = load("dictionaries/$(date_temp)_$(fx_name)_$(ndjson_name_temp)_maxAAmut$(max_AA_mut_temp)_maxRevs$(revs_thresh_temp)_qcMax$(qc_max_temp)_pango_unaliased_set.jld2", "pango_unaliased_set")
#    global pango_to_pango_unaliased = load("dictionaries/$(date_temp)_$(fx_name)_$(ndjson_name_temp)_maxAAmut$(max_AA_mut_temp)_maxRevs$(revs_thresh_temp)_qcMax$(qc_max_temp)_pango_to_pango_unaliased.jld2", "pango_to_pango_unaliased")
#    global clade_pango_set = load("dictionaries/$(date_temp)_$(fx_name)_$(ndjson_name_temp)_maxAAmut$(max_AA_mut_temp)_maxRevs$(revs_thresh_temp)_qcMax$(qc_max_temp)_clade_pango_set.jld2", "clade_pango_set")
#    global clade_pango_unaliased_set = load("dictionaries/$(date_temp)_$(fx_name)_$(ndjson_name_temp)_maxAAmut$(max_AA_mut_temp)_maxRevs$(revs_thresh_temp)_qcMax$(qc_max_temp)_clade_pango_unaliased_set.jld2", "clade_pango_unaliased_set")
#####################################################################################################################################
#    global seq_country_all = load("dictionaries/$(date_temp)_$(fx_name)_$(ndjson_name_temp)_maxAAmut$(max_AA_mut_temp)_maxRevs$(revs_thresh_temp)_qcMax$(qc_max_temp)_seq_country.jld2", "seq_country")
#    global seq_US_state_all = load("dictionaries/$(date_temp)_$(fx_name)_$(ndjson_name_temp)_maxAAmut$(max_AA_mut_temp)_maxRevs$(revs_thresh_temp)_qcMax$(qc_max_temp)_seq_US_state.jld2", "seq_US_state")
#    global seq_clade_all = load("dictionaries/$(date_temp)_$(fx_name)_$(ndjson_name_temp)_maxAAmut$(max_AA_mut_temp)_maxRevs$(revs_thresh_temp)_qcMax$(qc_max_temp)_seq_clade.jld2", "seq_clade")
############ Currently using low-qc seq_pango_all as it's more comprehensive ###########
#    global seq_pango_all = load("dictionaries/$(date_temp)_$(fx_name)_$(ndjson_name_temp)_maxAAmut$(max_AA_mut_temp)_maxRevs$(revs_thresh_temp)_qcMax$(qc_max_temp)_seq_pango.jld2", "seq_pango")
#    global seq_pango_unaliased_all = load("dictionaries/$(date_temp)_$(fx_name)_$(ndjson_name_temp)_maxAAmut$(max_AA_mut_temp)_maxRevs$(revs_thresh_temp)_qcMax$(qc_max_temp)_seq_pango_unaliased.jld2", "seq_pango_unaliased")
############ Currently using low-qc seq_date_index_all as it's more comprehensive ########################
#    global seq_date_index_all = load("dictionaries/2025-11-12_all_private_muts_EPI_ISL_400000_20080000_v5_maxAAmut90_maxRevs5_qcMax8000_seq_date_index.jld2", "seq_date_index")
#    global seq_date_tuple_all = load("dictionaries/2025-11-12_all_private_muts_EPI_ISL_400000_20080000_v5_maxAAmut90_maxRevs5_qcMax8000_seq_date_tuple.jld2", "seq_date_tuple")
#    global seq_collection_date_all = load("dictionaries/$(date_temp)_$(fx_name)_$(ndjson_name_temp)_maxAAmut$(max_AA_mut_temp)_maxRevs$(revs_thresh_temp)_qcMax$(qc_max_temp)_seq_collection_date.jld2", "seq_collection_date")
#    global seq_lab_dict_all = load("dictionaries/$(date_temp)_$(fx_name)_$(ndjson_name_temp)_maxAAmut$(max_AA_mut_temp)_maxRevs$(revs_thresh_temp)_qcMax$(qc_max_temp)_seq_lab_dict.jld2", "seq_lab_dict")
#    global seq_nuc_del_ranges_ct_all = load("dictionaries/$(date_temp)_$(fx_name)_$(ndjson_name_temp)_maxAAmut$(max_AA_mut_temp)_maxRevs$(revs_thresh_temp)_qcMax$(qc_max_temp)_seq_nuc_del_ranges_ct.jld2", "seq_nuc_del_ranges_ct")
#    global seq_lab_set = load("dictionaries/$(date_temp)_$(fx_name)_$(ndjson_name_temp)_maxAAmut$(max_AA_mut_temp)_maxRevs$(revs_thresh_temp)_qcMax$(qc_max_temp)_seq_lab_set.jld2", "seq_lab_set")
#####################################################################################################################################    
#    global pango_unaliased_date_index_ct = load("dictionaries/$(date_temp)_$(fx_name)_$(ndjson_name_temp)_maxAAmut$(max_AA_mut_temp)_maxRevs$(revs_thresh_temp)_qcMax$(qc_max_temp)_pango_unaliased_date_index_ct.jld2", "pango_unaliased_date_index_ct")
#    global country_clade_date_index_ct = load("dictionaries/$(date_temp)_$(fx_name)_$(ndjson_name_temp)_maxAAmut$(max_AA_mut_temp)_maxRevs$(revs_thresh_temp)_qcMax$(qc_max_temp)_country_clade_date_index_ct.jld2", "country_clade_date_index_ct")
#    global country_pango_date_index_ct = load("dictionaries/$(date_temp)_$(fx_name)_$(ndjson_name_temp)_maxAAmut$(max_AA_mut_temp)_maxRevs$(revs_thresh_temp)_qcMax$(qc_max_temp)_country_pango_date_index_ct.jld2", "country_pango_date_index_ct")
#    global country_pango_unaliased_date_index_ct = load("dictionaries/$(date_temp)_$(fx_name)_$(ndjson_name_temp)_maxAAmut$(max_AA_mut_temp)_maxRevs$(revs_thresh_temp)_qcMax$(qc_max_temp)_country_pango_unaliased_date_index_ct.jld2", "country_pango_unaliased_date_index_ct")
############################################################################################################################################################################
######################################################## Above: semi-permanent, unt#il new ndjson made #####################################################################
############################################################################################################################################################################
#    global ORF9b_CTD_muts_seq_all = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_ORF9b_CTD_muts_seq.jld2", "ORF9b_CTD_muts_seq")
    global multi_ORF9b_CTD_all = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_multi_ORF9b_CTD.jld2", "multi_ORF9b_CTD")
    global multi_ORF9b_CTD_ct_all = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_multi_ORF9b_CTD_ct.jld2", "multi_ORF9b_CTD_ct")
#    global qualifying_ORF9b_double_ct_all = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_qualifying_ORF9b_double_ct.jld2", "qualifying_ORF9b_double_ct")
    
#    global ORF9b_CTD_muts_seq_all_relaxed_qc_10_1_30 = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut8_maxRevs1_qcMax30_ORF9b_CTD_muts_seq_relaxed_qc_10_1_30.jld2", "ORF9b_CTD_muts_seq_relaxed_qc_10_1_30")
#    global multi_ORF9b_CTD_all_relaxed_qc_10_1_30 = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut8_maxRevs1_qcMax30_multi_ORF9b_CTD_relaxed_qc_10_1_30.jld2", "multi_ORF9b_CTD_relaxed_qc_10_1_30")
#    global multi_ORF9b_CTD_ct_all_relaxed_qc_10_1_30 = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut8_maxRevs1_qcMax30_multi_ORF9b_CTD_ct_relaxed_qc_10_1_30.jld2", "multi_ORF9b_CTD_ct_relaxed_qc_10_1_30")
#    global qualifying_ORF9b_double_ct_all_relaxed_qc_10_1_30 = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut8_maxRevs1_qcMax30_qualifying_ORF9b_double_ct_relaxed_qc_10_1_30.jld2", "qualifying_ORF9b_double_ct_relaxed_qc_10_1_30")

#    global ORF9b_CTD_muts_seq_all_relaxed_qc_15_1_50 = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut15_maxRevs1_qcMax50_ORF9b_CTD_muts_seq_relaxed_qc_15_1_50.jld2", "ORF9b_CTD_muts_seq_relaxed_qc_50_1_15")
#    global multi_ORF9b_CTD_all_relaxed_qc_15_1_50 = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut15_maxRevs1_qcMax50_multi_ORF9b_CTD_relaxed_qc_15_1_50.jld2", "multi_ORF9b_CTD_relaxed_qc_50_1_15")
#    global multi_ORF9b_CTD_ct_all_relaxed_qc_15_1_50 = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut15_maxRevs1_qcMax50_multi_ORF9b_CTD_ct_relaxed_qc_15_1_50.jld2", "multi_ORF9b_CTD_ct_relaxed_qc_50_1_15")
#    global qualifying_ORF9b_double_ct_all_relaxed_qc_15_1_50 = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut15_maxRevs1_qcMax50_qualifying_ORF9b_double_ct_relaxed_qc_15_1_50.jld2", "qualifying_ORF9b_double_ct_relaxed_qc_50_1_15")

#    global ORF9b_CTD_muts_seq_all_relaxed_qc_25_3_200 = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut25_maxRevs3_qcMax200_ORF9b_CTD_muts_seq_relaxed_qc_25_3_200.jld2", "ORF9b_CTD_muts_seq_relaxed_qc_200_3_25")
#    global multi_ORF9b_CTD_all_relaxed_qc_25_3_200 = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut25_maxRevs3_qcMax200_multi_ORF9b_CTD_relaxed_qc_25_3_200.jld2", "multi_ORF9b_CTD_relaxed_qc_200_3_25")
#    global multi_ORF9b_CTD_ct_all_relaxed_qc_25_3_200 = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut25_maxRevs3_qcMax200_multi_ORF9b_CTD_ct_relaxed_qc_25_3_200.jld2", "multi_ORF9b_CTD_ct_relaxed_qc_200_3_25")
#    global qualifying_ORF9b_double_ct_all_relaxed_qc_25_3_200 = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut25_maxRevs3_qcMax200_qualifying_ORF9b_double_ct_relaxed_qc_25_3_200.jld2", "qualifying_ORF9b_double_ct_relaxed_qc_200_3_25")
#    global avg_private_AA_per_circ_seq2 = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_avg_private_AA_per_circ_seq2.jld2", "avg_private_AA_per_circ_seq2")
    ##################### Below: Dicts for all seqs using higher (less restrictive) QC filters ##########################################
#    global AA_muts_seq_all_8000qc_filter = load("2026_01_07_all_private_muts_EPI_ISL_400001_20300000_maxAAmut90_maxRevs5_qcMax8000/2026_01_07_all_private_muts_EPI_ISL_400001_20300000_maxAAmut90_maxRevs5_qcMax8000_AA_muts_seq.jld2", "AA_muts_seq")
#    global AA_muts_seq_all_999qc_filter = load("dictionaries/2025-08-24_all_private_muts_EPI_ISL_400000_20080000_maxAAmut90_maxRevs4_qcMax999_AA_muts_seq.jld2", "AA_muts_seq")
#####################################################################################################################################
    println("Dictionaries loaded!");   println()
    finish_all = time() - start_all;    finish_all_rd = round(digits=1, finish_all)
    load_hms1, load_hms2 = seconds_to_hrs_min_sec(finish_all)
    println("Total Time to Load ALL Dictionaries = $(finish_all_rd) seconds")  # println("Total Time to Load ALL Dictionaries = $(load_hms1)")
    println("Total Time to Load ALL Dictionaries = $(load_hms2)")
end
######################################################################################################################################
date = "2026_02_08"
clade_pct_date_index = load("dictionaries/$(date)__clade_pct_date_index.jld2", "clade_pct_date_index")
clade_pct_date_tuple = load("dictionaries/$(date)__clade_pct_date_tuple.jld2", "clade_pct_date_tuple")
clade_pct_date_string = load("dictionaries/$(date)__clade_pct_date_string.jld2", "clade_pct_date_string")
pango_pct_date_index = load("dictionaries/$(date)__pango_pct_date_index.jld2", "pango_pct_date_index")
pango_pct_date_tuple = load("dictionaries/$(date)__pango_pct_date_tuple.jld2", "pango_pct_date_tuple")
pango_pct_date_string = load("dictionaries/$(date)__pango_pct_date_string.jld2", "pango_pct_date_string")
pango_unaliased_pct_date_index = load("dictionaries/$(date)__pango_unaliased_pct_date_index.jld2", "pango_unaliased_pct_date_index")
pango_unaliased_pct_date_tuple = load("dictionaries/$(date)__pango_unaliased_pct_date_tuple.jld2", "pango_unaliased_pct_date_tuple")
pango_unaliased_pct_date_string = load("dictionaries/$(date)__pango_unaliased_pct_date_string.jld2", "pango_unaliased_pct_date_string")
pango_to_pango_unaliased_v2 = load("dictionaries/$(date)__pango_to_pango_unaliased_v2.jld2", "pango_to_pango_unaliased_v2")
pango_unaliased_to_pango_prefix = load("dictionaries/$(date)__pango_unaliased_to_pango_prefix.jld2", "pango_unaliased_to_pango_prefix")
pango_unaliased_to_pango = load("dictionaries/$(date)__pango_unaliased_to_pango.jld2", "pango_unaliased_to_pango")
pango_unaliased_predecessor_meta_dict = load("dictionaries/$(date)__pango_unaliased_predecessor_meta_dict.jld2", "pango_unaliased_predecessor_meta_dict")
pango_predecessor_meta_dict = load("dictionaries/$(date)__pango_predecessor_meta_dict.jld2", "pango_predecessor_meta_dict")
pango_unaliased_set = load("dictionaries/$(date)__pango_unaliased_set.jld2", "pango_unaliased_set")
pango_unaliased_date_index_ct = load("dictionaries/$(date)__pango_unaliased_date_index_ct.jld2", "pango_unaliased_date_index_ct")
clade_date_index_cumul = load("dictionaries/$(date)__clade_date_index_cumul.jld2", "clade_date_index_cumul")
pango_date_index_cumul = load("dictionaries/$(date)__pango_date_index_cumul.jld2", "pango_date_index_cumul")
pango_unaliased_date_index_cumul = load("dictionaries/$(date)__pango_unaliased_date_index_cumul.jld2", "pango_unaliased_date_index_cumul")
clade_total = load("dictionaries/$(date)__clade_total.jld2", "clade_total")
pango_total = load("dictionaries/$(date)__pango_total.jld2", "pango_total")
pango_unaliased_total = load("dictionaries/$(date)__pango_unaliased_total.jld2", "pango_unaliased_total")
AAmut_date_index_cumul = load("dictionaries/$(date)__AAmut_date_index_cumul.jld2", "AAmut_date_index_cumul")
nucmut_date_index_cumul = load("dictionaries/$(date)__nucmut_date_index_cumul.jld2", "nucmut_date_index_cumul")
pango_nuc_sub_WT = load("dictionaries/$(date)__pango_nuc_sub_WT.jld2", "pango_nuc_sub_WT")
pango_nuc_del_WT = load("dictionaries/$(date)__pango_nuc_del_WT.jld2", "pango_nuc_del_WT")
pango_nuc_sub_private = load("dictionaries/$(date)__pango_nuc_sub_private.jld2", "pango_nuc_sub_private")
pango_nuc_del_private = load("dictionaries/$(date)__pango_nuc_del_private.jld2", "pango_nuc_del_private")
pango_nuc_sub_revs = load("dictionaries/$(date)__pango_nuc_sub_revs.jld2", "pango_nuc_sub_revs")
pango_nuc_del_revs = load("dictionaries/$(date)__pango_nuc_del_revs.jld2", "pango_nuc_del_revs")
pango_AAsub_WT = load("dictionaries/$(date)__pango_AAsub_WT.jld2", "pango_AAsub_WT")
pango_AAsub_WT_pos_only = load("dictionaries/$(date)__pango_AAsub_WT_pos_only.jld2", "pango_AAsub_WT_pos_only")
pango_AAdel_WT = load("dictionaries/$(date)__pango_AAdel_WT.jld2", "pango_AAdel_WT")
pango_AAsub_private = load("dictionaries/$(date)__pango_AAsub_private.jld2", "pango_AAsub_private")
pango_AAdel_private = load("dictionaries/$(date)__pango_AAdel_private.jld2", "pango_AAdel_private")
pango_AAsub_revs = load("dictionaries/$(date)__pango_AAsub_revs.jld2", "pango_AAsub_revs")
pango_AAdel_revs = load("dictionaries/$(date)__pango_AAdel_revs.jld2", "pango_AAdel_revs")
pango_frameshifts_WT = load("dictionaries/$(date)__pango_frameshifts_WT.jld2", "pango_frameshifts_WT")
pango_designation_date = load("dictionaries/$(date)__pango_designation_date.jld2", "pango_designation_date")
clade_pango_set = load("dictionaries/$(date)__clade_pango_set.jld2", "clade_pango_set")
delete!(pango_AAsub_WT["B.55"], "")
delete!(pango_AAsub_WT["B"], "")
println("Done!")
load_all_fx_runtime = time() - load_all_start; load_all_fx_runtime_rd = round(digits=3, load_all_fx_runtime)
load_all_fx_hms1, load_all_fx_hms2 = seconds_to_hrs_min_sec(load_all_fx_runtime)
println("Total Time to Run load_all Fx = $(load_all_fx_runtime_rd) seconds")  # println("Total Time to Run load_all Fx = $(load_all_fx_hms1)") 
println("Total Time to Run load_all Fx = $(load_all_fx_hms2)"); print("\n"^1)
######################################################################################################################################
######################################################################################################################################

2026_03_08__416PM
Done!
Total Time to Run load_all Fx = 8.23 seconds
Total Time to Run load_all Fx = 0 hr, 0 min, 08.23 sec



In [73]:
### Execute Load All Dicts Fx, EPI_ISL_400000_20300000 | 2026_02_08 | QC--5-1-5 | Runtime = 38 sec #########################
date_now = Dates.format(now(), "yyyy_mm_dd__IMMp"); println(date_now)
start_load_all_dicts = time()
#                     date          fx_name            ndjson_name     max_AA_mut,revs_thresh,qc_max
HQCS_date = "2026_01_06"
HQCS_fx_name = "all_private_muts"
HQCS_ndjson_name = "EPI_ISL_400001_20300000"
HQCS_max_AA_mut = 5
HQCS_revs_thresh = 1
HQCS_qc_max = 5
HQCS_qc_string = "$(HQCS_max_AA_mut)_$(HQCS_revs_thresh)_$(HQCS_qc_max)"
println("HQCS_qc_string = $(HQCS_qc_string)")
load_all_seq_dicts(HQCS_date, HQCS_fx_name, HQCS_ndjson_name, HQCS_max_AA_mut, HQCS_revs_thresh, HQCS_qc_max)

gene_protein_order = Dict{String, Int}("NSP1"=>1, "NSP2"=>2, "NSP3"=>3, "NSP4"=>4, "NSP5"=>5, "NSP6"=>6, "NSP7"=>7, "NSP8"=>8, "NSP9"=>9, "NSP10"=>10, "NSP12"=>12, "NSP13"=>13, "NSP14"=>14, "NSP15"=>15, "NSP16"=>16, "ORF3a"=>17, "ORF6"=>18, "ORF7a"=>19, "ORF7b"=>20, "ORF8"=>21, "ORF9b"=>22, "S"=>23, "E"=>24, "M"=>25, "N"=>26)
domain_order = Dict{String, Int}("NSP3_Ubl1"=>1, "NSP3_HVR"=>2, "NSP3_Mac1"=>3, "NSP3_Mac2"=>4, "NSP3_Mac3"=>5, "NSP3_DPUP"=>6, "NSP3_Ubl2"=>7, "NSP3_PLpro"=>8, "NSP3_NAB"=>9, "NSP3_BSM"=>10, "NSP3_TM1"=>11, "NSP3_Ecto3"=>12, "NSP3_TM234HLX"=>13, "NSP3_Y1"=>14, "NSP3_CoVY"=>15, "NSP4_TM1"=>16, "NSP4_Ecto4"=>17, "NSP4_TM2_TM6"=>18, "NSP4_CTD"=>19, "NSP6_AmphHlx"=>20, "NSP6_MAE"=>21, "NSP6_cyto_CTD"=>22, "NSP12_NiRAN"=>23, "NSP12_intrfce"=>24, "NSP12_fingers"=>25, "NSP12_palm"=>26, "NSP12_palmLnk"=>27, "NSP12_thumb"=>28, "NSP13_ZBD"=>29, "NSP13_stalk"=>30, "NSP13_1B"=>31, "NSP13_RecA1"=>32, "NSP13_RecA2"=>33, "NSP14_nsp10"=>34, "NSP14_EXON"=>35, "NSP14_hinge1"=>36, "NSP14_hinge2"=>37, "NSP14_N7MTase"=>38, "NSP15_NTD"=>39, "NSP15_MD"=>40, "NSP15_endoU"=>41, "S_S1"=>42, "S_S2"=>43, "S_NTD"=>44, "S_N2R"=>45, "S_RBD"=>46, "S_RBM"=>47, "S_SD1"=>48, "S_SD2"=>49, "S_630_loop"=>50, "S_FCS_region"=>51, "S_Beta1"=>52, "S_3H"=>53, "S_IL770"=>54, "S_FPPR"=>55, "S_FP"=>56, "S_HR1"=>57, "S_CH"=>58, "S_CD"=>59, "S_Beta2"=>60, "S_2turnHelix"=>61, "S_HR2"=>62, "S_TM"=>63, "S_CT"=>64, "ORF3a_SignalP"=>65, "ORF3a_NTD"=>66, "ORF3a_TM1"=>67, "ORF3a_TM12Lnk"=>68, "ORF3a_TM2"=>69, "ORF3a_TM3"=>70, "ORF3a_cytosl1"=>71, "ORF3a_Loop"=>72, "ORF3a_3DB"=>73, "ORF3a_CTD"=>74, "E_TM"=>75, "E_cytosol"=>76, "E_CTD"=>77, "N_N1"=>78, "N_N2"=>79, "N_N3"=>80, "N_N4"=>81, "N_N5"=>82, "N_SR"=>83, "N_L_helix"=>84, "N_CBP"=>85, "N_9b_overlap"=>86)    

finish_load_all_dicts = time() - start_load_all_dicts
finish_load_all_dicts_rd = round(digits=1, finish_load_all_dicts)
println("Total Time to Load ALL Dictionaries = $(finish_load_all_dicts_rd)")
####################################################################################################################################
#################################### Create  date_index_seq_ct_all  Dictionary #####################################################
############################# Only needed for special calculations, so commented out here ##########################################
####################################################################################################################################
#date_index_seq_ct_all = Dict{Int, Int}()
#for d_index in values(seq_date_index_all)
#    date_index_seq_ct_all[d_index] = get(date_index_seq_ct_all, d_index, 0) + 1
#end
function date_index_range_seq_ct(date1::Int, date2::Int)
    date1_date2_seq_ct = 0
    for d_index in date1:date2
        date1_date2_seq_ct += date_index_seq_ct_all[d_index]
    end
    return date1_date2_seq_ct
end   
###################################################################################################################################
############################### Create  nuc_muts_ct_pos_only_no_dels_all  Dictionary ##############################################
###################################################################################################################################
dict_make_start = time()
nuc_muts_ct_pos_only_no_dels_all = Dict{Int, Int}()
for i in 1:30000
    nuc_muts_ct_pos_only_no_dels_all[i] = 0
end
for (nuc_mut, count) in nuc_muts_ct_no_dels_all
    pos = nuc_mut_int_comprehensive_dict[nuc_mut]
    nuc_muts_ct_pos_only_no_dels_all[pos] += count
end
#############################################__Create   pango_index_date_set    ###################################################
pango_index_date_set = Set{String}()
for pango in keys(pango_date_index_ct)
    push!(pango_index_date_set, pango)
end
println(length(pango_index_date_set))
####################################################################################################################################
#function pango_prefix_to_unaliased_fx(fx_pango::String)
#    prefix_unaliased = ""
#    dotpts_ct = count('.', fx_pango)
#    if dotpts_ct ≥ 1
#        dotsplits = split(fx_pango, ".")
#        prefix = string(dotsplits[1])
#        prefix_unaliased = get(pango_prefix_to_pango_unaliased, prefix, prefix)
#        last_pts = join(dotsplits[2:end], ".")
#        unaliased = "$(prefix_unaliased).$(last_pts)"
#        return unaliased, prefix_unaliased
#    else
#        return fx_pango, prefix_unaliased
#    end
#end
########################################################################################################################
#for pango in pango_set
#    unali, new_prfx = pango_prefix_to_unaliased_fx(pango)
#    pango_to_pango_unaliased_v2[pango] = unali
#end
#################################################################################################################################
println("length(keys(pango_to_pango_unaliased_v2) = $(length(keys(pango_to_pango_unaliased_v2)))")
dict_make_runtime = round(digits=2, time() - dict_make_start)
println("Total Time to Make nuc_muts_ct_pos_only_no_dels_all Dict = $(dict_make_runtime) seconds"); print("\n"^1)
#################################################################################################################################
#################################################################################################################################

2026_03_08__220PM
HQCS_qc_string = 5_1_5
Dictionaries loaded!

Total Time to Load ALL Dictionaries = 43.8 seconds
Total Time to Load ALL Dictionaries = 0 hr, 0 min, 43.79 sec
Total Time to Load ALL Dictionaries = 43.9
4297
length(keys(pango_to_pango_unaliased_v2) = 5657
Total Time to Make nuc_muts_ct_pos_only_no_dels_all Dict = 0.1 seconds



In [87]:
### Fx: chronic_load_dicts2_DQ, 2026_03_01 version  | Loads EPCI dataset info |
date_now = Dates.format(now(), "yyyy_mm_dd__IMMp"); println(date_now)
start_chr_load_fx = time()
function chronic_load_dicts2_DQ(ndjson_name::String, folder_name::String, date::String, rep_thresh::Int, revs_thresh::Int, print_ct_thresh::Int, DQ_mut_thresh::Int, DatePctDQThresh::Int, abs_min_mut_thresh::Int)
    start_date = Dates.format(now(), "yyyy_mm_dd__IMMp"); println(start_date)
######################################################################################################################################
#    global cumulative_AA_mut_ct_by_date_index
#        cumulative_AA_mut_ct_by_date_index = load("$(folder_name)/$(folder_name)__cumulative_AA_mut_ct_by_date_index.jld2", "cumulative_AA_mut_ct_by_date_index")
#    global cumulative_nuc_mut_ct_by_date_index
#        cumulative_nuc_mut_ct_by_date_index = load("$(folder_name)/$(folder_name)__cumulative_nuc_mut_ct_by_date_index.jld2", "cumulative_nuc_mut_ct_by_date_index")
######################################################################################################################################
    global seq_AA_insertions_WT
        seq_AA_insertions_WT = load("$(folder_name)/$(folder_name)__seq_AA_insertions_WT.jld2", "seq_AA_insertions_WT")
    global seq_nuc_insertions_WT
        seq_nuc_insertions_WT = load("$(folder_name)/$(folder_name)__seq_nuc_insertions_WT.jld2", "seq_nuc_insertions_WT")    
######################################################################################################################################
    global total_chr_AA_subs
        total_chr_AA_subs = load("$(folder_name)/$(folder_name)__total_chr_AA_subs.jld2", "total_chr_AA_subs")
######################################################################################################################################
    global total_nuc_revs_seq
        total_nuc_revs_seq = load("$(folder_name)/$(folder_name)__total_nuc_revs_seq.jld2", "total_nuc_revs_seq")
    global seq_nuc_total_revs
        seq_nuc_total_revs = load("$(folder_name)/$(folder_name)__seq_nuc_total_revs.jld2", "seq_nuc_total_revs")
    global total_AA_revs_seq
        total_AA_revs_seq = load("$(folder_name)/$(folder_name)__total_AA_revs_seq.jld2", "total_AA_revs_seq")
    global seq_AA_total_revs
        seq_AA_total_revs = load("$(folder_name)/$(folder_name)__seq_AA_total_revs.jld2", "seq_AA_total_revs")
    global seq_AA_revs
        seq_AA_revs = load("$(folder_name)/$(folder_name)__seq_AA_revs.jld2", "seq_AA_revs")
######################################################################################################################################
    global AA_muts_ct_chr_all_ratio
        AA_muts_ct_chr_all_ratio = load("$(folder_name)/$(folder_name)__AA_muts_ct_chr_all_ratio.jld2", "AA_muts_ct_chr_all_ratio")
    global AA_muts_ct_no_dels_chr_all_ratio
        AA_muts_ct_no_dels_chr_all_ratio = load("$(folder_name)/$(folder_name)__AA_muts_ct_no_dels_chr_all_ratio.jld2", "AA_muts_ct_no_dels_chr_all_ratio")
    global AA_muts_ct_pos_only_no_dels_chr_all_ratio
        AA_muts_ct_pos_only_no_dels_chr_all_ratio = load("$(folder_name)/$(folder_name)__AA_muts_ct_pos_only_no_dels_chr_all_ratio.jld2", "AA_muts_ct_pos_only_no_dels_chr_all_ratio")
    global AA_muts_ct_no_dels_no_revs_chr_all_ratio
        AA_muts_ct_no_dels_no_revs_chr_all_ratio = load("$(folder_name)/$(folder_name)__AA_muts_ct_no_dels_no_revs_chr_all_ratio.jld2", "AA_muts_ct_no_dels_no_revs_chr_all_ratio")
    global AA_muts_ct_pos_only_no_dels_no_revs_chr_all_ratio
        AA_muts_ct_pos_only_no_dels_no_revs_chr_all_ratio = load("$(folder_name)/$(folder_name)__AA_muts_ct_pos_only_no_dels_no_revs_chr_all_ratio.jld2", "AA_muts_ct_pos_only_no_dels_no_revs_chr_all_ratio")
###################################################################################################################################### 
    global nuc_muts_ct_no_dels_chr_all_ratio
        nuc_muts_ct_no_dels_chr_all_ratio = load("$(folder_name)/$(folder_name)__nuc_muts_ct_no_dels_chr_all_ratio.jld2", "nuc_muts_ct_no_dels_chr_all_ratio")
    global nuc_muts_ct_pos_only_no_dels_chr_all_ratio
        nuc_muts_ct_pos_only_no_dels_chr_all_ratio = load("$(folder_name)/$(folder_name)__nuc_muts_ct_pos_only_no_dels_chr_all_ratio.jld2", "nuc_muts_ct_pos_only_no_dels_chr_all_ratio")
    global nuc_muts_ct_no_dels_no_revs_chr_all_ratio
        nuc_muts_ct_no_dels_no_revs_chr_all_ratio = load("$(folder_name)/$(folder_name)__nuc_muts_ct_no_dels_no_revs_chr_all_ratio.jld2", "nuc_muts_ct_no_dels_no_revs_chr_all_ratio")
    global nuc_muts_ct_pos_only_no_dels_no_revs_chr_all_ratio
        nuc_muts_ct_pos_only_no_dels_no_revs_chr_all_ratio = load("$(folder_name)/$(folder_name)__nuc_muts_ct_pos_only_no_dels_no_revs_chr_all_ratio.jld2", "nuc_muts_ct_pos_only_no_dels_no_revs_chr_all_ratio")
###################################################################################################################################### 
####################################################################################################################################
    global AA_muts_ct_pos_only_adj_score
        AA_muts_ct_pos_only_adj_score = load("$(folder_name)/$(folder_name)__AA_muts_ct_pos_only_adj_score.jld2", "AA_muts_ct_pos_only_adj_score")
    global AA_muts_ct_adj
        AA_muts_ct_adj = load("$(folder_name)/$(folder_name)__AA_muts_ct_adj.jld2", "AA_muts_ct_adj")
    global AA_muts_ct_pos_only_adj_score_no_dels
        AA_muts_ct_pos_only_adj_score_no_dels = load("$(folder_name)/$(folder_name)__AA_muts_ct_pos_only_adj_score_no_dels.jld2", "AA_muts_ct_pos_only_adj_score_no_dels")
    global nuc_muts_ct_adj
        nuc_muts_ct_adj = load("$(folder_name)/$(folder_name)__nuc_muts_ct_adj.jld2", "nuc_muts_ct_adj")
    global AA_muts_ct_adj_score
        AA_muts_ct_adj_score = load("$(folder_name)/$(folder_name)__AA_muts_ct_adj_score.jld2", "AA_muts_ct_adj_score")
    global AA_muts_ct_adj_score_no_dels
        AA_muts_ct_adj_score_no_dels = load("$(folder_name)/$(folder_name)__AA_muts_ct_adj_score_no_dels.jld2", "AA_muts_ct_adj_score_no_dels")
    global AA_muts_ct_pos_only_adj
        AA_muts_ct_pos_only_adj = load("$(folder_name)/$(folder_name)__AA_muts_ct_pos_only_adj.jld2", "AA_muts_ct_pos_only_adj")
    global AA_muts_ct_pos_only_adj_no_dels
        AA_muts_ct_pos_only_adj_no_dels = load("$(folder_name)/$(folder_name)__AA_muts_ct_pos_only_adj_no_dels.jld2", "AA_muts_ct_pos_only_adj_no_dels")
    global nuc_muts_ct_adj_score_no_dels
        nuc_muts_ct_adj_score_no_dels = load("$(folder_name)/$(folder_name)__nuc_muts_ct_adj_score_no_dels.jld2", "nuc_muts_ct_adj_score_no_dels")
    global nuc_muts_ct_adj_score
        nuc_muts_ct_adj_score = load("$(folder_name)/$(folder_name)__nuc_muts_ct_adj_score.jld2", "nuc_muts_ct_adj_score")
    global nuc_muts_ct_adj_no_dels
        nuc_muts_ct_adj_no_dels = load("$(folder_name)/$(folder_name)__nuc_muts_ct_adj_no_dels.jld2", "nuc_muts_ct_adj_no_dels")
    global AA_muts_ct_adj_no_dels
        AA_muts_ct_adj_no_dels = load("$(folder_name)/$(folder_name)__AA_muts_ct_adj_no_dels.jld2", "AA_muts_ct_adj_no_dels")
####################################################################################################################################
####################################################################################################################################
    global seq_ct_by_year
        seq_ct_by_year = load("$(folder_name)/$(folder_name)__seq_ct_by_year.jld2", "seq_ct_by_year")
    global seq_ct_by_year_month
        seq_ct_by_year_month = load("$(folder_name)/$(folder_name)__seq_ct_by_year_month.jld2", "seq_ct_by_year_month")
    global seq_ct_by_year_month_day
        seq_ct_by_year_month_day = load("$(folder_name)/$(folder_name)__seq_ct_by_year_month_day.jld2", "seq_ct_by_year_month_day")
    global seq_date_tuple
        seq_date_tuple = load("$(folder_name)/$(folder_name)__seq_date_tuple.jld2", "seq_date_tuple")
    global date_nuc_mut_ct_no_dels
        date_nuc_mut_ct_no_dels = load("$(folder_name)/$(folder_name)__date_nuc_mut_ct_no_dels.jld2", "date_nuc_mut_ct_no_dels")
    global date_AA_mut_ct_no_dels
        date_AA_mut_ct_no_dels = load("$(folder_name)/$(folder_name)__date_AA_mut_ct_no_dels.jld2", "date_AA_mut_ct_no_dels")
    global date_AA_mut_ct
        date_AA_mut_ct = load("$(folder_name)/$(folder_name)__date_AA_mut_ct.jld2", "date_AA_mut_ct")
    global date_AA_mut_ct_pos_only_no_dels
        date_AA_mut_ct_pos_only_no_dels = load("$(folder_name)/$(folder_name)__date_AA_mut_ct_pos_only_no_dels.jld2", "date_AA_mut_ct_pos_only_no_dels")
####################################################################################################################################
    global seq_collection_date
        seq_collection_date = load("$(folder_name)/$(folder_name)__seq_collection_date.jld2", "seq_collection_date")
    global seq_date_index
        seq_date_index = load("$(folder_name)/$(folder_name)__seq_date_index.jld2", "seq_date_index")
    global date_nuc_mut_ct
        date_nuc_mut_ct = load("$(folder_name)/$(folder_name)__date_nuc_mut_ct.jld2", "date_nuc_mut_ct")
####################################################################################################################################
    global seq_AA_dels
        seq_AA_dels = load("$(folder_name)/$(folder_name)__seq_AA_dels.jld2", "seq_AA_dels")
    global seq_AA_muts_pos_only_no_dels
        seq_AA_muts_pos_only_no_dels = load("$(folder_name)/$(folder_name)__seq_AA_muts_pos_only_no_dels.jld2", "seq_AA_muts_pos_only_no_dels")
    global seq_AA_muts_WT_pos_only
        seq_AA_muts_WT_pos_only = load("$(folder_name)/$(folder_name)__seq_AA_muts_WT_pos_only.jld2", "seq_AA_muts_WT_pos_only")
    global seq_AA_dels_WT
        seq_AA_dels_WT = load("$(folder_name)/$(folder_name)__seq_AA_dels_WT.jld2", "seq_AA_dels_WT")
    global seq_AA_muts_WT
        seq_AA_muts_WT = load("$(folder_name)/$(folder_name)__seq_AA_muts_WT.jld2", "seq_AA_muts_WT")
    global seq_AA_muts
        seq_AA_muts = load("$(folder_name)/$(folder_name)__seq_AA_muts.jld2", "seq_AA_muts")
    global seq_AA_muts_no_dels
        seq_AA_muts_no_dels = load("$(folder_name)/$(folder_name)__seq_AA_muts_no_dels.jld2", "seq_AA_muts_no_dels")
    global seq_AA_muts_pos_only
        seq_AA_muts_pos_only = load("$(folder_name)/$(folder_name)__seq_AA_muts_pos_only.jld2", "seq_AA_muts_pos_only")
    global seq_mixed_AA_muts
        seq_mixed_AA_muts = load("$(folder_name)/$(folder_name)__seq_mixed_AA_muts.jld2", "seq_mixed_AA_muts")
    global seq_unknown_AA
        seq_unknown_AA = load("$(folder_name)/$(folder_name)__seq_unknown_AA.jld2", "seq_unknown_AA")
    global seq_unknown_AA_ranges
        seq_unknown_AA_ranges = load("$(folder_name)/$(folder_name)__seq_unknown_AA_ranges.jld2", "seq_unknown_AA_ranges")
####################################################################################################################################
    global seq_nuc_muts
        seq_nuc_muts = load("$(folder_name)/$(folder_name)__seq_nuc_muts.jld2", "seq_nuc_muts")
    global seq_nuc_muts_WT
        seq_nuc_muts_WT = load("$(folder_name)/$(folder_name)__seq_nuc_muts_WT.jld2", "seq_nuc_muts_WT")
    global seq_nuc_del_ranges_WT
        seq_nuc_del_ranges_WT = load("$(folder_name)/$(folder_name)__seq_nuc_del_ranges_WT.jld2", "seq_nuc_del_ranges_WT")
    global seq_nuc_dropout
        seq_nuc_dropout = load("$(folder_name)/$(folder_name)__seq_nuc_dropout.jld2", "seq_nuc_dropout")
    global seq_nuc_del_ranges
        seq_nuc_del_ranges = load("$(folder_name)/$(folder_name)__seq_nuc_del_ranges.jld2", "seq_nuc_del_ranges")
    global seq_mixed_nucs
        seq_mixed_nucs = load("$(folder_name)/$(folder_name)__seq_mixed_nucs.jld2", "seq_mixed_nucs")
###################################################################################################################################
    global seq_clade
        seq_clade = load("$(folder_name)/$(folder_name)__seq_clade.jld2", "seq_clade")
    global seq_pango
        seq_pango = load("$(folder_name)/$(folder_name)__seq_pango.jld2", "seq_pango")
    global seq_pango_unaliased
        seq_pango_unaliased = load("$(folder_name)/$(folder_name)__seq_pango_unaliased.jld2", "seq_pango_unaliased")    
    global seq_clade_display
        seq_clade_display = load("$(folder_name)/$(folder_name)__seq_clade_display.jld2", "seq_clade_display")
    global seq_clade_ct
        seq_clade_ct = load("$(folder_name)/$(folder_name)__seq_clade_ct.jld2", "seq_clade_ct")
    global seq_pango_ct
        seq_pango_ct = load("$(folder_name)/$(folder_name)__seq_pango_ct.jld2", "seq_pango_ct")
    global seq_pango_unaliased_ct
        seq_pango_unaliased_ct = load("$(folder_name)/$(folder_name)__seq_pango_unaliased_ct.jld2", "seq_pango_unaliased_ct")
####################################################################################################################################
    global seq_US_state
        seq_US_state = load("$(folder_name)/$(folder_name)__seq_US_state.jld2", "seq_US_state")
    global seq_country
        seq_country = load("$(folder_name)/$(folder_name)__seq_country.jld2", "seq_country")
    global seq_lab_dict
        seq_lab_dict = load("$(folder_name)/$(folder_name)__seq_lab_dict.jld2", "seq_lab_dict")
####################################################################################################################################
    global gene_mut_density
        gene_mut_density = load("$(folder_name)/$(folder_name)__gene_mut_density.jld2", "gene_mut_density")
    global domain_mut_density
        domain_mut_density = load("$(folder_name)/$(folder_name)__domain_mut_density.jld2", "domain_mut_density")
####################################################################################################################################
    global nuc_muts_ct
        nuc_muts_ct = load("$(folder_name)/$(folder_name)__nuc_muts_ct.jld2", "nuc_muts_ct")
    global nuc_dels_ct
        nuc_dels_ct = load("$(folder_name)/$(folder_name)__nuc_dels_ct.jld2", "nuc_dels_ct")
    global nuc_muts_ct_no_dels
        nuc_muts_ct_no_dels = load("$(folder_name)/$(folder_name)__nuc_muts_ct_no_dels.jld2", "nuc_muts_ct_no_dels")
######################
    global nuc_dels_seq
        nuc_dels_seq = load("$(folder_name)/$(folder_name)__nuc_dels_seq.jld2", "nuc_dels_seq")
    global nuc_muts_seq
        nuc_muts_seq = load("$(folder_name)/$(folder_name)__nuc_muts_seq.jld2", "nuc_muts_seq")
    global nuc_dels_seq_WT
        nuc_dels_seq_WT = load("$(folder_name)/$(folder_name)__nuc_dels_seq_WT.jld2", "nuc_dels_seq_WT")
    global nuc_muts_seq_WT
        nuc_muts_seq_WT = load("$(folder_name)/$(folder_name)__nuc_muts_seq_WT.jld2", "nuc_muts_seq_WT")
##################################################################
    global AA_muts_ct
        AA_muts_ct = load("$(folder_name)/$(folder_name)__AA_muts_ct.jld2", "AA_muts_ct")
    global AA_muts_ct_no_dels_no_revs
        AA_muts_ct_no_dels_no_revs = load("$(folder_name)/$(folder_name)__AA_muts_ct_no_dels_no_revs.jld2", "AA_muts_ct_no_dels_no_revs")
    global AA_muts_ct_pos_only
        AA_muts_ct_pos_only = load("$(folder_name)/$(folder_name)__AA_muts_ct_pos_only.jld2", "AA_muts_ct_pos_only")
    global AA_dels_ct
        AA_dels_ct = load("$(folder_name)/$(folder_name)__AA_dels_ct.jld2", "AA_dels_ct")
    global AA_muts_ct_pos_only_no_dels
        AA_muts_ct_pos_only_no_dels = load("$(folder_name)/$(folder_name)__AA_muts_ct_pos_only_no_dels.jld2", "AA_muts_ct_pos_only_no_dels")
    global AA_muts_ct_no_dels
        AA_muts_ct_no_dels = load("$(folder_name)/$(folder_name)__AA_muts_ct_no_dels.jld2", "AA_muts_ct_no_dels")
############################## 
    global AA_muts_seq
        AA_muts_seq = load("$(folder_name)/$(folder_name)__AA_muts_seq.jld2", "AA_muts_seq")
    global AA_muts_seq_pos_only
        AA_muts_seq_pos_only = load("$(folder_name)/$(folder_name)__AA_muts_seq_pos_only.jld2", "AA_muts_seq_pos_only")
    global AA_muts_seq_pos_only_no_dels
        AA_muts_seq_pos_only_no_dels = load("$(folder_name)/$(folder_name)__AA_muts_seq_pos_only_no_dels.jld2", "AA_muts_seq_pos_only_no_dels")
    global AA_dels_seq
        AA_dels_seq = load("$(folder_name)/$(folder_name)__AA_dels_seq.jld2", "AA_dels_seq")
##############################
    global AA_muts_seq_WT
        AA_muts_seq_WT = load("$(folder_name)/$(folder_name)__AA_muts_seq_WT.jld2", "AA_muts_seq_WT")
    global AA_muts_seq_WT_pos_only
        AA_muts_seq_WT_pos_only = load("$(folder_name)/$(folder_name)__AA_muts_seq_WT_pos_only.jld2", "AA_muts_seq_WT_pos_only")
    global AA_dels_seq_WT
        AA_dels_seq_WT = load("$(folder_name)/$(folder_name)__AA_dels_seq_WT.jld2", "AA_dels_seq_WT")
#####################################################################################################################################
#####################################################################################################################################
    global NSP_muts
        NSP_muts = load("$(folder_name)/$(folder_name)__NSP_muts.jld2", "NSP_muts")
    global NSP_muts_no_dels
        NSP_muts_no_dels = load("$(folder_name)/$(folder_name)__NSP_muts_no_dels.jld2", "NSP_muts_no_dels")
#####################################################################################################################################
    global non_rep_seqs_AA
        non_rep_seqs_AA = load("$(folder_name)/$(folder_name)__non_rep_seqs_AA.jld2", "non_rep_seqs_AA")
    global non_rep_seqs_AA_pos_only
        non_rep_seqs_AA_pos_only = load("$(folder_name)/$(folder_name)__non_rep_seqs_AA_pos_only.jld2", "non_rep_seqs_AA_pos_only")
    global non_rep_seqs_AA_pos_only_no_dels
        non_rep_seqs_AA_pos_only_no_dels = load("$(folder_name)/$(folder_name)__non_rep_seqs_AA_pos_only_no_dels.jld2", "non_rep_seqs_AA_pos_only_no_dels")
#####################################################################################################################################
    global rep_seq_grps_AA_muts_WT
        rep_seq_grps_AA_muts_WT = load("$(folder_name)/$(folder_name)__rep_seq_grps_AA_muts_WT.jld2", "rep_seq_grps_AA_muts_WT")
    global rep_seq_grps_AA_muts_WT_pos_only
        rep_seq_grps_AA_muts_WT_pos_only = load("$(folder_name)/$(folder_name)__rep_seq_grps_AA_muts_WT_pos_only.jld2", "rep_seq_grps_AA_muts_WT_pos_only")
    global rep_seq_grps_AA_dels_WT
        rep_seq_grps_AA_dels_WT = load("$(folder_name)/$(folder_name)__rep_seq_grps_AA_dels_WT.jld2", "rep_seq_grps_AA_dels_WT")
    global rep_seq_grps_nuc_muts_WT
        rep_seq_grps_nuc_muts_WT = load("$(folder_name)/$(folder_name)__rep_seq_grps_nuc_muts_WT.jld2", "rep_seq_grps_nuc_muts_WT")
    global rep_seq_grps_nuc_dels_WT
        rep_seq_grps_nuc_dels_WT = load("$(folder_name)/$(folder_name)__rep_seq_grps_nuc_dels_WT.jld2", "rep_seq_grps_nuc_dels_WT")
    global nuc_muts_rep_seq_grps_WT
        nuc_muts_rep_seq_grps_WT = load("$(folder_name)/$(folder_name)__nuc_muts_rep_seq_grps_WT.jld2", "nuc_muts_rep_seq_grps_WT")
    global nuc_dels_rep_seq_grps_WT
        nuc_dels_rep_seq_grps_WT = load("$(folder_name)/$(folder_name)__nuc_dels_rep_seq_grps_WT.jld2", "nuc_dels_rep_seq_grps_WT")
    global AA_dels_rep_seq_grps_WT
        AA_dels_rep_seq_grps_WT = load("$(folder_name)/$(folder_name)__AA_dels_rep_seq_grps_WT.jld2", "AA_dels_rep_seq_grps_WT")
    global AA_muts_rep_seq_grps_WT
        AA_muts_rep_seq_grps_WT = load("$(folder_name)/$(folder_name)__AA_muts_rep_seq_grps_WT.jld2", "AA_muts_rep_seq_grps_WT")
    global AA_muts_rep_seq_grps_WT_pos_only
        AA_muts_rep_seq_grps_WT_pos_only = load("$(folder_name)/$(folder_name)__AA_muts_rep_seq_grps_WT_pos_only.jld2", "AA_muts_rep_seq_grps_WT_pos_only")
#####################################################################################################################################
#####################################################################################################################################
    global rep_seq_grps_maxmut_pango
        rep_seq_grps_maxmut_pango = load("$(folder_name)/$(folder_name)__rep_seq_grps_maxmut_pango.jld2", "rep_seq_grps_maxmut_pango")
    global rep_seq_grps_maxmut_clade
        rep_seq_grps_maxmut_clade = load("$(folder_name)/$(folder_name)__rep_seq_grps_maxmut_clade.jld2", "rep_seq_grps_maxmut_clade")
    global rep_seq_grps_maxmut_pango_unaliased
        rep_seq_grps_maxmut_pango_unaliased = load("$(folder_name)/$(folder_name)__rep_seq_grps_maxmut_pango_unaliased.jld2", "rep_seq_grps_maxmut_pango_unaliased")
#####################################################################################################################################
    global rep_seq_grps_maxmut_dels
        rep_seq_grps_maxmut_dels = load("$(folder_name)/$(folder_name)__rep_seq_grps_maxmut_dels.jld2", "rep_seq_grps_maxmut_dels")
    global rep_seq_grps_maxmut_AA_pos_only_no_dels
        rep_seq_grps_maxmut_AA_pos_only_no_dels = load("$(folder_name)/$(folder_name)__rep_seq_grps_maxmut_AA_pos_only_no_dels.jld2", "rep_seq_grps_maxmut_AA_pos_only_no_dels")
    global rep_seq_grps_maxmut_nuc
        rep_seq_grps_maxmut_nuc = load("$(folder_name)/$(folder_name)__rep_seq_grps_maxmut_nuc.jld2", "rep_seq_grps_maxmut_nuc")
    global rep_seq_grps_maxmut_AA_pos_only
        rep_seq_grps_maxmut_AA_pos_only = load("$(folder_name)/$(folder_name)__rep_seq_grps_maxmut_AA_pos_only.jld2", "rep_seq_grps_maxmut_AA_pos_only")
    global rep_seq_grps_maxmut_nuc_dropout
        rep_seq_grps_maxmut_nuc_dropout = load("$(folder_name)/$(folder_name)__rep_seq_grps_maxmut_nuc_dropout.jld2", "rep_seq_grps_maxmut_nuc_dropout")
    global rep_seq_grps_maxmut_nuc_no_dels
        rep_seq_grps_maxmut_nuc_no_dels = load("$(folder_name)/$(folder_name)__rep_seq_grps_maxmut_nuc_no_dels.jld2", "rep_seq_grps_maxmut_nuc_no_dels")
    global rep_seq_grps_maxmut_mixed_nucs
        rep_seq_grps_maxmut_mixed_nucs = load("$(folder_name)/$(folder_name)__rep_seq_grps_maxmut_mixed_nucs.jld2", "rep_seq_grps_maxmut_mixed_nucs")
    global rep_seq_grps_maxmut_AA_dels
        rep_seq_grps_maxmut_AA_dels = load("$(folder_name)/$(folder_name)__rep_seq_grps_maxmut_AA_dels.jld2", "rep_seq_grps_maxmut_AA_dels")
    global rep_seq_grps_maxmut_del_ranges_ct
        rep_seq_grps_maxmut_del_ranges_ct = load("$(folder_name)/$(folder_name)__rep_seq_grps_maxmut_del_ranges_ct.jld2", "rep_seq_grps_maxmut_del_ranges_ct")
    global rep_seq_grps_maxmut_AA
        rep_seq_grps_maxmut_AA = load("$(folder_name)/$(folder_name)__rep_seq_grps_maxmut_AA.jld2", "rep_seq_grps_maxmut_AA")
    global rep_seq_grps_maxmut_mixed_AA_muts
        rep_seq_grps_maxmut_mixed_AA_muts = load("$(folder_name)/$(folder_name)__rep_seq_grps_maxmut_mixed_AA_muts.jld2", "rep_seq_grps_maxmut_mixed_AA_muts")
    global rep_seq_grps_maxmut_unknown_AA
        rep_seq_grps_maxmut_unknown_AA = load("$(folder_name)/$(folder_name)__rep_seq_grps_maxmut_unknown_AA.jld2", "rep_seq_grps_maxmut_unknown_AA")
    global rep_seq_grps_maxmut_unknown_AA_ranges
        rep_seq_grps_maxmut_unknown_AA_ranges = load("$(folder_name)/$(folder_name)__rep_seq_grps_maxmut_unknown_AA_ranges.jld2", "rep_seq_grps_maxmut_unknown_AA_ranges")
    global rep_seq_grps_maxmut_seqs
        rep_seq_grps_maxmut_seqs = load("$(folder_name)/$(folder_name)__rep_seq_grps_maxmut_seqs.jld2", "rep_seq_grps_maxmut_seqs")
    global rep_seq_grps_maxmut_AA_no_dels
        rep_seq_grps_maxmut_AA_no_dels = load("$(folder_name)/$(folder_name)__rep_seq_grps_maxmut_AA_no_dels.jld2", "rep_seq_grps_maxmut_AA_no_dels")
    global rep_seq_grps_maxmut_nuc_muts_WT
        rep_seq_grps_maxmut_nuc_muts_WT = load("$(folder_name)/$(folder_name)__rep_seq_grps_maxmut_nuc_muts_WT.jld2", "rep_seq_grps_maxmut_nuc_muts_WT")
    global rep_seq_grps_maxmut_nuc_dels_WT
        rep_seq_grps_maxmut_nuc_dels_WT = load("$(folder_name)/$(folder_name)__rep_seq_grps_maxmut_nuc_dels_WT.jld2", "rep_seq_grps_maxmut_nuc_dels_WT")
    global rep_seq_grps_maxmut_AA_muts_WT
        rep_seq_grps_maxmut_AA_muts_WT = load("$(folder_name)/$(folder_name)__rep_seq_grps_maxmut_AA_muts_WT.jld2", "rep_seq_grps_maxmut_AA_muts_WT")
    global rep_seq_grps_maxmut_AA_dels_WT
        rep_seq_grps_maxmut_AA_dels_WT = load("$(folder_name)/$(folder_name)__rep_seq_grps_maxmut_AA_dels_WT.jld2", "rep_seq_grps_maxmut_AA_dels_WT")
    global rep_seq_grps_maxmut_AA_muts_WT_pos_only
        rep_seq_grps_maxmut_AA_muts_WT_pos_only = load("$(folder_name)/$(folder_name)__rep_seq_grps_maxmut_AA_muts_WT_pos_only.jld2", "rep_seq_grps_maxmut_AA_muts_WT_pos_only")
################################################################################################################################
#####################################################################################################################################
    global rep_seq_grps_seqs
        rep_seq_grps_seqs = load("$(folder_name)/$(folder_name)__rep_seq_grps_seqs.jld2", "rep_seq_grps_seqs")
    global rep_seq_grps_pango
        rep_seq_grps_pango = load("$(folder_name)/$(folder_name)__rep_seq_grps_pango.jld2", "rep_seq_grps_pango")
    global rep_seq_grps_clade
        rep_seq_grps_clade = load("$(folder_name)/$(folder_name)__rep_seq_grps_clade.jld2", "rep_seq_grps_clade")
    global rep_seq_grps_pango_unaliased
        rep_seq_grps_pango_unaliased = load("$(folder_name)/$(folder_name)__rep_seq_grps_pango_unaliased.jld2", "rep_seq_grps_pango_unaliased")
    global rep_seq_grps_muts
        rep_seq_grps_muts = load("$(folder_name)/$(folder_name)__rep_seq_grps_muts.jld2", "rep_seq_grps_muts")
    global rep_seq_grps_mixed_nucs
        rep_seq_grps_mixed_nucs = load("$(folder_name)/$(folder_name)__rep_seq_grps_mixed_nucs.jld2", "rep_seq_grps_mixed_nucs")
    global rep_seq_grps_muts_no_dels
        rep_seq_grps_muts_no_dels = load("$(folder_name)/$(folder_name)__rep_seq_grps_muts_no_dels.jld2", "rep_seq_grps_muts_no_dels")
    global rep_seq_grps_AA_pos_only
        rep_seq_grps_AA_pos_only = load("$(folder_name)/$(folder_name)__rep_seq_grps_AA_pos_only.jld2", "rep_seq_grps_AA_pos_only")
    global rep_seq_grps_AA_no_dels
        rep_seq_grps_AA_no_dels = load("$(folder_name)/$(folder_name)__rep_seq_grps_AA_no_dels.jld2", "rep_seq_grps_AA_no_dels")
    global rep_seq_grps_AA_dels
        rep_seq_grps_AA_dels = load("$(folder_name)/$(folder_name)__rep_seq_grps_AA_dels.jld2", "rep_seq_grps_AA_dels")
    global rep_seq_grps_dels
        rep_seq_grps_dels = load("$(folder_name)/$(folder_name)__rep_seq_grps_dels.jld2", "rep_seq_grps_dels")
    global rep_seq_grps_AA_pos_only_no_dels
        rep_seq_grps_AA_pos_only_no_dels = load("$(folder_name)/$(folder_name)__rep_seq_grps_AA_pos_only_no_dels.jld2", "rep_seq_grps_AA_pos_only_no_dels")
    global rep_seq_grps_del_ranges_ct
        rep_seq_grps_del_ranges_ct = load("$(folder_name)/$(folder_name)__rep_seq_grps_del_ranges_ct.jld2", "rep_seq_grps_del_ranges_ct")
    global rep_seq_grps_AA
        rep_seq_grps_AA = load("$(folder_name)/$(folder_name)__rep_seq_grps_AA.jld2", "rep_seq_grps_AA")
    global rep_seq_grps_nuc_dropout
        rep_seq_grps_nuc_dropout = load("$(folder_name)/$(folder_name)__rep_seq_grps_nuc_dropout.jld2", "rep_seq_grps_nuc_dropout")
#    global rep_seq_grps_unknown_AA
#        rep_seq_grps_unknown_AA = load("$(folder_name)/$(folder_name)__rep_seq_grps_unknown_AA.jld2", "rep_seq_grps_unknown_AA")
#    global rep_seq_grps_mixed_AA_muts
#        rep_seq_grps_mixed_AA_muts = load("$(folder_name)/$(folder_name)__rep_seq_grps_mixed_AA_muts.jld2", "rep_seq_grps_mixed_AA_muts")
    global AA_dels_rep_seq_grps
        AA_dels_rep_seq_grps = load("$(folder_name)/$(folder_name)__AA_dels_rep_seq_grps.jld2", "AA_dels_rep_seq_grps")
    global AA_muts_rep_seq_grps
        AA_muts_rep_seq_grps = load("$(folder_name)/$(folder_name)__AA_muts_rep_seq_grps.jld2", "AA_muts_rep_seq_grps")
    global AA_muts_rep_seq_grps_pos_only
        AA_muts_rep_seq_grps_pos_only = load("$(folder_name)/$(folder_name)__AA_muts_rep_seq_grps_pos_only.jld2", "AA_muts_rep_seq_grps_pos_only")
    global AA_muts_rep_seq_grps_no_dels
        AA_muts_rep_seq_grps_no_dels = load("$(folder_name)/$(folder_name)__AA_muts_rep_seq_grps_no_dels.jld2", "AA_muts_rep_seq_grps_no_dels")
    global nuc_muts_rep_seq_grps
        nuc_muts_rep_seq_grps = load("$(folder_name)/$(folder_name)__nuc_muts_rep_seq_grps.jld2", "nuc_muts_rep_seq_grps")
    global nuc_muts_rep_seq_grps_no_dels
        nuc_muts_rep_seq_grps_no_dels = load("$(folder_name)/$(folder_name)__nuc_muts_rep_seq_grps_no_dels.jld2", "nuc_muts_rep_seq_grps_no_dels")
    global nuc_dels_rep_seq_grps
        nuc_dels_rep_seq_grps = load("$(folder_name)/$(folder_name)__nuc_dels_rep_seq_grps.jld2", "nuc_dels_rep_seq_grps")    
##########################################################################################################################################################################
##########################################################################################################################################################################
################################################ Below: Used to be in ungodly @save form, now changed ####################################################################
##########################################################################################################################################################################
##########################################################################################################################################################################
    global AA_muts_ct_pos_only_adj_sort_by_site
        AA_muts_ct_pos_only_adj_sort_by_site = load("$(folder_name)/$(folder_name)__AA_muts_ct_pos_only_adj_sort_by_site.jld2", "AA_muts_ct_pos_only_adj_sort_by_site")
    global nuc_muts_ct_adj_score_no_dels_sort_by_site
        nuc_muts_ct_adj_score_no_dels_sort_by_site = load("$(folder_name)/$(folder_name)__nuc_muts_ct_adj_score_no_dels_sort_by_site.jld2", "nuc_muts_ct_adj_score_no_dels_sort_by_site")
    global nuc_muts_ct_adj_no_dels_sort_by_site
        nuc_muts_ct_adj_no_dels_sort_by_site = load("$(folder_name)/$(folder_name)__nuc_muts_ct_adj_no_dels_sort_by_site.jld2", "nuc_muts_ct_adj_no_dels_sort_by_site")
    global nuc_muts_ct_adj_sort_by_seq_ct
        nuc_muts_ct_adj_sort_by_seq_ct = load("$(folder_name)/$(folder_name)__nuc_muts_ct_adj_sort_by_seq_ct.jld2", "nuc_muts_ct_adj_sort_by_seq_ct")
    global nuc_muts_ct_adj_score_sort_by_score
        nuc_muts_ct_adj_score_sort_by_score = load("$(folder_name)/$(folder_name)__nuc_muts_ct_adj_score_sort_by_score.jld2", "nuc_muts_ct_adj_score_sort_by_score")
    global AA_muts_ct_adj_score_sort_by_site
        AA_muts_ct_adj_score_sort_by_site = load("$(folder_name)/$(folder_name)__AA_muts_ct_adj_score_sort_by_site.jld2", "AA_muts_ct_adj_score_sort_by_site")
    global AA_muts_ct_pos_only_adj_no_dels_sort_by_seq_ct
        AA_muts_ct_pos_only_adj_no_dels_sort_by_seq_ct = load("$(folder_name)/$(folder_name)__AA_muts_ct_pos_only_adj_no_dels_sort_by_seq_ct.jld2", "AA_muts_ct_pos_only_adj_no_dels_sort_by_seq_ct")
    global AA_muts_ct_pos_only_adj_score_sort_by_site
        AA_muts_ct_pos_only_adj_score_sort_by_site = load("$(folder_name)/$(folder_name)__AA_muts_ct_pos_only_adj_score_sort_by_site.jld2", "AA_muts_ct_pos_only_adj_score_sort_by_site")
    global AA_muts_ct_pos_only_adj_sort_by_seq_ct
        AA_muts_ct_pos_only_adj_sort_by_seq_ct = load("$(folder_name)/$(folder_name)__AA_muts_ct_pos_only_adj_sort_by_seq_ct.jld2", "AA_muts_ct_pos_only_adj_sort_by_seq_ct")
    global AA_muts_ct_adj_score_no_dels_sort_by_site
        AA_muts_ct_adj_score_no_dels_sort_by_site = load("$(folder_name)/$(folder_name)__AA_muts_ct_adj_score_no_dels_sort_by_site.jld2", "AA_muts_ct_adj_score_no_dels_sort_by_site")
    global AA_muts_ct_adj_sort_by_seq_ct
        AA_muts_ct_adj_sort_by_seq_ct = load("$(folder_name)/$(folder_name)__AA_muts_ct_adj_sort_by_seq_ct.jld2", "AA_muts_ct_adj_sort_by_seq_ct")
    global AA_muts_ct_adj_no_dels_sort_by_site
        AA_muts_ct_adj_no_dels_sort_by_site = load("$(folder_name)/$(folder_name)__AA_muts_ct_adj_no_dels_sort_by_site.jld2", "AA_muts_ct_adj_no_dels_sort_by_site")
    global AA_muts_ct_pos_only_adj_score_sort_by_score
        AA_muts_ct_pos_only_adj_score_sort_by_score = load("$(folder_name)/$(folder_name)__AA_muts_ct_pos_only_adj_score_sort_by_score.jld2", "AA_muts_ct_pos_only_adj_score_sort_by_score")
    global AA_muts_ct_pos_only_adj_score_no_dels_sort_by_score
        AA_muts_ct_pos_only_adj_score_no_dels_sort_by_score = load("$(folder_name)/$(folder_name)__AA_muts_ct_pos_only_adj_score_no_dels_sort_by_score.jld2", "AA_muts_ct_pos_only_adj_score_no_dels_sort_by_score")
    global nuc_muts_ct_adj_no_dels_sort_by_seq_ct
        nuc_muts_ct_adj_no_dels_sort_by_seq_ct = load("$(folder_name)/$(folder_name)__nuc_muts_ct_adj_no_dels_sort_by_seq_ct.jld2", "nuc_muts_ct_adj_no_dels_sort_by_seq_ct")
    global AA_muts_ct_adj_sort_by_site
        AA_muts_ct_adj_sort_by_site = load("$(folder_name)/$(folder_name)__AA_muts_ct_adj_sort_by_site.jld2", "AA_muts_ct_adj_sort_by_site")
    global AA_muts_ct_pos_only_adj_no_dels_sort_by_site
        AA_muts_ct_pos_only_adj_no_dels_sort_by_site = load("$(folder_name)/$(folder_name)__AA_muts_ct_pos_only_adj_no_dels_sort_by_site.jld2", "AA_muts_ct_pos_only_adj_no_dels_sort_by_site")
    global AA_muts_ct_adj_score_sort_by_score
        AA_muts_ct_adj_score_sort_by_score = load("$(folder_name)/$(folder_name)__AA_muts_ct_adj_score_sort_by_score.jld2", "AA_muts_ct_adj_score_sort_by_score")
    global nuc_muts_ct_adj_score_no_dels_sort_by_score
        nuc_muts_ct_adj_score_no_dels_sort_by_score = load("$(folder_name)/$(folder_name)__nuc_muts_ct_adj_score_no_dels_sort_by_score.jld2", "nuc_muts_ct_adj_score_no_dels_sort_by_score")
    global nuc_muts_ct_adj_sort_by_site
        nuc_muts_ct_adj_sort_by_site = load("$(folder_name)/$(folder_name)__nuc_muts_ct_adj_sort_by_site.jld2", "nuc_muts_ct_adj_sort_by_site")
    global AA_muts_ct_adj_score_no_dels_sort_by_score
        AA_muts_ct_adj_score_no_dels_sort_by_score = load("$(folder_name)/$(folder_name)__AA_muts_ct_adj_score_no_dels_sort_by_score.jld2", "AA_muts_ct_adj_score_no_dels_sort_by_score")
    global AA_muts_ct_adj_no_dels_sort_by_seq_ct
        AA_muts_ct_adj_no_dels_sort_by_seq_ct = load("$(folder_name)/$(folder_name)__AA_muts_ct_adj_no_dels_sort_by_seq_ct.jld2", "AA_muts_ct_adj_no_dels_sort_by_seq_ct")
    global AA_muts_ct_pos_only_adj_score_no_dels_sort_by_site
        AA_muts_ct_pos_only_adj_score_no_dels_sort_by_site = load("$(folder_name)/$(folder_name)__AA_muts_ct_pos_only_adj_score_no_dels_sort_by_site.jld2", "AA_muts_ct_pos_only_adj_score_no_dels_sort_by_site")
    global nuc_muts_ct_adj_score_sort_by_site
        nuc_muts_ct_adj_score_sort_by_site = load("$(folder_name)/$(folder_name)__nuc_muts_ct_adj_score_sort_by_site.jld2", "nuc_muts_ct_adj_score_sort_by_site")
###########################################################################################################################################################################
###########################################################################################################################################################################
    global AA_muts_ct_no_dels_sort_by_site
        AA_muts_ct_no_dels_sort_by_site = load("$(folder_name)/$(folder_name)__AA_muts_ct_no_dels_sort_by_site.jld2", "AA_muts_ct_no_dels_sort_by_site")
    global AA_muts_ct_pos_only_sort_by_seq_ct
        AA_muts_ct_pos_only_sort_by_seq_ct = load("$(folder_name)/$(folder_name)__AA_muts_ct_pos_only_sort_by_seq_ct.jld2", "AA_muts_ct_pos_only_sort_by_seq_ct")
    global nuc_muts_ct_sort_by_site
        nuc_muts_ct_sort_by_site = load("$(folder_name)/$(folder_name)__nuc_muts_ct_sort_by_site.jld2", "nuc_muts_ct_sort_by_site")
    global excluded_AA
        excluded_AA = load("$(folder_name)/$(folder_name)__excluded_AA.jld2", "excluded_AA")
    global chronic_search_muts
        chronic_search_muts = load("$(folder_name)/$(folder_name)__chronic_search_muts.jld2", "chronic_search_muts")
    global excluded_pos
        excluded_pos = load("$(folder_name)/$(folder_name)__excluded_pos.jld2", "excluded_pos")
    global rep_seqs
        rep_seqs = load("$(folder_name)/$(folder_name)__rep_seqs.jld2", "rep_seqs")
    global non_rep_seqs
        non_rep_seqs = load("$(folder_name)/$(folder_name)__non_rep_seqs.jld2", "non_rep_seqs")
    global all_seqs
        all_seqs = load("$(folder_name)/$(folder_name)__all_seqs.jld2", "all_seqs")
    global domain_mut_density_sort_by_gene
        domain_mut_density_sort_by_gene = load("$(folder_name)/$(folder_name)__domain_mut_density_sort_by_gene.jld2", "domain_mut_density_sort_by_gene")
    global gene_mut_density_sort_by_density
        gene_mut_density_sort_by_density = load("$(folder_name)/$(folder_name)__gene_mut_density_sort_by_density.jld2", "gene_mut_density_sort_by_density")
    global nuc_muts_ct_sort_by_seq_ct
        nuc_muts_ct_sort_by_seq_ct = load("$(folder_name)/$(folder_name)__nuc_muts_ct_sort_by_seq_ct.jld2", "nuc_muts_ct_sort_by_seq_ct")
    global AA_muts_ct_sort_by_seq_ct
        AA_muts_ct_sort_by_seq_ct = load("$(folder_name)/$(folder_name)__AA_muts_ct_sort_by_seq_ct.jld2", "AA_muts_ct_sort_by_seq_ct")
    global AA_muts_ct_no_dels_sort_by_seq_ct
        AA_muts_ct_no_dels_sort_by_seq_ct = load("$(folder_name)/$(folder_name)__AA_muts_ct_no_dels_sort_by_seq_ct.jld2", "AA_muts_ct_no_dels_sort_by_seq_ct")
    global AA_muts_ct_pos_only_no_dels_sort_by_site
        AA_muts_ct_pos_only_no_dels_sort_by_site = load("$(folder_name)/$(folder_name)__AA_muts_ct_pos_only_no_dels_sort_by_site.jld2", "AA_muts_ct_pos_only_no_dels_sort_by_site")
    global gene_mut_density_sort_by_gene
        gene_mut_density_sort_by_gene = load("$(folder_name)/$(folder_name)__gene_mut_density_sort_by_gene.jld2", "gene_mut_density_sort_by_gene")
    global AA_muts_ct_sort_by_site
        AA_muts_ct_sort_by_site = load("$(folder_name)/$(folder_name)__AA_muts_ct_sort_by_site.jld2", "AA_muts_ct_sort_by_site")
    global AA_muts_ct_pos_only_sort_by_site
        AA_muts_ct_pos_only_sort_by_site = load("$(folder_name)/$(folder_name)__AA_muts_ct_pos_only_sort_by_site.jld2", "AA_muts_ct_pos_only_sort_by_site")
    global domain_mut_density_sort_by_density
        domain_mut_density_sort_by_density = load("$(folder_name)/$(folder_name)__domain_mut_density_sort_by_density.jld2", "domain_mut_density_sort_by_density")
    global too_many_reversions
        too_many_reversions = load("$(folder_name)/$(folder_name)__too_many_reversions.jld2", "too_many_reversions")
    global AA_muts_ct_pos_only_no_dels_sort_by_seq_ct
        AA_muts_ct_pos_only_no_dels_sort_by_seq_ct = load("$(folder_name)/$(folder_name)__AA_muts_ct_pos_only_no_dels_sort_by_seq_ct.jld2", "AA_muts_ct_pos_only_no_dels_sort_by_seq_ct")

    global AA_muts_ct_chr_all_ratio_ct_sort
        AA_muts_ct_chr_all_ratio_ct_sort = load("$(folder_name)/$(folder_name)__AA_muts_ct_chr_all_ratio_ct_sort.jld2", "AA_muts_ct_chr_all_ratio_ct_sort")
    global AA_muts_ct_chr_all_ratio_pos_sort
        AA_muts_ct_chr_all_ratio_pos_sort = load("$(folder_name)/$(folder_name)__AA_muts_ct_chr_all_ratio_pos_sort.jld2", "AA_muts_ct_chr_all_ratio_pos_sort")    

    global AA_muts_ct_no_dels_chr_all_ratio_ct_sort
        AA_muts_ct_no_dels_chr_all_ratio_ct_sort = load("$(folder_name)/$(folder_name)__AA_muts_ct_no_dels_chr_all_ratio_ct_sort.jld2", "AA_muts_ct_no_dels_chr_all_ratio_ct_sort")
    global AA_muts_ct_no_dels_chr_all_ratio_pos_sort
        AA_muts_ct_no_dels_chr_all_ratio_pos_sort = load("$(folder_name)/$(folder_name)__AA_muts_ct_no_dels_chr_all_ratio_pos_sort.jld2", "AA_muts_ct_no_dels_chr_all_ratio_pos_sort")
    
    global AA_muts_ct_pos_only_no_dels_chr_all_ratio_pos_sort
        AA_muts_ct_pos_only_no_dels_chr_all_ratio_pos_sort = load("$(folder_name)/$(folder_name)__AA_muts_ct_pos_only_no_dels_chr_all_ratio_pos_sort.jld2", "AA_muts_ct_pos_only_no_dels_chr_all_ratio_pos_sort")
    global AA_muts_ct_pos_only_no_dels_chr_all_ratio_ct_sort
        AA_muts_ct_pos_only_no_dels_chr_all_ratio_ct_sort = load("$(folder_name)/$(folder_name)__AA_muts_ct_pos_only_no_dels_chr_all_ratio_ct_sort.jld2", "AA_muts_ct_pos_only_no_dels_chr_all_ratio_ct_sort")
    
    global avg_AA_subs_per_chr_seq
        avg_AA_subs_per_chr_seq = load("$(folder_name)/$(folder_name)__avg_AA_subs_per_chr_seq.jld2", "avg_AA_subs_per_chr_seq")
    global avg_AA_subs_per_circ_seq
        avg_AA_subs_per_circ_seq = load("$(folder_name)/$(folder_name)__avg_AA_subs_per_circ_seq.jld2", "avg_AA_subs_per_circ_seq")
###########################################################################################################################################################################
###########################################################################################################################################################################
    global NSP_muts_sortByCt_Arr
        NSP_muts_sortByCt_Arr = load("$(folder_name)/$(folder_name)__NSP_muts_sortByCt_Arr.jld2", "NSP_muts_sortByCt_Arr")
    global NSP_muts_sortByPos_Arr
        NSP_muts_sortByPos_Arr = load("$(folder_name)/$(folder_name)__NSP_muts_sortByPos_Arr.jld2", "NSP_muts_sortByPos_Arr")
    global NSP_muts_no_dels_sortByCt_Arr
        NSP_muts_no_dels_sortByCt_Arr = load("$(folder_name)/$(folder_name)__NSP_muts_no_dels_sortByCt_Arr.jld2", "NSP_muts_no_dels_sortByCt_Arr")
    global NSP_muts_no_dels_sortByPos_Arr
        NSP_muts_no_dels_sortByPos_Arr = load("$(folder_name)/$(folder_name)__NSP_muts_no_dels_sortByPos_Arr.jld2", "NSP_muts_no_dels_sortByPos_Arr")
    global NSP1_muts_sortByCt
        NSP1_muts_sortByCt = load("$(folder_name)/$(folder_name)__NSP1_muts_sortByCt.jld2", "NSP1_muts_sortByCt")
    global NSP1_muts_sortByPos
        NSP1_muts_sortByPos = load("$(folder_name)/$(folder_name)__NSP1_muts_sortByPos.jld2", "NSP1_muts_sortByPos")
    global NSP2_muts_sortByCt
        NSP2_muts_sortByCt = load("$(folder_name)/$(folder_name)__NSP2_muts_sortByCt.jld2", "NSP2_muts_sortByCt")
    global NSP2_muts_sortByPos
        NSP2_muts_sortByPos = load("$(folder_name)/$(folder_name)__NSP2_muts_sortByPos.jld2", "NSP2_muts_sortByPos")
    global NSP3_muts_sortByCt
        NSP3_muts_sortByCt = load("$(folder_name)/$(folder_name)__NSP3_muts_sortByCt.jld2", "NSP3_muts_sortByCt")
    global NSP3_muts_sortByPos
        NSP3_muts_sortByPos = load("$(folder_name)/$(folder_name)__NSP3_muts_sortByPos.jld2", "NSP3_muts_sortByPos")
    global NSP4_muts_sortByCt
        NSP4_muts_sortByCt = load("$(folder_name)/$(folder_name)__NSP4_muts_sortByCt.jld2", "NSP4_muts_sortByCt")
    global NSP4_muts_sortByPos
        NSP4_muts_sortByPos = load("$(folder_name)/$(folder_name)__NSP4_muts_sortByPos.jld2", "NSP4_muts_sortByPos")
    global NSP5_muts_sortByCt
        NSP5_muts_sortByCt = load("$(folder_name)/$(folder_name)__NSP5_muts_sortByCt.jld2", "NSP5_muts_sortByCt")
    global NSP5_muts_sortByPos
        NSP5_muts_sortByPos = load("$(folder_name)/$(folder_name)__NSP5_muts_sortByPos.jld2", "NSP5_muts_sortByPos")
    global NSP6_muts_sortByCt
        NSP6_muts_sortByCt = load("$(folder_name)/$(folder_name)__NSP6_muts_sortByCt.jld2", "NSP6_muts_sortByCt")
    global NSP6_muts_sortByPos
        NSP6_muts_sortByPos = load("$(folder_name)/$(folder_name)__NSP6_muts_sortByPos.jld2", "NSP6_muts_sortByPos")
    global NSP7_muts_sortByCt
        NSP7_muts_sortByCt = load("$(folder_name)/$(folder_name)__NSP7_muts_sortByCt.jld2", "NSP7_muts_sortByCt")
    global NSP7_muts_sortByPos
        NSP7_muts_sortByPos = load("$(folder_name)/$(folder_name)__NSP7_muts_sortByPos.jld2", "NSP7_muts_sortByPos")
    global NSP8_muts_sortByCt
        NSP8_muts_sortByCt = load("$(folder_name)/$(folder_name)__NSP8_muts_sortByCt.jld2", "NSP8_muts_sortByCt")
    global NSP8_muts_sortByPos
        NSP8_muts_sortByPos = load("$(folder_name)/$(folder_name)__NSP8_muts_sortByPos.jld2", "NSP8_muts_sortByPos")
    global NSP9_muts_sortByCt
        NSP9_muts_sortByCt = load("$(folder_name)/$(folder_name)__NSP9_muts_sortByCt.jld2", "NSP9_muts_sortByCt")
    global NSP9_muts_sortByPos
        NSP9_muts_sortByPos = load("$(folder_name)/$(folder_name)__NSP9_muts_sortByPos.jld2", "NSP9_muts_sortByPos")
    global NSP10_muts_sortByCt
        NSP10_muts_sortByCt = load("$(folder_name)/$(folder_name)__NSP10_muts_sortByCt.jld2", "NSP10_muts_sortByCt")
    global NSP10_muts_sortByPos
        NSP10_muts_sortByPos = load("$(folder_name)/$(folder_name)__NSP10_muts_sortByPos.jld2", "NSP10_muts_sortByPos")
    global NSP11_muts_sortByCt
        NSP11_muts_sortByCt = load("$(folder_name)/$(folder_name)__NSP11_muts_sortByCt.jld2", "NSP11_muts_sortByCt")
    global NSP11_muts_sortByPos
        NSP11_muts_sortByPos = load("$(folder_name)/$(folder_name)__NSP11_muts_sortByPos.jld2", "NSP11_muts_sortByPos")
    global NSP12_muts_sortByCt
        NSP12_muts_sortByCt = load("$(folder_name)/$(folder_name)__NSP12_muts_sortByCt.jld2", "NSP12_muts_sortByCt")
    global NSP12_muts_sortByPos
        NSP12_muts_sortByPos = load("$(folder_name)/$(folder_name)__NSP12_muts_sortByPos.jld2", "NSP12_muts_sortByPos")
    global NSP13_muts_sortByCt
        NSP13_muts_sortByCt = load("$(folder_name)/$(folder_name)__NSP13_muts_sortByCt.jld2", "NSP13_muts_sortByCt")
    global NSP13_muts_sortByPos
        NSP13_muts_sortByPos = load("$(folder_name)/$(folder_name)__NSP13_muts_sortByPos.jld2", "NSP13_muts_sortByPos")
    global NSP14_muts_sortByCt
        NSP14_muts_sortByCt = load("$(folder_name)/$(folder_name)__NSP14_muts_sortByCt.jld2", "NSP14_muts_sortByCt")
    global NSP14_muts_sortByPos
        NSP14_muts_sortByPos = load("$(folder_name)/$(folder_name)__NSP14_muts_sortByPos.jld2", "NSP14_muts_sortByPos")
    global NSP15_muts_sortByCt
        NSP15_muts_sortByCt = load("$(folder_name)/$(folder_name)__NSP15_muts_sortByCt.jld2", "NSP15_muts_sortByCt")
    global NSP15_muts_sortByPos
        NSP15_muts_sortByPos = load("$(folder_name)/$(folder_name)__NSP15_muts_sortByPos.jld2", "NSP15_muts_sortByPos")
    global NSP16_muts_sortByCt
        NSP16_muts_sortByCt = load("$(folder_name)/$(folder_name)__NSP16_muts_sortByCt.jld2", "NSP16_muts_sortByCt")
    global NSP16_muts_sortByPos
        NSP16_muts_sortByPos = load("$(folder_name)/$(folder_name)__NSP16_muts_sortByPos.jld2", "NSP16_muts_sortByPos")
    global NSP1_muts_no_dels_sortByCt
        NSP1_muts_no_dels_sortByCt = load("$(folder_name)/$(folder_name)__NSP1_muts_no_dels_sortByCt.jld2", "NSP1_muts_no_dels_sortByCt")
    global NSP1_muts_no_dels_sortByPos
        NSP1_muts_no_dels_sortByPos = load("$(folder_name)/$(folder_name)__NSP1_muts_no_dels_sortByPos.jld2", "NSP1_muts_no_dels_sortByPos")
    global NSP2_muts_no_dels_sortByCt
        NSP2_muts_no_dels_sortByCt = load("$(folder_name)/$(folder_name)__NSP2_muts_no_dels_sortByCt.jld2", "NSP2_muts_no_dels_sortByCt")
    global NSP2_muts_no_dels_sortByPos
        NSP2_muts_no_dels_sortByPos = load("$(folder_name)/$(folder_name)__NSP2_muts_no_dels_sortByPos.jld2", "NSP2_muts_no_dels_sortByPos")
    global NSP3_muts_no_dels_sortByCt
        NSP3_muts_no_dels_sortByCt = load("$(folder_name)/$(folder_name)__NSP3_muts_no_dels_sortByCt.jld2", "NSP3_muts_no_dels_sortByCt")
    global NSP3_muts_no_dels_sortByPos
        NSP3_muts_no_dels_sortByPos = load("$(folder_name)/$(folder_name)__NSP3_muts_no_dels_sortByPos.jld2", "NSP3_muts_no_dels_sortByPos")
    global NSP4_muts_no_dels_sortByCt
        NSP4_muts_no_dels_sortByCt = load("$(folder_name)/$(folder_name)__NSP4_muts_no_dels_sortByCt.jld2", "NSP4_muts_no_dels_sortByCt")
    global NSP4_muts_no_dels_sortByPos
        NSP4_muts_no_dels_sortByPos = load("$(folder_name)/$(folder_name)__NSP4_muts_no_dels_sortByPos.jld2", "NSP4_muts_no_dels_sortByPos")
    global NSP5_muts_no_dels_sortByCt
        NSP5_muts_no_dels_sortByCt = load("$(folder_name)/$(folder_name)__NSP5_muts_no_dels_sortByCt.jld2", "NSP5_muts_no_dels_sortByCt")
    global NSP5_muts_no_dels_sortByPos
        NSP5_muts_no_dels_sortByPos = load("$(folder_name)/$(folder_name)__NSP5_muts_no_dels_sortByPos.jld2", "NSP5_muts_no_dels_sortByPos")
    global NSP6_muts_no_dels_sortByCt
        NSP6_muts_no_dels_sortByCt = load("$(folder_name)/$(folder_name)__NSP6_muts_no_dels_sortByCt.jld2", "NSP6_muts_no_dels_sortByCt")
    global NSP6_muts_no_dels_sortByPos
        NSP6_muts_no_dels_sortByPos = load("$(folder_name)/$(folder_name)__NSP6_muts_no_dels_sortByPos.jld2", "NSP6_muts_no_dels_sortByPos")
    global NSP7_muts_no_dels_sortByCt
        NSP7_muts_no_dels_sortByCt = load("$(folder_name)/$(folder_name)__NSP7_muts_no_dels_sortByCt.jld2", "NSP7_muts_no_dels_sortByCt")
    global NSP7_muts_no_dels_sortByPos
        NSP7_muts_no_dels_sortByPos = load("$(folder_name)/$(folder_name)__NSP7_muts_no_dels_sortByPos.jld2", "NSP7_muts_no_dels_sortByPos")
    global NSP8_muts_no_dels_sortByCt
        NSP8_muts_no_dels_sortByCt = load("$(folder_name)/$(folder_name)__NSP8_muts_no_dels_sortByCt.jld2", "NSP8_muts_no_dels_sortByCt")
    global NSP8_muts_no_dels_sortByPos
        NSP8_muts_no_dels_sortByPos = load("$(folder_name)/$(folder_name)__NSP8_muts_no_dels_sortByPos.jld2", "NSP8_muts_no_dels_sortByPos")
    global NSP9_muts_no_dels_sortByCt
        NSP9_muts_no_dels_sortByCt = load("$(folder_name)/$(folder_name)__NSP9_muts_no_dels_sortByCt.jld2", "NSP9_muts_no_dels_sortByCt")
    global NSP9_muts_no_dels_sortByPos
        NSP9_muts_no_dels_sortByPos = load("$(folder_name)/$(folder_name)__NSP9_muts_no_dels_sortByPos.jld2", "NSP9_muts_no_dels_sortByPos")
    global NSP10_muts_no_dels_sortByCt
        NSP10_muts_no_dels_sortByCt = load("$(folder_name)/$(folder_name)__NSP10_muts_no_dels_sortByCt.jld2", "NSP10_muts_no_dels_sortByCt")
    global NSP10_muts_no_dels_sortByPos
        NSP10_muts_no_dels_sortByPos = load("$(folder_name)/$(folder_name)__NSP10_muts_no_dels_sortByPos.jld2", "NSP10_muts_no_dels_sortByPos")
    global NSP11_muts_no_dels_sortByCt
        NSP11_muts_no_dels_sortByCt = load("$(folder_name)/$(folder_name)__NSP11_muts_no_dels_sortByCt.jld2", "NSP11_muts_no_dels_sortByCt")
    global NSP11_muts_no_dels_sortByPos
        NSP11_muts_no_dels_sortByPos = load("$(folder_name)/$(folder_name)__NSP11_muts_no_dels_sortByPos.jld2", "NSP11_muts_no_dels_sortByPos")
    global NSP12_muts_no_dels_sortByCt
        NSP12_muts_no_dels_sortByCt = load("$(folder_name)/$(folder_name)__NSP12_muts_no_dels_sortByCt.jld2", "NSP12_muts_no_dels_sortByCt")
    global NSP12_muts_no_dels_sortByPos
        NSP12_muts_no_dels_sortByPos = load("$(folder_name)/$(folder_name)__NSP12_muts_no_dels_sortByPos.jld2", "NSP12_muts_no_dels_sortByPos")
    global NSP13_muts_no_dels_sortByCt
        NSP13_muts_no_dels_sortByCt = load("$(folder_name)/$(folder_name)__NSP13_muts_no_dels_sortByCt.jld2", "NSP13_muts_no_dels_sortByCt")
    global NSP13_muts_no_dels_sortByPos
        NSP13_muts_no_dels_sortByPos = load("$(folder_name)/$(folder_name)__NSP13_muts_no_dels_sortByPos.jld2", "NSP13_muts_no_dels_sortByPos")
    global NSP14_muts_no_dels_sortByCt
        NSP14_muts_no_dels_sortByCt = load("$(folder_name)/$(folder_name)__NSP14_muts_no_dels_sortByCt.jld2", "NSP14_muts_no_dels_sortByCt")
    global NSP14_muts_no_dels_sortByPos
        NSP14_muts_no_dels_sortByPos = load("$(folder_name)/$(folder_name)__NSP14_muts_no_dels_sortByPos.jld2", "NSP14_muts_no_dels_sortByPos")
    global NSP15_muts_no_dels_sortByCt
        NSP15_muts_no_dels_sortByCt = load("$(folder_name)/$(folder_name)__NSP15_muts_no_dels_sortByCt.jld2", "NSP15_muts_no_dels_sortByCt")
    global NSP15_muts_no_dels_sortByPos
        NSP15_muts_no_dels_sortByPos = load("$(folder_name)/$(folder_name)__NSP15_muts_no_dels_sortByPos.jld2", "NSP15_muts_no_dels_sortByPos")
    global NSP16_muts_no_dels_sortByCt
        NSP16_muts_no_dels_sortByCt = load("$(folder_name)/$(folder_name)__NSP16_muts_no_dels_sortByCt.jld2", "NSP16_muts_no_dels_sortByCt")
    global NSP16_muts_no_dels_sortByPos
        NSP16_muts_no_dels_sortByPos = load("$(folder_name)/$(folder_name)__NSP16_muts_no_dels_sortByPos.jld2", "NSP16_muts_no_dels_sortByPos")
####################################################################################################
    global all_seqs_set
        all_seqs_set = load("$(folder_name)/$(folder_name)__all_seqs_set.jld2", "all_seqs_set")
    global all_qualifying_seqs
        all_qualifying_seqs = load("$(folder_name)/$(folder_name)__all_qualifying_seqs.jld2", "all_qualifying_seqs")
    global all_qualifying_seqs_set
        all_qualifying_seqs_set = load("$(folder_name)/$(folder_name)__all_qualifying_seqs_set.jld2", "all_qualifying_seqs_set")
    global all_nonqualifying_seqs
        all_nonqualifying_seqs = load("$(folder_name)/$(folder_name)__all_nonqualifying_seqs.jld2", "all_nonqualifying_seqs")
    global all_nonqualifying_seqs_set
        all_nonqualifying_seqs_set = load("$(folder_name)/$(folder_name)__all_nonqualifying_seqs_set.jld2", "all_nonqualifying_seqs_set")
###########################################################################################################################################################################
    global AA_muts_ct_no_dels_no_revs_sort_by_site
        AA_muts_ct_no_dels_no_revs_sort_by_site = load("$(folder_name)/$(folder_name)__AA_muts_ct_no_dels_no_revs_sort_by_site.jld2", "AA_muts_ct_no_dels_no_revs_sort_by_site")
    global AA_muts_ct_no_dels_no_revs_sort_by_seq_ct
        AA_muts_ct_no_dels_no_revs_sort_by_seq_ct = load("$(folder_name)/$(folder_name)__AA_muts_ct_no_dels_no_revs_sort_by_seq_ct.jld2", "AA_muts_ct_no_dels_no_revs_sort_by_seq_ct")
    global chronic_search_muts_v2
        chronic_search_muts_v2 = load("$(folder_name)/$(folder_name)__chronic_search_muts_v2.jld2", "chronic_search_muts_v2")
    global avg_AA_subs_per_chr_seq_no_revs
        avg_AA_subs_per_chr_seq_no_revs = load("$(folder_name)/$(folder_name)__avg_AA_subs_per_chr_seq_no_revs.jld2", "avg_AA_subs_per_chr_seq_no_revs")
###########################################################################################################################################################################
    global nuc_muts_ct_no_dels_chr_all_ratio_ct_sort
        nuc_muts_ct_no_dels_chr_all_ratio_ct_sort = load("$(folder_name)/$(folder_name)__nuc_muts_ct_no_dels_chr_all_ratio_ct_sort.jld2", "nuc_muts_ct_no_dels_chr_all_ratio_ct_sort")
    global nuc_muts_ct_no_dels_chr_all_ratio_pos_sort
        nuc_muts_ct_no_dels_chr_all_ratio_pos_sort = load("$(folder_name)/$(folder_name)__nuc_muts_ct_no_dels_chr_all_ratio_pos_sort.jld2", "nuc_muts_ct_no_dels_chr_all_ratio_pos_sort")
    global nuc_muts_ct_pos_only_no_dels_chr_all_ratio_pos_sort
        nuc_muts_ct_pos_only_no_dels_chr_all_ratio_pos_sort = load("$(folder_name)/$(folder_name)__nuc_muts_ct_pos_only_no_dels_chr_all_ratio_pos_sort.jld2", "nuc_muts_ct_pos_only_no_dels_chr_all_ratio_pos_sort")
    global nuc_muts_ct_pos_only_no_dels_chr_all_ratio_ct_sort
        nuc_muts_ct_pos_only_no_dels_chr_all_ratio_ct_sort = load("$(folder_name)/$(folder_name)__nuc_muts_ct_pos_only_no_dels_chr_all_ratio_ct_sort.jld2", "nuc_muts_ct_pos_only_no_dels_chr_all_ratio_ct_sort")
    global avg_nuc_subs_per_chr_seq
        avg_nuc_subs_per_chr_seq = load("$(folder_name)/$(folder_name)__avg_nuc_subs_per_chr_seq.jld2", "avg_nuc_subs_per_chr_seq")
    global avg_nuc_subs_per_circ_seq
        avg_nuc_subs_per_circ_seq = load("$(folder_name)/$(folder_name)__avg_nuc_subs_per_circ_seq.jld2", "avg_nuc_subs_per_circ_seq")
###########################################################################################################################################################################
###########################################################################################################################################################################
    return date_nuc_mut_ct, date_nuc_mut_ct_no_dels, date_AA_mut_ct, date_AA_mut_ct_no_dels, date_AA_mut_ct_pos_only_no_dels, 
    seq_ct_by_year, seq_ct_by_year_month, seq_ct_by_year_month_day, 
    seq_collection_date, seq_date_index, seq_date_tuple, 
###################################################################################################################################
    seq_clade, seq_clade_display, seq_pango, 
    seq_pango_unaliased, seq_clade_ct, seq_pango_ct, seq_pango_unaliased_ct, 
###################################################################################################################################
    too_many_reversions, 
###################################################################################################################################
    seq_nuc_muts, seq_nuc_del_ranges, seq_nuc_dropout, seq_nuc_muts_WT, seq_nuc_del_ranges_WT, 
################################
    seq_AA_insertions_WT, seq_nuc_insertions_WT,
################################
    seq_AA_muts, seq_AA_muts_no_dels, seq_AA_muts_pos_only, seq_AA_dels, seq_mixed_AA_muts, 
    seq_AA_muts_WT, seq_AA_muts_WT_pos_only, seq_AA_dels_WT, seq_AA_muts_pos_only_no_dels, 
########################################################
    nuc_muts_seq, nuc_muts_seq_WT, 
    nuc_dels_seq, nuc_dels_seq_WT, 
    AA_muts_seq, AA_muts_seq_pos_only, AA_muts_seq_WT, AA_muts_seq_WT_pos_only, AA_muts_seq_pos_only_no_dels, 
    AA_dels_seq, AA_dels_seq_WT, 
######################################################## 
    seq_unknown_AA, seq_unknown_AA_ranges, seq_mixed_nucs, 
############################
    nuc_dels_ct, nuc_muts_ct, nuc_muts_ct_no_dels, 
###############
    AA_dels_ct, AA_muts_ct, AA_muts_ct_no_dels, AA_muts_ct_pos_only, AA_muts_ct_pos_only_no_dels, AA_muts_ct_no_dels_no_revs, 
############################
    nuc_muts_ct_sort_by_site, nuc_muts_ct_sort_by_seq_ct, 
############################ 
    AA_muts_ct_sort_by_site, AA_muts_ct_sort_by_seq_ct, AA_muts_ct_no_dels_sort_by_site, AA_muts_ct_no_dels_no_revs_sort_by_site, 
    AA_muts_ct_no_dels_no_revs_sort_by_seq_ct, AA_muts_ct_no_dels_sort_by_seq_ct,  AA_muts_ct_pos_only_sort_by_site, 
    AA_muts_ct_pos_only_sort_by_seq_ct, AA_muts_ct_pos_only_no_dels_sort_by_site, AA_muts_ct_pos_only_no_dels_sort_by_seq_ct, 
###################################################################################################################################
################################################################################################################################### 
    nuc_muts_ct_adj, nuc_muts_ct_adj_no_dels, nuc_muts_ct_adj_score, nuc_muts_ct_adj_score_no_dels,   
    nuc_muts_ct_adj_sort_by_seq_ct, nuc_muts_ct_adj_no_dels_sort_by_site, nuc_muts_ct_adj_no_dels_sort_by_seq_ct,
    nuc_muts_ct_adj_sort_by_site, nuc_muts_ct_adj_score_no_dels_sort_by_score, AA_muts_ct_pos_only_adj_score_no_dels_sort_by_score, 
    nuc_muts_ct_adj_score_sort_by_site, nuc_muts_ct_adj_score_sort_by_score, nuc_muts_ct_adj_score_no_dels_sort_by_site, 
    AA_muts_ct_adj, AA_muts_ct_adj_no_dels, AA_muts_ct_pos_only_adj, AA_muts_ct_pos_only_adj_no_dels, 
    AA_muts_ct_adj_score, AA_muts_ct_adj_score_no_dels, AA_muts_ct_pos_only_adj_score, AA_muts_ct_pos_only_adj_score_no_dels,
    AA_muts_ct_adj_sort_by_site, AA_muts_ct_adj_sort_by_seq_ct, AA_muts_ct_adj_no_dels_sort_by_site,
    AA_muts_ct_adj_no_dels_sort_by_seq_ct, AA_muts_ct_adj_score_sort_by_score, AA_muts_ct_adj_score_sort_by_site, 
    AA_muts_ct_adj_score_no_dels_sort_by_score, AA_muts_ct_adj_score_no_dels_sort_by_site, 
    AA_muts_ct_pos_only_adj_sort_by_site, AA_muts_ct_pos_only_adj_sort_by_seq_ct, AA_muts_ct_pos_only_adj_no_dels_sort_by_site, 
    AA_muts_ct_pos_only_adj_no_dels_sort_by_seq_ct, AA_muts_ct_pos_only_adj_score_sort_by_site, 
    AA_muts_ct_pos_only_adj_score_sort_by_score, AA_muts_ct_pos_only_adj_score_no_dels_sort_by_site, 
####################################################################################################################################
    AA_muts_ct_chr_all_ratio, AA_muts_ct_no_dels_chr_all_ratio, AA_muts_ct_no_dels_no_revs_chr_all_ratio, AA_muts_ct_pos_only_no_dels_no_revs_chr_all_ratio, 
########################
    AA_muts_ct_chr_all_ratio_ct_sort, AA_muts_ct_chr_all_ratio_pos_sort, 
    AA_muts_ct_no_dels_chr_all_ratio_ct_sort, AA_muts_ct_no_dels_chr_all_ratio_pos_sort, 
    AA_muts_ct_pos_only_no_dels_chr_all_ratio_pos_sort, AA_muts_ct_pos_only_no_dels_chr_all_ratio_ct_sort, 
###################################################### 
    nuc_muts_ct_no_dels_chr_all_ratio, nuc_muts_ct_no_dels_no_revs_chr_all_ratio, nuc_muts_ct_pos_only_no_dels_chr_all_ratio, nuc_muts_ct_pos_only_no_dels_no_revs_chr_all_ratio, 
#######################
    nuc_muts_ct_no_dels_chr_all_ratio_ct_sort, nuc_muts_ct_no_dels_chr_all_ratio_pos_sort, 
    nuc_muts_ct_pos_only_no_dels_chr_all_ratio_pos_sort, nuc_muts_ct_pos_only_no_dels_chr_all_ratio_ct_sort, 
####################################################################################################################################
    NSP_muts, NSP_muts_sortByCt_Arr, NSP_muts_sortByPos_Arr, NSP_muts_no_dels_sortByCt_Arr, NSP_muts_no_dels_sortByPos_Arr, 
    NSP1_muts_sortByCt, NSP2_muts_sortByCt, NSP3_muts_sortByCt, NSP4_muts_sortByCt, NSP5_muts_sortByCt, NSP6_muts_sortByCt,
    NSP7_muts_sortByCt, NSP8_muts_sortByCt, NSP9_muts_sortByCt, NSP10_muts_sortByCt, NSP11_muts_sortByCt, NSP12_muts_sortByCt, 
    NSP13_muts_sortByCt, NSP14_muts_sortByCt, NSP15_muts_sortByCt, NSP16_muts_sortByCt, NSP1_muts_sortByPos, NSP2_muts_sortByPos, 
    NSP3_muts_sortByPos, NSP4_muts_sortByPos, NSP5_muts_sortByPos, NSP6_muts_sortByPos, NSP7_muts_sortByPos, NSP8_muts_sortByPos, 
    NSP9_muts_sortByPos, NSP10_muts_sortByPos, NSP11_muts_sortByPos, NSP12_muts_sortByPos, NSP13_muts_sortByPos, NSP14_muts_sortByPos, 
    NSP15_muts_sortByPos, NSP16_muts_sortByPos, NSP1_muts_no_dels_sortByCt, NSP2_muts_no_dels_sortByCt, NSP3_muts_no_dels_sortByCt, 
    NSP4_muts_no_dels_sortByCt, NSP5_muts_no_dels_sortByCt, NSP6_muts_no_dels_sortByCt, NSP7_muts_no_dels_sortByCt, 
    NSP8_muts_no_dels_sortByCt, NSP9_muts_no_dels_sortByCt, NSP10_muts_no_dels_sortByCt, NSP11_muts_no_dels_sortByCt, 
    NSP12_muts_no_dels_sortByCt, NSP13_muts_no_dels_sortByCt, NSP14_muts_no_dels_sortByCt, NSP15_muts_no_dels_sortByCt, 
    NSP16_muts_no_dels_sortByCt, NSP1_muts_no_dels_sortByPos, NSP2_muts_no_dels_sortByPos, NSP3_muts_no_dels_sortByPos, 
    NSP4_muts_no_dels_sortByPos, NSP5_muts_no_dels_sortByPos, NSP6_muts_no_dels_sortByPos, NSP7_muts_no_dels_sortByPos, 
    NSP8_muts_no_dels_sortByPos, NSP9_muts_no_dels_sortByPos, NSP10_muts_no_dels_sortByPos, NSP11_muts_no_dels_sortByPos, 
    NSP12_muts_no_dels_sortByPos, NSP13_muts_no_dels_sortByPos, NSP14_muts_no_dels_sortByPos, NSP15_muts_no_dels_sortByPos, 
    NSP16_muts_no_dels_sortByPos,
####################################################################################################################################
    chronic_search_muts, chronic_search_muts_v2, 
####################################################################################################################################
    avg_AA_subs_per_circ_seq, avg_nuc_subs_per_circ_seq, avg_AA_subs_per_chr_seq, avg_nuc_subs_per_chr_seq,
####################################################################################################################################
    nuc_muts_rep_seq_grps, nuc_dels_rep_seq_grps, 
    nuc_muts_rep_seq_grps_no_dels, rep_seq_grps_dels, rep_seq_grps_muts_no_dels, rep_seq_grps_del_ranges_ct, rep_seq_grps_nuc_dropout, 
    rep_seq_grps_mixed_nucs, rep_seq_grps_AA_dels, rep_seq_grps_AA_no_dels, AA_muts_rep_seq_grps, AA_dels_rep_seq_grps, 
    AA_muts_rep_seq_grps_no_dels, AA_muts_rep_seq_grps_pos_only,  
    rep_seq_grps_nuc_muts_WT, rep_seq_grps_nuc_dels_WT, rep_seq_grps_AA_muts_WT, rep_seq_grps_AA_dels_WT, nuc_muts_rep_seq_grps_WT, 
    nuc_dels_rep_seq_grps_WT, AA_muts_rep_seq_grps_WT, AA_dels_rep_seq_grps_WT, rep_seq_grps_AA_muts_WT_pos_only, 
    AA_muts_rep_seq_grps_WT_pos_only, rep_seq_grps_pango, rep_seq_grps_pango_unaliased, excluded_pos, excluded_AA, all_seqs, rep_seqs, 
    non_rep_seqs, rep_seq_grps_seqs, rep_seq_grps_muts, rep_seq_grps_clade, rep_seq_grps_AA, rep_seq_grps_AA_pos_only, 
    rep_seq_grps_AA_pos_only_no_dels, non_rep_seqs_AA, non_rep_seqs_AA_pos_only, non_rep_seqs_AA_pos_only_no_dels, 
####################################################################################################################################
    rep_seq_grps_maxmut_nuc_muts_WT, rep_seq_grps_maxmut_nuc_dels_WT, 
    rep_seq_grps_maxmut_AA_muts_WT, rep_seq_grps_maxmut_AA_dels_WT, rep_seq_grps_maxmut_AA_muts_WT_pos_only,
    rep_seq_grps_maxmut_seqs, rep_seq_grps_maxmut_nuc, rep_seq_grps_maxmut_nuc_no_dels, rep_seq_grps_maxmut_dels, 
    rep_seq_grps_maxmut_nuc_dropout, rep_seq_grps_maxmut_mixed_nucs, rep_seq_grps_maxmut_AA, rep_seq_grps_maxmut_AA_no_dels, 
    rep_seq_grps_maxmut_AA_dels, rep_seq_grps_maxmut_AA_pos_only, rep_seq_grps_maxmut_AA_pos_only_no_dels, 
    rep_seq_grps_maxmut_unknown_AA, rep_seq_grps_maxmut_unknown_AA_ranges, rep_seq_grps_maxmut_mixed_AA_muts, rep_seq_grps_maxmut_del_ranges_ct,
####################################################################################################################################
    gene_mut_density_sort_by_gene, gene_mut_density_sort_by_density, domain_mut_density_sort_by_gene, domain_mut_density_sort_by_density, 
    gene_mut_density, domain_mut_density, 
####################################################################################################################################
    all_seqs_set, all_qualifying_seqs, all_qualifying_seqs_set, all_nonqualifying_seqs, all_nonqualifying_seqs_set
####################################################################################################################################
    total_chr_AA_subs, total_nuc_revs_seq, seq_nuc_total_revs, total_AA_revs_seq, seq_AA_total_revs, seq_AA_revs
end
# REMOVED: rep_seq_grps_unknown_AA, rep_seq_grps_mixed_AA_muts,
###########################################################################################################################################################################
###########################################################################################################################################################################
###########################################################################################################################################################################
###########################################################################################################################################################################
###########################################################################################################################################################################
###########################################################################################################################################################################
function gene_sub_pos_ranks_chr(gene_array::Vector{String}, gene_AA_lengths::Dict{String, Int}, AA_muts_ct_pos_only_no_dels::Dict{String, Int} )
    ORF1a_pos_ct = Dict{String, Int}()
    ORF1b_pos_ct = Dict{String, Int}()
    S_pos_ct = Dict{String, Int}()
    ORF3a_pos_ct = Dict{String, Int}()
    E_pos_ct = Dict{String, Int}()
    M_pos_ct = Dict{String, Int}()
    ORF6_pos_ct = Dict{String, Int}()
    ORF7a_pos_ct = Dict{String, Int}()
    ORF7b_pos_ct = Dict{String, Int}()
    ORF8_pos_ct = Dict{String, Int}()
    N_pos_ct = Dict{String, Int}()
    ORF9b_pos_ct = Dict{String, Int}()
    gene_pos_ct_dict_arr = [ORF1a_pos_ct, ORF1b_pos_ct, S_pos_ct, ORF3a_pos_ct, E_pos_ct, M_pos_ct, ORF6_pos_ct, ORF7a_pos_ct, ORF7b_pos_ct, ORF8_pos_ct, N_pos_ct, ORF9b_pos_ct]
######################################################
    ORF1a_pos_ct_v1 = Dict{Int, Int}()
    ORF1b_pos_ct_v1 = Dict{Int, Int}()
    S_pos_ct_v1 = Dict{Int, Int}()
    ORF3a_pos_ct_v1 = Dict{Int, Int}()
    E_pos_ct_v1 = Dict{Int, Int}()
    M_pos_ct_v1 = Dict{Int, Int}()
    ORF6_pos_ct_v1 = Dict{Int, Int}()
    ORF7a_pos_ct_v1 = Dict{Int, Int}()
    ORF7b_pos_ct_v1 = Dict{Int, Int}()
    ORF8_pos_ct_v1 = Dict{Int, Int}()
    N_pos_ct_v1 = Dict{Int, Int}()
    ORF9b_pos_ct_v1 = Dict{Int, Int}()
    gene_pos_ct_v1_dict_arr = [ORF1a_pos_ct_v1, ORF1b_pos_ct_v1, S_pos_ct_v1, ORF3a_pos_ct_v1, E_pos_ct_v1, M_pos_ct_v1, ORF6_pos_ct_v1, ORF7a_pos_ct_v1, ORF7b_pos_ct_v1, ORF8_pos_ct_v1, N_pos_ct_v1, ORF9b_pos_ct_v1]
######################################################
    for i in 1:length(gene_pos_ct_dict_arr)
        dict = gene_pos_ct_dict_arr[i]
        dict_v1 = gene_pos_ct_v1_dict_arr[i]
        gene = gene_array[i]
        gene_len = gene_AA_lengths[gene]
        for j in 1:gene_len
            site = gene*":"*"$(j)"
            dict[site] = 0
            dict_v1[j] = 0
        end
    end
#############################################################
#############################################################
    for (mut, ct) in AA_muts_ct_pos_only_no_dels
        pos = aa_pos_comprehensive_dict[mut]
        if aa_gene_comprehensive_dict[mut] == "ORF1a"
            ORF1a_pos_ct[mut] = ct
            ORF1a_pos_ct_v1[pos] = ct
            elseif aa_gene_comprehensive_dict[mut] == "ORF1b"
            ORF1b_pos_ct[mut] = ct
            ORF1b_pos_ct_v1[pos] = ct
            elseif aa_gene_comprehensive_dict[mut] == "S"
            S_pos_ct[mut] = ct
            S_pos_ct_v1[pos] = ct
            elseif aa_gene_comprehensive_dict[mut] == "ORF3a"
            ORF3a_pos_ct[mut] = ct
            ORF3a_pos_ct_v1[pos] = ct
            elseif aa_gene_comprehensive_dict[mut] == "E"
            E_pos_ct[mut] = ct
            E_pos_ct_v1[pos] = ct
            elseif aa_gene_comprehensive_dict[mut] == "M"
            M_pos_ct[mut] = ct
            M_pos_ct_v1[pos] = ct
            elseif aa_gene_comprehensive_dict[mut] == "ORF6"
            ORF6_pos_ct[mut] = ct
            ORF6_pos_ct_v1[pos] = ct
            elseif aa_gene_comprehensive_dict[mut] == "ORF7a"
            ORF7a_pos_ct[mut] = ct
            ORF7a_pos_ct_v1[pos] = ct
            elseif aa_gene_comprehensive_dict[mut] == "ORF7b"
            ORF7b_pos_ct[mut] = ct
            ORF7b_pos_ct_v1[pos] = ct
            elseif aa_gene_comprehensive_dict[mut] == "ORF8"
            ORF8_pos_ct[mut] = ct
            ORF8_pos_ct_v1[pos] = ct
            elseif aa_gene_comprehensive_dict[mut] == "N"
            N_pos_ct[mut] = ct
            N_pos_ct_v1[pos] = ct
            elseif aa_gene_comprehensive_dict[mut] == "ORF9b"
            ORF9b_pos_ct[mut] = ct
            ORF9b_pos_ct_v1[pos] = ct
        end
    end
######################### 
    global ORF1a_pos_ct_sort_by_pos = sort(collect(ORF1a_pos_ct), by = x -> aa_pos_comprehensive_dict[x[1]])
    global ORF1b_pos_ct_sort_by_pos = sort(collect(ORF1b_pos_ct), by = x -> aa_pos_comprehensive_dict[x[1]])
    global S_pos_ct_sort_by_pos = sort(collect(S_pos_ct), by = x -> aa_pos_comprehensive_dict[x[1]])
    global ORF3a_pos_ct_sort_by_pos = sort(collect(ORF3a_pos_ct), by = x -> aa_pos_comprehensive_dict[x[1]])
    global E_pos_ct_sort_by_pos = sort(collect(E_pos_ct), by = x -> aa_pos_comprehensive_dict[x[1]])
    global M_pos_ct_sort_by_pos = sort(collect(M_pos_ct), by = x -> aa_pos_comprehensive_dict[x[1]])
    global ORF6_pos_ct_sort_by_pos = sort(collect(ORF6_pos_ct), by = x -> aa_pos_comprehensive_dict[x[1]])
    global ORF7a_pos_ct_sort_by_pos = sort(collect(ORF7a_pos_ct), by = x -> aa_pos_comprehensive_dict[x[1]])
    global ORF7b_pos_ct_sort_by_pos = sort(collect(ORF7b_pos_ct), by = x -> aa_pos_comprehensive_dict[x[1]])
    global ORF8_pos_ct_sort_by_pos = sort(collect(ORF8_pos_ct), by = x -> aa_pos_comprehensive_dict[x[1]])
    global N_pos_ct_sort_by_pos = sort(collect(N_pos_ct), by = x -> aa_pos_comprehensive_dict[x[1]])
    global ORF9b_pos_ct_sort_by_pos = sort(collect(ORF9b_pos_ct), by = x -> aa_pos_comprehensive_dict[x[1]])
#########################
    global ORF1a_pos_ct_sort_by_ct = sort(collect(ORF1a_pos_ct), by = x -> x[2], rev=true)
    global ORF1b_pos_ct_sort_by_ct = sort(collect(ORF1b_pos_ct), by = x -> x[2], rev=true)
    global S_pos_ct_sort_by_ct = sort(collect(S_pos_ct), by = x -> x[2], rev=true)
    global ORF3a_pos_ct_sort_by_ct = sort(collect(ORF3a_pos_ct), by = x -> x[2], rev=true)
    global E_pos_ct_sort_by_ct = sort(collect(E_pos_ct), by = x -> x[2], rev=true)
    global M_pos_ct_sort_by_ct = sort(collect(M_pos_ct), by = x -> x[2], rev=true)
    global ORF6_pos_ct_sort_by_ct = sort(collect(ORF6_pos_ct), by = x -> x[2], rev=true)
    global ORF7a_pos_ct_sort_by_ct = sort(collect(ORF7a_pos_ct), by = x -> x[2], rev=true)
    global ORF7b_pos_ct_sort_by_ct = sort(collect(ORF7b_pos_ct), by = x -> x[2], rev=true)
    global ORF8_pos_ct_sort_by_ct = sort(collect(ORF8_pos_ct), by = x -> x[2], rev=true)
    global N_pos_ct_sort_by_ct = sort(collect(N_pos_ct), by = x -> x[2], rev=true)
    global ORF9b_pos_ct_sort_by_ct = sort(collect(ORF9b_pos_ct), by = x -> x[2], rev=true)
#########################
    global ORF1a_pos_ct_sort_by_pos_v1 = sort(collect(ORF1a_pos_ct_v1), by = x -> x[1])
    global ORF1b_pos_ct_sort_by_pos_v1 = sort(collect(ORF1b_pos_ct_v1), by = x -> x[1])
    global S_pos_ct_sort_by_pos_v1 = sort(collect(S_pos_ct_v1), by = x -> x[1])
    global ORF3a_pos_ct_sort_by_pos_v1 = sort(collect(ORF3a_pos_ct_v1), by = x -> x[1])
    global E_pos_ct_sort_by_pos_v1 = sort(collect(E_pos_ct_v1), by = x -> x[1])
    global M_pos_ct_sort_by_pos_v1 = sort(collect(M_pos_ct_v1), by = x -> x[1])
    global ORF6_pos_ct_sort_by_pos_v1 = sort(collect(ORF6_pos_ct_v1), by = x -> x[1])
    global ORF7a_pos_ct_sort_by_pos_v1 = sort(collect(ORF7a_pos_ct_v1), by = x -> x[1])
    global ORF7b_pos_ct_sort_by_pos_v1 = sort(collect(ORF7b_pos_ct_v1), by = x -> x[1])
    global ORF8_pos_ct_sort_by_pos_v1 = sort(collect(ORF8_pos_ct_v1), by = x -> x[1])
    global N_pos_ct_sort_by_pos_v1 = sort(collect(N_pos_ct_v1), by = x -> x[1])
    global ORF9b_pos_ct_sort_by_pos_v1 = sort(collect(ORF9b_pos_ct_v1), by = x -> x[1])
#########################
    global ORF1a_pos_ct_sort_by_ct_v1 = sort(collect(ORF1a_pos_ct_v1), by = x -> x[2], rev=true)
    global ORF1b_pos_ct_sort_by_ct_v1 = sort(collect(ORF1b_pos_ct_v1), by = x -> x[2], rev=true)
    global S_pos_ct_sort_by_ct_v1 = sort(collect(S_pos_ct_v1), by = x -> x[2], rev=true)
    global ORF3a_pos_ct_sort_by_ct_v1 = sort(collect(ORF3a_pos_ct_v1), by = x -> x[2], rev=true)
    global E_pos_ct_sort_by_ct_v1 = sort(collect(E_pos_ct_v1), by = x -> x[2], rev=true)
    global M_pos_ct_sort_by_ct_v1 = sort(collect(M_pos_ct_v1), by = x -> x[2], rev=true)
    global ORF6_pos_ct_sort_by_ct_v1 = sort(collect(ORF6_pos_ct_v1), by = x -> x[2], rev=true)
    global ORF7a_pos_ct_sort_by_ct_v1 = sort(collect(ORF7a_pos_ct_v1), by = x -> x[2], rev=true)
    global ORF7b_pos_ct_sort_by_ct_v1 = sort(collect(ORF7b_pos_ct_v1), by = x -> x[2], rev=true)
    global ORF8_pos_ct_sort_by_ct_v1 = sort(collect(ORF8_pos_ct_v1), by = x -> x[2], rev=true)
    global N_pos_ct_sort_by_ct_v1 = sort(collect(N_pos_ct_v1), by = x -> x[2], rev=true)
    global ORF9b_pos_ct_sort_by_ct_v1 = sort(collect(ORF9b_pos_ct_v1), by = x -> x[2], rev=true)
end
#####################################################################################################################################
#####################################################################################################################################
#####################################################################################################################################
function gene_sub_ranks_chr(gene_array::Vector{String}, sub_types_at_every_site_combined::Dict{String, Dict{Int, Set{String}}}, AA_muts_ct_no_dels::Dict{String, Int} )
    ORF1a_ct = Dict{String, Int}()
    ORF1b_ct = Dict{String, Int}()
    S_ct = Dict{String, Int}()
    ORF3a_ct = Dict{String, Int}()
    E_ct = Dict{String, Int}()
    M_ct = Dict{String, Int}()
    ORF6_ct = Dict{String, Int}()
    ORF7a_ct = Dict{String, Int}()
    ORF7b_ct = Dict{String, Int}()
    ORF8_ct = Dict{String, Int}()
    N_ct = Dict{String, Int}()
    ORF9b_ct = Dict{String, Int}()
#############################################################
    gene_ct_dict_arr = [ORF1a_ct, ORF1b_ct, S_ct, ORF3a_ct, E_ct, M_ct, ORF6_ct, ORF7a_ct, ORF7b_ct, ORF8_ct, N_ct, ORF9b_ct] 
#############################################################
#                                        Gene      AAsite  all_sub_types_at_AAsite (e.g., AV, AT, TA, etc)
# sub_types_at_every_site_combined = Dict{String, Dict{Int, Set{String}}}()
    for i in 1:length(gene_array)
        gene = gene_array[i]
        gene_ct_dict = gene_ct_dict_arr[i]
        for (AAsite, mut_type_set) in sub_types_at_every_site_combined[gene]
            for mut_type in mut_type_set
                sub = gene*":"*string(mut_type[1])*"$(AAsite)"*string(mut_type[end])
                gene_ct_dict[sub] = 0
            end
        end
    end
#############################################################
    for (mut, ct) in AA_muts_ct_no_dels
        if aa_gene_comprehensive_dict[mut] == "ORF1a"
            ORF1a_ct[mut] = ct
            elseif aa_gene_comprehensive_dict[mut] == "ORF1b"
            ORF1b_ct[mut] = ct
            elseif aa_gene_comprehensive_dict[mut] == "S"
            S_ct[mut] = ct
            elseif aa_gene_comprehensive_dict[mut] == "ORF3a"
            ORF3a_ct[mut] = ct
            elseif aa_gene_comprehensive_dict[mut] == "E"
            E_ct[mut] = ct
            elseif aa_gene_comprehensive_dict[mut] == "M"
            M_ct[mut] = ct
            elseif aa_gene_comprehensive_dict[mut] == "ORF6"
            ORF6_ct[mut] = ct
            elseif aa_gene_comprehensive_dict[mut] == "ORF7a"
            ORF7a_ct[mut] = ct
            elseif aa_gene_comprehensive_dict[mut] == "ORF7b"
            ORF7b_ct[mut] = ct
            elseif aa_gene_comprehensive_dict[mut] == "ORF8"
            ORF8_ct[mut] = ct
            elseif aa_gene_comprehensive_dict[mut] == "N"
            N_ct[mut] = ct
            elseif aa_gene_comprehensive_dict[mut] == "ORF9b"
            ORF9b_ct[mut] = ct
        end
    end
    fin_sortkey(m) = (aa_pos_comprehensive_dict[m], refAA_comprehensive_dict[m]*qryAA_comprehensive_dict[m])
########################################## 
    global ORF1a_ct_sort_by_pos = sort(collect(ORF1a_ct), by = x -> fin_sortkey(x[1]))
    global ORF1b_ct_sort_by_pos = sort(collect(ORF1b_ct), by = x -> fin_sortkey(x[1]))
    global S_ct_sort_by_pos = sort(collect(S_ct), by = x -> fin_sortkey(x[1]))
    global ORF3a_ct_sort_by_pos = sort(collect(ORF3a_ct), by = x -> fin_sortkey(x[1]))
    global E_ct_sort_by_pos = sort(collect(E_ct), by = x -> fin_sortkey(x[1]))
    global M_ct_sort_by_pos = sort(collect(M_ct), by = x -> fin_sortkey(x[1]))
    global ORF6_ct_sort_by_pos = sort(collect(ORF6_ct), by = x -> fin_sortkey(x[1]))
    global ORF7a_ct_sort_by_pos = sort(collect(ORF7a_ct), by = x -> fin_sortkey(x[1]))
    global ORF7b_ct_sort_by_pos = sort(collect(ORF7b_ct), by = x -> fin_sortkey(x[1]))
    global ORF8_ct_sort_by_pos = sort(collect(ORF8_ct), by = x -> fin_sortkey(x[1]))
    global N_ct_sort_by_pos = sort(collect(N_ct), by = x -> fin_sortkey(x[1]))
    global ORF9b_ct_sort_by_pos = sort(collect(ORF9b_ct), by = x -> fin_sortkey(x[1]))
##########################################
    global ORF1a_ct_sort_by_ct = sort(collect(ORF1a_ct), by = x -> x[2], rev=true)
    global ORF1b_ct_sort_by_ct = sort(collect(ORF1b_ct), by = x -> x[2], rev=true)
    global S_ct_sort_by_ct = sort(collect(S_ct), by = x -> x[2], rev=true)
    global ORF3a_ct_sort_by_ct = sort(collect(ORF3a_ct), by = x -> x[2], rev=true)
    global E_ct_sort_by_ct = sort(collect(E_ct), by = x -> x[2], rev=true)
    global M_ct_sort_by_ct = sort(collect(M_ct), by = x -> x[2], rev=true)
    global ORF6_ct_sort_by_ct = sort(collect(ORF6_ct), by = x -> x[2], rev=true)
    global ORF7a_ct_sort_by_ct = sort(collect(ORF7a_ct), by = x -> x[2], rev=true)
    global ORF7b_ct_sort_by_ct = sort(collect(ORF7b_ct), by = x -> x[2], rev=true)
    global ORF8_ct_sort_by_ct = sort(collect(ORF8_ct), by = x -> x[2], rev=true)
    global N_ct_sort_by_ct = sort(collect(N_ct), by = x -> x[2], rev=true)
    global ORF9b_ct_sort_by_ct = sort(collect(ORF9b_ct), by = x -> x[2], rev=true)
end
#####################################################################################################################################
function NSP_sub_ranks_chr(NSP_array::Vector{String}, NSP_sub_types_at_every_site_combined::Dict{String, Dict{Int, Set{String}}}, AA_muts_ct_no_dels::Dict{String, Int} )
    NSP1_ct = Dict{String, Int}()
    NSP2_ct = Dict{String, Int}()
    NSP3_ct = Dict{String, Int}()
    NSP4_ct = Dict{String, Int}()
    NSP5_ct = Dict{String, Int}()
    NSP6_ct = Dict{String, Int}()
    NSP7_ct = Dict{String, Int}()
    NSP8_ct = Dict{String, Int}()
    NSP9_ct = Dict{String, Int}()
    NSP10_ct = Dict{String, Int}()
    NSP11_ct = Dict{String, Int}()
    NSP12_ct = Dict{String, Int}()
    NSP13_ct = Dict{String, Int}()
    NSP14_ct = Dict{String, Int}()
    NSP15_ct = Dict{String, Int}()
    NSP16_ct = Dict{String, Int}()
#############################################################
    NSP_ct_dict_arr = [NSP1_ct, NSP2_ct, NSP3_ct, NSP4_ct, NSP5_ct, NSP6_ct, NSP7_ct, NSP8_ct, NSP9_ct, NSP10_ct, NSP11_ct, NSP12_ct, NSP13_ct, NSP14_ct, NSP15_ct, NSP16_ct]
#                                              NSP       AAsite  all_sub_types_at_AAsite (e.g., AV, AT, TA, etc)
# NSP_sub_types_at_every_site_combined = Dict{String, Dict{Int, Set{String}}}()   
    for i in 1:length(NSP_array)
        NSP = NSP_array[i]
        NSP_ct_dict = NSP_ct_dict_arr[i]
        for (AAsite, mut_type_set) in NSP_sub_types_at_every_site_combined[NSP]
            for mut_type in mut_type_set
                sub = NSP*":"*string(mut_type[1])*"$(AAsite)"*string(mut_type[end])
                NSP_ct_dict[sub] = 0
            end
        end
    end   
    for (sub, ct) in AA_muts_ct_no_dels
        gene = aa_gene_comprehensive_dict[sub]
        if gene == "ORF1a" || gene == "ORF1b"
            AA_site = aa_pos_comprehensive_dict[sub]
            if AA_site < 4402
                NSP_sub = ORF1abMut_to_NSP(sub)
                NSP = NSP_muts_gene_dict[NSP_sub]
                NSP_pos = NSP_muts_pos_dict[NSP_sub]
                NSP_num = parse(Int, split(NSP, "P")[2])
                NSP_dict = NSP_ct_dict_arr[NSP_num]
                NSP_dict[NSP_sub] = ct
            end
        end
    end
    fin_sortkey(m) = (NSP_muts_pos_dict[m], NSP_ref_AA_dict[m]*NSP_qry_AA_dict[m])
#############################
    global NSP1_ct_sort_by_pos = sort(collect(NSP1_ct), by = x -> fin_sortkey(x[1]))
    global NSP2_ct_sort_by_pos = sort(collect(NSP2_ct), by = x -> fin_sortkey(x[1]))
    global NSP3_ct_sort_by_pos = sort(collect(NSP3_ct), by = x -> fin_sortkey(x[1]))
    global NSP4_ct_sort_by_pos = sort(collect(NSP4_ct), by = x -> fin_sortkey(x[1]))
    global NSP5_ct_sort_by_pos = sort(collect(NSP5_ct), by = x -> fin_sortkey(x[1]))
    global NSP6_ct_sort_by_pos = sort(collect(NSP6_ct), by = x -> fin_sortkey(x[1]))
    global NSP7_ct_sort_by_pos = sort(collect(NSP7_ct), by = x -> fin_sortkey(x[1]))
    global NSP8_ct_sort_by_pos = sort(collect(NSP8_ct), by = x -> fin_sortkey(x[1]))
    global NSP9_ct_sort_by_pos = sort(collect(NSP9_ct), by = x -> fin_sortkey(x[1]))
    global NSP10_ct_sort_by_pos = sort(collect(NSP10_ct), by = x -> fin_sortkey(x[1]))
    global NSP11_ct_sort_by_pos = sort(collect(NSP11_ct), by = x -> fin_sortkey(x[1]))
    global NSP12_ct_sort_by_pos = sort(collect(NSP12_ct), by = x -> fin_sortkey(x[1]))
    global NSP13_ct_sort_by_pos = sort(collect(NSP13_ct), by = x -> fin_sortkey(x[1]))
    global NSP14_ct_sort_by_pos = sort(collect(NSP14_ct), by = x -> fin_sortkey(x[1]))
    global NSP15_ct_sort_by_pos = sort(collect(NSP15_ct), by = x -> fin_sortkey(x[1]))
    global NSP16_ct_sort_by_pos = sort(collect(NSP16_ct), by = x -> fin_sortkey(x[1]))
#############################
    global NSP1_ct_sort_by_ct = sort(collect(NSP1_ct), by = x -> x[2], rev=true)
    global NSP2_ct_sort_by_ct = sort(collect(NSP2_ct), by = x -> x[2], rev=true)
    global NSP3_ct_sort_by_ct = sort(collect(NSP3_ct), by = x -> x[2], rev=true)
    global NSP4_ct_sort_by_ct = sort(collect(NSP4_ct), by = x -> x[2], rev=true)
    global NSP5_ct_sort_by_ct = sort(collect(NSP5_ct), by = x -> x[2], rev=true)
    global NSP6_ct_sort_by_ct = sort(collect(NSP6_ct), by = x -> x[2], rev=true)
    global NSP7_ct_sort_by_ct = sort(collect(NSP7_ct), by = x -> x[2], rev=true)
    global NSP8_ct_sort_by_ct = sort(collect(NSP8_ct), by = x -> x[2], rev=true)
    global NSP9_ct_sort_by_ct = sort(collect(NSP9_ct), by = x -> x[2], rev=true)
    global NSP10_ct_sort_by_ct = sort(collect(NSP10_ct), by = x -> x[2], rev=true)
    global NSP11_ct_sort_by_ct = sort(collect(NSP11_ct), by = x -> x[2], rev=true)
    global NSP12_ct_sort_by_ct = sort(collect(NSP12_ct), by = x -> x[2], rev=true)
    global NSP13_ct_sort_by_ct = sort(collect(NSP13_ct), by = x -> x[2], rev=true)
    global NSP14_ct_sort_by_ct = sort(collect(NSP14_ct), by = x -> x[2], rev=true)
    global NSP15_ct_sort_by_ct = sort(collect(NSP15_ct), by = x -> x[2], rev=true)
    global NSP16_ct_sort_by_ct = sort(collect(NSP16_ct), by = x -> x[2], rev=true)
end
#####################################################################################################################################
function NSP_sub_pos_ranks_chr(NSP_array::Vector{String}, NSP_sub_types_at_every_site_combined::Dict{String, Dict{Int, Set{String}}}, AA_muts_ct_pos_only_no_dels::Dict{String, Int} )
    NSP1_pos_ct = Dict{String, Int}()
    NSP2_pos_ct = Dict{String, Int}()
    NSP3_pos_ct = Dict{String, Int}()
    NSP4_pos_ct = Dict{String, Int}()
    NSP5_pos_ct = Dict{String, Int}()
    NSP6_pos_ct = Dict{String, Int}()
    NSP7_pos_ct = Dict{String, Int}()
    NSP8_pos_ct = Dict{String, Int}()
    NSP9_pos_ct = Dict{String, Int}()
    NSP10_pos_ct = Dict{String, Int}()
    NSP11_pos_ct = Dict{String, Int}()
    NSP12_pos_ct = Dict{String, Int}()
    NSP13_pos_ct = Dict{String, Int}()
    NSP14_pos_ct = Dict{String, Int}()
    NSP15_pos_ct = Dict{String, Int}()
    NSP16_pos_ct = Dict{String, Int}()
    NSP_pos_ct_array = [NSP1_pos_ct, NSP2_pos_ct, NSP3_pos_ct, NSP4_pos_ct, NSP5_pos_ct, NSP6_pos_ct, NSP7_pos_ct, NSP8_pos_ct, NSP9_pos_ct, NSP10_pos_ct, NSP11_pos_ct, NSP12_pos_ct, NSP13_pos_ct, NSP14_pos_ct, NSP15_pos_ct, NSP16_pos_ct]
##########################################
    NSP1_pos_ct_v1 = Dict{Int, Int}()
    NSP2_pos_ct_v1 = Dict{Int, Int}()
    NSP3_pos_ct_v1 = Dict{Int, Int}()
    NSP4_pos_ct_v1 = Dict{Int, Int}()
    NSP5_pos_ct_v1 = Dict{Int, Int}()
    NSP6_pos_ct_v1 = Dict{Int, Int}()
    NSP7_pos_ct_v1 = Dict{Int, Int}()
    NSP8_pos_ct_v1 = Dict{Int, Int}()
    NSP9_pos_ct_v1 = Dict{Int, Int}()
    NSP10_pos_ct_v1 = Dict{Int, Int}()
    NSP11_pos_ct_v1 = Dict{Int, Int}()
    NSP12_pos_ct_v1 = Dict{Int, Int}()
    NSP13_pos_ct_v1 = Dict{Int, Int}()
    NSP14_pos_ct_v1 = Dict{Int, Int}()
    NSP15_pos_ct_v1 = Dict{Int, Int}()
    NSP16_pos_ct_v1 = Dict{Int, Int}()
    NSP_pos_ct_v1_array = [NSP1_pos_ct_v1, NSP2_pos_ct_v1, NSP3_pos_ct_v1, NSP4_pos_ct_v1, NSP5_pos_ct_v1, NSP6_pos_ct_v1, NSP7_pos_ct_v1, NSP8_pos_ct_v1, NSP9_pos_ct_v1, NSP10_pos_ct_v1, NSP11_pos_ct_v1, NSP12_pos_ct_v1, NSP13_pos_ct_v1, NSP14_pos_ct_v1, NSP15_pos_ct_v1, NSP16_pos_ct_v1]
#######################################################
    NSP_pos_ct = Dict{String, Int}()
    for i in 1:length(NSP_pos_ct_array)
        NSP_dict = NSP_pos_ct_array[i]
        NSP_dict_v1 = NSP_pos_ct_v1_array[i]
        NSP = NSP_array[i]
        NSP_len = NSP_AA_size[NSP]
        for j in 1:NSP_len
            NSP_pos = NSP*":"*"$(j)"
            NSP_dict[NSP_pos] = 0
            NSP_dict_v1[j] = 0
            NSP_pos_ct[NSP_pos] = 0
        end
    end
##########################################################################################################################
##########################################################################################################################
    NSP1_ct = Dict{String, Int}()
    NSP2_ct = Dict{String, Int}()
    NSP3_ct = Dict{String, Int}()
    NSP4_ct = Dict{String, Int}()
    NSP5_ct = Dict{String, Int}()
    NSP6_ct = Dict{String, Int}()
    NSP7_ct = Dict{String, Int}()
    NSP8_ct = Dict{String, Int}()
    NSP9_ct = Dict{String, Int}()
    NSP10_ct = Dict{String, Int}()
    NSP11_ct = Dict{String, Int}()
    NSP12_ct = Dict{String, Int}()
    NSP13_ct = Dict{String, Int}()
    NSP14_ct = Dict{String, Int}()
    NSP15_ct = Dict{String, Int}()
    NSP16_ct = Dict{String, Int}()
#############################################################
    NSP_ct_dict_arr = [NSP1_ct, NSP2_ct, NSP3_ct, NSP4_ct, NSP5_ct, NSP6_ct, NSP7_ct, NSP8_ct, NSP9_ct, NSP10_ct, NSP11_ct, NSP12_ct, NSP13_ct, NSP14_ct, NSP15_ct, NSP16_ct]
    for i in 1:length(NSP_array)
        NSP = NSP_array[i]
        NSP_ct_dict = NSP_ct_dict_arr[i]
        for (AAsite, mut_type_set) in NSP_sub_types_at_every_site_combined[NSP]
            for mut_type in mut_type_set
                sub = NSP*":"*string(mut_type[1])*"$(AAsite)"*string(mut_type[end])
                NSP_ct_dict[sub] = 0
            end
        end
    end   
    for (sub, ct) in AA_muts_ct_no_dels
        gene = aa_gene_comprehensive_dict[sub]
        if gene == "ORF1a" || gene == "ORF1b"
            NSP_sub = ORF1abMut_to_NSP(sub)
            NSP = NSP_muts_gene_dict[NSP_sub]
            NSP_pos = NSP_muts_pos_dict[NSP_sub]
            NSP_num = parse(Int, split(NSP, "P")[2])
            NSP_dict = NSP_ct_dict_arr[NSP_num]
            NSP_dict[NSP_sub] = ct
        end
    end
##########################################################################################################################
##########################################################################################################################
    for i in 1:length(NSP_ct_dict_arr)
        NSP = NSP_array[i]
        NSP_dict = NSP_ct_dict_arr[i]
        NSP_pos_dict = NSP_pos_ct_array[i]
        NSP_pos_dict_v1 = NSP_pos_ct_v1_array[i]
        for (sub, ct) in NSP_dict
            pos = NSP_muts_pos_dict[sub]
            sub_pos = NSP*":"*"$(pos)"
            NSP_pos_dict[sub_pos] += ct
            NSP_pos_dict_v1[pos] += ct
            NSP_pos_ct[sub_pos] += ct
        end
    end
    global NSP1_pos_ct_sort_by_pos = sort(collect(NSP1_pos_ct), by = x -> NSP_muts_pos_dict[x[1]])
    global NSP2_pos_ct_sort_by_pos = sort(collect(NSP2_pos_ct), by = x -> NSP_muts_pos_dict[x[1]])
    global NSP3_pos_ct_sort_by_pos = sort(collect(NSP3_pos_ct), by = x -> NSP_muts_pos_dict[x[1]])
    global NSP4_pos_ct_sort_by_pos = sort(collect(NSP4_pos_ct), by = x -> NSP_muts_pos_dict[x[1]])
    global NSP5_pos_ct_sort_by_pos = sort(collect(NSP5_pos_ct), by = x -> NSP_muts_pos_dict[x[1]])
    global NSP6_pos_ct_sort_by_pos = sort(collect(NSP6_pos_ct), by = x -> NSP_muts_pos_dict[x[1]])
    global NSP7_pos_ct_sort_by_pos = sort(collect(NSP7_pos_ct), by = x -> NSP_muts_pos_dict[x[1]])
    global NSP8_pos_ct_sort_by_pos = sort(collect(NSP8_pos_ct), by = x -> NSP_muts_pos_dict[x[1]])
    global NSP9_pos_ct_sort_by_pos = sort(collect(NSP9_pos_ct), by = x -> NSP_muts_pos_dict[x[1]])
    global NSP10_pos_ct_sort_by_pos = sort(collect(NSP10_pos_ct), by = x -> NSP_muts_pos_dict[x[1]])
    global NSP11_pos_ct_sort_by_pos = sort(collect(NSP11_pos_ct), by = x -> NSP_muts_pos_dict[x[1]])
    global NSP12_pos_ct_sort_by_pos = sort(collect(NSP12_pos_ct), by = x -> NSP_muts_pos_dict[x[1]])
    global NSP13_pos_ct_sort_by_pos = sort(collect(NSP13_pos_ct), by = x -> NSP_muts_pos_dict[x[1]])
    global NSP14_pos_ct_sort_by_pos = sort(collect(NSP14_pos_ct), by = x -> NSP_muts_pos_dict[x[1]])
    global NSP15_pos_ct_sort_by_pos = sort(collect(NSP15_pos_ct), by = x -> NSP_muts_pos_dict[x[1]])
    global NSP16_pos_ct_sort_by_pos = sort(collect(NSP16_pos_ct), by = x -> NSP_muts_pos_dict[x[1]])
#########################
    global NSP1_pos_ct_sort_by_ct = sort(collect(NSP1_pos_ct), by = x -> x[2], rev=true)
    global NSP2_pos_ct_sort_by_ct = sort(collect(NSP2_pos_ct), by = x -> x[2], rev=true)
    global NSP3_pos_ct_sort_by_ct = sort(collect(NSP3_pos_ct), by = x -> x[2], rev=true)
    global NSP4_pos_ct_sort_by_ct = sort(collect(NSP4_pos_ct), by = x -> x[2], rev=true)
    global NSP5_pos_ct_sort_by_ct = sort(collect(NSP5_pos_ct), by = x -> x[2], rev=true)
    global NSP6_pos_ct_sort_by_ct = sort(collect(NSP6_pos_ct), by = x -> x[2], rev=true)
    global NSP7_pos_ct_sort_by_ct = sort(collect(NSP7_pos_ct), by = x -> x[2], rev=true)
    global NSP8_pos_ct_sort_by_ct = sort(collect(NSP8_pos_ct), by = x -> x[2], rev=true)
    global NSP9_pos_ct_sort_by_ct = sort(collect(NSP9_pos_ct), by = x -> x[2], rev=true)
    global NSP10_pos_ct_sort_by_ct = sort(collect(NSP10_pos_ct), by = x -> x[2], rev=true)
    global NSP11_pos_ct_sort_by_ct = sort(collect(NSP11_pos_ct), by = x -> x[2], rev=true)
    global NSP12_pos_ct_sort_by_ct = sort(collect(NSP12_pos_ct), by = x -> x[2], rev=true)
    global NSP13_pos_ct_sort_by_ct = sort(collect(NSP13_pos_ct), by = x -> x[2], rev=true)
    global NSP14_pos_ct_sort_by_ct = sort(collect(NSP14_pos_ct), by = x -> x[2], rev=true)
    global NSP15_pos_ct_sort_by_ct = sort(collect(NSP15_pos_ct), by = x -> x[2], rev=true)
    global NSP16_pos_ct_sort_by_ct = sort(collect(NSP16_pos_ct), by = x -> x[2], rev=true)
#########################
    global NSP1_pos_ct_sort_by_pos_v1 = sort(collect(NSP1_pos_ct_v1), by = x -> x[1])
    global NSP2_pos_ct_sort_by_pos_v1 = sort(collect(NSP2_pos_ct_v1), by = x -> x[1])
    global NSP3_pos_ct_sort_by_pos_v1 = sort(collect(NSP3_pos_ct_v1), by = x -> x[1])
    global NSP4_pos_ct_sort_by_pos_v1 = sort(collect(NSP4_pos_ct_v1), by = x -> x[1])
    global NSP5_pos_ct_sort_by_pos_v1 = sort(collect(NSP5_pos_ct_v1), by = x -> x[1])
    global NSP6_pos_ct_sort_by_pos_v1 = sort(collect(NSP6_pos_ct_v1), by = x -> x[1])
    global NSP7_pos_ct_sort_by_pos_v1 = sort(collect(NSP7_pos_ct_v1), by = x -> x[1])
    global NSP8_pos_ct_sort_by_pos_v1 = sort(collect(NSP8_pos_ct_v1), by = x -> x[1])
    global NSP9_pos_ct_sort_by_pos_v1 = sort(collect(NSP9_pos_ct_v1), by = x -> x[1])
    global NSP10_pos_ct_sort_by_pos_v1 = sort(collect(NSP10_pos_ct_v1), by = x -> x[1])
    global NSP11_pos_ct_sort_by_pos_v1 = sort(collect(NSP11_pos_ct_v1), by = x -> x[1])
    global NSP12_pos_ct_sort_by_pos_v1 = sort(collect(NSP12_pos_ct_v1), by = x -> x[1])
    global NSP13_pos_ct_sort_by_pos_v1 = sort(collect(NSP13_pos_ct_v1), by = x -> x[1])
    global NSP14_pos_ct_sort_by_pos_v1 = sort(collect(NSP14_pos_ct_v1), by = x -> x[1])
    global NSP15_pos_ct_sort_by_pos_v1 = sort(collect(NSP15_pos_ct_v1), by = x -> x[1])
    global NSP16_pos_ct_sort_by_pos_v1 = sort(collect(NSP16_pos_ct_v1), by = x -> x[1])
end
###########################################################################################################################################################################
###########################################################################################################################################################################
###########################################################################################################################################################################
###########################################################################################################################################################################
function gene_sub_ranks_all(gene_array::Vector{String}, sub_types_at_every_site_combined::Dict{String, Dict{Int, Set{String}}}, AA_muts_ct_no_dels_all::Dict{String, Int} )
    ORF1a_ct_all = Dict{String, Int}()
    ORF1b_ct_all = Dict{String, Int}()
    S_ct_all = Dict{String, Int}()
    ORF3a_ct_all = Dict{String, Int}()
    E_ct_all = Dict{String, Int}()
    M_ct_all = Dict{String, Int}()
    ORF6_ct_all = Dict{String, Int}()
    ORF7a_ct_all = Dict{String, Int}()
    ORF7b_ct_all = Dict{String, Int}()
    ORF8_ct_all = Dict{String, Int}()
    N_ct_all = Dict{String, Int}()
    ORF9b_ct_all = Dict{String, Int}()
#############################################################
    #                                        Gene      AAsite  all_sub_types_at_AAsite (e.g., AV, AT, TA, etc)
# sub_types_at_every_site_combined = Dict{String, Dict{Int, Set{String}}}()
    gene_ct_dict_all_arr = [ORF1a_ct_all, ORF1b_ct_all, S_ct_all, ORF3a_ct_all, E_ct_all, M_ct_all, ORF6_ct_all, ORF7a_ct_all, ORF7b_ct_all, ORF8_ct_all, N_ct_all, ORF9b_ct_all]
    for i in 1:length(gene_array)
        gene = gene_array[i]
        gene_ct_dict = gene_ct_dict_all_arr[i]
        for (AAsite, mut_type_set) in sub_types_at_every_site_combined[gene]
            for mut_type in mut_type_set
                sub = gene*":"*string(mut_type[1])*"$(AAsite)"*string(mut_type[end])
                gene_ct_dict[sub] = 0
            end
        end
    end
#############################################################
    for (mut, ct) in AA_muts_ct_no_dels_all
        if aa_gene_comprehensive_dict[mut] == "ORF1a"
            ORF1a_ct_all[mut] = ct
            elseif aa_gene_comprehensive_dict[mut] == "ORF1b"
            ORF1b_ct_all[mut] = ct
            elseif aa_gene_comprehensive_dict[mut] == "S"
            S_ct_all[mut] = ct
            elseif aa_gene_comprehensive_dict[mut] == "ORF3a"
            ORF3a_ct_all[mut] = ct
            elseif aa_gene_comprehensive_dict[mut] == "E"
            E_ct_all[mut] = ct
            elseif aa_gene_comprehensive_dict[mut] == "M"
            M_ct_all[mut] = ct
            elseif aa_gene_comprehensive_dict[mut] == "ORF6"
            ORF6_ct_all[mut] = ct
            elseif aa_gene_comprehensive_dict[mut] == "ORF7a"
            ORF7a_ct_all[mut] = ct
            elseif aa_gene_comprehensive_dict[mut] == "ORF7b"
            ORF7b_ct_all[mut] = ct
            elseif aa_gene_comprehensive_dict[mut] == "ORF8"
            ORF8_ct_all[mut] = ct
            elseif aa_gene_comprehensive_dict[mut] == "N"
            N_ct_all[mut] = ct
            elseif aa_gene_comprehensive_dict[mut] == "ORF9b"
            ORF9b_ct_all[mut] = ct
        end
    end
    fin_sortkey(m) = (aa_pos_comprehensive_dict[m], refAA_comprehensive_dict[m]*qryAA_comprehensive_dict[m])
    global ORF1a_ct_all_sort_by_pos = sort(collect(ORF1a_ct_all), by = x -> fin_sortkey(x[1]))
    global ORF1b_ct_all_sort_by_pos = sort(collect(ORF1b_ct_all), by = x -> fin_sortkey(x[1]))
    global S_ct_all_sort_by_pos = sort(collect(S_ct_all), by = x -> fin_sortkey(x[1]))
    global ORF3a_ct_all_sort_by_pos = sort(collect(ORF3a_ct_all), by = x -> fin_sortkey(x[1]))
    global E_ct_all_sort_by_pos = sort(collect(E_ct_all), by = x -> fin_sortkey(x[1]))
    global M_ct_all_sort_by_pos = sort(collect(M_ct_all), by = x -> fin_sortkey(x[1]))
    global ORF6_ct_all_sort_by_pos = sort(collect(ORF6_ct_all), by = x -> fin_sortkey(x[1]))
    global ORF7a_ct_all_sort_by_pos = sort(collect(ORF7a_ct_all), by = x -> fin_sortkey(x[1]))
    global ORF7b_ct_all_sort_by_pos = sort(collect(ORF7b_ct_all), by = x -> fin_sortkey(x[1]))
    global ORF8_ct_all_sort_by_pos = sort(collect(ORF8_ct_all), by = x -> fin_sortkey(x[1]))
    global N_ct_all_sort_by_pos = sort(collect(N_ct_all), by = x -> fin_sortkey(x[1]))
    global ORF9b_ct_all_sort_by_pos = sort(collect(ORF9b_ct_all), by = x -> fin_sortkey(x[1]))

    global ORF1b_ct_all_sort_by_ct = sort(collect(ORF1b_ct_all), by = x -> x[2], rev=true)
    global ORF1a_ct_all_sort_by_ct = sort(collect(ORF1a_ct_all), by = x -> x[2], rev=true)
    global S_ct_all_sort_by_ct = sort(collect(S_ct_all), by = x -> x[2], rev=true)
    global ORF3a_ct_all_sort_by_ct = sort(collect(ORF3a_ct_all), by = x -> x[2], rev=true)
    global E_ct_all_sort_by_ct = sort(collect(E_ct_all), by = x -> x[2], rev=true)
    global M_ct_all_sort_by_ct = sort(collect(M_ct_all), by = x -> x[2], rev=true)
    global ORF6_ct_all_sort_by_ct = sort(collect(ORF6_ct_all), by = x -> x[2], rev=true)
    global ORF7a_ct_all_sort_by_ct = sort(collect(ORF7a_ct_all), by = x -> x[2], rev=true)
    global ORF7b_ct_all_sort_by_ct = sort(collect(ORF7b_ct_all), by = x -> x[2], rev=true)
    global ORF8_ct_all_sort_by_ct = sort(collect(ORF8_ct_all), by = x -> x[2], rev=true)
    global N_ct_all_sort_by_ct = sort(collect(N_ct_all), by = x -> x[2], rev=true)
    global ORF9b_ct_all_sort_by_ct = sort(collect(ORF9b_ct_all), by = x -> x[2], rev=true)
end
###########################################################################################################################################################################
###########################################################################################################################################################################
function gene_sub_pos_ranks_all(gene_array::Vector{String}, gene_AA_lengths::Dict{String, Int},  AA_muts_ct_pos_only_no_dels_all::Dict{String, Int})
    ORF1a_pos_ct_all = Dict{String, Int}()
    ORF1b_pos_ct_all = Dict{String, Int}()
    S_pos_ct_all = Dict{String, Int}()
    ORF3a_pos_ct_all = Dict{String, Int}()
    E_pos_ct_all = Dict{String, Int}()
    M_pos_ct_all = Dict{String, Int}()
    ORF6_pos_ct_all = Dict{String, Int}()
    ORF7a_pos_ct_all = Dict{String, Int}()
    ORF7b_pos_ct_all = Dict{String, Int}()
    ORF8_pos_ct_all = Dict{String, Int}()
    N_pos_ct_all = Dict{String, Int}()
    ORF9b_pos_ct_all = Dict{String, Int}()
    gene_ct_pos_all_dict_arr = [ORF1a_pos_ct_all, ORF1b_pos_ct_all, S_pos_ct_all, ORF3a_pos_ct_all, E_pos_ct_all, M_pos_ct_all, ORF6_pos_ct_all, ORF7a_pos_ct_all, ORF7b_pos_ct_all, ORF8_pos_ct_all, N_pos_ct_all, ORF9b_pos_ct_all] 
###################################################### 
    ORF1a_pos_ct_all_v1 = Dict{Int, Int}()
    ORF1b_pos_ct_all_v1 = Dict{Int, Int}()
    S_pos_ct_all_v1 = Dict{Int, Int}()
    ORF3a_pos_ct_all_v1 = Dict{Int, Int}()
    E_pos_ct_all_v1 = Dict{Int, Int}()
    M_pos_ct_all_v1 = Dict{Int, Int}()
    ORF6_pos_ct_all_v1 = Dict{Int, Int}()
    ORF7a_pos_ct_all_v1 = Dict{Int, Int}()
    ORF7b_pos_ct_all_v1 = Dict{Int, Int}()
    ORF8_pos_ct_all_v1 = Dict{Int, Int}()
    N_pos_ct_all_v1 = Dict{Int, Int}()
    ORF9b_pos_ct_all_v1 = Dict{Int, Int}()
    gene_pos_ct_all_v1_dict_arr = [ORF1a_pos_ct_all_v1, ORF1b_pos_ct_all_v1, S_pos_ct_all_v1, ORF3a_pos_ct_all_v1, E_pos_ct_all_v1, M_pos_ct_all_v1, ORF6_pos_ct_all_v1, ORF7a_pos_ct_all_v1, ORF7b_pos_ct_all_v1, ORF8_pos_ct_all_v1, N_pos_ct_all_v1, ORF9b_pos_ct_all_v1]
###################################################### 
    for i in 1:length(gene_ct_pos_all_dict_arr)
        dict = gene_ct_pos_all_dict_arr[i]
        dict_v1 = gene_pos_ct_all_v1_dict_arr[i]
        gene = gene_array[i]
        gene_len = gene_AA_lengths[gene]
        for j in 1:gene_len
            site = gene*":"*"$(j)"
            dict[site] = 0
            dict_v1[j] = 0
        end
    end
#############################################################
    for (mut, ct) in AA_muts_ct_pos_only_no_dels_all
        pos = aa_pos_comprehensive_dict[mut]
        if aa_gene_comprehensive_dict[mut] == "ORF1a"
            ORF1a_pos_ct_all[mut] = ct
            ORF1a_pos_ct_all_v1[pos] = ct
            elseif aa_gene_comprehensive_dict[mut] == "ORF1b"
            ORF1b_pos_ct_all[mut] = ct
            ORF1b_pos_ct_all_v1[pos] = ct
            elseif aa_gene_comprehensive_dict[mut] == "S"
            S_pos_ct_all[mut] = ct
            S_pos_ct_all_v1[pos] = ct
            elseif aa_gene_comprehensive_dict[mut] == "ORF3a"
            ORF3a_pos_ct_all[mut] = ct
            ORF3a_pos_ct_all_v1[pos] = ct
            elseif aa_gene_comprehensive_dict[mut] == "E"
            E_pos_ct_all[mut] = ct
            E_pos_ct_all_v1[pos] = ct
            elseif aa_gene_comprehensive_dict[mut] == "M"
            M_pos_ct_all[mut] = ct
            M_pos_ct_all_v1[pos] = ct
            elseif aa_gene_comprehensive_dict[mut] == "ORF6"
            ORF6_pos_ct_all[mut] = ct
            ORF6_pos_ct_all_v1[pos] = ct
            elseif aa_gene_comprehensive_dict[mut] == "ORF7a"
            ORF7a_pos_ct_all[mut] = ct
            ORF7a_pos_ct_all_v1[pos] = ct
            elseif aa_gene_comprehensive_dict[mut] == "ORF7b"
            ORF7b_pos_ct_all[mut] = ct
            ORF7b_pos_ct_all_v1[pos] = ct
            elseif aa_gene_comprehensive_dict[mut] == "ORF8"
            ORF8_pos_ct_all[mut] = ct
            ORF8_pos_ct_all_v1[pos] = ct
            elseif aa_gene_comprehensive_dict[mut] == "N"
            N_pos_ct_all[mut] = ct
            N_pos_ct_all_v1[pos] = ct
            elseif aa_gene_comprehensive_dict[mut] == "ORF9b"
            ORF9b_pos_ct_all[mut] = ct
            ORF9b_pos_ct_all_v1[pos] = ct
        end
    end
    global ORF1a_pos_ct_all_sort_by_pos = sort(collect(ORF1a_pos_ct_all), by = x -> aa_pos_comprehensive_dict[x[1]])
    global ORF1b_pos_ct_all_sort_by_pos = sort(collect(ORF1b_pos_ct_all), by = x -> aa_pos_comprehensive_dict[x[1]])
    global S_pos_ct_all_sort_by_pos = sort(collect(S_pos_ct_all), by = x -> aa_pos_comprehensive_dict[x[1]])
    global ORF3a_pos_ct_all_sort_by_pos = sort(collect(ORF3a_pos_ct_all), by = x -> aa_pos_comprehensive_dict[x[1]])
    global E_pos_ct_all_sort_by_pos = sort(collect(E_pos_ct_all), by = x -> aa_pos_comprehensive_dict[x[1]]) 
    global M_pos_ct_all_sort_by_pos = sort(collect(M_pos_ct_all), by = x -> aa_pos_comprehensive_dict[x[1]])
    global ORF6_pos_ct_all_sort_by_pos = sort(collect(ORF6_pos_ct_all), by = x -> aa_pos_comprehensive_dict[x[1]])
    global ORF7a_pos_ct_all_sort_by_pos = sort(collect(ORF7a_pos_ct_all), by = x -> aa_pos_comprehensive_dict[x[1]])
    global ORF7b_pos_ct_all_sort_by_pos = sort(collect(ORF7b_pos_ct_all), by = x -> aa_pos_comprehensive_dict[x[1]])
    global ORF8_pos_ct_all_sort_by_pos = sort(collect(ORF8_pos_ct_all), by = x -> aa_pos_comprehensive_dict[x[1]])
    global N_pos_ct_all_sort_by_pos = sort(collect(N_pos_ct_all), by = x -> aa_pos_comprehensive_dict[x[1]])   
    global ORF9b_pos_ct_all_sort_by_pos = sort(collect(ORF9b_pos_ct_all), by = x -> aa_pos_comprehensive_dict[x[1]])
################################## 
    global ORF1a_pos_ct_all_sort_by_ct = sort(collect(ORF1a_pos_ct_all), by = x -> x[2], rev=true)
    global ORF1b_pos_ct_all_sort_by_ct = sort(collect(ORF1b_pos_ct_all), by = x -> x[2], rev=true)
    global S_pos_ct_all_sort_by_ct = sort(collect(S_pos_ct_all), by = x -> x[2], rev=true)
    global ORF3a_pos_ct_all_sort_by_ct = sort(collect(ORF3a_pos_ct_all), by = x -> x[2], rev=true)
    global E_pos_ct_all_sort_by_ct = sort(collect(E_pos_ct_all), by = x -> x[2], rev=true)
    global M_pos_ct_all_sort_by_ct = sort(collect(M_pos_ct_all), by = x -> x[2], rev=true)
    global ORF6_pos_ct_all_sort_by_ct = sort(collect(ORF6_pos_ct_all), by = x -> x[2], rev=true)
    global ORF7a_pos_ct_all_sort_by_ct = sort(collect(ORF7a_pos_ct_all), by = x -> x[2], rev=true)
    global ORF7b_pos_ct_all_sort_by_ct = sort(collect(ORF7b_pos_ct_all), by = x -> x[2], rev=true)
    global ORF8_pos_ct_all_sort_by_ct = sort(collect(ORF8_pos_ct_all), by = x -> x[2], rev=true)
    global N_pos_ct_all_sort_by_ct = sort(collect(N_pos_ct_all), by = x -> x[2], rev=true)
    global ORF9b_pos_ct_all_sort_by_ct = sort(collect(ORF9b_pos_ct_all), by = x -> x[2], rev=true)
##################################
    global ORF1a_pos_ct_all_sort_by_pos_v1 = sort(collect(ORF1a_pos_ct_all_v1), by = x -> x[1])
    global ORF1b_pos_ct_all_sort_by_pos_v1 = sort(collect(ORF1b_pos_ct_all_v1), by = x -> x[1])
    global S_pos_ct_all_sort_by_pos_v1 = sort(collect(S_pos_ct_all_v1), by = x -> x[1])
    global ORF3a_pos_ct_all_sort_by_pos_v1 = sort(collect(ORF3a_pos_ct_all_v1), by = x -> x[1])
    global E_pos_ct_all_sort_by_pos_v1 = sort(collect(E_pos_ct_all_v1), by = x -> x[1]) 
    global M_pos_ct_all_sort_by_pos_v1 = sort(collect(M_pos_ct_all_v1), by = x -> x[1])
    global ORF6_pos_ct_all_sort_by_pos_v1 = sort(collect(ORF6_pos_ct_all_v1), by = x -> x[1])
    global ORF7a_pos_ct_all_sort_by_pos_v1 = sort(collect(ORF7a_pos_ct_all_v1), by = x -> x[1])
    global ORF7b_pos_ct_all_sort_by_pos_v1 = sort(collect(ORF7b_pos_ct_all_v1), by = x -> x[1])
    global ORF8_pos_ct_all_sort_by_pos_v1 = sort(collect(ORF8_pos_ct_all_v1), by = x -> x[1])
    global N_pos_ct_all_sort_by_pos_v1 = sort(collect(N_pos_ct_all_v1), by = x -> x[1])   
    global ORF9b_pos_ct_all_sort_by_pos_v1 = sort(collect(ORF9b_pos_ct_all_v1), by = x -> x[1])
################################## 
    global ORF1a_pos_ct_all_sort_by_ct_v1 = sort(collect(ORF1a_pos_ct_all_v1), by = x -> x[2], rev=true)
    global ORF1b_pos_ct_all_sort_by_ct_v1 = sort(collect(ORF1b_pos_ct_all_v1), by = x -> x[2], rev=true)
    global S_pos_ct_all_sort_by_ct_v1 = sort(collect(S_pos_ct_all_v1), by = x -> x[2], rev=true)
    global ORF3a_pos_ct_all_sort_by_ct_v1 = sort(collect(ORF3a_pos_ct_all_v1), by = x -> x[2], rev=true)
    global E_pos_ct_all_sort_by_ct_v1 = sort(collect(E_pos_ct_all_v1), by = x -> x[2], rev=true)
    global M_pos_ct_all_sort_by_ct_v1 = sort(collect(M_pos_ct_all_v1), by = x -> x[2], rev=true)
    global ORF6_pos_ct_all_sort_by_ct_v1 = sort(collect(ORF6_pos_ct_all_v1), by = x -> x[2], rev=true)
    global ORF7a_pos_ct_all_sort_by_ct_v1 = sort(collect(ORF7a_pos_ct_all_v1), by = x -> x[2], rev=true)
    global ORF7b_pos_ct_all_sort_by_ct_v1 = sort(collect(ORF7b_pos_ct_all_v1), by = x -> x[2], rev=true)
    global ORF8_pos_ct_all_sort_by_ct_v1 = sort(collect(ORF8_pos_ct_all_v1), by = x -> x[2], rev=true)
    global N_pos_ct_all_sort_by_ct_v1 = sort(collect(N_pos_ct_all_v1), by = x -> x[2], rev=true)
    global ORF9b_pos_ct_all_sort_by_ct_v1 = sort(collect(ORF9b_pos_ct_all_v1), by = x -> x[2], rev=true)
end
###########################################################################################################################################################################
###########################################################################################################################################################################
function NSP_sub_ranks_all(NSP_array::Vector{String}, NSP_sub_types_at_every_site_combined::Dict{String, Dict{Int, Set{String}}}, AA_muts_ct_no_dels_all::Dict{String, Int} )
    NSP1_ct_all = Dict{String, Int}()
    NSP2_ct_all = Dict{String, Int}()
    NSP3_ct_all = Dict{String, Int}()
    NSP4_ct_all = Dict{String, Int}()
    NSP5_ct_all = Dict{String, Int}()
    NSP6_ct_all = Dict{String, Int}()
    NSP7_ct_all = Dict{String, Int}()
    NSP8_ct_all = Dict{String, Int}()
    NSP9_ct_all = Dict{String, Int}()
    NSP10_ct_all = Dict{String, Int}()
    NSP11_ct_all = Dict{String, Int}()
    NSP12_ct_all = Dict{String, Int}()
    NSP13_ct_all = Dict{String, Int}()
    NSP14_ct_all = Dict{String, Int}()
    NSP15_ct_all = Dict{String, Int}()
    NSP16_ct_all = Dict{String, Int}()
#############################################################
    NSP_ct_all_dict_arr = [NSP1_ct_all, NSP2_ct_all, NSP3_ct_all, NSP4_ct_all, NSP5_ct_all, NSP6_ct_all, NSP7_ct_all, NSP8_ct_all, NSP9_ct_all, NSP10_ct_all, NSP11_ct_all, NSP12_ct_all, NSP13_ct_all, NSP14_ct_all, NSP15_ct_all, NSP16_ct_all]
#                                              NSP       AAsite  all_sub_types_at_AAsite (e.g., AV, AT, TA, etc)
# NSPsub_types_at_every_site_combined = Dict{String, Dict{Int, Set{String}}}()   
    for i in 1:length(NSP_array)
        NSP = NSP_array[i]
        NSP_ct_dict = NSP_ct_all_dict_arr[i]
        for (AAsite, mut_type_set) in NSP_sub_types_at_every_site_combined[NSP]
            for mut_type in mut_type_set
                sub = NSP*":"*string(mut_type[1])*"$(AAsite)"*string(mut_type[end])
                NSP_ct_dict[sub] = 0
            end
        end
    end   
    for (sub, ct) in AA_muts_ct_no_dels_all
        AA_site = aa_pos_comprehensive_dict[sub]
        if AA_site < 4402
            gene = aa_gene_comprehensive_dict[sub]
            if gene == "ORF1a" || gene == "ORF1b"
                NSP_sub = ORF1abMut_to_NSP(sub)
                NSP = NSP_muts_gene_dict[NSP_sub]
                NSP_pos = NSP_muts_pos_dict[NSP_sub]
                NSP_num = parse(Int, split(NSP, "P")[2])
                NSP_dict = NSP_ct_all_dict_arr[NSP_num]
                NSP_dict[NSP_sub] = ct
            end
        end
    end
#############################
    fin_sortkey(m) = (NSP_muts_pos_dict[m], NSP_ref_AA_dict[m]*NSP_qry_AA_dict[m])
    global NSP1_ct_all_sort_by_pos = sort(collect(NSP1_ct_all), by = x -> fin_sortkey(x[1]))
    global NSP2_ct_all_sort_by_pos = sort(collect(NSP2_ct_all), by = x -> fin_sortkey(x[1]))
    global NSP3_ct_all_sort_by_pos = sort(collect(NSP3_ct_all), by = x -> fin_sortkey(x[1]))
    global NSP4_ct_all_sort_by_pos = sort(collect(NSP4_ct_all), by = x -> fin_sortkey(x[1]))
    global NSP5_ct_all_sort_by_pos = sort(collect(NSP5_ct_all), by = x -> fin_sortkey(x[1]))
    global NSP6_ct_all_sort_by_pos = sort(collect(NSP6_ct_all), by = x -> fin_sortkey(x[1]))
    global NSP7_ct_all_sort_by_pos = sort(collect(NSP7_ct_all), by = x -> fin_sortkey(x[1]))
    global NSP8_ct_all_sort_by_pos = sort(collect(NSP8_ct_all), by = x -> fin_sortkey(x[1]))
    global NSP9_ct_all_sort_by_pos = sort(collect(NSP9_ct_all), by = x -> fin_sortkey(x[1]))
    global NSP10_ct_all_sort_by_pos = sort(collect(NSP10_ct_all), by = x -> fin_sortkey(x[1]))
    global NSP11_ct_all_sort_by_pos = sort(collect(NSP11_ct_all), by = x -> fin_sortkey(x[1]))
    global NSP12_ct_all_sort_by_pos = sort(collect(NSP12_ct_all), by = x -> fin_sortkey(x[1]))
    global NSP13_ct_all_sort_by_pos = sort(collect(NSP13_ct_all), by = x -> fin_sortkey(x[1]))
    global NSP14_ct_all_sort_by_pos = sort(collect(NSP14_ct_all), by = x -> fin_sortkey(x[1]))
    global NSP15_ct_all_sort_by_pos = sort(collect(NSP15_ct_all), by = x -> fin_sortkey(x[1]))
    global NSP16_ct_all_sort_by_pos = sort(collect(NSP16_ct_all), by = x -> fin_sortkey(x[1]))
#############################
    global NSP1_ct_all_sort_by_ct = sort(collect(NSP1_ct_all), by = x -> x[2], rev=true)
    global NSP2_ct_all_sort_by_ct = sort(collect(NSP2_ct_all), by = x -> x[2], rev=true)
    global NSP3_ct_all_sort_by_ct = sort(collect(NSP3_ct_all), by = x -> x[2], rev=true)
    global NSP4_ct_all_sort_by_ct = sort(collect(NSP4_ct_all), by = x -> x[2], rev=true)
    global NSP5_ct_all_sort_by_ct = sort(collect(NSP5_ct_all), by = x -> x[2], rev=true)
    global NSP6_ct_all_sort_by_ct = sort(collect(NSP6_ct_all), by = x -> x[2], rev=true)
    global NSP7_ct_all_sort_by_ct = sort(collect(NSP7_ct_all), by = x -> x[2], rev=true)
    global NSP8_ct_all_sort_by_ct = sort(collect(NSP8_ct_all), by = x -> x[2], rev=true)
    global NSP9_ct_all_sort_by_ct = sort(collect(NSP9_ct_all), by = x -> x[2], rev=true)
    global NSP10_ct_all_sort_by_ct = sort(collect(NSP10_ct_all), by = x -> x[2], rev=true)
    global NSP11_ct_all_sort_by_ct = sort(collect(NSP11_ct_all), by = x -> x[2], rev=true)
    global NSP12_ct_all_sort_by_ct = sort(collect(NSP12_ct_all), by = x -> x[2], rev=true)
    global NSP13_ct_all_sort_by_ct = sort(collect(NSP13_ct_all), by = x -> x[2], rev=true)
    global NSP14_ct_all_sort_by_ct = sort(collect(NSP14_ct_all), by = x -> x[2], rev=true)
    global NSP15_ct_all_sort_by_ct = sort(collect(NSP15_ct_all), by = x -> x[2], rev=true)
    global NSP16_ct_all_sort_by_ct = sort(collect(NSP16_ct_all), by = x -> x[2], rev=true)
end
###########################################################################################################################################################################
###########################################################################################################################################################################
# Removed from parameters: NSP_AA_size::Dict{String, Int}
function NSP_sub_pos_ranks_all(NSP_array::Vector{String}, NSP_sub_types_at_every_site_combined::Dict{String, Dict{Int, Set{String}}}, AA_muts_ct_no_dels_all::Dict{String, Int} )
    NSP1_pos_ct_all = Dict{String, Int}()
    NSP2_pos_ct_all = Dict{String, Int}()
    NSP3_pos_ct_all = Dict{String, Int}()
    NSP4_pos_ct_all = Dict{String, Int}()
    NSP5_pos_ct_all = Dict{String, Int}()
    NSP6_pos_ct_all = Dict{String, Int}()
    NSP7_pos_ct_all = Dict{String, Int}()
    NSP8_pos_ct_all = Dict{String, Int}()
    NSP9_pos_ct_all = Dict{String, Int}()
    NSP10_pos_ct_all = Dict{String, Int}()
    NSP11_pos_ct_all = Dict{String, Int}()
    NSP12_pos_ct_all = Dict{String, Int}()
    NSP13_pos_ct_all = Dict{String, Int}()
    NSP14_pos_ct_all = Dict{String, Int}()
    NSP15_pos_ct_all = Dict{String, Int}()
    NSP16_pos_ct_all = Dict{String, Int}()
    NSP_pos_ct_all_array = [NSP1_pos_ct_all, NSP2_pos_ct_all, NSP3_pos_ct_all, NSP4_pos_ct_all, NSP5_pos_ct_all, NSP6_pos_ct_all, NSP7_pos_ct_all, NSP8_pos_ct_all, NSP9_pos_ct_all, NSP10_pos_ct_all, NSP11_pos_ct_all, NSP12_pos_ct_all, NSP13_pos_ct_all, NSP14_pos_ct_all, NSP15_pos_ct_all, NSP16_pos_ct_all]
#######################################################
    NSP1_pos_ct_all_v1 = Dict{Int, Int}()
    NSP2_pos_ct_all_v1 = Dict{Int, Int}()
    NSP3_pos_ct_all_v1 = Dict{Int, Int}()
    NSP4_pos_ct_all_v1 = Dict{Int, Int}()
    NSP5_pos_ct_all_v1 = Dict{Int, Int}()
    NSP6_pos_ct_all_v1 = Dict{Int, Int}()
    NSP7_pos_ct_all_v1 = Dict{Int, Int}()
    NSP8_pos_ct_all_v1 = Dict{Int, Int}()
    NSP9_pos_ct_all_v1 = Dict{Int, Int}()
    NSP10_pos_ct_all_v1 = Dict{Int, Int}()
    NSP11_pos_ct_all_v1 = Dict{Int, Int}()
    NSP12_pos_ct_all_v1 = Dict{Int, Int}()
    NSP13_pos_ct_all_v1 = Dict{Int, Int}()
    NSP14_pos_ct_all_v1 = Dict{Int, Int}()
    NSP15_pos_ct_all_v1 = Dict{Int, Int}()
    NSP16_pos_ct_all_v1 = Dict{Int, Int}()
    NSP_pos_ct_all_v1_array = [NSP1_pos_ct_all_v1, NSP2_pos_ct_all_v1, NSP3_pos_ct_all_v1, NSP4_pos_ct_all_v1, NSP5_pos_ct_all_v1, NSP6_pos_ct_all_v1, NSP7_pos_ct_all_v1, NSP8_pos_ct_all_v1, NSP9_pos_ct_all_v1, NSP10_pos_ct_all_v1, NSP11_pos_ct_all_v1, NSP12_pos_ct_all_v1, NSP13_pos_ct_all_v1, NSP14_pos_ct_all_v1, NSP15_pos_ct_all_v1, NSP16_pos_ct_all_v1]
#######################################################
    NSP_pos_ct_all = Dict{String, Int}()
    for i in 1:length(NSP_pos_ct_all_array)
        NSP_all_dict = NSP_pos_ct_all_array[i]
        NSP_all_dict_v1 = NSP_pos_ct_all_v1_array[i]
        NSP = NSP_array[i]
        NSP_len = NSP_AA_size[NSP]
        for j in 1:NSP_len
            NSP_pos = NSP*":"*"$(j)"
            NSP_all_dict[NSP_pos] = 0
            NSP_pos_ct_all[NSP_pos] = 0
            NSP_all_dict_v1[j] = 0
        end
    end
############################################################
    NSP1_ct_all = Dict{String, Int}()
    NSP2_ct_all = Dict{String, Int}()
    NSP3_ct_all = Dict{String, Int}()
    NSP4_ct_all = Dict{String, Int}()
    NSP5_ct_all = Dict{String, Int}()
    NSP6_ct_all = Dict{String, Int}()
    NSP7_ct_all = Dict{String, Int}()
    NSP8_ct_all = Dict{String, Int}()
    NSP9_ct_all = Dict{String, Int}()
    NSP10_ct_all = Dict{String, Int}()
    NSP11_ct_all = Dict{String, Int}()
    NSP12_ct_all = Dict{String, Int}()
    NSP13_ct_all = Dict{String, Int}()
    NSP14_ct_all = Dict{String, Int}()
    NSP15_ct_all = Dict{String, Int}()
    NSP16_ct_all = Dict{String, Int}()
##################################################################################################################################
    NSP_ct_all_dict_arr = [NSP1_ct_all, NSP2_ct_all, NSP3_ct_all, NSP4_ct_all, NSP5_ct_all, NSP6_ct_all, NSP7_ct_all, NSP8_ct_all, NSP9_ct_all, NSP10_ct_all, NSP11_ct_all, NSP12_ct_all, NSP13_ct_all, NSP14_ct_all, NSP15_ct_all, NSP16_ct_all]
#                                              NSP       AAsite  all_sub_types_at_AAsite (e.g., AV, AT, TA, etc)
# NSPsub_types_at_every_site_combined = Dict{String, Dict{Int, Set{String}}}()   
    for i in 1:length(NSP_array)
        NSP = NSP_array[i]
        NSP_ct_dict = NSP_ct_all_dict_arr[i]
        for (AAsite, mut_type_set) in NSP_sub_types_at_every_site_combined[NSP]
            for mut_type in mut_type_set
                sub = NSP*":"*string(mut_type[1])*"$(AAsite)"*string(mut_type[end])
                NSP_ct_dict[sub] = 0
            end
        end
    end   
    for (sub, ct) in AA_muts_ct_no_dels_all
        gene = aa_gene_comprehensive_dict[sub]
        if gene == "ORF1a" || gene == "ORF1b"
            AA_site = parse(Int, split(sub, ":")[2][2:end-1])
            if AA_site < 4402
                NSP_sub = ORF1abMut_to_NSP(sub)
                NSP = NSP_muts_gene_dict[NSP_sub]
                NSP_pos = NSP_muts_pos_dict[NSP_sub]
                NSP_num = parse(Int, split(NSP, "P")[2])
                NSP_dict = NSP_ct_all_dict_arr[NSP_num]
                NSP_dict[NSP_sub] = ct
            end
        end
    end
###################################################################################################################################
    for i in 1:length(NSP_pos_ct_all_array)
        NSP = NSP_array[i]
        NSP_dict = NSP_ct_all_dict_arr[i]
        NSP_pos_dict = NSP_pos_ct_all_array[i]
        NSP_pos_dict_v1 = NSP_pos_ct_all_v1_array[i]
        for (sub, ct) in NSP_dict
            pos = parse(Int, split(sub, ":")[2][2:end-1])
            sub_pos = NSP*":"*"$(pos)"
            NSP_pos_dict[sub_pos] += ct
            NSP_pos_dict_v1[pos] += ct
            NSP_pos_ct_all[sub_pos] += ct
        end
    end
##############################################################
    global NSP1_pos_ct_all_sort_by_pos = sort(collect(NSP1_pos_ct_all), by = x -> NSP_muts_pos_dict[x[1]])
    global NSP2_pos_ct_all_sort_by_pos = sort(collect(NSP2_pos_ct_all), by = x -> NSP_muts_pos_dict[x[1]])
    global NSP3_pos_ct_all_sort_by_pos = sort(collect(NSP3_pos_ct_all), by = x -> NSP_muts_pos_dict[x[1]])
    global NSP4_pos_ct_all_sort_by_pos = sort(collect(NSP4_pos_ct_all), by = x -> NSP_muts_pos_dict[x[1]])
    global NSP5_pos_ct_all_sort_by_pos = sort(collect(NSP5_pos_ct_all), by = x -> NSP_muts_pos_dict[x[1]])
    global NSP6_pos_ct_all_sort_by_pos = sort(collect(NSP6_pos_ct_all), by = x -> NSP_muts_pos_dict[x[1]])
    global NSP7_pos_ct_all_sort_by_pos = sort(collect(NSP7_pos_ct_all), by = x -> NSP_muts_pos_dict[x[1]])
    global NSP8_pos_ct_all_sort_by_pos = sort(collect(NSP8_pos_ct_all), by = x -> NSP_muts_pos_dict[x[1]])
    global NSP9_pos_ct_all_sort_by_pos = sort(collect(NSP9_pos_ct_all), by = x -> NSP_muts_pos_dict[x[1]])
    global NSP10_pos_ct_all_sort_by_pos = sort(collect(NSP10_pos_ct_all), by = x -> NSP_muts_pos_dict[x[1]])
    global NSP11_pos_ct_all_sort_by_pos = sort(collect(NSP11_pos_ct_all), by = x -> NSP_muts_pos_dict[x[1]])
    global NSP12_pos_ct_all_sort_by_pos = sort(collect(NSP12_pos_ct_all), by = x -> NSP_muts_pos_dict[x[1]])
    global NSP13_pos_ct_all_sort_by_pos = sort(collect(NSP13_pos_ct_all), by = x -> NSP_muts_pos_dict[x[1]])
    global NSP14_pos_ct_all_sort_by_pos = sort(collect(NSP14_pos_ct_all), by = x -> NSP_muts_pos_dict[x[1]])
    global NSP15_pos_ct_all_sort_by_pos = sort(collect(NSP15_pos_ct_all), by = x -> NSP_muts_pos_dict[x[1]])
    global NSP16_pos_ct_all_sort_by_pos = sort(collect(NSP16_pos_ct_all), by = x -> NSP_muts_pos_dict[x[1]])
#############################
    global NSP1_pos_ct_all_sort_by_ct = sort(collect(NSP1_pos_ct_all), by = x -> x[2], rev=true)
    global NSP2_pos_ct_all_sort_by_ct = sort(collect(NSP2_pos_ct_all), by = x -> x[2], rev=true)
    global NSP3_pos_ct_all_sort_by_ct = sort(collect(NSP3_pos_ct_all), by = x -> x[2], rev=true)
    global NSP4_pos_ct_all_sort_by_ct = sort(collect(NSP4_pos_ct_all), by = x -> x[2], rev=true)
    global NSP5_pos_ct_all_sort_by_ct = sort(collect(NSP5_pos_ct_all), by = x -> x[2], rev=true)
    global NSP6_pos_ct_all_sort_by_ct = sort(collect(NSP6_pos_ct_all), by = x -> x[2], rev=true)
    global NSP7_pos_ct_all_sort_by_ct = sort(collect(NSP7_pos_ct_all), by = x -> x[2], rev=true)
    global NSP8_pos_ct_all_sort_by_ct = sort(collect(NSP8_pos_ct_all), by = x -> x[2], rev=true)
    global NSP9_pos_ct_all_sort_by_ct = sort(collect(NSP9_pos_ct_all), by = x -> x[2], rev=true)
    global NSP10_pos_ct_all_sort_by_ct = sort(collect(NSP10_pos_ct_all), by = x -> x[2], rev=true)
    global NSP11_pos_ct_all_sort_by_ct = sort(collect(NSP11_pos_ct_all), by = x -> x[2], rev=true)
    global NSP12_pos_ct_all_sort_by_ct = sort(collect(NSP12_pos_ct_all), by = x -> x[2], rev=true)
    global NSP13_pos_ct_all_sort_by_ct = sort(collect(NSP13_pos_ct_all), by = x -> x[2], rev=true)
    global NSP14_pos_ct_all_sort_by_ct = sort(collect(NSP14_pos_ct_all), by = x -> x[2], rev=true)
    global NSP15_pos_ct_all_sort_by_ct = sort(collect(NSP15_pos_ct_all), by = x -> x[2], rev=true)
    global NSP16_pos_ct_all_sort_by_ct = sort(collect(NSP16_pos_ct_all), by = x -> x[2], rev=true)
###############################
    global NSP1_pos_ct_all_sort_by_pos_v1 = sort(collect(NSP1_pos_ct_all_v1), by = x -> x[1])
    global NSP2_pos_ct_all_sort_by_pos_v1 = sort(collect(NSP2_pos_ct_all_v1), by = x -> x[1])
    global NSP3_pos_ct_all_sort_by_pos_v1 = sort(collect(NSP3_pos_ct_all_v1), by = x -> x[1])
    global NSP4_pos_ct_all_sort_by_pos_v1 = sort(collect(NSP4_pos_ct_all_v1), by = x -> x[1])
    global NSP5_pos_ct_all_sort_by_pos_v1 = sort(collect(NSP5_pos_ct_all_v1), by = x -> x[1])
    global NSP6_pos_ct_all_sort_by_pos_v1 = sort(collect(NSP6_pos_ct_all_v1), by = x -> x[1])
    global NSP7_pos_ct_all_sort_by_pos_v1 = sort(collect(NSP7_pos_ct_all_v1), by = x -> x[1])
    global NSP8_pos_ct_all_sort_by_pos_v1 = sort(collect(NSP8_pos_ct_all_v1), by = x -> x[1])
    global NSP9_pos_ct_all_sort_by_pos_v1 = sort(collect(NSP9_pos_ct_all_v1), by = x -> x[1])
    global NSP10_pos_ct_all_sort_by_pos_v1 = sort(collect(NSP10_pos_ct_all_v1), by = x -> x[1])
    global NSP11_pos_ct_all_sort_by_pos_v1 = sort(collect(NSP11_pos_ct_all_v1), by = x -> x[1])
    global NSP12_pos_ct_all_sort_by_pos_v1 = sort(collect(NSP12_pos_ct_all_v1), by = x -> x[1])
    global NSP13_pos_ct_all_sort_by_pos_v1 = sort(collect(NSP13_pos_ct_all_v1), by = x -> x[1])
    global NSP14_pos_ct_all_sort_by_pos_v1 = sort(collect(NSP14_pos_ct_all_v1), by = x -> x[1])
    global NSP15_pos_ct_all_sort_by_pos_v1 = sort(collect(NSP15_pos_ct_all_v1), by = x -> x[1])
    global NSP16_pos_ct_all_sort_by_pos_v1 = sort(collect(NSP16_pos_ct_all_v1), by = x -> x[1])
end
######################################################################################################################################
runtime = time() - start_chr_load_fx
runtime_rd = round(digits=2, runtime)
runtime1, runtime2 = seconds_to_hrs_min_sec(runtime); print("\n"^1)
println("Runtime v0 = $(runtime) seconds")
println("Runtime v2 = $(runtime2)"); print("\n"^1)
######################################################################################################################################
######################################################################################################################################

2026_03_08__409PM

Runtime v0 = 0.9259028434753418 seconds
Runtime v2 = 0 hr, 0 min, 00.93 sec



In [88]:
### Execute Load Chronic Dicts DQ Fx | 2026_02_22 version | chronics_2026_02_01__6153seq | 95—8—6 | Runtime: 1 min, 00.26 sec | 2026_2_13
date_now = Dates.format(now(), "yyyy_mm_dd__IMMp"); println(date_now)
chr_dict_load_start = time()

abs_min_mut_thresh = 8
DQ_mut_thresh = 10
date_pct_DQ_thresh = 95

rep_thresh = 5
revs_thresh = 6
HQCS_qc_str = HQCS_qc_string
EPCI_qc_str = "$(abs_min_mut_thresh)_$(DQ_mut_thresh)_$(date_pct_DQ_thresh)"

print_ct_thresh = 1
#date = "2026_02_21"
date = "2026_02_20"
ndjson_name = "chronics_2026_02_01__6153seq_v2"
folder_name = "$(date)__$(ndjson_name)__EPCI_$(EPCI_qc_str)__HQCS_$(HQCS_qc_string)"

println("EPCI_qc_str = $(EPCI_qc_str) | HQCS_qc_str = $(HQCS_qc_str)")

chronic_load_dicts2_DQ(ndjson_name, folder_name, date, rep_thresh, revs_thresh, print_ct_thresh, DQ_mut_thresh, date_pct_DQ_thresh, abs_min_mut_thresh)
# ndjson_name::String, folder_name::String, date::String, rep_thresh::Int, revs_thresh::Int, print_ct_thresh::Int, DQ_mut_thresh::Int, DatePctDQThresh::Float64, abs_min_mut_thresh::Int
######################################################################################################################################
seq_AA_muts["EPI_ISL_8725398"] = Set{String}();             seq_AA_muts["EPI_ISL_949208"] = Set{String}();   seq_pango["EPI_ISL_6281381"] = "AY.4"
######################################################################################################################################
### Create  all_unique_chr_seqs  &  all_unique_chr_seqs_set 
rep_seq_grps_maxmut_seqs_arr = collect(values(rep_seq_grps_maxmut_seqs))
all_unique_chr_seqs = union(rep_seq_grps_maxmut_seqs_arr, non_rep_seqs)
all_unique_chr_seqs_ct = length(all_unique_chr_seqs)
all_unique_chr_seqs_set = Set{String}()
for seq in all_unique_chr_seqs
    push!(all_unique_chr_seqs_set, seq)
end
println("length(all_unique_chr_seqs) = $(length(all_unique_chr_seqs))")
println("length(all_unique_chr_seqs_set)  $(length(all_unique_chr_seqs_set))"); print("\n"^2)
########################################################################################################################################
for seq in all_unique_chr_seqs_set 
    for del in seq_AA_dels[seq]
        aa_gene_comprehensive_dict[del] = string(split(del, ":")[1])
        firstdel = string(split(del, "-")[1])
        aa_pos_comprehensive_dict[del] = aa_pos_comprehensive_dict[firstdel]
    end
end
#############################################################################################################################################################################
#### Removing clearly artifactual reversions that occur directly next to dropout (and are almost always from labs that frequently have this problem)
FCS_fake_revs = Set(["S:K679N", "S:H681P", "S:R681P"])
FCS_fake_revs_pos = Set(["S:679", "S:681"])
for seq in all_qualifying_seqs_set
    if "S:683" in seq_unknown_AA[seq] || "S:676" in seq_unknown_AA[seq]
        if seq in all_unique_chr_seqs_set
            for mut in FCS_fake_revs
                if mut in seq_AA_muts[seq]
                    mut_pos = aa_gene_and_pos_comprehensive_dict[mut]
                    AA_muts_ct[mut] -= 1
                    AA_muts_ct_no_dels[mut] -= 1
                    AA_muts_ct_pos_only[mut_pos] -= 1
                    AA_muts_ct_pos_only_no_dels[mut_pos] -= 1
                end
            end
        end
        setdiff!(seq_AA_muts[seq], FCS_fake_revs)
        setdiff!(seq_AA_muts_no_dels[seq], FCS_fake_revs)
        setdiff!(seq_AA_muts_pos_only[seq], FCS_fake_revs_pos)
        setdiff!(seq_AA_muts_pos_only_no_dels[seq], FCS_fake_revs_pos)
    end
end
######################################################################################################################################
if haskey(AA_muts_ct_pos_only_no_dels, "")
    delete!(AA_muts_ct_pos_only_no_dels, "")
end
if haskey(AA_muts_ct_no_dels, "")
    delete!(AA_muts_ct_no_dels, "")
end
######################################################################################################################################
## Deletions royally screw up any attempt to find correlated mutations since they frequently occur in bunches, which are, of course, highly correlated, but only in the most trivial way. 
## The code below is a preliminary attempt to include common deletions by only allowing the inclusion of a single AA deletion, though most contain multiple consecutive 
## AA deletions. Also included are mutations that only occur as 1-AA deletions, such as E:I13- and E:V14-. It can be removed or inserted without substantially changing the results 
deletion_exceptions = list_to_strings_set("E:I13-, E:V14-, M:G6-, S:C15-, S:C136-, S:D138-, S:A243-, S:F371-, S:F375-, S:V483-, S:A484-, S:E484-, S:D1257-, ORF3a:T14-, ORF3a:V255-, ORF3a:V256-, ORF3a:N257-, ORF7a:I103-, ORF7a:*122-, ORF1a:N2081-, ORF1a:D2136-, ORF1a:S4398-")
pos_only_deletion_exception_ct_dict = Dict{String, Int}("E:I13-"=>0, "E:V14-"=>0, "M:G6-"=>0, "S:C15-"=>0, "S:C136-"=>0, "S:D138-"=>0, "S:A243-"=>0, "S:F371-"=>0, "S:F375-"=>0, "S:V483-"=>0, "S:A484-"=>0, "S:E484-"=>0, "S:D1257-"=>0, "ORF3a:T14-"=>0, "ORF3a:V255-"=>0, "ORF3a:V256-"=>0, "ORF3a:N257-"=>0, "ORF7a:I103-"=>0, "ORF7a:*122-"=>0, "ORF1a:N2081-"=>0, "ORF1a:D2136-"=>0, "ORF1a:S4398-"=>0)    
del_to_del_pos = Dict{String, String}("E:I13-"=>"E:13", "E:V14-"=>"E:14", "M:G6-"=>"M:6", "S:C15-"=>"S:15", "S:C136-"=>"S:136", "S:D138-"=>"S:138", "S:A243-"=>"S:243", "S:F371-"=>"S:371", "S:F375-"=>"S:375", "S:V483-"=>"S:483", "S:A484-"=>"S:484", "S:E484-"=>"S:484", "S:D1257-"=>"S:1257", "ORF3a:T14-"=>"ORF3a:14", "ORF3a:V255-"=>"ORF3a:255", "ORF3a:V256-"=>"ORF3a:256", "ORF3a:N257-"=>"ORF3a:257", "ORF7a:I103-"=>"ORF7a:103", "ORF7a:*122-"=>"ORF7a:122", "ORF1a:N2081-"=>"ORF1a:2081", "ORF1a:D2136-"=>"ORF1a:2136", "ORF1a:S4398-"=>"ORF1a:4398")
pos_only_exception_count::Int = 0
for seq in all_seqs_set
    for del in deletion_exceptions
        if del in seq_AA_muts[seq]
            del_pos = del_to_del_pos[del]
            push!(seq_AA_muts_pos_only_no_dels[seq], del_pos)
            pos_only_exception_count += 1
            pos_only_deletion_exception_ct_dict[del] += 1
        end
    end
end
######################################################################################################################################
for del in deletion_exceptions
    del_pos = del_to_del_pos[del]
    if !haskey(AA_muts_ct_pos_only_no_dels, del_pos)
        AA_muts_ct_pos_only_no_dels[del_pos] = 0
    end
    AA_muts_ct_pos_only_no_dels[del_pos] += pos_only_deletion_exception_ct_dict[del]
    AA_muts_ct_pos_only_no_dels_chr_all_ratio[del_pos] = 999
end
######################################################################################################################################
deletion_exception_ct_dict = Dict{String, Int}("E:I13-"=>0, "E:V14-"=>0, "M:G6-"=>0, "S:C15-"=>0, "S:C136-"=>0, "S:D138-"=>0, "S:A243-"=>0, "S:F371-"=>0, "S:F375-"=>0, "S:V483-"=>0, "S:A484-"=>0, "S:E484-"=>0, "S:D1257-"=>0, "ORF3a:T14-"=>0, "ORF3a:V255-"=>0, "ORF3a:V256-"=>0, "ORF3a:N257-"=>0, "ORF7a:I103-"=>0, "ORF7a:*122-"=>0, "ORF1a:N2081-"=>0, "ORF1a:D2136-"=>0, "ORF1a:S4398-"=>0)    
exception_count::Int = 0
for seq in all_unique_chr_seqs_set
    for del in deletion_exceptions
        if del in seq_AA_muts[seq]
            push!(seq_AA_muts_no_dels[seq], del)
            exception_count += 1
            deletion_exception_ct_dict[del] += 1
        end
    end
end
######################################################################################################################################
for del in deletion_exceptions
    AA_muts_ct_no_dels[del] = get(AA_muts_ct_no_dels, del, 0)
    AA_muts_ct_no_dels[del] += deletion_exception_ct_dict[del]
    AA_muts_ct_chr_all_ratio[del] = 999
end
######################################################################################################################################
blank_mutstrings = Set(["", " ", "  ", "   ", "    ", "     ", "      ", "       ", "        ", "         ", "          ", "           ", "            "])
for blnk in blank_mutstrings
    AA_muts_ct_chr_all_ratio[blnk] = 0
    AA_muts_ct_no_dels_chr_all_ratio[blnk] = 0
    AA_muts_ct_pos_only_no_dels_chr_all_ratio[blnk] = 0
end
#######################################################################################################################################
mut_gene_Dict = Dict{String, Int}("ORF1a"=>1, "ORF1b"=>2, "S"=>3, "E"=>4, "M"=>5, "N"=>6, "ORF3a"=>7, "ORF6"=>8, "ORF7a"=>9, "ORF7b"=>10, "ORF8"=>11, "ORF9b"=>12)
######################################################################################################################################
mp_AA_gene_sortKey(n) = (mut_gene_Dict[aa_gene_comprehensive_dict[n[1]]], aa_pos_comprehensive_dict[n[1]])
mp_AA_gene_sortKey_2(n) = (mut_gene_Dict[aa_gene_comprehensive_dict[n]], aa_pos_comprehensive_dict[n])
mp_AA_ct_sortKey1(n) = (1000÷mut_gene_Dict[aa_gene_comprehensive_dict[n[1]]], aa_pos_comprehensive_dict[n[1]])
mp_AA_ct_sortKey2(n) = (n[2], mp_AA_ct_sortKey1(n))
#########
mp_AA_gene_pos_only_sortKey(n) = (mut_gene_Dict[aa_gene_comprehensive_dict[n[1]]], aa_pos_comprehensive_dict[n[1]])
mp_AA_gene_pos_only_sortKey_2(n) = (mut_gene_Dict[aa_gene_comprehensive_dict[n]], aa_pos_comprehensive_dict[n])
mp_AA_ct_pos_only_sortKey1(n) = (1000÷mut_gene_Dict[aa_gene_comprehensive_dict[n[1]]], aa_pos_comprehensive_dict[n[1]])
mp_AA_ct_pos_only_sortKey2(n) = (n[2], mp_AA_ct_pos_only_sortKey1(n))
#################################
function mp_AApos_sort_key(n)
    if haskey(aa_pos_comprehensive_dict, n)
        return (aa_pos_comprehensive_dict[n], n)
    else
        return (9999, n)
    end
end
function sel_muts_pt1_sort_key(n)
    if n == "" || isempty(n)
        return (0, 0)  # Return a default sort key for empty strings
    else
        return mp_AA_gene_sortKey_2(string(split(n, ", ")[1]))
    end
end
##################################################################
println("Number of Deletion Exceptions Made = $(exception_count)")
deletion_exception_ct_dict_sort = sort(collect(deletion_exception_ct_dict), by = x -> x[2])
for del__ct in deletion_exception_ct_dict_sort
    del = del__ct[1]
    ct = del__ct[2]
    println("$(del) = $(ct)")
end
###########################################################################################
seq_privAA_len = Dict{String, Int}()
for (seq, AAsubvec) in seq_AA_muts_no_dels
    seqAAlen = length(AAsubvec)
    seq_privAA_len[seq] = seqAAlen
end
############################################################################################################################################################################
############################################################################################################################################################################
for (seq, date) in seq_collection_date
    date_arr = string.(collect(date))
    if sum(date_arr .== "-") ≠ 2
        println("Doesn't have two dashes in date = $(seq)")
    else
        year = string(split(date, "-")[1])
        month = string(split(date, "-")[2])
        day = string(split(date, "-")[3])
        if length(month) == 1 && month ≠ "0"
            month = add_leading_zero(month)
        end
        if length(day) == 1 && day ≠ "0"
            day = add_leading_zero(day)
        end
        seq_collection_date[seq] = year*"-"*month*"-"*day
    end
end 
############################################################################################################################################################################
############################################################################################################################################################################
corrected_count = 0
noncorrected_ct = 0
for seq in all_seqs_set
    if seq_date_index[seq] > 4000
#        println("seq_date_tuple = $(seq_date_tuple[seq]); seq_date_tuple[seq][1] = $(seq_date_tuple[seq][1]); seq_date_tuple[seq][2] = $(seq_date_tuple[seq][2]); seq_date_tuple[seq][3] = $(seq_date_tuple[seq][3])")
        if seq_date_tuple[seq][1] ≠ 0 && seq_date_tuple[seq][2] ≠ 0
            new_date_tuple = (seq_date_tuple[seq][1], seq_date_tuple[seq][2], 15)
            seq_date_tuple[seq] = new_date_tuple
            seq_date_index[seq] = tuple_to_index[new_date_tuple]
            corrected_count += 1
        elseif seq_date_tuple[seq][2] == 0
            noncorrected_ct += 1
        end
    end
end
total_baddie_ct = corrected_count + noncorrected_ct
println("seq_date_index corrected = $(corrected_count)")
println("seq_date_index not corrected = $(noncorrected_ct)")
println("total_baddie_ct = $(total_baddie_ct)")
############################################################################################################################################################################
############################################################################################################################################################################
chr_load_runtime = time() - chr_dict_load_start
chr_load_runtime_rd = round(digits=1, chr_load_runtime)
chr_load_hms1, chr_load_hms2 = seconds_to_hrs_min_sec(chr_load_runtime)
println("Total Time to Load Chronic Dictionaries = $(chr_load_runtime_rd)")
println("Total Time to Load Chronic Dictionaries = $(chr_load_hms1)")
println("Total Time to Load Chronic Dictionaries = $(chr_load_hms2)"); print("\n"^1)
println("Finished!")
date_now = Dates.format(now(), "yyyy_mm_dd__IMMp"); println(date_now); print("\n"^1)
######################################################################################################################################
######################################################################################################################################

2026_03_08__409PM
EPCI_qc_str = 8_10_95 | HQCS_qc_str = 5_1_5
2026_03_08__409PM
length(all_unique_chr_seqs) = 3297
length(all_unique_chr_seqs_set)  3297


Number of Deletion Exceptions Made = 838
S:E484- = 1
ORF1a:D2136- = 2
S:F371- = 4
ORF1a:N2081- = 6
ORF1a:S4398- = 7
E:I13- = 8
S:D1257- = 8
S:V483- = 9
ORF7a:I103- = 10
M:G6- = 12
ORF7a:*122- = 16
S:A484- = 23
S:F375- = 24
S:C136- = 32
ORF3a:V256- = 38
ORF3a:T14- = 40
ORF3a:N257- = 43
ORF3a:V255- = 51
S:D138- = 89
E:V14- = 90
S:C15- = 105
S:A243- = 220
seq_date_index corrected = 70
seq_date_index not corrected = 52
total_baddie_ct = 122
Total Time to Load Chronic Dictionaries = 15.6
Total Time to Load Chronic Dictionaries = 0:00:15.56
Total Time to Load Chronic Dictionaries = 0 hr, 0 min, 15.56 sec

Finished!
2026_03_08__409PM



In [76]:
### NEW | 2026_02_22 | Many, many functions (mixed_nuc, nuc_to_AA, etc) | Runtime: 1 min 33 sec 
### Timing Template
nowtime = Dates.format(now(), "I:MM.SS_p"); println(nowtime)
start = time()
######################################################################################################################################
AA_triplets = Dict{String, String}("TTT"=>"F", "TTC"=>"F", "TTA"=>"L", "TTG"=>"L", "TCT"=>"S", "TCC"=>"S", "TCA"=>"S", "TCG"=>"S", "TAT"=>"Y", "TAC"=>"Y", "TAA"=>"*", "TAG"=>"*", "TGT"=>"C", "TGC"=>"C", "TGA"=>"*", "TGG"=>"W", "CTT"=>"L", "CTC"=>"L", "CTA"=>"L", "CTG"=>"L", "CCT"=>"P", "CCC"=>"P", "CCA"=>"P", "CCG"=>"P", "CAT"=>"H", "CAC"=>"H", "CAA"=>"Q", "CAG"=>"Q", "CGT"=>"R", "CGC"=>"R", "CGA"=>"R", "CGG"=>"R", "ATT"=>"I", "ATC"=>"I", "ATA"=>"I", "ATG"=>"M", "ACT"=>"T", "ACC"=>"T", "ACA"=>"T", "ACG"=>"T", "AAT"=>"N", "AAC"=>"N", "AAA"=>"K", "AAG"=>"K", "AGT"=>"S", "AGC"=>"S", "AGA"=>"R", "AGG"=>"R", "GTT"=>"V", "GTC"=>"V", "GTA"=>"V", "GTG"=>"V", "GCT"=>"A", "GCC"=>"A", "GCA"=>"A", "GCG"=>"A", "GAT"=>"D", "GAC"=>"D", "GAA"=>"E", "GAG"=>"E", "GGT"=>"G", "GGC"=>"G", "GGA"=>"G", "GGG"=>"G", "TT-"=>"X", "TC-"=>"X", "TA-"=>"X", "TG-"=>"X", "T-T"=>"X", "T-C"=>"X", "T-A"=>"X", "T-G"=>"X", "T--"=>"X", "CT-"=>"X", "CC-"=>"X", "CA-"=>"X", "CG-"=>"X", "C-T"=>"X", "C-C"=>"X", "C-A"=>"X", "C-G"=>"X", "C--"=>"X", "AT-"=>"X", "AC-"=>"X", "AA-"=>"X", "AG-"=>"X", "A-T"=>"X", "A-C"=>"X", "A-A"=>"X", "A-G"=>"X", "A--"=>"X", "GT-"=>"X", "GC-"=>"X", "GA-"=>"X", "GG-"=>"X", "G-T"=>"X", "G-C"=>"X", "G-A"=>"X", "G-G"=>"X", "G--"=>"X", "-TT"=>"X", "-TC"=>"X", "-TA"=>"X", "-TG"=>"X", "-T-"=>"X", "-CT"=>"X", "-CC"=>"X", "-CA"=>"X", "-CG"=>"X", "-C-"=>"X", "-AT"=>"X", "-AC"=>"X", "-AA"=>"X", "-AG"=>"X", "-A-"=>"X", "-GT"=>"X", "-GC"=>"X", "-GA"=>"X", "-GG"=>"X", "-G-"=>"X", "--T"=>"X", "--C"=>"X", "--A"=>"X", "--G"=>"X", "---"=>"X", "NTT"=>"X", "TNT"=>"X", "TTN"=>"X", "NTC"=>"X", "TNC"=>"X", "TCN"=>"X", "NTA"=>"X", "TNA"=>"X", "TAN"=>"X", "NTG"=>"X", "TNG"=>"X", "TGN"=>"X", "NT-"=>"X", "TN-"=>"X", "T-N"=>"X", "NTN"=>"X", "TNN"=>"X", "NCT"=>"X", "CNT"=>"X", "CTN"=>"X", "NCC"=>"X", "CNC"=>"X", "CCN"=>"X", "NCA"=>"X", "CNA"=>"X", "CAN"=>"X", "NCG"=>"X", "CNG"=>"X", "CGN"=>"X", "NC-"=>"X", "CN-"=>"X", "C-N"=>"X", "NCN"=>"X", "CNN"=>"X", "NAT"=>"X", "ANT"=>"X", "ATN"=>"X", "NAC"=>"X", "ANC"=>"X", "ACN"=>"X", "NAA"=>"X", "ANA"=>"X", "AAN"=>"X", "NAG"=>"X", "ANG"=>"X", "AGN"=>"X", "NA-"=>"X", "AN-"=>"X", "A-N"=>"X", "NAN"=>"X", "ANN"=>"X", "NGT"=>"X", "GNT"=>"X", "GTN"=>"X", "NGC"=>"X", "GNC"=>"X", "GCN"=>"X", "NGA"=>"X", "GNA"=>"X", "GAN"=>"X", "NGG"=>"X", "GNG"=>"X", "GGN"=>"X", "NG-"=>"X", "GN-"=>"X", "G-N"=>"X", "NGN"=>"X", "GNN"=>"X", "N-T"=>"X", "-NT"=>"X", "-TN"=>"X", "N-C"=>"X", "-NC"=>"X", "-CN"=>"X", "N-A"=>"X", "-NA"=>"X", "-AN"=>"X", "N-G"=>"X", "-NG"=>"X", "-GN"=>"X", "N--"=>"X", "-N-"=>"X", "--N"=>"X", "N-N"=>"X", "-NN"=>"X", "NNT"=>"X", "NNC"=>"X", "NNA"=>"X", "NNG"=>"X", "NN-"=>"X", "NNN"=>"X")                 
AA_triplet_dels = Dict{String, String}("TTT"=>"F", "TTC"=>"F", "TTA"=>"L", "TTG"=>"L", "TCT"=>"S", "TCC"=>"S", "TCA"=>"S", "TCG"=>"S", "TAT"=>"Y", "TAC"=>"Y", "TAA"=>"*", "TAG"=>"*", "TGT"=>"C", "TGC"=>"C", "TGA"=>"*", "TGG"=>"W", "CTT"=>"L", "CTC"=>"L", "CTA"=>"L", "CTG"=>"L", "CCT"=>"P", "CCC"=>"P", "CCA"=>"P", "CCG"=>"P", "CAT"=>"H", "CAC"=>"H", "CAA"=>"Q", "CAG"=>"Q", "CGT"=>"R", "CGC"=>"R", "CGA"=>"R", "CGG"=>"R", "ATT"=>"I", "ATC"=>"I", "ATA"=>"I", "ATG"=>"M", "ACT"=>"T", "ACC"=>"T", "ACA"=>"T", "ACG"=>"T", "AAT"=>"N", "AAC"=>"N", "AAA"=>"K", "AAG"=>"K", "AGT"=>"S", "AGC"=>"S", "AGA"=>"R", "AGG"=>"R", "GTT"=>"V", "GTC"=>"V", "GTA"=>"V", "GTG"=>"V", "GCT"=>"A", "GCC"=>"A", "GCA"=>"A", "GCG"=>"A", "GAT"=>"D", "GAC"=>"D", "GAA"=>"E", "GAG"=>"E", "GGT"=>"G", "GGC"=>"G", "GGA"=>"G", "GGG"=>"G", "TT-"=>"X", "TC-"=>"X", "TA-"=>"X", "TG-"=>"X", "T-T"=>"X", "T-C"=>"X", "T-A"=>"X", "T-G"=>"X", "T--"=>"-", "CT-"=>"X", "CC-"=>"X", "CA-"=>"X", "CG-"=>"X", "C-T"=>"X", "C-C"=>"X", "C-A"=>"X", "C-G"=>"X", "C--"=>"-", "AT-"=>"X", "AC-"=>"X", "AA-"=>"X", "AG-"=>"X", "A-T"=>"X", "A-C"=>"X", "A-A"=>"X", "A-G"=>"X", "A--"=>"-", "GT-"=>"X", "GC-"=>"X", "GA-"=>"X", "GG-"=>"X", "G-T"=>"X", "G-C"=>"X", "G-A"=>"X", "G-G"=>"X", "G--"=>"-", "-TT"=>"X", "-TC"=>"X", "-TA"=>"X", "-TG"=>"X", "-T-"=>"X", "-CT"=>"X", "-CC"=>"X", "-CA"=>"X", "-CG"=>"X", "-C-"=>"X", "-AT"=>"X", "-AC"=>"X", "-AA"=>"X", "-AG"=>"X", "-A-"=>"X", "-GT"=>"X", "-GC"=>"X", "-GA"=>"X", "-GG"=>"X", "-G-"=>"X", "--T"=>"-", "--C"=>"-", "--A"=>"-", "--G"=>"-", "---"=>"-", "NTT"=>"X", "TNT"=>"X", "TTN"=>"X", "NTC"=>"X", "TNC"=>"X", "TCN"=>"X", "NTA"=>"X", "TNA"=>"X", "TAN"=>"X", "NTG"=>"X", "TNG"=>"X", "TGN"=>"X", "NT-"=>"X", "TN-"=>"X", "T-N"=>"X", "NTN"=>"X", "TNN"=>"X", "NCT"=>"X", "CNT"=>"X", "CTN"=>"X", "NCC"=>"X", "CNC"=>"X", "CCN"=>"X", "NCA"=>"X", "CNA"=>"X", "CAN"=>"X", "NCG"=>"X", "CNG"=>"X", "CGN"=>"X", "NC-"=>"X", "CN-"=>"X", "C-N"=>"X", "NCN"=>"X", "CNN"=>"X", "NAT"=>"X", "ANT"=>"X", "ATN"=>"X", "NAC"=>"X", "ANC"=>"X", "ACN"=>"X", "NAA"=>"X", "ANA"=>"X", "AAN"=>"X", "NAG"=>"X", "ANG"=>"X", "AGN"=>"X", "NA-"=>"X", "AN-"=>"X", "A-N"=>"X", "NAN"=>"X", "ANN"=>"X", "NGT"=>"X", "GNT"=>"X", "GTN"=>"X", "NGC"=>"X", "GNC"=>"X", "GCN"=>"X", "NGA"=>"X", "GNA"=>"X", "GAN"=>"X", "NGG"=>"X", "GNG"=>"X", "GGN"=>"X", "NG-"=>"X", "GN-"=>"X", "G-N"=>"X", "NGN"=>"X", "GNN"=>"X", "N-T"=>"X", "-NT"=>"X", "-TN"=>"X", "N-C"=>"X", "-NC"=>"X", "-CN"=>"X", "N-A"=>"X", "-NA"=>"X", "-AN"=>"X", "N-G"=>"X", "-NG"=>"X", "-GN"=>"X", "N--"=>"X", "-N-"=>"X", "--N"=>"X", "N-N"=>"X", "-NN"=>"X", "NNT"=>"X", "NNC"=>"X", "NNA"=>"X", "NNG"=>"X", "NN-"=>"X", "NNN"=>"X")                 
######################################################################################################################################
function list_to_strings_no_spaces(list::String)
    string_vec = string.(split(list, ","))
    return string_vec
end
######################################################################################################################################
function stringlist_to_strings_nonEPI(txt::String)
    arr_of_strings = Vector{String}()
    no_newlines = replace(txt, "\n" =>" ")
    for seq in split(no_newlines, ", ")
        push!(arr_of_strings, seq)
    end
    sort_arr_of_strings = sort(collect(arr_of_strings), by = x -> nuc_mut_int_comprehensive_dict[x])  
    return sort_arr_of_strings
end
######################################################################################################################################
function list_to_string_array(txt::String) # similar to stringlist_to_strings but not for EPIs
    no_newlines = replace(txt, "\n" =>" ")
    string_array = string.(split(no_newlines, ", "))
    return string_array
end
###########################################################################################################################################################################
###########################################################################################################################################################################
function get_ref_pango_nucseq_and_geneseqs(ref_pango::String)
    ref_seq = "..."
    refAA_ORF1a = "MESLVPGFNEKTHVQLSLPVLQVRDVLVRGFGDSVEEVLSEARQHLKDGTCGLVEVEKGVLPQLEQPYVFIKRSDARTAPHGHVMVELVAELEGIQYGRSGETLGVLVPHVGEIPVAYRKVLLRKNGNKGAGGHSYGADLKSFDLGDELGTDPYEDFQENWNTKHSSGVTRELMRELNGGAYTRYVDNNFCGPDGYPLECIKDLLARAGKASCTLSEQLDFIDTKRGVYCCREHEHEIAWYTERSEKSYELQTPFEIKLAKKFDTFNGECPNFVFPLNSIIKTIQPRVEKKKLDGFMGRIRSVYPVASPNECNQMCLSTLMKCDHCGETSWQTGDFVKATCEFCGTENLTKEGATTCGYLPQNAVVKIYCPACHNSEVGPEHSLAEYHNESGLKTILRKGGRTIAFGGCVFSYVGCHNKCAYWVPRASANIGCNHTGVVGEGSEGLNDNLLEILQKEKVNINIVGDFKLNEEIAIILASFSASTSAFVETVKGLDYKAFKQIVESCGNFKVTKGKAKKGAWNIGEQKSILSPLYAFASEAARVVRSIFSRTLETAQNSVRVLQKAAITILDGISQYSLRLIDAMMFTSDLATNNLVVMAYITGGVVQLTSQWLTNIFGTVYEKLKPVLDWLEEKFKEGVEFLRDGWEIVKFISTCACEIVGGQIVTCAKEIKESVQTFFKLVNKFLALCADSIIIGGAKLKALNLGETFVTHSKGLYRKCVKSREETGLLMPLKAPKEIIFLEGETLPTEVLTEEVVLKTGDLQPLEQPTSEAVEAPLVGTPVCINGLMLLEIKDTEKYCALAPNMMVTNNTFTLKGGAPTKVTFGDDTVIEVQGYKSVNITFELDERIDKVLNEKCSAYTVELGTEVNEFACVVADAVIKTLQPVSELLTPLGIDLDEWSMATYYLFDESGEFKLASHMYCSFYPPDEDEEEGDCEEEEFEPSTQYEYGTEDDYQGKPLEFGATSAALQPEEEQEEDWLDDDSQQTVGQQDGSEDNQTTTIQTIVEVQPQLEMELTPVVQTIEVNSFSGYLKLTDNVYIKNADIVEEAKKVKPTVVVNAANVYLKHGGGVAGALNKATNNAMQVESDDYIATNGPLKVGGSCVLSGHNLAKHCLHVVGPNVNKGEDIQLLKSAYENFNQHEVLLAPLLSAGIFGADPIHSLRVCVDTVRTNVYLAVFDKNLYDKLVSSFLEMKSEKQVEQKIAEIPKEEVKPFITESKPSVEQRKQDDKKIKACVEEVTTTLEETKFLTENLLLYIDINGNLHPDSATLVSDIDITFLKKDAPYIVGDVVQEGVLTAVVIPTKKAGGTTEMLAKALRKVPTDNYITTYPGQGLNGYTVEEAKTVLKKCKSAFYILPSIISNEKQEILGTVSWNLREMLAHAEETRKLMPVCVETKAIVSTIQRKYKGIKIQEGVVDYGARFYFYTSKTTVASLINTLNDLNETLVTMPLGYVTHGLNLEEAARYMRSLKVPATVSVSSPDAVTAYNGYLTSSSKTPEEHFIETISLAGSYKDWSYSGQSTQLGIEFLKRGDKSVYYTSNPTTFHLDGEVITFDNLKTLLSLREVRTIKVFTTVDNINLHTQVVDMSMTYGQQFGPTYLDGADVTKIKPHNSHEGKTFYVLPNDDTLRVEAFEYYHTTDPSFLGRYMSALNHTKKWKYPQVNGLTSIKWADNNCYLATALLTLQQIELKFNPPALQDAYYRARAGEAANFCALILAYCNKTVGELGDVRETMSYLFQHANLDSCKRVLNVVCKTCGQQQTTLKGVEAVMYMGTLSYEQFKKGVQIPCTCGKQATKYLVQQESPFVMMSAPPAQYELKHGTFTCASEYTGNYQCGHYKHITSKETLYCIDGALLTKSSEYKGPITDVFYKENSYTTTIKPVTYKLDGVVCTEIDPKLDNYYKKDNSYFTEQPIDLVPNQPYPNASFDNFKFVCDNIKFADDLNQLTGYKKPASRELKVTFFPDLNGDVVAIDYKHYTPSFKKGAKLLHKPIVWHVNNATNKATYKPNTWCIRCLWSTKPVETSNSFDVLKSEDAQGMDNLACEDLKPVSEEVVENPTIQKDVLECNVKTTEVVGDIILKPANNSLKITEEVGHTDLMAAYVDNSSLTIKKPNELSRVLGLKTLATHGLAAVNSVPWDTIANYAKPFLNKVVSTTTNIVTRCLNRVCTNYMPYFFTLLLQLCTFTRSTNSRIKASMPTTIAKNTVKSVGKFCLEASFNYLKSPNFSKLINIIIWFLLLSVCLGSLIYSTAALGVLMSNLGMPSYCTGYREGYLNSTNVTIATYCTGSIPCSVCLSGLDSLDTYPSLETIQITISSFKWDLTAFGLVAEWFLAYILFTRFFYVLGLAAIMQLFFSYFAVHFISNSWLMWLIINLVQMAPISAMVRMYIFFASFYYVWKSYVHVVDGCNSSTCMMCYKRNRATRVECTTIVNGVRRSFYVYANGGKGFCKLHNWNCVNCDTFCAGSTFISDEVARDLSLQFKRPINPTDQSSYIVDSVTVKNGSIHLYFDKAGQKTYERHSLSHFVNLDNLRANNTKGSLPINVIVFDGKSKCEESSAKSASVYYSQLMCQPILLLDQALVSDVGDSAEVAVKMFDAYVNTFSSTFNVPMEKLKTLVATAEAELAKNVSLDNVLSTFISAARQGFVDSDVETKDVVECLKLSHQSDIEVTGDSCNNYMLTYNKVENMTPRDLGACIDCSARHINAQVAKSHNIALIWNVKDFMSLSEQLRKQIRSAAKKNNLPFKLTCATTRQVVNVVTTKIALKGGKIVNNWLKQLIKVTLVFLFVAAIFYLITPVHVMSKHTDFSSEIIGYKAIDGGVTRDIASTDTCFANKHADFDTWFSQRGGSYTNDKACPLIAAVITREVGFVVPGLPGTILRTTNGDFLHFLPRVFSAVGNICYTPSKLIEYTDFATSACVLAAECTIFKDASGKPVPYCYDTNVLEGSVAYESLRPDTRYVLMDGSIIQFPNTYLEGSVRVVTTFDSEYCRHGTCERSEAGVCVSTSGRWVLNNDYYRSLPGVFCGVDAVNLLTNMFTPLIQPIGALDISASIVAGGIVAIVVTCLAYYFMRFRRAFGEYSHVVAFNTLLFLMSFTVLCLTPVYSFLPGVYSVIYLYLTFYLTNDVSFLAHIQWMVMFTPLVPFWITIAYIICISTKHFYWFFSNYLKRRVVFNGVSFSTFEEAALCTFLLNKEMYLKLRSDVLLPLTQYNRYLALYNKYKYFSGAMDTTSYREAACCHLAKALNDFSNSGSDVLYQPPQTSITSAVLQSGFRKMAFPSGKVEGCMVQVTCGTTTLNGLWLDDVVYCPRHVICTSEDMLNPNYEDLLIRKSNHNFLVQAGNVQLRVIGHSMQNCVLKLKVDTANPKTPKYKFVRIQPGQTFSVLACYNGSPSGVYQCAMRPNFTIKGSFLNGSCGSVGFNIDYDCVSFCYMHHMELPTGVHAGTDLEGNFYGPFVDRQTAQAAGTDTTITVNVLAWLYAAVINGDRWFLNRFTTTLNDFNLVAMKYNYEPLTQDHVDILGPLSAQTGIAVLDMCASLKELLQNGMNGRTILGSALLEDEFTPFDVVRQCSGVTFQSAVKRTIKGTHHWLLLTILTSLLVLVQSTQWSLFFFLYENAFLPFAMGIIAMSAFAMMFVKHKHAFLCLFLLPSLATVAYFNMVYMPASWVMRIMTWLDMVDTSLSGFKLKDCVMYASAVVLLILMTARTVYDDGARRVWTLMNVLTLVYKVYYGNALDQAISMWALIISVTSNYSGVVTTVMFLARGIVFMCVEYCPIFFITGNTLQCIMLVYCFLGYFCTCYFGLFCLLNRYFRLTLGVYDYLVSTQEFRYMNSQGLLPPKNSIDAFKLNIKLLGVGGKPCIKVATVQSKMSDVKCTSVVLLSVLQQLRVESSSKLWAQCVQLHNDILLAKDTTEAFEKMVSLLSVLLSMQGAVDINKLCEEMLDNRATLQAIASEFSSLPSYAAFATAQEAYEQAVANGDSEVVLKKLKKSLNVAKSEFDRDAAMQRKLEKMADQAMTQMYKQARSEDKRAKVTSAMQTMLFTMLRKLDNDALNNIINNARDGCVPLNIIPLTTAAKLMVVIPDYNTYKNTCDGTTFTYASALWEIQQVVDADSKIVQLSEISMDNSPNLAWPLIVTALRANSAVKLQNNELSPVALRQMSCAAGTTQTACTDDNALAYYNTTKGGRFVLALLSDLQDLKWARFPKSDGTGTIYTELEPPCRFVTDTPKGPKVKYLYFIKGLNNLNRGMVLGSLAATVRLQAGNATEVPANSTVLSFCAFAVDAAKAYKDYLASGGQPITNCVKMLCTHTGTGQAITVTPEANMDQESFGGASCCLYCRCHIDHPNPKGFCDLKGKYVQIPTTCANDPVGFTLKNTVCTVCGMWKGYGCSCDQLREPMLQSADAQSFLNGFAV"
    refAA_ORF1b = "NRVCGVSAARLTPCGTGTSTDVVYRAFDIYNDKVAGFAKFLKTNCCRFQEKDEDDNLIDSYFVVKRHTFSNYQHEETIYNLLKDCPAVAKHDFFKFRIDGDMVPHISRQRLTKYTMADLVYALRHFDEGNCDTLKEILVTYNCCDDDYFNKKDWYDFVENPDILRVYANLGERVRQALLKTVQFCDAMRNAGIVGVLTLDNQDLNGNWYDFGDFIQTTPGSGVPVVDSYYSLLMPILTLTRALTAESHVDTDLTKPYIKWDLLKYDFTEERLKLFDRYFKYWDQTYHPNCVNCLDDRCILHCANFNVLFSTVFPPTSFGPLVRKIFVDGVPFVVSTGYHFRELGVVHNQDVNLHSSRLSFKELLVYAADPAMHAASGNLLLDKRTTCFSVAALTNNVAFQTVKPGNFNKDFYDFAVSKGFFKEGSSVELKHFFFAQDGNAAISDYDYYRYNLPTMCDIRQLLFVVEVVDKYFDCYDGGCINANQVIVNNLDKSAGFPFNKWGKARLYYDSMSYEDQDALFAYTKRNVIPTITQMNLKYAISAKNRARTVAGVSICSTMTNRQFHQKLLKSIAATRGATVVIGTSKFYGGWHNMLKTVYSDVENPHLMGWDYPKCDRAMPNMLRIMASLVLARKHTTCCSLSHRFYRLANECAQVLSEMVMCGGSLYVKPGGTSSGDATTAYANSVFNICQAVTANVNALLSTDGNKIADKYVRNLQHRLYECLYRNRDVDTDFVNEFYAYLRKHFSMMILSDDAVVCFNSTYASQGLVASIKNFKSVLYYQNNVFMSEAKCWTETDLTKGPHEFCSQHTMLVKQGDDYVYLPYPDPSRILGAGCFVDDIVKTDGTLMIERFVSLAIDAYPLTKHPNQEYADVFHLYLQYIRKLHDELTGHMLDMYSVMLTNDNTSRYWEPEFYEAMYTPHTVLQAVGACVLCNSQTSLRCGACIRRPFLCCKCCYDHVISTSHKLVLSVNPYVCNAPGCDVTDVTQLYLGGMSYYCKSHKPPISFPLCANGQVFGLYKNTCVGSDNVTDFNAIATCDWTNAGDYILANTCTERLKLFAAETLKATEETFKLSYGIATVREVLSDRELHLSWEVGKPRPPLNRNYVFTGYRVTKNSKVQIGEYTFEKGDYGDAVVYRGTTTYKLNVGDYFVLTSHTVMPLSAPTLVPQEHYVRITGLYPTLNISDEFSSNVANYQKVGMQKYSTLQGPPGTGKSHFAIGLALYYPSARIVYTACSHAAVDALCEKALKYLPIDKCSRIIPARARVECFDKFKVNSTLEQYVFCTVNALPETTADIVVFDEISMATNYDLSVVNARLRAKHYVYIGDPAQLPAPRTLLTKGTLEPEYFNSVCRLMKTIGPDMFLGTCRRCPAEIVDTVSALVYDNKLKAHKDKSAQCFKMFYKGVITHDVSSAINRPQIGVVREFLTRNPAWRKAVFISPYNSQNAVASKILGLPTQTVDSSQGSEYDYVIFTQTTETAHSCNVNRFNVAITRAKVGILCIMSDRDLYDKLQFTSLEIPRRNVATLQAENVTGLFKDCSKVITGLHPTQAPTHLSVDTKFKTEGLCVDIPGIPKDMTYRRLISMMGFKMNYQVNGYPNMFITREEAIRHVRAWIGFDVEGCHATREAVGTNLPLQLGFSTGVNLVAVPTGYVDTPNNTDFSRVSAKPPPGDQFKHLIPLMYKGLPWNVVRIKIVQMLSDTLKNLSDRVVFVLWAHGFELTSMKYFVKIGPERTCCLCDRRATCFSTASDTYACWHHSIGFDYVYNPFMIDVQQWGFTGNLQSNHDLYCQVHGNAHVASCDAIMTRCLAVHECFVKRVDWTIEYPIIGDELKINAACRKVQHMVVKAALLADKFPVLHDIGNPKAIKCVPQADVEWKFYDAQPCSDKAYKIEELFYSYATHSDKFTDGVCLFWNCNVDRYPANSIVCRFDTRVLSNLNLPGCDGGSLYVNKHAFHTPAFDKSAFVNLKQLPFFYYSDSPCESHGKQVVSDIDYVPLKSATCITRCNLGGAVCRHHANEYRLYLDAYNMMISAGFSLWVYKQFDTYNLWNTFTRLQSLENVAFNVVNKGHFDGQQGEVPVSIINNTVYTKVDGVDVELFENKTTLPVNVAFELWAKRNIKPVPEVKILNNLGVDIAANTVIWDYKRDAPAHISTIGVCSMTDIAKKPTETICAPLTVFFDGRVDGQVDLFRNARNGVLITEGSVKGLQPSVGPKQASLNGVTLIGEAVKTQFNYYKKVDGVVQQLPETYFTQSRNLQEFKPRSQMEIDFLELAMDEFIERYKLEGYAFEHIVYGDFSHSQLGGLHLLIGLAKRFKESPFELEDFIPMDSTVKNYFITDAQTGSSKCVCSVIDLLLDDFVEIIKSQDLSVVSKVVKVTIDYTEISFMLWCKDGHVETFYPKLQSSQAWQPGVAMPNLYKMQRMLLEKCDLQNYGDSATLPKGIMMNVAKYTQLCQYLNTLTLAVPYNMRVIHFGAGSDKGVAPGTAVLRQWLPTGTLLVDSDLNDFVSDADSTLIGDCATVHTANKWDLIISDMYDPKTKNVTKENDSKEGFFTYICGFIQQKLALGGSVAIKITEHSWNADLYKLMGHFAWWTAFVTNVNASSSEAFLIGCNYLGKPREQIDGYVMHANYIFWRNTNPIQLSSYSLFDMSKFPLKLRGTAVMSLKEGQINDMILSLLSKGRLIIRENNRVVISSDVLVNN*"
    refAA_S = "MFVFLVLLPLVSSQCVNLTTRTQLPPAYTNSFTRGVYYPDKVFRSSVLHSTQDLFLPFFSNVTWFHAIHVSGTNGTKRFDNPVLPFNDGVYFASTEKSNIIRGWIFGTTLDSKTQSLLIVNNATNVVIKVCEFQFCNDPFLGVYYHKNNKSWMESEFRVYSSANNCTFEYVSQPFLMDLEGKQGNFKNLREFVFKNIDGYFKIYSKHTPINLVRDLPQGFSALEPLVDLPIGINITRFQTLLALHRSYLTPGDSSSGWTAGAAAYYVGYLQPRTFLLKYNENGTITDAVDCALDPLSETKCTLKSFTVEKGIYQTSNFRVQPTESIVRFPNITNLCPFGEVFNATRFASVYAWNRKRISNCVADYSVLYNSASFSTFKCYGVSPTKLNDLCFTNVYADSFVIRGDEVRQIAPGQTGKIADYNYKLPDDFTGCVIAWNSNNLDSKVGGNYNYLYRLFRKSNLKPFERDISTEIYQAGSTPCNGVEGFNCYFPLQSYGFQPTNGVGYQPYRVVVLSFELLHAPATVCGPKKSTNLVKNKCVNFNFNGLTGTGVLTESNKKFLPFQQFGRDIADTTDAVRDPQTLEILDITPCSFGGVSVITPGTNTSNQVAVLYQDVNCTEVPVAIHADQLTPTWRVYSTGSNVFQTRAGCLIGAEHVNNSYECDIPIGAGICASYQTQTNSPRRARSVASQSIIAYTMSLGAENSVAYSNNSIAIPTNFTISVTTEILPVSMTKTSVDCTMYICGDSTECSNLLLQYGSFCTQLNRALTGIAVEQDKNTQEVFAQVKQIYKTPPIKDFGGFNFSQILPDPSKPSKRSFIEDLLFNKVTLADAGFIKQYGDCLGDIAARDLICAQKFNGLTVLPPLLTDEMIAQYTSALLAGTITSGWTFGAGAALQIPFAMQMAYRFNGIGVTQNVLYENQKLIANQFNSAIGKIQDSLSSTASALGKLQDVVNQNAQALNTLVKQLSSNFGAISSVLNDILSRLDKVEAEVQIDRLITGRLQSLQTYVTQQLIRAAEIRASANLAATKMSECVLGQSKRVDFCGKGYHLMSFPQSAPHGVVFLHVTYVPAQEKNFTTAPAICHDGKAHFPREGVFVSNGTHWFVTQRNFYEPQIITTDNTFVSGNCDVVIGIVNNTVYDPLQPELDSFKEELDKYFKNHTSPDVDLGDISGINASVVNIQKEIDRLNEVAKNLNESLIDLQELGKYEQYIKWPWYIWLGFIAGLIAIVMVTIMLCCMTSCCSCLKGCCSCGSCCKFDEDDSEPVLKGVKLHYT*"
    refAA_ORF3a = "MDLFMRIFTIGTVTLKQGEIKDATPSDFVRATATIPIQASLPFGWLIVGVALLAVFQSASKIITLKKRWQLALSKGVHFVCNLLLLFVTVYSHLLLVAAGLEAPFLYLYALVYFLQSINFVRIIMRLWLCWKCRSKNPLLYDANYFLCWHTNCYDYCIPYNSVTSSIVITSGDGTTSPISEHDYQIGGYTEKWESGVKDCVVLHSYFTSDYYQLYSTQLSTDTGVEHVTFFIYNKIVDEPEEHVQIHTIDGSSGVVNPVMEPIYDEPTTTTSVPL*"
    refAA_E = "MYSFVSEETGTLIVNSVLLFLAFVVFLLVTLAILTALRLCAYCCNIVNVSLVKPSFYVYSRVKNLNSSRVPDLLV*"
    refAA_M = "MADSNGTITVEELKKLLEQWNLVIGFLFLTWICLLQFAYANRNRFLYIIKLIFLWLLWPVTLACFVLAAVYRINWITGGIAIAMACLVGLMWLSYFIASFRLFARTRSMWSFNPETNILLNVPLHGTILTRPLLESELVIGAVILRGHLRIAGHHLGRCDIKDLPKEITVATSRTLSYYKLGASQRVAGDSGFAAYSRYRIGNYKLNTDHSSSSDNIALLVQ*"
    refAA_ORF6 = "MFHLVDFQVTIAEILLIIMRTFKVSIWNLDYIINLIIKNLSKSLTENKYSQLDEEQPMEID*"
    refAA_ORF7a = "MKIILFLALITLATCELYHYQECVRGTTVLLKEPCSSGTYEGNSPFHPLADNKFALTCFSTQFAFACPDGVKHVYQLRARSVSPKLFIRQEEVQELYSPIFLIVAAIVFITLCFTLKRKTE*"
    refAA_ORF7b = "MIELSLIDFYLCFLAFLLFLVLIMLIIFWFSLELQDHNETCHA*"
    refAA_ORF8 = "MKFLVFLGIITTVAAFHQECSLQSCTQHQPYVVDDPCPIHFYSKWYIRVGARKSAPLIELCVDEAGSKSPIQYIDIGNYTVSCLPFTINCQEPKLGSLVVRCSFYEDFLEYHDVRVVLDFI*"
    refAA_N = "MSDNGPQNQRNAPRITFGGPSDSTGSNQNGERSGARSKQRRPQGLPNNTASWFTALTQHGKEDLKFPRGQGVPINTNSSPDDQIGYYRRATRRIRGGDGKMKDLSPRWYFYYLGTGPEAGLPYGANKDGIIWVATEGALNTPKDHIGTRNPANNAAIVLQLPQGTTLPKGFYAEGSRGGSQASSRSSSRSRNSSRNSTPGSSRGTSPARMAGNGGDAALALLLLDRLNQLESKMSGKGQQQQGQTVTKKSAAEASKKPRQKRTATKAYNVTQAFGRRGPEQTQGNFGDQELIRQGTDYKHWPQIAQFAPSASAFFGMSRIGMEVTPSGTWLTYTGAIKLDDKDPNFKDQVILLNKHIDAYKTFPPTEPKKDKKKKADETQALPQRQKKQQTVTLLPAADLDDFSKQLQQSMSSADSTQA*"
    refAA_ORF9b = "MDPKISEMHPALRLVDPQIQLAVTRMENAVGRDQNNVGPKVYPIILRLGSPLSLNMARKTLNSLEDKAFQLTPIAVQMTKLATTEELPDEFVVVTVK*"
    if ref_pango ≠ "Wuhan"
        if haskey(nuc_genome_pango_dict, ref_pango)
            ref_seq = nuc_genome_pango_dict[ref_pango]
        elseif haskey(pango_predecessor_meta_dict, ref_pango)
            minus1_pango = ""
            minus2_pango = ""
            minus3_pango = ""
            minus4_pango = ""
            if haskey(pango_predecessor_meta_dict[ref_pango], 1)
                minus1_pango = pango_predecessor_meta_dict[ref_pango][1]
            end
            if haskey(pango_predecessor_meta_dict[ref_pango], 2)
                minus2_pango = pango_predecessor_meta_dict[ref_pango][2]
            end
            if haskey(pango_predecessor_meta_dict[ref_pango], 3)
                minus3_pango = pango_predecessor_meta_dict[ref_pango][3]
            end
            if haskey(pango_predecessor_meta_dict[ref_pango], 4)
                minus4_pango = pango_predecessor_meta_dict[ref_pango][4]
            end
            minus_pango_vec = [minus1_pango, minus2_pango, minus3_pango, minus4_pango]
            for minus_pango_X in minus_pango_vec
                if haskey(nuc_genome_pango_dict, minus_pango_X)
                    ref_seq = nuc_genome_pango_dict[minus_pango_X]
                    break
                end
            end
        else
            for x in 1:5
                minus_pango_X = pango_minus_X_fx(ref_pango, x)
                if haskey(nuc_genome_pango_dict, minus_pango_X)
                    ref_seq = nuc_genome_pango_dict[minus_pango_X]
                    break
                end
            end
        end
##########################################################################################
        if haskey(gene_AA_pango_dict, ref_pango)
            refAA_ORF1a = gene_AA_pango_dict[ref_pango]["ORF1a"]
            refAA_ORF1b = gene_AA_pango_dict[ref_pango]["ORF1b"]
            refAA_S = gene_AA_pango_dict[ref_pango]["S"]
            refAA_ORF3a = gene_AA_pango_dict[ref_pango]["ORF3a"]
            refAA_E = gene_AA_pango_dict[ref_pango]["E"]
            refAA_M = gene_AA_pango_dict[ref_pango]["M"]
            refAA_ORF6 = gene_AA_pango_dict[ref_pango]["ORF6"]
            refAA_ORF7a = gene_AA_pango_dict[ref_pango]["ORF7a"]
            refAA_ORF7b = gene_AA_pango_dict[ref_pango]["ORF7b"]
            refAA_ORF8 = gene_AA_pango_dict[ref_pango]["ORF8"]
            refAA_N = gene_AA_pango_dict[ref_pango]["N"]
            refAA_ORF9b = gene_AA_pango_dict[ref_pango]["ORF9b"]
        elseif haskey(pango_predecessor_meta_dict, ref_pango)
            minus1_pango = ""
            minus2_pango = ""
            minus3_pango = ""
            minus4_pango = ""
            if haskey(pango_predecessor_meta_dict[ref_pango], 1)
                minus1_pango = pango_predecessor_meta_dict[ref_pango][1]
            end
            if haskey(pango_predecessor_meta_dict[ref_pango], 2)
                minus2_pango = pango_predecessor_meta_dict[ref_pango][2]
            end
            if haskey(pango_predecessor_meta_dict[ref_pango], 3)
                minus3_pango = pango_predecessor_meta_dict[ref_pango][3]
            end
            if haskey(pango_predecessor_meta_dict[ref_pango], 4)
                minus4_pango = pango_predecessor_meta_dict[ref_pango][4]
            end
            minus_pango_vec = [minus1_pango, minus2_pango, minus3_pango, minus4_pango]
            for minus_pango_X in minus_pango_vec
                if haskey(gene_AA_pango_dict, minus_pango_X)
                    refAA_ORF1a = gene_AA_pango_dict[minus_pango_X]["ORF1a"]
                    refAA_ORF1b = gene_AA_pango_dict[minus_pango_X]["ORF1b"]
                    refAA_S = gene_AA_pango_dict[minus_pango_X]["S"]
                    refAA_ORF3a = gene_AA_pango_dict[minus_pango_X]["ORF3a"]
                    refAA_E = gene_AA_pango_dict[minus_pango_X]["E"]
                    refAA_M = gene_AA_pango_dict[minus_pango_X]["M"]
                    refAA_ORF6 = gene_AA_pango_dict[minus_pango_X]["ORF6"]
                    refAA_ORF7a = gene_AA_pango_dict[minus_pango_X]["ORF7a"]
                    refAA_ORF7b = gene_AA_pango_dict[minus_pango_X]["ORF7b"]
                    refAA_ORF8 = gene_AA_pango_dict[minus_pango_X]["ORF8"]
                    refAA_N = gene_AA_pango_dict[minus_pango_X]["N"]
                    refAA_ORF9b = gene_AA_pango_dict[minus_pango_X]["ORF9b"]
                    break
                end
            end
        else
            for x in 1:5
                minus_pango_X = pango_minus_X_fx(ref_pango, x)
                if haskey(gene_AA_pango_dict, minus_pango_X)
                    refAA_ORF1a = gene_AA_pango_dict[minus_pango_X]["ORF1a"]
                    refAA_ORF1b = gene_AA_pango_dict[minus_pango_X]["ORF1b"]
                    refAA_S = gene_AA_pango_dict[minus_pango_X]["S"]
                    refAA_ORF3a = gene_AA_pango_dict[minus_pango_X]["ORF3a"]
                    refAA_E = gene_AA_pango_dict[minus_pango_X]["E"]
                    refAA_M = gene_AA_pango_dict[minus_pango_X]["M"]
                    refAA_ORF6 = gene_AA_pango_dict[minus_pango_X]["ORF6"]
                    refAA_ORF7a = gene_AA_pango_dict[minus_pango_X]["ORF7a"]
                    refAA_ORF7b = gene_AA_pango_dict[minus_pango_X]["ORF7b"]
                    refAA_ORF8 = gene_AA_pango_dict[minus_pango_X]["ORF8"]
                    refAA_N = gene_AA_pango_dict[minus_pango_X]["N"]
                    refAA_ORF9b = gene_AA_pango_dict[minus_pango_X]["ORF9b"]
                    break
                end
            end
        end
    end
    return ref_seq, refAA_ORF1a, refAA_ORF1b, refAA_S, refAA_ORF3a, refAA_E, refAA_M, refAA_ORF6, refAA_ORF7a, refAA_ORF7b, refAA_ORF8, refAA_N, refAA_ORF9b
end
###########################################################################################################################################################################
###########################################################################################################################################################################
function mixed_nucs_filter(mut_arr)
    mixed_nuc_arr = Vector{String}()
    mixed_nuc_res_list = Set(["Y", "R", "K", "M", "W", "S"])
    for mut in mut_arr
        if string(mut[end]) in mixed_nuc_res_list
            push!(mixed_nuc_arr, mut)
        end
    end
    return mixed_nuc_arr
end
#####################################################################################################################################
N3_syn = ["TCT", "TCC", "TCA", "TCG", "CTT", "CTC", "CTA", "CTG", "CCT", "CCC", "CCA", "CCG", "CGT", "CGC", "CGA", "CGG", "ACT", "ACC", "ACA", "ACG", "GTT", "GTC", "GTA", "GTG", "GCT", "GCC", "GCA", "GCG", "GGT", "GGC", "GGA", "GGG"]
N3_tv = ["TTT", "TTC", "TTA", "TTG", "TAT", "TAC", "TAA", "TAG", "AAT", "AAC", "AAA", "AAG", "AGT", "AGC", "AGA", "AGG", "GAT", "GAC", "GAA", "GAG"]
#####################################################################################################################################
function mixed2nuc(mix_mut::String)
    nuc_mut = mix_mut
    qrynuc = qry_nuc_comprehensive_dict[mix_mut]
    refnuc = ref_nuc_comprehensive_dict[mix_mut]
    pos_str = nuc_mut_int_string_comprehensive_dict[mix_mut]
    ref_n_pos = refnuc*pos_str
    if length(mix_mut) ≥ 4
        if qrynuc == "T"
            nuc_mut = mix_mut
        elseif qrynuc == "C"
            nuc_mut = mix_mut
        elseif qrynuc == "A"
            nuc_mut = mix_mut
        elseif qrynuc == "G"
            nuc_mut = mix_mut
        elseif qrynuc == "N"
            nuc_mut = mix_mut
        end   
        if refnuc == "T"
            if qrynuc == "Y"
                nuc_mut = ref_n_pos*"C"
            elseif qrynuc == "W"
                nuc_mut = ref_n_pos*"A"
            elseif qrynuc == "K"
                nuc_mut = ref_n_pos*"G"
            elseif qrynuc == "M"
                nuc_mut = "$(ref_n_pos)C, $(ref_n_pos)A"
            elseif qrynuc == "S"
                nuc_mut = "$(ref_n_pos)C, $(ref_n_pos)G"
            elseif qrynuc == "R"
                nuc_mut = "$(ref_n_pos)A, $(ref_n_pos)G"
            end
        end
        if refnuc == "C"
            if qrynuc == "Y"
                nuc_mut = ref_n_pos*"T"
            elseif qrynuc == "M"
                nuc_mut = ref_n_pos*"A"
            elseif qrynuc == "S"
                nuc_mut = ref_n_pos*"G"
            elseif qrynuc == "R"
                nuc_mut = "$(ref_n_pos)A, $(ref_n_pos)G"
            elseif qrynuc == "W"
                nuc_mut = "$(ref_n_pos)T, $(ref_n_pos)A"
            elseif qrynuc == "K"
                nuc_mut = "$(ref_n_pos)T, $(ref_n_pos)G"
            end
        end
        if refnuc == "A"
            if qrynuc == "R"
                nuc_mut = ref_n_pos*"G"
            elseif qrynuc == "W"
                nuc_mut = ref_n_pos*"T"
            elseif qrynuc == "M"
                nuc_mut = ref_n_pos*"C"
            elseif qrynuc == "Y"
                nuc_mut = "$(ref_n_pos)T, $(ref_n_pos)C"
            elseif qrynuc == "K"
                nuc_mut = "$(ref_n_pos)T, $(ref_n_pos)G"
            elseif qrynuc == "S"
                nuc_mut = "$(ref_n_pos)C, $(ref_n_pos)G"
            end
        end
        if refnuc == "G"
            if qrynuc == "R"
                nuc_mut = ref_n_pos*"A"
            elseif qrynuc == "K"
                nuc_mut = ref_n_pos*"T"
            elseif qrynuc == "S"
                nuc_mut = ref_n_pos*"C"
            elseif qrynuc == "Y"
                nuc_mut = "$(ref_n_pos)T, $(ref_n_pos)C"
            elseif qrynuc == "W"
                nuc_mut = "$(ref_n_pos)T, $(ref_n_pos)A"
            elseif qrynuc == "M"
                nuc_mut = "$(ref_n_pos)C, $(ref_n_pos)A"
            end
        end
    end
    return nuc_mut
end
######################################################################################################################################
function muts_to_strings(mut_list_in_string_form::String)
    mut_arr = split(mut_list_in_string_form, ", ")
    mut_array = Vector{String}()
    for mut in mut_arr
        if string(mut[end]) ≠ "-"
            mutstr = string(mut)
            push!(mut_array, mutstr)
        end
    end
    sortKey(n) = (length(n), nuc_mut_int_comprehensive_dict[n])  ## Fx ##
    mut_array_sort = sort(mut_array, by = x -> sortKey(x))
#    mixed_mut_arr = mixed_nucs_filter(mut_arr)
    return mut_array_sort
end
######################################################################################################################################
function mixed_mut_to_regular_mut(mixed_nuc_muts)            ### New, 2025-1-26 (entire function)
    ct = 0
    mixed_regular_muts = Vector{String}()
    for i in 1:length(mixed_nuc_muts)
        mut = mixed_nuc_muts[i]
        qrynuc = qry_nuc_comprehensive_dict[mut]
        refnuc = ref_nuc_comprehensive_dict[mut]
        pos_str = nuc_mut_int_string_comprehensive_dict[mut]
        ref_n_pos = refnuc*pos_str
        if refnuc == "T"
            if qrynuc == "Y"
                new_end = "C"
                new_mut = ref_n_pos*new_end
                push!(mixed_regular_muts, new_mut)
            end
            if qrynuc == "R"
                new_end1 = "A"
                new_end2 = "G"
                new_mut1 = ref_n_pos*new_end1
                new_mut2 = ref_n_pos*new_end2
                push!(mixed_regular_muts, new_mut1)
                push!(mixed_regular_muts, new_mut2)
            end
            if qrynuc == "K"
                new_end = "G"
                new_mut = ref_n_pos*new_end
                push!(mixed_regular_muts, new_mut)
            end
            if qrynuc == "M"
                new_end1 = "C"
                new_end2 = "A"
                new_mut1 = ref_n_pos*new_end1
                new_mut2 = ref_n_pos*new_end2
                push!(mixed_regular_muts, new_mut1)
                push!(mixed_regular_muts, new_mut2)
            end
            if qrynuc == "W"
                new_end = "A"
                new_mut = ref_n_pos*new_end
                push!(mixed_regular_muts, new_mut)
            end
            if qrynuc == "S"
                new_end1 = "C"
                new_end2 = "G"
                new_mut1 = ref_n_pos*new_end1
                new_mut2 = ref_n_pos*new_end2
                push!(mixed_regular_muts, new_mut1)
                push!(mixed_regular_muts, new_mut2)
            end
        end
        if refnuc == "C"
            if qrynuc == "Y"
                new_end = "T"
                new_mut = ref_n_pos*new_end
                push!(mixed_regular_muts, new_mut)
            end
            if qrynuc == "R"
                new_end1 = "A"
                new_end2 = "G"
                new_mut1 = ref_n_pos*new_end1
                new_mut2 = ref_n_pos*new_end2
                push!(mixed_regular_muts, new_mut1)
                push!(mixed_regular_muts, new_mut2)
            end
            if qrynuc == "K"
                new_end1 = "T"
                new_end2 = "G"
                new_mut1 = ref_n_pos*new_end1
                new_mut2 = ref_n_pos*new_end2
                push!(mixed_regular_muts, new_mut1)
                push!(mixed_regular_muts, new_mut2)
            end
            if qrynuc == "M"
                new_end = "A"
                new_mut = ref_n_pos*new_end
                push!(mixed_regular_muts, new_mut)
            end
            if qrynuc == "W"
                new_end1 = "T"
                new_end2 = "A"
                new_mut1 = ref_n_pos*new_end1
                new_mut2 = ref_n_pos*new_end2
                push!(mixed_regular_muts, new_mut1)
                push!(mixed_regular_muts, new_mut2)
            end
            if qrynuc == "S"
                new_end = "G"
                new_mut = ref_n_pos*new_end
                push!(mixed_regular_muts, new_mut)
            end
        end
        if refnuc == "A"
            if qrynuc == "Y"
                new_end1 = "T"
                new_end2 = "C"
                new_mut1 = ref_n_pos*new_end1
                new_mut2 = ref_n_pos*new_end2
                push!(mixed_regular_muts, new_mut1)
                push!(mixed_regular_muts, new_mut2)
            end
            if qrynuc == "R"
                new_end = "G"
                new_mut = ref_n_pos*new_end
                push!(mixed_regular_muts, new_mut)
            end
            if qrynuc == "K"
                new_end1 = "T"
                new_end2 = "G"
                new_mut1 = ref_n_pos*new_end1
                new_mut2 = ref_n_pos*new_end2
                push!(mixed_regular_muts, new_mut1)
                push!(mixed_regular_muts, new_mut2)
            end
            if qrynuc == "M"
                new_end = "C"
                new_mut = ref_n_pos*new_end
                push!(mixed_regular_muts, new_mut)
            end
            if qrynuc == "W"
                new_end = "T"
                new_mut = ref_n_pos*new_end
                push!(mixed_regular_muts, new_mut)
            end
            if qrynuc == "S"
                new_end1 = "C"
                new_end2 = "G"
                new_mut1 = ref_n_pos*new_end1
                new_mut2 = ref_n_pos*new_end2
                push!(mixed_regular_muts, new_mut1)
                push!(mixed_regular_muts, new_mut2)
            end
        end
        if refnuc == "G"
            if qrynuc == "Y"
                new_end1 = "T"
                new_end2 = "C"
                new_mut1 = ref_n_pos*new_end1
                new_mut2 = ref_n_pos*new_end2
                push!(mixed_regular_muts, new_mut1)
                push!(mixed_regular_muts, new_mut2)
            end
            if qrynuc == "R"
                new_end = "A"
                new_mut = ref_n_pos*new_end
                push!(mixed_regular_muts, new_mut)
            end
            if qrynuc == "K"
                new_end = "T"
                new_mut = ref_n_pos*new_end
                push!(mixed_regular_muts, new_mut)
            end
            if qrynuc == "M"
                new_end1 = "C"
                new_end2 = "A"
                new_mut1 = ref_n_pos*new_end1
                new_mut2 = ref_n_pos*new_end2
                push!(mixed_regular_muts, new_mut1)
                push!(mixed_regular_muts, new_mut2)
            end
            if qrynuc == "W"
                new_end1 = "T"
                new_end2 = "A"
                new_mut1 = ref_n_pos*new_end1
                new_mut2 = ref_n_pos*new_end2
                push!(mixed_regular_muts, new_mut1)
                push!(mixed_regular_muts, new_mut2)
            end
            if qrynuc == "S"
                new_end = "C"
                new_mut = ref_n_pos*new_end
                push!(mixed_regular_muts, new_mut)
            end
        end
    end
    return mixed_regular_muts
end
######################################################################################################################################
gene_print_array = ["S", "N", "E", "M", "ORF3a", "ORF6", "ORF7a", "ORF7b", "ORF8", "ORF9b", "ORF1a", "ORF1b"]
#######################################################################################################################################
function nuc_to_AA(ref_pango::String, muts::Vector{String})
    muts_filtered = filter(!isempty, muts)
#    all_muts_sort = sort(muts_filtered, by = x -> nuc_mut_int_comprehensive_dict[x])
    ref_seq, refAA_ORF1a, refAA_ORF1b, refAA_S, refAA_ORF3a, refAA_E, refAA_M, refAA_ORF6, refAA_ORF7a, refAA_ORF7b, refAA_ORF8, refAA_N, refAA_ORF9b = get_ref_pango_nucseq_and_geneseqs(ref_pango)
    coding_ranges = BitSet([266:13467..., 13469:21555..., 21563:25384..., 25393:26220..., 26245:26472..., 26523:27191..., 27202:27387..., 27394:27755..., 27760:27887..., 27894:28259..., 28274:29533..., 28284:28577...])
    noncoding_ranges = BitSet([1:265..., 21556:21562..., 25385:25392..., 26221:26244..., 26473:26522..., 27192:27201..., 27388:27393..., 27888:27893..., 28260:28273..., 29534:29830...])
    coding_range_9b = BitSet([28284:28577...])
    gene_nuc_starts = Dict{Int, Int}(0=>263, 1=>13465, 2=>21560, 3=>25390, 4=>26242, 5=>26520, 6=>27199, 7=>27391, 8=>27753, 9=>27891, 10=>28271, 11=>28281)
    ref_AA_genes = Dict{Int, String}(0=>refAA_ORF1a, 1=>refAA_ORF1b, 2=>refAA_S, 3=>refAA_ORF3a, 4=>refAA_E, 5=>refAA_M, 6=>refAA_ORF6, 7=>refAA_ORF7a, 8=>refAA_ORF7b, 9=>refAA_ORF8, 10=>refAA_N, 11=>refAA_ORF9b)
    gene_strings = Dict{Int, String}(0=>"ORF1a", 1=>"ORF1b", 2=>"S", 3=>"ORF3a", 4=>"E", 5=>"M", 6=>"ORF6", 7=>"ORF7a", 8=>"ORF7b", 9=>"ORF8", 10=>"N", 11=>"ORF9b")
    nuc_gene_num = Dict{Int, Int}()
    nuc_gene_num_9b = Dict{Int, Int}()
    nonsynonymous_nuc_to_AA_dict = Dict{String, String}()
    nonsynonymous_nuc_to_context_dict = Dict{String, String}()
    all_nuc_to_AA_dict = Dict{String, String}()
    all_nuc_to_context_dict = Dict{String, String}()
    synonymous_nuc_to_AA_dict = Dict{String, String}()
    synonymous_nuc_to_context_dict = Dict{String, String}()
    noncoding_nuc_to_context_dict = Dict{String, String}()
    noncoding_to_noncoding_region_dict = Dict{String, String}()
    noncoding_range_dict = Dict{Vector{Int}, String}([1, 265]=>"5' UTR", [21556, 21562]=>"Spike TRS", [25385, 25392]=>"ORF3a TRS", [26221, 26234]=>"ORF3a-E UTR", [26235, 26244]=>"E TRS", [26473, 26522]=>"E-M UTR", [27192, 27201]=>"M-ORF6 UTR", [27388, 27393]=>"ORF7a TRS", [27888, 27893]=>"ORF8 TRS", [28260, 28273]=>"N/ORF9b TRS", [29534, 29830]=>"3' UTR", [29794, 29801]=>"ONM")
    noncoding_nuc_vector = Vector{String}()
################################################        
    gene_nuc_arr = [[266:13467...], [13469:21555...], [21563:25384...], [25393:26220...], [26245:26472...], [26523:27191...], [27202:27387...], [27394:27755...], [27760:27887...], [27894:28259...], [28274:29533...], [28284:28577...]]
    for i in 1:length(gene_nuc_arr)-1
        for nuc_pos in gene_nuc_arr[i]
            nuc_gene_num[nuc_pos] = i-1
        end
    end
    for nuc_pos in gene_nuc_arr[end]
        nuc_gene_num_9b[nuc_pos] = 11
    end

    rem0_gene = [5, 8, 9, 11]
    rem1_gene = [1, 3, 4, 6, 7]
    rem2_gene = [0, 2, 10]
    rem0 = BitSet([26523:27191..., 27760:27887..., 27894:28259..., 28284:28577...])
    rem1 = BitSet([13469:21555..., 25393:26220..., 26245:26472..., 27202:27387..., 27394:27755...])
    rem2 = BitSet([266:13467..., 21563:25384..., 28274:29533...])
    rem9b = BitSet([28284:28577...])
    rem7ab = BitSet([27756:27759...])
    
    gene_num(nuc_mut) = nuc_gene_num[nuc_mut_int_comprehensive_dict[nuc_mut]]                                                          ### FUNCTION #####################
    nuc_to_AA_pos(nuc_mut) = string((nuc_mut_int_comprehensive_dict[nuc_mut] - gene_nuc_starts[gene_num(nuc_mut)])÷3)                  ### FUNCTION #####################
    nuc_to_AA_pos_9b(nuc_mut) = string((nuc_mut_int_comprehensive_dict[nuc_mut] - 28281)÷3)                                            ### FUNCTION #####################
    nuc2AA_ORF1a(nuc_mut, refAA, qryAA) = gene_strings[gene_num(nuc_mut)]*":"*refAA*nuc_to_AA_pos(nuc_mut)*qryAA   ### FUNCTION #####################
    nuc2AA_ORF9b(nuc_mut, refAA, qryAA) = "ORF9b:"*refAA*nuc_to_AA_pos_9b(nuc_mut)*qryAA
    
    nuc_codon_pos_dict = Dict{Int, Int}()
    for nuc_pos in coding_ranges
        gene_number = nuc_gene_num[nuc_pos]
        gene_start = gene_nuc_starts[gene_number]
        codon_num = (nuc_pos-gene_start)%3 + 1
        nuc_codon_pos_dict[nuc_pos] = codon_num
    end
    nuc_codon_pos_dict_9b = Dict{Int, Int}()
    for nuc_pos in coding_range_9b
        gene_number = 11
        gene_start = gene_nuc_starts[gene_number]
        codon_num = (nuc_pos-gene_start)%3 + 1
        nuc_codon_pos_dict_9b[nuc_pos] = codon_num
    end
#######################################################################################################################
    for nuc_mut in muts
        mut = mixed2nuc(nuc_mut)
        if ',' in mut
            mut1 = string(split(mut, ",")[1])
            mut2 = string(split(mut, ",")[2])
            push!(muts, mut1)
            push!(muts, mut2)
            filter!(x -> !(length(x)>6), muts)
        end
    end
    coding_ranges = BitSet([266:13467..., 13468:21555..., 21563:25384..., 25393:26220..., 26245:26472..., 26523:27191..., 27202:27387..., 27394:27755..., 27760:27887..., 27894:28259..., 28274:29533...])
    N_9b_synonymous = Set(["C28379A", "C28394A", "T28406C", "C28475A", "C28535A", "A28547C"])
    AA_mut_set = Set{String}()
    AA_mut = ""
    for nuc_mut in muts
        pos = nuc_mut_int_comprehensive_dict[nuc_mut]
        if pos in coding_ranges
            mut = mixed2nuc(nuc_mut)  
            gene_number = gene_num(mut)
            ref_triple = ""
            qry_triple = ""
            ref_triple_context = ""
            qry_triple_context = ""
            ref2qry_context = ""
            if nuc_codon_pos_dict[pos] == 1
                ref_triple = ref_nuc_comprehensive_dict[mut]*string(ref_seq[pos+1])  *string(ref_seq[pos+2])
                qry_triple = qry_nuc_comprehensive_dict[mut]*string(ref_seq[pos+1])*string(ref_seq[pos+2])
                ref_triple_context = string(ref_seq[pos-6])*string(ref_seq[pos-5])*string(ref_seq[pos-4])*string(ref_seq[pos-3])*string(ref_seq[pos-2])*string(ref_seq[pos-1])*ref_triple*string(ref_seq[pos+3])*string(ref_seq[pos+4])*string(ref_seq[pos+5])*string(ref_seq[pos+6])*string(ref_seq[pos+7])*string(ref_seq[pos+8])
                qry_triple_context = string(ref_seq[pos-6])*string(ref_seq[pos-5])*string(ref_seq[pos-4])*string(ref_seq[pos-3])*string(ref_seq[pos-2])*string(ref_seq[pos-1])*qry_triple*string(ref_seq[pos+3])*string(ref_seq[pos+4])*string(ref_seq[pos+5])*string(ref_seq[pos+6])*string(ref_seq[pos+7])*string(ref_seq[pos+8])
            elseif nuc_codon_pos_dict[pos] == 2
                ref_triple = string(ref_seq[pos-1])*ref_nuc_comprehensive_dict[mut]  *string(ref_seq[pos+1])
                qry_triple = string(ref_seq[pos-1])*qry_nuc_comprehensive_dict[mut]*string(ref_seq[pos+1])
                ref_triple_context = string(ref_seq[pos-7])*string(ref_seq[pos-6])*string(ref_seq[pos-5])*string(ref_seq[pos-4])*string(ref_seq[pos-3])*string(ref_seq[pos-2])*ref_triple*string(ref_seq[pos+2])*string(ref_seq[pos+3])*string(ref_seq[pos+4])*string(ref_seq[pos+5])*string(ref_seq[pos+6])*string(ref_seq[pos+7])
                qry_triple_context = string(ref_seq[pos-7])*string(ref_seq[pos-6])*string(ref_seq[pos-5])*string(ref_seq[pos-4])*string(ref_seq[pos-3])*string(ref_seq[pos-2])*qry_triple*string(ref_seq[pos+2])*string(ref_seq[pos+3])*string(ref_seq[pos+4])*string(ref_seq[pos+5])*string(ref_seq[pos+6])*string(ref_seq[pos+7])
            elseif nuc_codon_pos_dict[pos] == 3
                ref_triple = string(ref_seq[pos-2])*string(ref_seq[pos-1])*ref_nuc_comprehensive_dict[mut]
                qry_triple = string(ref_seq[pos-2])*string(ref_seq[pos-1])*qry_nuc_comprehensive_dict[mut]
                ref_triple_context = string(ref_seq[pos-8])*string(ref_seq[pos-7])*string(ref_seq[pos-6])*string(ref_seq[pos-5])*string(ref_seq[pos-4])*string(ref_seq[pos-3])*ref_triple*string(ref_seq[pos+1])*string(ref_seq[pos+2])*string(ref_seq[pos+3])*string(ref_seq[pos+4])*string(ref_seq[pos+5])*string(ref_seq[pos+6])
                qry_triple_context = string(ref_seq[pos-8])*string(ref_seq[pos-7])*string(ref_seq[pos-6])*string(ref_seq[pos-5])*string(ref_seq[pos-4])*string(ref_seq[pos-3])*qry_triple*string(ref_seq[pos+1])*string(ref_seq[pos+2])*string(ref_seq[pos+3])*string(ref_seq[pos+4])*string(ref_seq[pos+5])*string(ref_seq[pos+6])
            end
            refAA = AA_triplets[ref_triple]
            qryAA = AA_triplets[qry_triple]
            AA_mut = nuc2AA_ORF1a(mut, refAA, qryAA)
            push!(AA_mut_set, AA_mut)
            ref2qry_context = ref_triple_context*"-->"*qry_triple_context
            all_nuc_to_AA_dict[mut] = AA_mut
            if refAA == qryAA && !(pos in rem9b)
                synonymous_nuc_to_AA_dict[mut] = AA_mut
                synonymous_nuc_to_context_dict[mut] = ref2qry_context
            elseif refAA == qryAA && pos in rem9b && mut in N_9b_synonymous
                synonymous_nuc_to_AA_dict[mut] = AA_mut
                synonymous_nuc_to_context_dict[mut] = ref2qry_context
            else
                nonsynonymous_nuc_to_AA_dict[mut] = AA_mut
                nonsynonymous_nuc_to_context_dict[mut] = ref2qry_context
#                push!(nonsynonymous_nuc_muts, mut)
            end
            all_nuc_to_context_dict[mut] = ref2qry_context
###################################
            for nuc_mut2 in muts
                mut2 = mixed2nuc(nuc_mut2)
                pos2 = nuc_mut_int_comprehensive_dict[mut2]
                if pos2 in coding_ranges
                    gene_number2 = gene_num(mut2)
                    if mut ≠ mut2 && gene_number == gene_number2 && nuc_to_AA_pos(mut) == nuc_to_AA_pos(mut2)
                        if nuc_codon_pos_dict[pos] == 1 && nuc_codon_pos_dict[pos2] == 2
                            ref_triple = ref_nuc_comprehensive_dict[mut]*ref_nuc_comprehensive_dict[mut2]*string(ref_seq[pos+2])
                            qry_triple = qry_nuc_comprehensive_dict[mut]*qry_nuc_comprehensive_dict[mut2]*string(ref_seq[pos+2])
                            ref_triple_context = string(ref_seq[pos-3])*string(ref_seq[pos-2])*string(ref_seq[pos-1])*ref_triple*string(ref_seq[pos+3])*string(ref_seq[pos+4])*string(ref_seq[pos+5])
                            qry_triple_context = string(ref_seq[pos-3])*string(ref_seq[pos-2])*string(ref_seq[pos-1])*qry_triple*string(ref_seq[pos+3])*string(ref_seq[pos+4])*string(ref_seq[pos+5])
                        elseif nuc_codon_pos_dict[pos] == 1 && nuc_codon_pos_dict[pos2] == 3
                            ref_triple = ref_nuc_comprehensive_dict[mut]*string(ref_seq[pos+1])*ref_nuc_comprehensive_dict[mut2]
                            qry_triple = qry_nuc_comprehensive_dict[mut]*string(ref_seq[pos+1])*qry_nuc_comprehensive_dict[mut2]
                            ref_triple_context = string(ref_seq[pos-3])*string(ref_seq[pos-2])*string(ref_seq[pos-1])*ref_triple*string(ref_seq[pos+3])*string(ref_seq[pos+4])*string(ref_seq[pos+5])
                            qry_triple_context = string(ref_seq[pos-3])*string(ref_seq[pos-2])*string(ref_seq[pos-1])*qry_triple*string(ref_seq[pos+3])*string(ref_seq[pos+4])*string(ref_seq[pos+5])
                        elseif nuc_codon_pos_dict[pos] == 2 && nuc_codon_pos_dict[pos2] == 3
                            ref_triple = string(ref_seq[pos-1])*ref_nuc_comprehensive_dict[mut]*ref_nuc_comprehensive_dict[mut2]
                            qry_triple = string(ref_seq[pos-1])*qry_nuc_comprehensive_dict[mut]*qry_nuc_comprehensive_dict[mut2]
                            ref_triple_context = string(ref_seq[pos-4])*string(ref_seq[pos-3])*string(ref_seq[pos-2])*ref_triple*string(ref_seq[pos+2])*string(ref_seq[pos+3])*string(ref_seq[pos+4])
                            qry_triple_context = string(ref_seq[pos-4])*string(ref_seq[pos-3])*string(ref_seq[pos-2])*qry_triple*string(ref_seq[pos+2])*string(ref_seq[pos+3])*string(ref_seq[pos+4])
                        elseif nuc_codon_pos_dict[pos2] == 1 && nuc_codon_pos_dict[pos] == 2
                            ref_triple = ref_nuc_comprehensive_dict[mut2]*ref_nuc_comprehensive_dict[mut]*string(ref_seq[pos+1])
                            qry_triple = qry_nuc_comprehensive_dict[mut2]*qry_nuc_comprehensive_dict[mut]*string(ref_seq[pos+1])
                            ref_triple_context = string(ref_seq[pos-4])*string(ref_seq[pos-3])*string(ref_seq[pos-2])*ref_triple*string(ref_seq[pos+2])*string(ref_seq[pos+3])*string(ref_seq[pos+4])
                            qry_triple_context = string(ref_seq[pos-4])*string(ref_seq[pos-3])*string(ref_seq[pos-2])*qry_triple*string(ref_seq[pos+2])*string(ref_seq[pos+3])*string(ref_seq[pos+4])
                        elseif nuc_codon_pos_dict[pos2] == 1 && nuc_codon_pos_dict[pos] == 3
                            ref_triple = ref_nuc_comprehensive_dict[mut2]*string(ref_seq[pos-1])*ref_nuc_comprehensive_dict[mut]
                            qry_triple = qry_nuc_comprehensive_dict[mut2]*string(ref_seq[pos-1])*qry_nuc_comprehensive_dict[mut]
                            ref_triple_context = string(ref_seq[pos-5])*string(ref_seq[pos-4])*string(ref_seq[pos-3])*ref_triple*string(ref_seq[pos+1])*string(ref_seq[pos+2])*string(ref_seq[pos+3])
                            qry_triple_context = string(ref_seq[pos-5])*string(ref_seq[pos-4])*string(ref_seq[pos-3])*qry_triple*string(ref_seq[pos+1])*string(ref_seq[pos+2])*string(ref_seq[pos+3])
                        elseif nuc_codon_pos_dict[pos2] == 2 && nuc_codon_pos_dict[pos] == 3
                            ref_triple = string(ref_seq[pos-2])*ref_nuc_comprehensive_dict[mut2]*ref_nuc_comprehensive_dict[mut]
                            qry_triple = string(ref_seq[pos-2])*qry_nuc_comprehensive_dict[mut2]*qry_nuc_comprehensive_dict[mut]
                        end
                        refAA2 = AA_triplets[ref_triple]
                        qryAA2 = AA_triplets[qry_triple]
                        AA_mut2 = nuc2AA_ORF1a(mut2, refAA2, qryAA2)
                        push!(AA_mut_set, AA_mut2)
                        ref2qry_context = ref_triple_context*"-->"*qry_triple_context
                        all_nuc_to_AA_dict[mut2] = AA_mut2
                        all_nuc_to_AA_dict[mut] = AA_mut2
                        if refAA2 == qryAA2 && !(pos2 in rem9b)
                            synonymous_nuc_to_AA_dict[mut2] = AA_mut2
                            synonymous_nuc_to_context_dict[mut2] = ref2qry_context 
                        else
                            nonsynonymous_nuc_to_AA_dict[mut2] = AA_mut2
                            nonsynonymous_nuc_to_context_dict[mut2] = ref2qry_context
                            nonsynonymous_nuc_to_AA_dict[mut] = AA_mut2
                            nonsynonymous_nuc_to_context_dict[mut] = ref2qry_context
                        end
                        all_nuc_to_context_dict[mut] = ref2qry_context
                        all_nuc_to_context_dict[mut2] = ref2qry_context
                    end
                end
            end
        else                  
            npos = nuc_mut_int_comprehensive_dict[nuc_mut]
            qry_nuc = qry_nuc_comprehensive_dict[nuc_mut]
            ref_nc_nuc_context = string(ref_seq[npos-8])*string(ref_seq[npos-7])*string(ref_seq[npos-6])*string(ref_seq[npos-5])*string(ref_seq[npos-4])*string(ref_seq[npos-3])*string(ref_seq[npos-2])*string(ref_seq[npos-1])*string(ref_seq[npos])*string(ref_seq[npos+1])*string(ref_seq[npos+2])*string(ref_seq[npos+3])*string(ref_seq[npos+4])*string(ref_seq[npos+5])*string(ref_seq[npos+6])*string(ref_seq[npos+7])*string(ref_seq[npos+8])
            qry_nc_nuc_context = string(ref_seq[npos-8])*string(ref_seq[npos-7])*string(ref_seq[npos-6])*string(ref_seq[npos-5])*string(ref_seq[npos-4])*string(ref_seq[npos-3])*string(ref_seq[npos-2])*string(ref_seq[npos-1])*qry_nuc*string(ref_seq[npos+1])*string(ref_seq[npos+2])*string(ref_seq[npos+3])*string(ref_seq[npos+4])*string(ref_seq[npos+5])*string(ref_seq[npos+6])*string(ref_seq[npos+7])*string(ref_seq[npos+8])
            full_nc_context = ref_nc_nuc_context*"|"*qry_nc_nuc_context
            noncoding_nuc_to_context_dict[nuc_mut] = full_nc_context
            for (start_end, place) in noncoding_range_dict
                frst = start_end[1]
                last = start_end[2]
                if npos ≥ frst && npos ≤ last
                    noncoding_to_noncoding_region_dict[nuc_mut] = place
                    mut_vec = [nuc_mut, place]
                    push!(noncoding_nuc_vector, nuc_mut)
                end
            end
        end
    end
#########################################################################################################
    for nuc_mut in muts
        pos_9b = nuc_mut_int_comprehensive_dict[nuc_mut]
        if pos_9b in rem9b
            mut_9b = mixed2nuc(nuc_mut)
            pos_9b = nuc_mut_int_comprehensive_dict[mut_9b]   
            gene_number_9b = 11
            ref_triple_9b = ""
            qry_triple_9b = ""
            ref_triple_context_9b = ""
            qry_triple_context_9b = ""
            ref2qry_context_9b = ""
            if nuc_codon_pos_dict_9b[pos_9b] == 1
                ref_triple_9b = ref_nuc_comprehensive_dict[mut_9b]*string(ref_seq[pos_9b+1])  *string(ref_seq[pos_9b+2])
                qry_triple_9b = qry_nuc_comprehensive_dict[mut_9b]*string(ref_seq[pos_9b+1])*string(ref_seq[pos_9b+2])
                ref_triple_context_9b = string(ref_seq[pos_9b-6])*string(ref_seq[pos_9b-5])*string(ref_seq[pos_9b-4])*string(ref_seq[pos_9b-3])*string(ref_seq[pos_9b-2])*string(ref_seq[pos_9b-1])*ref_triple_9b*string(ref_seq[pos_9b+3])*string(ref_seq[pos_9b+4])*string(ref_seq[pos_9b+5])*string(ref_seq[pos_9b+6])*string(ref_seq[pos_9b+7])*string(ref_seq[pos_9b+8])
                qry_triple_context_9b = string(ref_seq[pos_9b-6])*string(ref_seq[pos_9b-5])*string(ref_seq[pos_9b-4])*string(ref_seq[pos_9b-3])*string(ref_seq[pos_9b-2])*string(ref_seq[pos_9b-1])*qry_triple_9b*string(ref_seq[pos_9b+3])*string(ref_seq[pos_9b+4])*string(ref_seq[pos_9b+5])*string(ref_seq[pos_9b+6])*string(ref_seq[pos_9b+7])*string(ref_seq[pos_9b+8])
            elseif nuc_codon_pos_dict_9b[pos_9b] == 2
                ref_triple_9b = string(ref_seq[pos_9b-1])*ref_nuc_comprehensive_dict[mut_9b]  *string(ref_seq[pos_9b+1])
                qry_triple_9b = string(ref_seq[pos_9b-1])*qry_nuc_comprehensive_dict[mut_9b]*string(ref_seq[pos_9b+1])
                ref_triple_context_9b = string(ref_seq[pos_9b-7])*string(ref_seq[pos_9b-6])*string(ref_seq[pos_9b-5])*string(ref_seq[pos_9b-4])*string(ref_seq[pos_9b-3])*string(ref_seq[pos_9b-2])*ref_triple_9b*string(ref_seq[pos_9b+2])*string(ref_seq[pos_9b+3])*string(ref_seq[pos_9b+4])*string(ref_seq[pos_9b+5])*string(ref_seq[pos_9b+6])*string(ref_seq[pos_9b+7])
                qry_triple_context_9b = string(ref_seq[pos_9b-7])*string(ref_seq[pos_9b-6])*string(ref_seq[pos_9b-5])*string(ref_seq[pos_9b-4])*string(ref_seq[pos_9b-3])*string(ref_seq[pos_9b-2])*qry_triple_9b*string(ref_seq[pos_9b+2])*string(ref_seq[pos_9b+3])*string(ref_seq[pos_9b+4])*string(ref_seq[pos_9b+5])*string(ref_seq[pos_9b+6])*string(ref_seq[pos_9b+7])
            elseif nuc_codon_pos_dict_9b[pos_9b] == 3
                ref_triple_9b = string(ref_seq[pos_9b-2])*string(ref_seq[pos_9b-1])*ref_nuc_comprehensive_dict[mut_9b]
                qry_triple_9b = string(ref_seq[pos_9b-2])*string(ref_seq[pos_9b-1])*qry_nuc_comprehensive_dict[mut_9b]
                ref_triple_context_9b = string(ref_seq[pos_9b-8])*string(ref_seq[pos_9b-7])*string(ref_seq[pos_9b-6])*string(ref_seq[pos_9b-5])*string(ref_seq[pos_9b-4])*string(ref_seq[pos_9b-3])*ref_triple_9b*string(ref_seq[pos_9b+1])*string(ref_seq[pos_9b+2])*string(ref_seq[pos_9b+3])*string(ref_seq[pos_9b+4])*string(ref_seq[pos_9b+5])*string(ref_seq[pos_9b+6])
                qry_triple_context_9b = string(ref_seq[pos_9b-8])*string(ref_seq[pos_9b-7])*string(ref_seq[pos_9b-6])*string(ref_seq[pos_9b-5])*string(ref_seq[pos_9b-4])*string(ref_seq[pos_9b-3])*qry_triple_9b*string(ref_seq[pos_9b+1])*string(ref_seq[pos_9b+2])*string(ref_seq[pos_9b+3])*string(ref_seq[pos_9b+4])*string(ref_seq[pos_9b+5])*string(ref_seq[pos_9b+6])
            end
            refAA_9b = AA_triplets[ref_triple_9b]
            qryAA_9b = AA_triplets[qry_triple_9b]
            AA_mut_9b = nuc2AA_ORF9b(mut_9b, refAA_9b, qryAA_9b)
            push!(AA_mut_set, AA_mut_9b)
            ref2qry_context_9b = ref_triple_context_9b*"-->"*qry_triple_context_9b
            all_nuc_to_AA_dict[mut_9b] = AA_mut_9b
            if refAA_9b == qryAA_9b && nuc_mut in N_9b_synonymous
                synonymous_nuc_to_AA_dict[mut_9b] = AA_mut_9b
                synonymous_nuc_to_context_dict[mut_9b] = ref2qry_context_9b
            end
            if refAA_9b ≠ qryAA_9b
                nonsynonymous_nuc_to_AA_dict[mut_9b] = AA_mut_9b
                nonsynonymous_nuc_to_context_dict[mut_9b] = ref2qry_context_9b
#                push!(nonsynonymous_nuc_muts, mut_9b)
            end
            all_nuc_to_context_dict[mut_9b] = ref2qry_context_9b
###################################
            for nuc_mut2_9b in muts
                mut2_9b = mixed2nuc(nuc_mut2_9b)
                pos2_9b = nuc_mut_int_comprehensive_dict[mut2_9b]
                if pos2_9b in rem9b
                    gene_number2_9b = 11
                    if mut_9b ≠ mut2_9b && nuc_to_AA_pos_9b(mut_9b) == nuc_to_AA_pos_9b(mut2_9b)
                        if nuc_codon_pos_dict_9b[pos_9b] == 1 && nuc_codon_pos_dict_9b[pos2_9b] == 2
                            ref_triple_9b = ref_nuc_comprehensive_dict[mut_9b]*ref_nuc_comprehensive_dict[mut2_9b]*string(ref_seq[pos_9b+2])
                            qry_triple_9b = qry_nuc_comprehensive_dict[mut_9b]*qry_nuc_comprehensive_dict[mut2_9b]*string(ref_seq[pos_9b+2])
                            ref_triple_context_9b = string(ref_seq[pos_9b-3])*string(ref_seq[pos_9b-2])*string(ref_seq[pos_9b-1])*ref_triple_9b*string(ref_seq[pos_9b+3])*string(ref_seq[pos_9b+4])*string(ref_seq[pos_9b+5])
                            qry_triple_context_9b = string(ref_seq[pos_9b-3])*string(ref_seq[pos_9b-2])*string(ref_seq[pos_9b-1])*qry_triple_9b*string(ref_seq[pos_9b+3])*string(ref_seq[pos_9b+4])*string(ref_seq[pos_9b+5])
                        elseif nuc_codon_pos_dict_9b[pos_9b] == 1 && nuc_codon_pos_dict_9b[pos2_9b] == 3
                            ref_triple_9b = ref_nuc_comprehensive_dict[mut_9b]*string(ref_seq[pos_9b+1])*ref_nuc_comprehensive_dict[mut2_9b]
                            qry_triple_9b = qry_nuc_comprehensive_dict[mut_9b]*string(ref_seq[pos_9b+1])*qry_nuc_comprehensive_dict[mut2_9b]
                            ref_triple_context_9b = string(ref_seq[pos_9b-3])*string(ref_seq[pos_9b-2])*string(ref_seq[pos_9b-1])*ref_triple_9b*string(ref_seq[pos_9b+3])*string(ref_seq[pos_9b+4])*string(ref_seq[pos_9b+5])
                            qry_triple_context_9b = string(ref_seq[pos_9b-3])*string(ref_seq[pos_9b-2])*string(ref_seq[pos_9b-1])*qry_triple_9b*string(ref_seq[pos_9b+3])*string(ref_seq[pos_9b+4])*string(ref_seq[pos_9b+5])
                        elseif nuc_codon_pos_dict_9b[pos_9b] == 2 && nuc_codon_pos_dict_9b[pos2_9b] == 3
                            ref_triple_9b = string(ref_seq[pos_9b-1])*ref_nuc_comprehensive_dict[mut_9b]*ref_nuc_comprehensive_dict[mut2_9b]
                            qry_triple_9b = string(ref_seq[pos_9b-1])*qry_nuc_comprehensive_dict[mut_9b]*qry_nuc_comprehensive_dict[mut2_9b]
                            ref_triple_context_9b = string(ref_seq[pos_9b-4])*string(ref_seq[pos_9b-3])*string(ref_seq[pos_9b-2])*ref_triple_9b*string(ref_seq[pos_9b+2])*string(ref_seq[pos_9b+3])*string(ref_seq[pos_9b+4])
                            qry_triple_context_9b = string(ref_seq[pos_9b-4])*string(ref_seq[pos_9b-3])*string(ref_seq[pos_9b-2])*qry_triple_9b*string(ref_seq[pos_9b+2])*string(ref_seq[pos_9b+3])*string(ref_seq[pos_9b+4])
                        elseif nuc_codon_pos_dict_9b[pos2_9b] == 1 && nuc_codon_pos_dict_9b[pos_9b] == 2
                            ref_triple_9b = ref_nuc_comprehensive_dict[mut2_9b]*ref_nuc_comprehensive_dict[mut_9b]*string(ref_seq[pos_9b+1])
                            qry_triple_9b = qry_nuc_comprehensive_dict[mut2_9b]*qry_nuc_comprehensive_dict[mut_9b]*string(ref_seq[pos_9b+1])
                            ref_triple_context_9b = string(ref_seq[pos_9b-4])*string(ref_seq[pos_9b-3])*string(ref_seq[pos_9b-2])*ref_triple_9b*string(ref_seq[pos_9b+2])*string(ref_seq[pos_9b+3])*string(ref_seq[pos_9b+4])
                            qry_triple_context_9b = string(ref_seq[pos_9b-4])*string(ref_seq[pos_9b-3])*string(ref_seq[pos_9b-2])*qry_triple_9b*string(ref_seq[pos_9b+2])*string(ref_seq[pos_9b+3])*string(ref_seq[pos_9b+4])
                        elseif nuc_codon_pos_dict_9b[pos2_9b] == 1 && nuc_codon_pos_dict_9b[pos_9b] == 3
                            ref_triple_9b = ref_nuc_comprehensive_dict[mut2_9b]*string(ref_seq[pos_9b-1])*ref_nuc_comprehensive_dict[mut_9b]
                            qry_triple_9b = qry_nuc_comprehensive_dict[mut2_9b]*string(ref_seq[pos_9b-1])*qry_nuc_comprehensive_dict[mut_9b]
                            ref_triple_context_9b = string(ref_seq[pos_9b-5])*string(ref_seq[pos_9b-4])*string(ref_seq[pos_9b-3])*ref_triple_9b*string(ref_seq[pos_9b+1])*string(ref_seq[pos_9b+2])*string(ref_seq[pos_9b+3])
                            qry_triple_context_9b = string(ref_seq[pos_9b-5])*string(ref_seq[pos_9b-4])*string(ref_seq[pos_9b-3])*qry_triple_9b*string(ref_seq[pos_9b+1])*string(ref_seq[pos_9b+2])*string(ref_seq[pos_9b+3])
                        elseif nuc_codon_pos_dict_9b[pos2_9b] == 2 && nuc_codon_pos_dict_9b[pos_9b] == 3
                            ref_triple_9b = string(ref_seq[pos_9b-2])*ref_nuc_comprehensive_dict[mut2_9b]*ref_nuc_comprehensive_dict[mut_9b]
                            qry_triple_9b = string(ref_seq[pos_9b-2])*qry_nuc_comprehensive_dict[mut2_9b]*qry_nuc_comprehensive_dict[mut_9b]
                        end
                        refAA2_9b = AA_triplets[ref_triple_9b]
                        qryAA2_9b = AA_triplets[qry_triple_9b]
                        AA_mut2_9b = nuc2AA_ORF9b(mut2_9b, refAA2_9b, qryAA2_9b)
                        push!(AA_mut_set, AA_mut2_9b)
                        ref2qry_context_9b = ref_triple_context_9b*"-->"*qry_triple_context_9b
                        if refAA2_9b == qryAA2_9b && nuc_mut2_9b in N_9b_synonymous 
                            synonymous_nuc_to_AA_dict[mut2_9b] = AA_mut2_9b
                            synonymous_nuc_to_context_dict[mut2_9b] = ref2qry_context_9b
                            synonymous_nuc_to_AA_dict[mut_9b] = AA_mut2_9b
                            synonymous_nuc_to_context_dict[mut_9b] = ref2qry_context_9b
                        else
                            nonsynonymous_nuc_to_AA_dict[mut2_9b] = AA_mut2_9b
                            nonsynonymous_nuc_to_context_dict[mut2_9b] = ref2qry_context_9b
                            nonsynonymous_nuc_to_AA_dict[mut_9b] = AA_mut2_9b
                            nonsynonymous_nuc_to_context_dict[mut_9b] = ref2qry_context_9b
                        end
                        all_nuc_to_context_dict[mut_9b] = ref2qry_context_9b
                        all_nuc_to_context_dict[mut2_9b] = ref2qry_context_9b
                    end
                end
            end
        end
    end
#################################################################################
    mut_vec_dict = Dict{String, Vector{String}}()
    for gene in gene_print_array
        mut_vec_dict[gene] = Vector{String}()
    end
    for mut in AA_mut_set
        jeen = aa_gene_comprehensive_dict[mut]
        mut_only = string(split(mut, ":")[2])
        push!(mut_vec_dict[jeen], mut_only) 
    end
    for gene in keys(mut_vec_dict)
        if !isempty(mut_vec_dict[gene])
            sort!(mut_vec_dict[gene], by = x -> x[2:end-1])
        end
    end
#####################################################################################################################################
    aasortkeyhere(m) = (gene_order_dict[aa_gene_comprehensive_dict[m]], aa_pos_comprehensive_dict[m])
    AA_sort = sort(collect(AA_mut_set), by = x -> aasortkeyhere(x))
    AA_sort2 = Vector{String}()
    for mut in AA_sort
        refAA = refAA_comprehensive_dict[mut]
        qryAA = qryAA_comprehensive_dict[mut]
        if !(refAA == qryAA)
            push!(AA_sort2, mut)
        end
    end
#    print("\n"^2)
#    println("##################### Amino Acid Mutations ######################")
#    for i in 1:length(AA_sort2)
#        mut = AA_sort2[i]
#        gene = aa_gene_comprehensive_dict[mut]
#        non_gene = string(split(mut, ":")[2])
#        if i == 1
#            print("               ", AA_sort2[i])
#        elseif i > 1
#            lastmut = AA_sort2[i-1]
#            last_gene = aa_gene_comprehensive_dict[lastmut]
#            last_non_gene = string(split(lastmut, ":")[2])
#            if gene == last_gene
#                print(", $(non_gene)")
#            else
#                println()
#                print("               $(mut)")
#            end
#        end
#    end
#    print("\n"^2)
#####################################################################################################################################
    all_nuc_to_AA_dict_sort = sort(collect(all_nuc_to_AA_dict), by = x -> nuc_mut_int_comprehensive_dict[x[1]])
    all_nuc_to_context_dict_sort = sort(collect(all_nuc_to_context_dict), by = x -> nuc_mut_int_comprehensive_dict[x[1]])
    nonsynonymous_nuc_to_AA_dict_sort = sort(collect(nonsynonymous_nuc_to_AA_dict), by = x -> nuc_mut_int_comprehensive_dict[x[1]])
    nonsynonymous_nuc_sort = sort(collect(keys(nonsynonymous_nuc_to_AA_dict)), by = x -> nuc_mut_int_comprehensive_dict[x])
    nonsynonymous_AA_sort = sort(collect(values(nonsynonymous_nuc_to_AA_dict)), by = x -> AA_gene_sortKey_2(x))
    nonsynonymous_nuc_to_context_dict_sort = sort(collect(nonsynonymous_nuc_to_context_dict), by = x -> nuc_mut_int_comprehensive_dict[x[1]])
    synonymous_nuc_to_AA_dict_sort = sort(collect(synonymous_nuc_to_AA_dict), by = x -> nuc_mut_int_comprehensive_dict[x[1]])
    synonymous_nuc_sort = sort(collect(keys(synonymous_nuc_to_AA_dict)), by = x -> nuc_mut_int_comprehensive_dict[x])
    synonymous_AA_sort = sort(collect(values(synonymous_nuc_to_AA_dict)), by = x -> AA_gene_sortKey_2(x))
    synonymous_nuc_to_context_dict_sort = sort(collect(synonymous_nuc_to_context_dict), by = x -> nuc_mut_int_comprehensive_dict[x[1]])
    noncoding_nuc_to_context_dict_sort = sort(collect(noncoding_nuc_to_context_dict), by = x -> nuc_mut_int_comprehensive_dict[x[1]])
    noncoding_nuc_vector_sort = sort(collect(noncoding_nuc_vector), by = x -> nuc_mut_int_comprehensive_dict[x])
    noncoding_to_noncoding_region_dict_sort = sort(collect(noncoding_to_noncoding_region_dict), by = x -> nuc_mut_int_comprehensive_dict[x[1]])
####################################################################################################################################
    nc_len = length(noncoding_to_noncoding_region_dict_sort)
#    println("NONCODING NUC MUTATIONS - Total:$(nc_len)")
    if !isempty(noncoding_to_noncoding_region_dict_sort)
        for i in 1:length(noncoding_to_noncoding_region_dict_sort)
            nc_nuc = noncoding_to_noncoding_region_dict_sort[i][1]
            nc_nuc_pad = rpad(nc_nuc, 7)
            nc_region = noncoding_to_noncoding_region_dict_sort[i][2]
            nc_region_len = length(nc_region)
            ncpad1 = (13 - nc_region_len)÷2
            ncpad12 = " "^ncpad1*nc_region
            nc_region_pad2 = rpad(ncpad12, 13)
            nc_context = noncoding_nuc_to_context_dict_sort[i][2]
            premut_context = ""
            postmut_context = ""
            context = split(nc_context, "-->")
            if !isempty(context)
                if length(context) >1
                    premut_context = split(nc_context, "-->")[1]
                    postmut_context = split(nc_context, "-->")[2]
                end
            end    
            postpad = lpad(postmut_context, 39)
#            println("$(nc_nuc_pad)|$(nc_region_pad2)|$(premut_context)")
#            println(postpad)
#                println(g, "$(nc_nuc_pad)|$(nc_region_pad2)|$(premut_context)")
#                println(g, postpad)
        end
#        println("#"^94); println() 
#            println(g, "#"^94); println(g)
    end
######################################################################################################
    total_syn = length(synonymous_nuc_to_AA_dict_sort)
#    println("SYNONYMOUS NUC MUTATIONS — Total:$(total_syn)")
    for i in 1:length(synonymous_nuc_to_AA_dict_sort)
        synnuc = synonymous_nuc_to_AA_dict_sort[i][1]
        synnuc_pad = rpad(synnuc, 7)
        synAA = synonymous_nuc_to_AA_dict_sort[i][2]
        AAlen = length(synAA)
        pad1 = (14 - AAlen)÷2
        pad12 = " "^pad1*synAA
        synAA_pad = rpad(pad12, 14)
        syncontext = synonymous_nuc_to_context_dict_sort[i][2]
        premut_context = split(syncontext, "-->")[1]
        postmut_context = split(syncontext, "-->")[2]
        postpad = lpad(postmut_context, 38)
#        println("$(synnuc_pad)|$(synAA_pad)|$(premut_context)")
#        println(postpad)
#            println(g, "$(synnuc_pad)|$(synAA_pad)|$(premut_context)")
#            println(g, postpad)
    end
#    println()
#################################################################
    total_syn = length(nonsynonymous_nuc_to_AA_dict_sort)
#    println("NONSYNONYMOUS NUC MUTATIONS — Total:$(total_syn)")
    for i in 1:length(nonsynonymous_nuc_to_AA_dict_sort)
        synnuc = nonsynonymous_nuc_to_AA_dict_sort[i][1]
        synnuc_pad = rpad(synnuc, 7)
        synAA = nonsynonymous_nuc_to_AA_dict_sort[i][2]
        AAlen = length(synAA)
        pad1 = (14 - AAlen)÷2
        pad12 = " "^pad1*synAA
        synAA_pad = rpad(pad12, 14)
        syncontext = nonsynonymous_nuc_to_context_dict_sort[i][2]
        premut_context = split(syncontext, "-->")[1]
        postmut_context = split(syncontext, "-->")[2]
        postpad = lpad(postmut_context, 38)
#        println("$(synnuc_pad)|$(synAA_pad)|$(premut_context)")
#        println(postpad)
#            println(g, "$(synnuc_pad)|$(synAA_pad)|$(premut_context)")
#            println(g, postpad)
    end
#    println()
###################################################################
    nonsynonymous_nuc_total = length(nonsynonymous_nuc_sort)
#    println("        Total Number of Non-synonymous Nuc Muts = $(nonsynonymous_nuc_total)")
    nonsynonymous_nuc_sort_join = join(nonsynonymous_nuc_sort, ", ")
#    println("################ Nonsynonymous Nuc Mutations ################")
#    println(nonsynonymous_nuc_sort_join)
#    print("\n"^1)
    for i in 1:length(nonsynonymous_nuc_sort)
#        println("               $(nonsynonymous_nuc_to_AA_dict_sort[i][1]) | $(nonsynonymous_nuc_to_AA_dict_sort[i][2])")
        premut_nucseq = split(nonsynonymous_nuc_to_context_dict_sort[i][2], "-->")[1]
        postmut_nucseq = split(nonsynonymous_nuc_to_context_dict_sort[i][2], "-->")[2]
#        println("                 $(premut_nucseq)")
#        println("                 $(postmut_nucseq)")
    end
#synonymous_nuc_to_context_dict[mut] = ref_triple_context*"-->"*qry_triple_context
    return synonymous_nuc_to_AA_dict_sort, nonsynonymous_nuc_to_AA_dict_sort, all_nuc_to_AA_dict_sort
end
###########################################################################################################################################################################
###########################################################################################################################################################################
function mixed_muts_to_AA(ref_pango::String, muts::String)
    mut_strings = muts_to_strings(muts)
    mixed_muts = mixed_nucs_filter(mut_strings)             ### New, 2025-1-26
    mixed_muts_regular = mixed_mut_to_regular_mut(mixed_muts)    ### New, 2025-1-26
    ct = 0
    total_mixed_muts = length(mixed_muts)
#    println("Total Mixed Nucs = $(total_mixed_muts)")
#    print("\n"^2)
    for i in 1:length(mixed_muts)
        if ct == 0
#            print(mixed_muts[i])
            ct = 1
        else
#            print(", ", mixed_muts[i])
        end
    end
    ct2 = 0
#    print("\n"^2)
    for i in 1:length(mixed_muts_regular)
        if ct2 == 0
#            print(mixed_muts_regular[i])
            ct2 = 1
#        else
#            print(", ", mixed_muts_regular[i])
        end
    end
#    println()
    syn_nuc_to_AA_dict_sort, nonsyn_nuc_to_AA_dict_sort, all_nuc_to_AA_dict_sort = nuc_to_AA(ref_pango, mixed_muts_regular)
    AA__AAprintlen__vec = []
    nonsynnuc__nonsynprintlen__vec = []
    for nuc___AA in nonsyn_nuc_to_AA_dict_sort
        nucmut = nuc___AA[1]
        nucmutlen = length(nucmut)
        AAmut = nuc___AA[2]
        AAmutlen = length(AAmut)
        push!(AA__AAprintlen__vec, [AAmut, AAmutlen])
        push!(nonsynnuc__nonsynprintlen__vec, [nucmut, nucmutlen])
    end
    aa_pad_vec = String[]
    nuc_pad_vec = String[]
    for i in 1:length(AA__AAprintlen__vec)
        aa = AA__AAprintlen__vec[i][1]
        nuc = nonsynnuc__nonsynprintlen__vec[i][1]
        aapad = AA__AAprintlen__vec[i][2]
        nucpad = nonsynnuc__nonsynprintlen__vec[i][2]
        pads = [nucpad, aapad]
        pad = maximum(pads)
        push!(aa_pad_vec, rpad(aa, pad))
        push!(nuc_pad_vec, rpad(nuc, pad))
    end
    aapad_join = join(aa_pad_vec, ", ")
    nucpad_join = join(nuc_pad_vec, ", ")
#    println("\n"^1)
#    println(aapad_join)
#    println(nucpad_join)
#    println("\n"^1)
    return syn_nuc_to_AA_dict_sort, nonsyn_nuc_to_AA_dict_sort, all_nuc_to_AA_dict_sort
end
######################################################################################################################################
function count_nuc_mut_types(mut_strings::Vector{String})
    mut_types_arr = ["TC", "TA", "TG", "CT", "CA", "CG", "AT", "AC", "AG", "GT", "GC", "GA"]
    mut_type_cts = Dict{String, Int}(mut_nuc_type=>0 for mut_nuc_type in mut_types_arr)
    for nuc_mut in mut_strings
        ref = ref_nuc_comprehensive_dict[nuc_mut]
        qry = ref_nuc_comprehensive_dict[nuc_mut]
        if ref == "T"
            if qry == "C"
                mut_type_cts["TC"] += 1
            elseif qry == "A"
                mut_type_cts["TA"] += 1
            elseif qry == "G"
                mut_type_cts["TG"] += 1
            end
        end
        if ref == "C"
            if qry == "T"
                mut_type_cts["CT"] += 1
            elseif qry == "A"
                mut_type_cts["CA"] += 1
            elseif qry == "G"
                mut_type_cts["CG"] += 1
            end
        end 
        if ref == "A"
            if qry == "T"
                mut_type_cts["AT"] += 1
            elseif qry == "C"
                mut_type_cts["AC"] += 1
            elseif qry == "G"
                mut_type_cts["AG"] += 1
            end
        end   
        if ref == "G"
            if qry == "T"
                mut_type_cts["GT"] += 1
            elseif qry == "C"
                mut_type_cts["GC"] += 1
            elseif qry == "A"
                mut_type_cts["GA"] += 1
            end
        end
    end
    mut_type_cts_sort_by_type = sort(collect(mut_type_cts), by = x -> x[1])
    mut_type_cts_sort_by_count = sort(collect(mut_type_cts), by = x -> x[2], rev=true)
    return mut_type_cts_sort_by_count
end
######################################################################################################################################
function AA_triple(pos::Int, rem0::BitSet, rem1::BitSet, rem2::BitSet, mut::String, ref_pango::String)
    ref_seq, refAA_ORF1a, refAA_ORF1b, refAA_S, refAA_ORF3a, refAA_E, refAA_M, refAA_ORF6, refAA_ORF7a, refAA_ORF7b, refAA_ORF8, refAA_N, refAA_ORF9b = get_ref_pango_nucseq_and_geneseqs(ref_pango)
    ref_triple = ""
    qry_triple = ""
    if pos in rem0 && pos%3 == 0
        ref_triple = ref_nuc_comprehensive_dict[mut]*string(ref_seq[pos+1])*string(ref_seq[pos+2])
        qry_triple = qry_nuc_comprehensive_dict[mut]*string(ref_seq[pos+1])*string(ref_seq[pos+2])
    elseif pos in rem1 && pos%3 == 1
        ref_triple = ref_nuc_comprehensive_dict[mut]*string(ref_seq[pos+1])*string(ref_seq[pos+2])
        qry_triple = qry_nuc_comprehensive_dict[mut]*string(ref_seq[pos+1])*string(ref_seq[pos+2])
    elseif pos in rem2 && pos%3 == 2
        ref_triple = ref_nuc_comprehensive_dict[mut]*string(ref_seq[pos+1])*string(ref_seq[pos+2])
        qry_triple = qry_nuc_comprehensive_dict[mut]*string(ref_seq[pos+1])*string(ref_seq[pos+2])
    elseif pos in rem0 && pos%3 == 1
        ref_triple = string(ref_seq[pos-1])*ref_nuc_comprehensive_dict[mut]*string(ref_seq[pos+1])
        qry_triple = string(ref_seq[pos-1])*qry_nuc_comprehensive_dict[mut]*string(ref_seq[pos+1])
    elseif pos in rem1 && pos%3 == 2
        ref_triple = string(ref_seq[pos-1])*ref_nuc_comprehensive_dict[mut]*string(ref_seq[pos+1])
        qry_triple = string(ref_seq[pos-1])*qry_nuc_comprehensive_dict[mut]*string(ref_seq[pos+1])
    elseif pos in rem2 && pos%3 == 0
        ref_triple = string(ref_seq[pos-1])*ref_nuc_comprehensive_dict[mut]*string(ref_seq[pos+1])
        qry_triple = string(ref_seq[pos-1])*qry_nuc_comprehensive_dict[mut]*string(ref_seq[pos+1])
    elseif pos in rem0 && pos%3 == 2
        ref_triple = string(ref_seq[pos-2])*string(ref_seq[pos-1])*ref_nuc_comprehensive_dict[mut]
        qry_triple = string(ref_seq[pos-2])*string(ref_seq[pos-1])*qry_nuc_comprehensive_dict[mut]
    elseif pos in rem1 && pos%3 == 0
        ref_triple = string(ref_seq[pos-2])*string(ref_seq[pos-1])*ref_nuc_comprehensive_dict[mut]
        qry_triple = string(ref_seq[pos-2])*string(ref_seq[pos-1])*qry_nuc_comprehensive_dict[mut]
    elseif pos in rem2 && pos%3 == 1
        ref_triple = string(ref_seq[pos-2])*string(ref_seq[pos-1])*ref_nuc_comprehensive_dict[mut]
        qry_triple = string(ref_seq[pos-2])*string(ref_seq[pos-1])*qry_nuc_comprehensive_dict[mut]
    end
    return ref_triple, qry_triple
end
######################################################################################################################################
### Fx: SIMPLER_nuc_to_AA (no context)
function SIMPLER_nuc_to_AA(ref_pango::String, muts::Vector{String})
    muts = filter(!isempty, muts)
    if !isempty(muts)
#        all_muts_sort = sort(collect(muts), by = x -> nuc_mut_int_comprehensive_dict[x])
        ref_seq, refAA_ORF1a, refAA_ORF1b, refAA_S, refAA_ORF3a, refAA_E, refAA_M, refAA_ORF6, refAA_ORF7a, refAA_ORF7b, refAA_ORF8, refAA_N, refAA_ORF9b = get_ref_pango_nucseq_and_geneseqs(ref_pango)
    ###############################################################################
        coding_ranges = BitSet([266:13467..., 13469:21555..., 21563:25384..., 25393:26220..., 26245:26472..., 26523:27191..., 27202:27387..., 27394:27755..., 27760:27887..., 27894:28259..., 28274:29533..., 28284:28577...])
        noncoding_ranges = BitSet([1:265..., 21556:21562..., 25385:25392..., 26221:26244..., 26473:26522..., 27192:27201..., 27388:27393..., 27888:27893..., 28260:28273..., 29534:29830...])
        coding_range_9b = BitSet([28284:28577...])
        gene_nuc_starts = Dict{Int, Int}(0=>263, 1=>13465, 2=>21560, 3=>25390, 4=>26242, 5=>26520, 6=>27199, 7=>27391, 8=>27753, 9=>27891, 10=>28271, 11=>28281)
        ref_AA_genes = Dict{Int, String}(0=>refAA_ORF1a, 1=>refAA_ORF1b, 2=>refAA_S, 3=>refAA_ORF3a, 4=>refAA_E, 5=>refAA_M, 6=>refAA_ORF6, 7=>refAA_ORF7a, 8=>refAA_ORF7b, 9=>refAA_ORF8, 10=>refAA_N, 11=>refAA_ORF9b)
        gene_strings = Dict{Int, String}(0=>"ORF1a", 1=>"ORF1b", 2=>"S", 3=>"ORF3a", 4=>"E", 5=>"M", 6=>"ORF6", 7=>"ORF7a", 8=>"ORF7b", 9=>"ORF8", 10=>"N", 11=>"ORF9b")
        nuc_gene_num = Dict{Int, Int}()
        nuc_gene_num_9b = Dict{Int, Int}()
        synonymous_nuc_to_AA_dict = Dict{String, String}()
        nonsynonymous_nuc_to_AA_dict = Dict{String, String}()
        all_nuc_to_AA_dict = Dict{String, String}()
        noncoding_range_dict = Dict{Vector{Int}, String}([1, 265]=>"5' UTR", [21556, 21562]=>"Spike TRS", [25385, 25392]=>"ORF3a TRS", [26221, 26234]=>"ORF3a-E UTR", [26235, 26244]=>"E TRS", [26473, 26522]=>"E-M UTR", [27192, 27201]=>"M-ORF6 UTR", [27388, 27393]=>"ORF7a TRS", [27888, 27893]=>"ORF8 TRS", [28260, 28273]=>"N/ORF9b TRS", [29534, 29830]=>"3' UTR", [29794, 29801]=>"Octanucleotide Motif")
################################################
        noncoding_nuc_vector = Vector{String}()
################################################ 
        gene_nuc_arr = [[266:13467...], [13469:21555...], [21563:25384...], [25393:26220...], [26245:26472...], [26523:27191...], [27202:27387...], [27394:27755...], [27760:27887...], [27894:28259...], [28274:29533...], [28284:28577...]]
        for i in 1:length(gene_nuc_arr)-1
            for nuc_pos in gene_nuc_arr[i]
                nuc_gene_num[nuc_pos] = i-1
            end
        end
        for nuc_pos in gene_nuc_arr[end]
            nuc_gene_num_9b[nuc_pos] = 11
        end
        rem0_gene = [5, 8, 9, 11]
        rem1_gene = [1, 3, 4, 6, 7]
        rem2_gene = [0, 2, 10]
        rem0 = BitSet([26523:27191..., 27760:27887..., 27894:28259..., 28284:28577...])
        rem1 = BitSet([13469:21555..., 25393:26220..., 26245:26472..., 27202:27387..., 27394:27755...])
        rem2 = BitSet([266:13467..., 21563:25384..., 28274:29533...])
        rem9b = BitSet([28284:28577...])
        rem7ab = BitSet([27756:27759...])

        gene_num(nuc_mut) = nuc_gene_num[nuc_mut_int_comprehensive_dict[nuc_mut]] ## Fx ##
        nuc_to_AA_pos(nuc_mut) = string((nuc_mut_int_comprehensive_dict[nuc_mut] - gene_nuc_starts[gene_num(nuc_mut)])÷3) ## Fx ##
        nuc_to_AA_pos_9b(nuc_mut) = string((nuc_mut_int_comprehensive_dict[nuc_mut] - 28281)÷3) ## Fx ##
        nuc2AA_ORF1a(nuc_mut, refAA, qryAA) = gene_strings[gene_num(nuc_mut)]*":"*refAA*nuc_to_AA_pos(nuc_mut)*qryAA ## Fx ##
        nuc2AA_ORF9b(nuc_mut, refAA, qryAA) = "ORF9b:"*refAA*nuc_to_AA_pos_9b(nuc_mut)*qryAA
        nuc_codon_pos_dict = Dict{Int, Int}()
        for nuc_pos in coding_ranges
            gene_number = nuc_gene_num[nuc_pos]
            gene_start = gene_nuc_starts[gene_number]
            codon_num = (nuc_pos-gene_start)%3 + 1
            nuc_codon_pos_dict[nuc_pos] = codon_num
        end
        nuc_codon_pos_dict_9b = Dict{Int, Int}()
        for nuc_pos in coding_range_9b
            gene_number = 11
            gene_start = gene_nuc_starts[gene_number]
            codon_num = (nuc_pos-gene_start)%3 + 1
            nuc_codon_pos_dict_9b[nuc_pos] = codon_num
        end    
        N3_syn = ["TCT", "TCC", "TCA", "TCG", "CTT", "CTC", "CTA", "CTG", "CCT", "CCC", "CCA", "CCG", "CGT", "CGC", "CGA", "CGG", "ACT", "ACC", "ACA", "ACG", "GTT", "GTC", "GTA", "GTG", "GCT", "GCC", "GCA", "GCG", "GGT", "GGC", "GGA", "GGG"]
        N3_tv = ["TTT", "TTC", "TTA", "TTG", "TAT", "TAC", "TAA", "TAG", "AAT", "AAC", "AAA", "AAG", "AGT", "AGC", "AGA", "AGG", "GAT", "GAC", "GAA", "GAG"]
        for nuc_mut in muts
            if !isempty(muts)
                mut = mixed2nuc(nuc_mut)
                if ',' in mut
                    mut1 = string(split(mut, ",")[1])
                    mut2 = string(split(mut, ",")[2])
                    push!(muts, mut1)
                    push!(muts, mut2)
                    filter!(x -> !(length(x)>6), muts)
                end
            end
        end
        coding_ranges = BitSet([266:13467..., 13468:21555..., 21563:25384..., 25393:26220..., 26245:26472..., 26523:27191..., 27202:27387..., 27394:27755..., 27760:27887..., 27894:28259..., 28274:29533...])
        N_9b_synonymous = Set(["C28379A", "C28394A", "T28406C", "C28475A", "C28535A", "A28547C"])
        AA_mut_set = Set{String}()
        AA_mut = ""
        for nuc_mut in muts
            pos = nuc_mut_int_comprehensive_dict[nuc_mut]
            if pos in coding_ranges
                mut = mixed2nuc(nuc_mut)  
                gene_number = gene_num(mut)
                ref_triple = ""
                qry_triple = ""
                if nuc_codon_pos_dict[pos] == 1
                    ref_triple = ref_nuc_comprehensive_dict[mut]*string(ref_seq[pos+1])*string(ref_seq[pos+2])
                    qry_triple = qry_nuc_comprehensive_dict[mut]*string(ref_seq[pos+1])*string(ref_seq[pos+2])
                elseif nuc_codon_pos_dict[pos] == 2
                    ref_triple = string(ref_seq[pos-1])*ref_nuc_comprehensive_dict[mut]*string(ref_seq[pos+1])
                    qry_triple = string(ref_seq[pos-1])*qry_nuc_comprehensive_dict[mut]*string(ref_seq[pos+1])
                elseif nuc_codon_pos_dict[pos] == 3
                    ref_triple = string(ref_seq[pos-2])*string(ref_seq[pos-1])*ref_nuc_comprehensive_dict[mut]
                    qry_triple = string(ref_seq[pos-2])*string(ref_seq[pos-1])*qry_nuc_comprehensive_dict[mut]
                end
                refAA = AA_triplets[ref_triple]
                qryAA = AA_triplets[qry_triple]
                AA_mut = nuc2AA_ORF1a(mut, refAA, qryAA)
                push!(AA_mut_set, AA_mut)
                all_nuc_to_AA_dict[mut] = AA_mut
                if refAA == qryAA && !(pos in rem9b)
                    synonymous_nuc_to_AA_dict[mut] = AA_mut
                elseif refAA == qryAA && pos in rem9b && mut in N_9b_synonymous
                    synonymous_nuc_to_AA_dict[mut] = AA_mut
                else
                    nonsynonymous_nuc_to_AA_dict[mut] = AA_mut
#                    push!(nonsynonymous_nuc_muts, mut)
                end
###################################
                for nuc_mut2 in muts
                    mut2 = mixed2nuc(nuc_mut2)
                    pos2 = nuc_mut_int_comprehensive_dict[mut2]
                    if pos2 in coding_ranges
                        gene_number2 = gene_num(mut2)
                        if mut ≠ mut2 && gene_number == gene_number2 && nuc_to_AA_pos(mut) == nuc_to_AA_pos(mut2)
                            if nuc_codon_pos_dict[pos] == 1 && nuc_codon_pos_dict[pos2] == 2
                                ref_triple = ref_nuc_comprehensive_dict[mut]*ref_nuc_comprehensive_dict[mut2]*string(ref_seq[pos+2])
                                qry_triple = qry_nuc_comprehensive_dict[mut]*qry_nuc_comprehensive_dict[mut2]*string(ref_seq[pos+2])
                            elseif nuc_codon_pos_dict[pos] == 1 && nuc_codon_pos_dict[pos2] == 3
                                ref_triple = ref_nuc_comprehensive_dict[mut]*string(ref_seq[pos+1])*ref_nuc_comprehensive_dict[mut2]
                                qry_triple = qry_nuc_comprehensive_dict[mut]*string(ref_seq[pos+1])*qry_nuc_comprehensive_dict[mut2]
                            elseif nuc_codon_pos_dict[pos] == 2 && nuc_codon_pos_dict[pos2] == 3
                                ref_triple = string(ref_seq[pos-1])*ref_nuc_comprehensive_dict[mut]*ref_nuc_comprehensive_dict[mut2]
                                qry_triple = string(ref_seq[pos-1])*qry_nuc_comprehensive_dict[mut]*qry_nuc_comprehensive_dict[mut2]
                            elseif nuc_codon_pos_dict[pos2] == 1 && nuc_codon_pos_dict[pos] == 2
                                ref_triple = ref_nuc_comprehensive_dict[mut2]*ref_nuc_comprehensive_dict[mut]*string(ref_seq[pos+1])
                                qry_triple = qry_nuc_comprehensive_dict[mut2]*qry_nuc_comprehensive_dict[mut]*string(ref_seq[pos+1])
                            elseif nuc_codon_pos_dict[pos2] == 1 && nuc_codon_pos_dict[pos] == 3
                                ref_triple = ref_nuc_comprehensive_dict[mut2]*string(ref_seq[pos-1])*ref_nuc_comprehensive_dict[mut]
                                qry_triple = qry_nuc_comprehensive_dict[mut2]*string(ref_seq[pos-1])*qry_nuc_comprehensive_dict[mut]
                            elseif nuc_codon_pos_dict[pos2] == 2 && nuc_codon_pos_dict[pos] == 3
                                ref_triple = string(ref_seq[pos-2])*ref_nuc_comprehensive_dict[mut2]*ref_nuc_comprehensive_dict[mut]
                                qry_triple = string(ref_seq[pos-2])*qry_nuc_comprehensive_dict[mut2]*qry_nuc_comprehensive_dict[mut]
                            end
                            refAA2 = AA_triplets[ref_triple]
                            qryAA2 = AA_triplets[qry_triple]
                            AA_mut2 = nuc2AA_ORF1a(mut2, refAA2, qryAA2)
                            push!(AA_mut_set, AA_mut2)
                            delete!(AA_mut_set, AA_mut)
                            all_nuc_to_AA_dict[mut2] = AA_mut2
                            all_nuc_to_AA_dict[mut] = AA_mut2
                            if refAA2 == qryAA2 && !(pos2 in rem9b)
                                synonymous_nuc_to_AA_dict[mut2] = AA_mut2
                            else
                                nonsynonymous_nuc_to_AA_dict[mut2] = AA_mut2
                                nonsynonymous_nuc_to_AA_dict[mut] = AA_mut2
                            end
                        end
                    end
                end
            elseif !isempty(nuc_mut)
                qry_nuc = qry_nuc_comprehensive_dict[nuc_mut]
                for (start_end, place) in noncoding_range_dict
                    frst = start_end[1]
                    last = start_end[2]
                    if pos ≥ frst && pos ≤ last
                        mut_vec = [nuc_mut, place]
                        push!(noncoding_nuc_vector, nuc_mut)
                    end
                end
            end
        end
#########################################################################################################
        for nuc_mut in muts
            pos_9b = nuc_mut_int_comprehensive_dict[nuc_mut]
            if pos_9b in rem9b
                mut_9b = mixed2nuc(nuc_mut)
                pos_9b = nuc_mut_int_comprehensive_dict[mut_9b]   
                gene_number_9b = 11
                ref_triple_9b = ""
                qry_triple_9b = ""
                if nuc_codon_pos_dict_9b[pos_9b] == 1
                    ref_triple_9b = ref_nuc_comprehensive_dict[mut_9b]*string(ref_seq[pos_9b+1])*string(ref_seq[pos_9b+2])
                    qry_triple_9b = qry_nuc_comprehensive_dict[mut_9b]*string(ref_seq[pos_9b+1])*string(ref_seq[pos_9b+2])
                elseif nuc_codon_pos_dict_9b[pos_9b] == 2
                    ref_triple_9b = string(ref_seq[pos_9b-1])*ref_nuc_comprehensive_dict[mut_9b]*string(ref_seq[pos_9b+1])
                    qry_triple_9b = string(ref_seq[pos_9b-1])*qry_nuc_comprehensive_dict[mut_9b]*string(ref_seq[pos_9b+1])
                elseif nuc_codon_pos_dict_9b[pos_9b] == 3
                    ref_triple_9b = string(ref_seq[pos_9b-2])*string(ref_seq[pos_9b-1])*ref_nuc_comprehensive_dict[mut_9b]
                    qry_triple_9b = string(ref_seq[pos_9b-2])*string(ref_seq[pos_9b-1])*qry_nuc_comprehensive_dict[mut_9b]
                end
                refAA_9b = AA_triplets[ref_triple_9b]
                qryAA_9b = AA_triplets[qry_triple_9b]
                AA_mut_9b = nuc2AA_ORF9b(mut_9b, refAA_9b, qryAA_9b)
                push!(AA_mut_set, AA_mut_9b)
                all_nuc_to_AA_dict[mut_9b] = AA_mut_9b
                if refAA_9b == qryAA_9b && nuc_mut in N_9b_synonymous
                    synonymous_nuc_to_AA_dict[mut_9b] = AA_mut_9b
                end
                if refAA_9b ≠ qryAA_9b
                    nonsynonymous_nuc_to_AA_dict[mut_9b] = AA_mut_9b
#                    push!(nonsynonymous_nuc_muts, mut_9b)
                end
###################################
                for nuc_mut2_9b in muts
                    mut2_9b = mixed2nuc(nuc_mut2_9b)
                    pos2_9b = nuc_mut_int_comprehensive_dict[mut2_9b]
                    if pos2_9b in rem9b
                        gene_number2_9b = 11
                        if mut_9b ≠ mut2_9b && nuc_to_AA_pos_9b(mut_9b) == nuc_to_AA_pos_9b(mut2_9b)
                            if nuc_codon_pos_dict_9b[pos_9b] == 1 && nuc_codon_pos_dict_9b[pos2_9b] == 2
                                ref_triple_9b = ref_nuc_comprehensive_dict[mut_9b]*ref_nuc_comprehensive_dict[mut2_9b]*string(ref_seq[pos_9b+2])
                                qry_triple_9b = qry_nuc_comprehensive_dict[mut_9b]*qry_nuc_comprehensive_dict[mut2_9b]*string(ref_seq[pos_9b+2])
                            elseif nuc_codon_pos_dict_9b[pos_9b] == 1 && nuc_codon_pos_dict_9b[pos2_9b] == 3
                                ref_triple_9b = ref_nuc_comprehensive_dict[mut_9b]*string(ref_seq[pos_9b+1])*ref_nuc_comprehensive_dict[mut2_9b]
                                qry_triple_9b = qry_nuc_comprehensive_dict[mut_9b]*string(ref_seq[pos_9b+1])*qry_nuc_comprehensive_dict[mut2_9b]
                            elseif nuc_codon_pos_dict_9b[pos_9b] == 2 && nuc_codon_pos_dict_9b[pos2_9b] == 3
                                ref_triple_9b = string(ref_seq[pos_9b-1])*ref_nuc_comprehensive_dict[mut_9b]*ref_nuc_comprehensive_dict[mut2_9b]
                                qry_triple_9b = string(ref_seq[pos_9b-1])*qry_nuc_comprehensive_dict[mut_9b]*qry_nuc_comprehensive_dict[mut2_9b]
                            elseif nuc_codon_pos_dict_9b[pos2_9b] == 1 && nuc_codon_pos_dict_9b[pos_9b] == 2
                                ref_triple_9b = ref_nuc_comprehensive_dict[mut2_9b]*ref_nuc_comprehensive_dict[mut_9b]*string(ref_seq[pos_9b+1])
                                qry_triple_9b = qry_nuc_comprehensive_dict[mut2_9b]*qry_nuc_comprehensive_dict[mut_9b]*string(ref_seq[pos_9b+1])
                            elseif nuc_codon_pos_dict_9b[pos2_9b] == 1 && nuc_codon_pos_dict_9b[pos_9b] == 3
                                ref_triple_9b = ref_nuc_comprehensive_dict[mut2_9b]*string(ref_seq[pos_9b-1])*ref_nuc_comprehensive_dict[mut_9b]
                                qry_triple_9b = qry_nuc_comprehensive_dict[mut2_9b]*string(ref_seq[pos_9b-1])*qry_nuc_comprehensive_dict[mut_9b]
                            elseif nuc_codon_pos_dict_9b[pos2_9b] == 2 && nuc_codon_pos_dict_9b[pos_9b] == 3
                                ref_triple_9b = string(ref_seq[pos_9b-2])*ref_nuc_comprehensive_dict[mut2_9b]*ref_nuc_comprehensive_dict[mut_9b]
                                qry_triple_9b = string(ref_seq[pos_9b-2])*qry_nuc_comprehensive_dict[mut2_9b]*qry_nuc_comprehensive_dict[mut_9b]
                            end
                            refAA2_9b = AA_triplets[ref_triple_9b]
                            qryAA2_9b = AA_triplets[qry_triple_9b]
                            AA_mut2_9b = nuc2AA_ORF9b(mut2_9b, refAA2_9b, qryAA2_9b)
                            push!(AA_mut_set, AA_mut2_9b)
                            delete!(AA_mut_set, AA_mut_9b)
                            if refAA2_9b == qryAA2_9b && nuc_mut2_9b in N_9b_synonymous 
                                synonymous_nuc_to_AA_dict[mut2_9b] = AA_mut2_9b
                                synonymous_nuc_to_AA_dict[mut_9b] = AA_mut2_9b
                            else
                                nonsynonymous_nuc_to_AA_dict[mut2_9b] = AA_mut2_9b
                                nonsynonymous_nuc_to_AA_dict[mut_9b] = AA_mut2_9b
                            end
                        end
                    end
                end
            end
        end
#####################################################################################################################################
        AA_sort = sort(collect(AA_mut_set), by = x -> AA_order_key(x))
        AA_sort2 = Vector{String}()
        for aa in AA_sort
            if !(refAA_comprehensive_dict[aa] == qryAA_comprehensive_dict)
                push!(AA_sort2, aa)
            end
        end
#        print("\n"^1)
#        ct = 0
#        println("############# AA Mutations #############")
#        for i in 1:length(AA_sort2)
#            mut = AA_sort2[i]
#            gene = string(split(mut, ":")[1])
#            non_gene = string(split(mut, ":")[2])
#            if i == 1
#                print("        ", AA_sort2[i])
#            elseif i > 1
#                lastmut = AA_sort2[i-1]
#                last_gene = string(split(lastmut, ":")[1])
#                last_non_gene = string(split(lastmut, ":")[2])
#                if gene == last_gene
#                    print(", $(non_gene)")
#                else
#                    print("        $(mut)")
#                end
#            end
#        end
#####################################################################################################################################
        all_nuc_to_AA_dict_sort = sort(collect(all_nuc_to_AA_dict), by = x -> nuc_mut_int_comprehensive_dict[x[1]])
        nonsynonymous_nuc_to_AA_dict_sort = sort(collect(nonsynonymous_nuc_to_AA_dict), by = x -> nuc_mut_int_comprehensive_dict[x[1]])
        nonsynonymous_nuc_sort = sort(collect(keys(nonsynonymous_nuc_to_AA_dict)), by = x -> nuc_mut_int_comprehensive_dict[x])
        nonsynonymous_AA_sort = sort(collect(values(nonsynonymous_nuc_to_AA_dict)), by = x -> AA_gene_sortKey_2(x))
        synonymous_nuc_to_AA_dict_sort = sort(collect(synonymous_nuc_to_AA_dict), by = x -> nuc_mut_int_comprehensive_dict[x[1]])
        synonymous_nuc_sort = sort(collect(keys(synonymous_nuc_to_AA_dict)), by = x -> nuc_mut_int_comprehensive_dict[x])
        synonymous_AA_sort = sort(collect(values(synonymous_nuc_to_AA_dict)), by = x -> AA_gene_sortKey_2(x))
        noncoding_nuc_vector_sort = sort(collect(noncoding_nuc_vector), by = x -> nuc_mut_int_comprehensive_dict[x])
        synonymous_nuc_total = length(synonymous_nuc_sort)
        for i in 1:length(synonymous_nuc_sort)
            nucpad = rpad("$(synonymous_nuc_to_AA_dict_sort[i][1])", 10)
        end
#        return synonymous_nuc_to_AA_dict, nonsynonymous_nuc_to_AA_dict_sort, all_nuc_to_AA_dict_sort
        return synonymous_nuc_to_AA_dict_sort, nonsynonymous_nuc_to_AA_dict_sort, all_nuc_to_AA_dict_sort
    end
    if isempty(muts)
        aa = Vector{Pair{String, String}}()
        bb = Vector{Pair{String, String}}()
        cc = Vector{Pair{String, String}}()
        return aa, bb, cc
    end 
end
###########################################################################################################################################################################
############################################################################################################################################################################
############################################################################################################################################################################
function pango_minus_1_fx(pango::String)
    if '.' in pango
        dot_ct = count(".", pango)
        dotsplits = split(pango, ".")
        minus_1 = join(dotsplits(1:dot_ct-1), ".")
        return minus_1
    else
        return pango
    end
end
############################################################################################################################################################################
### Fx: SIMPLER_syn_noncoding_nonsyn_nuc (no context)
function SIMPLER_syn_noncoding_nuc(ref_pango::String, muts::Set{String})
    B_1_1_list = ["B.1.1.53", "B.1.1.273"]
    if ref_pango in B_1_1_list
        ref_pango = "B.1.1"
    end
    if ref_pango == "XBB.1.5.82"
         ref_pango = "XBB.1.5"
    end
    if !isempty(muts)
#        all_muts_sort = sort(collect(muts), by = x -> nuc_mut_int_comprehensive_dict[x])
        ref_seq, refAA_ORF1a, refAA_ORF1b, refAA_S, refAA_ORF3a, refAA_E, refAA_M, refAA_ORF6, refAA_ORF7a, refAA_ORF7b, refAA_ORF8, refAA_N, refAA_ORF9b = get_ref_pango_nucseq_and_geneseqs(ref_pango)
###################################################################################################################################
        coding_ranges = BitSet([266:13467..., 13469:21555..., 21563:25384..., 25393:26220..., 26245:26472..., 26523:27191..., 27202:27387..., 27394:27755..., 27760:27887..., 27894:28259..., 28274:29533..., 28284:28577...])
        noncoding_ranges = BitSet([1:265..., 21556:21562..., 25385:25392..., 26221:26244..., 26473:26522..., 27192:27201..., 27388:27393..., 27888:27893..., 28260:28273..., 29534:29830...])
        coding_range_9b = BitSet([28284:28577...])
        gene_nuc_starts = Dict{Int, Int}(0=>263, 1=>13465, 2=>21560, 3=>25390, 4=>26242, 5=>26520, 6=>27199, 7=>27391, 8=>27753, 9=>27891, 10=>28271, 11=>28281)
        ref_AA_genes = Dict{Int, String}(0=>refAA_ORF1a, 1=>refAA_ORF1b, 2=>refAA_S, 3=>refAA_ORF3a, 4=>refAA_E, 5=>refAA_M, 6=>refAA_ORF6, 7=>refAA_ORF7a, 8=>refAA_ORF7b, 9=>refAA_ORF8, 10=>refAA_N, 11=>refAA_ORF9b)
        gene_strings = Dict{Int, String}(0=>"ORF1a", 1=>"ORF1b", 2=>"S", 3=>"ORF3a", 4=>"E", 5=>"M", 6=>"ORF6", 7=>"ORF7a", 8=>"ORF7b", 9=>"ORF8", 10=>"N", 11=>"ORF9b")
        nuc_gene_num = Dict{Int, Int}()
        nuc_gene_num_9b = Dict{Int, Int}()
        synonymous_nuc_to_AA_dict = Dict{String, String}()
        noncoding_range_dict = Dict{Vector{Int}, String}([1, 265]=>"5' UTR", [21556, 21562]=>"Spike TRS", [25385, 25392]=>"ORF3a TRS", [26221, 26234]=>"ORF3a-E UTR", [26235, 26244]=>"E TRS", [26473, 26522]=>"E-M UTR", [27192, 27201]=>"M-ORF6 UTR", [27388, 27393]=>"ORF7a TRS", [27888, 27893]=>"ORF8 TRS", [28260, 28273]=>"N/ORF9b TRS", [29534, 29830]=>"3' UTR", [29794, 29801]=>"Octanucleotide Motif")
################################################
        noncoding_nuc_vector = Vector{String}()
################################################ 
        gene_nuc_arr = [[266:13467...], [13469:21555...], [21563:25384...], [25393:26220...], [26245:26472...], [26523:27191...], [27202:27387...], [27394:27755...], [27760:27887...], [27894:28259...], [28274:29533...], [28284:28577...]]
        for i in 1:length(gene_nuc_arr)-1
            for nuc_pos in gene_nuc_arr[i]
                nuc_gene_num[nuc_pos] = i-1
            end
        end
        for nuc_pos in gene_nuc_arr[end]
            nuc_gene_num_9b[nuc_pos] = 11
        end
        rem0_gene = [5, 8, 9, 11]
        rem1_gene = [1, 3, 4, 6, 7]
        rem2_gene = [0, 2, 10]
        rem0 = BitSet([26523:27191..., 27760:27887..., 27894:28259..., 28284:28577...])
        rem1 = BitSet([13469:21555..., 25393:26220..., 26245:26472..., 27202:27387..., 27394:27755...])
        rem2 = BitSet([266:13467..., 21563:25384..., 28274:29533...])
        rem9b = BitSet([28284:28577...])
        rem7ab = BitSet([27756:27759...])

        gene_num(nuc_mut) = nuc_gene_num[nuc_mut_int_comprehensive_dict[nuc_mut]] ## Fx ##
        nuc_to_AA_pos(nuc_mut) = string((nuc_mut_int_comprehensive_dict[nuc_mut] - gene_nuc_starts[gene_num(nuc_mut)])÷3) ## Fx ##
        nuc_to_AA_pos_9b(nuc_mut) = string((nuc_mut_int_comprehensive_dict[nuc_mut] - 28281)÷3) ## Fx ##
        nuc2AA_ORF1a(nuc_mut, refAA, qryAA) = gene_strings[gene_num(nuc_mut)]*":"*refAA*nuc_to_AA_pos(nuc_mut)*qryAA ## Fx ##
        nuc2AA_ORF9b(nuc_mut, refAA, qryAA) = "ORF9b:"*refAA*nuc_to_AA_pos_9b(nuc_mut)*qryAA
        nuc_codon_pos_dict = Dict{Int, Int}()
        for nuc_pos in coding_ranges
            gene_number = nuc_gene_num[nuc_pos]
            gene_start = gene_nuc_starts[gene_number]
            codon_num = (nuc_pos-gene_start)%3 + 1
            nuc_codon_pos_dict[nuc_pos] = codon_num
        end
        nuc_codon_pos_dict_9b = Dict{Int, Int}()
        for nuc_pos in coding_range_9b
            gene_number = 11
            gene_start = gene_nuc_starts[gene_number]
            codon_num = (nuc_pos-gene_start)%3 + 1
            nuc_codon_pos_dict_9b[nuc_pos] = codon_num
        end    
        for nuc_mut in muts
            if !isempty(muts)
                mut = mixed2nuc(nuc_mut)
                if ',' in mut
                    mut1 = string(split(mut, ",")[1])
                    mut2 = string(split(mut, ",")[2])
                    push!(muts, mut1)
                    push!(muts, mut2)
                    filter!(x -> !(length(x)>6), muts)
                end
            end
        end
        coding_ranges = BitSet([266:13467..., 13468:21555..., 21563:25384..., 25393:26220..., 26245:26472..., 26523:27191..., 27202:27387..., 27394:27755..., 27760:27887..., 27894:28259..., 28274:29533...])
        N_9b_synonymous = Set(["C28379A", "C28394A", "T28406C", "C28475A", "C28535A", "A28547C"])
        AA_mut = ""
        for nuc_mut in muts
            if !isempty(muts) && !isempty(ref_seq) && nuc_mut ≠ ""
                pos = nuc_mut_int_comprehensive_dict[nuc_mut]
                if pos in coding_ranges
                    mut = mixed2nuc(nuc_mut)
                    if isempty(mut)
                        println(nuc_mut)
                        println(mut)
                        println("$(pango)")
                    end
                    try
                        nuc_mut_int_comprehensive_dict[mut]
                    catch e
                        println("Problematic mutation string: ", repr(mut))
                        println("Length: ", length(mut))
                        println("Extracted substring: ", repr(mut[2:end-1]))
                        rethrow(e)
                    end
                    gene_number = nuc_gene_num[pos]
                    ref_triple = ""
                    qry_triple = ""
                    if nuc_codon_pos_dict[pos] == 1
                        ref_triple = ref_nuc_comprehensive_dict[mut]*string(ref_seq[pos+1])*string(ref_seq[pos+2])
                        qry_triple = qry_nuc_comprehensive_dict[mut]*string(ref_seq[pos+1])*string(ref_seq[pos+2])
                    elseif nuc_codon_pos_dict[pos] == 2
                        ref_triple = string(ref_seq[pos-1])*ref_nuc_comprehensive_dict[mut]*string(ref_seq[pos+1])
                        qry_triple = string(ref_seq[pos-1])*qry_nuc_comprehensive_dict[mut]*string(ref_seq[pos+1])
                    elseif nuc_codon_pos_dict[pos] == 3
                        ref_triple = string(ref_seq[pos-2])*string(ref_seq[pos-1])*ref_nuc_comprehensive_dict[mut]
                        qry_triple = string(ref_seq[pos-2])*string(ref_seq[pos-1])*qry_nuc_comprehensive_dict[mut]
                    end
                    refAA = AA_triplets[ref_triple]
                    qryAA = AA_triplets[qry_triple]
                    AA_mut = nuc2AA_ORF1a(mut, refAA, qryAA)
                    if refAA == qryAA && !(pos in rem9b)
                        synonymous_nuc_to_AA_dict[mut] = AA_mut
                    elseif refAA == qryAA && pos in rem9b && mut in N_9b_synonymous
                        synonymous_nuc_to_AA_dict[mut] = AA_mut
                    end
###################################
                    for nuc_mut2 in muts
                        mut2 = mixed2nuc(nuc_mut2)
                        pos2 = nuc_mut_int_comprehensive_dict[mut2]
                        if pos2 in coding_ranges
                            gene_number2 = gene_num(mut2)
                            if mut ≠ mut2 && gene_number == gene_number2 && nuc_to_AA_pos(mut) == nuc_to_AA_pos(mut2)
                                if nuc_codon_pos_dict[pos] == 1 && nuc_codon_pos_dict[pos2] == 2
                                    ref_triple = ref_nuc_comprehensive_dict[mut]*ref_nuc_comprehensive_dict[mut2]*string(ref_seq[pos+2])
                                    qry_triple = qry_nuc_comprehensive_dict[mut]*qry_nuc_comprehensive_dict[mut2]*string(ref_seq[pos+2])
                                elseif nuc_codon_pos_dict[pos] == 1 && nuc_codon_pos_dict[pos2] == 3
                                    ref_triple = ref_nuc_comprehensive_dict[mut]*string(ref_seq[pos+1])*ref_nuc_comprehensive_dict[mut2]
                                    qry_triple = qry_nuc_comprehensive_dict[mut]*string(ref_seq[pos+1])*qry_nuc_comprehensive_dict[mut2]
                                elseif nuc_codon_pos_dict[pos] == 2 && nuc_codon_pos_dict[pos2] == 3
                                    ref_triple = string(ref_seq[pos-1])*ref_nuc_comprehensive_dict[mut]*ref_nuc_comprehensive_dict[mut2]
                                    qry_triple = string(ref_seq[pos-1])*qry_nuc_comprehensive_dict[mut]*qry_nuc_comprehensive_dict[mut2]
                                elseif nuc_codon_pos_dict[pos2] == 1 && nuc_codon_pos_dict[pos] == 2
                                    ref_triple = ref_nuc_comprehensive_dict[mut2]*ref_nuc_comprehensive_dict[mut]*string(ref_seq[pos+1])
                                    qry_triple = qry_nuc_comprehensive_dict[mut2]*qry_nuc_comprehensive_dict[mut]*string(ref_seq[pos+1])
                                elseif nuc_codon_pos_dict[pos2] == 1 && nuc_codon_pos_dict[pos] == 3
                                    ref_triple = ref_nuc_comprehensive_dict[mut2]*string(ref_seq[pos-1])*ref_nuc_comprehensive_dict[mut]
                                    qry_triple = qry_nuc_comprehensive_dict[mut2]*string(ref_seq[pos-1])*qry_nuc_comprehensive_dict[mut]
                                elseif nuc_codon_pos_dict[pos2] == 2 && nuc_codon_pos_dict[pos] == 3
                                    ref_triple = string(ref_seq[pos-2])*ref_nuc_comprehensive_dict[mut2]*ref_nuc_comprehensive_dict[mut]
                                    qry_triple = string(ref_seq[pos-2])*qry_nuc_comprehensive_dict[mut2]*qry_nuc_comprehensive_dict[mut]
                                end
                                refAA2 = AA_triplets[ref_triple]
                                qryAA2 = AA_triplets[qry_triple]
                                AA_mut2 = nuc2AA_ORF1a(mut2, refAA2, qryAA2)
                                if refAA2 == qryAA2 && !(pos2 in rem9b)
                                    synonymous_nuc_to_AA_dict[mut2] = AA_mut2
                                end
                            end
                        end
                    end
                else
                    qry_nuc = qry_nuc_comprehensive_dict[nuc_mut]
                    for (start_end, place) in noncoding_range_dict
                        frst = start_end[1]
                        last = start_end[2]
                        if pos ≥ frst && pos ≤ last
                            mut_vec = [nuc_mut, place]
                            push!(noncoding_nuc_vector, nuc_mut)
                        end
                    end
                end
            end
        end
#########################################################################################################
        for nuc_mut in muts
            pos_9b = nuc_mut_int_comprehensive_dict[nuc_mut]
            if pos_9b in rem9b && !isempty(ref_seq) && nuc_mut ≠ ""
                mut_9b = mixed2nuc(nuc_mut)
                pos_9b = nuc_mut_int_comprehensive_dict[mut_9b]   
                gene_number_9b = 11
                ref_triple_9b = ""
                qry_triple_9b = ""
                if nuc_codon_pos_dict_9b[pos_9b] == 1
                    ref_triple_9b = ref_nuc_comprehensive_dict[mut_9b]*string(ref_seq[pos_9b+1])*string(ref_seq[pos_9b+2])
                    qry_triple_9b = qry_nuc_comprehensive_dict[mut_9b]*string(ref_seq[pos_9b+1])*string(ref_seq[pos_9b+2])
                elseif nuc_codon_pos_dict_9b[pos_9b] == 2
                    ref_triple_9b = string(ref_seq[pos_9b-1])*ref_nuc_comprehensive_dict[mut_9b]*string(ref_seq[pos_9b+1])
                    qry_triple_9b = string(ref_seq[pos_9b-1])*qry_nuc_comprehensive_dict[mut_9b]*string(ref_seq[pos_9b+1])
                elseif nuc_codon_pos_dict_9b[pos_9b] == 3
                    ref_triple_9b = string(ref_seq[pos_9b-2])*string(ref_seq[pos_9b-1])*ref_nuc_comprehensive_dict[mut_9b]
                    qry_triple_9b = string(ref_seq[pos_9b-2])*string(ref_seq[pos_9b-1])*qry_nuc_comprehensive_dict[mut_9b]
                end
                refAA_9b = AA_triplets[ref_triple_9b]
                qryAA_9b = AA_triplets[qry_triple_9b]
                AA_mut_9b = nuc2AA_ORF9b(mut_9b, refAA_9b, qryAA_9b)
                if refAA_9b == qryAA_9b && nuc_mut in N_9b_synonymous
                    synonymous_nuc_to_AA_dict[mut_9b] = AA_mut_9b
                end
###################################
                for nuc_mut2_9b in muts
                    mut2_9b = mixed2nuc(nuc_mut2_9b)
                    pos2_9b = nuc_mut_int_comprehensive_dict[mut2_9b]
                    if pos2_9b in rem9b
                        gene_number2_9b = 11
                        if mut_9b ≠ mut2_9b && nuc_to_AA_pos_9b(mut_9b) == nuc_to_AA_pos_9b(mut2_9b)
                            if nuc_codon_pos_dict_9b[pos_9b] == 1 && nuc_codon_pos_dict_9b[pos2_9b] == 2
                                ref_triple_9b = ref_nuc_comprehensive_dict[mut_9b]*ref_nuc_comprehensive_dict[mut2_9b]*string(ref_seq[pos_9b+2])
                                qry_triple_9b = qry_nuc_comprehensive_dict[mut_9b]*qry_nuc_comprehensive_dict[mut2_9b]*string(ref_seq[pos_9b+2])
                            elseif nuc_codon_pos_dict_9b[pos_9b] == 1 && nuc_codon_pos_dict_9b[pos2_9b] == 3
                                ref_triple_9b = ref_nuc_comprehensive_dict[mut_9b]*string(ref_seq[pos_9b+1])*ref_nuc_comprehensive_dict[mut2_9b]
                                qry_triple_9b = qry_nuc_comprehensive_dict[mut_9b]*string(ref_seq[pos_9b+1])*qry_nuc_comprehensive_dict[mut2_9b]
                            elseif nuc_codon_pos_dict_9b[pos_9b] == 2 && nuc_codon_pos_dict_9b[pos2_9b] == 3
                                ref_triple_9b = string(ref_seq[pos_9b-1])*ref_nuc_comprehensive_dict[mut_9b]*ref_nuc_comprehensive_dict[mut2_9b]
                                qry_triple_9b = string(ref_seq[pos_9b-1])*qry_nuc_comprehensive_dict[mut_9b]*qry_nuc_comprehensive_dict[mut2_9b]
                            elseif nuc_codon_pos_dict_9b[pos2_9b] == 1 && nuc_codon_pos_dict_9b[pos_9b] == 2
                                ref_triple_9b = ref_nuc_comprehensive_dict[mut2_9b]*ref_nuc_comprehensive_dict[mut_9b]*string(ref_seq[pos_9b+1])
                                qry_triple_9b = qry_nuc_comprehensive_dict[mut2_9b]*qry_nuc_comprehensive_dict[mut_9b]*string(ref_seq[pos_9b+1])
                            elseif nuc_codon_pos_dict_9b[pos2_9b] == 1 && nuc_codon_pos_dict_9b[pos_9b] == 3
                                ref_triple_9b = ref_nuc_comprehensive_dict[mut2_9b]*string(ref_seq[pos_9b-1])*ref_nuc_comprehensive_dict[mut_9b]
                                qry_triple_9b = qry_nuc_comprehensive_dict[mut2_9b]*string(ref_seq[pos_9b-1])*qry_nuc_comprehensive_dict[mut_9b]
                            elseif nuc_codon_pos_dict_9b[pos2_9b] == 2 && nuc_codon_pos_dict_9b[pos_9b] == 3
                                ref_triple_9b = string(ref_seq[pos_9b-2])*ref_nuc_comprehensive_dict[mut2_9b]*ref_nuc_comprehensive_dict[mut_9b]
                                qry_triple_9b = string(ref_seq[pos_9b-2])*qry_nuc_comprehensive_dict[mut2_9b]*qry_nuc_comprehensive_dict[mut_9b]
                            end
                            refAA2_9b = AA_triplets[ref_triple_9b]
                            qryAA2_9b = AA_triplets[qry_triple_9b]
                            AA_mut2_9b = nuc2AA_ORF9b(mut2_9b, refAA2_9b, qryAA2_9b)
                            if refAA2_9b == qryAA2_9b && nuc_mut2_9b in N_9b_synonymous 
                                synonymous_nuc_to_AA_dict[mut2_9b] = AA_mut2_9b
                                synonymous_nuc_to_AA_dict[mut_9b] = AA_mut2_9b
                            end
                        end
                    end
                end
            end
        end
#####################################################################################################################################
        synonymous_nuc_to_AA_dict_sort = sort(collect(synonymous_nuc_to_AA_dict), by = x -> nuc_mut_int_comprehensive_dict[x[1]])
        synonymous_nuc_sort = sort(collect(keys(synonymous_nuc_to_AA_dict)), by = x -> nuc_mut_int_comprehensive_dict[x])
        synonymous_AA_sort = sort(collect(values(synonymous_nuc_to_AA_dict)), by = x -> AA_gene_sortKey_2(x))
        noncoding_nuc_vector_sort = sort(collect(noncoding_nuc_vector), by = x -> nuc_mut_int_comprehensive_dict[x])
        synonymous_nuc_total = length(synonymous_nuc_sort)
#        for i in 1:length(synonymous_nuc_sort)
#            nucpad = rpad("$(synonymous_nuc_to_AA_dict_sort[i][1])", 10)
#        end
        return synonymous_nuc_sort, noncoding_nuc_vector_sort
    else
        aa = Vector{Pair{String, String}}()
        bb = Vector{Pair{String, String}}()
        cc = Vector{Pair{String, String}}()
        return aa, bb, cc
    end 
end
#################################################################################
function add_leading_zero(int_str::String)
    int_str2 = int_str
    if length(int_str) == 1 && int_str ≠ "0"
        int_str2 = "0"*int_str
    end
    return int_str2
end     
##############################################################################################################################
###################################################################################################################
index_to_date_str = Dict{Int, String}()
date_str_to_index = Dict{String, Int}()
function convert_date_to_date_index(date_str::String)
    date_arr = string.(collect(date_str))
    date_tuple = nothing
### This counts the number of times "-" appears in the date_arr ---> sum(date_arr .== "-")
    if sum(date_arr .== "-") == 0
        year = parse(Int, date_str)
        date_tuple = (year, 0, 0)
    end
    if sum(date_arr .== "-") > 0
        year = parse(Int, split(date_str, "-")[1])
        month = parse(Int, split(date_str, "-")[2])
        if sum(date_arr .== "-") == 1
            date_tuple = (year, month, 0)
        else
            day = parse(Int, split(date_str, "-")[3])
            date_tuple = (year, month, day)
        end
    end
    date_index = tuple_to_index[date_tuple]
    return date_index
end
print("\n"^1)
println("Done Loading Functions (line #1618 as of 2022_02_22)!")
nowtime = Dates.format(now(), "I:MM.SS_p"); println(nowtime); print("\n"^1)
##############################################################################################################################################
##############################################################################################################################################
for tuple in keys(tuple_to_index)
    one = add_leading_zero(string(tuple[1]))
    two = add_leading_zero(string(tuple[2]))
    three = add_leading_zero(string(tuple[3]))
    date_string = one*"-"*two*"-"*three
    date_str_to_index[date_string] = convert_date_to_date_index(date_string)
end
for (index, tuple) in index_to_tuple
    one = add_leading_zero(string(tuple[1]))
    two = add_leading_zero(string(tuple[2]))
    three = add_leading_zero(string(tuple[3]))
    date_string = one*"-"*two*"-"*three
    index_to_date_str[index] = date_string
end
println("Done Making date_str_to_index & index_to_date_str (line #1641 as of 2022_02_22)!")
nowtime = Dates.format(now(), "I:MM.SS_p"); println(nowtime); print("\n"^1)
##################################################################
date_to_tuple = Dict{String, Tuple{Int, Int, Int}}()
tuple_to_date = Dict{Tuple{Int, Int, Int}, String}()
function convert_date_to_date_tuple(date_str::String)
    date_arr = string.(collect(date_str))
    date_tuple = nothing
### This counts the number of times "-" appears in the date_arr ---> sum(date_arr .== "-")
    if sum(date_arr .== "-") == 0
        year = parse(Int, date_str)
        date_tuple = (year, 0, 0)
    end
    if sum(date_arr .== "-") > 0
        year = parse(Int, split(date_str, "-")[1])
        month = parse(Int, split(date_str, "-")[2])
        if sum(date_arr .== "-") == 1
            date_tuple = (year, month, 0)
        else
            day = parse(Int, split(date_str, "-")[3])
            date_tuple = (year, month, day)
        end
    end
    return date_tuple
end
for date in keys(date_str_to_index)
    date_to_tuple[date] = convert_date_to_date_tuple(date)
end
for (date, date_tuple) in date_to_tuple
    tuple_to_date[date_tuple] = date
end
println("Done Making date_to_tuple & tuple_to_date (line #1672 as of 2022_02_22)!")
nowtime = Dates.format(now(), "I:MM.SS_p"); println(nowtime); print("\n"^1)
##################################################################################################################
function find_X_pct_date(clade_pango::String, pct::Float64, cp_date_cumul_dict::Dict{String, Dict{Int, Int}}, cp_total_dict::Dict{String, Int})
    cp_total = cp_total_dict[clade_pango]
    pct_date_index = 0
    pct_date_tuple = nothing
    for date_index in 1:2500
        cumulative_ct = cp_date_cumul_dict[clade_pango][date_index]
        if 100*cumulative_ct/cp_total ≥ pct
            pct_date_index = date_index
            pct_date_tuple = index_to_tuple[date_index]
            break
        end
    end
    return pct_date_index, pct_date_tuple
end
##################################################################################################################
### Truly necessary stuff from old megacell | 2026_02_08
############################################################################################################################################################################
###########################################################################################################################################################################
###########################################################################################################################################################################
all_chr_seqs_pangos = Dict{String, Int}()
all_chr_seqs_inherited = Dict{String, Int}()
all_chr_seqs_inherited_pos_only = Dict{String, Int}()
for seq in all_unique_chr_seqs
    pango = seq_pango[seq]
    all_chr_seqs_pangos[pango] = get(all_chr_seqs_pangos, pango, 0) + 1
    if !haskey(pango_AAsub_WT, pango)
        if haskey(pango_predecessor_meta_dict, pango)
            if haskey(pango_predecessor_meta_dict[pango], 2)
                pango = pango_predecessor_meta_dict[pango][2]
                if !haskey(pango_AAsub_WT, pango)
                    if haskey(pango_predecessor_meta_dict, pango)
                        if haskey(pango_predecessor_meta_dict[pango], 3)
                            pango = pango_predecessor_meta_dict[pango][3]
                        end
                    end
                end
            end
        end
    end
    if pango == "B.1.1.529"
        pango_AAsub_WT[pango] = union(pango_AAsub_WT["BA.1"], pango_AAsub_WT["BA.2"])
        for mut in pango_AAsub_WT[pango]
            all_chr_seqs_inherited[mut] = get(all_chr_seqs_inherited, mut, 0) + 1
        end
    elseif haskey(pango_AAsub_WT, pango)
        for mut in pango_AAsub_WT[pango]
            all_chr_seqs_inherited[mut] = get(all_chr_seqs_inherited, mut, 0) + 1
        end
    elseif haskey(pango_predecessor_meta_dict, pango)
        if haskey(pango_predecessor_meta_dict[pango], 1)
            pango_pre1 = pango_predecessor_meta_dict[pango][1]
            for mut in pango_AAsub_WT[pango_pre1]
                all_chr_seqs_inherited[mut] = get(all_chr_seqs_inherited, mut, 0) + 1
            end
        end
    end
end
println("Done Filling all_chr_seqs_pangos, all_chr_seqs_inherited, pango_AAsub_WT (leftovers)!")
nowtime = Dates.format(now(), "I:MM.SS_p"); println(nowtime); print("\n"^1)
############################################################################################################################################################################
############################################################################################################################################################################
############################################################################################################################################################################
seq_syn_nucs = Dict{String, Vector{String}}()
seq_noncoding_nucs = Dict{String, Vector{String}}()
for (seq, nuc_set) in seq_nuc_muts
    pango = seq_pango[seq]
#    if !haskey(nuc_genome_pango_dict, pango)
#        println("No nuc_genome_pango_dict key, $(pango)")
#    end
    synonymous_nucmuts, noncoding_nucmuts = SIMPLER_syn_noncoding_nuc(pango, nuc_set)
    seq_syn_nucs[seq] = synonymous_nucmuts
    seq_noncoding_nucs[seq] = noncoding_nucmuts
end
println("Done Filling seq_syn_nucs, seq_noncoding_nucs!")
nowtime = Dates.format(now(), "I:MM.SS_p"); println(nowtime); print("\n"^1)
############################################################################################################################################################################
runtime = time() - start
runtime_rd = round(digits=2, runtime)
runtime1, runtime2 = seconds_to_hrs_min_sec(runtime)
println("Runtime v0 = $(runtime) seconds")
println("Runtime v2 = $(runtime2)")
############################################################################################################################################################################
############################################################################################################################################################################
############################################################################################################################################################################
############################################################################################################################################################################
############################################################################################################################################################################
println("Finished!"); print("\n"^2)

2:22.00_PM

Done Loading Functions (line #1618 as of 2022_02_22)!
2:22.00_PM

Done Making date_str_to_index & index_to_date_str (line #1641 as of 2022_02_22)!
2:22.00_PM

Done Making date_to_tuple & tuple_to_date (line #1672 as of 2022_02_22)!
2:22.00_PM

Done Filling all_chr_seqs_pangos, all_chr_seqs_inherited, pango_AAsub_WT (leftovers)!
2:22.00_PM

Done Filling seq_syn_nucs, seq_noncoding_nucs!
2:23.16_PM

Runtime v0 = 76.03781700134277 seconds
Runtime v2 = 0 hr, 1 min, 16.04 sec
Finished!




In [48]:
### Making pango_to_pango_unaliased_v2 & pango consensus Dicts + MANY Others | Runtime = 154 seconds
####### Making pango_to_pango_unaliased_v2 —— using pango_variant_alias_key_2026_01_03_update.json from https://github.com/cov-lineages/pango-designation/blob/master/pango_designation/alias_key.json
start = time(); date_now = Dates.format(now(), "yyyy_mm_dd_IMMp"); println(date_now)
println("nuc_genome_pango_dict length = $(length(nuc_genome_pango_dict))")

missing_AA_dict_keys = Set{String}()
############################################################################################################################################################################
#################### Making/filling index_to_tuple, tuple_to_index, index_to_date_str, date_str_to_index...etc dictionaries ################################################
############################################################################################################################################################################
date_now = Dates.format(now(), "yyyy_mm_dd__IMMp"); println(date_now)
###################################
chronic_pango_set = Set{String}()
chronic_pango_unaliased_set = Set{String}()
chronic_clade_set = Set{String}()
for seq in all_qualifying_seqs_set
    push!(chronic_pango_set, seq_pango[seq])
    push!(chronic_pango_unaliased_set, seq_pango_unaliased[seq])
    push!(chronic_clade_set, seq_clade[seq])
end
chronic_pango_set_len = length(chronic_pango_set)
chronic_pango_unaliased_set_len = length(chronic_pango_unaliased_set)
chronic_clade_set_len = length(chronic_clade_set)
println("Number of Different Pango Lineages in Chronics = $(chronic_pango_set_len)")
println("Number of Different Pango_Unaliased Lineages in Chronics = $(chronic_pango_unaliased_set_len)")
println("Number of Different Clades in Chronics = $(chronic_clade_set_len)")
println()
#####################################################################################################################################
all_effing_pangos = union(pango_set, pango_index_date_set, chronic_pango_set)
push!(all_effing_pangos, "B.1.640")
push!(all_effing_pangos, "B.1.640.1")
push!(all_effing_pangos, "B.1.640.2")
push!(all_effing_pangos, "B.1.258.2")
push!(all_effing_pangos, "G.1")
############################################################################################################################################################################
pango_to_pango_unaliased_v2["G.1"] = "B.1.258.2.1"
pgct = 0
for (pngo, unal) in pango_to_pango_unaliased_v2
    pgct += 1
    if pgct < 15
        println("$(pngo) = $(unal)")
    end
end
println("length(keys(pango_to_pango_unaliased_v2) = $(length(keys(pango_to_pango_unaliased_v2)))")
########################################################################################################################################################################
########################################################################################################################################################################
date_now = Dates.format(now(), "yyyy_mm_dd__IMMp"); println(date_now)
date_hour = Dates.format(now(), "yyyy_mm_dd_Ip")
missing_keys = String[]
ndjson = "___pango_consensus_sequences/pango_consensus_sequences_summary_2025_06_25.ndjson"
for line in eachline(ndjson)
    j = JSON3.read(line)
    pango = j.lineage
    if !(pango in pango_set)
        push!(missing_keys, pango)
    end
    push!(pango_set, pango)
end
for pango in keys(pango_date_index_ct)
    if !(pango in keys(pango_to_pango_unaliased_v2))
        push!(pango_set, pango)
    end
end
date_now = Dates.format(now(), "yyyy_mm_dd__IMMp"); println(date_now)
#pango_to_pango_unaliased = Dict{String, String}()
clade_pango_set = Dict{String, Set{String}}()
#################################################################################
pango_nuc_sub_WT = Dict{String, Set{String}}()
pango_nuc_del_WT = Dict{String, Set{String}}()
pango_nuc_sub_private = Dict{String, Set{String}}()
pango_nuc_del_private = Dict{String, Set{String}}()
pango_nuc_sub_revs = Dict{String, Set{String}}()
pango_nuc_del_revs = Dict{String, Set{String}}()

pango_AAsub_WT = Dict{String, Set{String}}()
pango_AAsub_WT_pos_only = Dict{String, Set{String}}()
pango_AAdel_WT = Dict{String, Set{String}}()
pango_AAsub_private = Dict{String, Set{String}}()
pango_AAdel_private = Dict{String, Set{String}}()
pango_AAsub_revs = Dict{String, Set{String}}()
pango_AAdel_revs = Dict{String, Set{String}}()

pango_frameshifts_WT = Dict{String, Set{String}}()
pango_designation_date = Dict{String, String}()
#####################################################
for pango in pango_set
    pango_nuc_sub_WT[pango] = Set{String}()
    pango_nuc_del_WT[pango] = Set{String}()
    pango_nuc_sub_private[pango] = Set{String}()
    pango_nuc_del_private[pango] = Set{String}()
    pango_nuc_sub_revs[pango] = Set{String}()
    pango_nuc_del_revs[pango] = Set{String}()
########
    pango_AAsub_WT[pango] = Set{String}()
    pango_AAsub_WT_pos_only[pango] = Set{String}()
    pango_AAdel_WT[pango] = Set{String}()
    pango_AAsub_private[pango] = Set{String}()
    pango_AAdel_private[pango] = Set{String}()
    pango_AAsub_revs[pango] = Set{String}()
    pango_AAdel_revs[pango] = Set{String}()
#########
    pango_frameshifts_WT[pango] = Set{String}()
end
#################################################################################
for clade in clade_set
    clade_pango_set[clade] = Set{String}()
end
ndjson = "___pango_consensus_sequences/pango_consensus_sequences_summary_2025_06_25.ndjson"
missing_keys_for_WT = Set{String}()
###################################################################
#pango_consensus_nuc_subs_WT = 
#pango_consensus_nuc_dels_WT = 
#pango_consensus_nuc_subs = 
#pango_consensus_nuc_dels = 
#pango_consensus_nuc_revs = 
#pango_consensus_nuc_del_revs = 
#pango_consensus_AA_subs_WT = 
#pango_consensus_AA_dels_WT = 
#pango_consensus_AA_subs = 
#pango_consensus_AA_dels = 
#pango_consensus_AA_revs = 
#pango_consensus_AA_del_revs = 
#pango_consensus_
###################################################################
for line in eachline(ndjson)
    j = JSON3.read(line)
    pango = j.lineage
    unaliased = j.unaliased
    clade = j.nextstrainClade
    push!(clade_pango_set[clade], pango)
#    pango_to_pango_unaliased[pango] = unaliased
####################
    parent = j.parent
    children = j.children
####################
    nuc_subs_WT = j.nucSubstitutions
    nuc_dels_WT = j.nucDeletions
    nuc_subs = j.nucSubstitutionsNew
    nuc_dels = j.nucDeletionsNew
    nuc_revs = j.nucSubstitutionsReverted
    nuc_del_revs = j.nucDeletionsReverted
#####################
    for nuc_sub in nuc_subs_WT
        push!(pango_nuc_sub_WT[pango], nuc_sub)
    end
    for ndwt in nuc_dels_WT
        push!(pango_nuc_del_WT[pango], ndwt)
    end
    for n in nuc_subs
        push!(pango_nuc_sub_private[pango], n)
    end
    for n in nuc_dels
        push!(pango_nuc_del_private[pango], n)
    end
    for n in nuc_revs
        push!(pango_nuc_sub_revs[pango], n)
    end
    for n in nuc_del_revs
        push!(pango_nuc_del_revs[pango], n)
    end
########################################################
    AA_subs_WT = j.aaSubstitutions
    AA_dels_WT = Set(j.aaDeletions)
    AA_subs = j.aaSubstitutionsNew
    AA_dels = j.aaDeletionsNew
    AA_revs = j.aaSubstitutionsReverted
    AA_del_revs = j.aaDeletionsReverted
##################################################
    AA_subs_WT_pos_only = Set{String}()
    delete!(AA_dels_WT, "")
    for sub in AA_subs_WT
        if !haskey(AA_gene_and_pos_dict, sub)
            push!(missing_keys_for_WT, sub)
            continue
        end
        sub_po = AA_gene_and_pos_dict[sub]
        push!(AA_subs_WT_pos_only, sub_po)
    end 
    for AA_sub_po in AA_subs_WT_pos_only
        push!(pango_AAsub_WT_pos_only[pango], AA_sub_po)
    end
##################################################
    for AAsub in AA_subs_WT
        push!(pango_AAsub_WT[pango], AAsub)
    end
    for AAdel in AA_dels_WT
        push!(pango_AAdel_WT[pango], AAdel)
    end
    for AAsub in AA_subs
        push!(pango_AAsub_private[pango], AAsub)
    end
    for AAdel in AA_dels
        push!(pango_AAdel_private[pango], AAdel)
    end
    for AArev in AA_revs
        push!(pango_AAsub_revs[pango], AArev)
    end
    for AAdelrev in AA_del_revs
        push!(pango_AAdel_revs[pango], AAdelrev)
    end
    frameshiftsWT = j.frameShifts
    for fs in frameshiftsWT
        push!(pango_frameshifts_WT[pango], fs)
    end
    designation_date = j.designationDate
    pango_designation_date[pango] = designation_date
end
#####################################################################################################################################################################
#####################################################################################################################################################################
push!(pango_nuc_del_WT["B.1.617.2"], "28271")
push!(pango_nuc_del_private["B.1.617.2"], "28271")
#####################################################################################################################################################################
#####################################################################################################################################################################
println("Done Making Pango Consensus Sequences! (Part 1)"); print("\n"^1)
#####################################################################################################################################################################
println("Total Missing Keys for AA_pos_only_gene_and_pos_dict = $(length(missing_keys_for_WT))")
#missing_keys_for_WT_sort = sort(collect(missing_keys_for_WT), by = x -> mp_AA_gene_sortKey_2(x))
missing_keys_for_WT_join = join(collect(missing_keys_for_WT), ", ")
println("missing_keys_for_WT_join = ", "|", missing_keys_for_WT_join, "|"); print("\n"^1)
############################################################################################################################################################################
############################################################################################################################################################################
############################################################################################################################################################################
############################################################# NEW SECTION (2026-01-03)   #################################################################################
############################################################################################################################################################################
############################################################################################################################################################################
############################################################################################################################################################################
### Making consensus muts for new sequences (using Angie Hinrichs' tree mut TSV file) —— MUST RUN load_all_seq_dicts FIRST
df_tree_muts = CSV.read("___pango_consensus_sequences/pango_clade_mutations_from_Angie_Hinrichs_2026_01_03.tsv", DataFrame, delim='\t')
##################synonymous_nuc_AA_sorted_dict, nonsynonymous_nuc_AA_sorted_dict, all_nuc_AA_sorted_dict = nuc_to_AA(pango, new_muts)########################################################################################################################################
total_rows = nrow(df_tree_muts)
tree_pangos = df_tree_muts[!, 1]
tree_muts = df_tree_muts[!, 2]
#####################################################################################################################
#  Example of 1st, then 2nd column (i.e. tree_pangos & tree_muts) ——  AY.112.2 | AY.112 > G23012C > G4162T > G27516A
#####################################################################################################################
tree_pangos_set = Set(tree_pangos)
tree_muts_set = Set(tree_muts)
################################################
bad_pangos = Vector{String}()
################################################
OG_tree_pangos_set_len = length(tree_pangos_set)
println("OG_tree_pangos_set_len = $(OG_tree_pangos_set_len)")
excluded_bits = ["misc", "proposed", '_']
for tpango in tree_pangos_set
    for exclude in excluded_bits
        if occursin(exclude, tpango)
            push!(bad_pangos, tpango)
        end
    end
end
for baddy in bad_pangos
    delete!(tree_pangos_set, baddy)
end
NEW_tree_pangos_set_len = length(tree_pangos_set)
println("NEW_tree_pangos_set_len = $(NEW_tree_pangos_set_len)")
println("bad_pangos length = $(length(bad_pangos))")
bad_pangos_sort = sort(bad_pangos)
bad_pangos_sort_join = join(bad_pangos_sort, ", ")
bad_pangos_sort_join_newlines = join(bad_pangos_sort, "\n")
###########
println(bad_pangos_sort_join)
############################################################################################################################################################################
pango_unaliased_to_pango_prefix = Dict{String, String}()
pango_to_pango_unaliased_v2 = Dict{String, String}()
pango_unaliased_to_pango = Dict{String, String}()
##################################################################
pango_unaliased_predecessor_meta_dict = Dict{String, Dict{Int, String}}()
pango_predecessor_meta_dict = Dict{String, Dict{Int, String}}()
##################################################################
for (prefix, unaliased) in pango_prefix_to_pango_unaliased
    pango_unaliased_to_pango_prefix[unaliased] = prefix
end
for pango in tree_pangos_set
    dotpts_ct = count('.', pango)
    if dotpts_ct ≥ 1
        dotsplits = split(pango, ".")
        prefix = string(dotsplits[1])
        prefix_unaliased = get(pango_prefix_to_pango_unaliased, prefix, prefix)
        last_pts = join(dotsplits[2:end], ".")
        pango_to_pango_unaliased_v2[pango] = "$(prefix_unaliased).$(last_pts)"
    else
        pango_to_pango_unaliased_v2[pango] = pango
    end
end
#####################################################################
pango_to_pango_unaliased_v2_key_set = Set(collect(keys(pango_to_pango_unaliased_v2)))
no_key_pangos = Set(["C.1.2", "B.1.1.70", "B.1.177.46", "B.1.36", "B.1.551", "B.1.1.329", "B.1.1.170", "C.26", "B.1.214.3", "B.1.609", "B.55", "A.2.5.3", "B.1.1.117", "C.13", "B.1.1.523", "B.1.470", "B.1.1.67", "B.39", "B.1.36.31", "B.1.36.22", "B.4", "B.1.1.500", "B.1.258.17", "B.1.1.101", "B.1.1.141", "B.1.260", "B.1.1.360", "AE.1", "B.1.404", "B.1.351.1", "AE.8", "B.1.1.10", "B.1.36.19", "B.1.587", "B.1.465", "B.1.177.52", "B.1.1.404", "B.1.436", "A.5", "B.1.1.221", "B.1.396", "B.1.428", "C.4", "B.1.214.2", "B.1.568", "B.1.147", "B.1.1.442", "B.1.1.47", "B.1.575.1", "B.1.177.4", "B.1.527", "B.1.1.354", "B.1.530", "B.1.361", "B.1.36.36", "B.1.126", "B.1.1.297", "B.1.420", "B.1.1.406", "B.1.36.29", "B.1.560", "B.1.214.1", "W.1", "B.1.177.12", "B.1.629", "B.1.1.37", "B.1.1.192", "N.6", "A.23", "B.1.177.54", "A.2.5.1", "A.3", "B.1.1.263", "B.1.379", "B.1.1.376", "B.1.240.1", "B.1.466.1", "B.1.1.241", "B.1.1.368", "B.1.177.32", "B.1.1.420", "B.1.1.413", "B.1.236", "B.1.2", "B.23", "B.1.1.418", "B.1.416.1", "B.1.1.54", "B.1.540", "AZ.5", "B.1.389", "B.1.433", "B.1.1.205", "B.1.1.434", "B.1.1.359", "C.2", "B.1.600", "B.1.9", "B.1.575", "B.1.177.57", "B.1.627", "B.1.36.35", "A.19", "B.1.371", "B.1.462", "B.1.22", "B.1.111", "B.1.258", "B.1.427", "B.1.177.53", "B.1.626", "B.1.1.294", "B.1.36.38", "B.1.1.332", "B.1.1.417", "B.1.1.521", "B.1.558", "B.1.36.18", "B.1.544", "B.1.1.375", "B.1.1.412", "B.1.1.485", "B.1.1.326", "B.1.375", "B.1.1.243", "B.1.1.369", "B.1.336", "B.1.577", "B.1.409", "B.1.219", "B.1.1.301", "B.1.1.433", "B.1.128", "B.1.179", "B.1.1.519", "B.1.1.307", "B.1.537", "B.1.1.265", "C.38", "B.1.177.44", "B.1.620", "AD.2", "C.23", "B.1.189", "B.1.349", "B.1.567", "B.1.398", "AZ.2", "B.1.1.25", "B.1.1.163", "B.1.497", "B.1.480", "B.1.1.198", "B.1.320", "B.1.1.121", "B.1.402", "B.6.6", "B.1.8", "B.1.298", "B.1.177.8", "B.1.1.39", "B.1.324", "B.1.1.351", "B.1.1.161", "B.1.1.279", "B.1.1.46", "B.1.177.83", "B.1.177.15", "B.1.346", "B.1.566", "B.1.78", "B.1.239", "A", "B.1.1.318", "B.1.397", "B.1.177.10", "B.1.1.416", "B.1.1.27", "AZ.3", "B.1.319", "B.1.619", "B.1.177.7", "B.1.258.3", "B.1.1.317", "B.1.605", "B.1.146", "B.1.1.61", "A.25", "C.40", "B.3", "B.1.438", "B.1.1.33", "B.1.192", "B.1.468", "C.11", "B.1.177.18", "B.1.471", "B.1.1.52", "B.1.1.171", "C.1", "B.1.1.348", "B.1.1.507", "B.1.177.82", "B.1.533", "B.1.1.306", "B.1.524", "B.1.149", "B.1.619.1", "B.1.36.9", "B.1.570", "B.1.1.50", "B.1.517", "B.1.499", "B.1.177.60", "B.1.1.378", "B.1.177.62", "C.16", "C.36.3", "B.1.1.459", "B.1.177.75", "A.27", "B.1.1.207", "B.1.618", "B.1.391", "B.1.1.115", "A.29", "B.1.1.1", "L.3", "A.28", "B.1.1.302", "B.1.3", "C.2.1", "B.1.1.135", "B.1.1.316", "B.1.393", "B.1.177.87", "R.1", "B.1.1.176", "C.32", "B.1.636", "B.1.1.133", "B.1.1.462", "N.3", "B.1.399", "B.1.1.487", "B.1.1.311", "B.1.606", "B.1.617.1", "B.1.265", "B.1.177.86", "N.9", "B.1.1.374", "B.1.1.228", "B.1.177", "B.1.177.5", "A.2.5.2", "B.1.1.57", "C.33", "B.1.538", "B.1.258.14", "B.1.1.305", "B.1.505", "B.1.1.397", "A.1", "B.1.565", "AT.1", "B.1.1.214", "B.1.177.16", "C.35", "B.6", "B.1.356", "B.1.177.21", "B.1.1.153", "B.1.1.432", "B.1.1.181", "B.1.1.312", "B.1.1.216", "B.1.1.526", "A.21", "B.1.380", "B.1.210", "B.1.1.398", "B.1.232", "B.1.1.220", "C.36", "B.1.438.1", "B.1.503", "B.1.36.7", "B.1.426", "B.1.1.159", "B.1.1.231", "A.18", "B.1.177.77", "B.1.1.99", "B.1.1.298", "B.1.453", "B.1.1.253", "N.5", "B.1.367", "B.1.549", "C.14", "B.1.400", "B.1.1.284", "AZ.4", "B.1.459", "B.1.243", "C.36.3.1", "B.1.474", "B.1.221", "B.1.630", "C.29", "B.1.1.528", "B.1.1.203", "B.1.1.269", "B.1.340", "B.1.634", "B.1.1.174", "B.1.596.1", "B.1.311", "B.1.36.24", "B.1.509", "B.1.36.39", "B.1.1.189", "B.1.36.16", "B.1.441", "B.1.91", "B.1.416", "B.1.1.56", "B.1.258.11", "B.1.417", "B.1.525", "B.1.561", "B.1.1.525", "B.1.1.277", "B.1.177.35", "B.1.240", "B.1.1.382", "B.1.234", "B.1.1.273", "B.1.1.291", "N.4", "B.1.617.3", "A.2", "B.1.523", "B.1.279", "B.1.1.63", "B.1.1.51", "B.1.1.157", "B.1.415.1", "B.1.212", "B.1.177.73", "B.1.1.371", "A.2.5", "B.1.1.44", "B.1.1.464", "B.1.362", "D.2", "B.1.241", "B.1.1.486", "B.1.377", "B.1.1.53", "B.1.588", "B.1.1.391", "B.1.1.186", "B.1.1.274", "B.1.623", "B.1.36.1", "B.1.195", "B.1.237", "B.1.110", "B.1.1.111", "AZ.1", "B.1.395", "B.1.381", "C.39", "B.1.206", "B.1.177.81", "B.1.214", "B.40", "B.1.110.3", "B.1.1.83", "B.1.411", "B.1.625", "B.1.350"])
for pango in all_effing_pangos
    if !haskey(pango_to_pango_unaliased_v2, pango)
        dotpts_ct = count('.', pango)
        if dotpts_ct ≥ 1
            dotsplits = split(pango, ".")
            prefix = string(dotsplits[1])
            prefix_unaliased = get(pango_prefix_to_pango_unaliased, prefix, prefix)
            last_pts = join(dotsplits[2:end], ".")
            pango_to_pango_unaliased_v2[pango] = "$(prefix_unaliased).$(last_pts)"
        end
    end
end
pango_to_pango_unaliased_v2["B.1.1.529"] = "B.1.1.529"
pango_to_pango_unaliased_v2["LF.3"] = "B.1.1.529.2.86.1.1.16.1.3"
pango_to_pango_unaliased_v2["LF.3.1"] = "B.1.1.529.2.86.1.1.16.1.3.1"
pango_to_pango_unaliased_v2["B.1.469"] = "B.1.469"
pango_to_pango_unaliased_v2["A"] = "A"
pango_to_pango_unaliased_v2["B"] = "B"
pango_to_pango_unaliased_v2["B.1.1.482"] = "B.1.1.482"
pango_to_pango_unaliased_v2["B.1.1.370"] = "B.1.1.370"
pango_to_pango_unaliased_v2["B.1.596"] = "B.1.596"
pango_to_pango_unaliased_v2["B.1.415"] = "B.1.415"
####################################################################
pango_unaliased_set = Set{String}()
for (pango, unaliased) in pango_to_pango_unaliased_v2
    pango_unaliased_to_pango[unaliased] = pango
    push!(pango_unaliased_set, unaliased)
end
####################################################################
all_effing_pangos = union(all_effing_pangos, pango_to_pango_unaliased_v2_key_set)
for pango in all_effing_pangos
    unaliased = pango_to_pango_unaliased_v2[pango]
    pango_unaliased_predecessor_meta_dict[unaliased] = Dict{Int, String}()
#    println(pango)
    dotpts_ct = count('.', unaliased)
    if dotpts_ct ≥ 1
        dotsplits = split(unaliased, ".")
        for i in 1:dotpts_ct
            predecessor_i_unaliased = join(dotsplits[1:end-i], ".")
            pango_unaliased_predecessor_meta_dict[unaliased][i] = predecessor_i_unaliased
#            println("$(pango), $(unaliased), $(i) = $(predecessor_i_unaliased)")
        end
    end
end
pango_unaliased_predecessor_meta_dict["B.1.1.370"] = Dict{Int, String}()
pango_unaliased_predecessor_meta_dict["B.1.1.370"][1] = "B.1.1"
pango_unaliased_predecessor_meta_dict["B.1.1.370"][2] = "B.1"
pango_unaliased_predecessor_meta_dict["B.1.1.370"][3] = "B"
pango_unaliased_to_pango["B.1.1.370"] = "B.1.1.370"

pango_unaliased_predecessor_meta_dict["B.1.1.482"] = Dict{Int, String}()
pango_unaliased_predecessor_meta_dict["B.1.1.482"][1] = "B.1.1"
pango_unaliased_predecessor_meta_dict["B.1.1.482"][2] = "B.1"
pango_unaliased_predecessor_meta_dict["B.1.1.482"][3] = "B"
pango_unaliased_to_pango["B.1.1.482"] = "B.1.1.482"

pango_unaliased_predecessor_meta_dict["B.1.596"] = Dict{Int, String}()
pango_unaliased_predecessor_meta_dict["B.1.596"][1] = "B.1.1"
pango_unaliased_predecessor_meta_dict["B.1.596"][2] = "B.1"
pango_unaliased_predecessor_meta_dict["B.1.596"][3] = "B"
pango_unaliased_to_pango["B.1.596"] = "B.1.596"

pango_unaliased_predecessor_meta_dict["B.1.415"] = Dict{Int, String}()
pango_unaliased_predecessor_meta_dict["B.1.415"][1] = "B.1.1"
pango_unaliased_predecessor_meta_dict["B.1.415"][2] = "B.1"
pango_unaliased_predecessor_meta_dict["B.1.415"][3] = "B"
pango_unaliased_to_pango["B.1.415"] = "B.1.415"

for pango in all_effing_pangos
    pango_predecessor_meta_dict[pango] = Dict{Int, String}()
    unaliased = pango_to_pango_unaliased_v2[pango]
    for (i, predecessor_i) in pango_unaliased_predecessor_meta_dict[unaliased]
        predecessor_i_pango = pango_unaliased_to_pango[predecessor_i]
        pango_predecessor_meta_dict[pango][i] = predecessor_i_pango
    end
end
#######################################################
pango_unaliased_predecessor_meta_dict["B.1.1.529.2.86.1.1.16.1.3.1"] = Dict{Int, String}()
pango_unaliased_predecessor_meta_dict["B.1.1.529.2.86.1.1.16.1.3.1"][1] = "B.1.1.529.2.86.1.1.16.1.3"
pango_unaliased_predecessor_meta_dict["B.1.1.529.2.86.1.1.16.1.3.1"][2] = "B.1.1.529.2.86.1.1.16.1"
pango_unaliased_predecessor_meta_dict["B.1.1.529.2.86.1.1.16.1.3.1"][3] = "B.1.1.529.2.86.1.1.16"
pango_unaliased_predecessor_meta_dict["B.1.1.529.2.86.1.1.16.1.3.1"][4] = "B.1.1.529.2.86.1.1"
pango_unaliased_predecessor_meta_dict["B.1.1.529.2.86.1.1.16.1.3.1"][5] = "B.1.1.529.2.86.1"
pango_unaliased_predecessor_meta_dict["B.1.1.529.2.86.1.1.16.1.3.1"][6] = "B.1.1.529.2.86"
pango_unaliased_predecessor_meta_dict["B.1.1.529.2.86.1.1.16.1.3.1"][7] = "B.1.1.529.2"
pango_unaliased_predecessor_meta_dict["B.1.1.529.2.86.1.1.16.1.3.1"][8] = "B.1.1.529"
pango_unaliased_predecessor_meta_dict["B.1.1.529.2.86.1.1.16.1.3.1"][9] = "B.1.1"
pango_unaliased_predecessor_meta_dict["B.1.1.529.2.86.1.1.16.1.3.1"][10] = "B.1"
pango_unaliased_predecessor_meta_dict["B.1.1.529.2.86.1.1.16.1.3.1"][11] = "B"
pango_predecessor_meta_dict["LF.3.1"] = Dict{Int, String}()
pango_predecessor_meta_dict["LF.3.1"][1] = "LF.3"
pango_predecessor_meta_dict["LF.3.1"][2] = "JN.1.16.1"
pango_predecessor_meta_dict["LF.3.1"][3] = "JN.1.16"
pango_predecessor_meta_dict["LF.3.1"][4] = "JN.1"
pango_predecessor_meta_dict["LF.3.1"][5] = "BA.2.86.1"
pango_predecessor_meta_dict["LF.3.1"][6] = "BA.2.86"
pango_predecessor_meta_dict["LF.3.1"][7] = "BA.2"
pango_predecessor_meta_dict["LF.3.1"][8] = "B.1.1.529"
pango_predecessor_meta_dict["LF.3.1"][9] = "B.1.1"
pango_predecessor_meta_dict["LF.3.1"][10] = "B.1"
pango_predecessor_meta_dict["LF.3.1"][11] = "B"
pango_unaliased_predecessor_meta_dict["B.1.1.529"] = Dict{Int, String}()
pango_unaliased_predecessor_meta_dict["B.1.1.529"][1] = "B.1.1"
pango_unaliased_predecessor_meta_dict["B.1.1.529"][2] = "B.1"
pango_unaliased_predecessor_meta_dict["B.1.1.529"][3] = "B"
pango_predecessor_meta_dict["B.1.1.529"] = Dict{Int, String}()
pango_predecessor_meta_dict["B.1.1.529"][1] = "B.1.1"
pango_predecessor_meta_dict["B.1.1.529"][2] = "B.1"
pango_predecessor_meta_dict["B.1.1.529"][3] = "B"
############################################################################################################################################################################
forbidden_pangos = Set(["XCD", "XCT", "XEA", "XFR", "XFS", "XFT", "XFT.1", "XFU", "XGD"])
#######################################################################################################################################
pango_new_nuc_muts = Dict{String, Set{String}}()
pango_new_nuc_muts_WT = Dict{String, Set{String}}()
pango_new_nuc_muts_pos_only = Dict{String, Set{Int}}()
###########################
pango_new_AA_muts = Dict{String, Set{String}}()
pango_new_AA_muts_WT = Dict{String, Set{String}}()
pango_new_AA_muts_pos_only = Dict{String, Set{String}}()
###########################
nuc_mut_all_and_chr_set_missing = Set{String}()
new_pango_ct = 0
empty_string_pangos = Set{String}()
for i in 1:length(tree_muts)
    if tree_pangos[i] in tree_pangos_set
        pango = convert(String, tree_pangos[i])
        unaliased = pango_to_pango_unaliased_v2[pango]
        if ismissing(tree_muts[i])
            println("missing tree muts: $(pango)")
            continue
        end
        new_muts_pre0 = replace(tree_muts[i], r"        "=>",")
        new_muts_pre1 = replace(new_muts_pre0, r"[TCAGN] [TCAGN]"=>",")
        new_muts_pre2 = replace(new_muts_pre1, r"\s"=>"")
        new_muts_pre3 = replace(new_muts_pre2, ">"=>",")
        new_muts_pre4 = replace(new_muts_pre3, ",,"=>",")
#        println("new_muts_pre3 = $(new_muts_pre3)")
        comma_splits = string.(split(new_muts_pre3, ","))
        predecessor_vec = String[]
        predecessor = string(comma_splits[1])
        if '_' in predecessor
            predecessor = string(split(predecessor, "_")[1])
        end
#        predecessor_unaliased = pango_to_pango_unaliased_v2[predecessor]
        push!(predecessor_vec, predecessor)
#        println(predecessor)
        for j in 1:length(pango_unaliased_predecessor_meta_dict[unaliased])
            predecessor_j = pango_unaliased_predecessor_meta_dict[unaliased][j]
            predecessor_j_pango = pango_unaliased_to_pango[predecessor_j]
            push!(predecessor_vec, predecessor_j_pango)
        end
        new_muts_pre = comma_splits[2:end]
        new_muts = String[]
        for mut in new_muts_pre
            if !('N' in mut)
                push!(new_muts, mut)
            end
        end
#        println("new_muts = $(new_muts)")
#        if !haskey(pango_nuc_sub_private, pango) && !haskey(pango_AAsub_private, pango)
#            pango_nuc_sub_WT[pango] = Set{String}()
#            pango_nuc_del_WT[pango] = Set{String}()
#            pango_nuc_sub_private[pango] = Set{String}()
#            pango_nuc_del_private[pango] = Set{String}()
#            pango_AAsub_WT[pango] = Set{String}()
#            pango_AAsub_WT_pos_only[pango] = Set{String}()
#            pango_AAdel_WT[pango] = Set{String}()
#            pango_AAsub_private[pango] = Set{String}()
#            pango_AAdel_private[pango] = Set{String}()
#        end
#        new_muts = filter!(
        if !isempty(new_muts)
            for xmut in new_muts
                if xmut == ""
                    push!(empty_string_pangos, pango)
#                    println("empty string in $(pango)")
                    continue
                end
                if !(xmut in nuc_mut_all_and_chr_set)
                    push!(nuc_mut_all_and_chr_set_missing, xmut)
                    pos_int = nuc_mut_int_comprehensive_dict[xmut]
                    refnuc = ref_nuc_comprehensive_dict[xmut]
                    qrynuc = qry_nuc_comprehensive_dict[xmut]
                    nuc_mut_int_string_comprehensive_dict[xmut] = string(pos_int)
                end
            end
        end
        precedessor_vec_join = join(predecessor_vec, ", ")
        if pango == "BA.3.2"
            println("BA.3.2 new_muts = $(new_muts)")
            println("BA.3.2 precedessor_vec_join = $(precedessor_vec_join)")
        end
#        println(precedessor_vec_join)
        for predecessor_j in predecessor_vec
            if (!haskey(nuc_genome_pango_dict, pango) || !haskey(pango_AAsub_private, pango)) && haskey(nuc_genome_pango_dict, predecessor_j) && length(nuc_genome_pango_dict[predecessor_j]) ≥ 28600 # && predecessor_j in pango_consensus_set && !(pango in forbidden_pangos)
                new_pango_ct += 1
                if new_pango_ct%100 == 0
                    println("################## NEW PANGO COUNT = $(new_pango_ct) #######################")
                end
                gene_AA_pango_dict[pango] = Dict{String, String}()
                for gene in gene_array
                    gene_AA_pango_dict[pango][gene] = gene_AA_pango_dict[predecessor_j][gene]
                end
####################################################################################################################################################
#                println(new_muts)
                synonymous_nuc_AA_sorted_dict, nonsynonymous_nuc_AA_sorted_dict, all_nuc_AA_sorted_dict = SIMPLER_nuc_to_AA(predecessor_j, new_muts)
####################################################################################################################################################
                sorted_nuc_muts = String[]
                if !isempty(all_nuc_AA_sorted_dict)
                    pango_nuc_sub_private[pango] = Set{String}()
                    pango_nuc_sub_WT[pango] = Set{String}()
                    pango_new_nuc_muts_pos_only[pango] = Set{String}()
                    sorted_nuc_muts = [all_nuc_AA_sorted_dict[i][1] for i in 1:length(all_nuc_AA_sorted_dict)]
                    for mut in sorted_nuc_muts
                        nuc_mut_int = nuc_mut_int_comprehensive_dict[mut]
                        WT_ref_nuc = string(wuhan_ref_seq[nuc_mut_int])
#                        predecessor_ref_nuc = ref_nuc_comprehensive_dict[mut]
                        predecessor_ref_nuc = ref_nuc_comprehensive_dict[mut]
#                        qry_nuc = qry_nuc_comprehensive_dict[mut]
                        qry_nuc = qry_nuc_comprehensive_dict[mut]
                        nuc_mut_WT = ""
                        if WT_ref_nuc == predecessor_ref_nuc
                            nuc_mut_WT = mut
                        else
                            nuc_mut_WT = "$(WT_ref_nuc)$(nuc_mut_int)$(qry_nuc)"
                        end
                        push!(pango_nuc_sub_private[pango], mut)
                        push!(pango_nuc_sub_WT[pango], nuc_mut_WT)
                        push!(pango_new_nuc_muts_pos_only[pango], nuc_mut_int)
                    end
                end
###################################################################
                sorted_AA_muts = String[]
                if !isempty(nonsynonymous_nuc_AA_sorted_dict)
                    pango_AAsub_private[pango] = Set{String}()
                    pango_AAsub_WT[pango] = Set{String}()
                    pango_new_AA_muts_pos_only[pango] = Set{String}()
                    sorted_AA_muts = [nonsynonymous_nuc_AA_sorted_dict[i][2] for i in 1:length(nonsynonymous_nuc_AA_sorted_dict)]
                    for mut in sorted_AA_muts
                        mutgeen = aa_gene_comprehensive_dict[mut]
                        AAmut_int = aa_pos_comprehensive_dict[mut]
                        AAmut_pos_only = aa_gene_and_pos_comprehensive_dict[mut]
                        refAA = refAA_comprehensive_dict[mut]
                        qryAA = qryAA_comprehensive_dict[mut]
                        new_gene_AA_seq = gene_AA_pango_dict[pango][mutgeen][1:AAmut_int-1]*qryAA*gene_AA_pango_dict[pango][mutgeen][AAmut_int+1:end]
                        gene_AA_pango_dict[pango][mutgeen] = new_gene_AA_seq
#                        pango_new_AA_muts[pango] = Set{String}()
#                        pango_new_AA_muts_WT[pango] = Set{String}()
#                        pango_new_AA_muts_pos_only[pango] = Set{String}()
                        WT_ref_AA = string(gene_AA_pango_dict[predecessor_j][mutgeen][AAmut_int])
                        AA_mut_WT = ""
                        if WT_ref_AA == refAA
                            AA_mut_WT = mut
                        else
                            AA_mut_WT = "$(mutgeen):$(WT_ref_AA)$(AAmut_int)$(qryAA)"
                        end 
                        push!(pango_AAsub_private[pango], mut)
                        push!(pango_AAsub_WT[pango], AA_mut_WT)
                        push!(pango_new_AA_muts_pos_only[pango], AAmut_pos_only)
                    end
                end
##################################################################
                nuc_genome_pango_dict[pango] = nuc_genome_pango_dict[predecessor_j]
                if !isempty(new_muts)
                    for mut in new_muts
                        if mut ≠ ""
                            qryNuc = qry_nuc_comprehensive_dict[mut]
                            nuc_int = nuc_mut_int_comprehensive_dict[mut]
                            new_nuc_seq = nuc_genome_pango_dict[pango][1:nuc_int-1]*qryNuc*nuc_genome_pango_dict[pango][nuc_int+1:end]
                            nuc_genome_pango_dict[pango] = new_nuc_seq
                        end
                    end
                    push!(pango_consensus_set, pango)
                end
                break
            end
        end
    end
end
#for pango in all_effing_pangos
#    if !haskey(nuc_genome_pango_dict, pango)
#        unaliased = pango_to_pango_unaliased_v2[pango]
#        println("No nuc_genome_pango_dict for $(pango) | $(unaliased)")
#    end
#end
#for pango in all_effing_pangos
#    if !haskey(nuc_genome_pango_dict, pango)
#        dot_ct = count('.', unaliased)
#        if dot_ct ≥ 1
#            for i in 1:dot_ct
#                predecessor_i = pango_unaliased_predecessor_meta_dict[pango][i]
#                if haskey(nuc_genome_pango_dict, predecessor_i)
#                    nuc_genome_pango_dict[pango] = nuc_genome_pango_dict[predecessor_i]
#                    break
#                end
#            end
#        end
#    end
#end
############################################################################################################################################################################
############################################################################################################################################################################
############################################################################################################################################################################
#########################################################__END NEW SECTION (2026-01-03)    #################################################################################
############################################################################################################################################################################
############################################################################################################################################################################
############################################################################################################################################################################
############################################################################################################################################################################
############################### 2025-11-13 | Filling pango/clade/unaliased date/ct/country dicts  Runtime = 41.57 seconds ##################################################
############################################################################################################################################################################
date_now = Dates.format(now(), "yyyy_mm_dd_IMMp"); println(date_now)
index_len = length(values(tuple_to_index))
println("tuple_to_index_len = $(index_len)"); println("index_to_tuple[1] = $(index_to_tuple[1])"); println("index_to_tuple[2000] = $(index_to_tuple[2000])")
###########################################################################################################
pango_unaliased_date_index_ct = Dict{String, Dict{Int64, Int64}}()    
#                          clade     date_index  cumulative_count
clade_date_index_cumul = Dict{String, Dict{Int, Int}}()
pango_date_index_cumul = Dict{String, Dict{Int, Int}}()
pango_unaliased_date_index_cumul = Dict{String, Dict{Int, Int}}()
#                                country       clade    date_index  cumulative_count  
#country_clade_date_index_cumul = Dict{String, Dict{String, Dict{Int, Int}}}()
country_pango_date_index_cumul = Dict{String, Dict{String, Dict{Int, Int}}}()
country_pango_unaliased_date_index_cumul = Dict{String, Dict{String, Dict{Int, Int}}}()
#
clade_total = Dict{String, Int}()
pango_total = Dict{String, Int}()
pango_unaliased_total = Dict{String, Int}()
####################################################################################################
AAmut_date_index_cumul = Dict{String, Dict{Int, Int}}()
nucmut_date_index_cumul = Dict{String, Dict{Int, Int}}()
###################################################################################################################################
for clade in chronic_clade_set
    clade_date_index_cumul[clade] = Dict{Int, Int}()
    cumulative_clade_ct = 0
    for date_index in 1:2500
        if !haskey(clade_date_index_ct[clade], date_index)
            clade_date_index_cumul[clade][date_index] = cumulative_clade_ct
        end
        if haskey(clade_date_index_ct[clade], date_index)
            cumulative_clade_ct += clade_date_index_ct[clade][date_index]
            clade_date_index_cumul[clade][date_index] = cumulative_clade_ct
        end
    end
    clade_total[clade] = cumulative_clade_ct
end
################################################################################################################################################################
missing_pangos = Set{String}()
for pango in all_effing_pangos
    if !haskey(pango_date_index_ct, pango)
        push!(missing_pangos, pango)
    end
end     
################################################################################################################################################################
for pango in all_effing_pangos
    if pango in missing_pangos
        continue
    end
    pango_date_index_cumul[pango] = Dict{Int, Int}()
    cumulative_pango_ct = 0
    for date_index in 1:2500
        if !haskey(pango_date_index_ct[pango], date_index)
            pango_date_index_cumul[pango][date_index] = cumulative_pango_ct
        end
        if haskey(pango_date_index_ct[pango], date_index)
            cumulative_pango_ct += pango_date_index_ct[pango][date_index]
            pango_date_index_cumul[pango][date_index] = cumulative_pango_ct
        end
    end
    pango_total[pango] = cumulative_pango_ct
end
################################################################################
for pango in keys(pango_date_index_cumul)
    unaliased = pango_to_pango_unaliased_v2[pango]
    pango_unaliased_date_index_cumul[unaliased] = Dict{Int, Int}()
    for date_index in 1:2500
        pango_unaliased_date_index_cumul[unaliased][date_index] = pango_date_index_cumul[pango][date_index]
    end
    pango_unaliased_total[unaliased] = pango_total[pango]
end
println("Done Filling Pango Date Index Cts!")
################################################################################
clade_pct_date_index = Dict{String, Dict{Float64, Int}}()
clade_pct_date_tuple = Dict{String, Dict{Float64, Tuple{Int, Int, Int}}}()
clade_pct_date_string = Dict{String, Dict{Float64, String}}()
#####
pango_pct_date_index = Dict{String, Dict{Float64, Int}}()
pango_pct_date_tuple = Dict{String, Dict{Float64, Tuple{Int, Int, Int}}}()
pango_pct_date_string = Dict{String, Dict{Float64, String}}()
#####
pango_unaliased_pct_date_index = Dict{String, Dict{Float64, Int}}()
pango_unaliased_pct_date_tuple = Dict{String, Dict{Float64, Tuple{Int, Int, Int}}}()
pango_unaliased_pct_date_string = Dict{String, Dict{Float64, String}}()
######################################################################################################################
for clade in chronic_clade_set
    clade_pct_date_index[clade] = Dict{Float64, Int}()
    clade_pct_date_tuple[clade] = Dict{Float64, Tuple{Int, Int, Int}}()
    clade_pct_date_string[clade] = Dict{Float64, String}()
    cp_total = clade_total[clade]
    pct_date_index = 0
    pct_date_tuple = nothing
#    for pct in 950:999
#        pct = pct/10
    for pct in [1.0, 50.0, 95.0, 99.0]
        for date_index in 1:2500
            cumulative_ct = clade_date_index_cumul[clade][date_index]
            if 100*cumulative_ct/cp_total ≥ pct
                clade_pct_date_index[clade][pct] = date_index
                clade_pct_date_tuple[clade][pct] = index_to_tuple[date_index]
                clade_pct_date_string[clade][pct] = index_to_date_str[date_index]
                break
            end
        end
    end
end
##########################################################
for pango in all_effing_pangos
    if pango in missing_pangos
        continue
    end
    pango_pct_date_index[pango] = Dict{Float64, Int}()
    pango_pct_date_tuple[pango] = Dict{Float64, Tuple{Int, Int, Int}}()
    pango_pct_date_string[pango] = Dict{Float64, String}()
    cp_total = get(pango_total, pango, 0)
    pct_date_index = 0
    pct_date_tuple = nothing
#    for pct in [10.0, 25.0, 50.0, 75.0, 90.0, 95.0, 98.0, 99.0, 99.5, 99.7, 99.8, 99.9]
    for pct in [1.0, 50.0, 95.0, 99.0]
        for date_index in 1:2500
            cumulative_ct = 0
            cumulative_ct = pango_date_index_cumul[pango][date_index]
            if 100*cumulative_ct/cp_total ≥ pct
                pango_pct_date_index[pango][pct] = date_index
                pango_pct_date_tuple[pango][pct] = index_to_tuple[date_index]
                pango_pct_date_string[pango][pct] = index_to_date_str[date_index]
                break
            end
        end
    end
end
println("Done Filling pango_pct_date_index/tuple/string!")
##########################################################
for pango in keys(pango_pct_date_index)
    if pango in missing_pangos
        continue
    end
    unaliased = pango_to_pango_unaliased_v2[pango]
    pango_unaliased_pct_date_index[unaliased] = Dict{Float64, Int}()
    pango_unaliased_pct_date_tuple[unaliased] = Dict{Float64, Tuple{Int, Int, Int}}()
    pango_unaliased_pct_date_string[unaliased] = Dict{Float64, String}()
    cp_total = get(pango_total, pango, 0)
    pct_date_index = 0
    pct_date_tuple = nothing
#    for pct in 950:999
#    for pct in [10.0, 25.0, 50.0, 75.0, 90.0, 95.0, 98.0, 99.0, 99.5, 99.7, 99.8, 99.9]
    for pct in [1.0, 50.0, 95.0, 99.0]
        for date_index in 1:2500
            cumulative_ct = 0
            if pango == "B.1.1.529"
                cumulative_ct = pango_date_index_cumul["BA.2"][date_index]
            elseif pango == "LF.3.1"
                cumulative_ct = pango_date_index_cumul["LF.3"][date_index]
            else
                cumulative_ct = pango_date_index_cumul[pango][date_index]
            end
            if 100*cumulative_ct/cp_total ≥ pct
                pango_unaliased_pct_date_index[unaliased][pct] = date_index
                pango_unaliased_pct_date_tuple[unaliased][pct] = index_to_tuple[date_index]
                pango_unaliased_pct_date_string[unaliased][pct] = index_to_date_str[date_index]
                break
            end
        end
    end
end
println("Done Filling pango_unaliased_pct_date_index/tuple/string!")
########################################################################################################
seq_syn_nucs = Dict{String, Vector{String}}()
seq_noncoding_nucs = Dict{String, Vector{String}}()
for (seq, nuc_set) in seq_nuc_muts
    pango = seq_pango[seq]
#    if !haskey(nuc_genome_pango_dict, pango)
#        println("No nuc_genome_pango_dict key, $(pango)")
#    end
    synonymous_nucmuts, noncoding_nucmuts = SIMPLER_syn_noncoding_nuc(pango, nuc_set)
    seq_syn_nucs[seq] = synonymous_nucmuts
    seq_noncoding_nucs[seq] = noncoding_nucmuts
end
println("Done Filling seq_syn_nucs, seq_noncoding_nucs!")
###########################################################################################################################################################################
all_chr_seqs_pangos = Dict{String,Int}()
all_chr_seqs_inherited = Dict{String,Int}()
all_chr_seqs_inherited_pos_only = Dict{String,Int}()
for seq in all_unique_chr_seqs
    pango = seq_pango[seq]
    all_chr_seqs_pangos[pango] = get(all_chr_seqs_pangos, pango, 0) + 1
    if !haskey(pango_AAsub_WT, pango)
        if haskey(pango_predecessor_meta_dict, pango)
            if haskey(pango_predecessor_meta_dict[pango], 2)
                pango = pango_predecessor_meta_dict[pango][2]
                if !haskey(pango_AAsub_WT, pango)
                    if haskey(pango_predecessor_meta_dict, pango)
                        if haskey(pango_predecessor_meta_dict[pango], 3)
                            pango = pango_predecessor_meta_dict[pango][3]
                        end
                    end
                end
            end
        end
    end
    if pango == "B.1.1.529"
        pango_AAsub_WT[pango] = union(pango_AAsub_WT["BA.1"], pango_AAsub_WT["BA.2"])
        for mut in pango_AAsub_WT[pango]
            all_chr_seqs_inherited[mut] = get(all_chr_seqs_inherited, mut, 0) + 1
        end
    else
        for mut in pango_AAsub_WT[pango]
            all_chr_seqs_inherited[mut] = get(all_chr_seqs_inherited, mut, 0) + 1
        end
    end
end
println("Done Filling all_chr_seqs_pangos, all_chr_seqs_inherited, pango_AAsub_WT (leftovers)!")
############################################################################################################################################################################
############################################################################################################################################################################
nuc_genome_pango_dict["B.1.1.273"] = nuc_genome_pango_dict["B.1.1"]
nuc_genome_pango_dict["B.1.1.53"] = nuc_genome_pango_dict["B.1.1"]
############################################################################################################################################################################
finish = time() - start
finish_rd = round(digits=2, finish)
println("nuc_genome_pango_dict length = $(length(nuc_genome_pango_dict))")
println("Length all_effing_pangos = $(length(all_effing_pangos))")
print("\n"^1)
println("Total time to finish = $(finish_rd)"); print("\n"^1)
println("Finished!")
print("\n"^1)
temp_nuc_sort_key(n) = (length(n), nuc_mut_int_comprehensive_dict[n])
nuc_mut_all_and_chr_set_missing_sort = sort(collect(nuc_mut_all_and_chr_set_missing), by = x -> temp_nuc_sort_key(x))
nuc_mut_all_and_chr_set_missing_sort_join = join(nuc_mut_all_and_chr_set_missing_sort, ", ")
missing_AA_dict_keys_sort = sort(collect(missing_AA_dict_keys), by = x -> AA_gene_sortKey_2(x))
missing_AA_dict_keys_sort_join = join(missing_AA_dict_keys_sort, ", ")
empty_string_pangos_sort = sort(collect(empty_string_pangos))
empty_string_pangos_sort_join = join(empty_string_pangos_sort, ", ")
open("missing_nuc_dict_keys_$(date_now).txt", "w") do g
    println(g, "nuc_mut_all_and_chr_set_missing_sort_join = $(nuc_mut_all_and_chr_set_missing_sort_join)")
    println("nuc_mut_all_and_chr_set_missing_sort_join = $(nuc_mut_all_and_chr_set_missing_sort_join)")
    print("\n"^1)
    print(g, "\n"^1)
    println(g, "missing_AA_dict_keys_sort_join = $(missing_AA_dict_keys_sort_join)")
    println("missing_AA_dict_keys_sort_join = $(missing_AA_dict_keys_sort_join)")
    print("\n"^1)
    print(g, "\n"^1)
    println(g, "empty_string_pangos_sort_join = $(empty_string_pangos_sort_join)")
    println("empty_string_pangos_sort_join = $(empty_string_pangos_sort_join)")
end
print("\n"^1)
############################################################################################################################################################################
############################################################################################################################################################################

2026_02_08_403PM
nuc_genome_pango_dict length = 3586
2026_02_08__403PM
Number of Different Pango Lineages in Chronics = 834
Number of Different Pango_Unaliased Lineages in Chronics = 834
Number of Different Clades in Chronics = 49

PA.4 = B.1.1.529.2.86.1.1.11.1.3.1.1.10.1.1.4
AY.127 = B.1.617.2.127
XBB.1.5.33 = XBB.1.5.33
XES.1 = XES.1
C.17 = B.1.1.1.17
B.1.218 = B.1.218
XBY = XBY
B.1.469 = B.1.469
B.1.1.58 = B.1.1.58
BA.5.2.26 = B.1.1.529.5.2.26
B.1.1.346 = B.1.1.346
MC.36 = B.1.1.529.2.86.1.1.11.1.3.1.1.36
XFG.26 = XFG.26
B.30 = B.30
length(keys(pango_to_pango_unaliased_v2) = 5657
2026_02_08__403PM
2026_02_08__403PM
Done Making Pango Consensus Sequences! (Part 1)

Total Missing Keys for AA_pos_only_gene_and_pos_dict = 1
missing_keys_for_WT_join = ||

OG_tree_pangos_set_len = 4432
NEW_tree_pangos_set_len = 4372
bad_pangos length = 60
AY.43.8_18086, B.1.1.529_dropout, B.1.1.7_dropout, B.1.160.16_13923, BA.1_22877_22878, BA.1_dropout, BA.2.13_revs, BA.2.3.16_alt, BA.2.3.9_rev10198, BA.

LoadError: UndefVarError: `AA_order_key` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

In [80]:
### print_all_seq_info Fx + All gene/AA/nuc/key + synonymous_nuc_to_AA_and_noncoding_to_context #######################
date_now = Dates.format(now(), "yyyy_mm_dd__IMMp"); println(date_now)
for seq in all_unique_chr_seqs_set 
    for del in seq_AA_dels[seq]
        aa_gene_comprehensive_dict[del] = string(split(del, ":")[1])
        firstdel = string(split(del, "-")[1])
        aa_pos_comprehensive_dict[del] = aa_pos_comprehensive_dict[firstdel]
    end
end
######################################################################################################################################
AA_triplets = Dict{String, String}("TTT"=>"F", "TTC"=>"F", "TTA"=>"L", "TTG"=>"L", "TCT"=>"S", "TCC"=>"S", "TCA"=>"S", "TCG"=>"S", "TAT"=>"Y", "TAC"=>"Y", "TAA"=>"*", "TAG"=>"*", "TGT"=>"C", "TGC"=>"C", "TGA"=>"*", "TGG"=>"W", "CTT"=>"L", "CTC"=>"L", "CTA"=>"L", "CTG"=>"L", "CCT"=>"P", "CCC"=>"P", "CCA"=>"P", "CCG"=>"P", "CAT"=>"H", "CAC"=>"H", "CAA"=>"Q", "CAG"=>"Q", "CGT"=>"R", "CGC"=>"R", "CGA"=>"R", "CGG"=>"R", "ATT"=>"I", "ATC"=>"I", "ATA"=>"I", "ATG"=>"M", "ACT"=>"T", "ACC"=>"T", "ACA"=>"T", "ACG"=>"T", "AAT"=>"N", "AAC"=>"N", "AAA"=>"K", "AAG"=>"K", "AGT"=>"S", "AGC"=>"S", "AGA"=>"R", "AGG"=>"R", "GTT"=>"V", "GTC"=>"V", "GTA"=>"V", "GTG"=>"V", "GCT"=>"A", "GCC"=>"A", "GCA"=>"A", "GCG"=>"A", "GAT"=>"D", "GAC"=>"D", "GAA"=>"E", "GAG"=>"E", "GGT"=>"G", "GGC"=>"G", "GGA"=>"G", "GGG"=>"G", "TT-"=>"X", "TC-"=>"X", "TA-"=>"X", "TG-"=>"X", "T-T"=>"X", "T-C"=>"X", "T-A"=>"X", "T-G"=>"X", "T--"=>"X", "CT-"=>"X", "CC-"=>"X", "CA-"=>"X", "CG-"=>"X", "C-T"=>"X", "C-C"=>"X", "C-A"=>"X", "C-G"=>"X", "C--"=>"X", "AT-"=>"X", "AC-"=>"X", "AA-"=>"X", "AG-"=>"X", "A-T"=>"X", "A-C"=>"X", "A-A"=>"X", "A-G"=>"X", "A--"=>"X", "GT-"=>"X", "GC-"=>"X", "GA-"=>"X", "GG-"=>"X", "G-T"=>"X", "G-C"=>"X", "G-A"=>"X", "G-G"=>"X", "G--"=>"X", "-TT"=>"X", "-TC"=>"X", "-TA"=>"X", "-TG"=>"X", "-T-"=>"X", "-CT"=>"X", "-CC"=>"X", "-CA"=>"X", "-CG"=>"X", "-C-"=>"X", "-AT"=>"X", "-AC"=>"X", "-AA"=>"X", "-AG"=>"X", "-A-"=>"X", "-GT"=>"X", "-GC"=>"X", "-GA"=>"X", "-GG"=>"X", "-G-"=>"X", "--T"=>"X", "--C"=>"X", "--A"=>"X", "--G"=>"X", "---"=>"X", "NTT"=>"X", "TNT"=>"X", "TTN"=>"X", "NTC"=>"X", "TNC"=>"X", "TCN"=>"X", "NTA"=>"X", "TNA"=>"X", "TAN"=>"X", "NTG"=>"X", "TNG"=>"X", "TGN"=>"X", "NT-"=>"X", "TN-"=>"X", "T-N"=>"X", "NTN"=>"X", "TNN"=>"X", "NCT"=>"X", "CNT"=>"X", "CTN"=>"X", "NCC"=>"X", "CNC"=>"X", "CCN"=>"X", "NCA"=>"X", "CNA"=>"X", "CAN"=>"X", "NCG"=>"X", "CNG"=>"X", "CGN"=>"X", "NC-"=>"X", "CN-"=>"X", "C-N"=>"X", "NCN"=>"X", "CNN"=>"X", "NAT"=>"X", "ANT"=>"X", "ATN"=>"X", "NAC"=>"X", "ANC"=>"X", "ACN"=>"X", "NAA"=>"X", "ANA"=>"X", "AAN"=>"X", "NAG"=>"X", "ANG"=>"X", "AGN"=>"X", "NA-"=>"X", "AN-"=>"X", "A-N"=>"X", "NAN"=>"X", "ANN"=>"X", "NGT"=>"X", "GNT"=>"X", "GTN"=>"X", "NGC"=>"X", "GNC"=>"X", "GCN"=>"X", "NGA"=>"X", "GNA"=>"X", "GAN"=>"X", "NGG"=>"X", "GNG"=>"X", "GGN"=>"X", "NG-"=>"X", "GN-"=>"X", "G-N"=>"X", "NGN"=>"X", "GNN"=>"X", "N-T"=>"X", "-NT"=>"X", "-TN"=>"X", "N-C"=>"X", "-NC"=>"X", "-CN"=>"X", "N-A"=>"X", "-NA"=>"X", "-AN"=>"X", "N-G"=>"X", "-NG"=>"X", "-GN"=>"X", "N--"=>"X", "-N-"=>"X", "--N"=>"X", "N-N"=>"X", "-NN"=>"X", "NNT"=>"X", "NNC"=>"X", "NNA"=>"X", "NNG"=>"X", "NN-"=>"X", "NNN"=>"X")                 
AA_triplet_dels = Dict{String, String}("TTT"=>"F", "TTC"=>"F", "TTA"=>"L", "TTG"=>"L", "TCT"=>"S", "TCC"=>"S", "TCA"=>"S", "TCG"=>"S", "TAT"=>"Y", "TAC"=>"Y", "TAA"=>"*", "TAG"=>"*", "TGT"=>"C", "TGC"=>"C", "TGA"=>"*", "TGG"=>"W", "CTT"=>"L", "CTC"=>"L", "CTA"=>"L", "CTG"=>"L", "CCT"=>"P", "CCC"=>"P", "CCA"=>"P", "CCG"=>"P", "CAT"=>"H", "CAC"=>"H", "CAA"=>"Q", "CAG"=>"Q", "CGT"=>"R", "CGC"=>"R", "CGA"=>"R", "CGG"=>"R", "ATT"=>"I", "ATC"=>"I", "ATA"=>"I", "ATG"=>"M", "ACT"=>"T", "ACC"=>"T", "ACA"=>"T", "ACG"=>"T", "AAT"=>"N", "AAC"=>"N", "AAA"=>"K", "AAG"=>"K", "AGT"=>"S", "AGC"=>"S", "AGA"=>"R", "AGG"=>"R", "GTT"=>"V", "GTC"=>"V", "GTA"=>"V", "GTG"=>"V", "GCT"=>"A", "GCC"=>"A", "GCA"=>"A", "GCG"=>"A", "GAT"=>"D", "GAC"=>"D", "GAA"=>"E", "GAG"=>"E", "GGT"=>"G", "GGC"=>"G", "GGA"=>"G", "GGG"=>"G", "TT-"=>"X", "TC-"=>"X", "TA-"=>"X", "TG-"=>"X", "T-T"=>"X", "T-C"=>"X", "T-A"=>"X", "T-G"=>"X", "T--"=>"-", "CT-"=>"X", "CC-"=>"X", "CA-"=>"X", "CG-"=>"X", "C-T"=>"X", "C-C"=>"X", "C-A"=>"X", "C-G"=>"X", "C--"=>"-", "AT-"=>"X", "AC-"=>"X", "AA-"=>"X", "AG-"=>"X", "A-T"=>"X", "A-C"=>"X", "A-A"=>"X", "A-G"=>"X", "A--"=>"-", "GT-"=>"X", "GC-"=>"X", "GA-"=>"X", "GG-"=>"X", "G-T"=>"X", "G-C"=>"X", "G-A"=>"X", "G-G"=>"X", "G--"=>"-", "-TT"=>"X", "-TC"=>"X", "-TA"=>"X", "-TG"=>"X", "-T-"=>"X", "-CT"=>"X", "-CC"=>"X", "-CA"=>"X", "-CG"=>"X", "-C-"=>"X", "-AT"=>"X", "-AC"=>"X", "-AA"=>"X", "-AG"=>"X", "-A-"=>"X", "-GT"=>"X", "-GC"=>"X", "-GA"=>"X", "-GG"=>"X", "-G-"=>"X", "--T"=>"-", "--C"=>"-", "--A"=>"-", "--G"=>"-", "---"=>"-", "NTT"=>"X", "TNT"=>"X", "TTN"=>"X", "NTC"=>"X", "TNC"=>"X", "TCN"=>"X", "NTA"=>"X", "TNA"=>"X", "TAN"=>"X", "NTG"=>"X", "TNG"=>"X", "TGN"=>"X", "NT-"=>"X", "TN-"=>"X", "T-N"=>"X", "NTN"=>"X", "TNN"=>"X", "NCT"=>"X", "CNT"=>"X", "CTN"=>"X", "NCC"=>"X", "CNC"=>"X", "CCN"=>"X", "NCA"=>"X", "CNA"=>"X", "CAN"=>"X", "NCG"=>"X", "CNG"=>"X", "CGN"=>"X", "NC-"=>"X", "CN-"=>"X", "C-N"=>"X", "NCN"=>"X", "CNN"=>"X", "NAT"=>"X", "ANT"=>"X", "ATN"=>"X", "NAC"=>"X", "ANC"=>"X", "ACN"=>"X", "NAA"=>"X", "ANA"=>"X", "AAN"=>"X", "NAG"=>"X", "ANG"=>"X", "AGN"=>"X", "NA-"=>"X", "AN-"=>"X", "A-N"=>"X", "NAN"=>"X", "ANN"=>"X", "NGT"=>"X", "GNT"=>"X", "GTN"=>"X", "NGC"=>"X", "GNC"=>"X", "GCN"=>"X", "NGA"=>"X", "GNA"=>"X", "GAN"=>"X", "NGG"=>"X", "GNG"=>"X", "GGN"=>"X", "NG-"=>"X", "GN-"=>"X", "G-N"=>"X", "NGN"=>"X", "GNN"=>"X", "N-T"=>"X", "-NT"=>"X", "-TN"=>"X", "N-C"=>"X", "-NC"=>"X", "-CN"=>"X", "N-A"=>"X", "-NA"=>"X", "-AN"=>"X", "N-G"=>"X", "-NG"=>"X", "-GN"=>"X", "N--"=>"X", "-N-"=>"X", "--N"=>"X", "N-N"=>"X", "-NN"=>"X", "NNT"=>"X", "NNC"=>"X", "NNA"=>"X", "NNG"=>"X", "NN-"=>"X", "NNN"=>"X")                 
######################################################################################################################################
mut_gene_Dict = Dict{String, Int}("ORF1a"=>1, "ORF1b"=>2, "S"=>3, "E"=>4, "M"=>5, "N"=>6, "ORF3a"=>7, "ORF6"=>8, "ORF7a"=>9, "ORF7b"=>10, "ORF8"=>11, "ORF9b"=>12)
mut_gene_Dict2 = Dict{String, Int}("ORF1a"=>11, "ORF1b"=>12, "S"=>1, "E"=>2, "M"=>3, "N"=>4, "ORF3a"=>5, "ORF6"=>6, "ORF7a"=>7, "ORF7b"=>8, "ORF8"=>9, "ORF9b"=>10)
##############################################################################
nuc_del_range_sortkey(n) = parse(Int, split(n, "-")[1])
###########
AA_del_num(n) = parse(Int, split(split(n, "-")[1], ":")[2])
unknown_AA_first_pos(n) = string(split(n, "-")[1])
gene___AAnum_sortkey(n) = [mut_gene_Dict[aa_gene_comprehensive_dict[n]], aa_pos_comprehensive_dict[n]]
gene___AAnum_sortkey2(n) = [mut_gene_Dict2[aa_gene_comprehensive_dict[n]], aa_pos_comprehensive_dict[n]]
gene___AAdel_sortkey2(n) = [mut_gene_Dict2[aa_gene_comprehensive_dict[n]], aa_pos_comprehensive_dict[n]]
gene___AAposonly_sortkey(n) = [mut_gene_Dict[aa_gene_comprehensive_dict[n]], aa_pos_comprehensive_dict[n]]
unknown_AA_ranges_sort(n) = (mut_gene_Dict[aa_gene_comprehensive_dict[unknown_AA_first_pos(n)]], aa_pos_comprehensive_dict[unknown_AA_first_pos(n)])
##############################################################################
N3_syn = ["TCT", "TCC", "TCA", "TCG", "CTT", "CTC", "CTA", "CTG", "CCT", "CCC", "CCA", "CCG", "CGT", "CGC", "CGA", "CGG", "ACT", "ACC", "ACA", "ACG", "GTT", "GTC", "GTA", "GTG", "GCT", "GCC", "GCA", "GCG", "GGT", "GGC", "GGA", "GGG"]
N3_tv = ["TTT", "TTC", "TTA", "TTG", "TAT", "TAC", "TAA", "TAG", "AAT", "AAC", "AAA", "AAG", "AGT", "AGC", "AGA", "AGG", "GAT", "GAC", "GAA", "GAG"]
##############################################################################
EPI_ISL(n) = split(n, "|")[2]
country(n) = split(n, "/")[2]
sequence_date(n) = split(n, "|")[3]
seq_lab(n) = split(n, "/")[3]
US_state(n) = split(split(n, "/")[3], "-")[1]
##############################################################################
seq_nuc_muts["EPI_ISL_6281381"] = Set{String}()
seq_nuc_muts_no_dels = Dict{String, Set{String}}()
for (seq, mut_set) in seq_nuc_muts
    seq_nuc_muts_no_dels[seq] = Set{String}()
    for mut in mut_set
        if qry_nuc_comprehensive_dict[mut] ≠ "-"
            push!(seq_nuc_muts_no_dels[seq], mut)
        end
    end
end      
#####################################################################################################################################
gene_print_array = ["S", "N", "E", "M", "ORF3a", "ORF6", "ORF7a", "ORF7b", "ORF8", "ORF9b", "ORF1a", "ORF1b"]
#####################################################################################################################################
function print_all_seq_info(seq::String, txt_out::String)
    open(txt_out, "a") do g
        println("#"^94); println(g, "#"^94)
        cntry = seq_country[seq]
        pango = seq_pango[seq]
        USA_state = seq_US_state[seq] 
        coll_date = seq_collection_date[seq]
        date_index = seq_date_index[seq]
        lab = seq_lab_dict[seq]
#################################################################################
        aa_muts = seq_AA_muts_no_dels[seq]
        aa_del_ranges = seq_AA_dels[seq]
        nuc_muts = seq_nuc_muts[seq]
        nuc_muts_no_dels = seq_nuc_muts_no_dels[seq]
        nuc_del_ranges = seq_nuc_del_ranges[seq]
        aa_muts_sort = sort(collect(aa_muts), by = x -> gene___AAnum_sortkey(x))
        aa_muts_sort2 = sort(collect(aa_muts), by = x -> gene___AAnum_sortkey2(x))
        aa_del_ranges_sort = sort(collect(aa_del_ranges), by = x -> gene___AAdel_sortkey2(x))
        nuc_muts_sort = sort(collect(nuc_muts), by = x -> nuc_mut_int_comprehensive_dict[x])
        nuc_muts_no_dels_sort = sort(collect(nuc_muts_no_dels), by = x -> nuc_mut_int_comprehensive_dict[x])
        nuc_del_ranges_sort = sort(collect(nuc_del_ranges), by = x -> nuc_del_range_sortkey(x))
        seq_mixed_AA_muts_sort = sort(collect(seq_mixed_AA_muts[seq]), by = x -> gene___AAnum_sortkey(x))
        seq_mixed_nucs_sort = sort(collect(seq_mixed_nucs[seq]), by = x -> nuc_mut_int_comprehensive_dict[x])
        seq_unknown_AA_ranges_sort = sort(collect(seq_unknown_AA_ranges[seq]), by = x -> unknown_AA_ranges_sort(x))
        total_AA_subs_plus_del_ranges = length(aa_muts) + length(aa_del_ranges)
#################################################################################
        mut_vec_dict = Dict{String, Vector{String}}()
        for gene in gene_print_array
            mut_vec_dict[gene] = Vector{String}()
        end
        for mut in aa_muts
            gene = aa_gene_comprehensive_dict[mut]
            mut_only = string(split(mut, ":")[2])
            push!(mut_vec_dict[gene], mut_only) 
        end
        mutonly_sortkey(n) = parse(Int, n[2:end-1])
        for gene in keys(mut_vec_dict)
            if !isempty(mut_vec_dict[gene])
                sort!(mut_vec_dict[gene], by = x -> mutonly_sortkey(x))
            end
        end
#################################################################################
        if seq in rep_seqs
            for (grp_num, seq_set) in rep_seq_grps_seqs
                grp_size = length(seq_set)
                if seq in seq_set
                    seq_set_sort = sort(collect(seq_set), by = x -> (length(x), x))
                    seq_set_sort_join = join(seq_set_sort, ", ")
                    seq_set_sort_join_len = length(seq_set_sort_join)
#                    title_len = length("Group #$(grp_num), $(grp_size) related seqs--")
                        println("Group #$(grp_num), $(grp_size) related sequences")
                    println(g, "Group #$(grp_num), $(grp_size) related sequences")
                    if seq_set_sort_join_len ≤ 84
                           print("       ")
                        print(g, "       ")
                           println(seq_set_sort_join)
                        println(g, seq_set_sort_join)
                    elseif seq_set_sort_join_len > 84
                        start = 0
                        for z in 1:10
                            if seq_set_sort_join_len - start > 84
                                for i in start+85:-1:start+65
                                    if string(seq_set_sort_join[i]) == " "
                                           print("       ")
                                        print(g, "       ")
                                           println(seq_set_sort_join[start+1:i-1])
                                        println(g, seq_set_sort_join[start+1:i-1])
                                        start = i
                                        break
                                    end
                                end
                            end
                            if seq_set_sort_join_len - start ≤ 84
                                   print("       ")
                                print(g, "       ")
                                   println(seq_set_sort_join[start+1:end])
                                println(g, seq_set_sort_join[start+1:end])
                                break
                            end
                        end
                    end
                end
            end     
        else
            println("Singlet")
            println(g, "Singlet")
        end
########################################################################
           println("$(pango)|$(seq)|$(coll_date) | $(cntry)|$(USA_state)|$(lab) | AAsubs+DelRanges = $(total_AA_subs_plus_del_ranges)")
        println(g, "$(pango)|$(seq)|$(coll_date) | $(cntry)|$(USA_state)|$(lab) | AAsubs+DelRanges = $(total_AA_subs_plus_del_ranges)")
           println("----------------------------------- AA Substitutions -------------------------------------")
        println(g, "----------------------------------- AA Substitutions -------------------------------------")
################# Next bit is to figure out how many muts are in a given gene & how much print space they take up
        gene_print_length = Dict{String, Int}("ORF1a"=>0, "ORF1b"=>0, "ORF3a"=>0, "ORF6"=>0, "ORF7a"=>0, "ORF7b"=>0, "ORF8"=>0, "ORF9b"=>0, "S"=>0, "E"=>0, "M"=>0, "N"=>0)
        gene_mut_ct = Dict{String, Int}("ORF1a"=>0, "ORF1b"=>0, "ORF3a"=>0, "ORF6"=>0, "ORF7a"=>0, "ORF7b"=>0, "ORF8"=>0, "ORF9b"=>0, "S"=>0, "E"=>0, "M"=>0, "N"=>0)
        gene_set = Set{String}()
        for mut in aa_muts_sort2
            gene = split(mut, ":")[1]
            push!(gene_set, gene)
            gene_mut_ct[gene] += 1
            gene_print_length[gene] += length(mut) + 2
        end
        for gene in gene_set
            ## +4 is for the "--", the -2 b/c the last mut doesn't have ", " after it, & the +10 for 10 spaces btwn gene muts
            gene_print_length[gene] += length(gene) + 4 - 2
        end
################ Same thing as above but for deletions
        gene_del_print_length = Dict{String, Int}("ORF1a"=>0, "ORF1b"=>0, "ORF3a"=>0, "ORF6"=>0, "ORF7a"=>0, "ORF7b"=>0, "ORF8"=>0, "ORF9b"=>0, "S"=>0, "E"=>0, "M"=>0, "N"=>0)
        gene_AA_del_ct = Dict{String, Int}("ORF1a"=>0, "ORF1b"=>0, "ORF3a"=>0, "ORF6"=>0, "ORF7a"=>0, "ORF7b"=>0, "ORF8"=>0, "ORF9b"=>0, "S"=>0, "E"=>0, "M"=>0, "N"=>0)
        gene_del_set = Set{String}()
        for del in aa_del_ranges_sort
            gene = split(del, ":")[1]
            push!(gene_del_set, gene)
            del_rng = split(del, ":")[2]
            gene_del_print_length[gene] += length(del_rng) + 2
        end
        for gene in gene_del_set
            gene_del_print_length[gene] += length(gene) + 4 - 2
        end
############### Same thing as above but for unknown AA
        gene_unk_print_length = Dict{String, Int}("ORF1a"=>0, "ORF1b"=>0, "ORF3a"=>0, "ORF6"=>0, "ORF7a"=>0, "ORF7b"=>0, "ORF8"=>0, "ORF9b"=>0, "S"=>0, "E"=>0, "M"=>0, "N"=>0)
        gene_AA_unk_ct = Dict{String, Int}("ORF1a"=>0, "ORF1b"=>0, "ORF3a"=>0, "ORF6"=>0, "ORF7a"=>0, "ORF7b"=>0, "ORF8"=>0, "ORF9b"=>0, "S"=>0, "E"=>0, "M"=>0, "N"=>0)
        gene_unk_set = Set{String}()
        for unk in seq_unknown_AA_ranges_sort
            gene = split(unk, ":")[1]
            push!(gene_unk_set, gene)
            unk_rng = split(unk, ":")[2]
            gene_unk_print_length[gene] += length(unk_rng) + 2
        end
        for gene in gene_unk_set
            gene_unk_print_length[gene] += length(gene) + 4 - 2
        end
#####################################################################################################################################
        line_vec_dict = Dict{Int, Int}(1=>0, 2=>0, 3=>0, 4=>0, 5=>0, 6=>0, 7=>0, 8=>0, 9=>0, 10=>0, 11=>0, 12=>0, 13=>0, 14=>0, 15=>0, 16=>0)
        line_length = 0
        previous_gene = "S"
        midline = "no"
        for v in 1:length(gene_print_array)
            gene = gene_print_array[v]
            skip = "no"
            if !isempty(mut_vec_dict[gene])
                muts_joined = join(mut_vec_dict[gene], ", ")
                muts_line_empty = "$(gene)--$(muts_joined)"
                muts_line_nonempty = "     $(gene)--$(muts_joined)"
                muts_line_empty_len = length(muts_line_empty)
                muts_line_nonempty_len = length(muts_line_nonempty)
                if length(previous_gene) < 3 && length(gene) ≥ 4
                    skip = "yes"
                end
                previous_gene = gene
                if line_length > 0 && muts_line_nonempty_len ≤ (90-line_length) && skip == "yes"
                    println(); println(g)
                    print(muts_line_empty); print(g, muts_line_empty)
                    line_length = muts_line_empty_len
                    midline = "yes"
                    continue
                elseif line_length > 0 && muts_line_nonempty_len ≤ (90-line_length) && skip == "no"
                    print(muts_line_nonempty); print(g, muts_line_nonempty)
                    line_length = line_length + muts_line_nonempty_len
                    midline = "yes"
                    continue
                elseif line_length > 0 && muts_line_nonempty_len > (90-line_length) && muts_line_empty_len ≤ 90
                    println(); println(g)
                    print(muts_line_empty); print(g, muts_line_empty)
                    line_length = line_length + muts_line_nonempty_len
                    midline = "yes"
                    continue
                elseif line_length == 0 && muts_line_empty_len ≤ 90
                    print(muts_line_empty); print(g, muts_line_empty)
                    line_length = muts_line_empty_len
                    midline = "yes"
                    continue
                elseif line_length > 0 && muts_line_nonempty_len > (90-line_length) && muts_line_empty_len > 90
                    println(); println(g)
                    line_length = 0
                    midline = "no"
                    for i in 77:91
                        if string(muts_line_empty[i]) == " "
                            println(muts_line_empty[1:i-1]); println(g, muts_line_empty[1:i-1])
                            line_vec_dict[1] = i
                            print("      "); print(g, "      ") 
                            break
                        end
                    end
                    current_pos = line_vec_dict[1]
                    for j in 2:length(line_vec_dict)
                        line = line_vec_dict[j]
                        if muts_line_empty_len - current_pos ≤ 84
                            println(muts_line_empty[current_pos+1:end]); println(g, muts_line_empty[current_pos+1:end])
                            line_length = 0
                            midline = "no"
                            break
                        end
                        if muts_line_empty_len - current_pos > 84
                            for i in current_pos+71:current_pos+85
                                if string(muts_line_empty[i]) == " "
                                    println(muts_line_empty[current_pos+1:i-1]); println(g, muts_line_empty[current_pos+1:i-1])
                                    midline = "no"
                                    if v < length(gene_print_array) && !all(isempty(mut_vec_dict[gene_print_array[q]]) for q in v+1:length(gene_print_array))
                                        print("      "); print(g, "      ")
                                    end
                                    current_pos = i
                                    break
                                end
                            end
                        end
                    end
                elseif line_length == 0 && muts_line_empty_len > 90
                    for i in 77:91
                        if string(muts_line_empty[i]) == " "
                            println(muts_line_empty[1:i-1]); println(g, muts_line_empty[1:i-1])
                            line_vec_dict[1] = i
                            print("      "); print(g, "      ") 
                            break
                        end
                    end
                    current_pos = line_vec_dict[1]
                    for j in 2:length(line_vec_dict)
                        line = line_vec_dict[j]
                        if muts_line_empty_len - current_pos ≤ 84
                            println(muts_line_empty[current_pos+1:end]); println(g, muts_line_empty[current_pos+1:end])
                            line_length = 0
                            midline = "no"
                            break
                        end
                        if muts_line_empty_len - current_pos > 84
                            for i in current_pos+71:current_pos+85
                                if string(muts_line_empty[i]) == " "
                                    println(muts_line_empty[current_pos+1:i-1]); println(g, muts_line_empty[current_pos+1:i-1])
                                    midline = "no"
                                    if v < length(gene_print_array) && !all(isempty(mut_vec_dict[gene_print_array[q]]) for q in v+1:length(gene_print_array))
                                        print("      "); print(g, "      ")
                                    end
                                    current_pos = i
                                    break
                                end
                            end
                        end
                    end
                end
            end
        end
        if midline == "yes"
            println(); println(g)
        end
           println("------------------------------------------------------------------------------------------")
        println(g, "------------------------------------------------------------------------------------------")
##########################################################################################
#        print("\n"^1); print(g, "\n"^1)
        if !isempty(aa_del_ranges_sort)
            aadel_caps_pad = rpad("AA DELETIONS", 14)
            print("$(aadel_caps_pad)| ")
            print(g, "$(aadel_caps_pad)| ")
            line_AA_del_print_ct = 20
            for i in 1:length(aa_del_ranges_sort)
                del = aa_del_ranges_sort[i]
                gene = split(del, ":")[1]
                del2 = split(del, ":")[2]
                if length(aa_del_ranges_sort) == 1
                    println("$(gene)--$(del2)")
                    println(g, "$(gene)--$(del2)")
                end
                if length(aa_del_ranges_sort) > 1
                    if i == 1
                        if mut_gene(del) == mut_gene(aa_del_ranges_sort[i+1])
                            print("$(gene)--$(del2), ")
                            print(g, "$(gene)--$(del2), ")
                            line_AA_del_print_ct = length(gene) + 4 + length(del2) + 2
                        elseif mut_gene(del) ≠ mut_gene(aa_del_ranges_sort[i+1])
                            print("$(gene)--$(del2)")
                            print(g, "$(gene)--$(del2)")
                            line_AA_del_print_ct = length(gene) + 4 + length(del2)
                        end
                    end
                    if i ≠ 1 && i ≠ length(aa_del_ranges_sort)
                        if mut_gene(del) == mut_gene(aa_del_ranges_sort[i-1]) && mut_gene(del) == mut_gene(aa_del_ranges_sort[i+1])
                            line_AA_del_print_ct += length(del2) + 2
                            if line_AA_del_print_ct > 102 
                                println(); println(g)
                                print("          $(del2), ")
                                print(g, "          $(del2), ")
                                line_AA_del_print_ct = 10 + length(del2) + 2
                            else
                                print("$(del2), ")
                                print(g, "$(del2), ")
                            end
                        elseif mut_gene(del) == mut_gene(aa_del_ranges_sort[i-1]) && mut_gene(del) ≠ mut_gene(aa_del_ranges_sort[i+1])
                            line_AA_del_print_ct += length(del2)
                            if line_AA_del_print_ct > 102
                                println(); println(g)
                                println("          $(del2)")
                                println(g, "          $(del2)")
                                line_AA_del_print_ct = length(del2)
                            else
                                print("$(del2)")
                                print(g, "$(del2)")
                            end
                        elseif mut_gene(del) ≠ mut_gene(aa_del_ranges_sort[i-1]) && mut_gene(del) == mut_gene(aa_del_ranges_sort[i+1])
                            if line_AA_del_print_ct + gene_del_print_length[gene]+ 10 > 102
                                println(); println(g)
                                print("          $(gene)--$(del2), ")
                                print(g, "          $(gene)--$(del2), ")
                                line_AA_del_print_ct = 10 + length(gene) + 4 + length(del2) + 2
                            else
                                print("          $(gene)--$(del2), ")
                                print(g, "          $(gene)--$(del2), ")
                                line_AA_del_print_ct += 10 + length(gene) + 4 + length(del2) + 2
                            end
                        elseif mut_gene(del) ≠ mut_gene(aa_del_ranges_sort[i-1]) && mut_gene(del) ≠ mut_gene(aa_del_ranges_sort[i+1])
                            if line_AA_del_print_ct + gene_del_print_length[gene]+ 10 > 102
                                println(); println(g)
                                print("          $(gene)--$(del2)")
                                print(g, "          $(gene)--$(del2)")
                                line_AA_del_print_ct = 10 + length(gene) + 4 + length(del2)
                            else
                                print("          $(gene)--$(del2)")
                                print(g, "          $(gene)--$(del2)")
                                line_AA_del_print_ct += length(gene) + 4 + length(del2) + 10
                            end
                        end
                    end
                    if i == length(aa_del_ranges_sort)
                        if mut_gene(del) == mut_gene(aa_del_ranges_sort[i-1])
                            println("$(del2)")
                            println(g, "$(del2)")
                        elseif line_AA_del_print_ct + gene_del_print_length[gene] > 102
                            println(); println(g)
                            println("          $(gene)--$(del2)")
                            println(g, "          $(gene)--$(del2)")
                        else
                            println("          $(gene)--$(del2)")
                            println(g, "          $(gene)--$(del2)")
                        end
                    end
                end
            end
        end
#        println(); println(g)
##############################################################################################
##############################################################################################
        if !isempty(seq_mixed_AA_muts_sort)
            mixed_caps_pad = rpad("MIXED AA MUTS", 14)
            print("$(mixed_caps_pad)| ")
            print(g, "$(mixed_caps_pad)| ")
            mixed_print_ct = 21
            for i in 1:length(seq_mixed_AA_muts_sort)
                mix = seq_mixed_AA_muts_sort[i]
                mix1 = split(mix, ":")[1]
                mix2 = split(mix, ":")[2]
                if length(seq_mixed_AA_muts_sort) == 1
                    println("$(mix1)--$(mix2)")
                    println(g, "$(mix1)--$(mix2)")
                end
                if length(seq_mixed_AA_muts_sort) > 1
                    if i == 1
                        if aa_gene_comprehensive_dict[mix] == aa_gene_comprehensive_dict[seq_mixed_AA_muts_sort[i+1]]
                            print("$(mix1)--$(mix2), ")
                            print(g, "$(mix1)--$(mix2), ")
                        elseif aa_gene_comprehensive_dict[mix] ≠ aa_gene_comprehensive_dict[seq_mixed_AA_muts_sort[i+1]]
                            print("$(mix1)--$(mix2)")
                            print(g, "$(mix1)--$(mix2)")
                        end
                    end
                    if i ≠ 1 && i ≠ length(seq_mixed_AA_muts_sort)
                        if aa_gene_comprehensive_dict[mix] == aa_gene_comprehensive_dict[seq_mixed_AA_muts_sort[i-1]] && aa_gene_comprehensive_dict[mix] == aa_gene_comprehensive_dict[seq_mixed_AA_muts_sort[i+1]]
                            print("$(mix2), ")
                            print(g, "$(mix2), ")
                        elseif aa_gene_comprehensive_dict[mix] == aa_gene_comprehensive_dict[seq_mixed_AA_muts_sort[i-1]] && aa_gene_comprehensive_dict[mix] ≠ aa_gene_comprehensive_dict[seq_mixed_AA_muts_sort[i+1]]
                            println("$(mix2)")
                            println(g, "$(mix2)")
                        elseif aa_gene_comprehensive_dict[mix] ≠ aa_gene_comprehensive_dict[seq_mixed_AA_muts_sort[i-1]] && aa_gene_comprehensive_dict[mix] == aa_gene_comprehensive_dict[seq_mixed_AA_muts_sort[i+1]]
                            print("    $(mix1)--$(mix2), ")
                            print(g, "    $(mix1)--$(mix2), ")
                        elseif aa_gene_comprehensive_dict[mix] ≠ aa_gene_comprehensive_dict[seq_mixed_AA_muts_sort[i-1]] && aa_gene_comprehensive_dict[mix] ≠ aa_gene_comprehensive_dict[seq_mixed_AA_muts_sort[i+1]]
                            println("    $(mix1)--$(mix2)")
                            println(g, "    $(mix1)--$(mix2)")
                        end
                    end
                    if i == length(seq_mixed_AA_muts_sort)
                        if aa_gene_comprehensive_dict[mix] == aa_gene_comprehensive_dict[seq_mixed_AA_muts_sort[i-1]]
                            println("$(mix2)")
                            println(g, "$(mix2)")
                        else
                            println("    $(mix1)--$(mix2)   ")
                            println(g, "    $(mix1)--$(mix2)   ")
                        end
                    end
                end
            end
        end
##############################################################################################
        if !isempty(seq_unknown_AA_ranges_sort)
            aadrop_caps_pad = rpad("AA DROPOUT", 14)
            print("$(aadrop_caps_pad)| ")
            print(g, "$(aadrop_caps_pad)| ")
            line_unk_print_ct = 18
            for i in 1:length(seq_unknown_AA_ranges_sort)
                unk = seq_unknown_AA_ranges_sort[i]
                gene = split(unk, ":")[1]
                unk2 = split(unk, ":")[2]
                if length(seq_unknown_AA_ranges_sort) == 1
                    println("$(gene)--$(unk2)")
                    println(g, "$(gene)--$(unk2)")
                end
                if length(seq_unknown_AA_ranges_sort) > 1
                    if i == 1
                        if mut_gene(unk) == mut_gene(seq_unknown_AA_ranges_sort[i+1])
                            print("$(gene)--$(unk2), ")
                            print(g, "$(gene)--$(unk2), ")
                            line_unk_print_ct = length(gene) + 4 + length(unk2) + 2
                        elseif mut_gene(unk) ≠ mut_gene(seq_unknown_AA_ranges_sort[i+1])
                            print("$(gene)--$(unk2)")
                            print(g, "$(gene)--$(unk2)")
                            line_unk_print_ct = length(gene) + 4 + length(unk2)
                        end
                    end
                    if i ≠ 1 && i ≠ length(seq_unknown_AA_ranges_sort)
                        if mut_gene(unk) == mut_gene(seq_unknown_AA_ranges_sort[i-1]) && mut_gene(unk) == mut_gene(seq_unknown_AA_ranges_sort[i+1])
                            line_unk_print_ct += length(unk2) + 2
                            if line_unk_print_ct > 102 
                                println(); println(g)
                                print("      $(unk2), ")
                                print(g, "      $(unk2), ")
                                line_unk_print_ct = 10 + length(unk2) + 2
                            else
                                print("$(unk2), ")
                                print(g, "$(unk2), ")
                            end
                        elseif mut_gene(unk) == mut_gene(seq_unknown_AA_ranges_sort[i-1]) && mut_gene(unk) ≠ mut_gene(seq_unknown_AA_ranges_sort[i+1])
                            line_unk_print_ct += length(unk2)
                            if line_unk_print_ct > 102
                                println(); println(g)
                                println("      $(unk2)")
                                println(g, "      $(unk2)")
                                line_unk_print_ct = 0
                            else
                                print("$(unk2)")
                                print(g, "$(unk2)")
                            end
                        elseif mut_gene(unk) ≠ mut_gene(seq_unknown_AA_ranges_sort[i-1]) && mut_gene(unk) == mut_gene(seq_unknown_AA_ranges_sort[i+1])
                            if line_unk_print_ct + gene_unk_print_length[gene] + 10 > 102
                                println();  println(g)
                                print("      $(gene)--$(unk2), ")
                                print(g, "      $(gene)--$(unk2), ")
                                line_unk_print_ct = 10 + length(gene) + 4 + length(unk2) + 2
                            else
                                print("      $(gene)--$(unk2), ")
                                print(g, "      $(gene)--$(unk2), ")
                                line_unk_print_ct += length(gene) + 4 + length(unk2) + 2 + 10
                            end
                        elseif mut_gene(unk) ≠ mut_gene(seq_unknown_AA_ranges_sort[i-1]) && mut_gene(unk) ≠ mut_gene(seq_unknown_AA_ranges_sort[i+1])
                            if line_unk_print_ct + gene_unk_print_length[gene] + 10 > 102
                                println(); println(g)
                                print("      $(gene)--$(unk2)")
                                print(g, "      $(gene)--$(unk2)")
                                line_unk_print_ct = 10 + length(gene) + 4 + length(unk2)
                            else
                                print("      $(gene)--$(unk2)")
                                print(g, "      $(gene)--$(unk2)")
                                line_unk_print_ct += length(gene) + 4 + length(unk2) + 10
                            end
                        end
                    end
                    if i == length(seq_unknown_AA_ranges_sort)
                        if mut_gene(unk) == mut_gene(seq_unknown_AA_ranges_sort[i-1])
                            println("$(unk2)")
                            println(g, "$(unk2)")
                        elseif line_unk_print_ct + gene_unk_print_length[gene] > 102
                            println(); println(g)
                            println("    $(gene)--$(unk2)")
                            println(g, "    $(gene)--$(unk2)")
                        else
                            println("      $(gene)--$(unk2)")
                            println(g, "      $(gene)--$(unk2)")
                        end
                    end
                end
            end
            println(); println(g)
        end
###############################################################################################################################
        print("NUC MUTATIONS--")
        print(g, "NUC MUTATIONS--")
        nuc_muts_join = join(nuc_muts_no_dels_sort, ", ")
        nuc_mut_len = length(nuc_muts_join)
        line_length = 0
#        line1 = 0; line2 = 0; line3 = 0; line4 = 0; line5 = 0; line6 = 0; line7 = 0; line8 = 0
#        line9 = 0; line10 = 0; line11 = 0; line12 = 0; line13 = 0; line14 = 0; line15 = 0; line16 = 0
#        line_vec = [line1, line2, line3, line4, line5, line6, line7, line8, line9, line10, line11, line12, line13, line14, line15, line16]
        line_vec_dict = Dict{Int, Int}(1=>0, 2=>0, 3=>0, 4=>0, 5=>0, 6=>0, 7=>0, 8=>0, 9=>0, 10=>0, 11=>0, 12=>0, 13=>0, 14=>0, 15=>0, 16=>0)
        done = "no"
        if nuc_mut_len ≤ 75
               println(nuc_muts_join)
            println(g, nuc_muts_join)
            done = "yes"
        end
        if nuc_mut_len > 75 && done == "no"
            for j in 69:77
                if string(nuc_muts_join[j]) == " "
                       println(nuc_muts_join[1:(j-1)])
                    println(g, nuc_muts_join[1:(j-1)])
                    line_vec_dict[1] = j
                    break
                end
            end
        end
        for i in 1:length(line_vec_dict)
            line = line_vec_dict[i]
            if nuc_mut_len - line ≤ 84 && done == "no"
                   print("      ")
                print(g, "      ")
                   println(nuc_muts_join[(line+1):end])
                println(g, nuc_muts_join[(line+1):end])
                done = "yes"
                break
            end
            if nuc_mut_len - line > 84 && done == "no"
                   print("      ")
                print(g, "      ")
                for j in line+77:line+85
                    if string(nuc_muts_join[j]) == " "
                           println(nuc_muts_join[line+1:j-1])
                        println(g, nuc_muts_join[line+1:j-1])
                        if i ≠ length(line_vec_dict)
                            line_vec_dict[i+1] = j
                        end
                    end
                end
            end
        end
##############################################################################################
        if !isempty(nuc_del_ranges_sort)
            nucdel_caps = "NUC DELETIONS--"
            print("$(nucdel_caps)")
            print(g, "$(nucdel_caps)")
            print_length = 16
            for i in 1:length(nuc_del_ranges_sort)
                nuc_del = nuc_del_ranges_sort[i]
                print_length += length(nuc_del) + 2
                if print_length > 102
                    if i == 1
                        println("$(nuc_del), ")
                        println(g, "$(nuc_del), ")
                        print_length += 8
                    end
                    if i ≠ length(nuc_del_ranges_sort) && !(i == 1)
                        println("        $(nuc_del), ")
                        println(g, "        $(nuc_del), ")
                        print_length += 8
                    elseif !(i == 1)
                        println("        $(nuc_del)")
                        println(g, "        $(nuc_del)")
                        print_length += 8
                    end
                else
                    if i ≠ length(nuc_del_ranges_sort)
                        print("$(nuc_del), ")
                        print(g, "$(nuc_del), ")
                    else
                        println("$(nuc_del)")
                        println(g, "$(nuc_del)")
                    end
                end
            end
        end
        seqnucmutsnodels = collect(seq_nuc_muts_no_dels[seq])
        synonymous_nuc__AA_sort, synonymous_nuc__context_sort, noncoding__noncoding_region_sort, noncoding_nuc__context_sort = synonymous_nuc_to_AA_and_noncoding_to_context(seq_pango[seq], seqnucmutsnodels)
#        synonymous_nucs_vec = String[]
#        synonymous_AA_vec = String[]
#        synonymous_context_vec = String[]
######################################################################################################
        if !isempty(seq_mixed_nucs_sort)
            mixed_nucs_pad = rpad("MIXED NUCS", 14)
               print("$(mixed_nucs_pad)| ")
            print(g, "$(mixed_nucs_pad)| ")
            seq_mixed_nucs_sort_join = join(seq_mixed_nucs_sort, ", ")
               println(seq_mixed_nucs_sort_join)
            println(g, seq_mixed_nucs_sort_join)
            println(); println(g)
        end
##############################################################################################
############# Below = SHORT version of syn nuc print, WITHOUT AA positions and nuc context
        if !isempty(synonymous_nuc__AA_sort)
            total_syn = length(synonymous_nuc__AA_sort)
            println("SYNONYMOUS NUC MUTATIONS - Total:$(total_syn)")
            syn_nuc_mut_vec = String[]
            for i in 1:length(synonymous_nuc__AA_sort)
                synnuc = synonymous_nuc__AA_sort[i][1]
                push!(syn_nuc_mut_vec, synnuc)
            end
            syn_nuc_string = join(syn_nuc_mut_vec, ", ")
            println("     $syn_nuc_string")
############# Below = LONG version of syn nuc print, WITH AA positions & nuc context
#            for i in 1:length(synonymous_nuc__AA_sort)
#                synnuc = synonymous_nuc__AA_sort[i][1]
#                synnuc_pad = rpad(synnuc, 7)
#                synAA = synonymous_nuc__AA_sort[i][2]
#                AAlen = length(synAA)
#                pad1 = (14 - AAlen)÷2
#                pad12 = " "^pad1*synAA
#                synAA_pad = rpad(pad12, 14)
#                syncontext = synonymous_nuc__context_sort[i][2]
#                premut_context = split(syncontext, "-->")[1]
#                postmut_context = split(syncontext, "-->")[2]
#                postpad = lpad(postmut_context, 38)
#                println("$(synnuc_pad)|$(synAA_pad)|$(premut_context)")
#                println(postpad)
#                println(g, "$(synnuc_pad)|$(synAA_pad)|$(premut_context)")
#                println(g, postpad)
#            end
            println(); println(g)
        end
######################################################################################################
        if !isempty(noncoding__noncoding_region_sort)
            nc_len = length(noncoding__noncoding_region_sort)
            println("NONCODING NUC MUTATIONS - Total:$(nc_len)")
            for i in 1:length(noncoding__noncoding_region_sort)
                nc_nuc = noncoding__noncoding_region_sort[i][1]
                nc_nuc_pad = rpad(nc_nuc, 7)
                nc_region = noncoding__noncoding_region_sort[i][2]
                nc_region_len = length(nc_region)
                ncpad1 = (13 - nc_region_len)÷2
                ncpad12 = " "^ncpad1*nc_region
                nc_region_pad2 = rpad(ncpad12, 13)
                nc_context = noncoding_nuc__context_sort[i][2]
                premut_context = split(nc_context, "-->")[1]
                postmut_context = split(nc_context, "-->")[2]
                postpad = lpad(postmut_context, 39)
                println("$(nc_nuc_pad)|$(nc_region_pad2)|$(premut_context)")
                println(postpad)
                println(g, "$(nc_nuc_pad)|$(nc_region_pad2)|$(premut_context)")
                println(g, postpad)
            end
        end
        println("#"^94); println(g, "#"^94)
##############################################################################################
    end
end
############################################################################################################################################################################
############################################################################################################################################################################
############################################################################################################################################################################
############################################################################################################################################################################
############################################################################################################################################################################
############################################################################################################################################################################
############################################################################################################################################################################
############################################################################################################################################################################
############################################################################################################################################################################
############################################################################################################################################################################
### function synonymous_nuc_to_AA_and_noncoding_to_context #########################
date_now = Dates.format(now(), "yyyy_mm_dd__IMMp"); println(date_now)
function synonymous_nuc_to_AA_and_noncoding_to_context(ref_pango::String, muts::Vector{String})
#    all_muts_sort = sort(muts, by = x -> nuc_mut_int_comprehensive_dict[x])
    ref_seq, refAA_ORF1a, refAA_ORF1b, refAA_S, refAA_ORF3a, refAA_E, refAA_M, refAA_ORF6, refAA_ORF7a, refAA_ORF7b, refAA_ORF8, refAA_N, refAA_ORF9b = get_ref_pango_nucseq_and_geneseqs(ref_pango)
#########################################################################################################################
    coding_ranges = BitSet([266:13467..., 13469:21555..., 21563:25384..., 25393:26220..., 26245:26472..., 26523:27191..., 27202:27387..., 27394:27755..., 27760:27887..., 27894:28259..., 28274:29533..., 28284:28577...])
    noncoding_ranges = BitSet([1:265..., 21556:21562..., 25385:25392..., 26221:26244..., 26473:26522..., 27192:27201..., 27388:27393..., 27888:27893..., 28260:28273..., 29534:29830...])
    coding_range_9b = BitSet([28284:28577...])
    gene_nuc_starts = Dict{Int, Int}(0=>263, 1=>13465, 2=>21560, 3=>25390, 4=>26242, 5=>26520, 6=>27199, 7=>27391, 8=>27753, 9=>27891, 10=>28271, 11=>28281)
    ref_AA_genes = Dict{Int, String}(0=>refAA_ORF1a, 1=>refAA_ORF1b, 2=>refAA_S, 3=>refAA_ORF3a, 4=>refAA_E, 5=>refAA_M, 6=>refAA_ORF6, 7=>refAA_ORF7a, 8=>refAA_ORF7b, 9=>refAA_ORF8, 10=>refAA_N, 11=>refAA_ORF9b)
    gene_strings = Dict{Int, String}(0=>"ORF1a", 1=>"ORF1b", 2=>"S", 3=>"ORF3a", 4=>"E", 5=>"M", 6=>"ORF6", 7=>"ORF7a", 8=>"ORF7b", 9=>"ORF8", 10=>"N", 11=>"ORF9b")
    nuc_gene_num = Dict{Int, Int}()
    nuc_gene_num_9b = Dict{Int, Int}()
    synonymous_nuc_to_AA_dict = Dict{String, String}()
    synonymous_nuc_to_context_dict = Dict{String, String}()
    noncoding_nuc_to_context_dict = Dict{String, String}()
    noncoding_to_noncoding_region_dict = Dict{String, String}()
    noncoding_range_dict = Dict{Vector{Int}, String}([1, 265]=>"5' UTR", [21556, 21562]=>"Spike TRS", [25385, 25392]=>"ORF3a TRS", [26221, 26234]=>"ORF3a-E UTR", [26235, 26244]=>"E TRS", [26473, 26522]=>"E-M UTR", [27192, 27201]=>"M-ORF6 UTR", [27388, 27393]=>"ORF7a TRS", [27888, 27893]=>"ORF8 TRS", [28260, 28273]=>"N/ORF9b TRS", [29534, 29830]=>"3' UTR", [29794, 29801]=>"ONM")
    noncoding_nuc_vector = Vector{String}()
#########################################################################################################################
    gene_nuc_arr = [[266:13467...], [13469:21555...], [21563:25384...], [25393:26220...], [26245:26472...], [26523:27191...], [27202:27387...], [27394:27755...], [27760:27887...], [27894:28259...], [28274:29533...], [28284:28577...]]
    for i in 1:length(gene_nuc_arr)-1
        for nucpos in gene_nuc_arr[i]
            nuc_gene_num[nucpos] = i-1
        end
    end
    for nucpos in gene_nuc_arr[end]
        nuc_gene_num_9b[nucpos] = 11
    end
    rem0_gene = [5, 8, 9, 11]
    rem1_gene = [1, 3, 4, 6, 7]
    rem2_gene = [0, 2, 10]
    rem0 = BitSet([26523:27191..., 27760:27887..., 27894:28259..., 28284:28577...])
    rem1 = BitSet([13469:21555..., 25393:26220..., 26245:26472..., 27202:27387..., 27394:27755...])
    rem2 = BitSet([266:13467..., 21563:25384..., 28274:29533...])
    rem9b = BitSet([28284:28577...])
    rem7ab = BitSet([27756:27759...])
######################################################################################################################################
    gene_num(nuc_mut) = nuc_gene_num[nuc_mut_int_comprehensive_dict[nuc_mut]]                                           ## Fx ##
    nuc_to_AA_pos(nuc_mut) = string((nuc_mut_int_comprehensive_dict[nuc_mut] - gene_nuc_starts[gene_num(nuc_mut)])÷3)   ## Fx ##
    nuc_to_AA_pos_9b(nuc_mut) = string((nuc_mut_int_comprehensive_dict[nuc_mut] - 28281)÷3)                             ## Fx ##
    nuc2AA_ORF1a(nuc_mut,refAA,qryAA) = "$(gene_strings[gene_num(nuc_mut)]):$(refAA*nuc_to_AA_pos(nuc_mut))$qryAA" ### FUNCTION #####################
    nuc2AA_ORF9b(nuc_mut, refAA, qryAA) = "ORF9b:$(refAA)$(nuc_to_AA_pos_9b(nuc_mut))$(qryAA)"   ## Fx ##
######################################################################################################################################
    nuc_codon_pos_dict = Dict{Int, Int}()
    for nucpos in coding_ranges
        gene_number = nuc_gene_num[nucpos]
        gene_start = gene_nuc_starts[gene_number]
        codon_num = (nucpos-gene_start)%3 + 1
        nuc_codon_pos_dict[nucpos] = codon_num
    end
    nuc_codon_pos_dict_9b = Dict{Int, Int}()
    for nucpos in coding_range_9b
        gene_number = 11
        gene_start = gene_nuc_starts[gene_number]
        codon_num = (nucpos-gene_start)%3 + 1
        nuc_codon_pos_dict_9b[nucpos] = codon_num
    end
#########################################################################
#    muts_v2 = Vector{String}()
    for nuc_mut in muts
        mut = mixed2nuc(nuc_mut)
        if ',' in mut
            mut1 = string(split(mut, ", ")[1])
            mut2 = string(split(mut, ", ")[2])
            push!(muts, mut1)
            push!(muts, mut2)
            filter!(x -> !(length(x)>9), muts)
        end
    end
######################################################################### 
    coding_ranges = BitSet([266:13467..., 13468:21555..., 21563:25384..., 25393:26220..., 26245:26472..., 26523:27191..., 27202:27387..., 27394:27755..., 27760:27887..., 27894:28259..., 28274:29533...])
    N_9b_synonymous = Set(["C28379A", "C28394A", "T28406C", "C28475A", "C28535A", "A28547C"])
    AA_mut_set = Set{String}()
    AA_mut = ""
    for nuc_mut in muts
        pos = nuc_mut_int_comprehensive_dict[nuc_mut]
        if pos in coding_ranges
#            mut = mixed2nuc(nuc_mut)  
            mut = nuc_mut
            gene_number = gene_num(mut)
            ref_triple = ""
            qry_triple = ""
            ref_triple_context = ""
            qry_triple_context = ""
            ref2qry_context = ""
            if nuc_codon_pos_dict[pos] == 1
                ref_triple = ref_nuc_comprehensive_dict[mut]*string(ref_seq[pos+1])  *string(ref_seq[pos+2])
                qry_triple = qry_nuc_comprehensive_dict[mut]*string(ref_seq[pos+1])*string(ref_seq[pos+2])
                ref_triple_context = string(ref_seq[pos-6])*string(ref_seq[pos-5])*string(ref_seq[pos-4])*string(ref_seq[pos-3])*string(ref_seq[pos-2])*string(ref_seq[pos-1])*ref_triple*string(ref_seq[pos+3])*string(ref_seq[pos+4])*string(ref_seq[pos+5])*string(ref_seq[pos+6])*string(ref_seq[pos+7])*string(ref_seq[pos+8])
                qry_triple_context = string(ref_seq[pos-6])*string(ref_seq[pos-5])*string(ref_seq[pos-4])*string(ref_seq[pos-3])*string(ref_seq[pos-2])*string(ref_seq[pos-1])*qry_triple*string(ref_seq[pos+3])*string(ref_seq[pos+4])*string(ref_seq[pos+5])*string(ref_seq[pos+6])*string(ref_seq[pos+7])*string(ref_seq[pos+8])
            elseif nuc_codon_pos_dict[pos] == 2
                ref_triple = string(ref_seq[pos-1])*ref_nuc_comprehensive_dict[mut]  *string(ref_seq[pos+1])
                qry_triple = string(ref_seq[pos-1])*qry_nuc_comprehensive_dict[mut]*string(ref_seq[pos+1])
                ref_triple_context = string(ref_seq[pos-7])*string(ref_seq[pos-6])*string(ref_seq[pos-5])*string(ref_seq[pos-4])*string(ref_seq[pos-3])*string(ref_seq[pos-2])*ref_triple*string(ref_seq[pos+2])*string(ref_seq[pos+3])*string(ref_seq[pos+4])*string(ref_seq[pos+5])*string(ref_seq[pos+6])*string(ref_seq[pos+7])
                qry_triple_context = string(ref_seq[pos-7])*string(ref_seq[pos-6])*string(ref_seq[pos-5])*string(ref_seq[pos-4])*string(ref_seq[pos-3])*string(ref_seq[pos-2])*qry_triple*string(ref_seq[pos+2])*string(ref_seq[pos+3])*string(ref_seq[pos+4])*string(ref_seq[pos+5])*string(ref_seq[pos+6])*string(ref_seq[pos+7])
            elseif nuc_codon_pos_dict[pos] == 3
                ref_triple = string(ref_seq[pos-2])*string(ref_seq[pos-1])*ref_nuc_comprehensive_dict[mut]
                qry_triple = string(ref_seq[pos-2])*string(ref_seq[pos-1])*qry_nuc_comprehensive_dict[mut]
                ref_triple_context = string(ref_seq[pos-8])*string(ref_seq[pos-7])*string(ref_seq[pos-6])*string(ref_seq[pos-5])*string(ref_seq[pos-4])*string(ref_seq[pos-3])*ref_triple*string(ref_seq[pos+1])*string(ref_seq[pos+2])*string(ref_seq[pos+3])*string(ref_seq[pos+4])*string(ref_seq[pos+5])*string(ref_seq[pos+6])
                qry_triple_context = string(ref_seq[pos-8])*string(ref_seq[pos-7])*string(ref_seq[pos-6])*string(ref_seq[pos-5])*string(ref_seq[pos-4])*string(ref_seq[pos-3])*qry_triple*string(ref_seq[pos+1])*string(ref_seq[pos+2])*string(ref_seq[pos+3])*string(ref_seq[pos+4])*string(ref_seq[pos+5])*string(ref_seq[pos+6])
            end

            refAA = AA_triplets[ref_triple]
            qryAA = AA_triplets[qry_triple]
            AA_mut = nuc2AA_ORF1a(mut, refAA, qryAA)
            ref2qry_context = ref_triple_context*"-->"*qry_triple_context
            if refAA == qryAA && !(pos in rem9b)
                synonymous_nuc_to_AA_dict[mut] = AA_mut
                synonymous_nuc_to_context_dict[mut] = ref2qry_context
            elseif refAA == qryAA && pos in rem9b && mut in N_9b_synonymous
                synonymous_nuc_to_AA_dict[mut] = AA_mut
                synonymous_nuc_to_context_dict[mut] = ref2qry_context
            else
                push!(AA_mut_set, AA_mut)
            end
###################################
            for nuc_mut2 in muts
                mut2 = nuc_mut2
                pos2 = nuc_mut_int_comprehensive_dict[mut2]
                if pos2 in coding_ranges
                    gene_number2 = gene_num(mut2)
                    if mut ≠ mut2 && gene_number == gene_number2 && nuc_to_AA_pos(mut) == nuc_to_AA_pos(mut2)
                        if nuc_codon_pos_dict[pos] == 1 && nuc_codon_pos_dict[pos2] == 2
                            ref_triple = ref_nuc_comprehensive_dict[mut]*ref_nuc_comprehensive_dict[mut2]*string(ref_seq[pos+2])
                            qry_triple = qry_nuc_comprehensive_dict[mut]*qry_nuc_comprehensive_dict[mut2]*string(ref_seq[pos+2])
                            ref_triple_context = string(ref_seq[pos-3])*string(ref_seq[pos-2])*string(ref_seq[pos-1])*ref_triple*string(ref_seq[pos+3])*string(ref_seq[pos+4])*string(ref_seq[pos+5])
                            qry_triple_context = string(ref_seq[pos-3])*string(ref_seq[pos-2])*string(ref_seq[pos-1])*qry_triple*string(ref_seq[pos+3])*string(ref_seq[pos+4])*string(ref_seq[pos+5])
                        elseif nuc_codon_pos_dict[pos] == 1 && nuc_codon_pos_dict[pos2] == 3
                            ref_triple = ref_nuc_comprehensive_dict[mut]*string(ref_seq[pos+1])*ref_nuc_comprehensive_dict[mut2]
                            qry_triple = qry_nuc_comprehensive_dict[mut]*string(ref_seq[pos+1])*qry_nuc_comprehensive_dict[mut2]
                            ref_triple_context = string(ref_seq[pos-3])*string(ref_seq[pos-2])*string(ref_seq[pos-1])*ref_triple*string(ref_seq[pos+3])*string(ref_seq[pos+4])*string(ref_seq[pos+5])
                            qry_triple_context = string(ref_seq[pos-3])*string(ref_seq[pos-2])*string(ref_seq[pos-1])*qry_triple*string(ref_seq[pos+3])*string(ref_seq[pos+4])*string(ref_seq[pos+5])
                        elseif nuc_codon_pos_dict[pos] == 2 && nuc_codon_pos_dict[pos2] == 3
                            ref_triple = string(ref_seq[pos-1])*ref_nuc_comprehensive_dict[mut]*ref_nuc_comprehensive_dict[mut2]
                            qry_triple = string(ref_seq[pos-1])*qry_nuc_comprehensive_dict[mut]*qry_nuc_comprehensive_dict[mut2]
                            ref_triple_context = string(ref_seq[pos-4])*string(ref_seq[pos-3])*string(ref_seq[pos-2])*ref_triple*string(ref_seq[pos+2])*string(ref_seq[pos+3])*string(ref_seq[pos+4])
                            qry_triple_context = string(ref_seq[pos-4])*string(ref_seq[pos-3])*string(ref_seq[pos-2])*qry_triple*string(ref_seq[pos+2])*string(ref_seq[pos+3])*string(ref_seq[pos+4])
                        elseif nuc_codon_pos_dict[pos2] == 1 && nuc_codon_pos_dict[pos] == 2
                            ref_triple = ref_nuc_comprehensive_dict[mut2]*ref_nuc_comprehensive_dict[mut]*string(ref_seq[pos+1])
                            qry_triple = qry_nuc_comprehensive_dict[mut2]*qry_nuc_comprehensive_dict[mut]*string(ref_seq[pos+1])
                            ref_triple_context = string(ref_seq[pos-4])*string(ref_seq[pos-3])*string(ref_seq[pos-2])*ref_triple*string(ref_seq[pos+2])*string(ref_seq[pos+3])*string(ref_seq[pos+4])
                            qry_triple_context = string(ref_seq[pos-4])*string(ref_seq[pos-3])*string(ref_seq[pos-2])*qry_triple*string(ref_seq[pos+2])*string(ref_seq[pos+3])*string(ref_seq[pos+4])
                        elseif nuc_codon_pos_dict[pos2] == 1 && nuc_codon_pos_dict[pos] == 3
                            ref_triple = ref_nuc_comprehensive_dict[mut2]*string(ref_seq[pos-1])*ref_nuc_comprehensive_dict[mut]
                            qry_triple = qry_nuc_comprehensive_dict[mut2]*string(ref_seq[pos-1])*qry_nuc_comprehensive_dict[mut]
                            ref_triple_context = string(ref_seq[pos-5])*string(ref_seq[pos-4])*string(ref_seq[pos-3])*ref_triple*string(ref_seq[pos+1])*string(ref_seq[pos+2])*string(ref_seq[pos+3])
                            qry_triple_context = string(ref_seq[pos-5])*string(ref_seq[pos-4])*string(ref_seq[pos-3])*qry_triple*string(ref_seq[pos+1])*string(ref_seq[pos+2])*string(ref_seq[pos+3])
                        elseif nuc_codon_pos_dict[pos2] == 2 && nuc_codon_pos_dict[pos] == 3
                            ref_triple = string(ref_seq[pos-2])*ref_nuc_comprehensive_dict[mut2]*ref_nuc_comprehensive_dict[mut]
                            qry_triple = string(ref_seq[pos-2])*qry_nuc_comprehensive_dict[mut2]*qry_nuc_comprehensive_dict[mut]
                        end
                        refAA2 = AA_triplets[ref_triple]
                        qryAA2 = AA_triplets[qry_triple]
                        AA_mut2 = nuc2AA_ORF1a(mut2, refAA2, qryAA2)
                        ref2qry_context = ref_triple_context*"-->"*qry_triple_context
                        if refAA2 == qryAA2 && !(pos2 in rem9b)
                            synonymous_nuc_to_AA_dict[mut2] = AA_mut2
                            synonymous_nuc_to_context_dict[mut2] = ref2qry_context
                        else
                            push!(AA_mut_set, AA_mut2)
                        end
                    end
                end
            end
        else
            npos = nuc_mut_int_comprehensive_dict[nuc_mut]
            if npos < 29890
                qry_nuc = qry_nuc_comprehensive_dict[nuc_mut]
                ref_nc_nuc_context = string(ref_seq[npos-8])*string(ref_seq[npos-7])*string(ref_seq[npos-6])*string(ref_seq[npos-5])*string(ref_seq[npos-4])*string(ref_seq[npos-3])*string(ref_seq[npos-2])*string(ref_seq[npos-1])*string(ref_seq[npos])*string(ref_seq[npos+1])*string(ref_seq[npos+2])*string(ref_seq[npos+3])*string(ref_seq[npos+4])*string(ref_seq[npos+5])*string(ref_seq[npos+6])*string(ref_seq[npos+7])*string(ref_seq[npos+8])
                qry_nc_nuc_context = string(ref_seq[npos-8])*string(ref_seq[npos-7])*string(ref_seq[npos-6])*string(ref_seq[npos-5])*string(ref_seq[npos-4])*string(ref_seq[npos-3])*string(ref_seq[npos-2])*string(ref_seq[npos-1])*qry_nuc*string(ref_seq[npos+1])*string(ref_seq[npos+2])*string(ref_seq[npos+3])*string(ref_seq[npos+4])*string(ref_seq[npos+5])*string(ref_seq[npos+6])*string(ref_seq[npos+7])*string(ref_seq[npos+8])
                full_nc_context = ref_nc_nuc_context*"-->"*qry_nc_nuc_context
                noncoding_nuc_to_context_dict[nuc_mut] = full_nc_context
                for (start_end, place) in noncoding_range_dict
                    frst = start_end[1]
                    last = start_end[2]
                    if npos ≥ frst && npos ≤ last
                        noncoding_to_noncoding_region_dict[nuc_mut] = place
                        mut_vec = [nuc_mut, place]
                        push!(noncoding_nuc_vector, nuc_mut)
                    end
                end
            end
        end
    end
#########################################################################################################
    for nuc_mut in muts
        pos_9b = nuc_mut_int_comprehensive_dict[nuc_mut]
        if pos_9b in rem9b
#            mut_9b = mixed2nuc(nuc_mut)
            mut_9b = nuc_mut
            pos_9b = nuc_mut_int_comprehensive_dict[mut_9b]   
            gene_number_9b = 11
            ref_triple_9b = ""
            qry_triple_9b = ""
            ref_triple_context_9b = ""
            qry_triple_context_9b = ""
            ref2qry_context_9b = ""
            if nuc_codon_pos_dict_9b[pos_9b] == 1
                ref_triple_9b = ref_nuc_comprehensive_dict[mut_9b]*string(ref_seq[pos_9b+1])*string(ref_seq[pos_9b+2])
                qry_triple_9b = qry_nuc_comprehensive_dict[mut_9b]*string(ref_seq[pos_9b+1])*string(ref_seq[pos_9b+2])
                ref_triple_context_9b = string(ref_seq[pos_9b-6])*string(ref_seq[pos_9b-5])*string(ref_seq[pos_9b-4])*string(ref_seq[pos_9b-3])*string(ref_seq[pos_9b-2])*string(ref_seq[pos_9b-1])*ref_triple_9b*string(ref_seq[pos_9b+3])*string(ref_seq[pos_9b+4])*string(ref_seq[pos_9b+5])*string(ref_seq[pos_9b+6])*string(ref_seq[pos_9b+7])*string(ref_seq[pos_9b+8])
                qry_triple_context_9b = string(ref_seq[pos_9b-6])*string(ref_seq[pos_9b-5])*string(ref_seq[pos_9b-4])*string(ref_seq[pos_9b-3])*string(ref_seq[pos_9b-2])*string(ref_seq[pos_9b-1])*qry_triple_9b*string(ref_seq[pos_9b+3])*string(ref_seq[pos_9b+4])*string(ref_seq[pos_9b+5])*string(ref_seq[pos_9b+6])*string(ref_seq[pos_9b+7])*string(ref_seq[pos_9b+8])
            elseif nuc_codon_pos_dict_9b[pos_9b] == 2
                ref_triple_9b = string(ref_seq[pos_9b-1])*ref_nuc_comprehensive_dict[mut_9b]  *string(ref_seq[pos_9b+1])
                qry_triple_9b = string(ref_seq[pos_9b-1])*qry_nuc_comprehensive_dict[mut_9b]*string(ref_seq[pos_9b+1])
                ref_triple_context_9b = string(ref_seq[pos_9b-7])*string(ref_seq[pos_9b-6])*string(ref_seq[pos_9b-5])*string(ref_seq[pos_9b-4])*string(ref_seq[pos_9b-3])*string(ref_seq[pos_9b-2])*ref_triple_9b*string(ref_seq[pos_9b+2])*string(ref_seq[pos_9b+3])*string(ref_seq[pos_9b+4])*string(ref_seq[pos_9b+5])*string(ref_seq[pos_9b+6])*string(ref_seq[pos_9b+7])
                qry_triple_context_9b = string(ref_seq[pos_9b-7])*string(ref_seq[pos_9b-6])*string(ref_seq[pos_9b-5])*string(ref_seq[pos_9b-4])*string(ref_seq[pos_9b-3])*string(ref_seq[pos_9b-2])*qry_triple_9b*string(ref_seq[pos_9b+2])*string(ref_seq[pos_9b+3])*string(ref_seq[pos_9b+4])*string(ref_seq[pos_9b+5])*string(ref_seq[pos_9b+6])*string(ref_seq[pos_9b+7])
            elseif nuc_codon_pos_dict_9b[pos_9b] == 3
                ref_triple_9b = string(ref_seq[pos_9b-2])*string(ref_seq[pos_9b-1])*ref_nuc_comprehensive_dict[mut_9b]
                qry_triple_9b = string(ref_seq[pos_9b-2])*string(ref_seq[pos_9b-1])*qry_nuc_comprehensive_dict[mut_9b]
                ref_triple_context_9b = string(ref_seq[pos_9b-8])*string(ref_seq[pos_9b-7])*string(ref_seq[pos_9b-6])*string(ref_seq[pos_9b-5])*string(ref_seq[pos_9b-4])*string(ref_seq[pos_9b-3])*ref_triple_9b*string(ref_seq[pos_9b+1])*string(ref_seq[pos_9b+2])*string(ref_seq[pos_9b+3])*string(ref_seq[pos_9b+4])*string(ref_seq[pos_9b+5])*string(ref_seq[pos_9b+6])
                qry_triple_context_9b = string(ref_seq[pos_9b-8])*string(ref_seq[pos_9b-7])*string(ref_seq[pos_9b-6])*string(ref_seq[pos_9b-5])*string(ref_seq[pos_9b-4])*string(ref_seq[pos_9b-3])*qry_triple_9b*string(ref_seq[pos_9b+1])*string(ref_seq[pos_9b+2])*string(ref_seq[pos_9b+3])*string(ref_seq[pos_9b+4])*string(ref_seq[pos_9b+5])*string(ref_seq[pos_9b+6])
            end
            refAA_9b = AA_triplets[ref_triple_9b]
            qryAA_9b = AA_triplets[qry_triple_9b]
            AA_mut_9b = nuc2AA_ORF9b(mut_9b, refAA_9b, qryAA_9b)
            ref2qry_context_9b = ref_triple_context_9b*"-->"*qry_triple_context_9b
            if refAA_9b == qryAA_9b && nuc_mut in N_9b_synonymous
                synonymous_nuc_to_AA_dict[mut_9b] = AA_mut_9b
                synonymous_nuc_to_context_dict[mut_9b] = ref2qry_context_9b
            else
                push!(AA_mut_set, AA_mut_9b)
            end
#############################################################################
            for nuc_mut2_9b in muts
#                mut2_9b = mixed2nuc(nuc_mut2_9b)
                mut2_9b = nuc_mut2_9b
                pos2_9b = nuc_mut_int_comprehensive_dict[mut2_9b]
                if pos2_9b in rem9b
                    gene_number2_9b = 11
                    if mut_9b ≠ mut2_9b && nuc_to_AA_pos_9b(mut_9b) == nuc_to_AA_pos_9b(mut2_9b)
                        if nuc_codon_pos_dict_9b[pos_9b] == 1 && nuc_codon_pos_dict_9b[pos2_9b] == 2
                            ref_triple_9b = ref_nuc_comprehensive_dict[mut_9b]*ref_nuc_comprehensive_dict[mut2_9b]*string(ref_seq[pos_9b+2])
                            qry_triple_9b = qry_nuc_comprehensive_dict[mut_9b]*qry_nuc_comprehensive_dict[mut2_9b]*string(ref_seq[pos_9b+2])
                            ref_triple_context_9b = string(ref_seq[pos_9b-3])*string(ref_seq[pos_9b-2])*string(ref_seq[pos_9b-1])*ref_triple_9b*string(ref_seq[pos_9b+3])*string(ref_seq[pos_9b+4])*string(ref_seq[pos_9b+5])
                            qry_triple_context_9b = string(ref_seq[pos_9b-3])*string(ref_seq[pos_9b-2])*string(ref_seq[pos_9b-1])*qry_triple_9b*string(ref_seq[pos_9b+3])*string(ref_seq[pos_9b+4])*string(ref_seq[pos_9b+5])
                        elseif nuc_codon_pos_dict_9b[pos_9b] == 1 && nuc_codon_pos_dict_9b[pos2_9b] == 3
                            ref_triple_9b = ref_nuc_comprehensive_dict[mut_9b]*string(ref_seq[pos_9b+1])*ref_nuc_comprehensive_dict[mut2_9b]
                            qry_triple_9b = qry_nuc_comprehensive_dict[mut_9b]*string(ref_seq[pos_9b+1])*qry_nuc_comprehensive_dict[mut2_9b]
                            ref_triple_context_9b = string(ref_seq[pos_9b-3])*string(ref_seq[pos_9b-2])*string(ref_seq[pos_9b-1])*ref_triple_9b*string(ref_seq[pos_9b+3])*string(ref_seq[pos_9b+4])*string(ref_seq[pos_9b+5])
                            qry_triple_context_9b = string(ref_seq[pos_9b-3])*string(ref_seq[pos_9b-2])*string(ref_seq[pos_9b-1])*qry_triple_9b*string(ref_seq[pos_9b+3])*string(ref_seq[pos_9b+4])*string(ref_seq[pos_9b+5])
                        elseif nuc_codon_pos_dict_9b[pos_9b] == 2 && nuc_codon_pos_dict_9b[pos2_9b] == 3
                            ref_triple_9b = string(ref_seq[pos_9b-1])*ref_nuc_comprehensive_dict[mut_9b]*ref_nuc_comprehensive_dict[mut2_9b]
                            qry_triple_9b = string(ref_seq[pos_9b-1])*qry_nuc_comprehensive_dict[mut_9b]*qry_nuc_comprehensive_dict[mut2_9b]
                            ref_triple_context_9b = string(ref_seq[pos_9b-4])*string(ref_seq[pos_9b-3])*string(ref_seq[pos_9b-2])*ref_triple_9b*string(ref_seq[pos_9b+2])*string(ref_seq[pos_9b+3])*string(ref_seq[pos_9b+4])
                            qry_triple_context_9b = string(ref_seq[pos_9b-4])*string(ref_seq[pos_9b-3])*string(ref_seq[pos_9b-2])*qry_triple_9b*string(ref_seq[pos_9b+2])*string(ref_seq[pos_9b+3])*string(ref_seq[pos_9b+4])
                        elseif nuc_codon_pos_dict_9b[pos2_9b] == 1 && nuc_codon_pos_dict_9b[pos_9b] == 2
                            ref_triple_9b = ref_nuc_comprehensive_dict[mut2_9b]*ref_nuc_comprehensive_dict[mut_9b]*string(ref_seq[pos_9b+1])
                            qry_triple_9b = qry_nuc_comprehensive_dict[mut2_9b]*qry_nuc_comprehensive_dict[mut_9b]*string(ref_seq[pos_9b+1])
                            ref_triple_context_9b = string(ref_seq[pos_9b-4])*string(ref_seq[pos_9b-3])*string(ref_seq[pos_9b-2])*ref_triple_9b*string(ref_seq[pos_9b+2])*string(ref_seq[pos_9b+3])*string(ref_seq[pos_9b+4])
                            qry_triple_context_9b = string(ref_seq[pos_9b-4])*string(ref_seq[pos_9b-3])*string(ref_seq[pos_9b-2])*qry_triple_9b*string(ref_seq[pos_9b+2])*string(ref_seq[pos_9b+3])*string(ref_seq[pos_9b+4])
                        elseif nuc_codon_pos_dict_9b[pos2_9b] == 1 && nuc_codon_pos_dict_9b[pos_9b] == 3
                            ref_triple_9b = ref_nuc_comprehensive_dict[mut2_9b]*string(ref_seq[pos_9b-1])*ref_nuc_comprehensive_dict[mut_9b]
                            qry_triple_9b = qry_nuc_comprehensive_dict[mut2_9b]*string(ref_seq[pos_9b-1])*qry_nuc_comprehensive_dict[mut_9b]
                            ref_triple_context_9b = string(ref_seq[pos_9b-5])*string(ref_seq[pos_9b-4])*string(ref_seq[pos_9b-3])*ref_triple_9b*string(ref_seq[pos_9b+1])*string(ref_seq[pos_9b+2])*string(ref_seq[pos_9b+3])
                            qry_triple_context_9b = string(ref_seq[pos_9b-5])*string(ref_seq[pos_9b-4])*string(ref_seq[pos_9b-3])*qry_triple_9b*string(ref_seq[pos_9b+1])*string(ref_seq[pos_9b+2])*string(ref_seq[pos_9b+3])
                        elseif nuc_codon_pos_dict_9b[pos2_9b] == 2 && nuc_codon_pos_dict_9b[pos_9b] == 3
                            ref_triple_9b = string(ref_seq[pos_9b-2])*ref_nuc_comprehensive_dict[mut2_9b]*ref_nuc_comprehensive_dict[mut_9b]
                            qry_triple_9b = string(ref_seq[pos_9b-2])*qry_nuc_comprehensive_dict[mut2_9b]*qry_nuc_comprehensive_dict[mut_9b]
                        end
                        refAA2_9b = AA_triplets[ref_triple_9b]
                        qryAA2_9b = AA_triplets[qry_triple_9b]
                        AA_mut2_9b = nuc2AA_ORF9b(mut2_9b, refAA2_9b, qryAA2_9b)
                        ref2qry_context_9b = ref_triple_context_9b*"-->"*qry_triple_context_9b
                        if refAA2_9b == qryAA2_9b && nuc_mut2_9b in N_9b_synonymous 
                            synonymous_nuc_to_AA_dict[mut2_9b] = AA_mut2_9b
                            synonymous_nuc_to_context_dict[mut2_9b] = ref2qry_context_9b
                            synonymous_nuc_to_AA_dict[mut_9b] = AA_mut2_9b
                            synonymous_nuc_to_context_dict[mut_9b] = ref2qry_context_9b
                        else
                            push!(AA_mut_set, AA_mut2_9b)
                        end
                    end
                end
            end
        end
    end
#####################################################################################################################################
#    for mut in AA_mut_set
#        if !(mut in aa_mut_all_and_chr_set)
#            push!(missing_gene_pos_AA_keys, mut)
#        end
#    end
#    AA_sort = sort(collect(AA_mut_set), by = x -> AA_order_key(x))
#    AA_sort2 = Vector{String}()
#    for mut in AA_sort
#        refAA = ref_AA_dict[mut]
#        qryAA = qry_AA_dict[mut]
#        if !(refAA == qryAA)
#            push!(AA_sort2, mut)
#        end
#    end
#####################################################################################################################################
    synonymous_nuc_to_AA_dict_sort = sort(collect(synonymous_nuc_to_AA_dict), by = x -> nuc_mut_int_comprehensive_dict[x[1]])
    synonymous_nuc_sort = sort(collect(keys(synonymous_nuc_to_AA_dict)), by = x -> nuc_mut_int_comprehensive_dict[x])
    synonymous_AA_sort = sort(collect(values(synonymous_nuc_to_AA_dict)), by = x -> AA_gene_sortKey_2(x))
    synonymous_nuc_to_context_dict_sort = sort(collect(synonymous_nuc_to_context_dict), by = x -> nuc_mut_int_comprehensive_dict[x[1]])
    noncoding_nuc_to_context_dict_sort = sort(collect(noncoding_nuc_to_context_dict), by = x -> nuc_mut_int_comprehensive_dict[x[1]])
    noncoding_nuc_vector_sort = sort(collect(noncoding_nuc_vector), by = x -> nuc_mut_int_comprehensive_dict[x])
    noncoding_to_noncoding_region_dict_sort = sort(collect(noncoding_to_noncoding_region_dict), by = x -> nuc_mut_int_comprehensive_dict[x[1]])
############################################################
#    println("################## Noncoding Nuc Mutations ##################")
#    noncoding_nuc_total = length(keys(noncoding_nuc_to_context_dict))
#    println("    Total Number of Noncoding Nucs = $(noncoding_nuc_total)")
#    for nc_mut in noncoding_nuc_to_context_dict_sort
#        print("     $(nc_mut[1]) ($(nc_mut[2])), ")
#    end
#    println()
#    for i in 1:length(noncoding_nuc_to_context_dict_sort)
#        println("$(noncoding_nuc_to_context_dict_sort[i][1]) | $(noncoding_nuc_to_context_dict_sort[i][2])")
#        premut_nucseq = split(noncoding_nuc_to_context_dict_sort[i][2], "|")[1]
#        postmut_nucseq = split(noncoding_nuc_to_context_dict_sort[i][2], "|")[2]
#        println("    $(premut_nucseq)")
#        println("    $(postmut_nucseq)")
#    end
#    println("################# Synonymous Nuc Mutations #################")
#    synonymous_nuc_total = length(synonymous_nuc_sort)
#    println("  Total Number of Synonymous Nuc Muts = $(synonymous_nuc_total)")
#    synonymous_nuc_sort_join = join(synonymous_nuc_sort, ", ")
#    println("  ", synonymous_nuc_sort_join)
#    print("\n"^1)
#    for i in 1:length(synonymous_nuc_sort)
#        premut_nucseq = split(synonymous_nuc_to_context_dict_sort[i][2], "-->")[1]
#        postmut_nucseq = split(synonymous_nuc_to_context_dict_sort[i][2], "-->")[2]
#        postmut_cntxt_pad = lpad(postmut_nucseq, 13)
#        nucpad = rpad(synonymous_nuc_to_AA_dict_sort[i][1], 7)
#        println("     $(nucpad)|$(premut_nucseq)")
#        println("$(postmut_cntxt_pad)")
#    end
#    print("\n"^1)
#synonymous_nuc_to_context_dict[mut] = ref_triple_context*"-->"*qry_triple_context
    return synonymous_nuc_to_AA_dict_sort, synonymous_nuc_to_context_dict_sort, noncoding_to_noncoding_region_dict_sort, noncoding_nuc_to_context_dict_sort
end
#############################################################################

2026_02_28__1242PM
2026_02_28__1242PM


synonymous_nuc_to_AA_and_noncoding_to_context (generic function with 1 method)

In [7]:
### Calculates and makes dictionaries for the density of mutations in EPCI and HQCS datasets and the ratios between the two for each gene/protein
date_now = Dates.format(now(), "yyyy_mm_dd__IMMp"); println(date_now)
start_time_post_load_cell = time()
######################################################################################################################################
gene_protein_order = Dict{String, Int}("NSP1"=>1, "NSP2"=>2, "NSP3"=>3, "NSP4"=>4, "NSP5"=>5, "NSP6"=>6, "NSP7"=>7, "NSP8"=>8, "NSP9"=>9, "NSP10"=>10, "NSP12"=>12, "NSP13"=>13, "NSP14"=>14, "NSP15"=>15, "NSP16"=>16, "ORF3a"=>17, "ORF6"=>18, "ORF7a"=>19, "ORF7b"=>20, "ORF8"=>21, "ORF9b"=>22, "S"=>23, "E"=>24, "M"=>25, "N"=>26)

### domain_order **WITH** RBD (added 2025-08-31)
domain_order = Dict{String, Int}("NSP3_Ubl1"=>1, "NSP3_HVR"=>2, "NSP3_Mac1"=>3, "NSP3_Mac2"=>4, "NSP3_Mac3"=>5, "NSP3_DPUP"=>6, "NSP3_Ubl2"=>7, "NSP3_PLpro"=>8, "NSP3_NAB"=>9, "NSP3_BSM"=>10, "NSP3_TM1"=>11, "NSP3_Ecto3"=>12, "NSP3_TM234HLX"=>13, "NSP3_Y1"=>14, "NSP3_CoVY"=>15, "NSP4_TM1"=>16, "NSP4_Ecto4"=>17, "NSP4_TM2_TM6"=>18, "NSP4_CTD"=>19, "NSP6_AmphHlx"=>20, "NSP6_MAE"=>21, "NSP6_cyto_CTD"=>22, "NSP12_NiRAN"=>23, "NSP12_intrfce"=>24, "NSP12_fingers"=>25, "NSP12_palm"=>26, "NSP12_palmLnk"=>27, "NSP12_thumb"=>28, "NSP13_ZBD"=>29, "NSP13_stalk"=>30, "NSP13_1B"=>31, "NSP13_RecA1"=>32, "NSP13_RecA2"=>33, "NSP14_nsp10"=>34, "NSP14_EXON"=>35, "NSP14_hinge1"=>36, "NSP14_hinge2"=>37, "NSP14_N7MTase"=>38, "NSP15_NTD"=>39, "NSP15_MD"=>40, "NSP15_endoU"=>41, "S_S1"=>42, "S_S2"=>43, "S_NTD"=>44, "S_N2R"=>45, "S_RBD"=>46, "S_RBM"=>47, "S_SD1"=>48, "S_SD2"=>49, "S_630_loop"=>50, "S_FCS_region"=>51, "S_Beta1"=>52, "S_3H"=>53, "S_IL770"=>54, "S_FPPR"=>55, "S_FP"=>56, "S_HR1"=>57, "S_CH"=>58, "S_CD"=>59, "S_Beta2"=>60, "S_2turnHelix"=>61, "S_HR2"=>62, "S_TM"=>63, "S_CT"=>64, "ORF3a_SignalP"=>65, "ORF3a_NTD"=>66, "ORF3a_TM1"=>67, "ORF3a_TM12Lnk"=>68, "ORF3a_TM2"=>69, "ORF3a_TM3"=>70, "ORF3a_cytosl1"=>71, "ORF3a_Loop"=>72, "ORF3a_3DB"=>73, "ORF3a_CTD"=>74, "E_TM"=>75, "E_cytosol"=>76, "E_CTD"=>77, "N_N1"=>78, "N_N2"=>79, "N_N3"=>80, "N_N4"=>81, "N_N5"=>82, "N_SR"=>83, "N_L_helix"=>84, "N_CBP"=>85, "N_9b_overlap"=>86)    

### domain_order **WITHOUT** RBD (added 2025-08-31)
#domain_order = Dict{String, Int}("NSP3_Ubl1"=>1, "NSP3_HVR"=>2, "NSP3_Mac1"=>3, "NSP3_Mac2"=>4, "NSP3_Mac3"=>5, "NSP3_DPUP"=>6, "NSP3_Ubl2"=>7, "NSP3_PLpro"=>8, "NSP3_NAB"=>9, "NSP3_BSM"=>10, "NSP3_TM1"=>11, "NSP3_Ecto3"=>12, "NSP3_TM234HLX"=>13, "NSP3_Y1"=>14, "NSP3_CoVY"=>15, "NSP4_TM1"=>16, "NSP4_Ecto4"=>17, "NSP4_TM2_TM6"=>18, "NSP4_CTD"=>19, "NSP6_AmphHlx"=>20, "NSP6_MAE"=>21, "NSP6_cyto_CTD"=>22, "NSP12_NiRAN"=>23, "NSP12_intrfce"=>24, "NSP12_fingers"=>25, "NSP12_palm"=>26, "NSP12_palmLnk"=>27, "NSP12_thumb"=>28, "NSP13_ZBD"=>29, "NSP13_stalk"=>30, "NSP13_1B"=>31, "NSP13_RecA1"=>32, "NSP13_RecA2"=>33, "NSP14_nsp10"=>34, "NSP14_EXON"=>35, "NSP14_hinge1"=>36, "NSP14_hinge2"=>37, "NSP14_N7MTase"=>38, "NSP15_NTD"=>39, "NSP15_MD"=>40, "NSP15_endoU"=>41, "S_S1"=>42, "S_S2"=>43, "S_NTD"=>44, "S_N2R"=>45, "S_RBD"=>46, "S_SD1"=>47, "S_SD2"=>48, "S_630_loop"=>49, "S_FCS_region"=>50, "S_Beta1"=>51, "S_3H"=>52, "S_IL770"=>53, "S_FPPR"=>54, "S_FP"=>55, "S_HR1"=>56, "S_CH"=>57, "S_CD"=>58, "S_Beta2"=>59, "S_2turnHelix"=>60, "S_HR2"=>61, "S_TM"=>62, "S_CT"=>63, "ORF3a_SignalP"=>64, "ORF3a_NTD"=>65, "ORF3a_TM1"=>66, "ORF3a_TM12Lnk"=>67, "ORF3a_TM2"=>68, "ORF3a_TM3"=>69, "ORF3a_cytosl1"=>70, "ORF3a_Loop"=>71, "ORF3a_3DB"=>72, "ORF3a_CTD"=>73, "E_TM"=>74, "E_cytosol"=>75, "E_CTD"=>76, "N_N1"=>77, "N_N2"=>78, "N_N3"=>79, "N_N4"=>80, "N_N5"=>81, "N_SR"=>82, "N_L_helix"=>83, "N_CBP"=>84, "N_9b_overlap"=>85)    
######################################################################################################################################
gene_array = ["ORF1a", "ORF1b", "S", "ORF3a", "E", "M", "ORF6", "ORF7a", "ORF7b", "ORF8", "N", "ORF9b"]
#                                        Gene      AAsite  all_sub_types_at_AAsite (e.g., AV, AT, TA, etc)
######################################################################################################################################
######################################################################################################################################
##### Filling all types of NSP muts in all seqs——both circulating & chronic——for DataFrame purposes (to make # of rows the same) #####
######################################################################################################################################
######################################################################################################################################
sub_types_at_every_site_combined = Dict{String, Dict{Int, Set{String}}}()
#                                           Gene      AAsite sub_type_ct
sub_types_at_every_site_combined_ct = Dict{String, Dict{Int, Int}}()
# gene_array = ["ORF1a", "ORF1b", "S", "ORF3a", "E", "M", "ORF6", "ORF7a", "ORF7b", "ORF8", "N", "ORF9b"]
for gene in gene_array
    sub_types_at_every_site_combined[gene] = Dict{Int, Set{String}}()
    sub_types_at_every_site_combined_ct[gene] = Dict{Int, Int}()
end
gene_AA_lengths = Dict{String, Int}()
for (gene, AAseq) in gene_AA_dict
    gene_AA_lengths[gene] = length(AAseq)
end
for gene in gene_array
    gene_len = gene_AA_lengths[gene]
    for i in 1:gene_len
        sub_types_at_every_site_combined[gene][i] = Set{String}()
        sub_types_at_every_site_combined_ct[gene][i] = 0
    end
end
for sub in keys(AA_muts_ct_no_dels_all)
    gene = split(sub, ":")[1]
    pos = parse(Int, split(sub, ":")[2][2:end-1])
    mut_type = string(split(sub, ":")[2][1])*string(split(sub, ":")[2][end])
    push!(sub_types_at_every_site_combined[gene][pos], mut_type)
end
for sub in keys(AA_muts_ct_no_dels)
    gene = split(sub, ":")[1]
    pos = parse(Int, split(sub, ":")[2][2:end-1])
    mut_type = string(split(sub, ":")[2][1])*string(split(sub, ":")[2][end])
    push!(sub_types_at_every_site_combined[gene][pos], mut_type)
end
for gene in gene_array
    for (AApos, mut_type_set) in sub_types_at_every_site_combined[gene]
        sub_type_ct = length(mut_type_set)
        sub_types_at_every_site_combined_ct[gene][AApos] = sub_type_ct
    end
end
######################################################################################################################################
######################################################################################################################################
##### Filling all types of NSP muts in all seqs——both circulating & chronic——for DataFrame purposes (to make # of rows the same) #####
######################################################################################################################################
######################################################################################################################################
#                                             NSP      AAsite  all_sub_types_at_AAsite (e.g., AV, AT, TA, etc)
NSP_sub_types_at_every_site_combined = Dict{String, Dict{Int, Set{String}}}()
#                                               NSP      AAsite sub_type_ct
NSP_sub_types_at_every_site_combined_ct = Dict{String, Dict{Int, Int}}()
NSP_array = ["NSP1", "NSP2", "NSP3", "NSP4", "NSP5", "NSP6", "NSP7", "NSP8", "NSP9", "NSP10", "NSP11", "NSP12", "NSP13", "NSP14", "NSP15", "NSP16"]
for NSP in NSP_array
    NSP_sub_types_at_every_site_combined[NSP] = Dict{Int, Set{String}}()
    NSP_sub_types_at_every_site_combined_ct[NSP] = Dict{Int, Int}()
end
for NSP in NSP_array
    NSP_len = NSP_AA_size[NSP]
    for i in 1:NSP_len
        NSP_sub_types_at_every_site_combined[NSP][i] = Set{String}()
        NSP_sub_types_at_every_site_combined_ct[NSP][i] = 0
    end
end
for sub in keys(AA_muts_ct_no_dels_all)
    gene = split(sub, ":")[1]
    if gene == "ORF1a" || gene == "ORF1b"
        AA_site = parse(Int, split(sub, ":")[2][2:end-1])
        if AA_site < 4402
            NSP_sub = ORF1abMut_to_NSP(sub)
            NSP = split(NSP_sub, ":")[1]
            NSP_pos = parse(Int, split(NSP_sub, ":")[2][2:end-1])
            sub_type = string(split(NSP_sub, ":")[2][1])*string(split(NSP_sub, ":")[2][end])
            push!(NSP_sub_types_at_every_site_combined[NSP][NSP_pos], sub_type)
        end
    end
end
for sub in keys(AA_muts_ct_no_dels)
    gene = split(sub, ":")[1]
    if gene == "ORF1a" || gene == "ORF1b"
        AA_site = parse(Int, split(sub, ":")[2][2:end-1])
        if AA_site < 4402
            NSP_sub = ORF1abMut_to_NSP(sub)
            NSP = split(NSP_sub, ":")[1]
            NSP_pos = parse(Int, split(NSP_sub, ":")[2][2:end-1])
            sub_type = string(split(NSP_sub, ":")[2][1])*string(split(NSP_sub, ":")[2][end])
            push!(NSP_sub_types_at_every_site_combined[NSP][NSP_pos], sub_type)
        end
    end
end
for NSP in NSP_array
    for (NSP_pos, mut_type_set) in NSP_sub_types_at_every_site_combined[NSP]
        sub_type_ct = length(mut_type_set)
        NSP_sub_types_at_every_site_combined_ct[NSP][NSP_pos] = sub_type_ct
    end
end
#####################################################################################################################################        
#####################################################################################################################################
#####################################################################################################################################
counted_seq_total_private_AA_subs = Dict{String, Int}()
total_private_AA_subs_all_counted_chronics = 0
total_counted_sequences = 0
for name in all_unique_chr_seqs_set
    total_counted_sequences += 1
    total_AA = length(seq_AA_muts_no_dels[name])
    total_private_AA_subs_all_counted_chronics += total_AA
    counted_seq_total_private_AA_subs[name] = total_AA
end 
global chronic_avg_AA_subs_per_seq = total_private_AA_subs_all_counted_chronics/total_counted_sequences
chronic_avg_AA_subs_per_seq_rd = round(digits=2, chronic_avg_AA_subs_per_seq)
avg_private_AA_per_circ_seq_rd = round(digits=3, avg_private_AA_per_circ_seq)
println()
println("Avg AA Subs per Chronic Sequence = ", chronic_avg_AA_subs_per_seq_rd)
println("Avg AA Subs per Circulating Sequence = ", avg_private_AA_per_circ_seq_rd)
date = Dates.format(today(), "yyyy_mm_dd")
fx_name = "chr_to_circ_mut_density_ratios"
chr_to_circ_gene_density_ratio = Dict{String, Float64}()
chr_to_circ_domain_density_ratio = Dict{String, Float64}()
# The adjusted dict below adjusts for the greater number of AA muts per chronic seq compared to circ seq
adj_factor = avg_private_AA_per_circ_seq/chronic_avg_AA_subs_per_seq
adj_factor_rd = round(digits=4, adj_factor)

simple_chr_to_circ_adj_factor = total_AA_subs_all/total_chr_AA_subs
simple_chr_to_circ_adj_factor_rd = round(digits=2, simple_chr_to_circ_adj_factor)

println("simple_chr_to_circ_adj_factor = $(simple_chr_to_circ_adj_factor_rd)")
println("Adjustment Factor for greater number of AA muts per chronic seq compared to circ seq = $(adj_factor_rd)")
#####################################################################################################################################
#####################################################################################################################################
tot_singlets = length(non_rep_seqs)
tot_groups = length(keys(rep_seq_grps_maxmut_seqs) )
tot_chronics = tot_singlets + tot_groups
tot_all = qualifying_seq_ct_all
total_independent_DQ_qualifying_chronic_seqs = length(all_unique_chr_seqs)
circ_to_chronic_seq_total_ratio = qualifying_seq_ct_all/tot_chronics
combined_adjustment_factor = circ_to_chronic_seq_total_ratio*adj_factor
combined_adjustment_factor_check = (avg_private_AA_per_circ_seq*qualifying_seq_ct_all)/(chronic_avg_AA_subs_per_seq*tot_chronics)
combined_adjustment_factor_rd = round(digits=3, combined_adjustment_factor)
combined_adjustment_factor_check_rd = round(digits=3, combined_adjustment_factor_check)
println()
print("Total Singlets = $(tot_singlets) | ")
print("Total Groups = $(tot_groups) | ")
println("Total Independent Chronics = $(tot_chronics)")
println("Total Qualifying Circulating Sequences = $(qualifying_seq_ct_all)")
circ_to_chronic_seq_total_ratio_rd = round(digits=2, circ_to_chronic_seq_total_ratio)
println("(Total Qualifying Circulating Sequences)/(Total Independent Chronics) = $(circ_to_chronic_seq_total_ratio_rd)")
print("Combined Adjustment Factor = $(combined_adjustment_factor_rd)  |  ")
println("Combined Adjustment Factor Check = $(combined_adjustment_factor_check_rd)")
#####################################################################################################################################
########### Note: the 1/1000 factor below is to make up for the 1000 factor that's used in the current ALL_SEQS function ############
#####################################################################################################################################
for (gene, chr_density) in gene_mut_density
    all_density = gene_mut_density_all[gene]
    chr_circ_ratio = (chr_density/all_density)*simple_chr_to_circ_adj_factor
    ratio_rd = round(digits=2, chr_circ_ratio)
    chr_to_circ_gene_density_ratio[gene] = ratio_rd
end
for (domain, chr_density) in domain_mut_density
    all_density = domain_mut_density_all[domain]
    chr_circ_ratio = chr_density/all_density*simple_chr_to_circ_adj_factor
    ratio_rd = round(digits=2, chr_circ_ratio)
    chr_to_circ_domain_density_ratio[domain] = ratio_rd
end
######################################################################################################################################
chr_to_circ_gene_density_ratio_sort_by_gene = sort(collect(chr_to_circ_gene_density_ratio), by = x -> gene_protein_order[x[1]])
chr_to_circ_gene_density_ratio_sort_by_ratio = sort(collect(chr_to_circ_gene_density_ratio), by = x -> x[2], rev=true)
chr_to_circ_domain_density_ratio_sort_by_domain = sort(collect(chr_to_circ_domain_density_ratio), by = x -> domain_order[x[1]])
chr_to_circ_domain_density_ratio_sort_by_ratio = sort(collect(chr_to_circ_domain_density_ratio), by = x -> x[2], rev=true)
######################################################################################################################################
save("$(date)_chr_to_circ_gene_density_ratio.jld2", "chr_to_circ_gene_density_ratio", chr_to_circ_gene_density_ratio)
save("$(date)_chr_to_circ_domain_density_ratio.jld2", "chr_to_circ_domain_density_ratio", chr_to_circ_domain_density_ratio)
@save("$(date)_chr_to_circ_gene_density_ratio_sort_by_gene.jld2", chr_to_circ_gene_density_ratio_sort_by_gene)
@save("$(date)_chr_to_circ_gene_density_ratio_sort_by_ratio.jld2", chr_to_circ_gene_density_ratio_sort_by_ratio)
@save("$(date)_chr_to_circ_domain_density_ratio_sort_by_domain.jld2", chr_to_circ_domain_density_ratio_sort_by_domain)
@save("$(date)_chr_to_circ_domain_density_ratio_sort_by_ratio.jld2", chr_to_circ_domain_density_ratio_sort_by_ratio)
######################################################################################################################################

##### domain_residues_NSP/ORF *WITH* RBM (added 2025-08-31)
domain_residues_NSP = Dict{String, String}("NSP3_Ubl1"=>"NSP3:1-111", "NSP3_HVR"=>"NSP3:112-206", "NSP3_Mac1"=>"NSP3:207-374", "NSP3_Mac2"=>"NSP3:375-551", "NSP3_Mac3"=>"NSP3:552-673", "NSP3_DPUP"=>"NSP3:674-743", "NSP3_Ubl2"=>"NSP3:744-803", "NSP3_PLpro"=>"NSP3:804-1052", "NSP3_NAB"=>"NSP3:1053-1203", "NSP3_BSM"=>"NSP3:1204-1334", "NSP3_TM1"=>"NSP3:1335-1440", "NSP3_Ecto3"=>"NSP3:1441-1499", "NSP3_TM234HLX"=>"NSP3:1500-1586", "NSP3_Y1"=>"NSP3:1587-1764", "NSP3_CoVY"=>"NSP3:1765-1945", "NSP4_TM1"=>"NSP4:1-31", "NSP4_Ecto4"=>"NSP4:32-271", "NSP4_TM2_TM6"=>"NSP4:272-410", "NSP4_CTD"=>"NSP4:411-500", "NSP6_AmphHlx"=>"NSP6:219-236", "NSP6_MAE"=>"NSP6:237-276", "NSP6_cyto_CTD"=>"NSP6:277-290", "NSP12_NiRAN"=>"NSP12:1-250", "NSP12_intrfce"=>"NSP12:251-398", "NSP12_fingers"=>"NSP12:399-581", "NSP12_palm"=>"NSP12:582-627", "NSP12_palmLnk"=>"NSP12:628-687", "NSP12_palm2"=>"NSP12:688-812", "NSP12_thumb"=>"NSP12:813-932", "NSP13_ZBD"=>"NSP13:1-101", "NSP13_stalk"=>"NSP13:102-150", "NSP13_1B"=>"NSP13:151-226", "NSP13_RecA1"=>"NSP13:261-442", "NSP13_RecA2"=>"NSP13:443-601", "NSP14_nsp10"=>"NSP14:1-58", "NSP14_EXON"=>"NSP14:67-282 ", "NSP14_hinge1"=>"NSP14:286-300", "NSP14_hinge2"=>"NSP14:411-434", "NSP14_N7MTase"=>"NSP14:302-527", "NSP15_NTD"=>"NSP15:1-64", "NSP15_MD"=>"NSP15:65-182", "NSP15_endoU"=>"NSP15:207-347", "S_S1"=>"S:2-686", "S_S2"=>"S:687-1273", "S_NTD"=>"S:2-305", "S_N2R"=>"S:306-334", "S_RBD"=>"S:335-528", "S_RBM"=>"S:438-506", "S_SD1"=>"S:529-591", "S_SD2"=>"S:592-674", "S_630_loop"=>"S:619-642", "S_FCS_region"=>"S:675-691", "S_Beta1"=>"S:718-729", "S_3H"=>"S:747-769", "S_IL770"=>"S:770-831", "S_FPPR"=>"S:832-866", "S_FP"=>"S:867-909", "S_HR1"=>"S:910-985", "S_CH"=>"S:986-1035", "S_CD"=>"S:1036-1068", "S_Beta2"=>"S:1127-1135", "S_2turnHelix"=>"S:1148-1155", "S_HR2"=>"S:1164-1211", "S_TM"=>"S:1212-1234", "S_CT"=>"S:1235-1273", "ORF3a_SignalP"=>"ORF3a:1-14", "ORF3a_NTD"=>"ORF3a:15-33", "ORF3a_TM1"=>"ORF3a:34-56", "ORF3a_TM12Lnk"=>"ORF3a:57-76", "ORF3a_TM2"=>"ORF3a:77-99", "ORF3a_TM3"=>"ORF3a:103-125", "ORF3a_cytosl1"=>"ORF3a:126-169", "ORF3a_Loop"=>"ORF3a:170-183", "ORF3a_3DB"=>"ORF3a:170-222", "ORF3a_CTD"=>"ORF3a:223-275", "E_TM"=>"E:8-38", "E_cytosol"=>"E:54-65", "E_CTD"=>"E:69-75", "N_N1"=>"N:2-44", "N_N2"=>"N:45-175", "N_N3"=>"N:176-246", "N_N4"=>"N:247-364", "N_N5"=>"N:365-419", "N_SR"=>"N:176-206", "N_L_helix"=>"N:215-235", "N_CBP"=>"N:247-279", "N_9b_overlap"=>"N:4-101")
domain_residues_ORF = Dict{String, String}("ORF3a_SignalP"=>"ORF3a:1-14", "ORF3a_NTD"=>"ORF3a:15-33", "ORF3a_TM1"=>"ORF3a:34-56", "ORF3a_TM12Lnk"=>"ORF3a:57-76", "ORF3a_TM2"=>"ORF3a:77-99", "ORF3a_TM3"=>"ORF3a:103-125", "ORF3a_cytosl1"=>"ORF3a:126-169", "ORF3a_Loop"=>"ORF3a:170-183", "ORF3a_3DB"=>"ORF3a:170-222", "ORF3a_CTD"=>"ORF3a:223-275", "NSP3_Ubl1"=>"ORF1a:819-929", "NSP3_HVR"=>"ORF1a:930-1024", "NSP3_Mac1"=>"ORF1a:1025-1192", "NSP3_Mac2"=>"ORF1a:1193-1369", "NSP3_Mac3"=>"ORF1a:1370-1491", "NSP3_DPUP"=>"ORF1a:1492-1561", "NSP3_Ubl2"=>"ORF1a:1562-1621", "NSP3_PLpro"=>"ORF1a:1622-1870", "NSP3_NAB"=>"ORF1a:1871-2021", "NSP3_BSM"=>"ORF1a:2022-2152", "NSP3_TM1"=>"ORF1a:2153-2258", "NSP3_Ecto3"=>"ORF1a:2259-2317", "NSP3_TM234HLX"=>"ORF1a:2318-2404", "NSP3_Y1"=>"ORF1a:2405-2582", "NSP3_CoVY"=>"ORF1a:2583-2763", "NSP4_TM1"=>"ORF1a:2764-2794", "NSP4_Ecto4"=>"ORF1a:2795-3034", "NSP4_TM2_TM6"=>"ORF1a:3035-3173", "NSP4_CTD"=>"ORF1a:3174-3263", "NSP6_AmphHlx"=>"ORF1a:3788-3805", "NSP6_MAE"=>"ORF1a:3806-3845", "NSP6_cyto_CTD"=>"ORF1a:3846-3859", "NSP12_NiRAN"=>"1a:4393-1b:241", "NSP12_intrfce"=>"ORF1b:242-389", "NSP12_fingers"=>"ORF1b:390-572", "NSP12_palm"=>"ORF1b:573-618", "NSP12_palmLnk"=>"ORF1b:619-678", "NSP12_thumb"=>"ORF1b:804-923", "NSP13_ZBD"=>"ORF1b:924-1024", "NSP13_stalk"=>"ORF1b:1025-1073", "NSP13_1B"=>"ORF1b:1074-1149", "NSP13_RecA1"=>"ORF1b:1184-1365", "NSP13_RecA2"=>"ORF1b:1366-1524", "NSP14_nsp10"=>"ORF1b:1525-1582", "NSP14_EXON"=>"ORF1b:1591-1806", "NSP14_hinge1"=>"ORF1b:1810-1824", "NSP14_hinge2"=>"ORF1b:1935-1958", "NSP14_N7MTase"=>"ORF1b:1826-2051", "NSP15_NTD"=>"ORF1b:2052-2115", "NSP15_MD"=>"ORF1b:2116-2233", "NSP15_endoU"=>"ORF1b:2258-2398", "S_S1"=>"S:2-686", "S_NTD"=>"S:2-305", "S_N2R"=>"S:306-334", "S_RBD"=>"S:335-528", "S_RBM"=>"S:438-506", "S_SD1"=>"S:529-591", "S_SD2"=>"S:592-674", "S_630_loop"=>"S:619-642", "S_FCS_region"=>"S:675-691", "S_S2"=>"S:687-1273", "S_Beta1"=>"S:718-729", "S_3H"=>"S:747-769", "S_IL770"=>"S:770-831", "S_FPPR"=>"S:832-866", "S_FP"=>"S:867-909", "S_HR1"=>"S:910-985", "S_CH"=>"S:986-1035", "S_CD"=>"S:1036-1068", "S_Beta2"=>"S:1127-1135", "S_2turnHelix"=>"S:1148-1155", "S_HR2"=>"S:1164-1211", "S_TM"=>"S:1212-1234", "S_CT"=>"S:1235-1273", "E_TM"=>"E:8-38", "E_cytosol"=>"E:54-65", "E_CTD"=>"E:69-75", "N_9b_overlap"=>"N:4-101", "N_N1"=>"N:2-44", "N_N2"=>"N:45-175", "N_SR"=>"N:176-206", "N_N3"=>"N:176-246", "N_L_helix"=>"N:215-235", "N_CBP"=>"N:247-279", "N_N4"=>"N:247-364", "N_N5"=>"N:365-419" )

##### domain_residues_NSP/ORF *WITHOUT* RBM (added 2025-08-31)
#domain_residues_NSP = Dict{String, String}("NSP3_Ubl1"=>"NSP3:1-111", "NSP3_HVR"=>"NSP3:112-206", "NSP3_Mac1"=>"NSP3:207-374", "NSP3_Mac2"=>"NSP3:375-551", "NSP3_Mac3"=>"NSP3:552-673", "NSP3_DPUP"=>"NSP3:674-743", "NSP3_Ubl2"=>"NSP3:744-803", "NSP3_PLpro"=>"NSP3:804-1052", "NSP3_NAB"=>"NSP3:1053-1203", "NSP3_BSM"=>"NSP3:1204-1334", "NSP3_TM1"=>"NSP3:1335-1440", "NSP3_Ecto3"=>"NSP3:1441-1499", "NSP3_TM234HLX"=>"NSP3:1500-1586", "NSP3_Y1"=>"NSP3:1587-1764", "NSP3_CoVY"=>"NSP3:1765-1945", "NSP4_TM1"=>"NSP4:1-31", "NSP4_Ecto4"=>"NSP4:32-271", "NSP4_TM2_TM6"=>"NSP4:272-410", "NSP4_CTD"=>"NSP4:411-500", "NSP6_AmphHlx"=>"NSP6:219-236", "NSP6_MAE"=>"NSP6:237-276", "NSP6_cyto_CTD"=>"NSP6:277-290", "NSP12_NiRAN"=>"NSP12:1-250", "NSP12_intrfce"=>"NSP12:251-398", "NSP12_fingers"=>"NSP12:399-581", "NSP12_palm"=>"NSP12:582-627", "NSP12_palmLnk"=>"NSP12:628-687", "NSP12_palm2"=>"NSP12:688-812", "NSP12_thumb"=>"NSP12:813-932", "NSP13_ZBD"=>"NSP13:1-101", "NSP13_stalk"=>"NSP13:102-150", "NSP13_1B"=>"NSP13:151-226", "NSP13_RecA1"=>"NSP13:261-442", "NSP13_RecA2"=>"NSP13:443-601", "NSP14_nsp10"=>"NSP14:1-58", "NSP14_EXON"=>"NSP14:67-282 ", "NSP14_hinge1"=>"NSP14:286-300", "NSP14_hinge2"=>"NSP14:411-434", "NSP14_N7MTase"=>"NSP14:302-527", "NSP15_NTD"=>"NSP15:1-64", "NSP15_MD"=>"NSP15:65-182", "NSP15_endoU"=>"NSP15:207-347", "S_S1"=>"S:2-686", "S_S2"=>"S:687-1273", "S_NTD"=>"S:2-305", "S_N2R"=>"S:306-334", "S_RBD"=>"S:335-528", "S_SD1"=>"S:529-591", "S_SD2"=>"S:592-674", "S_630_loop"=>"S:619-642", "S_FCS_region"=>"S:675-691", "S_Beta1"=>"S:718-729", "S_3H"=>"S:747-769", "S_IL770"=>"S:770-831", "S_FPPR"=>"S:832-866", "S_FP"=>"S:867-909", "S_HR1"=>"S:910-985", "S_CH"=>"S:986-1035", "S_CD"=>"S:1036-1068", "S_Beta2"=>"S:1127-1135", "S_2turnHelix"=>"S:1148-1155", "S_HR2"=>"S:1164-1211", "S_TM"=>"S:1212-1234", "S_CT"=>"S:1235-1273", "ORF3a_SignalP"=>"ORF3a:1-14", "ORF3a_NTD"=>"ORF3a:15-33", "ORF3a_TM1"=>"ORF3a:34-56", "ORF3a_TM12Lnk"=>"ORF3a:57-76", "ORF3a_TM2"=>"ORF3a:77-99", "ORF3a_TM3"=>"ORF3a:103-125", "ORF3a_cytosl1"=>"ORF3a:126-169", "ORF3a_Loop"=>"ORF3a:170-183", "ORF3a_3DB"=>"ORF3a:170-222", "ORF3a_CTD"=>"ORF3a:223-275", "E_TM"=>"E:8-38", "E_cytosol"=>"E:54-65", "E_CTD"=>"E:69-75", "N_N1"=>"N:2-44", "N_N2"=>"N:45-175", "N_N3"=>"N:176-246", "N_N4"=>"N:247-364", "N_N5"=>"N:365-419", "N_SR"=>"N:176-206", "N_L_helix"=>"N:215-235", "N_CBP"=>"N:247-279", "N_9b_overlap"=>"N:4-101")
#domain_residues_ORF = Dict{String, String}("ORF3a_SignalP"=>"ORF3a:1-14", "ORF3a_NTD"=>"ORF3a:15-33", "ORF3a_TM1"=>"ORF3a:34-56", "ORF3a_TM12Lnk"=>"ORF3a:57-76", "ORF3a_TM2"=>"ORF3a:77-99", "ORF3a_TM3"=>"ORF3a:103-125", "ORF3a_cytosl1"=>"ORF3a:126-169", "ORF3a_Loop"=>"ORF3a:170-183", "ORF3a_3DB"=>"ORF3a:170-222", "ORF3a_CTD"=>"ORF3a:223-275", "NSP3_Ubl1"=>"ORF1a:819-929", "NSP3_HVR"=>"ORF1a:930-1024", "NSP3_Mac1"=>"ORF1a:1025-1192", "NSP3_Mac2"=>"ORF1a:1193-1369", "NSP3_Mac3"=>"ORF1a:1370-1491", "NSP3_DPUP"=>"ORF1a:1492-1561", "NSP3_Ubl2"=>"ORF1a:1562-1621", "NSP3_PLpro"=>"ORF1a:1622-1870", "NSP3_NAB"=>"ORF1a:1871-2021", "NSP3_BSM"=>"ORF1a:2022-2152", "NSP3_TM1"=>"ORF1a:2153-2258", "NSP3_Ecto3"=>"ORF1a:2259-2317", "NSP3_TM234HLX"=>"ORF1a:2318-2404", "NSP3_Y1"=>"ORF1a:2405-2582", "NSP3_CoVY"=>"ORF1a:2583-2763", "NSP4_TM1"=>"ORF1a:2764-2794", "NSP4_Ecto4"=>"ORF1a:2795-3034", "NSP4_TM2_TM6"=>"ORF1a:3035-3173", "NSP4_CTD"=>"ORF1a:3174-3263", "NSP6_AmphHlx"=>"ORF1a:3788-3805", "NSP6_MAE"=>"ORF1a:3806-3845", "NSP6_cyto_CTD"=>"ORF1a:3846-3859", "NSP12_NiRAN"=>"1a:4393-1b:241", "NSP12_intrfce"=>"ORF1b:242-389", "NSP12_fingers"=>"ORF1b:390-572", "NSP12_palm"=>"ORF1b:573-618", "NSP12_palmLnk"=>"ORF1b:619-678", "NSP12_thumb"=>"ORF1b:804-923", "NSP13_ZBD"=>"ORF1b:924-1024", "NSP13_stalk"=>"ORF1b:1025-1073", "NSP13_1B"=>"ORF1b:1074-1149", "NSP13_RecA1"=>"ORF1b:1184-1365", "NSP13_RecA2"=>"ORF1b:1366-1524", "NSP14_nsp10"=>"ORF1b:1525-1582", "NSP14_EXON"=>"ORF1b:1591-1806", "NSP14_hinge1"=>"ORF1b:1810-1824", "NSP14_hinge2"=>"ORF1b:1935-1958", "NSP14_N7MTase"=>"ORF1b:1826-2051", "NSP15_NTD"=>"ORF1b:2052-2115", "NSP15_MD"=>"ORF1b:2116-2233", "NSP15_endoU"=>"ORF1b:2258-2398", "S_S1"=>"S:2-686", "S_NTD"=>"S:2-305", "S_N2R"=>"S:306-334", "S_RBD"=>"S:335-528", "S_SD1"=>"S:529-591", "S_SD2"=>"S:592-674", "S_630_loop"=>"S:619-642", "S_FCS_region"=>"S:675-691", "S_S2"=>"S:687-1273", "S_Beta1"=>"S:718-729", "S_3H"=>"S:747-769", "S_IL770"=>"S:770-831", "S_FPPR"=>"S:832-866", "S_FP"=>"S:867-909", "S_HR1"=>"S:910-985", "S_CH"=>"S:986-1035", "S_CD"=>"S:1036-1068", "S_Beta2"=>"S:1127-1135", "S_2turnHelix"=>"S:1148-1155", "S_HR2"=>"S:1164-1211", "S_TM"=>"S:1212-1234", "S_CT"=>"S:1235-1273", "E_TM"=>"E:8-38", "E_cytosol"=>"E:54-65", "E_CTD"=>"E:69-75", "N_9b_overlap"=>"N:4-101", "N_N1"=>"N:2-44", "N_N2"=>"N:45-175", "N_SR"=>"N:176-206", "N_N3"=>"N:176-246", "N_L_helix"=>"N:215-235", "N_CBP"=>"N:247-279", "N_N4"=>"N:247-364", "N_N5"=>"N:365-419" )
######################################################################################################################################
function NSP_to_ORF(domain::String, NSP::Int)
    ORFadd = NSP1ab_add[NSP]
    first1 = minimum(domain_residues_NSP_BitSet[domain])
    last1 = maximum(domain_residues_NSP_BitSet[domain])
    first = first1 + ORFadd
    last = last1 + ORFadd
    ORF_range_BitSet = BitSet(first:last)
    firstORF = string(first)
    lastORF = string(last)
    ORF_range = ""
    if NSP < 12
        ORF_range = "ORF1a:$(firstORF)-$(lastORF)"
    else
        ORF_range = "ORF1b:$(firstORF)-$(lastORF)"
    end
    return ORF_range, ORF_range_BitSet
end
#####################################################################################################################################
sorted_ratio_gene_dicts = [chr_to_circ_gene_density_ratio_sort_by_gene, chr_to_circ_gene_density_ratio_sort_by_ratio] 
sorted_ratio_domain_dicts = [chr_to_circ_domain_density_ratio_sort_by_domain, chr_to_circ_domain_density_ratio_sort_by_ratio]
headers_gene = ["###### Chronic-to-Circulating Gene Mutation Density Ratios, Sorted by Gene ######", "###### Chronic-to-Circulating Gene Mutation Density Ratios, Sorted by Ratio ######"]
headers_domain = ["######### Chronic-to-Circulating Domain Mutation Density Ratios, Sorted by Domain #########", "######### Chronic-to-Circulating Domain Mutation Density Ratios, Sorted by Ratio #########"]
open("$(date)_$(fx_name).txt", "w") do g
    ct = 0
    for sorted_dict in sorted_ratio_gene_dicts
        ct += 1
        println()
        println(headers_gene[ct])
        println(g)
        println(g, headers_gene[ct])
        for w in sorted_dict
            gene = w[1]
            ratio1 = string(Int(w[2]÷1) )
            ratio21 = string(round(digits=2, w[2]%1) )
            ratio22 = split(ratio21, ".")[2]
            if length(ratio22) == 1
                ratio22 = ratio22*"0"
            end
            ratio = "$(ratio1).$(ratio22)"
            println("                  ", rpad("$(gene)", 5), " = ", lpad("$(ratio)", 5), "       ", lpad("$(NSP_ranges[gene])", 16) )
            println(g, "                  ", rpad("$(gene)", 5), " = ", lpad("$(ratio)", 5), "       ", lpad("$(NSP_ranges[gene])", 16) )
        end
    end
    ct2 = 0
    for sorted_dict in sorted_ratio_domain_dicts
        ct2 += 1
        println()
        println(headers_domain[ct2])
        println(g)
        println(g, headers_domain[ct2])
        for w in sorted_dict
            gene = w[1]
            ratio1 = string(Int(w[2]÷1) )
            ratio21 = string(round(digits=2, w[2]%1) )
            ratio22 = split(ratio21, ".")[2]
            if length(ratio22) == 1
                ratio22 = ratio22*"0"
            end
            ratio = "$(ratio1).$(ratio22)"
            println("               ", rpad("$(gene)", 13), " = ", lpad("$(ratio)", 5), "   ", rpad("$(domain_residues_NSP[gene])", 15), "  ", rpad("$(domain_residues_ORF[gene])", 17) )
            println(g, "               ", rpad("$(gene)", 13), " = ", lpad("$(ratio)", 5), "   ", rpad("$(domain_residues_NSP[gene])", 15), "  ", rpad("$(domain_residues_ORF[gene])", 17) )
        end
    end
end
#####################################################################################################################################
AApos(m) = parse(Int, split(m, ":")[2][2:end-1])
AA_sub_type(m) = "$(m[1])"*"$(m[end])"
gene_name(m) = split(m, ":")[1]
######################################################################################################################################
######################################################################################################################################        
######################################################################################################################################
######################################################################################################################################
gene_sub_ranks_chr(gene_array, sub_types_at_every_site_combined, AA_muts_ct_no_dels)
gene_sub_ranks_all(gene_array, sub_types_at_every_site_combined, AA_muts_ct_no_dels_all)
gene_sub_pos_ranks_all(gene_array, gene_AA_lengths,  AA_muts_ct_pos_only_no_dels_all)
gene_sub_pos_ranks_chr(gene_array, gene_AA_lengths, AA_muts_ct_pos_only_no_dels )
NSP_sub_ranks_chr(NSP_array, NSP_sub_types_at_every_site_combined, AA_muts_ct_no_dels)
NSP_sub_ranks_all(NSP_array, NSP_sub_types_at_every_site_combined, AA_muts_ct_no_dels_all)
NSP_sub_pos_ranks_all(NSP_array, NSP_sub_types_at_every_site_combined, AA_muts_ct_no_dels_all)
NSP_sub_pos_ranks_chr(NSP_array, NSP_sub_types_at_every_site_combined, AA_muts_ct_no_dels)
#NSP_sub_pos_ranks_all(NSP_array, NSP_AA_size, NSP_sub_types_at_every_site_combined, AA_muts_ct_no_dels_all)
#NSP_sub_pos_ranks_chr(NSP_array, NSP_AA_size, NSP_sub_types_at_every_site_combined, AA_muts_ct_no_dels)
######################################################################################################################################
println()
print(sub_types_at_every_site_combined["ORF9b"][2])
print(" | ", sub_types_at_every_site_combined_ct["ORF9b"][2]);    println()
print(sub_types_at_every_site_combined["ORF9b"][92])
print(" | ", sub_types_at_every_site_combined_ct["ORF9b"][92]);   println()
###################################################################################################################################
###################################################################################################################################
###################################################################################################################################
###################################################################################################################################
gene_AA_sites = Dict{String, BitSet}()
for (gene, length) in gene_AA_lengths
    gene_AA_sites[gene] = BitSet([1:length...])
end
#for (gene, bitset) in gene_AA_sites; println(gene); println(bitset); end
############################################################
gene_AA_sites_used = Dict{String, BitSet}()
for gene in gene_array
    gene_AA_sites_used[gene] = BitSet()
end
for (sub, ct) in AA_muts_ct_no_dels_all
    gene = split(sub, ":")[1]
    subby = split(sub, ":")[2]
    AA_site = parse(Int, subby[2:end-1])
    push!(gene_AA_sites_used[gene], AA_site)
end
############################################################
gene_AA_sites_not_used = Dict{String, BitSet}()
for gene in gene_array
    gene_AA_sites_not_used[gene] = BitSet()
end
for (gene, bitset) in gene_AA_sites
    bitset_used = gene_AA_sites_used[gene]
    unused_AA_sites = setdiff(bitset, bitset_used)
    gene_AA_sites_not_used[gene] = unused_AA_sites
end
###########################################################
for (gene, unused_AA_sites) in gene_AA_sites_not_used
    print("$(gene) = ")
    print(" | ")
    for unused in unused_AA_sites
        print("$(unused) |")
    end
end
for (gene, unused_AA_sites) in gene_AA_sites_not_used
    for AA_site in unused_AA_sites
        placeholder_sub = "$(gene):W$(AA_site)W"
        AA_muts_ct_no_dels_all[placeholder_sub] = 0
    end
end
######################################################################################################################################
######################################################################################################################################
######################################################################################################################################
######################################################################################################################################
#NSP_array = ["NSP1", "NSP2", "NSP3", "NSP4", "NSP5", "NSP6", "NSP7", "NSP8", "NSP9", "NSP10", "NSP11", "NSP12", "NSP13", "NSP14", "NSP15", "NSP16"]
NSP1_muts_ct_no_dels_all = Dict{String, Int}()
NSP2_muts_ct_no_dels_all = Dict{String, Int}()
NSP3_muts_ct_no_dels_all = Dict{String, Int}()
NSP4_muts_ct_no_dels_all = Dict{String, Int}()
NSP5_muts_ct_no_dels_all = Dict{String, Int}()
NSP6_muts_ct_no_dels_all = Dict{String, Int}()
NSP7_muts_ct_no_dels_all = Dict{String, Int}()
NSP8_muts_ct_no_dels_all = Dict{String, Int}()
NSP9_muts_ct_no_dels_all = Dict{String, Int}()
NSP10_muts_ct_no_dels_all = Dict{String, Int}()
NSP11_muts_ct_no_dels_all = Dict{String, Int}()
NSP12_muts_ct_no_dels_all = Dict{String, Int}()
NSP13_muts_ct_no_dels_all = Dict{String, Int}()
NSP14_muts_ct_no_dels_all = Dict{String, Int}()
NSP15_muts_ct_no_dels_all = Dict{String, Int}()
NSP16_muts_ct_no_dels_all = Dict{String, Int}()
NSP_muts_ct_no_dels_all_array = [NSP1_muts_ct_no_dels_all, NSP2_muts_ct_no_dels_all, NSP3_muts_ct_no_dels_all, NSP4_muts_ct_no_dels_all, NSP5_muts_ct_no_dels_all, NSP6_muts_ct_no_dels_all, NSP7_muts_ct_no_dels_all, NSP8_muts_ct_no_dels_all, NSP9_muts_ct_no_dels_all, NSP10_muts_ct_no_dels_all, NSP11_muts_ct_no_dels_all, NSP12_muts_ct_no_dels_all, NSP13_muts_ct_no_dels_all, NSP14_muts_ct_no_dels_all, NSP15_muts_ct_no_dels_all, NSP16_muts_ct_no_dels_all]       
NSP_muts_ct_no_dels_all = Dict{String, Int}()
for (sub, count) in AA_muts_ct_no_dels_all
#    println(sub)
    gene = split(sub, ":")[1]
    if gene == "ORF1a" || gene == "ORF1b"
        AA_site = parse(Int, split(sub, ":")[2][2:end-1])
        if AA_site < 4402
            NSP_sub = ORF1abMut_to_NSP(sub)
            NSP_muts_ct_no_dels_all[NSP_sub] = count
            NSP = split(NSP_sub, ":")[1]
#            println(NSP_sub)
            NSP_num = parse(Int, NSP[4:end])
            NSP_dict = NSP_muts_ct_no_dels_all_array[NSP_num]
            NSP_dict[NSP_sub] = count
        end
    end
end
###################################################################################################################################
NSP1_muts_ct_pos_only_no_dels_all = Dict{String, Int}()
NSP2_muts_ct_pos_only_no_dels_all = Dict{String, Int}()
NSP3_muts_ct_pos_only_no_dels_all = Dict{String, Int}()
NSP4_muts_ct_pos_only_no_dels_all = Dict{String, Int}()
NSP5_muts_ct_pos_only_no_dels_all = Dict{String, Int}()
NSP6_muts_ct_pos_only_no_dels_all = Dict{String, Int}()
NSP7_muts_ct_pos_only_no_dels_all = Dict{String, Int}()
NSP8_muts_ct_pos_only_no_dels_all = Dict{String, Int}()
NSP9_muts_ct_pos_only_no_dels_all = Dict{String, Int}()
NSP10_muts_ct_pos_only_no_dels_all = Dict{String, Int}()
NSP11_muts_ct_pos_only_no_dels_all = Dict{String, Int}()
NSP12_muts_ct_pos_only_no_dels_all = Dict{String, Int}()
NSP13_muts_ct_pos_only_no_dels_all = Dict{String, Int}()
NSP14_muts_ct_pos_only_no_dels_all = Dict{String, Int}()
NSP15_muts_ct_pos_only_no_dels_all = Dict{String, Int}()
NSP16_muts_ct_pos_only_no_dels_all = Dict{String, Int}()
NSP_muts_ct_pos_only_no_dels_all_array = [NSP1_muts_ct_pos_only_no_dels_all, NSP2_muts_ct_pos_only_no_dels_all, NSP3_muts_ct_pos_only_no_dels_all, NSP4_muts_ct_pos_only_no_dels_all, NSP5_muts_ct_pos_only_no_dels_all, NSP6_muts_ct_pos_only_no_dels_all, NSP7_muts_ct_pos_only_no_dels_all, NSP8_muts_ct_pos_only_no_dels_all, NSP9_muts_ct_pos_only_no_dels_all, NSP10_muts_ct_pos_only_no_dels_all, NSP11_muts_ct_pos_only_no_dels_all, NSP12_muts_ct_pos_only_no_dels_all, NSP13_muts_ct_pos_only_no_dels_all, NSP14_muts_ct_pos_only_no_dels_all, NSP15_muts_ct_pos_only_no_dels_all, NSP16_muts_ct_pos_only_no_dels_all]
NSP_muts_ct_pos_only_no_dels_all = Dict{String, Int}()
for i in 1:length(NSP_muts_ct_pos_only_no_dels_all_array)
    NSP_all_dict = NSP_muts_ct_pos_only_no_dels_all_array[i]
    NSP = NSP_array[i]
    NSP_len = NSP_AA_size[NSP]
    for j in 1:NSP_len
        NSP_pos = NSP*":"*"$(j)"
        NSP_all_dict[NSP_pos] = 0
        NSP_muts_ct_pos_only_no_dels_all[NSP_pos] = 0
    end
end
for i in 1:length(NSP_muts_ct_pos_only_no_dels_all_array)
    NSP = NSP_array[i]
    NSP_dict = NSP_muts_ct_no_dels_all_array[i]
    NSP_pos_dict = NSP_muts_ct_pos_only_no_dels_all_array[i]
    for (sub, ct) in NSP_dict
        pos = parse(Int, split(sub, ":")[2][2:end-1])
        sub_pos = NSP*":"*"$(pos)"
        NSP_pos_dict[sub_pos] += ct
        NSP_muts_ct_pos_only_no_dels_all[sub_pos] += ct
    end
end
total_time_post_load_cell = time() - start_time_post_load_cell
total_time_post_load_cell_rd = round(digits=1, total_time_post_load_cell)
println()
println("Total Time for Post-Load Cell = $(total_time_post_load_cell_rd) Seconds")
# Total Time for Post-Load Cell = 139.3 Seconds (2025-08-18)

2025-09-17__328PM

Avg AA Subs per Chronic Sequence = 18.01
Avg AA Subs per Circulating Sequence = 1.788
simple_chr_to_circ_adj_factor = 189.67
Adjustment Factor for greater number of AA muts per chronic seq compared to circ seq = 0.0993

Total Singlets = 2132 | Total Groups = 891 | Total Independent Chronics = 3023
Total Qualifying Circulating Sequences = 10154885
(Total Qualifying Circulating Sequences)/(Total Independent Chronics) = 3359.21
Combined Adjustment Factor = 333.577  |  Combined Adjustment Factor Check = 333.577

###### Chronic-to-Circulating Gene Mutation Density Ratios, Sorted by Gene ######
                  NSP1  =  0.59            ORF1a:1-180
                  NSP2  =  0.26          ORF1a:181-818
                  NSP3  =  0.48         ORF1a:819-2763
                  NSP4  =  0.56        ORF1a:2764-3263
                  NSP5  =  0.40        ORF1a:3264-3569
                  NSP6  =  0.49        ORF1a:3570-3859
                  NSP7  =  0.53        ORF1a:3860-3942


In [106]:
### Getting all RBM & RBD muts + ORF9b/N Doubles + artifactual_private_muts_subs + subs vs pos_only determination | Runtime = 
start = time(); print("\n"^1)
date_now = Dates.format(now(), "yyyy_mm_dd__IMMp"); println(date_now); date_hour = Dates.format(now(), "yyyy_mm_dd_Ip"); print("\n"^1)
###########################################################################################################################################################################
###########################################################################################################################################################################
###########################################################################################################################################################################
###########################################################################################################################################################################
###########################################################################################################################################################################
###########################################################################################################################################################################
###########################################################################################################################################################################
###########################################################################################################################################################################
###########################################################################################################################################################################
                                                        sub_0__posonly_1 = 0
                                                        normal_0__spikeonly_1 = 0
                                                        noBAL_0__withBAL_1 = 1
###########################################################################################################################################################################
###########################################################################################################################################################################
###########################################################################################################################################################################
###########################################################################################################################################################################
###########################################################################################################################################################################
###########################################################################################################################################################################
###########################################################################################################################################################################
###########################################################################################################################################################################
###########################################################################################################################################################################
#--------------------------------------------------------------------------------------------------------------------------------------------------------------------------
########################################################################################################################################################################### 
############################################################# Begin EXCLUDED MUTS SECTION #################################################################################
########################################################################################################################################################################### 
##### The mutations added below are ones that have been found to produce artifactual correlations. For example, 18/19 EPCI sequences 
####      with ORF1a:M2606I are B.1.2 (the other is likely a B.1.2 miscategorized as a B.1), and seven of these have N:D377Y, indicating
####      that these are both inherited (not private) mutations, which can be confirmed by viewing the Usher Tree for these sequences. 
####      Nextclade miscategorizes both of these as private mutations. Similarly, 5/6 sequences with ORF1a:E3764D are B.1.258, and all five also 
####      have ORF3a:A72S. Both are inherited and mischaracterized as private by Nextclade. 
global artifactual_private_muts_subs = Set{String}()
#push!(artifactual_private_muts_subs, "ORF1a:T2501I") ## Artifactual reversion in B.1 sequences (possibly real? Unclear)
push!(artifactual_private_muts_subs, "ORF1a:M2606I") ## Inherited B.1.2 mutation
push!(artifactual_private_muts_subs, "N:D377Y")      ## Inherited B.1.2 mutation
push!(artifactual_private_muts_subs, "ORF3a:A72S")   ## Inherited B.1.258 mutation
push!(artifactual_private_muts_subs, "ORF1a:E3764D") ## Inherited B.1.258 mutation
push!(artifactual_private_muts_subs, "ORF1a:I2283T") ## Artifactual reversion in XEC miscategorized as KP.3 or KP.3.3 by Nextclade
push!(artifactual_private_muts_subs, "ORF1a:A599T")  ## Artifactual "private" mutation in XEC miscategorized as KP.3 or KP.3.3 by Nextclade
push!(artifactual_private_muts_subs, "S:F59S")       ## Artifactual "private" mutation in XEC miscategorized as KP.3 or KP.3.3 by Nextclade
push!(artifactual_private_muts_subs, "ORF8:L60F")    ## Artifactual "private" mutation in AY.44. Over 125000 seqs have both ORF8:L60F & ORF1a:H2125Y
push!(artifactual_private_muts_subs, "ORF1a:H2125Y") ## Artifactual "private" mutation in AY.44. Over 125000 seqs have both ORF8:L60F & ORF1a:H2125Y
push!(artifactual_private_muts_subs, "S:Q146K")      ## Extremely homoplasic XBB mutation and/or artifact. Impossible to tell if inherited or private. 
push!(artifactual_private_muts_subs, "N:M203K")      ## Recombinant Delta/Omicron "mutation"
push!(artifactual_private_muts_subs, "N:G204R")      ## Recombinant Delta/Omicron "mutation"
push!(artifactual_private_muts_subs, "N:R204G")      ## Recombinant Delta/Omicron reversion "mutation"
push!(artifactual_private_muts_subs, "ORF1b:M1156I") ## BQ.1 mutation misattributed as private in 3 sequences
push!(artifactual_private_muts_subs, "ORF6:L61D")    ## Artifactual 3-nuc reversion, often in BA.1/2 or BA.2/5 recombinants
push!(artifactual_private_muts_subs, "ORF1a:S135R")  ## Omicron mut misattributed as private in 3 recombinants
push!(artifactual_private_muts_subs, "M:A30T")       ## Very common artifactual reversion in BA.2.86* lineages
push!(artifactual_private_muts_subs, "ORF1a:S1612L") ## Inherited Beta mutation miscategorized by Nextclade as private
push!(artifactual_private_muts_subs, "ORF7b:E39*")   ## Inherited Beta mutation miscategorized by Nextclade as private
push!(artifactual_private_muts_subs, "S:F18L")       ## All five on list in Beta, likely artifactual
push!(artifactual_private_muts_subs, "ORF1a:A2554V") ## Inherited AY.44 mutation miscategorized by Nextclade as private 
push!(artifactual_private_muts_subs, "ORF1b:H1087Y") ## Inherited AY.44 mutation miscategorized by Nextclade as private 
push!(artifactual_private_muts_subs, "ORF1a:M2796T") ## Inherited BA.1.17 mutation miscategorized by Nextclade as private 
push!(artifactual_private_muts_subs, "ORF1a:P1803S") ## Inherited BA.1.17 mutation miscategorized by Nextclade as private
push!(artifactual_private_muts_subs, "ORF3a:P104S")  ## Inherited AY.103 mutation miscategorized by Nextclade as private 
push!(artifactual_private_muts_subs, "N:A208S")      ## Inherited AY.103 mutation miscategorized by Nextclade as private 
push!(artifactual_private_muts_subs, "ORF8:K68*")    ## Inherited B.1.1.7 mutation miscategorized by Nextclade as private
push!(artifactual_private_muts_subs, "ORF1b:P1975S") ## Inherited B.1.1.7 mutation miscategorized by Nextclade as private
push!(artifactual_private_muts_subs, "ORF1b:V2178F") ## Inherited BA.1.1 mutation miscategorized by Nextclade as private
push!(artifactual_private_muts_subs, "ORF1b:I97V")   ## Inherited BA.1.1 mutation miscategorized by Nextclade as private
push!(artifactual_private_muts_subs, "ORF1b:A2589T") ## Inherited BA.5.1 mutation miscategorized by Nextclade as private
push!(artifactual_private_muts_subs, "ORF9b:T83I")   ## Inherited BA.5.1 mutation miscategorized by Nextclade as private
push!(artifactual_private_muts_subs, "ORF1a:T842I")  ## From 5' End Recombination with BA.2
push!(artifactual_private_muts_subs, "ORF1a:S135R")  ## From 5' End Recombination with BA.2
push!(artifactual_private_muts_subs, "ORF3a:V48F")   ## Inherited BE.1 mutation miscategorized by Nextclade as private
push!(artifactual_private_muts_subs, "ORF3a:G49C")   ## Inherited BE.1 mutation miscategorized by Nextclade as private
push!(artifactual_private_muts_subs, "ORF1a:A1204T") ## Inherited BE.1 mutation miscategorized by Nextclade as private
push!(artifactual_private_muts_subs, "S:A376P")      ## Butchered mutation interpretation by Nextclade for ∆374-375 or ∆374-376
push!(artifactual_private_muts_subs, "S:T376P")      ## Butchered mutation interpretation by Nextclade for ∆374-375 or ∆374-376
push!(artifactual_private_muts_subs, "S:F375A")      ## Butchered mutation interpretation by Nextclade for ∆374-375 or ∆374-376
push!(artifactual_private_muts_subs, "S:S375A")      ## Butchered mutation interpretation by Nextclade for ∆374-375 or ∆374-376
push!(artifactual_private_muts_subs, "S:D339G")      ## An artifactual reversion >90% of the time, likely 100%
push!(artifactual_private_muts_subs, "S:N417K")      ## An artifactual reversion >90% of the time, likely 100%
push!(artifactual_private_muts_subs, "S:K440N")      ## An artifactual reversion >90% of the time, likely 100%
push!(artifactual_private_muts_subs, "ORF1b:T1555I") ## Inherited B.1.2 mutation miscategorized by Nextclade as private
push!(artifactual_private_muts_subs, "ORF1a:A4285V") ## Inherited B.1.2 mutation miscategorized by Nextclade as private
push!(artifactual_private_muts_subs, "ORF1a:N2361K") ## Inherited B.1.2 mutation miscategorized by Nextclade as private
push!(artifactual_private_muts_subs, "ORF1a:V3782I") ## Inherited BA.1 mutation miscategorized by Nextclade as private
push!(artifactual_private_muts_subs, "S:K1191N")     ## Inherited B.1.1.7 mutation miscategorized by Nextclade as private
push!(artifactual_private_muts_subs, "S:R158G")      ## Artifactual Delta reversion
push!(artifactual_private_muts_subs, "ORF1a:I1023-") ## Dumb mistake I made that I have to undo with this.
push!(artifactual_private_muts_subs, "ORF1a:T3646A") ## Inherited Delta mut, miscategorized in recombinants 
push!(artifactual_private_muts_subs, "ORF1b:A1918V") ## Inherited Delta mut, miscategorized in recombinants
push!(artifactual_private_muts_subs, "ORF7b:I2V")    ## Corresponds to ORF7a:*122W
push!(artifactual_private_muts_subs, "ORF1b:P218L")  ## Inherited B.1.1.7 mutation miscategorized by Nextclade as private
push!(artifactual_private_muts_subs, "ORF1a:V3782I") ## Inherited mutation in Canadian BA.1 branch miscategorized by Nextclade as private
push!(artifactual_private_muts_subs, "ORF1a:G2118D") ## B.1 Inherited mutation miscategorized by Nextclade as private (co-occurs with S:P681H)
push!(artifactual_private_muts_subs, "ORF1a:T3646A") ## Inherited Delta mut, miscategorized in recombinants 
push!(artifactual_private_muts_subs, "ORF1b:A1918V") ## Inherited Delta mut, miscategorized in recombinants
push!(artifactual_private_muts_subs, "ORF1a:A1298V") ## Inherited BA.1 mut, miscategorized by Nextclade as private


push!(artifactual_private_muts_subs, "ORF3a:V112F")  ## Inherited B.1.1 mutation miscategorized by Nextclade as private
push!(artifactual_private_muts_subs, "ORF1b:N2695I") ## Inherited B.1.1 mutation miscategorized by Nextclade as private
push!(artifactual_private_muts_subs, "ORF3a:I35K")   ## Inherited XBB.1.16.11 mutation miscategorized by Nextclade as private
push!(artifactual_private_muts_subs, "ORF1b:E1871G") ## Inherited B.1.1.7 mutation miscategorized by Nextclade as private
push!(artifactual_private_muts_subs, "ORF1b:D1903Y") ## Inherited XBB.1.16.31 mutation miscategorized by Nextclade as private
push!(artifactual_private_muts_subs, "ORF1a:K2219R") ## Inherited XBB.1.16.31 mutation miscategorized by Nextclade as private
push!(artifactual_private_muts_subs, "ORF1b:K1094R") ## Inherited B.1.2 mutation miscategorized by Nextclade as private
push!(artifactual_private_muts_subs, "ORF1a:S771I")  ## Inherited XBB.1.5 mutation miscategorized by Nextclade as private
push!(artifactual_private_muts_subs, "N:S26C")       ## Inherited BA.5 mutation miscategorized by Nextclade as private
push!(artifactual_private_muts_subs, "ORF1a:V108I")  ## Inherited BA.5.1 mutation in Scandinavian branch miscategorized by Nextclade as private
push!(artifactual_private_muts_subs, "ORF1b:G128S")  ## Inherited BA.5.1 mutation in Scandinavian branch miscategorized by Nextclade as private
push!(artifactual_private_muts_subs, "ORF1b:N2452S") ## Inherited BA.5.1 mutation in Scandinavian branch miscategorized by Nextclade as private
push!(artifactual_private_muts_subs, "ORF1b:S1182T") ## Inherited FL.4 mutation in Australian branch miscategorized by Nextclade as private
push!(artifactual_private_muts_subs, "ORF1b:P1570T") ## Inherited BA.1.1.1 mutation in Belgian branch miscategorized by Nextclade as private
push!(artifactual_private_muts_subs, "ORF1b:E2355Q") ## Inherited BA.1.1.1 mutation in Belgian branch miscategorized by Nextclade as private
push!(artifactual_private_muts_subs, "S:T19R")       ## Delta mutation present in recombinants, falsely labeled as private mutation
push!(artifactual_private_muts_subs, "ORF7a:A66V")   ## Inherited B.1.429 mutation miscategorized by Nextclade as private
push!(artifactual_private_muts_subs, "ORF1a:T3459M") ## Inherited B.1.429 mutation miscategorized by Nextclade as private
push!(artifactual_private_muts_subs, "ORF7b:M1T")    ## Corresponds to ORF7a:*122R
push!(artifactual_private_muts_subs, "ORF1b:A869V")  ## Inherited KF.1 (FL.15.1.1.1) mutation miscategorized by Nextclade as private
push!(artifactual_private_muts_subs, "ORF1a:E1015G") ## Inherited KF.1 (FL.15.1.1.1) mutation miscategorized by Nextclade as private
push!(artifactual_private_muts_subs, "ORF7b:L4F")    ## Inherited BA.1 mutation on Swedish branch miscategorized by Nextclade as private
push!(artifactual_private_muts_subs, "ORF1a:V365I")  ## Inherited B.1.2 mutation miscategorized by Nextclade as private
push!(artifactual_private_muts_subs, "ORF1b:K1094R") ## Inherited B.1.2 mutation miscategorized by Nextclade as private
push!(artifactual_private_muts_subs, "ORF1b:D1869Y") ## Inherited C.37 and AY.85 mutation miscategorized by Nextclade as private
push!(artifactual_private_muts_subs, "ORF3a:L65H")   ## Inherited BN.1.3.1 mutation (Italian branch) miscategorized by Nextclade as private
   
push!(artifactual_private_muts_subs, "ORF3a:S220I")  ## Inherited B.1.466 (Indonesia) mutation miscategorized by Nextclade as private
push!(artifactual_private_muts_subs, "ORF1b:A176T")  ## Inherited EG.5.1 mutation miscategorized by Nextclade as private
push!(artifactual_private_muts_subs, "ORF1a:E352D")  ## Inherited B.1.1.176 (Canada) mutation miscategorized by Nextclade as private
push!(artifactual_private_muts_subs, "ORF1b:A434S")  ## Inherited B.1.1.176 (Canada) mutation miscategorized by Nextclade as private
push!(artifactual_private_muts_subs, "ORF1a:G3676C") ## Inherited B.1.1.176 (Canada) mutation miscategorized by Nextclade as private
push!(artifactual_private_muts_subs, "ORF1a:Q991L")  ## Inherited BA.5.2 (North America) mutation miscategorized by Nextclade as private
######################################################################################################################################
######################################################################################################################################
#### Double_N_ORF9b_muts are muts that result in both N and ORF9b mutations, which of course correlate perfectly but trivially.
####    Default is to block N muts, but it's also possible to change that and instead block ORF9b muts to view their N equivalents
####    by commenting out the top list and uncommenting out the bottom list.
#### This list was created in one of the above cells (the one with the ORF9b_N_nucmuts_to_AAres function) & then copied & pasted here.
global Double_N_ORF9b_muts = Set(["N:G63D", "N:F33S, N:L13-", "N:L13I", "N:L13H", "N:L13S", "N:N4I", "N:N4T", "N:N4S", "N:N4K", "N:N4K", "N:G5*", "N:G5R", "N:G5R", "N:G5V", "N:G5A", "N:G5E", "N:P6T", "N:P6A", "N:P6L", "N:P6H", "N:P6R", "N:Q7L", "N:Q7P", "N:Q7R", "N:Q7H", "N:Q7H", "N:N8Y", "N:N8H", "N:N8I", "N:N8T", "N:N8S", "N:N8K", "N:N8K", "N:Q9E", "N:Q9L", "N:Q9P", "N:Q9R", "N:Q9H", "N:Q9H", "N:R10G", "N:R10L", "N:R10P", "N:R10Q", "N:N11Y", "N:N11H", "N:N11I", "N:N11T", "N:N11S", "N:N11K", "N:N11K", "N:A12S", "N:A12P", "N:A12T", "N:A12V", "N:A12E", "N:A12G", "N:P13T", "N:P13A", "N:P13L", "N:L13P", "N:P13H", "N:P13R", "N:R14L", "N:R14P", "N:R14H", "N:I15N", "N:I15S", "N:I15M", "N:T16S", "N:T16P", "N:T16M", "N:T16K", "N:T16R", "N:F17Y", "N:F17C", "N:F17L", "N:F17L", "N:G18C", "N:G18R", "N:G18V", "N:G18A", "N:G18D", "N:G19V", "N:G19A", "N:G19E", "N:P20T", "N:P20A", "N:P20L", "N:P20H", "N:P20R", "N:S21L", "N:S21*", "N:S21*", "N:D22Y", "N:D22H", "N:D22V", "N:D22A", "N:D22G", "N:D22E", "N:D22E", "N:S23A", "N:S23L", "N:S23*", "N:S23*", "N:T24S", "N:T24P", "N:T24N", "N:T24S", "N:G25V", "N:G25A", "N:G25D", "N:S26I", "N:S26T", "N:S26N", "N:S26R", "N:S26R", "N:N27I", "N:N27T", "N:N27S", "N:N27K", "N:N27K", "N:Q28L", "N:Q28R", "N:Q28H", "N:Q28H", "N:N29Y", "N:N29H", "N:N29I", "N:N29T", "N:N29S", "N:N29K", "N:N29K", "N:G30*", "N:G30R", "N:G30R", "N:G30V", "N:G30A", "N:G30E", "N:E31*", "N:E31Q", "N:E31V", "N:E31A", "N:E31G", "N:E31D", "N:E31D", "N:R32S", "N:R32G", "N:R32L", "N:R32P", "N:R32H", "N:S33I", "N:S33T", "N:S33N", "N:S33R", "N:S33R", "N:G34V", "N:G34A", "N:G34E", "N:A35V", "N:A35E", "N:A35G", "N:R36L", "N:R36P", "N:R36Q", "N:S37T", "N:S37A", "N:S37L", "N:S37*", "N:S37*", "N:K38*", "N:K38Q", "N:K38I", "N:K38T", "N:K38R", "N:K38N", "N:K38N", "N:Q39K", "N:Q39E", "N:Q39L", "N:Q39P", "N:Q39R", "N:Q39H", "N:Q39H", "N:R40S", "N:R40G", "N:R40L", "N:R40P", "N:R40H", "N:R41L", "N:R41P", "N:R41Q", "N:P42L", "N:P42H", "N:P42R", "N:Q43L", "N:Q43P", "N:Q43R", "N:Q43H", "N:Q43H", "N:G44C", "N:G44R", "N:G44V", "N:G44A", "N:G44D", "N:L45S", "N:L45*", "N:L45*", "N:L45F", "N:L45F", "N:P46T", "N:P46A", "N:P46L", "N:P46H", "N:P46R", "N:N47I", "N:N47T", "N:N47S", "N:N47K", "N:N47K", "N:N48D", "N:N48I", "N:N48T", "N:N48S", "N:N48K", "N:N48K", "N:T49A", "N:T49N", "N:T49S", "N:A50V", "N:A50E", "N:A50G", "N:S51F", "N:S51Y", "N:S51C", "N:W52L", "N:W52S", "N:W52*", "N:W52C", "N:W52C", "N:W52*", "N:F53S", "N:F53Y", "N:F53C", "N:F53L", "N:F53L", "N:T54I", "N:T54N", "N:T54S", "N:A55V", "N:A55D", "N:A55G", "N:L56P", "N:L56H", "N:L56R", "N:T57I", "N:T57N", "N:T57S", "N:Q58L", "N:Q58P", "N:Q58R", "N:Q58H", "N:Q58H", "N:H59N", "N:H59D", "N:H59L", "N:H59P", "N:H59R", "N:H59Q", "N:H59Q", "N:G60C", "N:G60R", "N:G60S", "N:G60V", "N:G60A", "N:G60D", "N:K61M", "N:K61R", "N:K61N", "N:K61N", "N:E62*", "N:E62Q", "N:E62V", "N:E62A", "N:E62G", "N:E62D", "N:E62D", "N:D63Y", "N:D63H", "N:D63V", "N:D63A", "N:D63G", "N:D63E", "N:D63E", "N:G63D", "N:L64H", "N:L64R", "N:K65*", "N:K65Q", "N:K65I", "N:K65T", "N:K65R", "N:K65N", "N:K65N", "N:F66I", "N:F66V", "N:F66S", "N:F66Y", "N:F66C", "N:F66L", "N:F66L", "N:P67L", "N:P67H", "N:P67R", "N:R68L", "N:R68P", "N:R68Q", "N:G69*", "N:G69R", "N:G69V", "N:G69A", "N:G69E", "N:Q70K", "N:Q70E", "N:Q70L", "N:Q70P", "N:Q70R", "N:Q70H", "N:Q70H", "N:G71C", "N:G71R", "N:G71V", "N:G71A", "N:G71D", "N:V72A", "N:V72D", "N:V72G", "N:P73T", "N:P73A", "N:P73L", "N:P73Q", "N:P73R", "N:I74F", "N:I74L", "N:I74N", "N:I74S", "N:I74M", "N:N75Y", "N:N75H", "N:N75I", "N:N75T", "N:N75S", "N:N75K", "N:N75K", "N:T76I", "N:T76N", "N:T76S", "N:N77I", "N:N77T", "N:N77S", "N:N77K", "N:N77K", "N:S78G", "N:S78I", "N:S78T", "N:S78N", "N:S78R", "N:S78R", "N:S79I", "N:S79T", "N:S79N", "N:S79R", "N:S79R", "N:P80L", "N:P80Q", "N:P80R", "N:D81Y", "N:D81H", "N:D81V", "N:D81A", "N:D81G", "N:D81E", "N:D81E", "N:D82Y", "N:D82H", "N:D82N", "N:D82V", "N:D82A", "N:D82G", "N:D82E", "N:D82E", "N:Q83L", "N:Q83P", "N:Q83R", "N:Q83H", "N:Q83H", "N:I84F", "N:I84L", "N:I84N", "N:I84S", "N:I84M", "N:G85C", "N:G85R", "N:G85V", "N:G85A", "N:G85D", "N:Y86F", "N:Y86S", "N:Y86C", "N:Y86*", "N:Y86*", "N:Y87F", "N:Y87S", "N:Y87C", "N:Y87*", "N:Y87*", "N:R88L", "N:R88P", "N:R88Q", "N:R89*", "N:R89I", "N:R89T", "N:R89K", "N:R89S", "N:R89S", "N:A90S", "N:A90P", "N:A90D", "N:A90G", "N:T91I", "N:T91N", "N:T91S", "N:R92I", "N:R92T", "N:R92K", "N:R92S", "N:R92S", "N:R93G", "N:R93L", "N:R93P", "N:R93Q", "N:I94F", "N:I94L", "N:I94T", "N:I94N", "N:I94S", "N:I94M", "N:R95S", "N:R95G", "N:R95L", "N:R95P", "N:R95H", "N:G96V", "N:G96A", "N:G96D", "N:G97V", "N:G97A", "N:G97D", "N:D98V", "N:D98A", "N:D98G", "N:D98E", "N:D98E", "N:G99V", "N:G99A", "N:G99D", "N:K100I", "N:K100T", "N:K100R", "N:K100N", "N:K100N", "N:M101L", "N:M101L", "N:M101T", "N:M101K", "N:M101R", "N:M101I", "N:M101I", "N:K102*", "N:K102Q", "N:K102E"])            
# global  Double_N_ORF9b_muts = Set(["ORF9b:M1L", "ORF9b:M1L", "ORF9b:M1V", "ORF9b:M1K", "ORF9b:M1R", "ORF9b:M1I", "ORF9b:M1I", "ORF9b:M1I", "ORF9b:D2Y", "ORF9b:D2H", "ORF9b:D2N", "ORF9b:D2E", "ORF9b:D2E", "ORF9b:P3S", "ORF9b:P3T", "ORF9b:P3A", "ORF9b:K4*", "ORF9b:K4Q", "ORF9b:K4E", "ORF9b:K4I", "ORF9b:K4T", "ORF9b:K4N", "ORF9b:K4N", "ORF9b:I5F", "ORF9b:I5L", "ORF9b:I5V", "ORF9b:I5N", "ORF9b:I5S", "ORF9b:I5M", "ORF9b:S6C", "ORF9b:S6R", "ORF9b:S6G", "ORF9b:S6I", "ORF9b:S6T", "ORF9b:S6R", "ORF9b:E7*", "ORF9b:E7Q", "ORF9b:E7K", "ORF9b:E7D", "ORF9b:E7D", "ORF9b:M8L", "ORF9b:M8L", "ORF9b:M8V", "ORF9b:M8K", "ORF9b:M8R", "ORF9b:M8I", "ORF9b:M8I", "ORF9b:M8I", "ORF9b:H9Y", "ORF9b:H9N", "ORF9b:H9D", "ORF9b:H9Q", "ORF9b:P10S", "ORF9b:P10T", "ORF9b:P10A", "ORF9b:A11S", "ORF9b:A11P", "ORF9b:A11T", "ORF9b:L12I", "ORF9b:L12V", "ORF9b:L12*", "ORF9b:L12F", "ORF9b:L12F", "ORF9b:R13C", "ORF9b:R13S", "ORF9b:R13G", "ORF9b:L14M", "ORF9b:L14V", "ORF9b:L14*", "ORF9b:L14W", "ORF9b:L14F", "ORF9b:L14F", "ORF9b:V15L", "ORF9b:V15L", "ORF9b:V15M", "ORF9b:D16Y", "ORF9b:D16H", "ORF9b:D16N", "ORF9b:D16E", "ORF9b:D16E", "ORF9b:P17S", "ORF9b:P17T", "ORF9b:P17A", "ORF9b:Q18*", "ORF9b:Q18K", "ORF9b:Q18E", "ORF9b:Q18H", "ORF9b:Q18H", "ORF9b:I19F", "ORF9b:I19L", "ORF9b:I19V", "ORF9b:I19N", "ORF9b:I19S", "ORF9b:I19M", "ORF9b:Q20*", "ORF9b:Q20K", "ORF9b:Q20E", "ORF9b:Q20H", "ORF9b:Q20H", "ORF9b:L21M", "ORF9b:L21V", "ORF9b:A22S", "ORF9b:A22P", "ORF9b:A22T", "ORF9b:V23L", "ORF9b:V23L", "ORF9b:V23I", "ORF9b:V23E", "ORF9b:V23G", "ORF9b:T24S", "ORF9b:T24P", "ORF9b:T24A", "ORF9b:T24N", "ORF9b:T24S", "ORF9b:R25*", "ORF9b:R25G", "ORF9b:R25I", "ORF9b:R25T", "ORF9b:R25S", "ORF9b:R25S", "ORF9b:M26L", "ORF9b:M26L", "ORF9b:M26V", "ORF9b:M26K", "ORF9b:M26R", "ORF9b:M26I", "ORF9b:M26I", "ORF9b:M26I", "ORF9b:E27*", "ORF9b:E27Q", "ORF9b:E27K", "ORF9b:E27D", "ORF9b:E27D", "ORF9b:N28Y", "ORF9b:N28H", "ORF9b:N28D", "ORF9b:N28I", "ORF9b:N28T", "ORF9b:N28K", "ORF9b:N28K", "ORF9b:A29S", "ORF9b:A29P", "ORF9b:A29T", "ORF9b:V30L", "ORF9b:V30L", "ORF9b:V30M", "ORF9b:V30E", "ORF9b:V30G", "ORF9b:G31W", "ORF9b:G31R", "ORF9b:G31R", "ORF9b:R32C", "ORF9b:R32S", "ORF9b:R32G", "ORF9b:D33Y", "ORF9b:D33H", "ORF9b:D33N", "ORF9b:D33E", "ORF9b:D33E", "ORF9b:Q34*", "ORF9b:Q34K", "ORF9b:Q34E", "ORF9b:Q34H", "ORF9b:Q34H", "ORF9b:N35Y", "ORF9b:N35H", "ORF9b:N35D", "ORF9b:N35I", "ORF9b:N35T", "ORF9b:N35K", "ORF9b:N35K", "ORF9b:N36Y", "ORF9b:N36H", "ORF9b:N36D", "ORF9b:N36I", "ORF9b:N36T", "ORF9b:N36K", "ORF9b:N36K", "ORF9b:V37F", "ORF9b:V37L", "ORF9b:V37I", "ORF9b:G38C", "ORF9b:G38R", "ORF9b:G38S", "ORF9b:P39S", "ORF9b:P39T", "ORF9b:P39A", "ORF9b:K40*", "ORF9b:K40Q", "ORF9b:K40E", "ORF9b:K40M", "ORF9b:K40T", "ORF9b:K40N", "ORF9b:K40N", "ORF9b:V41F", "ORF9b:V41L", "ORF9b:V41I", "ORF9b:Y42H", "ORF9b:Y42N", "ORF9b:Y42D", "ORF9b:Y42F", "ORF9b:Y42S", "ORF9b:Y42*", "ORF9b:Y42*", "ORF9b:P43S", "ORF9b:P43T", "ORF9b:P43A", "ORF9b:I44L", "ORF9b:I44L", "ORF9b:I44V", "ORF9b:I44K", "ORF9b:I44R", "ORF9b:I44M", "ORF9b:I45L", "ORF9b:I45L", "ORF9b:I45V", "ORF9b:I45K", "ORF9b:I45R", "ORF9b:I45M", "ORF9b:L46M", "ORF9b:L46V", "ORF9b:R47C", "ORF9b:R47S", "ORF9b:R47G", "ORF9b:L48F", "ORF9b:L48I", "ORF9b:L48V", "ORF9b:G49C", "ORF9b:G49R", "ORF9b:G49S", "ORF9b:G49V", "ORF9b:G49A", "ORF9b:G49D", "ORF9b:S50P", "ORF9b:S50T", "ORF9b:S50A", "ORF9b:S50*", "ORF9b:S50*", "ORF9b:P51S", "ORF9b:P51T", "ORF9b:P51A", "ORF9b:L52F", "ORF9b:L52I", "ORF9b:L52V", "ORF9b:S53P", "ORF9b:S53T", "ORF9b:S53A", "ORF9b:L54F", "ORF9b:L54I", "ORF9b:L54V", "ORF9b:N55Y", "ORF9b:N55H", "ORF9b:N55D", "ORF9b:N55I", "ORF9b:N55T", "ORF9b:N55K", "ORF9b:N55K", "ORF9b:M56L", "ORF9b:M56L", "ORF9b:M56V", "ORF9b:M56K", "ORF9b:M56R", "ORF9b:M56I", "ORF9b:M56I", "ORF9b:M56I", "ORF9b:A57S", "ORF9b:A57P", "ORF9b:A57T", "ORF9b:R58W", "ORF9b:R58G", "ORF9b:R58M", "ORF9b:R58T", "ORF9b:R58S", "ORF9b:R58S", "ORF9b:K59*", "ORF9b:K59Q", "ORF9b:K59E", "ORF9b:K59M", "ORF9b:K59T", "ORF9b:K59N", "ORF9b:K59N", "ORF9b:T60S", "ORF9b:T60P", "ORF9b:T60A", "ORF9b:T60N", "ORF9b:T60S", "ORF9b:L61I", "ORF9b:L61V", "ORF9b:L61F", "ORF9b:L61F", "ORF9b:N62Y", "ORF9b:N62H", "ORF9b:N62D", "ORF9b:N62I", "ORF9b:N62T", "ORF9b:N62K", "ORF9b:N62K", "ORF9b:S63P", "ORF9b:S63T", "ORF9b:S63A", "ORF9b:S63Y", "ORF9b:S63C", "ORF9b:L64F", "ORF9b:L64I", "ORF9b:L64V", "ORF9b:E65*", "ORF9b:E65Q", "ORF9b:E65K", "ORF9b:E65D", "ORF9b:E65D", "ORF9b:D66Y", "ORF9b:D66H", "ORF9b:D66N", "ORF9b:D66E", "ORF9b:D66E", "ORF9b:K67*", "ORF9b:K67Q", "ORF9b:K67E", "ORF9b:K67M", "ORF9b:K67T", "ORF9b:K67N", "ORF9b:K67N", "ORF9b:A68S", "ORF9b:A68P", "ORF9b:A68T", "ORF9b:F69L", "ORF9b:F69I", "ORF9b:F69V", "ORF9b:F69L", "ORF9b:F69L", "ORF9b:Q70*", "ORF9b:Q70K", "ORF9b:Q70E", "ORF9b:Q70H", "ORF9b:Q70H", "ORF9b:L71I", "ORF9b:L71V", "ORF9b:L71*", "ORF9b:L71F", "ORF9b:L71F", "ORF9b:T72S", "ORF9b:T72P", "ORF9b:T72A", "ORF9b:T72K", "ORF9b:T72R", "ORF9b:P73S", "ORF9b:P73T", "ORF9b:P73A", "ORF9b:I74L", "ORF9b:I74L", "ORF9b:I74V", "ORF9b:I74K", "ORF9b:I74R", "ORF9b:I74M", "ORF9b:A75S", "ORF9b:A75P", "ORF9b:A75T", "ORF9b:A75E", "ORF9b:A75G", "ORF9b:V76F", "ORF9b:V76L", "ORF9b:V76I", "ORF9b:V76D", "ORF9b:V76G", "ORF9b:Q77*", "ORF9b:Q77K", "ORF9b:Q77E", "ORF9b:Q77H", "ORF9b:Q77H", "ORF9b:M78L", "ORF9b:M78L", "ORF9b:M78V", "ORF9b:M78K", "ORF9b:M78R", "ORF9b:M78I", "ORF9b:M78I", "ORF9b:M78I", "ORF9b:T79S", "ORF9b:T79P", "ORF9b:T79A", "ORF9b:T79N", "ORF9b:T79S", "ORF9b:K80*", "ORF9b:K80Q", "ORF9b:K80E", "ORF9b:K80I", "ORF9b:K80T", "ORF9b:K80N", "ORF9b:K80N", "ORF9b:L81M", "ORF9b:L81V", "ORF9b:L81W", "ORF9b:L81F", "ORF9b:L81F", "ORF9b:A82S", "ORF9b:A82P", "ORF9b:A82T", "ORF9b:T83S", "ORF9b:T83P", "ORF9b:T83A", "ORF9b:T83N", "ORF9b:T83S", "ORF9b:T84S", "ORF9b:T84P", "ORF9b:T84A", "ORF9b:T84N", "ORF9b:T84S", "ORF9b:E85*", "ORF9b:E85Q", "ORF9b:E85K", "ORF9b:E85D", "ORF9b:E86*", "ORF9b:E86Q", "ORF9b:E86K", "ORF9b:E86V", "ORF9b:E86A", "ORF9b:E86D", "ORF9b:E86D", "ORF9b:L87I", "ORF9b:L87V", "ORF9b:P88S", "ORF9b:P88T", "ORF9b:P88A", "ORF9b:D89Y", "ORF9b:D89H", "ORF9b:D89N", "ORF9b:D89V", "ORF9b:D89A", "ORF9b:D89E", "ORF9b:E90*", "ORF9b:E90Q", "ORF9b:E90K", "ORF9b:E90D", "ORF9b:E90D", "ORF9b:F91L", "ORF9b:F91I", "ORF9b:F91V", "ORF9b:F91C", "ORF9b:F91L", "ORF9b:F91L", "ORF9b:V92L", "ORF9b:V92L", "ORF9b:V92M", "ORF9b:V93L", "ORF9b:V93L", "ORF9b:V93M", "ORF9b:V94L", "ORF9b:V94L", "ORF9b:V94M", "ORF9b:T95S", "ORF9b:T95P", "ORF9b:T95A", "ORF9b:T95K", "ORF9b:T95R", "ORF9b:V96L", "ORF9b:V96L", "ORF9b:V96I", "ORF9b:K97*", "ORF9b:K97Q", "ORF9b:K97E", "ORF9b:K97I", "ORF9b:K97T", "ORF9b:K97N", "ORF9b:K97N", "ORF9b:*98R", "ORF9b:*98R", "ORF9b:*98G", "ORF9b:*98L", "ORF9b:*98S", "ORF9b:*98C", "ORF9b:*98C", "ORF9b:*98W"])
######################################################################################################################################
#### BAL_muts are all the mutations found that are part of the bronchoalveolar lavage mutation pattern (determined by a separate
####    function described in the paper and which is available for viewing on Github).
#### Like RBM_muts and artifactual_private_muts, the original mutation patterns are entirely unaffected by these mutations. However,
####    it is necessary to exclude them from serving **as seed mutations** in the control function, which tests all other mutations 
####    that occur more than 5 times in the EPCI dataset. Otherwise, a large number of BAL-associated mut patterns are reproduced. 
####    A few will sneak by anyway.
global BAL_muts_OG = list_to_strings_set("ORF1a:T2183I, ORF1a:S2972F, ORF1a:S2972P, ORF1a:A3049V, ORF1a:T3058I, ORF1a:A3070V, ORF1a:G3072C, ORF1a:H3076Y, ORF1a:L3123F, ORF1a:S3195G, ORF1a:P3359L, ORF1a:A3454V, ORF1a:A3456V, ORF1a:Q4110R, ORF1a:T4175I, ORF1a:P4197S, ORF1a:I4205V, ORF1a:T4207A, ORF1a:R4387S, ORF1b:L314P, ORF1b:I1181T, ORF1b:Y1247C, ORF1b:T1424I, ORF1b:S2339F, ORF1b:K2340N, ORF1b:T2537I, S:S50L, S:P330S, S:N354D, S:V367F, S:F374L, S:F375-, S:A376V, S:N405D, S:Y508H, S:V551I, S:T573I, S:A647S, S:A653V, S:N657K, S:S659P, S:A668V, S:T859I, S:A944T, S:A958D, S:N978D, S:I1169T, S:I1179T, S:L1186F, E:V5A, E:V5F, E:T9I, E:G10S, E:G10C, E:I13-, E:V14-, E:S16N, E:F23S, E:T30I, E:A36V, E:Y42C, M:A2V, M:S4P, M:V10I, M:F28S, M:T77N, M:S94R, M:S94N, M:H125Y, M:H148R, M:A188T, N:P80L, N:S416L, N:T417I, ORF7a:T115I, ORF9b:M1T, ORF9b:Q77*") 
artifactual_private_muts_pos_only = Set{String}()
#push!(artifactual_private_muts_pos_only, "ORF1a:2501) ## Artifactual reversion in B.1 sequences (possibly real? Unclear)
push!(artifactual_private_muts_pos_only, "ORF1a:2606") ## Inherited B.1.2 mutation
push!(artifactual_private_muts_pos_only, "N:377")      ## Inherited B.1.2 mutation
push!(artifactual_private_muts_pos_only, "ORF3a:72")   ## Inherited B.1.25 8 mutation
push!(artifactual_private_muts_pos_only, "ORF1a:3764") ## Inherited B.1.258 mutation
push!(artifactual_private_muts_pos_only, "ORF1a:2283") ## Artifactual reversion in XEC miscategorized as KP.3 or KP.3.3 by Nextclade
push!(artifactual_private_muts_pos_only, "ORF1a:599")  ## Artifactual "private" mutation in XEC miscategorized as KP.3 or KP.3.3 by Nextclade
push!(artifactual_private_muts_pos_only, "S:59")       ## Artifactual "private" mutation in XEC miscategorized as KP.3 or KP.3.3 by Nextclade
push!(artifactual_private_muts_pos_only, "ORF8:L60")   ## Artifactual "private" mutation in AY.44. Over 125000 seqs have both ORF8:L60F & ORF1a:2125Y
push!(artifactual_private_muts_pos_only, "ORF1a:2125") ## Artifactual "private" mutation in AY.44. Over 125000 seqs have both ORF8:L60F & ORF1a:2125Y
push!(artifactual_private_muts_pos_only, "S:146")      ## Extremely homoplasic XBB mutation and/or artifact. Impossible to tell if inherited or private. 
#push!(artifactual_private_muts_pos_only, "N:203")      ## Recombinant Delta/Omicron "mutation"
#push!(artifactual_private_muts_pos_only, "N:204")      ## Recombinant Delta/Omicron "mutation"
#push!(artifactual_private_muts_pos_only, "N:204")      ## Recombinant Delta/Omicron reversion "mutation"
push!(artifactual_private_muts_pos_only, "ORF1b:1156") ## BQ.1 mutation misattributed as private in 3 sequences
push!(artifactual_private_muts_pos_only, "ORF:61")     ## Artifactual 3-nuc reversion
push!(artifactual_private_muts_pos_only, "ORF1a:135")  ## Omicron mut misattributed as private in 3 recombinants
#push!(artifactual_private_muts_pos_only, "M:30")       ## Very common artifactual reversion in BA.2.86* lineages
push!(artifactual_private_muts_pos_only, "ORF1a:1612") ## Inherited Beta mutation miscategorized by Nextclade as private
push!(artifactual_private_muts_pos_only, "ORF7b:39")   ## Inherited Beta mutation miscategorized by Nextclade as private
push!(artifactual_private_muts_pos_only, "S:18")       ## All five on list in Beta, likely artifactual
push!(artifactual_private_muts_pos_only, "ORF1a:2554") ## Inherited AY.44 mutation miscategorized by Nextclade as private 
push!(artifactual_private_muts_pos_only, "ORF1b:1087") ## Inherited AY.44 mutation miscategorized by Nextclade as private 
push!(artifactual_private_muts_pos_only, "ORF1a:2796") ## Inherited BA.1.17 mutation miscategorized by Nextclade as private 
push!(artifactual_private_muts_pos_only, "ORF1a:1803") ## Inherited BA.1.17 mutation miscategorized by Nextclade as private
push!(artifactual_private_muts_pos_only, "ORF3a:104")  ## Inherited AY.103 mutation miscategorized by Nextclade as private 
push!(artifactual_private_muts_pos_only, "N:208")      ## Inherited AY.103 mutation miscategorized by Nextclade as private 
push!(artifactual_private_muts_pos_only, "ORF8:68")    ## Inherited B.1.1.7 mutation miscategorized by Nextclade as private
push!(artifactual_private_muts_pos_only, "ORF1b:1975") ## Inherited B.1.1.7 mutation miscategorized by Nextclade as private
push!(artifactual_private_muts_pos_only, "ORF1b:2178") ## Inherited BA.1.1 mutation miscategorized by Nextclade as private
push!(artifactual_private_muts_pos_only, "ORF1b:97")   ## Inherited BA.1.1 mutation miscategorized by Nextclade as private
push!(artifactual_private_muts_pos_only, "ORF1b:2589") ## Inherited BA.5.1 mutation miscategorized by Nextclade as private
push!(artifactual_private_muts_pos_only, "ORF9b:83")   ## Inherited BA.5.1 mutation miscategorized by Nextclade as private
push!(artifactual_private_muts_pos_only, "ORF1a:842")  ## From 5' End Recombination with BA.2
push!(artifactual_private_muts_pos_only, "ORF1a:135")  ## From 5' End Recombination with BA.2
push!(artifactual_private_muts_pos_only, "ORF3a:48")   ## Inherited BE.1 mutation miscategorized by Nextclade as private
push!(artifactual_private_muts_pos_only, "ORF3a:49")   ## Inherited BE.1 mutation miscategorized by Nextclade as private
push!(artifactual_private_muts_pos_only, "ORF1a:1204") ## Inherited BE.1 mutation miscategorized by Nextclade as private
#push!(artifactual_private_muts_pos_only, "S:376")      ## Butchered mutation interpretation by Nextclade for ∆374-375 or ∆374-376
#push!(artifactual_private_muts_pos_only, "S:376")      ## Butchered mutation interpretation by Nextclade for ∆374-375 or ∆374-376
#push!(artifactual_private_muts_pos_only, "S:375")      ## Butchered mutation interpretation by Nextclade for ∆374-375 or ∆374-376
#push!(artifactual_private_muts_pos_only, "S:375")      ## Butchered mutation interpretation by Nextclade for ∆374-375 or ∆374-376
#push!(artifactual_private_muts_pos_only, "S:339")      ## An artifactual reversion >90% of the time, likely 100%
#push!(artifactual_private_muts_pos_only, "S:417")      ## An artifactual reversion >90% of the time, likely 100%
#push!(artifactual_private_muts_pos_only, "S:440")      ## An artifactual reversion >90% of the time, likely 100%
push!(artifactual_private_muts_pos_only, "ORF1b:1555") ## Inherited B.1.2 mutation miscategorized by Nextclade as private
push!(artifactual_private_muts_pos_only, "ORF1a:4285") ## Inherited B.1.2 mutation miscategorized by Nextclade as private
push!(artifactual_private_muts_pos_only, "ORF1a:2361") ## Inherited B.1.2 mutation miscategorized by Nextclade as private
push!(artifactual_private_muts_pos_only, "ORF1a:3782") ## Inherited BA.1 mutation miscategorized by Nextclade as private
push!(artifactual_private_muts_pos_only, "S:1191")     ## Inherited B.1.1.7 mutation miscategorized by Nextclade as private
push!(artifactual_private_muts_pos_only, "ORF7b:44")   ## Deletions here that maintain the stop codon get misinterpreted as substitutions
push!(artifactual_private_muts_pos_only, "ORF1a:3646") ## Inherited Delta mut, miscategorized in recombinants 
push!(artifactual_private_muts_pos_only, "ORF1b:1918") ## Inherited Delta mut, miscategorized in recombinants 
push!(artifactual_private_muts_pos_only, "ORF7b:1")    ## Necessarily overlaps with ORF7a:122 mutations/deletions
push!(artifactual_private_muts_pos_only, "ORF7b:2")    ## Overlaps with ORF7a:122 mutations/deletions
push!(artifactual_private_muts_pos_only, "ORF1b:218")  ## Inherited B.1.1.7 mutation miscategorized by Nextclade as private
push!(artifactual_private_muts_pos_only, "ORF1a:3782") ## Inherited mutation in Canadian BA.1 branch miscategorized by Nextclade as private
push!(artifactual_private_muts_pos_only, "ORF1a:2118") ## B.1 Inherited mutation miscategorized by Nextclade as private (co-occurs with S:P681H)
push!(artifactual_private_muts_pos_only, "ORF1a:3646") ## Inherited Delta mut, miscategorized in recombinants 
push!(artifactual_private_muts_pos_only, "ORF1b:1918") ## Inherited Delta mut, miscategorized in recombinants
push!(artifactual_private_muts_pos_only, "ORF1a:1298") ## Inherited BA.1 mut, miscategorized by Nextclade as private

push!(artifactual_private_muts_pos_only, "ORF3a:112")  ## Inherited B.1.1 mutation miscategorized by Nextclade as private
push!(artifactual_private_muts_pos_only, "ORF1b:2695") ## Inherited B.1.1 mutation miscategorized by Nextclade as private
push!(artifactual_private_muts_pos_only, "ORF3a:35")   ## Inherited XBB.1.16.11 mutation miscategorized by Nextclade as private
push!(artifactual_private_muts_pos_only, "ORF1b:1903") ## Inherited XBB.1.16.31 mutation miscategorized by Nextclade as private
push!(artifactual_private_muts_pos_only, "ORF1b:1094") ## Inherited B.1.2 mutation miscategorized by Nextclade as private
push!(artifactual_private_muts_pos_only, "ORF1a:771")  ## Inherited XBB.1.5 mutation miscategorized by Nextclade as private 

push!(artifactual_private_muts_pos_only, "ORF1a:108")  ## Inherited BA.5.1 mutation in Scandinavian branch miscategorized by Nextclade as private
push!(artifactual_private_muts_pos_only, "ORF1a:365")  ## Inherited B.1.2 mutation miscategorized by Nextclade as private
push!(artifactual_private_muts_pos_only, "ORF1a:397")  ## Inherited B.1.466 (Indonesia) mutation miscategorized by Nextclade as private
push!(artifactual_private_muts_pos_only, "ORF1a:925")  ## Inherited BN.1.3.1 mutation (Italian branch) miscategorized by Nextclade as private
push!(artifactual_private_muts_pos_only, "ORF1a:1015") ## Inherited KF.1 (FL.15.1.1.1) mutation miscategorized by Nextclade as private
push!(artifactual_private_muts_pos_only, "ORF1a:3459") ## Inherited B.1.429 mutation miscategorized by Nextclade as private
push!(artifactual_private_muts_pos_only, "ORF1b:128")  ## Inherited BA.5.1 mutation in Scandinavian branch miscategorized by Nextclade as private
push!(artifactual_private_muts_pos_only, "ORF1b:869")  ## Inherited KF.1 (FL.15.1.1.1) mutation miscategorized by Nextclade as private
push!(artifactual_private_muts_pos_only, "ORF1b:1094") ## Inherited B.1.2 mutation miscategorized by Nextclade as private
push!(artifactual_private_muts_pos_only, "ORF1b:1871") ## Inherited B.1.1.7 mutation miscategorized by Nextclade as private
push!(artifactual_private_muts_pos_only, "ORF1b:2452") ## Inherited BA.5.1 mutation in Scandinavian branch miscategorized by Nextclade as private
push!(artifactual_private_muts_pos_only, "ORF7b:4")    ## Inherited BA.1 mutation on Swedish branch miscategorized by Nextclade as private
######################################################################################################################################
#### Double_N_ORF9b_muts are muts that result in both N and ORF9b mutations, which of course correlate perfectly but trivially.
####    Default is to block N muts, but it's also possible to change that and instead block ORF9b muts to view their N equivalents
####    by commenting out the top list and uncommenting out the bottom list.
global Double_N_ORF9b_muts_pos_only = Set(["N:4", "N:5", "N:6", "N:7", "N:8", "N:9", "N:10", "N:11", "N:12", "N:13", "N:14", "N:15", "N:16", "N:17", "N:18", "N:19", "N:20", "N:21", "N:22", "N:23", "N:24", "N:25", "N:26", "N:27", "N:28", "N:29", "N:30", "N:31", "N:32", "N:33", "N:34", "N:35", "N:36", "N:37", "N:38", "N:39", "N:40", "N:41", "N:42", "N:43", "N:44", "N:45", "N:46", "N:47", "N:48", "N:49", "N:50", "N:51", "N:52", "N:53", "N:54", "N:55", "N:56", "N:57", "N:58", "N:59", "N:60", "N:61", "N:62", "N:63", "N:64", "N:65", "N:66", "N:67", "N:68", "N:69", "N:70", "N:71", "N:72", "N:73", "N:74", "N:75", "N:76", "N:77", "N:78", "N:79", "N:80", "N:81", "N:82", "N:83", "N:84", "N:85", "N:86", "N:87", "N:88", "N:89", "N:90", "N:91", "N:92", "N:93", "N:94", "N:95", "N:96", "N:97", "N:98", "N:99", "N:100", "N:101"])
# global  Double_N_ORF9b_muts = Set(["ORF9b:1", "ORF9b:2", "ORF9b:3", "ORF9b:4", "ORF9b:5", "ORF9b:6", "ORF9b:7", "ORF9b:8", "ORF9b:9", "ORF9b:10", "ORF9b:11", "ORF9b:12", "ORF9b:13", "ORF9b:14", "ORF9b:15", "ORF9b:16", "ORF9b:17", "ORF9b:18", "ORF9b:19", "ORF9b:20", "ORF9b:21", "ORF9b:22", "ORF9b:23", "ORF9b:24", "ORF9b:25", "ORF9b:26", "ORF9b:27", "ORF9b:28", "ORF9b:29", "ORF9b:30", "ORF9b:31", "ORF9b:32", "ORF9b:33", "ORF9b:34", "ORF9b:35", "ORF9b:36", "ORF9b:37", "ORF9b:38", "ORF9b:39", "ORF9b:40", "ORF9b:41", "ORF9b:42", "ORF9b:43", "ORF9b:44", "ORF9b:45", "ORF9b:46", "ORF9b:47", "ORF9b:48", "ORF9b:49", "ORF9b:50", "ORF9b:51", "ORF9b:52", "ORF9b:53", "ORF9b:54", "ORF9b:55", "ORF9b:56", "ORF9b:57", "ORF9b:58", "ORF9b:59", "ORF9b:60", "ORF9b:61", "ORF9b:62", "ORF9b:63", "ORF9b:64", "ORF9b:65", "ORF9b:66", "ORF9b:67", "ORF9b:68", "ORF9b:69", "ORF9b:70", "ORF9b:71", "ORF9b:72", "ORF9b:73", "ORF9b:74", "ORF9b:75", "ORF9b:76", "ORF9b:77", "ORF9b:78", "ORF9b:79", "ORF9b:80", "ORF9b:81", "ORF9b:82", "ORF9b:83", "ORF9b:84", "ORF9b:85", "ORF9b:86", "ORF9b:87", "ORF9b:88", "ORF9b:89", "ORF9b:90", "ORF9b:91", "ORF9b:92", "ORF9b:93", "ORF9b:94", "ORF9b:95", "ORF9b:96", "ORF9b:97", "ORF9b:98"])
######################################################################################################################################
#### BAL_muts are all the mutations found that are part of the bronchoalveolar lavage mutation pattern (determined by a separate
####    function described in the paper and which is available for viewing on Github).
#### Like RBM_muts and artifactual_private_muts, the original mutation patterns are entirely unaffected by these mutations. However,
####    it is necessary to exclude them from serving **as seed mutations** in the control function, which tests all other mutations 
####    that occur more than 5 times in the EPCI dataset. Otherwise, a large number of BAL-associated mut patterns are reproduced. 
####    A few will sneak by anyway.
global BAL_muts_pos_only = list_to_strings_set("ORF1a:2183, ORF1a:2972, ORF1a:2972, ORF1a:3049, ORF1a:3058, ORF1a:3070, ORF1a:3072, ORF1a:3076, ORF1a:3123, ORF1a:3195, ORF1a:3359, ORF1a:3454, ORF1a:3456, ORF1a:4110, ORF1a:4175, ORF1a:4197, ORF1a:4205, ORF1a:4207, ORF1a:4387, ORF1a:314, ORF1b:1181, ORF1b:1247, ORF1b:1424, ORF1b:2339, ORF1b:2340, ORF1b:2537, S:50, S:330, S:354, S:367, S:374, S:375-, S:376, S:405, S:508, S:551, S:573, S:647, S:653, S:657, S:659, S:668, S:859, S:944, S:958, S:978, S:1169, S:1179, S:1186, E:5, E:5, E:9, E:10, E:10, E:13-, E:14-, E:16, E:23, E:30, E:36, E:42, M:2, M:4, M:10, M:28, M:77, M:94, M:94, M:125, M:148, M:188, N:80, N:416, N:417, ORF7a:115, ORF9b:1, ORF9b:77")
###########################################################################################################################################################################
#####################################################   BEGIN Sub/pos_only Section   ######################################################################################
###########################################################################################################################################################################
artifactual_private_muts = Set{String}()
pango_AAsub_WT_universal = Dict{String, Set{String}}
#pango_AAdel_WT_universal = Dict{String, String}()
global mp_folder_universal = ""
mp_chr_all_ratio = Dict()
if sub_0__posonly_1 == 0
    pango_AAsub_WT_universal = pango_AAsub_WT
    artifactual_private_muts = artifactual_private_muts_subs
    Double_N_ORF9b_muts = Double_N_ORF9b_muts
    BAL_muts = BAL_muts_OG
    mp_chr_all_ratio = AA_muts_ct_chr_all_ratio
elseif sub_0__posonly_1 == 1
    pango_AAsub_WT_universal = pango_AAsub_WT_pos_only
    artifactual_private_muts = artifactual_private_muts_pos_only
    Double_N_ORF9b_muts = Double_N_ORF9b_muts_pos_only
    BAL_muts = BAL_muts_pos_only
    mp_chr_all_ratio = AA_muts_ct_pos_only_no_dels_chr_all_ratio
end
for pango in pango_set
    mutset = pango_AAsub_WT_universal[pango]
    if "" in mutset
        println(pango)
    end
end
Bset = 
delete!(pango_AAsub_WT_universal["B"], "")
#################################################################################################
function sel_muts_pt1_pos_only_sort_key(n)
    if n == "" || isempty(n)
        return (0, 0)  # Return a default sort key for empty strings
    else
        return mp_AA_gene_pos_only_sortKey_2(string(split(n, ", ")[1]))
    end
end
############################################################################################
function sel_muts_pt1_sort_key_universal(n::String, sub_0__posonly_1::Int)
    if sub_0__posonly_1 == 0
        return sel_muts_pt1_sort_key(n)
    elseif sub_0__posonly_1 == 1
        return sel_muts_pt1_pos_only_sort_key(n)
    end
end
############################################################
function mp_AA_gene_sortKey_2_universal(n::String, sub_0__posonly_1::Int)
    if sub_0__posonly_1 == 0
        return mp_AA_gene_sortKey_2(n)
    elseif sub_0__posonly_1 == 1
        return mp_AA_gene_pos_only_sortKey_2(n)
    end
end    
############################################################
function AA_muts_ct_no_dels__sub__or__pos_only(sub_0__posonly_1::Int)
    if sub_0__posonly_1 == 0
        return AA_muts_ct_no_dels
    end
    if sub_0__posonly_1 == 1
        return AA_muts_ct_pos_only_no_dels
    end
end
AA_muts_ct_no_dels_universal = AA_muts_ct_no_dels__sub__or__pos_only(sub_0__posonly_1)
if AA_muts_ct_no_dels_universal == AA_muts_ct_no_dels
    println("AA_muts_ct_no_dels_universal == AA_muts_ct_no_dels")
else
    println("AA_muts_ct_no_dels_universal ≠ AA_muts_ct_no_dels")
end
if AA_muts_ct_no_dels_universal == AA_muts_ct_pos_only_no_dels
    println("AA_muts_ct_no_dels_universal == AA_muts_ct_pos_only_no_dels")
else
    println("AA_muts_ct_no_dels_universal ≠ AA_muts_ct_pos_only_no_dels")
end
############################################################
function seq_AA_muts_no_dels__sub__or__pos_only(sub_0__posonly_1::Int)
    if sub_0__posonly_1 == 0
        return seq_AA_muts_no_dels
    elseif sub_0__posonly_1 == 1
        return seq_AA_muts_pos_only_no_dels
    end
end
seq_AA_muts_no_dels_universal = seq_AA_muts_no_dels__sub__or__pos_only(sub_0__posonly_1)
####################################################################################
artifacts_ORF9bdoubles = union(artifactual_private_muts, Double_N_ORF9b_muts)
###########################################################################################################################################################################
###########################################################################################################################################################################
global RBM_muts = Set{String}()
global RBD_muts = Set{String}()
global RBD_not_RBM_muts = Set{String}()
global spike_not_RBD_muts = Set{String}()
global spike_muts = Set{String}()
global nonspike_muts = Set{String}()
RBD_sites = BitSet(335:528)
RBD_no_RBM_1 = BitSet(335:437)
RBD_no_RBM_2 = BitSet(507:528)
RBD_not_RBM_sites = union(RBD_no_RBM_1, RBD_no_RBM_2)
RBM_sites = BitSet(438:506)
spike_not_RBD1 = BitSet(1:334)
spike_not_RBD2 = BitSet(529:1273)
spike_not_RBD_sites = union(spike_not_RBD1, spike_not_RBD2)
for mut in keys(AA_muts_ct_no_dels_universal)
    if aa_gene_comprehensive_dict[mut] == "S"
        push!(spike_muts, mut)
    else
        push!(nonspike_muts, mut)
    end
    if aa_gene_comprehensive_dict[mut] == "S" && aa_pos_comprehensive_dict[mut] in RBD_not_RBM_sites
        push!(RBD_not_RBM_muts, mut)
    end
    if aa_gene_comprehensive_dict[mut] == "S" && aa_pos_comprehensive_dict[mut] in spike_not_RBD_sites
        push!(spike_not_RBD_muts, mut)
    end
    if aa_gene_comprehensive_dict[mut] == "S" && aa_pos_comprehensive_dict[mut] in RBM_sites
        push!(RBM_muts, mut)
    end
    if aa_gene_comprehensive_dict[mut] == "S" && aa_pos_comprehensive_dict[mut] in RBD_sites
        push!(RBD_muts, mut)
    end
end
###########################################################################################################################################################################
###########################################################################################################################################################################
global purespikebanned_0__purespikeallowed_1 = 0
global nonspike_seqfactor__0off__1RBM__2RBD__3SpikeNonRBD = 0
global all_excluded_muts = Set{String}()
if normal_0__spikeonly_1 == 0
    purespikebanned_0__purespikeallowed_1 = 0
    nonspike_seqfactor__0off__1RBM__2RBD__3SpikeNonRBD = 0
    if noBAL_0__withBAL_1 == 0
        all_excluded_muts = union(artifactual_private_muts, Double_N_ORF9b_muts, RBD_muts, BAL_muts)
    elseif noBAL_0__withBAL_1 == 1
        all_excluded_muts = union(artifactual_private_muts, Double_N_ORF9b_muts, RBD_muts)
    end
elseif normal_0__spikeonly_1 == 1
    purespikebanned_0__purespikeallowed_1 = 1
    nonspike_seqfactor__0off__1RBM__2RBD__3SpikeNonRBD = 3
    if noBAL_0__withBAL_1 == 0
        all_excluded_muts = union(artifactual_private_muts, Double_N_ORF9b_muts, RBD_muts, BAL_muts, nonspike_muts)
    elseif noBAL_0__withBAL_1 == 1
        all_excluded_muts = union(artifactual_private_muts, Double_N_ORF9b_muts, RBD_muts, nonspike_muts)
    end
end
###########################################################################################################################################################################
###########################################################################################################################################################################
RBD_not_RBM_muts_sort = sort(collect(RBD_not_RBM_muts), by = x -> aa_pos_comprehensive_dict[x])
RBD_not_RBM_tot = length(RBD_not_RBM_muts_sort)
println("Total Different RBD-not-RBM Muts = $(RBD_not_RBM_tot)")
RBD_not_RBM_muts_sort_join = join(RBD_not_RBM_muts_sort, ", ")
########################
RBM_muts_sort = sort(collect(RBM_muts), by = x -> aa_pos_comprehensive_dict[x])
RBM_tot = length(RBM_muts_sort)
println("Total Different RBM Muts = $(RBM_tot)")
RBM_muts_sort_join = join(RBM_muts_sort, ", ")
########################
RBD_muts_sort = sort(collect(RBD_muts), by = x -> aa_pos_comprehensive_dict[x])
RBD_tot = length(RBD_muts_sort)
println("Total Different RBD Muts = $(RBD_tot)")
RBD_muts_sort_join = join(RBD_muts_sort, ", ")
########################
spike_not_RBD_muts_sort = sort(collect(spike_not_RBD_muts), by = x -> aa_pos_comprehensive_dict[x])
spike_not_RBD_tot = length(spike_not_RBD_muts_sort)
println("Total Different spike_not_RBD Muts = $(spike_not_RBD_tot)")
spike_not_RBD_muts_sort_join = join(spike_not_RBD_muts_sort, ", ")
#####################################################################################################################################
ORF9bNdoubles_artifacts = union(Double_N_ORF9b_muts, artifactual_private_muts)
tot_grp_ct = length(rep_seq_grps_maxmut_seqs)
tot_single_ct = length(non_rep_seqs)
tot_chr_seq_ct = tot_grp_ct + tot_single_ct
total_chronic_AA_ct = 0
total_chronic_AA_ct_nonRBM = 0
total_chronic_AA_ct_nonRBD = 0
total_chronic_AA_ct_spike_nonRBD = 0
total_chronic_AA_ct_nonspike = 0
for (mut, ct) in AA_muts_ct_no_dels_universal
    if !(mut in artifacts_ORF9bdoubles)
        total_chronic_AA_ct += ct
        if aa_gene_comprehensive_dict[mut] ≠ "S"
            total_chronic_AA_ct_nonspike += ct
            total_chronic_AA_ct_nonRBM += ct
            total_chronic_AA_ct_nonRBD += ct
        end
        if aa_gene_comprehensive_dict[mut] == "S" && !(aa_pos_comprehensive_dict[mut] in RBM_sites)
            total_chronic_AA_ct_nonRBM += ct
        end
        if aa_gene_comprehensive_dict[mut] == "S" && !(aa_pos_comprehensive_dict[mut] in RBD_sites)
            total_chronic_AA_ct_nonRBD += ct
            total_chronic_AA_ct_spike_nonRBD += ct
        end
    end
end
#####################################################################################################################
avg_AA_sub_ct_per_chronic_seq = round(digits=2, total_chronic_AA_ct/tot_chr_seq_ct)
println("avg_AA_sub_ct_per_chronic_seq = $(avg_AA_sub_ct_per_chronic_seq)")
avg_AA_sub_ct_per_chronic_seq_nonspike = round(digits=2, total_chronic_AA_ct_nonspike/tot_chr_seq_ct)
println("avg_AA_sub_ct_per_chronic_seq_nonspike = $(avg_AA_sub_ct_per_chronic_seq_nonspike)")
avg_AA_sub_ct_per_chronic_seq_nonRBM = round(digits=2, total_chronic_AA_ct_nonRBM/tot_chr_seq_ct)
println("avg_AA_sub_ct_per_chronic_seq_nonRBM = $(avg_AA_sub_ct_per_chronic_seq_nonRBM)")
avg_AA_sub_ct_per_chronic_seq_nonRBD = round(digits=2, total_chronic_AA_ct_nonRBD/tot_chr_seq_ct)
println("avg_AA_sub_ct_per_chronic_seq_nonRBD = $(avg_AA_sub_ct_per_chronic_seq_nonRBD)")
avg_AA_sub_ct_per_chronic_seq_spike_nonRBD = round(digits=2, total_chronic_AA_ct_spike_nonRBD/tot_chr_seq_ct)
println("avg_AA_sub_ct_per_chronic_seq_spike_nonRBD = $(avg_AA_sub_ct_per_chronic_seq_spike_nonRBD)")
######################################################################################################################################
#seq_privAA_nonRBD_len = Dict{String, Int}()
#for (seq, AAsubvec) in seq_AA_muts_no_dels_universal
#    seq_nonRBD_ct = 0
#    for sub in AAsubvec
#        if !(sub in RBD_muts)
#            seq_nonRBD_ct += 1
#        end
#    end
#    seq_privAA_nonRBD_len[seq] = seq_nonRBD_ct
#end
######################################################################################################################################
#seq_privAA_nonRBD_len_relative = Dict{String, Float64}()
#for (seq, ct) in seq_privAA_nonRBD_len
#    relative_nonRBD = round(digits=2, ct/avg_AA_sub_ct_per_chronic_seq_nonRBD)
#    seq_privAA_nonRBD_len_relative[seq] = relative_nonRBD
#end
#####################################################################################################################
global avg_AA_sub_ct_per_chronic_seq_for_main_fx = 0.0
#####################################################################################################################
if normal_0__spikeonly_1 == 0
    avg_AA_sub_ct_per_chronic_seq_for_main_fx = avg_AA_sub_ct_per_chronic_seq_nonRBD
elseif normal_0__spikeonly_1 == 1
    avg_AA_sub_ct_per_chronic_seq_for_main_fx = avg_AA_sub_ct_per_chronic_seq_spike_nonRBD
end
#####################################################################################################################
# all_excluded_muts
seq_privAA_len = Dict{String, Int}()
for (seq, AAsubvec) in seq_AA_muts_no_dels_universal
    seq_mut_ct = 0
    for sub in AAsubvec
        if !(sub in all_excluded_muts)
            seq_mut_ct += 1
        end
    end
    seq_privAA_len[seq] = seq_mut_ct
end
#####################################################################################################################
seq_privAA_len_relative = Dict{String, Float64}()
for (seq, ct) in seq_privAA_len
    relative_muts = round(digits=2, ct/avg_AA_sub_ct_per_chronic_seq_for_main_fx)
    seq_privAA_len_relative[seq] = relative_muts
end
#####################################################################################################################
total_chronic_AA_ct_v2 = 0
total_chronic_AA_ct_v2_nonRBM = 0
total_chronic_AA_ct_v2_nonRBD = 0
total_chronic_AA_ct_v2_spike_nonRBD = 0
total_chronic_AA_ct_v2_nonspike = 0
for seq in all_unique_chr_seqs_set
    for mut in seq_AA_muts_no_dels_universal[seq]
        if !(mut in artifacts_ORF9bdoubles)
            total_chronic_AA_ct_v2 += 1
            if aa_gene_comprehensive_dict[mut] ≠ "S"
                total_chronic_AA_ct_v2_nonspike += 1
                total_chronic_AA_ct_v2_nonRBM += 1
                total_chronic_AA_ct_v2_nonRBD += 1
            end
            if aa_gene_comprehensive_dict[mut] == "S" && !(aa_pos_comprehensive_dict[mut] in RBM_sites)
                total_chronic_AA_ct_v2_nonRBM += 1
            end
            if aa_gene_comprehensive_dict[mut] == "S" && !(aa_pos_comprehensive_dict[mut] in RBD_sites)
                total_chronic_AA_ct_v2_nonRBD += 1
                total_chronic_AA_ct_v2_spike_nonRBD += 1
            end
        end
    end
end
####################################################################################################################################
avg_AA_sub_ct_per_chronic_seq_v2 = round(digits=2, total_chronic_AA_ct_v2/all_unique_chr_seqs_ct)  #### Double checking the first count
println("avg_AA_sub_ct_per_chronic_seq_v2 = $(avg_AA_sub_ct_per_chronic_seq_v2)")
println("total_chronic_AA_ct = $(total_chronic_AA_ct)")
println("total_chronic_AA_ct_v2 = $(total_chronic_AA_ct_v2)")
avg_AA_sub_ct_per_chronic_seq_v2_nonspike = round(digits=2, total_chronic_AA_ct_v2_nonspike/all_unique_chr_seqs_ct)
println("avg_AA_sub_ct_per_chronic_seq_v2_nonspike = $(avg_AA_sub_ct_per_chronic_seq_v2_nonspike)")
println("total_chronic_AA_ct_nonspike = $(total_chronic_AA_ct_nonspike)")
println("total_chronic_AA_ct_v2_nonspike = $(total_chronic_AA_ct_v2_nonspike)")
avg_AA_sub_ct_per_chronic_seq_v2_nonRBM = round(digits=2, total_chronic_AA_ct_v2_nonRBM/all_unique_chr_seqs_ct)
println("avg_AA_sub_ct_per_chronic_seq_v2_nonRBM = $(avg_AA_sub_ct_per_chronic_seq_v2_nonRBM)")
println("total_chronic_AA_ct_nonRBM = $(total_chronic_AA_ct_nonRBM)")
println("total_chronic_AA_ct_v2_nonRBM = $(total_chronic_AA_ct_v2_nonRBM)")
avg_AA_sub_ct_per_chronic_seq_v2_nonRBD = round(digits=2, total_chronic_AA_ct_v2_nonRBD/all_unique_chr_seqs_ct)
println("avg_AA_sub_ct_per_chronic_seq_v2_nonRBD = $(avg_AA_sub_ct_per_chronic_seq_v2_nonRBD)")
println("total_chronic_AA_ct_nonRBD = $(total_chronic_AA_ct_nonRBD)")
println("total_chronic_AA_ct_v2_nonRBD = $(total_chronic_AA_ct_v2_nonRBD)")
avg_AA_sub_ct_per_chronic_seq_v2_spike_nonRBD = round(digits=2, total_chronic_AA_ct_v2_spike_nonRBD/all_unique_chr_seqs_ct)
println("avg_AA_sub_ct_per_chronic_seq_v2_spike_nonRBD = $(avg_AA_sub_ct_per_chronic_seq_v2_spike_nonRBD)")
println("total_chronic_AA_ct_spike_nonRBD = $(total_chronic_AA_ct_spike_nonRBD)")
println("total_chronic_AA_ct_v2_spike_nonRBD = $(total_chronic_AA_ct_v2_spike_nonRBD)"); print("\n"^1)
println("Finished!"); print("\n"^1)
##############################################################################################
runtime = time() - start
runtime_rd = round(digits=2, runtime)
runtime1, runtime2 = seconds_to_hrs_min_sec(runtime)
println("Runtime v0 = $(runtime) seconds")
println("Runtime v2 = $(runtime2)")
date_now = Dates.format(now(), "yyyy_mm_dd__IMMp"); println(date_now)
date_hour = Dates.format(now(), "yyyy_mm_dd_Ip"); print("\n"^1)
####################################################################################################################################
####################################################################################################################################


2026_03_08__420PM

AA_muts_ct_no_dels_universal == AA_muts_ct_no_dels
AA_muts_ct_no_dels_universal ≠ AA_muts_ct_pos_only_no_dels
Total Different RBD-not-RBM Muts = 278
Total Different RBM Muts = 337
Total Different RBD Muts = 615
Total Different spike_not_RBD Muts = 2143
avg_AA_sub_ct_per_chronic_seq = 17.62
avg_AA_sub_ct_per_chronic_seq_nonspike = 11.79
avg_AA_sub_ct_per_chronic_seq_nonRBM = 16.23
avg_AA_sub_ct_per_chronic_seq_nonRBD = 15.35
avg_AA_sub_ct_per_chronic_seq_spike_nonRBD = 3.56
avg_AA_sub_ct_per_chronic_seq_v2 = 17.62
total_chronic_AA_ct = 58077
total_chronic_AA_ct_v2 = 58077
avg_AA_sub_ct_per_chronic_seq_v2_nonspike = 11.79
total_chronic_AA_ct_nonspike = 38860
total_chronic_AA_ct_v2_nonspike = 38860
avg_AA_sub_ct_per_chronic_seq_v2_nonRBM = 16.23
total_chronic_AA_ct_nonRBM = 53525
total_chronic_AA_ct_v2_nonRBM = 53525
avg_AA_sub_ct_per_chronic_seq_v2_nonRBD = 15.35
total_chronic_AA_ct_nonRBD = 50611
total_chronic_AA_ct_v2_nonRBD = 50611
avg_AA_sub_ct_per_chronic_seq_v2_

In [7]:
### avg_AAsubs_per_seq_by_mutation fx #####################
date = Dates.format(today(), "yyyy_mm_dd")
avg_AAsubs_per_seq_by_mutation(5)




S:Q498Y      = 54.20 [seq_ct=  5]
S:L828F      = 51.27 [seq_ct= 11]
ORF3a:Q213L  = 50.20 [seq_ct=  5]
N:L221F      = 46.00 [seq_ct=  5]
ORF1a:L92F   = 44.80 [seq_ct=  5]
ORF1a:L3606V = 44.50 [seq_ct=  8]
S:D1153A     = 44.44 [seq_ct=  9]
S:F186I      = 43.50 [seq_ct=  6]
ORF1a:V38A   = 43.27 [seq_ct= 22]
ORF1b:F2314L = 42.33 [seq_ct= 21]
S:K1266N     = 42.00 [seq_ct=  5]
S:A264D      = 41.80 [seq_ct=  5]
S:H66Q       = 41.57 [seq_ct=  7]
S:A575S      = 41.20 [seq_ct=  5]
S:G181E      = 40.20 [seq_ct=  5]
S:A706V      = 40.20 [seq_ct=  5]
ORF1b:D870A  = 39.88 [seq_ct=  8]
S:D571G      = 39.20 [seq_ct=  5]
S:K417T      = 38.90 [seq_ct= 20]
M:I48M       = 38.80 [seq_ct=  5]
S:T19A       = 38.60 [seq_ct=  5]
ORF1a:C3059S = 37.83 [seq_ct=  6]
S:K187E      = 37.75 [seq_ct=  8]
ORF1a:S3732Y = 37.67 [seq_ct=  6]
S:I651V      = 37.67 [seq_ct=  6]
S:A1070V     = 37.57 [seq_ct=  7]
S:V1189A     = 37.57 [seq_ct=  7]
ORF1a:D4085E = 37.40 [seq_ct=  5]
S:K679R      = 37.36 [seq_ct= 11]
S:Q498H    

ORF8:P38S    = 24.44 [seq_ct=  9]
S:M153T      = 24.41 [seq_ct= 29]
ORF1a:I1276T = 24.40 [seq_ct= 20]
ORF1a:S167G  = 24.40 [seq_ct= 10]
ORF3a:S171L  = 24.38 [seq_ct= 34]
N:T366I      = 24.38 [seq_ct= 13]
ORF9b:A75V   = 24.38 [seq_ct=  8]
S:Y453F      = 24.37 [seq_ct= 59]
S:V486A      = 24.36 [seq_ct= 28]
ORF3a:T175I  = 24.36 [seq_ct= 14]
S:A1020V     = 24.33 [seq_ct= 15]
S:A372T      = 24.30 [seq_ct= 20]
S:T240I      = 24.30 [seq_ct= 10]
ORF1a:H3580Y = 24.27 [seq_ct= 44]
ORF1a:A516V  = 24.27 [seq_ct= 11]
S:S939F      = 24.26 [seq_ct= 35]
ORF9b:K67R   = 24.25 [seq_ct=  8]
ORF7a:S37F   = 24.25 [seq_ct=  8]
E:G10S       = 24.24 [seq_ct= 21]
N:A173V      = 24.22 [seq_ct=  9]
ORF1b:S1003G = 24.21 [seq_ct= 24]
ORF8:L118V   = 24.21 [seq_ct= 24]
S:D339G      = 24.21 [seq_ct= 19]
ORF1a:Q4099L = 24.21 [seq_ct= 14]
ORF1a:R3164H = 24.20 [seq_ct= 20]
S:P486F      = 24.17 [seq_ct= 12]
S:A570V      = 24.15 [seq_ct= 26]
ORF1b:L1220F = 24.14 [seq_ct= 21]
ORF7a:E95Q   = 24.14 [seq_ct= 14]
ORF9b:I5T    =

E:T11A       = 21.91 [seq_ct= 22]
S:A484E      = 21.90 [seq_ct= 52]
ORF1b:T76I   = 21.90 [seq_ct= 20]
ORF1a:T283I  = 21.89 [seq_ct=  9]
S:S940F      = 21.89 [seq_ct=  9]
ORF1a:A3657V = 21.88 [seq_ct= 17]
S:D1260N     = 21.88 [seq_ct=  8]
S:N556K      = 21.85 [seq_ct= 13]
ORF1a:S2900L = 21.84 [seq_ct= 19]
S:N370K      = 21.82 [seq_ct= 11]
ORF1a:I4205V = 21.80 [seq_ct= 10]
ORF1a:A3705V = 21.78 [seq_ct=  9]
ORF7a:A105V  = 21.77 [seq_ct= 66]
S:F157L      = 21.76 [seq_ct= 58]
ORF1a:T1322A = 21.76 [seq_ct= 37]
S:F375S      = 21.76 [seq_ct= 34]
ORF1a:T4088I = 21.76 [seq_ct= 25]
ORF3a:S40L   = 21.76 [seq_ct= 17]
S:E1258Q     = 21.75 [seq_ct= 12]
ORF1a:S1539N = 21.75 [seq_ct=  8]
ORF9b:T83I   = 21.75 [seq_ct=  8]
ORF9b:V93L   = 21.74 [seq_ct= 78]
ORF1a:T2093I = 21.67 [seq_ct= 12]
ORF1a:R4387S = 21.67 [seq_ct= 12]
N:P6T        = 21.67 [seq_ct= 12]
ORF9b:D2E    = 21.67 [seq_ct= 12]
S:I834V      = 21.67 [seq_ct=  9]
S:G446S      = 21.65 [seq_ct= 68]
M:L138I      = 21.65 [seq_ct= 17]
S:N960S      =

ORF1a:P1921L = 19.40 [seq_ct= 10]
S:R498Q      = 19.39 [seq_ct= 23]
S:K444T      = 19.36 [seq_ct= 36]
S:D839N      = 19.36 [seq_ct= 14]
S:G446D      = 19.35 [seq_ct= 34]
S:L5F        = 19.33 [seq_ct= 90]
S:L582F      = 19.33 [seq_ct=  9]
ORF1a:D4165A = 19.32 [seq_ct= 22]
ORF1b:P1452L = 19.32 [seq_ct= 19]
M:S4F        = 19.31 [seq_ct=113]
ORF1a:D2472E = 19.31 [seq_ct= 13]
S:D936H      = 19.30 [seq_ct=100]
S:P1162L     = 19.30 [seq_ct= 23]
S:N417I      = 19.29 [seq_ct= 35]
S:F490L      = 19.29 [seq_ct= 21]
M:D3G        = 19.29 [seq_ct= 21]
S:A67V       = 19.29 [seq_ct= 14]
S:P174S      = 19.27 [seq_ct= 15]
ORF1b:A1131V = 19.27 [seq_ct= 11]
S:H519R      = 19.25 [seq_ct=  8]
S:P25S       = 19.25 [seq_ct=  8]
S:P9L        = 19.24 [seq_ct=178]
S:G446R      = 19.24 [seq_ct= 21]
S:N405D      = 19.23 [seq_ct= 39]
S:R190K      = 19.22 [seq_ct= 23]
ORF1b:T385M  = 19.22 [seq_ct=  9]
ORF1a:A540V  = 19.22 [seq_ct=  9]
N:A208V      = 19.22 [seq_ct=  9]
ORF1a:T1572I = 19.22 [seq_ct=  9]
S:G75V       =

In [4]:
### Getting & printing all N/ORF9b Double Muts (separately, in two arrays) #################
date_now = Dates.format(now(), "yyyy_mm_dd__IMMp"); println(date_now)
############ Calculating ORF9b & N Double Muts #################
function ORF9b_N_nucmuts_to_AAres(pos::Int, mut::String, ref_pango::String)
    ref_seq = ""; refAA_ORF1a = ""; refAA_ORF1b = ""; refAA_S = ""; refAA_ORF3a = ""; refAA_E = ""; refAA_M = ""
    refAA_ORF6 = ""; refAA_ORF7a = ""; refAA_ORF7b = ""; refAA_ORF8 = ""; refAA_N = ""; refAA_ORF9b = ""
    if ref_pango == "Wuhan"
        ref_seq = "..."
        refAA_ORF1a = "MESLVPGFNEKTHVQLSLPVLQVRDVLVRGFGDSVEEVLSEARQHLKDGTCGLVEVEKGVLPQLEQPYVFIKRSDARTAPHGHVMVELVAELEGIQYGRSGETLGVLVPHVGEIPVAYRKVLLRKNGNKGAGGHSYGADLKSFDLGDELGTDPYEDFQENWNTKHSSGVTRELMRELNGGAYTRYVDNNFCGPDGYPLECIKDLLARAGKASCTLSEQLDFIDTKRGVYCCREHEHEIAWYTERSEKSYELQTPFEIKLAKKFDTFNGECPNFVFPLNSIIKTIQPRVEKKKLDGFMGRIRSVYPVASPNECNQMCLSTLMKCDHCGETSWQTGDFVKATCEFCGTENLTKEGATTCGYLPQNAVVKIYCPACHNSEVGPEHSLAEYHNESGLKTILRKGGRTIAFGGCVFSYVGCHNKCAYWVPRASANIGCNHTGVVGEGSEGLNDNLLEILQKEKVNINIVGDFKLNEEIAIILASFSASTSAFVETVKGLDYKAFKQIVESCGNFKVTKGKAKKGAWNIGEQKSILSPLYAFASEAARVVRSIFSRTLETAQNSVRVLQKAAITILDGISQYSLRLIDAMMFTSDLATNNLVVMAYITGGVVQLTSQWLTNIFGTVYEKLKPVLDWLEEKFKEGVEFLRDGWEIVKFISTCACEIVGGQIVTCAKEIKESVQTFFKLVNKFLALCADSIIIGGAKLKALNLGETFVTHSKGLYRKCVKSREETGLLMPLKAPKEIIFLEGETLPTEVLTEEVVLKTGDLQPLEQPTSEAVEAPLVGTPVCINGLMLLEIKDTEKYCALAPNMMVTNNTFTLKGGAPTKVTFGDDTVIEVQGYKSVNITFELDERIDKVLNEKCSAYTVELGTEVNEFACVVADAVIKTLQPVSELLTPLGIDLDEWSMATYYLFDESGEFKLASHMYCSFYPPDEDEEEGDCEEEEFEPSTQYEYGTEDDYQGKPLEFGATSAALQPEEEQEEDWLDDDSQQTVGQQDGSEDNQTTTIQTIVEVQPQLEMELTPVVQTIEVNSFSGYLKLTDNVYIKNADIVEEAKKVKPTVVVNAANVYLKHGGGVAGALNKATNNAMQVESDDYIATNGPLKVGGSCVLSGHNLAKHCLHVVGPNVNKGEDIQLLKSAYENFNQHEVLLAPLLSAGIFGADPIHSLRVCVDTVRTNVYLAVFDKNLYDKLVSSFLEMKSEKQVEQKIAEIPKEEVKPFITESKPSVEQRKQDDKKIKACVEEVTTTLEETKFLTENLLLYIDINGNLHPDSATLVSDIDITFLKKDAPYIVGDVVQEGVLTAVVIPTKKAGGTTEMLAKALRKVPTDNYITTYPGQGLNGYTVEEAKTVLKKCKSAFYILPSIISNEKQEILGTVSWNLREMLAHAEETRKLMPVCVETKAIVSTIQRKYKGIKIQEGVVDYGARFYFYTSKTTVASLINTLNDLNETLVTMPLGYVTHGLNLEEAARYMRSLKVPATVSVSSPDAVTAYNGYLTSSSKTPEEHFIETISLAGSYKDWSYSGQSTQLGIEFLKRGDKSVYYTSNPTTFHLDGEVITFDNLKTLLSLREVRTIKVFTTVDNINLHTQVVDMSMTYGQQFGPTYLDGADVTKIKPHNSHEGKTFYVLPNDDTLRVEAFEYYHTTDPSFLGRYMSALNHTKKWKYPQVNGLTSIKWADNNCYLATALLTLQQIELKFNPPALQDAYYRARAGEAANFCALILAYCNKTVGELGDVRETMSYLFQHANLDSCKRVLNVVCKTCGQQQTTLKGVEAVMYMGTLSYEQFKKGVQIPCTCGKQATKYLVQQESPFVMMSAPPAQYELKHGTFTCASEYTGNYQCGHYKHITSKETLYCIDGALLTKSSEYKGPITDVFYKENSYTTTIKPVTYKLDGVVCTEIDPKLDNYYKKDNSYFTEQPIDLVPNQPYPNASFDNFKFVCDNIKFADDLNQLTGYKKPASRELKVTFFPDLNGDVVAIDYKHYTPSFKKGAKLLHKPIVWHVNNATNKATYKPNTWCIRCLWSTKPVETSNSFDVLKSEDAQGMDNLACEDLKPVSEEVVENPTIQKDVLECNVKTTEVVGDIILKPANNSLKITEEVGHTDLMAAYVDNSSLTIKKPNELSRVLGLKTLATHGLAAVNSVPWDTIANYAKPFLNKVVSTTTNIVTRCLNRVCTNYMPYFFTLLLQLCTFTRSTNSRIKASMPTTIAKNTVKSVGKFCLEASFNYLKSPNFSKLINIIIWFLLLSVCLGSLIYSTAALGVLMSNLGMPSYCTGYREGYLNSTNVTIATYCTGSIPCSVCLSGLDSLDTYPSLETIQITISSFKWDLTAFGLVAEWFLAYILFTRFFYVLGLAAIMQLFFSYFAVHFISNSWLMWLIINLVQMAPISAMVRMYIFFASFYYVWKSYVHVVDGCNSSTCMMCYKRNRATRVECTTIVNGVRRSFYVYANGGKGFCKLHNWNCVNCDTFCAGSTFISDEVARDLSLQFKRPINPTDQSSYIVDSVTVKNGSIHLYFDKAGQKTYERHSLSHFVNLDNLRANNTKGSLPINVIVFDGKSKCEESSAKSASVYYSQLMCQPILLLDQALVSDVGDSAEVAVKMFDAYVNTFSSTFNVPMEKLKTLVATAEAELAKNVSLDNVLSTFISAARQGFVDSDVETKDVVECLKLSHQSDIEVTGDSCNNYMLTYNKVENMTPRDLGACIDCSARHINAQVAKSHNIALIWNVKDFMSLSEQLRKQIRSAAKKNNLPFKLTCATTRQVVNVVTTKIALKGGKIVNNWLKQLIKVTLVFLFVAAIFYLITPVHVMSKHTDFSSEIIGYKAIDGGVTRDIASTDTCFANKHADFDTWFSQRGGSYTNDKACPLIAAVITREVGFVVPGLPGTILRTTNGDFLHFLPRVFSAVGNICYTPSKLIEYTDFATSACVLAAECTIFKDASGKPVPYCYDTNVLEGSVAYESLRPDTRYVLMDGSIIQFPNTYLEGSVRVVTTFDSEYCRHGTCERSEAGVCVSTSGRWVLNNDYYRSLPGVFCGVDAVNLLTNMFTPLIQPIGALDISASIVAGGIVAIVVTCLAYYFMRFRRAFGEYSHVVAFNTLLFLMSFTVLCLTPVYSFLPGVYSVIYLYLTFYLTNDVSFLAHIQWMVMFTPLVPFWITIAYIICISTKHFYWFFSNYLKRRVVFNGVSFSTFEEAALCTFLLNKEMYLKLRSDVLLPLTQYNRYLALYNKYKYFSGAMDTTSYREAACCHLAKALNDFSNSGSDVLYQPPQTSITSAVLQSGFRKMAFPSGKVEGCMVQVTCGTTTLNGLWLDDVVYCPRHVICTSEDMLNPNYEDLLIRKSNHNFLVQAGNVQLRVIGHSMQNCVLKLKVDTANPKTPKYKFVRIQPGQTFSVLACYNGSPSGVYQCAMRPNFTIKGSFLNGSCGSVGFNIDYDCVSFCYMHHMELPTGVHAGTDLEGNFYGPFVDRQTAQAAGTDTTITVNVLAWLYAAVINGDRWFLNRFTTTLNDFNLVAMKYNYEPLTQDHVDILGPLSAQTGIAVLDMCASLKELLQNGMNGRTILGSALLEDEFTPFDVVRQCSGVTFQSAVKRTIKGTHHWLLLTILTSLLVLVQSTQWSLFFFLYENAFLPFAMGIIAMSAFAMMFVKHKHAFLCLFLLPSLATVAYFNMVYMPASWVMRIMTWLDMVDTSLSGFKLKDCVMYASAVVLLILMTARTVYDDGARRVWTLMNVLTLVYKVYYGNALDQAISMWALIISVTSNYSGVVTTVMFLARGIVFMCVEYCPIFFITGNTLQCIMLVYCFLGYFCTCYFGLFCLLNRYFRLTLGVYDYLVSTQEFRYMNSQGLLPPKNSIDAFKLNIKLLGVGGKPCIKVATVQSKMSDVKCTSVVLLSVLQQLRVESSSKLWAQCVQLHNDILLAKDTTEAFEKMVSLLSVLLSMQGAVDINKLCEEMLDNRATLQAIASEFSSLPSYAAFATAQEAYEQAVANGDSEVVLKKLKKSLNVAKSEFDRDAAMQRKLEKMADQAMTQMYKQARSEDKRAKVTSAMQTMLFTMLRKLDNDALNNIINNARDGCVPLNIIPLTTAAKLMVVIPDYNTYKNTCDGTTFTYASALWEIQQVVDADSKIVQLSEISMDNSPNLAWPLIVTALRANSAVKLQNNELSPVALRQMSCAAGTTQTACTDDNALAYYNTTKGGRFVLALLSDLQDLKWARFPKSDGTGTIYTELEPPCRFVTDTPKGPKVKYLYFIKGLNNLNRGMVLGSLAATVRLQAGNATEVPANSTVLSFCAFAVDAAKAYKDYLASGGQPITNCVKMLCTHTGTGQAITVTPEANMDQESFGGASCCLYCRCHIDHPNPKGFCDLKGKYVQIPTTCANDPVGFTLKNTVCTVCGMWKGYGCSCDQLREPMLQSADAQSFLNGFAV"
        refAA_ORF1b = "NRVCGVSAARLTPCGTGTSTDVVYRAFDIYNDKVAGFAKFLKTNCCRFQEKDEDDNLIDSYFVVKRHTFSNYQHEETIYNLLKDCPAVAKHDFFKFRIDGDMVPHISRQRLTKYTMADLVYALRHFDEGNCDTLKEILVTYNCCDDDYFNKKDWYDFVENPDILRVYANLGERVRQALLKTVQFCDAMRNAGIVGVLTLDNQDLNGNWYDFGDFIQTTPGSGVPVVDSYYSLLMPILTLTRALTAESHVDTDLTKPYIKWDLLKYDFTEERLKLFDRYFKYWDQTYHPNCVNCLDDRCILHCANFNVLFSTVFPPTSFGPLVRKIFVDGVPFVVSTGYHFRELGVVHNQDVNLHSSRLSFKELLVYAADPAMHAASGNLLLDKRTTCFSVAALTNNVAFQTVKPGNFNKDFYDFAVSKGFFKEGSSVELKHFFFAQDGNAAISDYDYYRYNLPTMCDIRQLLFVVEVVDKYFDCYDGGCINANQVIVNNLDKSAGFPFNKWGKARLYYDSMSYEDQDALFAYTKRNVIPTITQMNLKYAISAKNRARTVAGVSICSTMTNRQFHQKLLKSIAATRGATVVIGTSKFYGGWHNMLKTVYSDVENPHLMGWDYPKCDRAMPNMLRIMASLVLARKHTTCCSLSHRFYRLANECAQVLSEMVMCGGSLYVKPGGTSSGDATTAYANSVFNICQAVTANVNALLSTDGNKIADKYVRNLQHRLYECLYRNRDVDTDFVNEFYAYLRKHFSMMILSDDAVVCFNSTYASQGLVASIKNFKSVLYYQNNVFMSEAKCWTETDLTKGPHEFCSQHTMLVKQGDDYVYLPYPDPSRILGAGCFVDDIVKTDGTLMIERFVSLAIDAYPLTKHPNQEYADVFHLYLQYIRKLHDELTGHMLDMYSVMLTNDNTSRYWEPEFYEAMYTPHTVLQAVGACVLCNSQTSLRCGACIRRPFLCCKCCYDHVISTSHKLVLSVNPYVCNAPGCDVTDVTQLYLGGMSYYCKSHKPPISFPLCANGQVFGLYKNTCVGSDNVTDFNAIATCDWTNAGDYILANTCTERLKLFAAETLKATEETFKLSYGIATVREVLSDRELHLSWEVGKPRPPLNRNYVFTGYRVTKNSKVQIGEYTFEKGDYGDAVVYRGTTTYKLNVGDYFVLTSHTVMPLSAPTLVPQEHYVRITGLYPTLNISDEFSSNVANYQKVGMQKYSTLQGPPGTGKSHFAIGLALYYPSARIVYTACSHAAVDALCEKALKYLPIDKCSRIIPARARVECFDKFKVNSTLEQYVFCTVNALPETTADIVVFDEISMATNYDLSVVNARLRAKHYVYIGDPAQLPAPRTLLTKGTLEPEYFNSVCRLMKTIGPDMFLGTCRRCPAEIVDTVSALVYDNKLKAHKDKSAQCFKMFYKGVITHDVSSAINRPQIGVVREFLTRNPAWRKAVFISPYNSQNAVASKILGLPTQTVDSSQGSEYDYVIFTQTTETAHSCNVNRFNVAITRAKVGILCIMSDRDLYDKLQFTSLEIPRRNVATLQAENVTGLFKDCSKVITGLHPTQAPTHLSVDTKFKTEGLCVDIPGIPKDMTYRRLISMMGFKMNYQVNGYPNMFITREEAIRHVRAWIGFDVEGCHATREAVGTNLPLQLGFSTGVNLVAVPTGYVDTPNNTDFSRVSAKPPPGDQFKHLIPLMYKGLPWNVVRIKIVQMLSDTLKNLSDRVVFVLWAHGFELTSMKYFVKIGPERTCCLCDRRATCFSTASDTYACWHHSIGFDYVYNPFMIDVQQWGFTGNLQSNHDLYCQVHGNAHVASCDAIMTRCLAVHECFVKRVDWTIEYPIIGDELKINAACRKVQHMVVKAALLADKFPVLHDIGNPKAIKCVPQADVEWKFYDAQPCSDKAYKIEELFYSYATHSDKFTDGVCLFWNCNVDRYPANSIVCRFDTRVLSNLNLPGCDGGSLYVNKHAFHTPAFDKSAFVNLKQLPFFYYSDSPCESHGKQVVSDIDYVPLKSATCITRCNLGGAVCRHHANEYRLYLDAYNMMISAGFSLWVYKQFDTYNLWNTFTRLQSLENVAFNVVNKGHFDGQQGEVPVSIINNTVYTKVDGVDVELFENKTTLPVNVAFELWAKRNIKPVPEVKILNNLGVDIAANTVIWDYKRDAPAHISTIGVCSMTDIAKKPTETICAPLTVFFDGRVDGQVDLFRNARNGVLITEGSVKGLQPSVGPKQASLNGVTLIGEAVKTQFNYYKKVDGVVQQLPETYFTQSRNLQEFKPRSQMEIDFLELAMDEFIERYKLEGYAFEHIVYGDFSHSQLGGLHLLIGLAKRFKESPFELEDFIPMDSTVKNYFITDAQTGSSKCVCSVIDLLLDDFVEIIKSQDLSVVSKVVKVTIDYTEISFMLWCKDGHVETFYPKLQSSQAWQPGVAMPNLYKMQRMLLEKCDLQNYGDSATLPKGIMMNVAKYTQLCQYLNTLTLAVPYNMRVIHFGAGSDKGVAPGTAVLRQWLPTGTLLVDSDLNDFVSDADSTLIGDCATVHTANKWDLIISDMYDPKTKNVTKENDSKEGFFTYICGFIQQKLALGGSVAIKITEHSWNADLYKLMGHFAWWTAFVTNVNASSSEAFLIGCNYLGKPREQIDGYVMHANYIFWRNTNPIQLSSYSLFDMSKFPLKLRGTAVMSLKEGQINDMILSLLSKGRLIIRENNRVVISSDVLVNN"
        refAA_S = "MFVFLVLLPLVSSQCVNLTTRTQLPPAYTNSFTRGVYYPDKVFRSSVLHSTQDLFLPFFSNVTWFHAIHVSGTNGTKRFDNPVLPFNDGVYFASTEKSNIIRGWIFGTTLDSKTQSLLIVNNATNVVIKVCEFQFCNDPFLGVYYHKNNKSWMESEFRVYSSANNCTFEYVSQPFLMDLEGKQGNFKNLREFVFKNIDGYFKIYSKHTPINLVRDLPQGFSALEPLVDLPIGINITRFQTLLALHRSYLTPGDSSSGWTAGAAAYYVGYLQPRTFLLKYNENGTITDAVDCALDPLSETKCTLKSFTVEKGIYQTSNFRVQPTESIVRFPNITNLCPFGEVFNATRFASVYAWNRKRISNCVADYSVLYNSASFSTFKCYGVSPTKLNDLCFTNVYADSFVIRGDEVRQIAPGQTGKIADYNYKLPDDFTGCVIAWNSNNLDSKVGGNYNYLYRLFRKSNLKPFERDISTEIYQAGSTPCNGVEGFNCYFPLQSYGFQPTNGVGYQPYRVVVLSFELLHAPATVCGPKKSTNLVKNKCVNFNFNGLTGTGVLTESNKKFLPFQQFGRDIADTTDAVRDPQTLEILDITPCSFGGVSVITPGTNTSNQVAVLYQDVNCTEVPVAIHADQLTPTWRVYSTGSNVFQTRAGCLIGAEHVNNSYECDIPIGAGICASYQTQTNSPRRARSVASQSIIAYTMSLGAENSVAYSNNSIAIPTNFTISVTTEILPVSMTKTSVDCTMYICGDSTECSNLLLQYGSFCTQLNRALTGIAVEQDKNTQEVFAQVKQIYKTPPIKDFGGFNFSQILPDPSKPSKRSFIEDLLFNKVTLADAGFIKQYGDCLGDIAARDLICAQKFNGLTVLPPLLTDEMIAQYTSALLAGTITSGWTFGAGAALQIPFAMQMAYRFNGIGVTQNVLYENQKLIANQFNSAIGKIQDSLSSTASALGKLQDVVNQNAQALNTLVKQLSSNFGAISSVLNDILSRLDKVEAEVQIDRLITGRLQSLQTYVTQQLIRAAEIRASANLAATKMSECVLGQSKRVDFCGKGYHLMSFPQSAPHGVVFLHVTYVPAQEKNFTTAPAICHDGKAHFPREGVFVSNGTHWFVTQRNFYEPQIITTDNTFVSGNCDVVIGIVNNTVYDPLQPELDSFKEELDKYFKNHTSPDVDLGDISGINASVVNIQKEIDRLNEVAKNLNESLIDLQELGKYEQYIKWPWYIWLGFIAGLIAIVMVTIMLCCMTSCCSCLKGCCSCGSCCKFDEDDSEPVLKGVKLHYT"
        refAA_ORF3a = "MDLFMRIFTIGTVTLKQGEIKDATPSDFVRATATIPIQASLPFGWLIVGVALLAVFQSASKIITLKKRWQLALSKGVHFVCNLLLLFVTVYSHLLLVAAGLEAPFLYLYALVYFLQSINFVRIIMRLWLCWKCRSKNPLLYDANYFLCWHTNCYDYCIPYNSVTSSIVITSGDGTTSPISEHDYQIGGYTEKWESGVKDCVVLHSYFTSDYYQLYSTQLSTDTGVEHVTFFIYNKIVDEPEEHVQIHTIDGSSGVVNPVMEPIYDEPTTTTSVPL"
        refAA_E = "MYSFVSEETGTLIVNSVLLFLAFVVFLLVTLAILTALRLCAYCCNIVNVSLVKPSFYVYSRVKNLNSSRVPDLLV"
        refAA_M = "MADSNGTITVEELKKLLEQWNLVIGFLFLTWICLLQFAYANRNRFLYIIKLIFLWLLWPVTLACFVLAAVYRINWITGGIAIAMACLVGLMWLSYFIASFRLFARTRSMWSFNPETNILLNVPLHGTILTRPLLESELVIGAVILRGHLRIAGHHLGRCDIKDLPKEITVATSRTLSYYKLGASQRVAGDSGFAAYSRYRIGNYKLNTDHSSSSDNIALLVQ"
        refAA_ORF6 = "MFHLVDFQVTIAEILLIIMRTFKVSIWNLDYIINLIIKNLSKSLTENKYSQLDEEQPMEID"
        refAA_ORF7a = "MKIILFLALITLATCELYHYQECVRGTTVLLKEPCSSGTYEGNSPFHPLADNKFALTCFSTQFAFACPDGVKHVYQLRARSVSPKLFIRQEEVQELYSPIFLIVAAIVFITLCFTLKRKTE"
        refAA_ORF7b = "MIELSLIDFYLCFLAFLLFLVLIMLIIFWFSLELQDHNETCHA"
        refAA_ORF8 = "MKFLVFLGIITTVAAFHQECSLQSCTQHQPYVVDDPCPIHFYSKWYIRVGARKSAPLIELCVDEAGSKSPIQYIDIGNYTVSCLPFTINCQEPKLGSLVVRCSFYEDFLEYHDVRVVLDFI"
        refAA_N = "MSDNGPQNQRNAPRITFGGPSDSTGSNQNGERSGARSKQRRPQGLPNNTASWFTALTQHGKEDLKFPRGQGVPINTNSSPDDQIGYYRRATRRIRGGDGKMKDLSPRWYFYYLGTGPEAGLPYGANKDGIIWVATEGALNTPKDHIGTRNPANNAAIVLQLPQGTTLPKGFYAEGSRGGSQASSRSSSRSRNSSRNSTPGSSRGTSPARMAGNGGDAALALLLLDRLNQLESKMSGKGQQQQGQTVTKKSAAEASKKPRQKRTATKAYNVTQAFGRRGPEQTQGNFGDQELIRQGTDYKHWPQIAQFAPSASAFFGMSRIGMEVTPSGTWLTYTGAIKLDDKDPNFKDQVILLNKHIDAYKTFPPTEPKKDKKKKADETQALPQRQKKQQTVTLLPAADLDDFSKQLQQSMSSADSTQA"
        refAA_ORF9b = "MDPKISEMHPALRLVDPQIQLAVTRMENAVGRDQNNVGPKVYPIILRLGSPLSLNMARKTLNSLEDKAFQLTPIAVQMTKLATTEELPDEFVVVTVK"
    else
        ref_seq = nuc_genome_pango_dict[ref_pango]
        refAA_ORF1a = gene_AA_pango_dict[ref_pango]["ORF1a"]
        refAA_ORF1b = gene_AA_pango_dict[ref_pango]["ORF1b"]
        refAA_S = gene_AA_pango_dict[ref_pango]["S"]
        refAA_ORF3a = gene_AA_pango_dict[ref_pango]["ORF3a"]
        refAA_E = gene_AA_pango_dict[ref_pango]["E"]
        refAA_M = gene_AA_pango_dict[ref_pango]["M"]
        refAA_ORF6 = gene_AA_pango_dict[ref_pango]["ORF6"]
        refAA_ORF7a = gene_AA_pango_dict[ref_pango]["ORF7a"]
        refAA_ORF7b = gene_AA_pango_dict[ref_pango]["ORF7b"]
        refAA_ORF8 = gene_AA_pango_dict[ref_pango]["ORF8"]
        refAA_N = gene_AA_pango_dict[ref_pango]["N"]
        refAA_ORF9b = gene_AA_pango_dict[ref_pango]["ORF9b"]
    end
    ORF9b_ref_triple = ""
    ORF9b_qry_triple = ""
    if pos%3 == 0
        ORF9b_ref_triple = string(mut[1])*string(ref_seq[pos+1])*string(ref_seq[pos+2])
        ORF9b_qry_triple = string(mut[end])*string(ref_seq[pos+1])*string(ref_seq[pos+2])
    elseif pos%3 == 1
        ORF9b_ref_triple = string(ref_seq[pos-1])*string(mut[1])*string(ref_seq[pos+1])
        ORF9b_qry_triple = string(ref_seq[pos-1])*string(mut[end])*string(ref_seq[pos+1])
    elseif pos%3 == 2
        ORF9b_ref_triple = string(ref_seq[pos-2])*string(ref_seq[pos-1])*string(mut[1])
        ORF9b_qry_triple = string(ref_seq[pos-2])*string(ref_seq[pos-1])*string(mut[end])
    end
#################################################################################################
    N_ref_triple = ""
    N_qry_triple = ""
    if pos%3 == 2
        N_ref_triple = string(mut[1])*string(ref_seq[pos+1])*string(ref_seq[pos+2])
        N_qry_triple = string(mut[end])*string(ref_seq[pos+1])*string(ref_seq[pos+2])
    elseif pos%3 == 0
        N_ref_triple = string(ref_seq[pos-1])*string(mut[1])*string(ref_seq[pos+1])
        N_qry_triple = string(ref_seq[pos-1])*string(mut[end])*string(ref_seq[pos+1])
    elseif pos%3 == 1
        N_ref_triple = string(ref_seq[pos-2])*string(ref_seq[pos-1])*string(mut[1])
        N_qry_triple = string(ref_seq[pos-2])*string(ref_seq[pos-1])*string(mut[end])
    end
    ORF9b_ref_AA = AA_triplets[ORF9b_ref_triple]
    ORF9b_qry_AA = AA_triplets[ORF9b_qry_triple]
    N_ref_AA = AA_triplets[N_ref_triple]
    N_qry_AA = AA_triplets[N_qry_triple]
    return ORF9b_ref_AA, ORF9b_qry_AA, N_ref_AA, N_qry_AA
end
###################################################################################################################################
###################################################################################################################################
###################################################################################################################################
ref_seq = "..."
coding_ranges = BitSet([266:13467..., 13469:21555..., 21563:25384..., 25393:26220..., 26245:26472..., 26523:27191..., 27202:27387..., 27394:27755..., 27760:27887..., 27894:28259..., 28274:29533..., 28284:28577...])
noncoding_ranges = BitSet([1:265..., 21556:21562..., 25385:25392..., 26221:26244..., 26473:26522..., 27192:27201..., 27388:27393..., 27888:27893..., 28260:28273..., 29534:29830...])
coding_range_9b = BitSet([28284:28577...])
gene_nuc_starts = Dict{Int, Int}(0=>263, 1=>13465, 2=>21560, 3=>25390, 4=>26242, 5=>26520, 6=>27199, 7=>27391, 8=>27753, 9=>27891, 10=>28271, 11=>28281)
#ref_AA_genes = Dict{Int, String}(0=>refAA_ORF1a, 1=>refAA_ORF1b, 2=>refAA_S, 3=>refAA_ORF3a, 4=>refAA_E, 5=>refAA_M, 6=>refAA_ORF6, 7=>refAA_ORF7a, 8=>refAA_ORF7b, 9=>refAA_ORF8, 10=>refAA_N, 11=>refAA_ORF9b)
gene_strings = Dict{Int, String}(0=>"ORF1a", 1=>"ORF1b", 2=>"S", 3=>"ORF3a", 4=>"E", 5=>"M", 6=>"ORF6", 7=>"ORF7a", 8=>"ORF7b", 9=>"ORF8", 10=>"N", 11=>"ORF9b")
AA_gene(aaM) = split(aaM, ":")[1] ### Fx: #####################
AA_pos(aam) = parse(Int, split(aam, ":")[2][2:end-1]) ### Fx: #####################
AA_sort_key(AAm) = (AA_gene(AAm), AA_pos(AAm)) ### Fx: #####################
nuc_posit(n) = parse(Int, n[2:end-1]) ### Fx: #####################
nuc_to_AA_pos(nuc_mut) = string((nuc_posit(nuc_mut) - gene_nuc_starts[gene_num(nuc_mut)])÷3) ### Fx: #####################
nuc_to_AA_pos_N(nuc_mut) = string((nuc_posit(nuc_mut) - 28271)÷3) ### Fx: #####################
nuc_to_AA_pos_9b(nuc_mut) = string((nuc_posit(nuc_mut) - 28281)÷3) ### Fx: #####################
#############################################################
ORF9b_doubles = Vector{String}()
N_doubles = Vector{String}()
#############################################################
coding_range_9b = BitSet([28284:28577...])
nucs = ["T", "C", "A", "G"]
for nuc_site in coding_range_9b
    ref_nuc = string(ref_seq[nuc_site])
    for qry_nuc in nucs
        nuc_mut = "$(ref_nuc)$(nuc_site)$(qry_nuc)"
        ORF9b_refAA, ORF9b_qryAA, N_refAA, N_qryAA = ORF9b_N_nucmuts_to_AAres(nuc_site, nuc_mut, "Wuhan")
        N_AA_site = nuc_to_AA_pos_N(nuc_mut)
        ORF9b_AA_site = nuc_to_AA_pos_9b(nuc_mut)
        if ORF9b_refAA ≠ ORF9b_qryAA && N_refAA ≠ N_qryAA
            ORF9b_AA_mut = "ORF9b:$(ORF9b_refAA)$(ORF9b_AA_site)$(ORF9b_qryAA)"
            N_AA_mut = "N:$(N_refAA)$(N_AA_site)$(N_qryAA)"
            push!(ORF9b_doubles, ORF9b_AA_mut)
            push!(N_doubles, N_AA_mut)
        end
    end
end

ORF9b_doubles_join = join(ORF9b_doubles, "\", \"")
N_doubles_join = join(N_doubles, "\", \"")

print("\n"^2)
print("\"")
print(ORF9b_doubles_join)
println("\"")
print("\n"^6)
print("\"")
print(N_doubles_join)
println("\"")
print("\n"^2)
open("ORF9b_double_muts_$(date_hour).txt", "w") do g
    print(g, "\"")
    print(g, ORF9b_doubles_join)
    println(g, "\"")
end

open("N_double_muts_$(date_hour).txt", "w") do g
    print(g, "\"")
    print(g, N_doubles_join)
    println(g, "\"")
end
#####################################################################################################################################

2025-11-06__832PM


"ORF9b:M1L", "ORF9b:M1L", "ORF9b:M1V", "ORF9b:M1K", "ORF9b:M1R", "ORF9b:M1I", "ORF9b:M1I", "ORF9b:M1I", "ORF9b:D2Y", "ORF9b:D2H", "ORF9b:D2N", "ORF9b:D2E", "ORF9b:D2E", "ORF9b:P3S", "ORF9b:P3T", "ORF9b:P3A", "ORF9b:K4*", "ORF9b:K4Q", "ORF9b:K4E", "ORF9b:K4I", "ORF9b:K4T", "ORF9b:K4N", "ORF9b:K4N", "ORF9b:I5F", "ORF9b:I5L", "ORF9b:I5V", "ORF9b:I5N", "ORF9b:I5S", "ORF9b:I5M", "ORF9b:S6C", "ORF9b:S6R", "ORF9b:S6G", "ORF9b:S6I", "ORF9b:S6T", "ORF9b:S6R", "ORF9b:E7*", "ORF9b:E7Q", "ORF9b:E7K", "ORF9b:E7D", "ORF9b:E7D", "ORF9b:M8L", "ORF9b:M8L", "ORF9b:M8V", "ORF9b:M8K", "ORF9b:M8R", "ORF9b:M8I", "ORF9b:M8I", "ORF9b:M8I", "ORF9b:H9Y", "ORF9b:H9N", "ORF9b:H9D", "ORF9b:H9Q", "ORF9b:H9Q", "ORF9b:P10S", "ORF9b:P10T", "ORF9b:P10A", "ORF9b:A11S", "ORF9b:A11P", "ORF9b:A11T", "ORF9b:L12I", "ORF9b:L12V", "ORF9b:L12*", "ORF9b:L12F", "ORF9b:L12F", "ORF9b:R13C", "ORF9b:R13S", "ORF9b:R13G", "ORF9b:L14M", "ORF9b:L14V", "ORF9b:L14*", "ORF9b:L14W", "ORF9b:L14F", "ORF9b:L14F", "ORF9b:V15L

In [159]:
### CREATING FAKE CHRONICS LISTS FOR CONTROLS  |  Typical Runtime for 100 Runs = ~23 seconds 
date_now = Dates.format(now(), "yyyy_mm_dd__IMMp"); println(date_now)
date = Dates.format(today(), "yyyy_mm_dd")
start = time()
using Random                                                      
artifacts_ORF9bdoubles = union(artifactual_private_muts, Double_N_ORF9b_muts)
mp_meta_fake_chr_dict_DQ = Dict{Int, Dict{Int, Vector{String}}}()
mp_meta_fake_chr_DQ_bad_cts = Dict{Int, Int}()
number_of_runs = 100
         sizehint!(mp_meta_fake_chr_dict_DQ, number_of_runs)
         sizehint!(mp_meta_fake_chr_DQ_bad_cts, number_of_runs)
for p in 1:number_of_runs
    mp_meta_fake_chr_dict_DQ[p] = Dict{Int, Vector{String}}()
end
############################################################################################################################################################################
seq_AA_muts_not_excluded_fake = Dict{String, Set{String}}()
seq_AA_muts_not_excluded_fake_nonRBM = Dict{String, Set{String}}()
seq_AA_muts_not_excluded_fake_nonRBD = Dict{String, Set{String}}()
seq_AA_muts_not_excluded_fake_nonspike = Dict{String, Set{String}}()
seq_AA_muts_not_excluded_fake_RBM = Dict{String, Set{String}}()
seq_AA_muts_not_excluded_fake_RBD = Dict{String, Set{String}}()
seq_AA_muts_not_excluded_fake_spike = Dict{String, Set{String}}()
seq_AA_muts_not_excluded_fake_spike_nonRBD = Dict{String, Set{String}}()
for (seq, mutset) in seq_AA_muts_no_dels_universal
    seq_AA_muts_not_excluded_fake[seq] = Set{String}()
    seq_AA_muts_not_excluded_fake_nonRBM[seq] = Set{String}()
    seq_AA_muts_not_excluded_fake_nonRBD[seq] = Set{String}()
    seq_AA_muts_not_excluded_fake_nonspike[seq] = Set{String}()
    seq_AA_muts_not_excluded_fake_RBM[seq] = Set{String}()
    seq_AA_muts_not_excluded_fake_RBD[seq] = Set{String}()
    seq_AA_muts_not_excluded_fake_spike[seq] = Set{String}()
    seq_AA_muts_not_excluded_fake_spike_nonRBD[seq] = Set{String}()
    for mut in mutset
        if !(mut in artifacts_ORF9bdoubles)
            push!(seq_AA_muts_not_excluded_fake[seq], mut)
            if !(mut in RBM_muts)
                push!(seq_AA_muts_not_excluded_fake_nonRBM[seq], mut)
            else
                push!(seq_AA_muts_not_excluded_fake_RBM[seq], mut)
            end
            if !(mut in RBD_muts)
                push!(seq_AA_muts_not_excluded_fake_nonRBD[seq], mut)
            else
                push!(seq_AA_muts_not_excluded_fake_RBD[seq], mut)
            end
            if !(mut in spike_muts)
                push!(seq_AA_muts_not_excluded_fake_nonspike[seq], mut)
            else
                push!(seq_AA_muts_not_excluded_fake_spike[seq], mut)
            end
            if mut in spike_muts && !(mut in RBD_muts)
                push!(seq_AA_muts_not_excluded_fake_spike_nonRBD[seq], mut)
            end
        end
    end
end
###########################################################################################################################################################################
AA_ct_total = 0
AA_nonspike_ct_total = 0
AA_spike_not_RBD_ct_total = 0
AA_RBD_ct_total = 0
AA_RBD_not_RBM_ct_total = 0
AA_RBM_ct_total = 0
#########################################################################################################
##            seqindex  Tuple: 1=AAct  2=nonspike_ct  3=spike_not_RBD_ct  4=RBD_not_RBM_ct  5=RBM_ct   
seq_AA_cts = Dict{Int, Tuple{Int, Int, Int, Int, Int}}()
            sizehint!(seq_AA_cts, all_unique_chr_seqs_ct)
########################################################################################################
for i in 1:all_unique_chr_seqs_ct
    seq = all_unique_chr_seqs[i]
    all_AA_seq_ct = 0
    nonspike_ct = 0 
    spike_not_RBD_ct = 0
    RBD_not_RBM_ct = 0
    RBM_ct = 0
    RBD_ct = 0
    for mut in seq_AA_muts_not_excluded_fake[seq]
        if !(mut in artifacts_ORF9bdoubles)
            all_AA_seq_ct += 1
#                mutpos = aa_pos_comprehensive_dict[mut]
            mutgene = aa_gene_comprehensive_dict[mut]
            if mutgene ≠ "S"
                nonspike_ct += 1
                continue
            end
            if mut in spike_not_RBD_muts
                spike_not_RBD_ct += 1
            end
            if mut in RBD_not_RBM_muts
                RBD_not_RBM_ct += 1
            end
            if mut in RBM_muts
                RBM_ct += 1
            end
            if mut in RBD_muts
                RBD_ct += 1
            end
        end
    end
    seq_AA_cts[i] = (all_AA_seq_ct, nonspike_ct, spike_not_RBD_ct, RBD_not_RBM_ct, RBM_ct)
    AA_ct_total += all_AA_seq_ct
    AA_nonspike_ct_total += nonspike_ct
    AA_spike_not_RBD_ct_total += spike_not_RBD_ct
    AA_RBD_not_RBM_ct_total += RBD_not_RBM_ct
    AA_RBM_ct_total += RBM_ct
    AA_RBD_ct_total += RBD_ct
end
#####################################################################################################################################
#####################################################################################################################################
AA_array = String[]
nonspike_array = String[]
spike_not_RBD_array = String[]
RBD_not_RBM_array = String[]
RBM_array = String[]
RBD_array = String[]
total_mcnd_AA_ct = 0
AA_muts_ct_no_dels_counter = Dict{Int, Dict{String,Int}}()
for i in 1:100
    AA_muts_ct_no_dels_counter[i] = Dict{String,Int}()
end
for (mut, ct) in AA_muts_ct_no_dels_universal
    if !(mut in artifacts_ORF9bdoubles)
        for i in 1:100
            AA_muts_ct_no_dels_counter[i][mut] = ct
        end
        total_mcnd_AA_ct += ct
        mutgene = aa_gene_comprehensive_dict[mut]
        mutpos = aa_pos_comprehensive_dict[mut]
        for i in 1:ct
            push!(AA_array, mut)
        end
###########
        if mutgene ≠ "S"
            for i in 1:ct
                push!(nonspike_array, mut)
            end
        end
###########
        if mutgene == "S"
#            for i in 1:ct
#                push!(spike_array, mut)
#            end
            if mutpos in spike_not_RBD_sites
                for i in 1:ct
                    push!(spike_not_RBD_array, mut)
                end
            end
            if mutpos in RBD_not_RBM_sites
                for i in 1:ct
                    push!(RBD_not_RBM_array, mut)
                end
            end
            if mutpos in RBM_sites
                for i in 1:ct
                    push!(RBM_array, mut)
                end
            end
        end
    end
end
AA_len = length(AA_array)
nonspike_len = length(nonspike_array)
spike_not_RBD_len = length(spike_not_RBD_array)
RBD_not_RBM_len = length(RBD_not_RBM_array)
RBM_len = length(RBM_array)
sum_composite = nonspike_len + spike_not_RBD_len + RBD_not_RBM_len + RBM_len
println("total_mcnd_AA_ct = $(total_mcnd_AA_ct) | AA_ct_total = $(AA_ct_total)")
println("AA_array Length = $(AA_len)")
println("nonspike_array Length = $(length(nonspike_array)) | AA_nonspike_ct_total = $(AA_nonspike_ct_total)")
println("spike_not_RBD_array Length = $(length(spike_not_RBD_array)) | AA_spike_not_RBD_ct_total = $(AA_spike_not_RBD_ct_total)")
println("RBD_not_RBM_array Length = $(length(RBD_not_RBM_array)) | AA_RBD_not_RBM_ct_total = $(AA_RBD_not_RBM_ct_total)")
println("RBM_array Length = $(length(RBM_array)) | AA_RBM_ct_total = $(AA_RBM_ct_total)")
println("nonspike + spike_not_RBD + RBD_not_RBM + RBM = $(sum_composite)")
######################################################################################################################################
######################################################################################################################################
######################################################################################################################################
######################################################################################################################################
####################################################   BEGIN RUNS   ##################################################################
####################################################   BEGIN RUNS   ##################################################################
######################################################################################################################################
######################################################################################################################################
######################################################################################################################################
######################################################################################################################################
missing_mut_ct = 0
not_enough_muts_ct = 0
zero_muts_seq_ct = 0
missing_mut_ct_dict = Dict{Int, Int}()
not_enough_muts_ct_dict = Dict{Int, Int}()
zero_muts_seq_ct_dict = Dict{Int, Int}()
for v in 1:number_of_runs
    missing_mut_ct_dict[v] = 0
    not_enough_muts_ct_dict[v] = 0
    zero_muts_seq_ct_dict[v] = 0
end
for e in 1:number_of_runs
##############################
##############################
    AA_array = shuffle!(AA_array)
    nonspike_array = shuffle!(nonspike_array)
    spike_not_RBD_array = shuffle!(spike_not_RBD_array)
    RBD_not_RBM_array = shuffle!(RBD_not_RBM_array)
    RBM_array = shuffle!(RBM_array)
##############################
    AA_array_2 = copy(AA_array)
    nonspike_array_2 = copy(nonspike_array)
    spike_not_RBD_array_2 = copy(spike_not_RBD_array)
    RBD_not_RBM_array_2 = copy(RBD_not_RBM_array)
    RBM_array_2 = copy(RBM_array)
    mut_region_array_array = [RBM_array_2, RBD_not_RBM_array_2, spike_not_RBD_array_2, nonspike_array_2]
###############################
    used_RBM_index_BitSet = BitSet()
    used_RBD_not_RBM_index_BitSet = BitSet()
    used_spike_not_RBD_index_BitSet = BitSet()
    used_nonspike_index_BitSet = BitSet()
    used_index_BitSet_array = [used_RBM_index_BitSet, used_RBD_not_RBM_index_BitSet, used_spike_not_RBD_index_BitSet, used_nonspike_index_BitSet]
###############################
    RBM_index_BitSet = BitSet([1:RBM_len...])
    RBD_not_RBM_index_BitSet = BitSet([1:RBD_not_RBM_len...])
    spike_not_RBD_index_BitSet = BitSet([1:spike_not_RBD_len...])
    nonspike_index_BitSet = BitSet([1:nonspike_len...])
    index_BitSet_array = [RBM_index_BitSet, RBD_not_RBM_index_BitSet, spike_not_RBD_index_BitSet, nonspike_index_BitSet]
###############################
    index_ct_array = [0, 0, 0, 0]
###############################
#    mut_type_cts_array = [RBM_len, RBD_not_RBM_len, spike_not_RBD_len, nonspike_len]
    start_rd = time()
    fake_seq_AA_muts = Dict{Int, Vector{String}}()
                sizehint!(fake_seq_AA_muts, all_unique_chr_seqs_ct)
    mp_meta_fake_chr_DQ_bad_cts[e] = 0
    for i in 1:all_unique_chr_seqs_ct
        fake_seq_AA_muts[i] = Vector{String}()
    end
    bad_ct = 0; good_ct = 0; finished_ct = 0; bad_seq = 0
    for z in 1:all_unique_chr_seqs_ct
        finished_ct += 1
###################
        seq_cts_tuple = seq_AA_cts[z]
        seq_AA_Ct = seq_cts_tuple[1]
        seq_nonspike_ct = seq_cts_tuple[2]
        seq_spike_not_RBD_ct = seq_cts_tuple[3]
        seq_RBD_not_RBM_ct = seq_cts_tuple[4]
        seq_RBM_ct = seq_cts_tuple[5]
        seq_cts_array = [seq_RBM_ct, seq_RBD_not_RBM_ct, seq_spike_not_RBD_ct, seq_nonspike_ct]
###################
        gene_pos_used = String[]
        for i in 1:length(seq_cts_array)
            if seq_cts_array[i] > 0
                count = 0
                index_attempts = 0
                for j in 1:seq_cts_array[i]
                    if count ≥ seq_cts_array[i]
                        break
                    end
                    indx = 1
                    if !isempty(index_BitSet_array[i])
                        indx = rand(index_BitSet_array[i])
                    else
                        not_enough_muts_ct += 1
                        not_enough_muts_ct_dict[e] += 1
                        break
                    end
                    mut = mut_region_array_array[i][indx]
                    gene_pos = aa_gene_and_pos_comprehensive_dict[mut]
                    if !(gene_pos in gene_pos_used)
                        push!(fake_seq_AA_muts[z], mut)
                        push!(gene_pos_used, gene_pos)
                        count += 1
                        push!(used_index_BitSet_array[i], indx)
                        index_BitSet_array[i] = setdiff(index_BitSet_array[i], indx)
                    else
                        index_attempts += 1
                        if length(index_BitSet_array[i]) ≤ index_attempts
                            missing_mut_ct += index_attempts
                            missing_mut_ct_dict[e] += index_attempts
                            break
                        end
                        for a in 1:20
                            if length(index_BitSet_array[i]) > index_attempts
                                indx2 = 1
                                for b in 1:20
                                    indx2 = rand(index_BitSet_array[i])
                                    if indx2 == indx
                                        continue
                                    else
                                        break
                                    end
                                end
                                mut2 = mut_region_array_array[i][indx2]
                                gene_pos2 = aa_gene_and_pos_comprehensive_dict[mut2]
                                if !(gene_pos2 in gene_pos_used)
                                    push!(fake_seq_AA_muts[z], mut)
                                    push!(gene_pos_used, gene_pos)
                                    count += 1
                                    push!(used_index_BitSet_array[i], indx2)
                                    index_BitSet_array[i] = setdiff(index_BitSet_array[i], indx2)
                                    break
                                else
                                    index_attempts += 1
                                    continue
                                end
                            end
                        end
                    end
                end
            end
        end
    end
#            if seq_cts_array[i] > 0
#                count = 0
#                h = 1
#                for j in 1:seq_cts_array[i]*2
#                    if length(array_array[i]) == 0
#                        break
#                    end
#                    if count ≥ seq_cts_array[i]
#                        break
#                    end
#                    mut = ""
#                    arr_len = length(array_array[i])
#                    if arr_len ≥ h
#                        mut = array_array[i][h]
#                    else
#                        println("####### run = $(e), sequence = $(z)")
#                        println("####### $(i) array in array_array, size = $(arr_len), h = $(h)")
#                        println("####### array_array[i][1] = $(array_array[i][1])")
#                        if arr_len ≥ 2
#                            println("array_array[i][2] = $(array_array[i][2])")
#                        end
#                        if arr_len ≥ 3
#                            println("array_array[i][3] = $(array_array[i][3])")
#                        end
#                        mp_meta_fake_chr_DQ_bad_cts[e] += 1
#                    end
#                    gene_plus_pos = ""
#                    if mut ≠ ""
#                        gene_plus_pos = aa_gene_and_pos_comprehensive_dict[mut]
#                    end
#                    if !(gene_plus_pos in gene_pos_used) && gene_plus_pos ≠ ""
#                        count += 1
#                        push!(fake_seq_AA_muts[z], mut)
#                        push!(gene_pos_used, gene_plus_pos)
##                        push!(usedkey_array[i], rand_index)
#                        deleteat!(array_array[i], h)
#                    else 
#                        h += 1
#                        continue 
#                    end
#                end
#            end
#        end
#    end
#    push!(mp_meta_fake_chr_dict_DQ[e], fake_seq_AA_muts)
    mp_meta_fake_chr_dict_DQ[e] = fake_seq_AA_muts
    total_RUN_time = round(digits=2, time() - start_rd)
    println("total_RUN_time = $(total_RUN_time)")
#############################################################################
#    for i in 1:1
#        println("print_count $(i) finished")
#        println("Sequence $i muts: ")
#        print("      ")
#        muts_arr = fake_seq_AA_muts[i]
#        for mut in muts_arr
#            print("$(mut), ")
#        end
#        print("\n"^1)
#    end
    print("\n"^1)
    fake_total_seq_AA = 0
    fake_nonspike_ct = 0
    fake_spike_not_RBD_ct = 0
    fake_RBD_not_RBM_ct = 0
    fake_RBM_ct = 0
    for (seq, AA_vec) in fake_seq_AA_muts
        fake_total_seq_AA += length(AA_vec)
        for mut in AA_vec
            mutpos = aa_pos_comprehensive_dict[mut]
            mutgene = aa_gene_comprehensive_dict[mut]
            if mutgene ≠ "S"
                fake_nonspike_ct += 1
                continue
            end
            if mutgene == "S"
                if mutpos in spike_not_RBD_sites
                    fake_spike_not_RBD_ct += 1
                end
                if mutpos in RBD_not_RBM_sites
                    fake_RBD_not_RBM_ct += 1
                end
                if mutpos in RBM_sites
                    fake_RBM_ct += 1
                end
            end
        end
    end
    for (seq, AAvec) in fake_seq_AA_muts
        AAtot = length(AAvec)
        if AAtot == 0
            zero_muts_seq_ct += 1
            zero_muts_seq_ct_dict[e] += 1
        end
    end
    avg_AAsub_seq_ct_fake = round(digits=2, fake_total_seq_AA/all_unique_chr_seqs_ct)
    println("avg_AAsub_seq_ct_fake = $(avg_AAsub_seq_ct_fake)")
    avg_fake_nonspike_ct = round(digits=2, fake_nonspike_ct/all_unique_chr_seqs_ct)
    println("avg_fake_nonspike_ct = $(avg_fake_nonspike_ct)")
    avg_fake_spike_not_RBD_ct = round(digits=2, fake_spike_not_RBD_ct/all_unique_chr_seqs_ct)
    println("avg_fake_spike_not_RBD_ct = $(avg_fake_spike_not_RBD_ct)")
    avg_fake_RBD_not_RBM_ct = round(digits=2, fake_RBD_not_RBM_ct/all_unique_chr_seqs_ct)
    println("avg_fake_RBD_not_RBM_ct = $(avg_fake_RBD_not_RBM_ct)")
    avg_fake_RBM_ct = round(digits=2, fake_RBM_ct/all_unique_chr_seqs_ct)
    println("avg_fake_RBM_ct = $(avg_fake_RBM_ct)")
#    print("\n"^1)
#####################################################################################################################################
    avg_AAsub_seq_ct = round(digits=2, AA_ct_total/all_unique_chr_seqs_ct)
    avg_nonspike_AAsub_seq_ct = round(digits=2, AA_nonspike_ct_total/all_unique_chr_seqs_ct)
    avg_spike_not_RBD_AAsub_seq_ct = round(digits=2, AA_spike_not_RBD_ct_total/all_unique_chr_seqs_ct)
    avg_RBD_not_RBM_AAsub_seq_ct = round(digits=2, AA_RBD_not_RBM_ct_total/all_unique_chr_seqs_ct)
    avg_RBM_AAsub_seq_ct = round(digits=2, AA_RBM_ct_total/all_unique_chr_seqs_ct)
    println("Avg # of AA Subs/Seq =       $(avg_AAsub_seq_ct)")
    println("Avg # of Nonspike Subs/Seq = $(avg_nonspike_AAsub_seq_ct)")
    println("Avg # of Spike-Not-RBD/Seq = $(avg_spike_not_RBD_AAsub_seq_ct)")
    println("Avg # of RBD_not_RBM/Seq =   $(avg_RBD_not_RBM_AAsub_seq_ct)")
    println("Avg # of RBM AA Subs/Seq =   $(avg_RBM_AAsub_seq_ct)")
###############################################################################
    avg_AAsub_seq_ct_fake = round(digits=2, AA_ct_total/all_unique_chr_seqs_ct)
    avg_nonspike_AAsub_seq_ct = round(digits=2, AA_nonspike_ct_total/all_unique_chr_seqs_ct)
    avg_spike_not_RBD_AAsub_seq_ct = round(digits=2, AA_spike_not_RBD_ct_total/all_unique_chr_seqs_ct)
    avg_RBD_not_RBM_AAsub_seq_ct = round(digits=2, AA_RBD_not_RBM_ct_total/all_unique_chr_seqs_ct)
    avg_RBM_AAsub_seq_ct = round(digits=2, AA_RBM_ct_total/all_unique_chr_seqs_ct)
    println("Total Number of Sequences in Round $(e) = $(length(keys(fake_seq_AA_muts)))")
    println("Total not_enough_muts_ct in Round $(e) = $(not_enough_muts_ct_dict[e])")
    println("Total missing_mut_ct in Round $(e) = $(missing_mut_ct_dict[e])")
    println("Total zero_muts_seq_ct in Round $(e) = $(zero_muts_seq_ct_dict[e])")
end
print("\n"^2)
println("Final missing_mut_ct = $(missing_mut_ct)")
println("Final not_enough_muts_ct = $(not_enough_muts_ct)")
println("Final zero_muts_seq_ct = $(zero_muts_seq_ct)")
print("\n"^2)
#####################################################################################################################################
folder_name = ""
save_name = ""
if sub_0__posonly_1 == 0
    if normal_0__spikeonly_1 == 0
        folder_name = "mp_fake__EPCI_$(EPCI_qc_str)__HQCS_$(HQCS_qc_string)"
        save_name = "mp_meta_fake_chr_dict_DQ__EPCI_$(EPCI_qc_str)__HQCS_$(HQCS_qc_string)"
    elseif normal_0__spikeonly_1 == 1
        folder_name = "mp_fake_spike_only__EPCI_$(EPCI_qc_str)__HQCS_$(HQCS_qc_string)"
        save_name = "mp_meta_fake_chr_dict_DQ__EPCI_$(EPCI_qc_str)__HQCS_$(HQCS_qc_string)"
    end
elseif sub_0__posonly_1 == 1
    if normal_0__spikeonly_1 == 0
        folder_name = "mp_fake_pos_only__EPCI_$(EPCI_qc_str)__HQCS_$(HQCS_qc_string)"
        save_name = "mp_meta_fake_pos_only_chr_dict_DQ__EPCI_$(EPCI_qc_str)__HQCS_$(HQCS_qc_string)"
    elseif normal_0__spikeonly_1 == 1
        folder_name = "mp_fake_pos_only_spike_only__EPCI_$(EPCI_qc_str)__HQCS_$(HQCS_qc_string)"
        save_name = "mp_meta_fake_pos_only_spike_only_chr_dict_DQ__EPCI_$(EPCI_qc_str)__HQCS_$(HQCS_qc_string)"
    end
end
save("$(folder_name)/$(save_name)_$(date).jld2", "mp_meta_fake_chr_dict_DQ", mp_meta_fake_chr_dict_DQ)
save("$(folder_name)/$(save_name)_bad_cts_$(date).jld2", "mp_meta_fake_chr_DQ_bad_cts", mp_meta_fake_chr_DQ_bad_cts)
#####################################################################################################################################
runtime = time() - start
runtime_rd = round(digits=2, runtime)
runtime1, runtime2 = seconds_to_hrs_min_sec(runtime)
println("Runtime v0 = $(runtime) seconds")
println("Runtime v1 = $(runtime1)")
println("Runtime v2 = $(runtime2)")
##############################################################################
##############################################################################
    #    println()
##############################################################################
#    AA_ct_dict_sort_by_mut_ct = sort(collect(AA_ct_dict), by = x -> x[1])
#    AA_ct_dict_sort_by_seq_ct = sort(collect(AA_ct_dict), by = x -> x[2], rev=true)
#############
#    AA_nonspike_ct_dict_sort_by_mut_ct = sort(collect(AA_nonspike_ct_dict), by = x -> x[1])
#    AA_nonspike_ct_dict_sort_by_seq_ct = sort(collect(AA_nonspike_ct_dict), by = x -> x[2], rev=true)
#############
#    AA_nonRBD_ct_dict_sort_by_mut_ct = sort(collect(AA_nonRBD_ct_dict), by = x -> x[1])
#    AA_nonRBM_ct_dict_sort_by_seq_ct = sort(collect(AA_nonRBM_ct_dict), by = x -> x[2], rev=true)
#############
#    AA_nonRBD_ct_dict_sort_by_mut_ct = sort(collect(AA_nonRBD_ct_dict), by = x -> x[1])
#    AA_nonRBM_ct_dict_sort_by_seq_ct = sort(collect(AA_nonRBM_ct_dict), by = x -> x[2], rev=true)
##############################################################################
#println("Keys in mp_meta_fake_chr_dict_DQ = $(length(keys(mp_meta_fake_chr_dict_DQ)))")
#println("Keys in mp_meta_fake_chr_DQ_bad_cts = $(length(keys(mp_meta_fake_chr_DQ_bad_cts)))")
#println("Keys in mp_meta_fake_chr_dict_DQ 1 = $(length(keys(mp_meta_fake_chr_dict_DQ[1])))")
#println("Keys in mp_meta_fake_chr_dict_DQ 2 = $(length(keys(mp_meta_fake_chr_dict_DQ[2])))")
#println("Keys in mp_meta_fake_chr_dict_DQ 3 = $(length(keys(mp_meta_fake_chr_dict_DQ[3])))")
#println("Keys in mp_meta_fake_chr_DQ_bad_cts 1 = $(length(keys(mp_meta_fake_chr_DQ_bad_cts[1])))")
#println("Keys in mp_meta_fake_chr_DQ_bad_cts 2 = $(length(keys(mp_meta_fake_chr_DQ_bad_cts[2])))")
#println("Keys in mp_meta_fake_chr_DQ_bad_cts 3 = $(length(keys(mp_meta_fake_chr_DQ_bad_cts[3])))")
#####################################################################################################################################
#####################################################################################################################################
#####################################################################################################################################

2026_02_23__656AM
total_mcnd_AA_ct = 49885 | AA_ct_total = 49229
AA_array Length = 49885
nonspike_array Length = 33636 | AA_nonspike_ct_total = 33390
spike_not_RBD_array Length = 10192 | AA_spike_not_RBD_ct_total = 9816
RBD_not_RBM_array Length = 2374 | AA_RBD_not_RBM_ct_total = 2367
RBM_array Length = 3683 | AA_RBM_ct_total = 3656
nonspike + spike_not_RBD + RBD_not_RBM + RBM = 49885
total_RUN_time = 1.25

avg_AAsub_seq_ct_fake = 19.94
avg_fake_nonspike_ct = 13.52
avg_fake_spike_not_RBD_ct = 3.98
avg_fake_RBD_not_RBM_ct = 0.96
avg_fake_RBM_ct = 1.48
Avg # of AA Subs/Seq =       19.94
Avg # of Nonspike Subs/Seq = 13.52
Avg # of Spike-Not-RBD/Seq = 3.98
Avg # of RBD_not_RBM/Seq =   0.96
Avg # of RBM AA Subs/Seq =   1.48
Total Number of Sequences in Round 1 = 2469
Total not_enough_muts_ct in Round 1 = 0
Total missing_mut_ct in Round 1 = 0
Total zero_muts_seq_ct in Round 1 = 0
total_RUN_time = 0.09

avg_AAsub_seq_ct_fake = 19.94
avg_fake_nonspike_ct = 13.52
avg_fake_spike_not_RBD_ct = 3.98

In [107]:
### MAIN FUNCTION Part 1 ######################################
####################################################################################################################################
################################### Abbreviations used in this function (including in comments) ####################################
#                       mut = mutation, which in this function means amino acid substitution (not insertions/deletions)
#                       EPCI = manually curated list of confirmed & likely chronic-infection sequences
#                       chronic = chronic-infection sequence or lineage, i.e. EPCI sequence
#                       mp = mutation pattern
#                       AA = amino acid
#                       del = deletion
#                       nd = no deletions (also no_dels)
#                       po = position only (also pos_only); indicates all AA substitutions at a given AA position
#                       ct = count
#                       seq = sequence
#                       grp = group
#                       RBD = spike receptor-binding domain (S:335-519)
#                       RBM = spike receptor-binding motif (S:438-506)
#                       log10pvFISH = negative log10 p-value for Fisher's exact test
#                       chi2 = chi-squared test (also Chi2 or Chi)
#                       dict = Dictionary (type of Julia coding object)
#                       qual = qualifying (i.e. meeting the necessary requirements)
#                       coAAmut = co-occuring amino acid substitutions (i.e. in the same sequence)
#                       priv = private (as in private mutations, i.e. those not inherited from documented sequences, according to the best phylogenetic reconstructions)
#                       min = minimum (as in min_mut_ct, i.e. minimum mutation count requirement)
#                       max = maximum
#                       avg = average, i.e. mean (for all you stats snobs, who had to invent a new word for average that already had a dozen other meanings)
#                       tot = total
#                       chr = chronic (see above)
#                       vec = Vector type (array or tuple)
#                       str = string type
#                       df = DataFrame type
#                       fake = from the artificially created list of EPCI-like sequences (called APCI in methods section)
#                       BAL = bronchoalveoloar lavage, as in mutations associated with bronchoalveoloar lavage-sample sequences
#################################################################################################################################################
date_now = Dates.format(now(), "yyyy_mm_dd__IMMp"); println(date_now)
date_hour = Dates.format(now(), "yyyy_mm_dd_Ip")
date = Dates.format(today(), "yyyy_mm_dd")
#### When testing the original mut patterns, zero RBD or RBM mutations appeared in any of the final or intermediary results. 
####    Their exclusion, therefore, has no effect. 
#### However, when testing all other mutations that occur in the EPCI dataset more than five times (for control purposes), 
####    RBM mutation groups frequently occur in enormous "mutation patterns", which is simply a result of the extremely dense
####    mutation rate in the RBM in a large number of EPCI sequences. These "patterns" are therefore not very meaningful and only
####    serve to identify the mutations present in sequences with a high density of mutations in the RBM.
#### RBM mutations are therefore entirely prohibited as seed mutations AND excluded from inclusion in any mut pattern. 
###########################################################################################################################################################################
###########################################################################################################################################################################
####### Below dicts were an attempt to use a more sophisticated & less blunt exclusion of N/9b double muts. Not currently used.
                        # ORF9b_N_doubles = Dict{String, Vector{String}}("ORF9b:D2E"=>["N:P6T"], "ORF9b:P10S"=>["N:P13L"], "ORF9b:S10P"=>["N:L13P"], "ORF9b:T60A"=>["N:D63G"], "ORF9b:Q77E"=>["N:P80R"], "ORF9b:E86D"=>["N:A90S", "N:A90P"], "ORF9b:V92M"=>["N:R95H"], "ORF9b:V93L"=>["N:G96V", "N:G96A"],            "ORF9b:V93M"=>["N:G96D"], "ORF9b:V94M"=>["N:G97D"],      "ORF9b:V94L"=>["N:G97V", "N:G97A"],    "ORF9b:T95A"=>["N:D98G"], "ORF9b:K97E"=>["N:K100R"], "ORF9b:K97N"=>["N:M101L"])  # "ORF9b:"=>["N:"], "ORF9b:"=>["N:"]
                        # N_ORF9b_doubles = Dict{String, Vector{String}}("N:P6T"=>["ORF9b:D2E"], "N:P13L"=>["ORF9b:P10S"], "N:L13P"=>["ORF9b:S10P"], "N:D63G"=>["ORF9b:T60A"], "N:P80R"=>["ORF9b:Q77E"], "N:A90S"=>["ORF9b:E86D"], "N:R95H"=>["ORF9b:V92M"], "N:G96V"=>["ORF9b:V93L"], "N:G96A"=>["ORF9b:V93L"], "N:G96D"=>["ORF9b:V93M"], "N:G97D"=>["ORF9b:V94M"], "N:G97V"=>["ORF9b:V94L"], "N:G97A"=>["ORF9b:V94L"], "N:D98G"=>["ORF9b:T95A"], "N:K100R"=>["ORF9b:K97E"], "N:M101L"=>["ORF9b:K97N"])
######################################################################################################################################
#open("$(mp_folder_universal)/mp_mp_round_by_round_debug/round_by_round_debug_seed$(seedmut)_$(mut_pattern_name)__EPCI_$(EPCI_qc_str)__HQCS_$(HQCS_qc_str)__plusminus$(plus_minus)_minGrpFish$(min_grp_fish)_minFish$(min_log_pv_fish)_seqfac$(seqfac)_$(date).txt", "w") do g; println(g); end
######################################################################################################################################
####       This fx begins with a minimal number of mutations that have a clear & obvious connection (e.g. ORF1a:T1322I & ORF1a:T1638I)
####   A minimum number of mutations is required for a sequence to qualify. The exact minimum depends on the number of 
####   genomic regions contained in the searched mutations (i.e. the parameter sel_muts_pt1). If there are 1-2 regions represented,
####   a sequence must have ≥1 mutation in the 2 regions. For 3-4 regions, a sequence must have ≥2 muts. If ≥5 regions are represented,
####   the minimum is 3 muts. 
####       Once the qualifying sequences have been determined, all private mutations in those sequences are tallied. Each mutation 
####   in this dataset is then assessed to find whether it is statistically overrepresented in the sequence set. A Chi^2 value is
####   calculated based on the number of times the mutation appears in the "qualifying sequences" dataset, the number of times
####   the mutation appears in the entire chronic-sequences list (EPCI), and the number of qualifying sequences. 
####       If a mutation is in the same region as one of the existing sel_muts_pt1 regions (defined as being within 4 AA of one of the
####   muts in a sel_muts_pt1 region), it must have a Chi^2 value ≥ 12 to qualify for inclusion in that region. If it is not near any 
####   of the sel_muts_pt1 regions, then it must meet the stricter criteria of having a -log10 p-value of for the Fisher's exact test
####   in order to be included as a new mutation group in sel_muts_pt1. 
####       When the new mutations (sel_muts2_pt1) have been determined, they become the mutations to begin the next round. The entire 
####   process is repeated until there is no change from the previous round, at which point this part of the 2-part function stops and
####   its data fed into part 2, which is described below. 
######################################################################################################################################
##### NOTE: Mutations in mp_masked_muts are excluded from inclusion in the mut pattern and from the sequence-qualifying process; 
##        i.e. if E:T9I is in mp_masked_muts, E:T9I is BLOCKED from inclusion in the mut pattern.
##    The justification for mp_masked_muts (which is rarely used) is that sometimes one or more extremely common chronic mutations
##        might have a strong statistical association with a mutation pattern, even though its fold increase might be relatively small
##        compared to other mutations——maybe 2-fold higher compared to >10-fold for others. But because it is such a common mutation,
##        the statistical association appears strong. The problem is that the sequence-selection process becomes dominated by these
##        common chronic mutations. For example, ORF1a:K1795Q and ORF7a:T39I are both very common in chronics (288 and 200 sequences, 
##        respectively) and highly correlated with each other. They also both have somewhat less intense correlations with the
##        3-7a pattern (which involves 5 other regions). ORF1a:K1795Q and ORF7a:T39I, because of how common they are and how often
##        they appear together, dominate the sequence-selection process, drowning out the other regions/mutations, which are the most
##        distinctive parts of the 3-7a mutational pattern. mp_masked_muts allows us to test the 3-7a mutational pattern both with 
##        and without ORF1a:K1795Q and ORF7a:T39I.
######################################################################################################################################
#### ind_or_grp is either "ind" or "grp". The former ("ind") means all muts counted individually; 
####     The latter ("grp") means each group/region is only counted once, no matter how many mutations are in that group.
####     Right now, "grp" is the ony option that is ever used.
###########################################################################################################################################################################
###########################################################################################################################################################################
#####################################################   Part 2 Description   ##############################################################################################
###########################################################################################################################################################################
###########################################################################################################################################################################
######   Part 2 looks at each mut region individually by first removing that region from the list & then reassessing the statistical
######   significance of each mut in that region (for how it correlates w/sequences w/muts in the remaining region(s)). The same
######   Chi^2 value of ≥ 12 is used to determine if a mutation qualifies, though this value can be adjusted up or down if one wishes.
######   It does this for each region in turn. A round is finished once all regions have been assessed this way. Mutations can
######   be added or removed during this process.
######       Once the final region is finished, it checks to see if the round just finished caused any changes in the full mutation 
######   list relative to the beginning of the round.  If there were changes, it goes through another full round. 
######   If there were no changes in the round, it finishes and prints a bunch of shit to record the final results.
######       Part 1 took care of locating the regions to be analyzed, so no new regions are added in Part 2, though individual 
######   mutations are lost or gained.
######   Originally, Part 1 and Part 2 were separate functions, but due to the inexplicable inability of Part 1 to correctly
######       deliver its results to Part 2, the two functions were combined into a single function that uses global variables
######       instead of parameters. 
######################################################################################################################################
##  • mp_masked_muts are excluded mutations. They are not eligible for inclusion in the mutational pattern. (See part 1 for justification
##        and explanation for mp_masked_muts.)
##  • ind_or_grp is either "ind" or "grp". The former ("ind") means all muts counted individually; *only* grp is currently used
##    The latter ("grp") means each group/region is only counted once, not matter how many mutations are in that group
##  • all_muts_round_dict_pt2 (line below) keeps track of all the muts after each full round. Used to determine when to stop iterating.
##  • pt2_iteration_ct keeps track of the total number of iterations. Each round has as many iterations as the mut pattern has regions.
##    E.g., if there are 5 regions in the mut pattern, there will be 5 iterations in each round, one for each region
##  • pt2_region_analyzed tells which region is being analyzed; e.g. if there are 4 regions
##    pt2_region_analyzed should *always* be set to *one* when starting the function.
###########################################################################################################################################################################
       fake_dict_num_ct = 1
######### pt1_round_ct = round # we are on in part 1.
       pt1_round_ct = 0
       pt1_finished = 0
######################################################################################################################################
####### all_muts_round_dict_pt2 records all qualifying muts at the end of each round (i.e. after all regions have been independently checked)
#######    The stats for each mutation MUST ONLY BE RECORDED IN THE ITERATION DURING WHICH ITS GROUP IS EXCLUDED
       all_muts_round_dict_pt2 = Dict{Int, Set{String}}()   # all_muts_round_dict_pt2 stores the qualifying mutations at the end of each round
#####################################################################################################################################
####### all_muts_round_mutgroup_dict_pt2 records all qualifying *groups* of mutations at the end of each round
####### As with the original sel_muts_pt2, each group is a string of comma-separated mutations
       all_muts_round_mutgroup_dict_pt2 = Dict{Int, Vector{String}}()   
#####################################################################################################################################
#### df_props_dict_pt2 is needed to record all the specific stats for each mutation in each round (for printing results & csv/tsv purposes)
#                             Round RegionAnalyzed  Mutation      EPCI_pct, MP_pct, totChrMutCt, MP_Tot_Mut_ct, Adjusted_MP_seq_ct, Chi^2,  pvFish, log10pvFISH, non_MP_pct
global df_props_dict_pt2 = Dict{Int, Dict{Int, Dict{String, Tuple{Float64,Float64,Int,Int,Int,Float64,Float64,Float64,Union{Float64, String},Float64}}}}()
global pt2_round_ct = 1
global pt2_iteration_ct = 0
global pt2_region_analyzed = 1
global sel_muts_pt2 = Vector{String}()
global sel_muts_pt2_sort = Vector{String}()
global sel_muts_pt2_ind = Vector{Vector{String}}()
global sel_muts_pt2_ind_dict = Dict{Int, Vector{String}}()
global sel_muts_pt2_ind_sort = Vector{Vector{String}}()
global region_ct2
global sel_muts_pt1_round_dict = Dict{Int, Vector{String}}()
global sel_muts_pt1_global = Vector{String}()
global sel_muts_pt1_sort = Vector{String}()
global sel_muts_pt1_ind = Vector{Vector{String}}()
global pt1_seed_mut = ""
global coAAmut_ct_pt1 = Dict{String,Int}()
global coAAmut_ct_pt2 = Dict{String,Int}()
global coAAmut_ct_pt1_unknown = Dict{String,Int}()
global coAAmut_ct_pt2_unknown = Dict{String,Int}()
############################################################################################################################################################################
#                         Mutation        EPCI_pct, MP_pct, totChrMutCt, MP_Tot_Mut_ct, Adjusted_MP_seq_ct, Chi^2,  pvFish, log10pvFISH, fold_incr, non_MP_pct
global prop_dict_pt1 = Dict{String, Tuple{Float64,Float64,Int,Int,Int,Float64,Float64,Float64,Union{Float64, String},Float64} }()
global prop_dict_pt2 = Dict{String, Tuple{Float64,Float64,Int,Int,Int,Float64,Float64,Float64,Union{Float64, String},Float64} }()
############################################################################################################################################################################
####  The mp_pango_50_dates_pt2 record the 50% date index for each Pango lineage that is included in the mut pattern.
####     The 50% date index is the date on which 50% of a given Pango lineages total sequences had been recorded.
####     E.g., if there are 5000 seqs for variant RG.1, then the day on which the 2500th sequence of RG.1 was recorded
####     would be its 50% date index.
############################################################################################################################################################################
global cumulative_seq_round_set_pt2 = Set{String}() ### Set of all seqs that qualify during any iteration of a round (erased upon the start of each round so that only the final round's results are left at end)  
global cumulative_seq_round_set_pt2_sort = Vector{String}()
global cumulative_seq_round_set_pt2_sort_join = ""
############################################################################################################################################################################
global pango_vec_pt2 = Vector{String}()  ### Array of Pango lineage for all seqs that qualify during last round of pt2
global clade_vec_pt2 = Vector{String}()  ### Array of clades for all seqs that qualify during last round of pt2
global cladepango_vec_pt2 = Vector{String}()  ### Array of corresponding Pango lineage for all clades of all seqs that qualify during last round of pt2 (e.g. "21L" = "BA.2")
global collection_date_index_vec_pt2 = Vector{Int}()  ### Array of all collection dates for sequences that qualify during last round of pt2
global mp_pango_50_dates_pt2 = Vector{Int}()
global mp_seq_dict = Dict{String, Set{String}}()
global df_mp = DataFrame(SeedMut = String[], MP_Tot_Mut_ct = Int[], MP_Tot_Reg_ct = Int[], Reg_Tot_Mut_ct = Int[], MP_Region_Number = Int[], Mutation = String[], MP_Mut_ct = Float64[], EPCI_Mut_ct = Int[], Adjusted_MP_seq_ct = Int[], EPCI_pct = Float64[], 
    MP_pct = Float64[], Fishers_Exact_Test_pval = Float64[], log10pvFISH = Float64[], Fold_Incr = Union{Float64, String}[], Chi2 = Float64[], EPCI_HQCS_Ratio = Float64[], avg_NonRBD_AAct = Float64[], avg_nonRBD_rel = Float64[], 
    PangoDateIndex50_Avg = Int[], PangoDateStr50_Avg = String[], CollDateIndex_Avg = Int[], CollDateString_Avg = String[], Est_Infx_Length = Int[], 
    PangoDateIndex50_min = Int[], PangoDateIndex50_p10 = Int[], PangoDateIndex50_Q1 = Int[], PangoDateIndex50_Q2 = Int[], PangoDateIndex50_Q3 = Int[], PangoDateIndex50_p90 = Int[], PangoDateIndex50_max = Int[], 
    PangoDateStr50_min = String[], PangoDateStr50_p10 = String[], PangoDateStr50_Q1 = String[], PangoDateStr50_Q2 = String[], PangoDateStr50_Q3 = String[], PangoDateStr50_p90 = String[], PangoDateStr50_max = String[], 
    CollDateIndex_min = Int[], CollDateIndex_p10 = Int[], CollDateIndex_Q1 = Int[], CollDateIndex_Q2 = Int[], CollDateIndex_Q3 = Int[], CollDateIndex_p90 = Int[], CollDateIndex_max = Int[], 
    CollDateString_min = String[], CollDateString_p10 = String[], CollDateString_Q1 = String[], CollDateString_Q2 = String[], CollDateString_Q3 = String[], CollDateString_p90 = String[], CollDateString_max = String[])
global df_mp_unique = DataFrame(SeedMut = String[], MP_Tot_Mut_ct = Int[], MP_Tot_Reg_ct = Int[], Reg_Tot_Mut_ct = Int[], MP_Region_Number = Int[], Mutation = String[], MP_Mut_ct = Float64[], EPCI_Mut_ct = Int[], Adjusted_MP_seq_ct = Int[], EPCI_pct = Float64[], 
    MP_pct = Float64[], Fishers_Exact_Test_pval = Float64[], log10pvFISH = Float64[], Fold_Incr = Union{Float64, String}[], Chi2 = Float64[], EPCI_HQCS_Ratio = Float64[], avg_NonRBD_AAct = Float64[], avg_nonRBD_rel = Float64[], 
    PangoDateIndex50_Avg = Int[], PangoDateStr50_Avg = String[], CollDateIndex_Avg = Int[], CollDateString_Avg = String[], Est_Infx_Length = Int[], 
    PangoDateIndex50_min = Int[], PangoDateIndex50_p10 = Int[], PangoDateIndex50_Q1 = Int[], PangoDateIndex50_Q2 = Int[], PangoDateIndex50_Q3 = Int[], PangoDateIndex50_p90 = Int[], PangoDateIndex50_max = Int[], 
    PangoDateStr50_min = String[], PangoDateStr50_p10 = String[], PangoDateStr50_Q1 = String[], PangoDateStr50_Q2 = String[], PangoDateStr50_Q3 = String[], PangoDateStr50_p90 = String[], PangoDateStr50_max = String[], 
    CollDateIndex_min = Int[], CollDateIndex_p10 = Int[], CollDateIndex_Q1 = Int[], CollDateIndex_Q2 = Int[], CollDateIndex_Q3 = Int[], CollDateIndex_p90 = Int[], CollDateIndex_max = Int[], 
    CollDateString_min = String[], CollDateString_p10 = String[], CollDateString_Q1 = String[], CollDateString_Q2 = String[], CollDateString_Q3 = String[], CollDateString_p90 = String[], CollDateString_max = String[])
global df_mp_META
global df_mp_META_sort
global df_mp_META_unique
global df_mp_META_sort_unique
global df_mp_META_set_set = Set{Set{String}}()                                                           
global df_mp_stats = DataFrame(Mut_Pattern_or_Seed_Mut = String[], Total_Regions = Int[], Total_Mutations = Int[], Total_Sequences = Int[], Pango50_DateIndex_Avg = Int[], Pango50_Date_Avg = String[], Collection_DateIndex_Avg = Int[], Collection_Date_Avg = String[], Est_Infx_Length = Int[], Pango50_DateIndex_Q2 = Int[], Pango50_Date_Q2 = String[], Collection_DateIndex_Q2 = Int[], Collection_Date_Q2 = String[], EPCI_HQCS_Ratio = Float64[])
#Tuple{String,Int64,Int64,Int64,String,Float64,Int64,Int64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Int64,String,Int64,String,Int64,Int64,Int64,Int64,Int64,Int64,Int64,Int64,String,String,String,String,String,String,String,Int64,Int64,Int64,Int64,Int64,Int64,Int64,String,String,String,String,String,String,String,String}
global df_mp_META_groups_dict = Dict{String,Vector{Tuple{String,Int,Int,Int,Int,String,Int,Int,Int,Float64,Float64,Float64,Float64,Float64,Union{Float64, String},Float64,Float64,Float64,Float64,Int,String,Int,String,Int,Int,Int,Int,Int,Int,Int,Int,String,String,String,String,String,String,String,Int,Int,Int,Int,Int,Int,Int,String,String,String,String,String,String,String,String}}}()
#global df_mp_META_groups_dict = Dict{String,Vector{Tuple{String,Int,Int,Int,String,Float64,Int,Int,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Int,String,Int,String,Int,Int,Int,Int,Int,Int,Int,Int,String,String,String,String,String,String,String,Int,Int,Int,Int,Int,Int,Int,String,String,String,String,String,String,String,String}}}()
#global df_mp_META_groups_dict_unique = Dict{String,Vector{Tuple{String,Int,Int,Int,String,Float64, Int,Int,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Int,String,Int,String,Int,Int,Int,Int,Int,Int,Int,Int,String,String,String,String,String,String,String,Int,Int,Int,Int,Int,Int,Int,String,String,String,String,String,String,String,String}}}()
global df_mp_META_groups_dict_unique = Dict{String,Vector{Tuple{String,Int,Int,Int,Int,String,Int,Int,Int,Float64,Float64,Float64,Float64,Float64,Union{Float64, String},Float64,Float64,Float64,Float64,Int,String,Int,String,Int,Int,Int,Int,Int,Int,Int,Int,String,String,String,String,String,String,String,Int,Int,Int,Int,Int,Int,Int,String,String,String,String,String,String,String,String}}}()
############################################################################################################################################################################
global total_EPCI_seq_ct = length(all_unique_chr_seqs_set)
global mp_index
######################################################################################################################################
seq_AA_muts_not_excluded = Dict{String, Set{String}}()
for (seq, mutset) in seq_AA_muts_no_dels_universal
    seq_AA_muts_not_excluded[seq] = Set{String}()
    for mut in mutset
        if !(mut in all_excluded_muts)
            push!(seq_AA_muts_not_excluded[seq], mut)
        end
    end
end
###########################################################################################################################################################################
min_log_pv_fish = 5.0;  min_grp_fish = 2.0; plus_minus = 5
global mp_folder_universal = ""
if sub_0__posonly_1 == 0
    if normal_0__spikeonly_1 == 0
        mp_folder_universal = "mp_subs_$(date_hour)__EPCI_$(EPCI_qc_str)__HQCS_$(HQCS_qc_str)__minFsh$(min_log_pv_fish)_grpFsh_$(min_grp_fish)"
    elseif normal_0__spikeonly_1 == 1
        mp_folder_universal = "mp_subs_spike_only_$(date_hour)__EPCI_$(EPCI_qc_str)__HQCS_$(HQCS_qc_str)__minFsh$(min_log_pv_fish)_grpFsh_$(min_grp_fish)"
    end
elseif sub_0__posonly_1 == 1
    if normal_0__spikeonly_1 == 0
        mp_folder_universal = "mp_pos_only_$(date_hour)__EPCI_$(EPCI_qc_str)__HQCS_$(HQCS_qc_str)__minFsh$(min_log_pv_fish)_grpFsh_$(min_grp_fish)"
    elseif normal_0__spikeonly_1 == 1
        mp_folder_universal = "mp_pos_only_spike_only_$(date_hour)__EPCI_$(EPCI_qc_str)__HQCS_$(HQCS_qc_str)__minFsh$(min_log_pv_fish)_grpFsh_$(min_grp_fish)"
    end
end
pango_counts_folder = "$(mp_folder_universal)/pango_counts"
tsv_folder = "$(mp_folder_universal)/TSV"
global pt1_seed_mut_set = Set{String}()
###########################################################################################################################################################################
###########################################################################################################################################################################
###########################################################################################################################################################################
function AA_2plus__2025_11_19__UNIVERSAL(min_match_ct::Int, pt1_ct_mltplr::Float64, pt2_ct_mltplr::Float64, purespikebanned_0__purespikeallowed_1::Int, nonmulti0__multi1::Int, notrandom0__random1::Int, nonspike_seqfactor__0off__1RBM__2RBD__3SpikeNonRBD::Int, plus_minus::Int, mut_pattern_name::String, ind_or_grp::String, min_grp_fish::Float64, min_log_pv_fish::Float64, mp_masked_muts::Set{String}, sel_muts_pt1::Vector{String}, seq_factor_OFF_HALF_ON__0_1_2::Int) # , random_or_notrandom::String)
    mkpath(mp_folder_universal); mkpath(tsv_folder); mkpath(pango_counts_folder)
    start_time = time()
    global sel_muts_pt1_round_dict # = Dict{Int, Vector{String}}()
    global sel_muts_pt1_global # = Vector{String}()
    global sel_muts_pt1_sort # = Vector{String}()
    global sel_muts_pt1_ind # = Vector{Vector{String}}()
    global sel_muts_pt2 # = Vector{String}()
    global sel_muts_pt2_sort # = Vector{String}()
    global sel_muts_pt2_ind # = Vector{Vector{String}}()
    global sel_muts_pt2_ind_dict # = Dict{Int, Vector{String}}()
    global sel_muts_pt2_ind_sort # = Vector{Vector{String}}()
    global pt1_round_ct # = 0
    global pt1_finished # = 0
    global pt2_round_ct # = 0
    global pt2_iteration_ct # = 0
    global pt2_region_analyzed # = 1
    global all_muts_round_dict_pt2 # = Dict{Int, Set{String}}()
    global all_muts_round_mutgroup_dict_pt2 # = Dict{Int, Vector{String}}()
    global df_props_dict_pt2 # = Dict{Int, Dict{Int, Dict{String,Tuple{Float64,Float64,Int,Int,Int,Float64,Float64,Float64,Union{Float64, String},Float64}}}}()
    global coAAmut_ct_pt1 # = Dict{String,Int}()
    global coAAmut_ct_pt2 # = Dict{String,Int}()
    global coAAmut_ct_pt1_unknown # = Dict{String,Int}()
    global coAAmut_ct_pt2_unknown # = Dict{String,Int}()
    global prop_dict_pt1 # = Dict{String,Tuple{Float64,Float64,Int,Int,Int,Float64,Float64,Float64,Float64}}()
    global prop_dict_pt2 # = Dict{String,Tuple{Float64,Float64,Int,Int,Int,Float64,Float64,Float64,Float64}}()
    global cumulative_seq_round_set_pt2 # = Set{String}()
    global cumulative_seq_round_set_pt2_sort # = Vector{String}()
    global cumulative_seq_round_set_pt2_sort_join # = ""
    
    global pango_vec_pt2 # = Vector{String}()
    global clade_vec_pt2
    global cladepango_vec_pt2
    global collection_date_index_vec_pt2 # = Vector{Int}()
    global mp_pango_50_dates_pt2 # = Vector{Int}()
    global df_mp # = DataFrame(SeedMut = String[], MP_Tot_Mut_ct = Int[], MP_Tot_Reg_ct = Int[], Reg_Tot_Mut_ct = Int[], MP_Region_Number = Int[], Mutation = String[], MP_Mut_ct = Float64[], EPCI_Mut_ct = Int[], Adjusted_MP_seq_ct = Int[], EPCI_pct = Float64[], MP_pct = Float64[], non_MP_pct = Float64[], Fishers_Exact_Test_pval = Float64[], log10pvFISH = Float64[], Fold_Incr = Union{Float64, String}[], Chi2 = Float64[], EPCI_HQCS_Ratio = Float64[], avg_NonRBD_AAct = Float64[], avg_nonRBD_rel = Float64[], PangoDateIndex50_Avg = Int[], PangoDateStr50_Avg = String[], CollDateIndex_Avg = Int[], CollDateString_Avg = String[], Est_Infx_Length = Int[], PangoDateIndex50_min = Int[], PangoDateIndex50_p10 = Int[], PangoDateIndex50_Q1 = Int[], PangoDateIndex50_Q2 = Int[], PangoDateIndex50_Q3 = Int[], PangoDateIndex50_p90 = Int[], PangoDateIndex50_max = Int[], PangoDateStr50_min = String[], PangoDateStr50_p10 = String[], PangoDateStr50_Q1 = String[], PangoDateStr50_Q2 = String[], PangoDateStr50_Q3 = String[], PangoDateStr50_p90 = String[], PangoDateStr50_max = String[], CollDateIndex_min = Int[], CollDateIndex_p10 = Int[], CollDateIndex_Q1 = Int[], CollDateIndex_Q2 = Int[], CollDateIndex_Q3 = Int[], CollDateIndex_p90 = Int[], CollDateIndex_max = Int[], CollDateString_min = String[], CollDateString_p10 = String[], CollDateString_Q1 = String[], CollDateString_Q2 = String[], CollDateString_Q3 = String[], CollDateString_p90 = String[], CollDateString_max = String[], ALL_mp_Sequences = String[])
#    global pango_ct_dict # = Dict{String,Int}()  # THIS IS THE CRITICAL ONE
    global mp_seq_dict = Dict{String, Set{String}}()
    global all_excluded_muts
    global purespikebanned_0__purespikeallowed_1
    global nonspike_seqfactor__0off__1RBM__2RBD__3SpikeNonRBD
    global mp_folder_universal
    global df_mp_META_seedmut_set_dict
    global pt1_seed_mut_set
#    global region_ct2

#   global fake_dict_num_ct    
#   fake_seq_AA_muts = mp_meta_fake_chr_dict_DQ[fake_dict_num_ct]
#   global mp_index
#   global df_mp_META
#   global df_mp_META_META
#   global df_rand_corr_muts_META
#   global seq_privAA_len
#   global seq_privAA_len_meta
#   global pt1_seed_mut
    
    pt1_seed_mut = ""
    date_and_AAlen_seqs = Set{String}()
    all_muts_set_pt2 = Set{String}()
    chr_all_ratio = 0.0
    min_abs_mut_ct = 1
    max_pt1_rounds = 16
    seqfac_dict = Dict(0=>"OFF", 1=>"ON", 2=>"FULLBLAST")
    seqfac_dict_spike = Dict(0=>"OFF", 1=>"RBM", 2=>"RBD", 3=>"SpikeNonRBD")
    seqfac = seqfac_dict[seq_factor_OFF_HALF_ON__0_1_2]
    SpikeFac = seqfac_dict_spike[nonspike_seqfactor__0off__1RBM__2RBD__3SpikeNonRBD]
    error_ct = 0
    avg_mp_pango_date_index_int_50 = 0
    avg_mp_pango_date_string_50 = "00-00-00"
    pango_ct_dict = Dict{String,Int}()
    pango_ct_dict_sort = Vector{Pair{String,Int64}}()
    clade_ct_dict = Dict{String,Int}()
    clade_ct_dict_sort = Vector{Pair{String,Int64}}()
    cladepango_ct_dict = Dict{String,Int}()
    cladepango_ct_dict_sort = Vector{Pair{String,Int64}}()    
    seedmut = sel_muts_pt1[1]
    min_date_raw_ct = 2
######################################################################################################################################
#    all_excluded_muts_plus_mp_masked_muts = union(mp_masked_muts, all_excluded_muts)
    seq_AA_muts_not_excluded2 = Dict{String, Set{String}}()
    for (seq, mutset) in seq_AA_muts_not_excluded
        seq_AA_muts_not_excluded2[seq] = Set{String}()
        for mut in mutset
            if !(mut in mp_masked_muts)
                push!(seq_AA_muts_not_excluded2[seq], mut)
            end
        end
    end
######################################################################################################################################
### These dictionaries determine the number of MP mutations required for a sequence to qualify to be part of the set of sequences included in the MP
### As the number of regions in an MP increases the additional mutations required foreach additional MP (which starts at 0.4) is decreased to 0.3
### for each additional MP region from 5-8, then 0.2 for each additional MP region beyond 8       
    min_mut_ct_dict_pt1 = Dict{Int,Float64}(i=>round(digits=1, (max(i, 1)*pt1_ct_mltplr)) for i in 0:4)
    for i in 5:8
        min_mut_ct_dict_pt1[i] = 4*pt1_ct_mltplr + (i-4)*(pt1_ct_mltplr - 0.1)
    end
    for i in 9:20
        min_mut_ct_dict_pt1[i] = 4*pt1_ct_mltplr + 4*(pt1_ct_mltplr - 0.1) + (i-8)*(pt1_ct_mltplr - 0.2)
    end
    min_abs_mut_ct_dict_pt1 = Dict{Int,Int}(-1=>1, 0=>1, 1=>1, 2=>1, 3=>1, 4=>2, 5=>2, 6=>2, 7=>2, 8=>3, 9=>3, 10=>3, 11=>3, 12=>3, 13=>3, 14=>4, 15=>4, 16=>4, 17=>4, 18=>5, 19=>5, 20=>5)
    min_date_abs_ct_dict_pt1 = Dict{Int,Int}(-1=>1, 0=>1, 1=>1, 2=>2, 3=>2, 4=>2, 5=>3, 6=>3, 7=>3, 8=>3, 9=>4, 10=>4, 11=>4, 12=>4, 13=>5, 14=>5, 15=>5, 16=>5, 17=>5, 18=>5, 19=>5, 20=>5)
######################################################################################################################################
    min_mut_ct_dict_pt2 = Dict{Int,Float64}(i=>round(digits=1, (max(i, 1)*pt2_ct_mltplr)) for i in -1:3)
    for i in 4:7 
        min_mut_ct_dict_pt2[i] = 3*pt2_ct_mltplr + (i-3)*(pt2_ct_mltplr - 0.1)
    end
    for i in 8:20
        min_mut_ct_dict_pt2[i] = 3*pt2_ct_mltplr + 4*(pt2_ct_mltplr - 0.1) + (i-7)*(pt2_ct_mltplr - 0.2)
    end
    min_abs_mut_ct_dict_pt2 = Dict{Int,Int}(-1=>1, 0=>1, 1=>1, 2=>1, 3=>1, 4=>2, 5=>2, 6=>2, 7=>2, 8=>3, 9=>3, 10=>3, 11=>3, 12=>3, 13=>3, 14=>4, 15=>4, 16=>4, 17=>4, 18=>5, 19=>5, 20=>5)
    min_date_abs_ct_dict_pt2 = Dict{Int,Int}(-1=>1, 0=>1, 1=>1, 2=>2, 3=>2, 4=>2, 5=>3, 6=>3, 7=>3, 8=>3, 9=>4, 10=>4, 11=>4, 12=>4, 13=>4, 14=>4, 15=>5, 16=>5, 17=>5, 18=>5, 19=>5, 20=>5)   
######################################################################################################################################
    while true
        if pt2_region_analyzed == 1 || pt1_finished == 0
            date_and_AAlen_seqs = Set{String}()   
            all_muts_round_dict_pt2[pt2_round_ct] = Set{String}()
            cumulative_seq_round_set_pt2 = Set{String}()
            mp_pango_50_dates_pt2 = Vector{Int}()
            pango_vec_pt2 = Vector{String}()
            clade_vec_pt2 = Vector{String}()
            cladepango_vec_pt2 = Vector{String}()
            collection_date_index_vec_pt2 = Vector{Int}()
        end 
        coAAmut_ct_pt1 = Dict{String,Int}()
        coAAmut_ct_pt2 = Dict{String,Int}()
        coAAmut_ct_pt1_unknown = Dict{String,Int}()
        coAAmut_ct_pt2_unknown = Dict{String,Int}()
        prop_dict_pt1 = Dict{String,Tuple{Float64,Float64,Int,Int,Int,Float64,Float64,Float64,Union{Float64, String},Float64} }()
        prop_dict_pt2 = Dict{String,Tuple{Float64,Float64,Int,Int,Int,Float64,Float64,Float64,Union{Float64, String},Float64} }()
        df_mp = DataFrame(SeedMut = String[], MP_Tot_Mut_ct = Int[], MP_Tot_Reg_ct = Int[], Reg_Tot_Mut_ct = Int[], MP_Region_Number = Int[], Mutation = String[], 
            MP_Mut_ct = Float64[], EPCI_Mut_ct = Int[], Adjusted_MP_seq_ct = Int[], EPCI_pct = Float64[], MP_pct = Float64[], non_MP_pct = Float64[], Fishers_Exact_Test_pval = Float64[], 
            log10pvFISH = Float64[], Fold_Incr = Union{Float64, String}[], Chi2 = Float64[], EPCI_HQCS_Ratio = Float64[], avg_NonRBD_AAct = Float64[], avg_nonRBD_rel = Float64[], 
            PangoDateIndex50_Avg = Int[], PangoDateStr50_Avg = String[], CollDateIndex_Avg = Int[], CollDateString_Avg = String[], Est_Infx_Length = Int[], 
            PangoDateIndex50_min = Int[], PangoDateIndex50_p10 = Int[], PangoDateIndex50_Q1 = Int[], PangoDateIndex50_Q2 = Int[], PangoDateIndex50_Q3 = Int[], PangoDateIndex50_p90 = Int[], PangoDateIndex50_max = Int[], 
            PangoDateStr50_min = String[], PangoDateStr50_p10 = String[], PangoDateStr50_Q1 = String[], PangoDateStr50_Q2 = String[], PangoDateStr50_Q3 = String[], PangoDateStr50_p90 = String[], PangoDateStr50_max = String[], 
            CollDateIndex_min = Int[], CollDateIndex_p10 = Int[], CollDateIndex_Q1 = Int[], CollDateIndex_Q2 = Int[], CollDateIndex_Q3 = Int[], CollDateIndex_p90 = Int[], CollDateIndex_max = Int[], 
            CollDateString_min = String[], CollDateString_p10 = String[], CollDateString_Q1 = String[], CollDateString_Q2 = String[], CollDateString_Q3 = String[], CollDateString_p90 = String[], CollDateString_max = String[], ALL_mp_Sequences = String[])
    ########### df_mp columns key #################
    #### Mutation = mutation being analyzed
    #### MP_Tot_Mut_ct = total qualifying sequences with given mutation
    #### EPCI_Mut_ct = total chronic seqs w/given mutation
    #### Adjusted_MP_seq_ct = number of qualifying seqs (with ≥ min_mut_ct_pt2 in mut pattern)
    #### EPCI_pct = % of all chronic seqs w/given AA mut that appear in qualifying seqs (e.g. if 10 seqs have E:T30I & 4 of those are in seqs meeting the criteria, EPCI_prop = 0.40, EPCI_pct = 40%)
    #### MP_pct = % of all qualifying chronic seqs that have a given mut (e.g. if 100 seqs meet criteria & 4 of those have E:T30I, MP_prop = 0.04, MP_pct = 4%)
    #### log10pvFISH = -log10 Fisher's exact test p-value
    #### Chi2 = chi-squared
    #### PangoDateIndex50 = avg date_index_50 for pango lineage for all seqs in mut pattern (date_index_25 = date when 50% of all GISAID seqs of a Pango lineage were recorded)
    #### dateString50 = avg date_index_50 in "year-month-day" format  
        masked_muts = union(mp_masked_muts, all_excluded_muts)
        if pt1_round_ct == 0
            sel_muts_pt1_global = sel_muts_pt1
            pt1_seed_mut = join(sel_muts_pt1_global, ", ")
        end
        if pt1_finished == 0  
##################################################################################################################### 
#### Debug required to prevent empty "groups" from appearing in sel_muts_pt1_global
            blank_indices = Int[]
            for i in 1:length(sel_muts_pt1_global)
                mutstring = sel_muts_pt1_global[i]
                if mutstring == ""
                    push!(blank_indices, i)
                end
            end
            temp_sel_muts_pt1_global_standin = String[]
            for i in 1:length(sel_muts_pt1_global)
                mutstring = sel_muts_pt1_global[i]
                if !(i in blank_indices)
                    push!(temp_sel_muts_pt1_global_standin, mutstring)
                end
            end
            sel_muts_pt1_global = temp_sel_muts_pt1_global_standin
#####################################################################################################################
#### Rules for minimum required mutation count (min_mut_ct) for mut patterns w/a given number of regions (region_ct)
            region_ct = length(sel_muts_pt1_global)
            min_mut_ct = min_mut_ct_dict_pt1[region_ct]
            min_abs_mut_ct = min_abs_mut_ct_dict_pt1[region_ct]
            min_date_raw_ct = min_date_abs_ct_dict_pt1[region_ct]
#####################################################################################################################################
#### Splitting all mutation groups into individual mutations in order to determine the range for each region.
#### This is necessary because the qualifying criteria for mutations in a groups's range are different (less strict) than for mutations outside it. 
            sel_muts_pt1_ind = Vector{Vector{String}}()
            dict_ct = 1
            sel_muts_pt1_sort = sort(sel_muts_pt1_global, by = x -> sel_muts_pt1_sort_key_universal(x, sub_0__posonly_1))
            for mut_grp_str in sel_muts_pt1_sort
                if mut_grp_str ≠ ""
                    grp_muts = string.(split(mut_grp_str, ", "))
                    tmp_mut_vec = Vector{String}()
                    for mut in grp_muts
                        push!(tmp_mut_vec, mut)
                    end
                    tmp_mut_vec_sort = sort(tmp_mut_vec, by = x -> aa_pos_comprehensive_dict[x])
                    push!(sel_muts_pt1_ind, tmp_mut_vec_sort)
                end
            end
#####################################################################################################################################
#### This determines the start & end of a given mutation region's range (which extends plus_minus AA up & downstream its current limits)
#                                            mut_grp  gene   min  max
            sel_muts_pt1_min_max = Vector{Tuple{Int,String,Int,Int}}()
            for grp_num in 1:length(sel_muts_pt1_ind)
                mut_vec = sel_muts_pt1_ind[grp_num]
                AA_num_vec = Int[]
                gene = ""
                for mut in mut_vec
                    if mut ≠ ""
                        pos = aa_pos_comprehensive_dict[mut]
                        push!(AA_num_vec, pos)
                        gene = aa_gene_comprehensive_dict[mut]
                    end
                end
                if !isempty(AA_num_vec)
                    min_site = minimum(AA_num_vec) - plus_minus
                    max_site = maximum(AA_num_vec) + plus_minus
                    minmaxtup = (grp_num, gene, min_site, max_site)
                    push!(sel_muts_pt1_min_max, minmaxtup)
                else
                    minmaxtup = (grp_num, "", 0, 0)
                    push!(sel_muts_pt1_min_max, minmaxtup)
                end
            end
#####################################################################################################################################
            total_mut_ct = 0
#            total_mut_ct_nonspike = 0
#            total_mut_ct_nonRBD = 0
#            total_mut_ct_spike_nonRBD = 0
#            total_mut_ct_nonRBM = 0
######################################################################################################################################
## coAAmut_ct_pt1 = number of sequences meeting specified criteria (i.e. that have ≥X AA muts from a given mutational pattern) that possess a given mut. 
#                                AA_mut   Count  
            coAAmut_ct_pt1 = Dict{String,Int}()                  
###################################
            mut_groups_ln = length(sel_muts_pt1_global)   ## Total # of mut regions searched for (= # of muts if all indicated individually, e.g. "E:16, E:30, E:31"=1 region, i.e. length=1; "E:16", "E:30", "E:31"=3 mutations, i.e. length=3)
###################################
#### Each mut_group is assigned a number, and the value for each group number is a vector containing all the mutations in that group.
#### If every mutation is listed as a separate string in the `sel_muts_pt1_global` parameter, then each mutation is its own mut_group.
            mut_groups = Dict{Int, Vector{String}}(i => Vector{String}() for i in 1:mut_groups_ln)  ## 
######################################################################################################################################
#### all_sel_muts is a list of all individual mutations in sel_muts_pt1_global
### sel_muts_pt1_global = selected mutations/mut groups. If muts are listed individually (e.g. "E:F4L", "E:V5I", "E:I9T", etc) then each mutation
#      is a "group." If all the mutations in a given region are grouped together (e.g. "E:F4L, E:V5I, E:I9T") each group is a group.
#      only the "grp" optio is currently used. May just dump the "ind" version at some point. 
            mut_grp_ct = 0
            for mut_grp in sel_muts_pt1_global
                if mut_grp ≠ ""
                    mut_grp_ct += 1
                    muts = split(mut_grp, ", ")
                    for m in muts
                        str = string(m)
                        push!(mut_groups[mut_grp_ct], str)
                    end
                end
            end
######################################################################################################################################
            MP_seqs = Set{String}()  ## MP_seqs = all seqs that meet criteria
            MP_seqs_pangos = Dict{String,Int}()  ## MP_seqs_pangos = Count for Pango lineages for all seqs that meet criteria
            MP_seqs_inherited = Dict{String,Int}()  ## MP_seqs_inherited = Counts for inherited mutations among sequences on list. 
#            MP_seqs_priv_AA_adjustment_factors = Set{Float64}()  ## factors used to adjust for # of priv AA muts in a sequence
######################################################################################################################################
#### masked_muts are masked mutations. They are excluded from being included in the mutational pattern (see above for more info).
#### MP_seqs is a set containing all seqs that meet qualifying criteria (e.g. that have ≥ min_mut_ct muts in sel_muts_pt1_global)
            if ind_or_grp == "grp"
                for seq in all_unique_chr_seqs
                    seq_factor = 0
#####################################################
                    seq_priv_AA_mut_total = length(seq_AA_muts_not_excluded2[seq])
                    seq_factor = clamp(0.6, (avg_AA_sub_ct_per_chronic_seq_for_main_fx/seq_priv_AA_mut_total), 1.5)
                    half_seq_factor = (seq_factor+1)/2
                    if seq_factor_OFF_HALF_ON__0_1_2 == 0
                        seq_factor = 1
                    end
                    if seq_factor_OFF_HALF_ON__0_1_2 == 1
                        seq_factor = half_seq_factor
                    end
#####################################################
                    seq_grp_mut_ct = 0
                    abs_grp_mut_ct = 0
                    for (group, muts) in mut_groups
                        group_ct = 0
                        for mut in muts
                            if mut in seq_AA_muts_not_excluded2[seq] # && !(mut in masked_muts)
                                group_ct += 1
                            end
                        end
                        if group_ct > 0
                            seq_grp_mut_ct += 1*seq_factor
                            abs_grp_mut_ct += 1
                        end
                    end
                    if seq_grp_mut_ct ≥ min_mut_ct && abs_grp_mut_ct ≥ min_abs_mut_ct
                        push!(MP_seqs, seq)
                    end
                end
######################################################################################################
            elseif ind_or_grp == "ind"
                for seq in all_unique_chr_seqs
                    seq_factor = 0
#####################################################
                    seq_priv_AA_mut_total = length(seq_AA_muts_not_excluded2[seq])
                    seq_factor = clamp(0.6, (avg_AA_sub_ct_per_chronic_seq_for_main_fx/seq_priv_AA_mut_total), 1.5)
                    half_seq_factor = (seq_factor+1)/2
                    if seq_factor_OFF_HALF_ON__0_1_2 == 0
                        seq_factor = 1
                    end
                    if seq_factor_OFF_HALF_ON__0_1_2 == 1
                        seq_factor = half_seq_factor
                    end
#####################################################
                    seq_mut_ct = 0
                    abs_mut_ct = 0
                    for (group, muts) in mut_groups
                        for mut in muts
                            if mut in seq_AA_muts_not_excluded2[seq] # && !(mut in masked_muts)
                                seq_mut_ct +=1*seq_factor
                                abs_mut_ct += 1
                            end
                        end
                    end
                    if seq_mut_ct ≥ min_mut_ct && abs_mut_ct ≥ min_abs_mut_ct
                        push!(MP_seqs, seq)
                    end
                end
            end
            pt1_qual_seq_number = length(MP_seqs)
            if notrandom0__random1 == 0
                print("MP_seqs pt1 = $(pt1_qual_seq_number) sequences | ")
            end
###########################################################################################################################
            for seq in MP_seqs
                total_mut_ct += length(seq_AA_muts_not_excluded2[seq])
#### For each qualifying sequence, all of its private mutations are tallied in the coAAmut_ct_pt1 dictionary 
                for mut in seq_AA_muts_not_excluded2[seq]
                    coAAmut_ct_pt1[mut] = get(coAAmut_ct_pt1, mut, 0) + 1
                end
            end
######################################################################################################################################
#### This counts the pango lineages for each sequence on the list, then counts all the mutations inherited by that pango lineage.
#### Similar to the "unknown" dicts below, the purpose here is to remove from the denominator sequences that could not possibly have had a given 
####    private mutation (because you can't acquire a mutation you already have). For example, let's say we're testing whether the private mutation 
####    S:H655Y correlates with other mutations in a list of 100 sequences. If 70 of thoser sequences are Omicron, all of which inherited S:H655Y,
####    then only the 30 non-Omicron sequences will be used in calculating the Fisher's exact test (as well as all other calculations).
            for seq in MP_seqs
                pango = seq_pango[seq]
                MP_seqs_pangos[pango] = get(MP_seqs_pangos, pango, 0) + 1
                if !haskey(pango_AAsub_WT_universal, pango)
                    pango = pango_predecessor_meta_dict[pango][1]
                    if !haskey(pango_AAsub_WT_universal, pango)
                        pango = pango_predecessor_meta_dict[pango][1]
                    end
                end
                for mut in pango_AAsub_WT_universal[pango]
                    mutpo = aa_gene_and_pos_comprehensive_dict[mut]
                    if !(mutpo in seq_unknown_AA[seq])
                        MP_seqs_inherited[mut] = get(MP_seqs_inherited, mut, 0) + 1
                    end
                end
                for del in pango_AAdel_WT[pango]
                    delpo = aa_gene_and_pos_comprehensive_dict[del]
                    if !(delpo in seq_unknown_AA[seq])
                        MP_seqs_inherited[del] = get(MP_seqs_inherited, del, 0) + 1
                    end
                end     
            end
######################################################################################################################################
#### coAAmut_ct_pt1_unknown counts the # of times a given AA position is "unknown," either due to dropout or a mixed nucleotide for the sequences that qualify
#                                 unknown_AA_pos | count
            coAAmut_ct_pt1_unknown = Dict{String,Int}()
            total_AApos_ct_unknown = Dict{String,Int}()  ### Total unknown count for each AA position in each independent chronic lineage
            for seq in all_unique_chr_seqs_set
                for pos in seq_unknown_AA[seq]
                    if !haskey(total_AApos_ct_unknown, pos)
                        total_AApos_ct_unknown[pos] = 1
                    else
                        total_AApos_ct_unknown[pos] += 1
                    end
                end
            end
##############################################################################################################
### MP_seqs_unknown_region_ct tallies the number of mutational regions with at least one unknown AA
#                                                    EPI_ISL | total_regions_with_unknown_AA_count_in_seq
#            MP_seqs_unknown_region_ct = Dict{String,Int}()
#            total_unknown_region_ct = 0
##########################################################
            for seq in MP_seqs
#                MP_seqs_unknown_region_ct[seq] = 0
                for region_num in 1:length(sel_muts_pt1_ind)
#                   region_unkn_ct = 0
                    mut_vec = sel_muts_pt1_ind[region_num]
                    for mut in mut_vec
                        if mut ≠ ""
                            mutpo = aa_gene_and_pos_comprehensive_dict[mut]
                            if mutpo in seq_unknown_AA[seq]
#                                region_unkn_ct += 1
                                coAAmut_ct_pt1_unknown[mutpo] = get(coAAmut_ct_pt1_unknown, mutpo, 0) + 1
                            end
                        end
                    end
#                    if region_unkn_ct > 0
#                        MP_seqs_unknown_region_ct[seq] += 1
#                        total_unknown_region_ct += 1
#                    end
                end
            end
###################################################################################################################################### 
            MP_seq_ct = length(MP_seqs)
            avg_mutations_per_seq_no_dels = round(digits=2, total_mut_ct/MP_seq_ct)
            ratio_of_AAmuts_in_Grp_vs_AAmuts_in_all_chronics = avg_mutations_per_seq_no_dels/avg_AA_sub_ct_per_chronic_seq_for_main_fx
            ratio_of_AAmuts_in_Grp_vs_AAmuts_in_all_chronics_rd = round(digits = 2, ratio_of_AAmuts_in_Grp_vs_AAmuts_in_all_chronics)
            MP_seqs_sort = sort(collect(MP_seqs), by = x -> (length(x), x) )
######################################################################################################################################
#### for prop_dict_pt1, keys = mutations; values = tuple of (EPCI_pct_rd, MP_pct_rd, ct, chi_squared, pv_fish, log_pv_fish)
#### See below for meanings of each part of this tuple
            prop_dict_pt1 = Dict{String, Tuple{Float64,Float64,Int,Int,Int,Float64,Float64,Float64,Union{Float64, String},Float64} }()
#### prop_dict_pt1_unknown: keys = mutations, values = percentage of qualifying sequences that have an unknown AA at that position
            prop_dict_pt1_unknown = Dict{String,Float64}()
###### EPCI_prop = proportion of all chronic seqs with a given AA mut that appear in seqs meeting the criteria (e.g. if 10 seqs have E:T30I & 4 of those are in seqs meeting the criteria, EPCI_prop = 0.40)
###### MP_prop = proportion of all chronic seqs that meet the criteria that have a given AA mut (e.g. if 100 seqs meet the criteria & 4 of those have E:T30I, MP_prop = 0.04)
###### EPCI_pct_rd = same as qry prop but in % form and rounded to 1 decimal place
###### MP_pct_rd = same as qry prop but in % form and rounded to 1 decimal place
###### mut_prop_of_EPCI = proportion of all chronic seqs that possess given AA mut
###### MP_prop_of_EPCI = proportion of all chronic seqs that meet criteria
###### fold_inc = fold-change where 1.0 = expected proportion, i.e. MP_prop_of_EPCI = mut_prop_of_EPCI, or (equivalently), EPCI_pct = MP_pct
            for (mut, ct) in coAAmut_ct_pt1_unknown
                MP_pct_unk = ct/MP_seq_ct
                prop_dict_pt1_unknown[mut] = MP_pct_unk
            end
            total_EPCI_seq_ct = length(all_unique_chr_seqs_set)
######################################################################################################################################
            for (mut, co_mut_count) in coAAmut_ct_pt1
                mut_po = aa_gene_and_pos_comprehensive_dict[mut]
                unknown = get(coAAmut_ct_pt1_unknown, mut_po, 0)                                           ### unknown = total number of qualifying sequences w/an unknown AA at the mut_po position
                total_unknown = get(total_AApos_ct_unknown, mut_po, 0)                                     ### total_unknown = total number of sequences in entire EPCI list w/an unknown AA at the mut_po position
                inherited = 0
                all_chr_inherited = 0
                if sub_0__posonly_1 == 0
                    inherited = get(MP_seqs_inherited, mut, 0)
                    all_chr_inherited = get(all_chr_seqs_inherited, mut, 0)
                end
#### This section accounts for inherited muts that make the mut being analyzed impossible. E.g., all BA.2 & XBB have ORF1a:L3201F.
#### This means it is impossible for any BA.2/XBB sequence to have ORF1a:L3201P, so they must be removed from the denominator.
#### The whole pango section is to prevent double-counting muts that are both inherited and either lack coverage or have mixed nuc muts there (i.e. are "unknown")
                for inherited_mut in keys(MP_seqs_inherited)
                    if inherited_mut ≠ mut && aa_gene_and_pos_comprehensive_dict[mut] == mut_po && qryAA_comprehensive_dict[inherited_mut] ≠ refAA_comprehensive_dict[mut]
                        inherited_incompatible_mut_ct = get(MP_seqs_inherited, inherited_mut, 0)
                        inherited += inherited_incompatible_mut_ct
                    end
                    inherited_mut_po = aa_gene_and_pos_comprehensive_dict[inherited_mut]
                    for seq in MP_seqs
                        pango = seq_pango[seq]
                        if !haskey(pango_AAsub_WT_universal, pango)
                            pango = pango_predecessor_meta_dict[pango][1]
                            if !haskey(pango_AAsub_WT_universal, pango)
                                pango = pango_predecessor_meta_dict[pango][1]
                            end
                        end
                        if inherited_mut_po in seq_unknown_AA[seq] && inherited_mut in pango_AAsub_WT_universal[pango]
                            inherited -= 1
                        end
                    end         
                end
                adjusted_MP_seq_ct = MP_seq_ct - unknown - inherited                                   ### MP_seq_ct = All qualifying seqs (which have ≥ min_mut_ct muts from current mut pattern list)
                adjusted_EPCI_seq_ct = total_EPCI_seq_ct - total_unknown - all_chr_inherited           ### total_EPCI_seq_ct = total # of independent chronic seqs in entire list                                                                
                EPCI_mut_ct = AA_muts_ct_no_dels_universal[mut]                                        ### AA_muts_ct_no_dels_universal[mut] = total AA mut ct in all independent chronic seqs/lineages
                EPCI_prop = co_mut_count/AA_muts_ct_no_dels_universal[mut]                             ### EPCI_prop = proportion of the total EPCI count of a mut that appears in the mp sequence list
                MP_prop = co_mut_count/adjusted_MP_seq_ct                                              ### MP_prop = proportion of all qualifying MP sequences that have the mut as a private mutation
                non_MP_prop = (EPCI_mut_ct - co_mut_count)/(adjusted_EPCI_seq_ct - adjusted_MP_seq_ct) ### non_MP_prop = proportion of all non-MP EPCI sequences that have the mut as a private mutation 
                EPCI_pct = 100*EPCI_prop 
                MP_pct = 100*MP_prop
                non_MP_pct = 100*non_MP_prop                             
                mut_prop_of_EPCI = AA_muts_ct_no_dels_universal[mut]/adjusted_EPCI_seq_ct                  
                MP_prop_of_EPCI = adjusted_MP_seq_ct/adjusted_EPCI_seq_ct
                fold_incr::Union{Float64, String} = 0.0
                if non_MP_prop ≠ 0
                    fold_incr = MP_prop/non_MP_prop
                elseif non_MP_prop == 0
                    fake_non_MP_prop = 1/(adjusted_EPCI_seq_ct - adjusted_MP_seq_ct)
                    fake_fold_incr = MP_prop/fake_non_MP_prop
                    fake_fold_incr_rd = round(digits=2, fake_fold_incr)
                    fold_incr = ">$(fake_fold_incr_rd)"
                end
#                fold_incr_rd = round(digits=2, fold_incr)
                if fold_incr == ">Inf"
                    println("For $(mut), fold_incr = $(fold_incr) [should show >Inf]")
                end
                observed = co_mut_count
                expected = mut_prop_of_EPCI*adjusted_MP_seq_ct
                if observed ≥ expected
#            diff = observed-expected; chi_squared = diff^2/expected; chi_squared_rd = round(digits=1, chi_squared)
                    fishexact_up_left = co_mut_count                                                                                           ### seq with mut + meet search criteria (i.e. ≥ min_mut_ct muts from current mut pattern list)
                    fishexact_up_rt = max(adjusted_MP_seq_ct - co_mut_count, 0)                                                                ### seq without mut + meet search criteria (i.e. ≥ min_mut_ct muts from current mut pattern list)
                    fishexact_down_left = max(AA_muts_ct_no_dels_universal[mut] - co_mut_count, 0)                                             ### seq with mut + does not meet search criteria (i.e. < min_mut_ct muts from current mut pattern list)
                    fishexact_down_rt = max(adjusted_EPCI_seq_ct - AA_muts_ct_no_dels_universal[mut] - adjusted_MP_seq_ct + co_mut_count, 0)   ### seq w/o mut + does not meet search criteria (i.e. < min_mut_ct muts from current mut pattern list)
                    fish = FisherExactTest(fishexact_up_left, fishexact_up_rt, fishexact_down_left, fishexact_down_rt)
                    pv_fish = pvalue(fish)
                    log_pv_fish = -log10(pv_fish)
################# Chi-squared formula —— I got this straight from Claude.ai which says I was doing it all wrong before. No perceivable difference in the results though.
                    a = fishexact_up_left; b = fishexact_up_rt; c = fishexact_down_left; d = fishexact_down_rt
                    n = a + b + c + d    # total sample size
                    chi_2_top = n*(a*d - b*c)^2
                    chi_2_bottom = (a+b)*(c+d)*(a+c)*(b+d)
                    chi_squared = 0.0
                    chi_squared_rd = 0.0
                    if chi_2_bottom > 0
                        chi_squared = chi_2_top/chi_2_bottom
                        expected_a = (a + b) * (a + c) / n
                        expected_b = (a + b) * (b + d) / n
                        expected_c = (c + d) * (a + c) / n
                        expected_d = (c + d) * (b + d) / n
                        if min(expected_a, expected_b, expected_c, expected_d) < 5   # Yates' continuity correction (another Claude command)
                            corrected_numerator = n * (abs(a * d - b * c) - n/2)^2
                            chi_squared_yates = corrected_numerator/chi_2_bottom
                            chi_squared = chi_squared_yates
                            chi_squared_rd = round(digits=1, chi_squared)
                        end
#                        chi_squared_pvalue = 1 - cdf(Chisq(1), chi_squared)
                    end
################# End Claude's Chi-squared formula calculations ################
                    props = (EPCI_pct, MP_pct, EPCI_mut_ct, co_mut_count, adjusted_MP_seq_ct, chi_squared, pv_fish, log_pv_fish, fold_incr, non_MP_pct)
                    prop_dict_pt1[mut] = props
                end
            end
#####################################################################################################################################
##### sel_muts2_pt1_sort is the updated, sorted set of mutation regions to be included in the next round. 
##### sel_muts2_pt1 is the unsorted list, which is later sorted to make sel_muts2_pt1_sort
            sel_muts2_pt1 = Vector{String}() 
            sel_muts2_pt1_sort = Vector{String}()                                  
#### sel_muts2_pt1_pre_dict = Each key represents a mutation group; values are mut grp vectors (later joined to become strings in sel_muts2_pt1)                      
            sel_muts2_pt1_pre_dict = Dict{Int, Vector{String}}()  
#### Two sets below created to distinguish between mutations within one of the mut_grp ranges and those not within an existing group
####    Mutations within an existing group are subjected to a less stringent qualification requirement.
            prop_dict_pt1_muts_set = Set{String}()
            MP_prop_dict_pt1_muts_set = Set{String}()
            for mut in keys(prop_dict_pt1)
                push!(prop_dict_pt1_muts_set, mut)
            end
            for (mut, props) in prop_dict_pt1
                mut_count = props[4]
                log_pv_fish = props[8]
                mut_gene = aa_gene_comprehensive_dict[mut]
                mut_pos = aa_pos_comprehensive_dict[mut]
                grp_join_ct = 0
                for grp_num in 1:length(sel_muts_pt1_min_max)
                    if grp_join_ct == 0
                        gene = sel_muts_pt1_min_max[grp_num][2]
                        min_site = sel_muts_pt1_min_max[grp_num][3]
                        max_site = sel_muts_pt1_min_max[grp_num][4]
                        if mut_gene == gene && mut_pos ≥ min_site && mut_pos ≤ max_site
                            push!(MP_prop_dict_pt1_muts_set, mut)
                            if log_pv_fish ≥ min_grp_fish && mut_count ≥ min_match_ct                            
                                if haskey(sel_muts2_pt1_pre_dict, grp_num)
                                    push!(sel_muts2_pt1_pre_dict[grp_num], mut)
                                else
                                    sel_muts2_pt1_pre_dict[grp_num] = Vector{String}()
                                    push!(sel_muts2_pt1_pre_dict[grp_num], mut)
                                end
                            end
                            grp_join_ct += 1
                        end
                    end
                end
            end
#### non_MP_prop_dict_pt1_muts_set are the muts not within or near one of the existing mut regions (which must meet stricter stat criteria)
            non_MP_prop_dict_pt1_muts_set = setdiff(prop_dict_pt1_muts_set, MP_prop_dict_pt1_muts_set)
            new_region_ct = 0
            for (mut, props) in prop_dict_pt1
                if new_region_ct == 0
                    if mut in non_MP_prop_dict_pt1_muts_set
                        log_pv_fish = props[8]
                        mut_count = props[4]
                        if log_pv_fish ≥ min_log_pv_fish && mut_count ≥ min_match_ct                   
                            mut_gene = aa_gene_comprehensive_dict[mut]
                            mut_pos = aa_pos_comprehensive_dict[mut]
                            grp_num = length(keys(sel_muts2_pt1_pre_dict)) + 1
                            sel_muts2_pt1_pre_dict[grp_num] = Vector{String}()
                            push!(sel_muts2_pt1_pre_dict[grp_num], mut)
                            new_region_ct += 1
                        end
#                        if mut == "ORF7a:E22D" && "ORF7a:T39I" in sel_muts_pt1_global || mut == "ORF7a:T39I" && "ORF7a:E22D" in sel_muts_pt1_global
#                            mut_gene = aa_gene_comprehensive_dict[mut]
#                            mut_pos = aa_pos_comprehensive_dict[mut]
#                            grp_num = length(keys(sel_muts2_pt1_pre_dict))
#                            sel_muts2_pt1_pre_dict[grp_num] = Vector{String}()
#                            push!(sel_muts2_pt1_pre_dict[grp_num], mut)
#                            new_region_ct += 1
#                        end
#                        if mut == "ORF7a:E22D" && "ORF7a:F59I" in sel_muts_pt1_global || mut == "ORF7a:F59I" && "ORF7a:E22D" in sel_muts_pt1_global
#                            mut_gene = aa_gene_comprehensive_dict[mut]
#                            mut_pos = aa_pos_comprehensive_dict[mut]
#                            grp_num = length(keys(sel_muts2_pt1_pre_dict))
#                            sel_muts2_pt1_pre_dict[grp_num] = Vector{String}()
#                            push!(sel_muts2_pt1_pre_dict[grp_num], mut)
#                            new_region_ct += 1
#                        end
#                        if mut == "ORF7a:T39I" && "ORF7a:F59I" in sel_muts_pt1_global || mut == "ORF7a:F59I" && "ORF7a:T39I" in sel_muts_pt1_global
#                            mut_gene = aa_gene_comprehensive_dict[mut]
#                            mut_pos = aa_pos_comprehensive_dict[mut]
#                            grp_num = length(keys(sel_muts2_pt1_pre_dict))
#                            sel_muts2_pt1_pre_dict[grp_num] = Vector{String}()
#                            push!(sel_muts2_pt1_pre_dict[grp_num], mut)
#                            new_region_ct += 1
#                        end
                    end
                end
            end
#### Sorting muts within each mut group so they're easier to make sense of
            for (grp_num, mut_vec) in sel_muts2_pt1_pre_dict
                mut_vec_sort = sort(mut_vec, by = x -> (aa_gene_comprehensive_dict[x], aa_pos_comprehensive_dict[x]))
                sel_muts2_pt1_pre_dict[grp_num] = mut_vec_sort
            end
#### Turning mut groups into strings so they can be used in the next round of the function
#### Also excluding all groups that consist only of spike muts, if that parameter (purespikebanned_0__purespikeallowed_1) is enabled
            grp_1stmut_set = Set{String}()
            for (grp_num, mut_vec) in sel_muts2_pt1_pre_dict
                push!(grp_1stmut_set, mut_vec[1])
            end
            for (grp_num, mut_vec) in sel_muts2_pt1_pre_dict
                if purespikebanned_0__purespikeallowed_1 == 1
                    sel_mut_str = join(mut_vec, ", ")
                    push!(sel_muts2_pt1, sel_mut_str)
                elseif purespikebanned_0__purespikeallowed_1 == 0
                    if !all(aa_gene_comprehensive_dict[mut] == "S" for mut in grp_1stmut_set)
                        sel_mut_str = join(mut_vec, ", ")
                        push!(sel_muts2_pt1, sel_mut_str)
                    end
                end     
            end
#####################################################################################################################################
            sel_muts_pt1_sort = sort(sel_muts_pt1_global, by = x -> sel_muts_pt1_sort_key_universal(x, sub_0__posonly_1))
            sel_muts2_pt1_sort = sort(sel_muts2_pt1, by = x -> sel_muts_pt1_sort_key_universal(x, sub_0__posonly_1))
            pt1_round_ct += 1
            sel_muts_pt1_round_dict[pt1_round_ct] = sel_muts2_pt1_sort
#####################################################################################################################################
#### If there are no changes from previous round, we're finished & everything is sorted & printed. Otherwise, the whole thing runs again with the new mutation set.
            sel_muts_pt1_set = Set{String}()
            sel_muts2_pt1_set = Set{String}()
            for m in sel_muts_pt1_global
                push!(sel_muts_pt1_set, m)
            end
            for m in sel_muts2_pt1
                push!(sel_muts2_pt1_set, m)
            end
            sel_muts_pt1_global = sel_muts2_pt1_sort
            sel_muts_pt2 = sel_muts_pt1_round_dict[pt1_round_ct] ### already sorted!!
            if sel_muts2_pt1_set ≠ sel_muts_pt1_set && pt1_round_ct < max_pt1_rounds
                continue
            elseif pt1_round_ct < 3
                continue
            else
##############################################################################################################
                pt1_finished += 1
##############################################################################################################
                ### Checking to make sure there are actually two regions before doing part 2 (Claude.ai suggestion)
                blank_group_ct = 0
                for i in 1:length(sel_muts_pt2)
                    mutstring = sel_muts_pt2[i]
                    blank_mutstrings = Set(["", " ", "  ", "   ", "    ", "     ", "      ", "       ", "        ", "         ", "          ", "           ", "            "])
                    if mutstring in blank_mutstrings || isempty(mutstring)
                        blank_group_ct += 1
                    end
                end
                real_region_ct = length(sel_muts_pt2) - blank_group_ct
                if real_region_ct < 2
                    return (Dict{Int, Dict{String, Tuple{Float64,Float64,Int,Int,Int,Float64,Float64,Float64,Float64}}}(), "", 0, "00-00-00", Pair{String,Int64}[], Pair{String,Int64}[], Pair{String,Int64}[])
                end
##############################################################################################################
                all_muts_set_pt2 = Set{String}()
                sel_muts_pt2_ind_dict = Dict{Int, Vector{String}}()
                for i in 1:length(sel_muts_pt2)
                    grp_str = sel_muts_pt2[i]
                    grp_muts = string.(split(grp_str, ", "))
                    tmp_mvec = String[]
                    for mut in grp_muts
                        push!(tmp_mvec, mut)
                        push!(all_muts_set_pt2, mut)
                    end
                    tmp_mvec_sort = sort(tmp_mvec, by = x -> aa_pos_comprehensive_dict[x])
                    sel_muts_pt2_ind_dict[i] = tmp_mvec_sort
                end
                if !(pt1_seed_mut in all_muts_set_pt2)
#                    break
                    return (Dict{Int, Dict{String, Tuple{Float64,Float64,Int,Int,Int,Float64,Float64,Float64,Float64}}}(), "", 0, "00-00-00", Pair{String,Int64}[], Pair{String,Int64}[], Pair{String,Int64}[])
                end
                continue
            end
        end
        if pt1_finished > 0 && !isempty(sel_muts_pt2) && sel_muts_pt2 ≠ [""]
###########################################################################################################################################################################
###########################################################################################################################################################################
###########################################################################################################################################################################
####################################################################   PART 2   ###########################################################################################
####################################################################   PART 2   ###########################################################################################
####################################################################   PART 2   ###########################################################################################
###########################################################################################################################################################################
###########################################################################################################################################################################
###########################################################################################################################################################################
            pt2_iteration_ct += 1
            blank_group_ct = 0
            blank_mutstrings = Set(["", " ", "  ", "   ", "    ", "     ", "      ", "       ", "        ", "         ", "          ", "           ", "            "])
            for i in 1:length(sel_muts_pt2)
                mutstring = sel_muts_pt2[i]
                if mutstring in blank_mutstrings || isempty(mutstring)
                    blank_group_ct += 1
                end
            end
            region_ct2 = length(sel_muts_pt2) - blank_group_ct
            min_mut_ct_pt2 = 0.1
            min_mut_ct_pt2 = min_mut_ct_dict_pt2[region_ct2-1]
            min_abs_mut_ct = min_abs_mut_ct_dict_pt2[region_ct2-1]
            min_date_raw_ct = min_date_abs_ct_dict_pt2[region_ct2-1]
#################################################################################################################################### 
#### During each iteration, we are only testing the region that is excluded when selecting the sequences used to test correlations.
####################################################################################################################################
#            mut_file = "$(mp_folder_universal)/$(mut_pattern_name)_muts_by_rd__EPCI_$(EPCI_qc_str)__HQCS_$(HQCS_qc_str)__plusminus$(plus_minus)_minGrpFish$(min_grp_fish)_minFish$(min_log_pv_fish)_seqfac$(seqfac)_$(date).txt"
#            final_mut_file = "$(mp_folder_universal)/$(mut_pattern_name)_FINAL_MUTATION_STATS__EPCI_$(EPCI_qc_str)__HQCS_$(HQCS_qc_str)__plusminus$(plus_minus)_minGrpFish$(min_grp_fish)_minFish$(min_log_pv_fish)_seqfac$(seqfac)_$(date).txt"
#            pango_file = ""
#            if notrandom0__random1 == 1
#                mut_file = "$(rand_folder_ind)/mp_$(mut_pattern_name)_muts_by_rd__EPCI_$(EPCI_qc_str)__HQCS_$(HQCS_qc_str)__plusminus$(plus_minus)_minGrpFish$(min_grp_fish)_minFish$(min_log_pv_fish)_seqfac$(seqfac)_$(date).txt"
#                final_mut_file = "$(rand_folder_ind)/mp_$(mut_pattern_name)_FINAL_MUTATION_STATS__EPCI_$(EPCI_qc_str)__HQCS_$(HQCS_qc_str)__plusminus$(plus_minus)_minGrpFish$(min_grp_fish)_minFish$(min_log_pv_fish)_seqfac$(seqfac)_$(date).txt"
#                pango_file = "$(pango_counts_folder_rand_txt)/mp_$(mut_pattern_name)_PANGO_CTS__EPCI_$(EPCI_qc_str)__HQCS_$(HQCS_qc_str)__plusminus$(plus_minus)_minGrpFish$(min_grp_fish)_minFish$(min_log_pv_fish)_seqfac$(seqfac)_$(date).txt"
#            else
#                pango_file = "$(pango_counts_folder_rand_txt)/$(mut_pattern_name)_PANGO_CTS__EPCI_$(EPCI_qc_str)__HQCS_$(HQCS_qc_str)__plusminus$(plus_minus)_minGrpFish$(min_grp_fish)_minFish$(min_log_pv_fish)_seqfac$(seqfac)_$(date).txt"
#            end
#### Splitting all mutation groups into individual mutations in order to determine the range for each region.
#### Also making all_muts_set_pt2 to update all_muts_round_dict_pt2 with (if we've just finished a full round).
####   This is necessary because the qualifying criteria for mutations in a groups's range are different than for mutations outside it.
####   Has to be done before removing the region being analyzed so that the favorable criteria apply to that region.
####################################################################################################################################
### This is to determine the start & end of a given mutation region's range (which extends 4 AA up & downstream its current limits)
####################################################################################################################################
#                                     mut_grp  gene   min  max
            sel_muts_pt2_min_max = Tuple{Int,String,Int,Int}
            mut_vec = sel_muts_pt2_ind_dict[pt2_region_analyzed]
            AA_num_vec = Int[]
            gene = ""
            for mut in mut_vec
                if mut ≠ ""
                    pos = aa_pos_comprehensive_dict[mut]
                    push!(AA_num_vec, pos)
                    gene = aa_gene_comprehensive_dict[mut]
                end
            end
            min_site = 1
            max_site = 1
            if !isempty(AA_num_vec)
                min_site = minimum(AA_num_vec) - plus_minus
                max_site = maximum(AA_num_vec) + plus_minus
                sel_muts_pt2_min_max = (pt2_region_analyzed, gene, min_site, max_site)
            else
                sel_muts_pt2_min_max = (pt2_region_analyzed, gene, min_site, max_site)
#                push!(sel_muts_pt2_min_max, (pt2_region_analyzed, gene, min_site, max_site))
            end
############################################ End New Version (2025-11-1) #####################################################
#####################################################################################################################################
            total_mut_ct_pt2 = 0             ## Total mutations in all *counted* (i.e. qualifying) seqs
#####################################################################################################################################
## coAAmut_ct_pt2 = number of seqs meeting specified criteria (i.e. that have ≥min_mut_ct AA muts from a given mutational pattern) that possess a given mut. 
#                                 AA_mut   Count  
            coAAmut_ct_pt2 = Dict{String,Int}()                  
######################################################################################################################################
            MP_seqs_pt2 = Set{String}()  ## MP_seqs_pt2 = all seqs that meet criteria
            MP_seqs_pt2_pangos = Dict{String,Int}()  ## MP_seqs_pangos = Count for Pango lineages for all seqs that meet criteria
            MP_seqs_pt2_inherited = Dict{String,Int}()  ## MP_seqs_inherited = Counts for inherited mutations among sequences on list.
#            MP_seqs_pt2_priv_AA_adjustment_factors = Set{Float64}()  ## factors used to adjust for # of priv AA muts in a sequence
######################################################################################################################################
#### all_unique_chr_seqs combines all rep_seq_grps_maxmut_seqs and non_rep_seqs (both explained below). 
#### rep_seq_grps_maxmut_seqs is a dictionary containing one sequence that represents each group of related chronic sequences. 
#### Each representative sequence is the one with the most private AA mutations in the group. Its keys are the group's numbers 
####   (which are arbitrary) and its values are the EPI_ISL number for the sequence.
#### non_rep_seqs are chronic singlets (i.e. seqs with no closely related sequences from the same patient)
#### MP_seqs_pt2 is a set containing all sequences that meet the qualifying criteria (e.g. that have ≥2 mutations in sel_muts_pt2)
#### masked_muts are excluded from inclusion in the mutational pattern. (See above for explanation & justification.)
######################################################################################################################################
            if ind_or_grp == "grp"
                for seq in all_unique_chr_seqs
                    seq_factor = 0
#####################################################
#                    if nonspike_seqfactor__0off__1RBM__2RBD__3SpikeNonRBD == 0
                    seq_priv_AA_mut_total = length(seq_AA_muts_not_excluded2[seq])
                    seq_factor = clamp(0.6, (avg_AA_sub_ct_per_chronic_seq_for_main_fx/seq_priv_AA_mut_total), 1.5)
                    half_seq_factor = (seq_factor+1)/2
                    if seq_factor_OFF_HALF_ON__0_1_2 == 0
                        seq_factor = 1
                    end
                    if seq_factor_OFF_HALF_ON__0_1_2 == 1
                        seq_factor = half_seq_factor
                    end
##################################################### 
                    seq_grp_mut_ct = 0
                    abs_grp_mut_ct = 0
                    for grp_num in 1:length(sel_muts_pt2_ind_dict)
                        if !(grp_num == pt2_region_analyzed)
                            mut_vec = sel_muts_pt2_ind_dict[grp_num]
                            group_ct = 0
                            for mut in mut_vec
                                if mut in seq_AA_muts_not_excluded2[seq] # && !(mut in masked_muts)
                                    group_ct += 1
                                end
                            end
                            if group_ct > 0
                                seq_grp_mut_ct += 1*seq_factor
                                abs_grp_mut_ct += 1
                            end
                        end
                        if seq_grp_mut_ct ≥ min_mut_ct_pt2 && abs_grp_mut_ct ≥ min_abs_mut_ct
                            push!(MP_seqs_pt2, seq)
                            push!(cumulative_seq_round_set_pt2, seq)
                        end
                        if abs_grp_mut_ct ≥ min_date_raw_ct
                            push!(date_and_AAlen_seqs, seq)
                        end
                    end
                end
############ Part below only utilized if we want to count multiple mutations in each region (not used in normal calculations)
            elseif ind_or_grp == "ind"
                for seq in all_unique_chr_seqs
##############################################
                    seq_priv_AA_mut_total = length(seq_AA_muts_not_excluded2[seq])
                    seq_factor = clamp(0.6, (avg_AA_sub_ct_per_chronic_seq_for_main_fx/seq_priv_AA_mut_total), 1.5)
                    half_seq_factor = (seq_factor+1)/2
                    if seq_factor_OFF_HALF_ON__0_1_2 == 0
                        seq_factor = 1
                    end
                    if seq_factor_OFF_HALF_ON__0_1_2 == 1
                        seq_factor = half_seq_factor
                    end
###################################################
                    seq_mut_ct = 0
                    abs_mut_ct = 0
                    for grp_num in 1:length(sel_muts_pt2_ind_dict)
                        if !(grp_num == pt2_region_analyzed)
                            mut_vec = sel_muts_pt2_ind_dict[grp_num]
                            for mut in mut_vec
                                if mut in seq_AA_muts_not_excluded2[seq] # && !(mut in masked_muts)
                                    seq_mut_ct +=1*seq_factor
                                    abs_mut_ct += 1
                                end
                            end
                        end
                        if seq_mut_ct ≥ min_mut_ct_pt2 && abs_mut_ct ≥ min_abs_mut_ct
                            push!(MP_seqs_pt2, seq)
                            push!(cumulative_seq_round_set_pt2, seq) 
                        end
                        if abs_grp_mut_ct ≥ min_date_raw_ct
                            push!(date_and_AAlen_seqs, seq)
                        end
                    end
                end
            end
            if notrandom0__random1 == 0
                println("MP_seqs_pt2 length = $(length(MP_seqs_pt2)) | region_ct2 = $(region_ct2) | ")
            end
#####################################################################################
            for seq in MP_seqs_pt2
                total_mut_ct_pt2 += length(seq_AA_muts_not_excluded2[seq])
#### For each qualifying sequence, all of its private mutations are tallied in the coAAmut_ct_pt2 dictionary (keys = mutations, values = counts)
                for mut in seq_AA_muts_not_excluded2[seq]
                    #if !(mut in masked_muts)
                    coAAmut_ct_pt2[mut] = get(coAAmut_ct_pt2, mut, 0) + 1
                    #end
                end
            end
######################################################################################################################################
            for seq in MP_seqs_pt2
                WT_universal = Set{String}()
                WT_del_universal = Set{String}()
                pango = seq_pango[seq]
                MP_seqs_pt2_pangos[pango] = get(MP_seqs_pt2_pangos, pango, 0) + 1
                if !haskey(pango_AAsub_WT_universal, pango)
                    for i in 1:5
                        if haskey(pango_predecessor_meta_dict, pango)
                            if haskey(pango_predecessor_meta_dict[pango], i)
                                pango_i = pango_predecessor_meta_dict[pango][i]
                                if haskey(pango_AAsub_WT_universal, pango_i)
                                    WT_universal = pango_AAsub_WT_universal[pango_i]
                                    WT_del_universal = pango_AAdel_WT[pango_i]
                                    break
                                end
                            end
                        end
                    end
                else
                    WT_universal = pango_AAsub_WT_universal[pango]
                    WT_del_universal = pango_AAdel_WT[pango]
                end
                for mut in WT_universal
                    mutpo = aa_gene_and_pos_comprehensive_dict[mut]
                    if !(mutpo in seq_unknown_AA[seq])
                        MP_seqs_pt2_inherited[mut] = get(MP_seqs_pt2_inherited, mut, 0) + 1
                    end
                end
                for del in WT_del_universal
                    delpo = aa_gene_and_pos_comprehensive_dict[del]
                    if !(delpo in seq_unknown_AA[seq])
                        MP_seqs_pt2_inherited[del] = get(MP_seqs_pt2_inherited, del, 0) + 1
                    end
                end
            end
######################################################################################################################################
#### coAAmut_ct_pt2_unknown counts the number of timesa given AA position is "unknown," either due to dropout or a mixed nucleotide for
#       the sequences that qualify as matching a given mutational pattern 
#                                  unknown_AA_pos|count
            coAAmut_ct_pt2_unknown = Dict{String,Int}()
            total_AApos_ct_unknown_pt2 = Dict{String,Int}()  ### Total unknown count for each AA position in each independent chronic lineage
            for seq in all_unique_chr_seqs_set
                for pos in seq_unknown_AA[seq]
                    total_AApos_ct_unknown_pt2[pos] = get(total_AApos_ct_unknown_pt2, pos, 0) + 1
                end
            end
### MP_seqs_pt2_unknown_region_ct tallies the number of mutational regions with at least one unknown AA (not strictly necessary, tbh)
#                                                        EPI_ISL|total_regions_with_unknown_AA_count_in_seq
#            MP_seqs_pt2_unknown_region_ct = Dict{String,Int}()
#            total_unknown_region_ct_pt2 = 0
###################################################################
            for seq in MP_seqs_pt2
#                MP_seqs_pt2_unknown_region_ct[seq] = 0
                for region_num in 1:length(sel_muts_pt2_ind_dict)
#                    region_mut_ct = 0
                    mut_vec = sel_muts_pt2_ind_dict[region_num]
                    for mut in mut_vec
                        if mut ≠ ""
                            mutpo = aa_gene_and_pos_comprehensive_dict[mut]
                            if mutpo in seq_unknown_AA[seq]
#                                region_mut_ct += 1
                                coAAmut_ct_pt2_unknown[mutpo] = get(coAAmut_ct_pt2_unknown, mutpo, 0) + 1
                            end
                        end
                    end
#                    if region_mut_ct > 0
#                        MP_seqs_pt2_unknown_region_ct[seq] += 1
#                        total_unknown_region_ct_pt2 += 1
#                    end
                end
            end
###################################################################################################################################### 
            MP_seq_ct_pt2 = length(MP_seqs_pt2)
#           println("MP_seq_ct_pt2 = $(MP_seq_ct_pt2) | Seed Mut = $(pt1_seed_mut) ")
###################
#           avg_mutations_per_seq_pt2 = round(digits=2, total_mut_ct_pt2/MP_seq_ct_pt2)
#           ratio_of_AAmuts_in_Grp_vs_AAmuts_in_all_chronics_pt2 = avg_mutations_per_seq_pt2/avg_AA_sub_ct_per_chronic_seq
#           ratio_of_AAmuts_in_Grp_vs_AAmuts_in_all_chronics_pt2_rd = round(digits = 2, ratio_of_AAmuts_in_Grp_vs_AAmuts_in_all_chronics_pt2)
###################
            MP_seqs_pt2_sort = sort(collect(MP_seqs_pt2), by = x -> (length(x), x) )
######################################################################################################################################
#### for prop_dict_pt2, keys = mutations; values = tuple (EPCI_pct_rd, MP_pct_rd, ct, chi_squared, pv_fish, log_pv_fish))
#### See below for meanings of each part of this tuple
            prop_dict_pt2 = Dict{String, Tuple{Float64,Float64,Int,Int,Int,Float64,Float64,Float64,Union{Float64, String},Float64} }()
#### prop_dict_pt2_unknown: keys = mutations, values = percentage of qualifying sequences that have an unknown AA at that position
            prop_dict_pt2_unknown = Dict{String,Float64}()
##### EPCI_prop = proportion of all chronic seqs w/given AA mut that appear in qualifying seqs (e.g. if 10 seqs have E:T30I & 4 of those are in seqs meeting the criteria, EPCI_prop = 0.40)
##### MP_prop = proportion of all qualifying chronic seqs w/given mut (e.g. if 100 seqs meet criteria & 4 of those have E:T30I, MP_prop = 0.04)
##### EPCI_pct_rd = same as qry prop but in % form and rounded to 1 decimal place
##### MP_pct_rd = same as qry prop but in % form and rounded to 1 decimal place
##### mut_prop_of_EPCI = proportion of all chronic seqs that possess given AA mut
##### MP_prop_of_EPCI = proportion of all chronic seqs that meet criteria
            for (mut, ct) in coAAmut_ct_pt2_unknown
                MP_pct_unk = ct/MP_seq_ct_pt2
                prop_dict_pt2_unknown[mut] = MP_pct_unk
            end
            total_EPCI_seq_ct = length(all_unique_chr_seqs_set)
######################################################################################################################################
            for (mut, co_mut_count) in coAAmut_ct_pt2
                prop_dict_pt2[mut] = (0.0, 0.0, 0, 0, 0, 0.0, 0.0, 0.0, 0.0, 0.0)
###########################################################################
                unknown = 0                                                                               ### = total number of qualifying sequences w/an unknown AA at the mut_po position
                mut_po = aa_gene_and_pos_comprehensive_dict[mut]
                if haskey(coAAmut_ct_pt2_unknown, mut_po)
                    unknown = coAAmut_ct_pt2_unknown[mut_po]
                else
                    coAAmut_ct_pt2_unknown[mut_po] = 0
                end
                total_unknown = 0                                                                         ### total_unknown = total number of sequences in  EPCIw/an unknown AA at the mut_po position
                if haskey(total_AApos_ct_unknown_pt2, mut_po)
                    total_unknown = total_AApos_ct_unknown_pt2[mut_po]
                end
                inherited = 0
                all_chr_inherited = 0
                if sub_0__posonly_1 == 0
                    inherited = get(MP_seqs_pt2_inherited, mut, 0)
                    all_chr_inherited = get(all_chr_seqs_inherited, mut, 0)
                end
#### This section accounts for inherited muts that make the mut being analyzed impossible. E.g., all BA.2 & XBB have ORF1a:L3201F.
#### This means it is impossible for any BA.2/XBB sequence to have ORF1a:L3201P, so they must be removed from the denominator.
                for inherited_mut in keys(MP_seqs_pt2_inherited)
                    if inherited_mut ≠ mut && aa_gene_and_pos_comprehensive_dict[mut] == mut_po && qryAA_comprehensive_dict[inherited_mut] ≠ refAA_comprehensive_dict[mut]
                        inherited_incompatible_mut_ct = get(MP_seqs_pt2_inherited, inherited_mut, 0)
                        inherited += inherited_incompatible_mut_ct
                    end
                    inherited_mut_po = aa_gene_and_pos_comprehensive_dict[inherited_mut]
                    for seq in MP_seqs_pt2
                        pango = seq_pango[seq]
                        if !haskey(pango_AAsub_WT_universal, pango)
                            pango = pango_predecessor_meta_dict[pango][1]
                            if !haskey(pango_AAsub_WT_universal, pango)
                                pango = pango_predecessor_meta_dict[pango][1]
                            end
                        end
                        if inherited_mut_po in seq_unknown_AA[seq] && inherited_mut in pango_AAsub_WT_universal[pango]
                            inherited -= 1
                        end
                    end    
                end
                adjusted_MP_seq_ct = MP_seq_ct_pt2 - unknown - inherited                                   ### MP_seq_ct_pt2 = total # of qualifying sequences
                adjusted_EPCI_seq_ct = total_EPCI_seq_ct - total_unknown - all_chr_inherited               ### total_EPCI_seq_ct = total # of EPCI seqs
###################################################################
                EPCI_mut_ct = AA_muts_ct_no_dels_universal[mut]                                        
                EPCI_prop = co_mut_count/AA_muts_ct_no_dels_universal[mut]                                 ### AA_muts_ct_no_dels_universal[mut] = total AA mut count in all independent chronic seqs/lineages
                MP_prop = co_mut_count/adjusted_MP_seq_ct                                                  ### MP_prop = proportion of all qualifying MP sequences that have the mut as a private mutation
                non_MP_prop = (EPCI_mut_ct - co_mut_count)/(adjusted_EPCI_seq_ct - adjusted_MP_seq_ct) ### non_MP_prop = proportion of all non-MP EPCI sequences that have the mut as a private mutation
                EPCI_pct = 100*EPCI_prop
                MP_pct = 100*MP_prop
                non_MP_pct = 100*non_MP_prop
                mut_prop_of_EPCI = AA_muts_ct_no_dels_universal[mut]/adjusted_EPCI_seq_ct                  
                MP_prop_of_EPCI = adjusted_MP_seq_ct/adjusted_EPCI_seq_ct 
                fold_incr::Union{Float64, String} = 0.0
                if non_MP_prop ≠ 0
                    fold_incr = MP_prop/non_MP_prop
                elseif non_MP_prop == 0
                    fake_non_MP_prop = 1/(adjusted_EPCI_seq_ct - adjusted_MP_seq_ct)
                    fake_fold_incr = MP_prop/fake_non_MP_prop
                    fake_fold_incr_rd = round(digits=2, fake_fold_incr)
                    fold_incr = ">$(fake_fold_incr_rd)"
                end
#                fold_incr_rd = round(digits=2, fold_incr)
                if fold_incr == ">Inf"
                    println("For $(mut), fold_incr = $(fold_incr) [should show >Inf]")
                end
                observed = co_mut_count
                expected = mut_prop_of_EPCI*adjusted_MP_seq_ct
                if observed ≥ expected
                    fishexact_up_left = co_mut_count                                                                                            ### seq with mut + meet search criteria 
                    fishexact_up_rt = max(adjusted_MP_seq_ct - co_mut_count, 0)                                                                 ### seq without mut + meet search criteria 
                    fishexact_down_left = max(AA_muts_ct_no_dels_universal[mut] - co_mut_count, 0)                                              ### seq with mut + does not meet search criteria 
                    fishexact_down_rt = max(adjusted_EPCI_seq_ct - AA_muts_ct_no_dels_universal[mut] - adjusted_MP_seq_ct + co_mut_count , 0)   ### seq w/o mut + does not meet search criteria (i.e. < min_mut_ct_pt2 muts in current mut pattern list)
                    fish = FisherExactTest(fishexact_up_left, fishexact_up_rt, fishexact_down_left, fishexact_down_rt)
                    pv_fish = pvalue(fish)
                    log_pv_fish = -log10(pv_fish)
#### Chi2 formula —— Straight from Claude.ai which says I was doing it all wrong before. Seems to make ~zero difference though ########
                    a = fishexact_up_left;  b = fishexact_up_rt;  c = fishexact_down_left;  d = fishexact_down_rt
                    n = a + b + c + d  # total sample size
                    chi_2_top = n*(a*d - b*c)^2
                    chi_2_bottom = (a+b)*(c+d)*(a+c)*(b+d)
                    chi_squared = 0.0
                    chi_squared_rd = 0.0
                    if chi_2_bottom > 0
                        chi_squared = chi_2_top/chi_2_bottom
                        expected_a = (a + b) * (a + c) / n
                        expected_b = (a + b) * (b + d) / n
                        expected_c = (c + d) * (a + c) / n
                        expected_d = (c + d) * (b + d) / n
                        if min(expected_a, expected_b, expected_c, expected_d) < 5   # Yates' continuity correction (another Claude command)
                            corrected_numerator = n * (abs(a * d - b * c) - n/2)^2
                            chi_squared_yates = corrected_numerator/chi_2_bottom
                            chi_squared = chi_squared_yates
                            chi_squared_rd = round(digits=1, chi_squared)
                        end
#                        chi_squared_pvalue = 1 - cdf(Chisq(1), chi_squared)
                    end
################# End Claude's Chi-squared formula calculations ################
                    props_pt2 = (EPCI_pct, MP_pct, EPCI_mut_ct, co_mut_count, adjusted_MP_seq_ct, chi_squared, pv_fish, log_pv_fish, fold_incr, non_MP_pct)
                    prop_dict_pt2[mut] = props_pt2
                end
            end
###################################################################################################################################
#### sel_muts_pt2 is the updated set of mutation regions to be included in the next round of the function.
#### sel_muts_pt2_pre_dict = Each key represents a mutation group; values are mut grp vectors (later joined to become strings in sel_muts_pt2)                      
#### Two sets below created in order to distinguish between mutations within one of the mut_grp ranges and those not
#####################################################################################################################################
#### This mini-section is to determine if any mutations in the region being analyzed qualify. If none do, we move on.
            sel_muts2_pt2 = Vector{String}()
            reg_analyzed_qualified = 0
            for (mut, mut_props_pt2) in prop_dict_pt2
                co_mut_count = mut_props_pt2[4]
                log_pv_fish = mut_props_pt2[8]
                mut_gene = aa_gene_comprehensive_dict[mut]
                mut_pos = aa_pos_comprehensive_dict[mut]
################
                gene_reg_analyzed = sel_muts_pt2_min_max[2]
                min_reg_analyzed = sel_muts_pt2_min_max[3]
                max_reg_analyzed = sel_muts_pt2_min_max[4]
                if co_mut_count ≥ min_match_ct && mut_gene == gene_reg_analyzed && mut_pos ≥ min_reg_analyzed && mut_pos ≤ max_reg_analyzed && log_pv_fish ≥ min_log_pv_fish
                    reg_analyzed_qualified = 1
                    break
                end
            end
###############################
#### If the region qualifies, we examine all mutations in that region to see if they meet the minimum requirements to be included in the MP (namely, log_pv_fish ≥ min_grp_fish)
            if reg_analyzed_qualified == 1
                for (mut, mut_props_pt2) in prop_dict_pt2
                    co_mut_count = mut_props_pt2[4]
                    log_pv_fish = mut_props_pt2[8]
                    mut_gene = aa_gene_comprehensive_dict[mut]
                    mut_pos = aa_pos_comprehensive_dict[mut]
##################
                    gene_reg_analyzed = sel_muts_pt2_min_max[2]
                    min_reg_analyzed = sel_muts_pt2_min_max[3]
                    max_reg_analyzed = sel_muts_pt2_min_max[4]
                    if co_mut_count ≥ min_match_ct && mut_gene == gene_reg_analyzed && mut_pos ≥ min_reg_analyzed && mut_pos ≤ max_reg_analyzed && log_pv_fish ≥ min_grp_fish
                        push!(sel_muts2_pt2, mut)
                        push!(all_muts_round_dict_pt2[pt2_round_ct], mut)
                    end
                end
            end
#########################################################################################################################################
############ Sorting these so that the results of each round an iteration are easy to compare——also the final results
            sel_muts2_pt2_sort = sort(sel_muts2_pt2, by = x -> sel_muts_pt1_sort_key_universal(x, sub_0__posonly_1))
            sel_muts_pt2_ind_dict[pt2_region_analyzed] = sel_muts2_pt2_sort
#########################################################################################################################################
#################################################### Below: Debug Code ############################################
            if notrandom0__random1 == 0
                if length(sel_muts2_pt2) ≥ 1
                    println("length of sel_muts_pt2 = $(length(sel_muts_pt2))  |  number of regions = $(region_ct2)")
                    println("## sel_muts2_pt2, Seed Mut = $(pt1_seed_mut)  |  Region #$(pt2_region_analyzed) ")
                    sel_muts2_pt2_string = join(sel_muts2_pt2_sort, ", ")
                    println(sel_muts2_pt2_string); println()
                end
            end
#################################################### Above: Debug Code ############################################
###################################################################################################################################################
            if length(keys(prop_dict_pt2)) > 1         
                qual_mut_ct = 0
                for mut in sel_muts2_pt2
                    props2 = prop_dict_pt2[mut]                      
                    EPCI_pct = props2[1]
                    MP_pct = props2[2]
                    EPCI_mut_ct = props2[3]
                    qual_mut_ct = props2[4]
                    qual_seq_ct = props2[5]
                    Chi2 = props2[6]
                    pvFish = props2[7]
                    log10pvFISH = props2[8]
                    fold_incr = props2[9]
                    non_MP_pct = props2[10]
                    chr_all_ratio = mp_chr_all_ratio[mut]
                    if !haskey(df_props_dict_pt2, pt2_round_ct)
                        df_props_dict_pt2[pt2_round_ct] = Dict{Int, Dict{String, Tuple{Float64,Float64,Int,Int,Int,Float64,Float64,Float64,Union{Float64, String},Float64}}}()
                    end
                    if !haskey(df_props_dict_pt2[pt2_round_ct], pt2_region_analyzed)
                        df_props_dict_pt2[pt2_round_ct][pt2_region_analyzed] = Dict{String, Tuple{Float64,Float64,Int,Int,Int,Float64,Float64,Float64,Union{Float64, String},Float64}}()
                    end
                    df_props_dict_pt2[pt2_round_ct][pt2_region_analyzed][mut] = (EPCI_pct, MP_pct, EPCI_mut_ct, qual_mut_ct, qual_seq_ct, Chi2, pvFish, log10pvFISH, fold_incr, non_MP_pct)
                end
            end
#####################################################################################################################################
#### Move on to the next region (as long as we didn't just finish the final region)
#### Sel_muts_pt2 has to be turned into a vector of strings, with each string containing all a region's mutations, separated by commas
#            prev_round_pt2 = pt2_round_ct-1
            all_muts_set_pt2 = Set{String}()
            sel_muts_pt2 = Vector{String}()
            for grp in 1:length(sel_muts_pt2_ind_dict)
                mut_vec = sel_muts_pt2_ind_dict[grp]
                for mut in mut_vec
                    push!(all_muts_set_pt2, mut)
                end
                mut_join = join(mut_vec, ", ")
                push!(sel_muts_pt2, mut_join)
            end
            if pt2_region_analyzed < region_ct2 + blank_group_ct
                pt2_region_analyzed += 1
                continue
#### If on last region, start over on 1st region w/updated list——unless there were 0 changes this round, in which case we're done.
            elseif pt2_region_analyzed > region_ct2 + blank_group_ct #DEBUG
                print("\n"^8); println("ERROR ALERT: pt2_region_analyzed is greater than region_ct2; SHOULD NOT HAPPEN | Seed Mut = $(pt1_seed_mut) "); print("\n"^8)  #DEBUG 
            elseif pt2_round_ct < 3 && pt2_region_analyzed == region_ct2 + blank_group_ct
                pt2_round_ct += 1
                pt2_region_analyzed = 1
                continue
            elseif pt2_region_analyzed == region_ct2 + blank_group_ct && all_muts_round_dict_pt2[pt2_round_ct] ≠ all_muts_round_dict_pt2[pt2_round_ct-1] && pt2_round_ct ≤ 12 # && all_muts_round_dict_pt2[pt2_round_ct] ≠ all_muts_round_dict_pt2[pt2_round_ct-2]
                pt2_round_ct += 1
                pt2_region_analyzed = 1
                continue
            elseif (pt2_region_analyzed == region_ct2 + blank_group_ct && all_muts_round_dict_pt2[pt2_round_ct] == all_muts_round_dict_pt2[pt2_round_ct-1]) || (pt2_region_analyzed == region_ct2 + blank_group_ct && pt2_round_ct ≥ 12)
############################################################################################################################################################################
###################################################### Begin: Pango lineage & collection date section ######################################################################
############################################################################################################################################################################
                seq_index_errors = 0
                cumulative_seq_round_set_pt2_sort = sort(collect(cumulative_seq_round_set_pt2), by = x -> (length(x), x))
                cumulative_seq_round_set_pt2_sort_join = join(cumulative_seq_round_set_pt2_sort, ", ") 
                for seq in cumulative_seq_round_set_pt2
                    pango = seq_pango[seq]
                    clade = seq_clade[seq]
                    cladepango = clade_to_pango[clade]
                    coll_date = seq_date_index[seq]
                    push!(pango_vec_pt2, pango)
                    push!(clade_vec_pt2, clade)
                    push!(cladepango_vec_pt2, cladepango)
                    if coll_date > 0 && coll_date < 4000
                        push!(collection_date_index_vec_pt2, coll_date)
                    else
                        seq_index_errors += 1
                    end
                    clade_ct_dict[clade] = get(clade_ct_dict, clade, 0) + 1
                    cladepango_ct_dict[cladepango] = get(cladepango_ct_dict, cladepango, 0) + 1
                end
                for pango in pango_vec_pt2
                     date_index_50 = pango_pct_date_index[pango][50.0]
                     push!(mp_pango_50_dates_pt2, date_index_50)
                     pango_ct_dict[pango] = get(pango_ct_dict, pango, 0) + 1
                end
############################################################################################
                pango_ct_dict_sort = sort(collect(pango_ct_dict), by = x -> x[2], rev=true)
                clade_ct_dict_sort = sort(collect(clade_ct_dict), by = x -> x[2], rev=true)
                cladepango_ct_dict_sort = sort(collect(cladepango_ct_dict), by = x -> x[2], rev=true)
############################################################################################             
                mp_pango_50_dates_pt2_sort = sort(mp_pango_50_dates_pt2)
                collection_date_index_vec_pt2_sort = sort(collection_date_index_vec_pt2)
                total_pango_50 = 0
                pango_date_index_sum_50 = 0
                avg_mp_pango_date_index_50 = 0.0
                avg_mp_pango_date_index_rd_50 = 0
                avg_mp_pango_date_index_int_50 = 0
                avg_mp_pango_date_tuple_50 = (0, 0, 0)
                avg_mp_pango_date_string_50 = "00-00-00"
                collection_date_index_sum = 0
                avg_collection_date_index = 0
                avg_collection_date_index_int = 0
                avg_collection_date_string = "00-00-00"
                pango50_date_index_percentiles = Int[]
                pango50_date_string_percentiles = String[]
                collection_date_index_percentiles = Int[]
                collection_date_string_percentiles = String[]
                est_infx_len_at_collection = 0
                if !(length(mp_pango_50_dates_pt2) == 0)   
####################################
                     total_pango_50 = length(mp_pango_50_dates_pt2)
                     pango_date_index_sum_50 = sum(mp_pango_50_dates_pt2)
                     avg_mp_pango_date_index_50 = pango_date_index_sum_50/total_pango_50
                     avg_mp_pango_date_index_rd_50 = round(avg_mp_pango_date_index_50)
                     avg_mp_pango_date_index_int_50 = Int(avg_mp_pango_date_index_rd_50)
                     avg_mp_pango_date_tuple_50 = index_to_tuple[avg_mp_pango_date_index_int_50]
                     avg_mp_pango_date_string_50 = index_to_date_str[avg_mp_pango_date_index_int_50]
####################################
                     total_collection_date_seqs = length(collection_date_index_vec_pt2)
                     collection_date_index_sum = sum(collection_date_index_vec_pt2)
                     avg_collection_date_index = collection_date_index_sum/total_collection_date_seqs
                     avg_collection_date_index_rd = round(avg_collection_date_index)
                     avg_collection_date_index_int = Int(avg_collection_date_index_rd)
                     avg_collection_date_string = index_to_date_str[avg_collection_date_index_int]
                     if total_pango_50 - seq_index_errors ≠ total_collection_date_seqs
                         println("total_pango_50 - seq_index_errors ≠ total_collection_date_seqs, SOMETHING IS WRONG")
                     end
####################################
                     pango50_date_index_percentiles = quantile(mp_pango_50_dates_pt2_sort, [0.0, 0.10, 0.25, 0.5, 0.75, 0.90, 1.0])
                     collection_date_index_percentiles = quantile(collection_date_index_vec_pt2_sort, [0.0, 0.10, 0.25, 0.5, 0.75, 0.90, 1.0])
                     pango50_date_index_percentiles = round.(pango50_date_index_percentiles)
                     pango50_date_index_percentiles = Int.(pango50_date_index_percentiles)
                     collection_date_index_percentiles = round.(collection_date_index_percentiles)
                     collection_date_index_percentiles = Int.(collection_date_index_percentiles)
                     for date_index in pango50_date_index_percentiles
                         date_index_rd = round(date_index)
                         date_index_int = Int(date_index_rd)
                         date_str = index_to_date_str[date_index_int]
                         push!(pango50_date_string_percentiles, date_str)
                     end
                     for date_index in collection_date_index_percentiles
                         date_index_rd = round(date_index)
                         date_index_int = Int(date_index_rd)
                         date_str = index_to_date_str[date_index_int]
                         push!(collection_date_string_percentiles, date_str)
                     end
                     est_infx_len_at_collection = avg_collection_date_index_int - avg_mp_pango_date_index_int_50
####################################
                else
                     error_ct += 1
                end
############################################################################################################################################################################
###################################################### End: Pango lineage & collection date section ########################################################################
############################################################################################################################################################################
#df_mp = DataFrame(SeedMut = String[], MP_Tot_Mut_ct = Int[], MP_Tot_Reg_ct = Int[], Reg_Tot_Mut_ct = Int[], MP_Region_Number = Int[], Mutation = String[], MP_Mut_ct = Float64[], EPCI_Mut_ct = Int[], Adjusted_MP_seq_ct = Int[], EPCI_pct = Float64[], MP_pct = Float64[], non_MP_pct = Float64[], Fishers_Exact_Test_pval = Float64[], log10pvFISH = Float64[], Fold_Incr = Union{Float64, String}[], Chi2 = Float64[], EPCI_HQCS_Ratio = Float64[], avg_NonRBD_AAct = Float64[], avg_nonRBD_rel = Float64[], 
#        PangoDateIndex50_Avg = Int[], PangoDateStr50_Avg = String[], CollDateIndex_Avg = Int[], CollDateString_Avg = String[], 
#        Est_Infx_Length = Int[], 
#        PangoDateIndex50_min = Int[], PangoDateIndex50_p10 = Int[], PangoDateIndex50_Q1 = Int[], PangoDateIndex50_Q2 = Int[], PangoDateIndex50_Q3 = Int[], PangoDateIndex50_p90 = Int[], PangoDateIndex50_max = Int[], 
#        PangoDateStr50_min = String[], PangoDateStr50_p10 = String[], PangoDateStr50_Q1 = String[], PangoDateStr50_Q2 = String[], PangoDateStr50_Q3 = String[], PangoDateStr50_p90 = String[], PangoDateStr50_max = String[], 
#        CollDateIndex_min = Int[], CollDateIndex_p10 = Int[], CollDateIndex_Q1 = Int[], CollDateIndex_Q2 = Int[], CollDateIndex_Q3 = Int[], CollDateIndex_p90 = Int[], CollDateIndex_max = Int[], 
#        CollDateString_min = String[], CollDateString_p10 = String[], CollDateString_Q1 = String[], CollDateString_Q2 = String[], CollDateString_Q3 = String[], CollDateString_p90 = String[], CollDateString_max = String[], ALL_mp_Sequences = String[])
############################################################################################################################################################################
###################################### Begin: Rigorous Avg_AA, Avg_AA_nonRBD len, & collection date section ################################################################
############################################################################################################################################################################
                mp_date_index_vec = Int[]
                mp_AA_len_vec = Int[]
#                    mp_AA_nonRBD_len_vec = Int[]
                seq_index_error_count = 0
                avg_hard_mp_seq_date_index = 0
                avg_hard_mp_seq_date_string = ""
                avg_hard_mp_AA_len = 0
                avg_hard_mp_AA_len_raw = 0
                avg_hard_mp_AA_len_relative = 0
#                    avg_hard_mp_AA_nonRBD_len = 0
#                    avg_hard_mp_AA_nonRBD_len_raw = 0
#                    avg_hard_mp_AA_nonRBD_len_relative = 0
                if !isempty(date_and_AAlen_seqs)
                    for seq in date_and_AAlen_seqs
                        seq_dindex = seq_date_index[seq]
                        if seq_dindex > 0 && seq_dindex < 4000
                            push!(mp_date_index_vec, seq_dindex)
                        else
                            seq_index_error_count += 1
                        end
                        seqaa_len = seq_privAA_len[seq]
                        push!(mp_AA_len_vec, seqaa_len)
#                            seqaa_len_rel = seq_privAA_len_relative[seq]
#                            push!(mp_AA_len_vec, seqaa_len)
                    end
                    total_date_index_sum = 0
                    date_seq_count = 0
                    if !isempty(mp_date_index_vec) 
                        total_date_index_sum = sum(mp_date_index_vec)
                        date_seq_count = length(mp_date_index_vec)
                    end
                    avg_hard_mp_seq_date_index_pre = 0
                    if total_date_index_sum ≠ 0 && date_seq_count ≠ 0
                        avg_hard_mp_seq_date_index_pre = round(digits=0, total_date_index_sum/date_seq_count)
                    end
                    if isfinite(avg_hard_mp_seq_date_index_pre) && avg_hard_mp_seq_date_index_pre > 0
                        avg_hard_mp_seq_date_index = Int(avg_hard_mp_seq_date_index_pre)
                        avg_hard_mp_seq_date_string = get(index_to_date_str, avg_hard_mp_seq_date_index, "")
                    else
                       println("non-functioning avg_hard_mp_seq_date_index_pre = $(avg_hard_mp_seq_date_index_pre)")
                    end
############################
                    total_AA_len = 0
                    AA_len_seq_count = 0
                    if !isempty(mp_AA_len_vec)
                        total_AA_len = sum(mp_AA_len_vec)
                        AA_len_seq_count = length(mp_AA_len_vec)
                    end
                    if total_AA_len ≠ 0 && AA_len_seq_count ≠ 0
                        avg_hard_mp_AA_len_raw = total_AA_len/AA_len_seq_count
                        avg_hard_mp_AA_len = round(digits=2, avg_hard_mp_AA_len_raw)
                    end
                    avg_hard_mp_AA_len_relative = round(digits=2, avg_hard_mp_AA_len_raw/avg_AA_sub_ct_per_chronic_seq_for_main_fx)
                end
############################################################################################################################################################################
################################################# End: Rigorous Avg AA len & collection date section #######################################################################
############################################################################################################################################################################
                if !isempty(keys(df_props_dict_pt2)) && region_ct2 > 1 && haskey(df_props_dict_pt2, pt2_round_ct)
                    region_mut_array_dict = Dict{Int, Vector{String}}()
                    region_key_sort = sort(collect(keys(df_props_dict_pt2[pt2_round_ct])))
                    for region in region_key_sort
                        region_mut_array_dict[region] = Vector{String}()
                        for mut in keys(df_props_dict_pt2[pt2_round_ct][region])
                            push!(region_mut_array_dict[region], mut)
                        end
                        region_mut_array_sort = sort(region_mut_array_dict[region], by = x -> (aa_gene_comprehensive_dict[x], aa_pos_comprehensive_dict[x]))
                        region_mut_array_dict[region] = region_mut_array_sort
                    end
########################### Begin: Section to eliminate duplicate mut patterns #############################
                    df_mp_META_set = Set{String}()
                    for region in region_key_sort   
                        for mut in region_mut_array_dict[region]
                            push!(df_mp_META_set, mut)
                        end
                    end
                    if nonmulti0__multi1 == 1
                        if df_mp_META_set in df_mp_META_set_set
                            return (Dict{Int, Dict{String, Tuple{Float64,Float64,Int,Int,Int,Float64,Float64,Float64,Union{Float64, String},Float64}}}(), "", 0, "", Pair{String,Int64}[], Pair{String,Int64}[], Pair{String,Int64}[])
                        end
                    end
                    overlap = 0
                    proportion_share = 0.0
                    unique = 1
                    for mp_set in df_mp_META_set_set
                        smaller = Set()
                        if length(mp_set) > length(df_mp_META_set)
                            smaller = df_mp_META_set
                        else
                            smaller = mp_set
                        end
                        overlap = length(intersect(df_mp_META_set, mp_set))
                        proportion_share = overlap/length(smaller)
                        if overlap ≥ 5 && proportion_share > 0.5 && length(mp_set) > length(df_mp_META_set)
                            unique = 0
                            break
                        end
                    end
########################### End: Section to eliminate duplicate mut patterns ##############################
                    if pt1_seed_mut in df_mp_META_set
                        if unique == 1
                            df_mp_META_groups_dict_unique[pt1_seed_mut] = Vector{Tuple{String,Int,Int,Int,Int,String,Float64,Int,Int,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Int,String,Int,String,Int,Int,Int,Int,Int,Int,Int,Int,String,String,String,String,String,String,String,Int,Int,Int,Int,Int,Int,Int,String,String,String,String,String,String,String,String}}()
                        end
                        df_mp_META_groups_dict[pt1_seed_mut] = Vector{Tuple{String,Int,Int,Int,Int,String,Float64,Int,Int,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Int,String,Int,String,Int,Int,Int,Int,Int,Int,Int,Int,String,String,String,String,String,String,String,Int,Int,Int,Int,Int,Int,Int,String,String,String,String,String,String,String,String}}()
                        mp_tot_region_ct = length(region_key_sort)
                        mp_total_mut_ct = length(df_mp_META_set)
                        for region in region_key_sort
                            region_total_mut_ct = length(region_mut_array_dict[region])
                            for mut in region_mut_array_dict[region]
                                EPCI_pct__MP_pct__totChrMutCt__MP_Tot_Mut_ct__Adjusted_MP_seq_ct__Chi2__pvFish__log10pvFISH = df_props_dict_pt2[pt2_round_ct][region][mut]
                                EPCI_pct = EPCI_pct__MP_pct__totChrMutCt__MP_Tot_Mut_ct__Adjusted_MP_seq_ct__Chi2__pvFish__log10pvFISH[1]
                                MP_pct = EPCI_pct__MP_pct__totChrMutCt__MP_Tot_Mut_ct__Adjusted_MP_seq_ct__Chi2__pvFish__log10pvFISH[2]
                                EPCI_mut_ct = EPCI_pct__MP_pct__totChrMutCt__MP_Tot_Mut_ct__Adjusted_MP_seq_ct__Chi2__pvFish__log10pvFISH[3]
                                qual_mut_ct = EPCI_pct__MP_pct__totChrMutCt__MP_Tot_Mut_ct__Adjusted_MP_seq_ct__Chi2__pvFish__log10pvFISH[4]
                                qual_seq_ct = EPCI_pct__MP_pct__totChrMutCt__MP_Tot_Mut_ct__Adjusted_MP_seq_ct__Chi2__pvFish__log10pvFISH[5]
                                Chi2 = EPCI_pct__MP_pct__totChrMutCt__MP_Tot_Mut_ct__Adjusted_MP_seq_ct__Chi2__pvFish__log10pvFISH[6]
                                pvFish = EPCI_pct__MP_pct__totChrMutCt__MP_Tot_Mut_ct__Adjusted_MP_seq_ct__Chi2__pvFish__log10pvFISH[7]
                                log10pvFISH = EPCI_pct__MP_pct__totChrMutCt__MP_Tot_Mut_ct__Adjusted_MP_seq_ct__Chi2__pvFish__log10pvFISH[8]
                                fold_incr = EPCI_pct__MP_pct__totChrMutCt__MP_Tot_Mut_ct__Adjusted_MP_seq_ct__Chi2__pvFish__log10pvFISH[9]
                                non_MP_pct = EPCI_pct__MP_pct__totChrMutCt__MP_Tot_Mut_ct__Adjusted_MP_seq_ct__Chi2__pvFish__log10pvFISH[10]
                                chr_all_ratio = mp_chr_all_ratio[mut]
                                if mp_total_mut_ct == 2 && qual_mut_ct < 4
                                    return (Dict{Int, Dict{String, Tuple{Float64,Float64,Int,Int,Int,Float64,Float64,Float64,Union{Float64, String},Float64}}}(), "", 0, "", Pair{String,Int64}[], Pair{String,Int64}[], Pair{String,Int64}[])
                                end
    ################################################################
                                push!(df_mp, (pt1_seed_mut, mp_total_mut_ct, mp_tot_region_ct, region_total_mut_ct, region, mut, qual_mut_ct, EPCI_mut_ct, qual_seq_ct, EPCI_pct, MP_pct, non_MP_pct, pvFish, log10pvFISH, fold_incr, Chi2, chr_all_ratio, avg_hard_mp_AA_len, avg_hard_mp_AA_len_relative, avg_mp_pango_date_index_int_50, avg_mp_pango_date_string_50, avg_collection_date_index_int, avg_collection_date_string, est_infx_len_at_collection, pango50_date_index_percentiles[1], pango50_date_index_percentiles[2], pango50_date_index_percentiles[3], pango50_date_index_percentiles[4], pango50_date_index_percentiles[5], pango50_date_index_percentiles[6], pango50_date_index_percentiles[7], pango50_date_string_percentiles[1], pango50_date_string_percentiles[2], pango50_date_string_percentiles[3], pango50_date_string_percentiles[4], pango50_date_string_percentiles[5], pango50_date_string_percentiles[6], pango50_date_string_percentiles[7], collection_date_index_percentiles[1], collection_date_index_percentiles[2], collection_date_index_percentiles[3], collection_date_index_percentiles[4], collection_date_index_percentiles[5], collection_date_index_percentiles[6], collection_date_index_percentiles[7], collection_date_string_percentiles[1], collection_date_string_percentiles[2], collection_date_string_percentiles[3], collection_date_string_percentiles[4], collection_date_string_percentiles[5], collection_date_string_percentiles[6], collection_date_string_percentiles[7], cumulative_seq_round_set_pt2_sort_join))
                                if nonmulti0__multi1 == 1
                                    push!(pt1_seed_mut_set, pt1_seed_mut)
                                    push!(df_mp_META, (pt1_seed_mut, mp_total_mut_ct, mp_tot_region_ct, region_total_mut_ct, region, mut, qual_mut_ct, EPCI_mut_ct, qual_seq_ct, EPCI_pct, MP_pct, non_MP_pct, pvFish, log10pvFISH, fold_incr, Chi2, chr_all_ratio, avg_hard_mp_AA_len, avg_hard_mp_AA_len_relative, avg_mp_pango_date_index_int_50, avg_mp_pango_date_string_50, avg_collection_date_index_int, avg_collection_date_string, est_infx_len_at_collection, pango50_date_index_percentiles[1], pango50_date_index_percentiles[2], pango50_date_index_percentiles[3], pango50_date_index_percentiles[4], pango50_date_index_percentiles[5], pango50_date_index_percentiles[6], pango50_date_index_percentiles[7], pango50_date_string_percentiles[1], pango50_date_string_percentiles[2], pango50_date_string_percentiles[3], pango50_date_string_percentiles[4], pango50_date_string_percentiles[5], pango50_date_string_percentiles[6], pango50_date_string_percentiles[7], collection_date_index_percentiles[1], collection_date_index_percentiles[2], collection_date_index_percentiles[3], collection_date_index_percentiles[4], collection_date_index_percentiles[5], collection_date_index_percentiles[6], collection_date_index_percentiles[7], collection_date_string_percentiles[1], collection_date_string_percentiles[2], collection_date_string_percentiles[3], collection_date_string_percentiles[4], collection_date_string_percentiles[5], collection_date_string_percentiles[6], collection_date_string_percentiles[7], cumulative_seq_round_set_pt2_sort_join))
                                    push!(df_mp_META_groups_dict[pt1_seed_mut], (pt1_seed_mut, mp_total_mut_ct, mp_tot_region_ct, region_total_mut_ct, region, mut, qual_mut_ct, EPCI_mut_ct, qual_seq_ct, EPCI_pct, MP_pct, non_MP_pct, pvFish, log10pvFISH, fold_incr, Chi2, chr_all_ratio, avg_hard_mp_AA_len, avg_hard_mp_AA_len_relative, avg_mp_pango_date_index_int_50, avg_mp_pango_date_string_50, avg_collection_date_index_int, avg_collection_date_string, est_infx_len_at_collection, pango50_date_index_percentiles[1], pango50_date_index_percentiles[2], pango50_date_index_percentiles[3], pango50_date_index_percentiles[4], pango50_date_index_percentiles[5], pango50_date_index_percentiles[6], pango50_date_index_percentiles[7], pango50_date_string_percentiles[1], pango50_date_string_percentiles[2], pango50_date_string_percentiles[3], pango50_date_string_percentiles[4], pango50_date_string_percentiles[5], pango50_date_string_percentiles[6], pango50_date_string_percentiles[7], collection_date_index_percentiles[1], collection_date_index_percentiles[2], collection_date_index_percentiles[3], collection_date_index_percentiles[4], collection_date_index_percentiles[5], collection_date_index_percentiles[6], collection_date_index_percentiles[7], collection_date_string_percentiles[1], collection_date_string_percentiles[2], collection_date_string_percentiles[3], collection_date_string_percentiles[4], collection_date_string_percentiles[5], collection_date_string_percentiles[6], collection_date_string_percentiles[7], cumulative_seq_round_set_pt2_sort_join))
                                    if unique == 1
                                        push!(df_mp_unique, (pt1_seed_mut, mp_total_mut_ct, mp_tot_region_ct, region_total_mut_ct, region, mut, qual_mut_ct, EPCI_mut_ct, qual_seq_ct, EPCI_pct, MP_pct, non_MP_pct, pvFish, log10pvFISH, fold_incr, Chi2, chr_all_ratio, avg_hard_mp_AA_len, avg_hard_mp_AA_len_relative, avg_mp_pango_date_index_int_50, avg_mp_pango_date_string_50, avg_collection_date_index_int, avg_collection_date_string, est_infx_len_at_collection, pango50_date_index_percentiles[1], pango50_date_index_percentiles[2], pango50_date_index_percentiles[3], pango50_date_index_percentiles[4], pango50_date_index_percentiles[5], pango50_date_index_percentiles[6], pango50_date_index_percentiles[7], pango50_date_string_percentiles[1], pango50_date_string_percentiles[2], pango50_date_string_percentiles[3], pango50_date_string_percentiles[4], pango50_date_string_percentiles[5], pango50_date_string_percentiles[6], pango50_date_string_percentiles[7], collection_date_index_percentiles[1], collection_date_index_percentiles[2], collection_date_index_percentiles[3], collection_date_index_percentiles[4], collection_date_index_percentiles[5], collection_date_index_percentiles[6], collection_date_index_percentiles[7], collection_date_string_percentiles[1], collection_date_string_percentiles[2], collection_date_string_percentiles[3], collection_date_string_percentiles[4], collection_date_string_percentiles[5], collection_date_string_percentiles[6], collection_date_string_percentiles[7], cumulative_seq_round_set_pt2_sort_join))
                                        push!(df_mp_META_unique, (pt1_seed_mut, mp_total_mut_ct, mp_tot_region_ct, region_total_mut_ct, region, mut, qual_mut_ct, EPCI_mut_ct, qual_seq_ct, EPCI_pct, MP_pct, non_MP_pct, pvFish, log10pvFISH, fold_incr, Chi2, chr_all_ratio, avg_hard_mp_AA_len, avg_hard_mp_AA_len_relative, avg_mp_pango_date_index_int_50, avg_mp_pango_date_string_50, avg_collection_date_index_int, avg_collection_date_string, est_infx_len_at_collection, pango50_date_index_percentiles[1], pango50_date_index_percentiles[2], pango50_date_index_percentiles[3], pango50_date_index_percentiles[4], pango50_date_index_percentiles[5], pango50_date_index_percentiles[6], pango50_date_index_percentiles[7], pango50_date_string_percentiles[1], pango50_date_string_percentiles[2], pango50_date_string_percentiles[3], pango50_date_string_percentiles[4], pango50_date_string_percentiles[5], pango50_date_string_percentiles[6], pango50_date_string_percentiles[7], collection_date_index_percentiles[1], collection_date_index_percentiles[2], collection_date_index_percentiles[3], collection_date_index_percentiles[4], collection_date_index_percentiles[5], collection_date_index_percentiles[6], collection_date_index_percentiles[7], collection_date_string_percentiles[1], collection_date_string_percentiles[2], collection_date_string_percentiles[3], collection_date_string_percentiles[4], collection_date_string_percentiles[5], collection_date_string_percentiles[6], collection_date_string_percentiles[7], cumulative_seq_round_set_pt2_sort_join))
                                        push!(df_mp_META_groups_dict_unique[pt1_seed_mut], (pt1_seed_mut, mp_total_mut_ct, mp_tot_region_ct, region_total_mut_ct, region, mut, qual_mut_ct, EPCI_mut_ct, qual_seq_ct, EPCI_pct, MP_pct, non_MP_pct, pvFish, log10pvFISH, fold_incr, Chi2, chr_all_ratio, avg_hard_mp_AA_len, avg_hard_mp_AA_len_relative, avg_mp_pango_date_index_int_50, avg_mp_pango_date_string_50, avg_collection_date_index_int, avg_collection_date_string, est_infx_len_at_collection, pango50_date_index_percentiles[1], pango50_date_index_percentiles[2], pango50_date_index_percentiles[3], pango50_date_index_percentiles[4], pango50_date_index_percentiles[5], pango50_date_index_percentiles[6], pango50_date_index_percentiles[7], pango50_date_string_percentiles[1], pango50_date_string_percentiles[2], pango50_date_string_percentiles[3], pango50_date_string_percentiles[4], pango50_date_string_percentiles[5], pango50_date_string_percentiles[6], pango50_date_string_percentiles[7], collection_date_index_percentiles[1], collection_date_index_percentiles[2], collection_date_index_percentiles[3], collection_date_index_percentiles[4], collection_date_index_percentiles[5], collection_date_index_percentiles[6], collection_date_index_percentiles[7], collection_date_string_percentiles[1], collection_date_string_percentiles[2], collection_date_string_percentiles[3], collection_date_string_percentiles[4], collection_date_string_percentiles[5], collection_date_string_percentiles[6], collection_date_string_percentiles[7], cumulative_seq_round_set_pt2_sort_join))     
                                    end
                                end
                            end
                        end    
                        push!(df_mp_META_set_set, df_mp_META_set)
                    end
                end
######################################################################################################################################
#                    if length(keys(df_props_dict_pt2)) ≥ 2 && notrandom0__random1 == 0
#                        CSV.write("$(mp_folder_universal)/mp_df_CSV_TSV/df_$(mut_pattern_name)_$(date).csv", df_mp)
#                        CSV.write("$(mp_folder_universal)/mp_df_CSV_TSV/df_$(mut_pattern_name)_$(date).tsv", df_mp, delim='\t')
#                        total_mp_regions = region_ct2
#                        total_mp_mutations = length(all_muts_set_pt2)
#                        total_mp_sequences = length(cumulative_seq_round_set_pt2)
#                        push!(df_mp_stats, (mut_pattern_name, total_mp_regions, total_mp_mutations, total_mp_sequences, avg_mp_pango_date_index_int_50, avg_mp_pango_date_string_50, avg_collection_date_index_int, avg_collection_date_string, est_infx_len_at_collection, pango50_date_index_percentiles[4], pango50_date_string_percentiles[4], collection_date_index_percentiles[4], collection_date_string_percentiles[4], chr_all_ratio))
#                    end
######################################################################################################################################
############ All the sections directly below just sort the dictionaries according to their various components ########################
###################################################################################################################################### 
#                MP_seq_ct_pt2 = length(MP_seqs_pt2)
#                mut_perc_pt2 = MP_seq_ct_pt2/total_EPCI_seq_ct
#                mut_perc_pt2_rd = round(digits=1, mut_perc_pt2)
######################################################################################################################################
#### Counts the clades/Pango lineages of each qualifying sequence; ADJ dictionaries sort by the proportion of a given clade/Pango lineage that qualify
                pango_dict_for_muts_pt2 = Dict{String,Int}()
                clade_dict_for_muts_pt2 = Dict{String,Int}()
                clade_dict_for_muts_pt2_ADJ = Dict{String,Float64}()
                clade_ADJ_factor_dict_pt2 = Dict{String,Float64}()
                for clade in clade_set_complete
                    if !haskey(seq_clade_ct, clade)
                        seq_clade_ct[clade] = 0
                    end
                end
                for clade in clade_set_complete
                    if seq_clade_ct[clade] == 0
                        clade_ADJ_factor_dict_pt2[clade] = 0
                    end
                    if seq_clade_ct[clade] ≠ 0
                        clade_ADJ_factor_dict_pt2[clade] = 1000/seq_clade_ct[clade]
                    end
                    clade_dict_for_muts_pt2[clade] = 0
                    clade_dict_for_muts_pt2_ADJ[clade] = 0
                end
#############################################################################################################      
                for seq in MP_seqs_pt2
                    if haskey(pango_dict_for_muts_pt2, seq_pango[seq])
                        pango_dict_for_muts_pt2[seq_pango[seq]] += 1
                    else
                        pango_dict_for_muts_pt2[seq_pango[seq]] = 1
                    end
                    if haskey(clade_dict_for_muts_pt2, seq_clade[seq])
                        clade_dict_for_muts_pt2[seq_clade[seq]] += 1
                    else
                        clade_dict_for_muts_pt2[seq_clade[seq]] = 1
                    end
                end
#############################################################################################################
                for clade in clade_set_complete
                    if haskey(clade_dict_for_muts_pt2_ADJ, clade)
                        clade_dict_for_muts_pt2_ADJ[clade] = round(digits=1, clade_dict_for_muts_pt2[clade]*clade_ADJ_factor_dict_pt2[clade])
                    else
                        clade_dict_for_muts_pt2_ADJ[clade] = 0
                    end
                end
#############################################################################################################      
#                    pango_dict_for_muts_pt2_sort_by_ct = sort(collect(pango_dict_for_muts_pt2), by = x -> x[2], rev=true)
#                    pango_dict_for_muts_pt2_sort_by_pango = sort(collect(pango_dict_for_muts_pt2), by = x -> x[1])
#                    clade_dict_for_muts_pt2_sort_by_ct = sort(collect(clade_dict_for_muts_pt2), by = x -> x[2], rev=true)
#                    clade_dict_for_muts_pt2_sort_by_clade = sort(collect(clade_dict_for_muts_pt2), by = x -> x[1])
#                    clade_dict_for_muts_pt2_ADJ_sort_by_ct = sort(collect(clade_dict_for_muts_pt2_ADJ), by = x -> x[2], rev=true)
#                    clade_dict_for_muts_pt2_ADJ_sort_by_clade = sort(collect(clade_dict_for_muts_pt2_ADJ), by = x -> x[1])
#####################################################################################################################################
############################ Everything Below Just Prints the Results in various Ways ###############################################
#####################################################################################################################################
                mutPerc_pt2 = 100*MP_seq_ct_pt2/total_EPCI_seq_ct
                mutPerc_pt2_rd = round(digits=1, mutPerc_pt2)
                padd = 15; seq_title_pad = 24; clade_title_pad = 5
######################################################################################################################################
            else
                println("No Conditions Satisified: Something is seriously wrong with the recursion conditionals"^8)
            end
        end
        if !haskey(df_props_dict_pt2, pt2_round_ct)
            df_props_dict_pt2[pt2_round_ct] = Dict{Int, Dict{String, Tuple{Float64,Float64,Int,Int,Int,Float64,Float64,Float64,Union{Float64, String},Float64}}}()
            df_props_dict_pt2[pt2_round_ct][0] = Dict{String, Tuple{Float64,Float64,Int,Int,Int,Float64,Float64,Float64,Union{Float64, String},Float64}}()
            df_props_dict_pt2[pt2_round_ct][0][""] = (0.0, 0.0, 0, 0, 0, 0.0, 0.0, 0.0, 0.0, 0.0)
        end
        return df_props_dict_pt2[pt2_round_ct], cumulative_seq_round_set_pt2_sort_join, avg_mp_pango_date_index_int_50, avg_mp_pango_date_string_50, pango_ct_dict_sort, clade_ct_dict_sort, cladepango_ct_dict_sort
    end
    return (Dict{Int, Dict{String, Tuple{Float64,Float64,Int,Int,Int,Float64,Float64,Float64,Union{Float64, String},Float64}}}(), "", 0, "00-00-00", Pair{String,Int64}[], Pair{String,Int64}[], Pair{String,Int64}[])
end
date_now = Dates.format(now(), "yyyy_mm_dd__IMMp"); println(date_now); println("Finished."); print("\n"^1)
#############################################################################################################################
#############################################################################################################################

2026_03_08__421PM
2026_03_08__421PM
Finished.



In [21]:
### Making megaclade_EPCI_ct_dict and megaclade_EPCI_proportion_dict
date_now = Dates.format(now(), "yyyy_mm_dd__IMMp"); println(date_now)
nowtime = Dates.format(now(), "I:MM.SS_p"); println(nowtime)
x_minor_ct = 0
x_minor_dict_ct = Dict{String, Int}()
unknown_EPCI_ct = 0
pre_VOC_ct = 0
pre_VOC_dict_ct = Dict{String, Int}()
pre_VOC_vec = String[]
total_EPCI_seq_ct = length(all_unique_chr_seqs_set)
megaclade_EPCI_ct_dict = Dict{String, Int}()
megaclade_EPCI_proportion_dict = Dict{String, Float64}()
for seq in all_unique_chr_seqs_set
    complete = 0
    pango = seq_pango[seq]
    unaliased = get(pango_to_pango_unaliased_v2, pango, pango)
    if length(unaliased) ≥ 3
        if unaliased[1:3] == "XBB"
            megaclade_EPCI_ct_dict["XBB"] = get(megaclade_EPCI_ct_dict, "XBB", 0) + 1
            complete = 1
        elseif unaliased[1:3] == "XEC" || unaliased[1:3] == "XEK" || unaliased[1:3] == "XFG" || unaliased[1:3] == "XFC" || unaliased[1:3] == "XFJ"
            megaclade_EPCI_ct_dict["BA.2.86"] = get(megaclade_EPCI_ct_dict, "BA.2.86", 0) + 1
            complete = 1
        elseif unaliased[1:3] == "XCH"
            megaclade_EPCI_ct_dict["BA.5"] = get(megaclade_EPCI_ct_dict, "BA.5", 0) + 1
            complete = 1
        end
        if length(unaliased) ≥ 7
            if unaliased[1:7] == "B.1.1.7"
                megaclade_EPCI_ct_dict["B.1.1.7"] = get(megaclade_EPCI_ct_dict, "B.1.1.7", 0) + 1
                complete = 1
            elseif unaliased[1:7] == "B.1.351"
                megaclade_EPCI_ct_dict["B.1.351"] = get(megaclade_EPCI_ct_dict, "B.1.351", 0) + 1
                complete = 1
            elseif unaliased[1:7] == "B.1.617"
                megaclade_EPCI_ct_dict["B.1.617"] = get(megaclade_EPCI_ct_dict, "B.1.617", 0) + 1
                complete = 1
            end
            if length(unaliased) ≥ 9
                if unaliased == "B.1.1.529"
                    megaclade_EPCI_ct_dict["Omicron_Unknown"] = get(megaclade_EPCI_ct_dict, "Omicron_Unknown", 0) + 1
                    complete = 1
                end
                if length(unaliased) ≥ 10
                    if unaliased[1:10] == "B.1.1.28.1"
                        megaclade_EPCI_ct_dict["P.1"] = get(megaclade_EPCI_ct_dict, "P.1", 0) + 1
                        complete = 1
                    end
                    if length(unaliased) ≥ 11
                        if unaliased[1:11] == "B.1.1.529.1"
                            megaclade_EPCI_ct_dict["BA.1"] = get(megaclade_EPCI_ct_dict, "BA.1", 0) + 1
                            complete = 1
                        elseif unaliased[1:11] == "B.1.1.529.2"
                            megaclade_EPCI_ct_dict["BA.2"] = get(megaclade_EPCI_ct_dict, "BA.2", 0) + 1
                            complete = 1
                        elseif unaliased[1:11] == "B.1.1.529.4"
                            megaclade_EPCI_ct_dict["BA.5"] = get(megaclade_EPCI_ct_dict, "BA.5", 0) + 1
                            complete = 1
                        elseif unaliased[1:11] == "B.1.1.529.5"
                            megaclade_EPCI_ct_dict["BA.5"] = get(megaclade_EPCI_ct_dict, "BA.5", 0) + 1
                            complete = 1
                        end
                        if length(unaliased) ≥ 14
                            if unaliased[1:14] == "B.1.1.529.2.86"
                                megaclade_EPCI_ct_dict["BA.2.86"] = get(megaclade_EPCI_ct_dict, "BA.2.86", 0) + 1
                                megaclade_EPCI_ct_dict["BA.2"] = megaclade_EPCI_ct_dict["BA.2"] - 1
                                complete = 1
                            end
                        end
                    end
                end
            end
        end
    end
    if complete == 0
        if unaliased[1] == 'X'
            x_minor_ct += 1
            x_minor_dict_ct[unaliased] = get(x_minor_dict_ct, unaliased, 0) + 1
            megaclade_EPCI_ct_dict["Minor_Recombinant"] = get(megaclade_EPCI_ct_dict, "Minor_Recombinant", 0) + 1
            complete = 1
        end
        if unaliased[1] == 'A' || unaliased[1] == 'B'
            pre_VOC_ct += 1
            pre_VOC_dict_ct[unaliased] = get(pre_VOC_dict_ct, unaliased, 0) + 1
#            println(pango)
            megaclade_EPCI_ct_dict["pre_VOC"] = get(megaclade_EPCI_ct_dict, "pre_VOC", 0) + 1
            complete = 1
        end
    end
    if complete == 0
        println("Messed Up Pango = $(pango)")
        unknown_EPCI_ct += 1
    end
end
######################################################################################################################
for (megaclade, count) in megaclade_EPCI_ct_dict
    megaclade_EPCI_proportion_dict[megaclade] = count/total_EPCI_seq_ct
end
######################################################################################################################
print("\n"^2)
println("unknown_EPCI_ct = $(unknown_EPCI_ct)")
println("x_minor_ct = $(x_minor_ct)")
println("pre_VOC_ct =$(pre_VOC_ct)")
print("\n"^2)
x_minor_dict_ct_sort = sort(collect(x_minor_dict_ct), by = x -> x[1])
for minorX___ct in x_minor_dict_ct_sort
    minorX_pad = rpad(minorX___ct[1], 12)
    count_pad = lpad(minorX___ct[2], 2)
#    println("$(minorX_pad) = $(count_pad)")
end
print("\n"^2)   
pre_VOC_dict_ct_sort = sort(collect(pre_VOC_dict_ct), by = x -> x[1])
for preVOC___ct in pre_VOC_dict_ct_sort
    preVOC_pad = rpad(preVOC___ct[1], 12)
    count_pad = lpad(preVOC___ct[2], 2)
#    println("$(preVOC_pad) = $(count_pad)")
end
print("\n"^2) 
######################################################################################################################
megaclade_EPCI_ct_dict_sort = sort(collect(megaclade_EPCI_ct_dict), by = x -> x[2], rev=true)
for megaclade____count in megaclade_EPCI_ct_dict_sort
    megaclade = megaclade____count[1]
    count = megaclade____count[2]
    println("$(megaclade) = $(count)")
end
print("\n"^2)
date_now = Dates.format(now(), "yyyy_mm_dd__IMMp"); println(date_now)
nowtime = Dates.format(now(), "I:MM.SS_p"); println(nowtime)
print("\n"^1)
######################################################################################################################
######################################################################################################################

2026_03_04__1045AM
10:45.09_AM



unknown_EPCI_ct = 0
x_minor_ct = 28
pre_VOC_ct =292



XAC          =  1
XAE          =  1
XAF          =  1
XAH          =  1
XAP          =  1
XAZ          =  1
XBC.1        =  2
XBC.1.1      =  1
XBC.1.6.1    =  1
XBF          =  2
XBF.2        =  1
XBF.3        =  2
XBF.9        =  1
XBK          =  1
XCT.1        =  1
XDD          =  2
XDV.1.12     =  1
XDV.1.5.1.1.8.1.1 =  1
XDV.1.5.1.1.8.1.17 =  2
XDV.1.5.1.1.8.1.8.1 =  1
XM           =  1
XQ           =  2



A            =  1
A.2          =  3
B            =  4
B.1          = 45
B.1.1        = 36
B.1.1.1      =  2
B.1.1.1.14   =  1
B.1.1.1.37   =  5
B.1.1.176    =  2
B.1.1.192    =  1
B.1.1.214    =  2
B.1.1.232    =  1
B.1.1.241    =  2
B.1.1.25     =  2
B.1.1.273    =  1
B.1.1.277    =  2
B.1.1.28     =  3
B.1.1.28.2   =  1
B.1.1.284    =  2
B.1.1.294    =  1
B.1.1.306.1  =  1
B.1.1.316.1  =  1
B.1.1.317    =  1
B.1.1.318    =  1
B.1.1.33     =  1
B.1.1.33.4   =  1
B.1.1.351    =  1
B.1.1.37

In [108]:
### NEW (2025-11-18) v2 for candidate muts: only Double_N_ORF9b_muts & artifactual_private_muts excluded
## Reminder: all_excluded_muts is a global variable defined in RBD cell & takes on its value depending on the values of: 
#                                                        sub_0__posonly_1 = 0
#                                                        normal_0__spikeonly_1 = 0
#                                                        noBAL_0__withBAL_1 = 1
date_now = Dates.format(now(), "yyyy_mm_dd__IMMp"); println(date_now)
nowtime = Dates.format(now(), "I:MM.SS_p"); println(nowtime)
min_ct = 3
max_ct = 999
mut_cand_ct = 0
spike_muts_to_exclude = list_to_strings_set("S:D145Y, S:I95T, S:I212L, S:V67A, S:A27S")
candidate_test_muts = Set{String}()
for (mut, ct) in AA_muts_ct_no_dels_universal
    if !(mut in all_excluded_muts) && !(mut in spike_muts_to_exclude) && ct ≤ max_ct && ct ≥ min_ct
        push!(candidate_test_muts, mut)
    end
end
print("\n"^2)
cand_length = length(candidate_test_muts)
println("Total Number of Test Mut Candidates = $cand_length")
print("\n"^3)
for mut in candidate_test_muts
    print("$(mut), ")
end
print("\n"^2)
nowtime = Dates.format(now(), "I:MM.SS_p"); println(nowtime)
println("Finished")
print("\n"^2)

2026_03_08__421PM
4:21.14_PM


Total Number of Test Mut Candidates = 3747



ORF1a:Y621C, ORF1b:I1227M, S:T76I, S:R214S, ORF7a:N43T, S:T22N, ORF1b:A2222V, ORF1a:A1306S, ORF1a:A2710V, ORF1a:V2133I, ORF1a:V111M, ORF1a:L747F, S:E154Q, ORF1b:P1727S, ORF1b:L314P, ORF1a:S318L, S:S98F, S:N185S, ORF1b:V2073L, ORF1a:A3620V, ORF7b:E3K, S:H681Y, ORF1a:S3149F, S:K187E, ORF1a:G3072C, ORF8:A55V, M:K162R, ORF1a:T3058I, ORF1a:S723P, ORF1b:A1219S, ORF8:I74T, ORF1a:R398C, ORF1a:K2761R, S:I1081V, S:V952F, ORF1a:V1570L, ORF1a:L16F, S:W152S, ORF1a:D1157N, S:I1183T, ORF1a:L1368I, ORF1b:P1325S, ORF1a:P193S, ORF1a:V2430I, S:S172F, S:K1266R, M:G6V, S:T632I, S:V1164I, N:G124D, S:S691P, S:T618I, M:S94N, ORF1a:P1921S, N:K387R, ORF1a:A520V, N:L230F, ORF1a:A2325V, S:V1122A, ORF9b:D16N, ORF1a:R2159Q, ORF1b:I2303V, M:S4L, ORF1a:L1853F, S:D737E, S:A575S, S:N657D, ORF1b:P85L, ORF3a:L127F, N:R413L, ORF1a:I3341V, M:I8S, ORF1a:K120N, N:A252S, ORF1a:A3726T, ORF9b:M78T, ORF7a:L96R, ORF1a:L2781F, M:A98S, ORF1b:L314F, ORF1b:L

In [ ]:
###  Testing ALL EPCI mutations for mutational patterns         Runtime = 1 hr, 3 min, 18.75 sec, 2026-02-21   ########
date_now = Dates.format(now(), "yyyy_mm_dd__IMMp"); print("\n"^1); println(date_now)
nowtime = Dates.format(now(), "I:MM.SS_p"); println(nowtime)
date = Dates.format(today(), "yyyy_mm_dd")
date_hour = Dates.format(now(), "yyyy_mm_dd_Ip")
start = time()
############################################################################################################################################################################
#global pango_ct_dict
#global pango_ct_dict_sort
global fake_dict_num_ct = 0
global total_mp_ct = 0
global mp_index
#global df_rand_corr_muts_META = DataFrame(
#    Index = Int[], Seed_Mut = String[], 
#    Region1 = String[], mutCt1 = Int[], chrMutCt1 = Int[], mutPct1 = Float64[], qualSeqPct1 = Float64[], log10fisherPval1 = Float64[], chrAllRatio1 = Float64[],   
#    Region2 = String[], mutCt2 = Int[], chrMutCt2 = Int[], mutPct2 = Float64[], qualSeqPct2 = Float64[], log10fisherPval2 = Float64[], chrAllRatio2 = Float64[], 
#    Region3 = String[], mutCt3 = Int[], chrMutCt3 = Int[], mutPct3 = Float64[], qualSeqPct3 = Float64[], log10fisherPval3 = Float64[], chrAllRatio3 = Float64[], 
#    Region4 = String[], mutCt4 = Int[], chrMutCt4 = Int[], mutPct4 = Float64[], qualSeqPct4 = Float64[], log10fisherPval4 = Float64[], chrAllRatio4 = Float64[], 
#    Region5 = String[], mutCt5 = Int[], chrMutCt5 = Int[], mutPct5 = Float64[], qualSeqPct5 = Float64[], log10fisherPval5 = Float64[], chrAllRatio5 = Float64[], 
#    Region6 = String[], mutCt6 = Int[], chrMutCt6 = Int[], mutPct6 = Float64[], qualSeqPct6 = Float64[], log10fisherPval6 = Float64[], chrAllRatio6 = Float64[], 
#    Region7 = String[], mutCt7 = Int[], chrMutCt7 = Int[], mutPct7 = Float64[], qualSeqPct7 = Float64[], log10fisherPval7 = Float64[], chrAllRatio7 = Float64[], 
#    Region8 = String[], mutCt8 = Int[], chrMutCt8 = Int[], mutPct8 = Float64[], qualSeqPct8 = Float64[], log10fisherPval8 = Float64[], chrAllRatio8 = Float64[], 
#    Region9 = String[], mutCt9 = Int[], chrMutCt9 = Int[], mutPct9 = Float64[], qualSeqPct9 = Float64[], log10fisherPval9 = Float64[], chrAllRatio9 = Float64[], 
#    Region10 = String[],mutCt10 = Int[],chrMutCt10 = Int[],mutPct10 = Float64[],qualSeqPct10 = Float64[],log10fisherPval10 = Float64[],chrAllRatio10 = Float64[],
#    Region11 = String[],mutCt11 = Int[],chrMutCt11 = Int[],mutPct11 = Float64[],qualSeqPct11 = Float64[],log10fisherPval11 = Float64[],chrAllRatio11 = Float64[],   
#    Region12 = String[],mutCt12 = Int[],chrMutCt12 = Int[],mutPct12 = Float64[],qualSeqPct12 = Float64[],log10fisherPval12 = Float64[],chrAllRatio12 = Float64[],
#    Region13 = String[],mutCt13 = Int[],chrMutCt13 = Int[],mutPct13 = Float64[],qualSeqPct13 = Float64[],log10fisherPval13 = Float64[],chrAllRatio13 = Float64[],
#    Region14 = String[],mutCt14 = Int[],chrMutCt14 = Int[],mutPct14 = Float64[],qualSeqPct14 = Float64[],log10fisherPval14 = Float64[],chrAllRatio14 = Float64[],
#    Region15 = String[],mutCt15 = Int[],chrMutCt15 = Int[],mutPct15 = Float64[],qualSeqPct15 = Float64[],log10fisherPval15 = Float64[],chrAllRatio15 = Float64[],
#    Region16 = String[],mutCt16 = Int[],chrMutCt16 = Int[],mutPct16 = Float64[],qualSeqPct16 = Float64[],log10fisherPval16 = Float64[],chrAllRatio16 = Float64[],
#    Region17 = String[],mutCt17 = Int[],chrMutCt17 = Int[],mutPct17 = Float64[],qualSeqPct17 = Float64[],log10fisherPval17 = Float64[],chrAllRatio17 = Float64[],
#    Region18 = String[],mutCt18 = Int[],chrMutCt18 = Int[],mutPct18 = Float64[],qualSeqPct18 = Float64[],log10fisherPval18 = Float64[],chrAllRatio18 = Float64[],
#    Region19 = String[],mutCt19 = Int[],chrMutCt19 = Int[],mutPct19 = Float64[],qualSeqPct19 = Float64[],log10fisherPval19 = Float64[],chrAllRatio19 = Float64[],
#    Region20 = String[],mutCt20 = Int[],chrMutCt20 = Int[],mutPct20 = Float64[],qualSeqPct20 = Float64[],log10fisherPval20 = Float64[],chrAllRatio20 = Float64[],
#    ALL_mp_Sequences = String[])
#global df_mp_META_META = DataFrame(Index = Int[], Mutation = String[], MP_Mut_ct = Float64[], EPCI_Mut_ct = Int[], Adjusted_MP_seq_ct = Int[], EPCI_pct = Float64[], MP_pct = Float64[], non_MP_pct = Float64[], Fishers_Exact_Test_pval = Float64[], log10pvFISH = Float64[], Fold_Incr = Union{Float64, String}[], Chi2 = Float64[], EPCI_HQCS_Ratio = Float64[])
############################################################################################################################################################################
global rand_folder = "$(mp_folder_universal)_RANDOM"
mkpath(rand_folder)
global pango_counts_folder_rand_csv  = "$(rand_folder)/pango_counts_csv"
mkpath(pango_counts_folder_rand_csv)
#global pango_counts_folder_rand_txt  = "$(rand_folder)/pango_counts_txt"
#mkpath(pango_counts_folder_rand_txt)
#global rand_folder_ind = "$(mp_folder_universal)_RANDOM/specific_mut_stats"
#mkpath(rand_folder_ind)
############################################################################################################################################################################
mut_gene_Dict = Dict{String, Int}("ORF1a"=>1, "ORF1b"=>2, "S"=>3, "E"=>4, "M"=>5, "N"=>6, "ORF3a"=>7, "ORF6"=>8, "ORF7a"=>9, "ORF7b"=>10, "ORF8"=>11, "ORF9b"=>12)
#################################################################
pure_spike_ct = 0
cand_length = length(candidate_test_muts)
println("Total Number of Candidate Test Mutations = $(cand_length)"); print("\n"^1)
random_sample_size = cand_length
min_mut_ct = 5
max_mut_ct = 999
tot_tested_muts = random_sample_size
total_EPCI_seq_ct = length(all_unique_chr_seqs_set)
############################################################################################################################################################################
    min_match_ct = 2
    pt1_ct_mltplr = 0.4
    pt2_ct_mltplr = 0.4
    nonmulti0__multi1 = 1        
    notrandom0__random1 = 1
    plus_minus = 5
    if normal_0__spikeonly_1 == 1
        plus_minus = 2
    end
    ind_or_grp = "grp"
    min_grp_fish = 2.0
    min_log_pv_fish = 5.0
    mp_masked_muts = Set{String}()
### see `starter` below for sel_muts
    seq_factor_OFF_HALF_ON__0_1_2 = 1
    if normal_0__spikeonly_1 == 1
        seq_factor_OFF_HALF_ON__0_1_2 = 1
    end
#################################################################################
    seqfac_dict = Dict(0=>"OFF", 1=>"ON", 2=>"FULLBLAST")
    seqfac_dict_spike = Dict(0=>"OFF", 1=>"RBM", 2=>"RBD", 3=>"SpikeNonRBD")
    seqfac = seqfac_dict[seq_factor_OFF_HALF_ON__0_1_2]
    SpikeFac = seqfac_dict_spike[nonspike_seqfactor__0off__1RBM__2RBD__3SpikeNonRBD]
############################################################################################################################################################################
nowtime1 = Dates.format(now(), "I:MM.SS_p") #; println(nowtime1)
time1 = time()
###############################
#open("$(rand_folder)/rand_muts_mp_$(random_sample_size)mutsTested_plusminus$(plus_minus)_minChi$(min_grp_fish)_minLogPVfish$(min_log_pv_fish)_seqfac$(seqfac)_$(date).txt", "w") do g
#    println(g, "$(random_sample_size) Random Muts in ≥$(min_mut_ct) && ≤$(max_mut_ct) Sequences in Chronic List, Tested for Correlated Muts")
#end
global df_mp_stats = DataFrame(Mut_Pattern_or_Seed_Mut = String[], Total_Regions = Int[], Total_Mutations = Int[], Total_Sequences = Int[], Pango50_DateIndex_Avg = Int[], Pango50_Date_Avg = String[], Collection_DateIndex_Avg = Int[], Collection_Date_Avg = String[], Est_Infx_Length = Int[], Pango50_DateIndex_Q2 = Int[], Pango50_Date_Q2 = String[], Collection_DateIndex_Q2 = Int[], Collection_Date_Q2 = String[], EPCI_HQCS_Ratio = Float64[])
global df_mp = DataFrame(SeedMut = String[], MP_Tot_Mut_ct = Int[], MP_Tot_Reg_ct = Int[], Reg_Tot_Mut_ct = Int[], MP_Region_Number = Int[], Mutation = String[], MP_Mut_ct = Float64[], EPCI_Mut_ct = Int[], Adjusted_MP_seq_ct = Int[], EPCI_pct = Float64[], MP_pct = Float64[], non_MP_pct = Float64[], Fishers_Exact_Test_pval = Float64[], log10pvFISH = Float64[], Fold_Incr = Union{Float64, String}[], Chi2 = Float64[], EPCI_HQCS_Ratio = Float64[], avg_NonRBD_AAct = Float64[], avg_nonRBD_rel = Float64[], 
    PangoDateIndex50_Avg = Int[], PangoDateStr50_Avg = String[], CollDateIndex_Avg = Int[], CollDateString_Avg = String[], Est_Infx_Length = Int[], 
    PangoDateIndex50_min = Int[], PangoDateIndex50_p10 = Int[], PangoDateIndex50_Q1 = Int[], PangoDateIndex50_Q2 = Int[], PangoDateIndex50_Q3 = Int[], PangoDateIndex50_p90 = Int[], PangoDateIndex50_max = Int[], 
    PangoDateStr50_min = String[], PangoDateStr50_p10 = String[], PangoDateStr50_Q1 = String[], PangoDateStr50_Q2 = String[], PangoDateStr50_Q3 = String[], PangoDateStr50_p90 = String[], PangoDateStr50_max = String[], 
    CollDateIndex_min = Int[], CollDateIndex_p10 = Int[], CollDateIndex_Q1 = Int[], CollDateIndex_Q2 = Int[], CollDateIndex_Q3 = Int[], CollDateIndex_p90 = Int[], CollDateIndex_max = Int[], 
    CollDateString_min = String[], CollDateString_p10 = String[], CollDateString_Q1 = String[], CollDateString_Q2 = String[], CollDateString_Q3 = String[], CollDateString_p90 = String[], CollDateString_max = String[], ALL_mp_Sequences = String[])
global df_mp_unique = DataFrame(SeedMut = String[], MP_Tot_Mut_ct = Int[], MP_Tot_Reg_ct = Int[], Reg_Tot_Mut_ct = Int[], MP_Region_Number = Int[], Mutation = String[], MP_Mut_ct = Float64[], EPCI_Mut_ct = Int[], Adjusted_MP_seq_ct = Int[], EPCI_pct = Float64[], MP_pct = Float64[], non_MP_pct = Float64[], Fishers_Exact_Test_pval = Float64[], log10pvFISH = Float64[], Fold_Incr = Union{Float64, String}[], Chi2 = Float64[], EPCI_HQCS_Ratio = Float64[], avg_NonRBD_AAct = Float64[], avg_nonRBD_rel = Float64[], 
    PangoDateIndex50_Avg = Int[], PangoDateStr50_Avg = String[], CollDateIndex_Avg = Int[], CollDateString_Avg = String[], Est_Infx_Length = Int[], 
    PangoDateIndex50_min = Int[], PangoDateIndex50_p10 = Int[], PangoDateIndex50_Q1 = Int[], PangoDateIndex50_Q2 = Int[], PangoDateIndex50_Q3 = Int[], PangoDateIndex50_p90 = Int[], PangoDateIndex50_max = Int[], 
    PangoDateStr50_min = String[], PangoDateStr50_p10 = String[], PangoDateStr50_Q1 = String[], PangoDateStr50_Q2 = String[], PangoDateStr50_Q3 = String[], PangoDateStr50_p90 = String[], PangoDateStr50_max = String[], 
    CollDateIndex_min = Int[], CollDateIndex_p10 = Int[], CollDateIndex_Q1 = Int[], CollDateIndex_Q2 = Int[], CollDateIndex_Q3 = Int[], CollDateIndex_p90 = Int[], CollDateIndex_max = Int[], 
    CollDateString_min = String[], CollDateString_p10 = String[], CollDateString_Q1 = String[], CollDateString_Q2 = String[], CollDateString_Q3 = String[], CollDateString_p90 = String[], CollDateString_max = String[], ALL_mp_Sequences = String[])
# pt1_seed_mut, mp_total_mut_ct, mp_tot_region_ct, region_total_mut_ct, mut, qual_mut_ct, EPCI_mut_ct, 
global df_mp_META = DataFrame(SeedMut = String[], MP_Tot_Mut_ct = Int[], MP_Tot_Reg_ct = Int[], Reg_Tot_Mut_ct = Int[], MP_Region_Number = Int[], Mutation = String[], MP_Mut_ct = Float64[], EPCI_Mut_ct = Int[], Adjusted_MP_seq_ct = Int[], EPCI_pct = Float64[], MP_pct = Float64[], non_MP_pct = Float64[], Fishers_Exact_Test_pval = Float64[], log10pvFISH = Float64[], Fold_Incr = Union{Float64, String}[], Chi2 = Float64[], EPCI_HQCS_Ratio = Float64[], avg_NonRBD_AAct = Float64[], avg_nonRBD_rel = Float64[], PangoDateIndex50_Avg = Int[], PangoDateStr50_Avg = String[], CollDateIndex_Avg = Int[], CollDateString_Avg = String[], Est_Infx_Length = Int[], PangoDateIndex50_min = Int[], PangoDateIndex50_p10 = Int[], PangoDateIndex50_Q1 = Int[], PangoDateIndex50_Q2 = Int[], PangoDateIndex50_Q3 = Int[], PangoDateIndex50_p90 = Int[], PangoDateIndex50_max = Int[], PangoDateStr50_min = String[], PangoDateStr50_p10 = String[], PangoDateStr50_Q1 = String[], PangoDateStr50_Q2 = String[], PangoDateStr50_Q3 = String[], PangoDateStr50_p90 = String[], PangoDateStr50_max = String[], CollDateIndex_min = Int[], CollDateIndex_p10 = Int[], CollDateIndex_Q1 = Int[], CollDateIndex_Q2 = Int[], CollDateIndex_Q3 = Int[], CollDateIndex_p90 = Int[], CollDateIndex_max = Int[], CollDateString_min = String[], CollDateString_p10 = String[], CollDateString_Q1 = String[], CollDateString_Q2 = String[], CollDateString_Q3 = String[], CollDateString_p90 = String[], CollDateString_max = String[], ALL_mp_Sequences = String[])
global df_mp_META_sort = DataFrame(SeedMut = String[], MP_Tot_Mut_ct = Int[], MP_Tot_Reg_ct = Int[], Reg_Tot_Mut_ct = Int[], MP_Region_Number = Int[], Mutation = String[], MP_Mut_ct = Float64[], EPCI_Mut_ct = Int[], Adjusted_MP_seq_ct = Int[], EPCI_pct = Float64[], MP_pct = Float64[], non_MP_pct = Float64[], Fishers_Exact_Test_pval = Float64[], log10pvFISH = Float64[], Fold_Incr = Union{Float64, String}[], Chi2 = Float64[], EPCI_HQCS_Ratio = Float64[], avg_NonRBD_AAct = Float64[], avg_nonRBD_rel = Float64[], PangoDateIndex50_Avg = Int[], PangoDateStr50_Avg = String[], CollDateIndex_Avg = Int[], CollDateString_Avg = String[], Est_Infx_Length = Int[], PangoDateIndex50_min = Int[], PangoDateIndex50_p10 = Int[], PangoDateIndex50_Q1 = Int[], PangoDateIndex50_Q2 = Int[], PangoDateIndex50_Q3 = Int[], PangoDateIndex50_p90 = Int[], PangoDateIndex50_max = Int[], PangoDateStr50_min = String[], PangoDateStr50_p10 = String[], PangoDateStr50_Q1 = String[], PangoDateStr50_Q2 = String[], PangoDateStr50_Q3 = String[], PangoDateStr50_p90 = String[], PangoDateStr50_max = String[], CollDateIndex_min = Int[], CollDateIndex_p10 = Int[], CollDateIndex_Q1 = Int[], CollDateIndex_Q2 = Int[], CollDateIndex_Q3 = Int[], CollDateIndex_p90 = Int[], CollDateIndex_max = Int[], CollDateString_min = String[], CollDateString_p10 = String[], CollDateString_Q1 = String[], CollDateString_Q2 = String[], CollDateString_Q3 = String[], CollDateString_p90 = String[], CollDateString_max = String[], ALL_mp_Sequences = String[])
global df_mp_META_META
global df_mp_META_unique = DataFrame(SeedMut = String[], MP_Tot_Mut_ct = Int[], MP_Tot_Reg_ct = Int[], Reg_Tot_Mut_ct = Int[], MP_Region_Number = Int[], Mutation = String[], MP_Mut_ct = Float64[], EPCI_Mut_ct = Int[], Adjusted_MP_seq_ct = Int[], EPCI_pct = Float64[], MP_pct = Float64[], non_MP_pct = Float64[], Fishers_Exact_Test_pval = Float64[], log10pvFISH = Float64[], Fold_Incr = Union{Float64, String}[], Chi2 = Float64[], EPCI_HQCS_Ratio = Float64[], avg_NonRBD_AAct = Float64[], avg_nonRBD_rel = Float64[], PangoDateIndex50_Avg = Int[], PangoDateStr50_Avg = String[], CollDateIndex_Avg = Int[], CollDateString_Avg = String[], Est_Infx_Length = Int[], PangoDateIndex50_min = Int[], PangoDateIndex50_p10 = Int[], PangoDateIndex50_Q1 = Int[], PangoDateIndex50_Q2 = Int[], PangoDateIndex50_Q3 = Int[], PangoDateIndex50_p90 = Int[], PangoDateIndex50_max = Int[], PangoDateStr50_min = String[], PangoDateStr50_p10 = String[], PangoDateStr50_Q1 = String[], PangoDateStr50_Q2 = String[], PangoDateStr50_Q3 = String[], PangoDateStr50_p90 = String[], PangoDateStr50_max = String[], CollDateIndex_min = Int[], CollDateIndex_p10 = Int[], CollDateIndex_Q1 = Int[], CollDateIndex_Q2 = Int[], CollDateIndex_Q3 = Int[], CollDateIndex_p90 = Int[], CollDateIndex_max = Int[], CollDateString_min = String[], CollDateString_p10 = String[], CollDateString_Q1 = String[], CollDateString_Q2 = String[], CollDateString_Q3 = String[], CollDateString_p90 = String[], CollDateString_max = String[], ALL_mp_Sequences = String[])
global df_mp_META_sort_unique = DataFrame(SeedMut = String[], MP_Tot_Mut_ct = Int[], MP_Tot_Reg_ct = Int[], Reg_Tot_Mut_ct = Int[], MP_Region_Number = Int[], Mutation = String[], MP_Mut_ct = Float64[], EPCI_Mut_ct = Int[], Adjusted_MP_seq_ct = Int[], EPCI_pct = Float64[], MP_pct = Float64[], non_MP_pct = Float64[], Fishers_Exact_Test_pval = Float64[], log10pvFISH = Float64[], Fold_Incr = Union{Float64, String}[], Chi2 = Float64[], EPCI_HQCS_Ratio = Float64[], avg_NonRBD_AAct = Float64[], avg_nonRBD_rel = Float64[], PangoDateIndex50_Avg = Int[], PangoDateStr50_Avg = String[], CollDateIndex_Avg = Int[], CollDateString_Avg = String[], Est_Infx_Length = Int[], PangoDateIndex50_min = Int[], PangoDateIndex50_p10 = Int[], PangoDateIndex50_Q1 = Int[], PangoDateIndex50_Q2 = Int[], PangoDateIndex50_Q3 = Int[], PangoDateIndex50_p90 = Int[], PangoDateIndex50_max = Int[], PangoDateStr50_min = String[], PangoDateStr50_p10 = String[], PangoDateStr50_Q1 = String[], PangoDateStr50_Q2 = String[], PangoDateStr50_Q3 = String[], PangoDateStr50_p90 = String[], PangoDateStr50_max = String[], CollDateIndex_min = Int[], CollDateIndex_p10 = Int[], CollDateIndex_Q1 = Int[], CollDateIndex_Q2 = Int[], CollDateIndex_Q3 = Int[], CollDateIndex_p90 = Int[], CollDateIndex_max = Int[], CollDateString_min = String[], CollDateString_p10 = String[], CollDateString_Q1 = String[], CollDateString_Q2 = String[], CollDateString_Q3 = String[], CollDateString_p90 = String[], CollDateString_max = String[], ALL_mp_Sequences = String[])
#global df_rand_corr_muts_META
global df_mp_META_set
global df_mp_META_set_set
global df_mp_META_seedmut_set_dict = Dict{String, Set{String}}()
global pt1_seed_mut_set
################################ Put in later: avgMutPerSeq1 = Float64[] —— requires changing main fx so that only the qualifying seqs with this mut are included. Likely need it in in props/props_dict2.
#df_rand_corr_muts = DataFrame(
#    Index = Int[], Seed_Mut = String[], 
#    Region1 = String[], mutCt1 = Int[], chrMutCt1 = Int[], mutPct1 = Float64[], qualSeqPct1 = Float64[], log10fisherPval1 = Float64[], chrAllRatio1 = Float64[],   
#    Region2 = String[], mutCt2 = Int[], chrMutCt2 = Int[], mutPct2 = Float64[], qualSeqPct2 = Float64[], log10fisherPval2 = Float64[], chrAllRatio2 = Float64[], 
#    Region3 = String[], mutCt3 = Int[], chrMutCt3 = Int[], mutPct3 = Float64[], qualSeqPct3 = Float64[], log10fisherPval3 = Float64[], chrAllRatio3 = Float64[], 
#    Region4 = String[], mutCt4 = Int[], chrMutCt4 = Int[], mutPct4 = Float64[], qualSeqPct4 = Float64[], log10fisherPval4 = Float64[], chrAllRatio4 = Float64[], 
#    Region5 = String[], mutCt5 = Int[], chrMutCt5 = Int[], mutPct5 = Float64[], qualSeqPct5 = Float64[], log10fisherPval5 = Float64[], chrAllRatio5 = Float64[], 
#    Region6 = String[], mutCt6 = Int[], chrMutCt6 = Int[], mutPct6 = Float64[], qualSeqPct6 = Float64[], log10fisherPval6 = Float64[], chrAllRatio6 = Float64[], 
#    Region7 = String[], mutCt7 = Int[], chrMutCt7 = Int[], mutPct7 = Float64[], qualSeqPct7 = Float64[], log10fisherPval7 = Float64[], chrAllRatio7 = Float64[], 
#    Region8 = String[], mutCt8 = Int[], chrMutCt8 = Int[], mutPct8 = Float64[], qualSeqPct8 = Float64[], log10fisherPval8 = Float64[], chrAllRatio8 = Float64[], 
#    Region9 = String[], mutCt9 = Int[], chrMutCt9 = Int[], mutPct9 = Float64[], qualSeqPct9 = Float64[], log10fisherPval9 = Float64[], chrAllRatio9 = Float64[], 
#    Region10 = String[],mutCt10 = Int[],chrMutCt10 = Int[],mutPct10 = Float64[],qualSeqPct10 = Float64[],log10fisherPval10 = Float64[],chrAllRatio10 = Float64[],
#    Region11 = String[],mutCt11 = Int[],chrMutCt11 = Int[],mutPct11 = Float64[],qualSeqPct11 = Float64[],log10fisherPval11 = Float64[],chrAllRatio11 = Float64[],   
#    Region12 = String[],mutCt12 = Int[],chrMutCt12 = Int[],mutPct12 = Float64[],qualSeqPct12 = Float64[],log10fisherPval12 = Float64[],chrAllRatio12 = Float64[],
#    Region13 = String[],mutCt13 = Int[],chrMutCt13 = Int[],mutPct13 = Float64[],qualSeqPct13 = Float64[],log10fisherPval13 = Float64[],chrAllRatio13 = Float64[],
#    Region14 = String[],mutCt14 = Int[],chrMutCt14 = Int[],mutPct14 = Float64[],qualSeqPct14 = Float64[],log10fisherPval14 = Float64[],chrAllRatio14 = Float64[],
#    Region15 = String[],mutCt15 = Int[],chrMutCt15 = Int[],mutPct15 = Float64[],qualSeqPct15 = Float64[],log10fisherPval15 = Float64[],chrAllRatio15 = Float64[],
#    Region16 = String[],mutCt16 = Int[],chrMutCt16 = Int[],mutPct16 = Float64[],qualSeqPct16 = Float64[],log10fisherPval16 = Float64[],chrAllRatio16 = Float64[],
#    Region17 = String[],mutCt17 = Int[],chrMutCt17 = Int[],mutPct17 = Float64[],qualSeqPct17 = Float64[],log10fisherPval17 = Float64[],chrAllRatio17 = Float64[],
#    Region18 = String[],mutCt18 = Int[],chrMutCt18 = Int[],mutPct18 = Float64[],qualSeqPct18 = Float64[],log10fisherPval18 = Float64[],chrAllRatio18 = Float64[],
#    Region19 = String[],mutCt19 = Int[],chrMutCt19 = Int[],mutPct19 = Float64[],qualSeqPct19 = Float64[],log10fisherPval19 = Float64[],chrAllRatio19 = Float64[],
#    Region20 = String[],mutCt20 = Int[],chrMutCt20 = Int[],mutPct20 = Float64[],qualSeqPct20 = Float64[],log10fisherPval20 = Float64[],chrAllRatio20 = Float64[],
#    ALL_mp_Sequences = String[]) 
############################################################################################################################################################################
#     In df_rand_dict, each seed_mut key must have 10 Region keys (1-10) with vectors as values. This is in order to fill the required
#        number of columns in the DataFrame. Beyond the number of correlated mut groups, values will all be: [""]
#                   seed_mut   column#         variousStats 
df_rand_dict = Dict{String, Dict{Int, Union{String, Int, Float64}}}()
#                     mp_index     seed_mut   column#         variousStats 
df_rand_dict_META = Dict{Int, Dict{String, Dict{Int, Union{String, Int, Float64}}}}()
chr_all_ratio = 0.0
all_correlated_rand_muts_set = Set{String}()
#    all_correlated_rand_mutstrings_set = Set{String}()
random_mut_correlated_muts_dict = Dict{String, Set{String}}()
random_mut_correlated_mutstrings_dict = Dict{String, Vector{String}}()
mp_index = 0
seedmut_mut_set_set = Set{Set{String}}()
blank_mutstrings = Set(["", " ", "  ", "   ", "    ", "     ", "      ", "       ", "        ", "         ", "          ", "           ", "            "])
blank_mut_ct = 0
repeat_ct = 0
for seed_mut in candidate_test_muts
    mp_index +=1
    if mp_index%100 == 0
        print("mp_index = $(mp_index)     |    ")
        nowtime = Dates.format(now(), "I:MM.SS_p"); println(nowtime)
    end
###############################################################
    global pt1_round_ct = 0
    global pt1_finished = 0
    global sel_muts_pt1_round_dict = Dict{Int, Vector{String}}()
    global all_muts_round_dict_pt2 = Dict{Int, Set{String}}()   # all_muts_round_dict_pt2 stores the qualifying mutations at the end of each round
    global all_muts_round_mutgroup_dict_pt2 = Dict{Int, Vector{String}}()
#                                 Round RegionAnalyzed  Mutation      EPCI_pct, MP_pct, totChrMutCt, MP_Tot_Mut_ct, Adjusted_MP_seq_ct, Chi^2,  pvFish, log10pvFISH
    global df_props_dict_pt2 = Dict{Int, Dict{Int, Dict{String, Tuple{Float64,Float64,Int,Int,Int,Float64,Float64,Float64,Union{Float64, String},Float64}}}}()
    global pt2_round_ct = 0
    global pt2_iteration_ct = 0
    global pt2_region_analyzed = 1
    global sel_muts_pt2 = Vector{String}()
    global sel_muts_pt2_sort = Vector{String}()
    global sel_muts_pt2_ind = Vector{Vector{String}}()
    global sel_muts_pt2_ind_sort = Vector{Vector{String}}()
    global sel_muts_pt1_global = Vector{String}()
    global sel_muts_pt1_sort = Vector{String}()
    global sel_muts_pt1_ind = Vector{Vector{String}}()
    global region_ct2
    global pt1_seed_mut = ""
    global coAAmut_ct_pt1 = Dict{String, Int}()
    global coAAmut_ct_pt2 = Dict{String, Int}()
    global coAAmut_ct_pt1_unknown = Dict{String, Int}()
    global coAAmut_ct_pt2_unknown = Dict{String, Int}()
    global prop_dict_pt1 = Dict{String, Tuple{Float64,Float64,Int,Int,Int,Float64,Float64,Float64,Union{Float64, String},Float64} }()
    global prop_dict_pt2 = Dict{String, Tuple{Float64,Float64,Int,Int,Int,Float64,Float64,Float64,Union{Float64, String},Float64} }()
    global cumulative_seq_round_set_pt2 = Set{String}() ### Set of all seqs that qualify during any iteration of a round (erased upon the start of each round so that only the final round's results are left at end)
    global df_mp = DataFrame(SeedMut = String[], MP_Tot_Mut_ct = Int[], MP_Tot_Reg_ct = Int[], Reg_Tot_Mut_ct = Int[], MP_Region_Number = Int[], Mutation = String[], MP_Mut_ct = Float64[], EPCI_Mut_ct = Int[], Adjusted_MP_seq_ct = Int[], EPCI_pct = Float64[], MP_pct = Float64[], non_MP_pct = Float64[], Fishers_Exact_Test_pval = Float64[], log10pvFISH = Float64[], Fold_Incr = Union{Float64, String}[], Chi2 = Float64[], EPCI_HQCS_Ratio = Float64[], avg_NonRBD_AAct = Float64[], avg_nonRBD_rel = Float64[], 
        PangoDateIndex50_Avg = Int[], PangoDateStr50_Avg = String[], CollDateIndex_Avg = Int[], CollDateString_Avg = String[], Est_Infx_Length = Int[], 
        PangoDateIndex50_min = Int[], PangoDateIndex50_p10 = Int[], PangoDateIndex50_Q1 = Int[], PangoDateIndex50_Q2 = Int[], PangoDateIndex50_Q3 = Int[], PangoDateIndex50_p90 = Int[], PangoDateIndex50_max = Int[], 
        PangoDateStr50_min = String[], PangoDateStr50_p10 = String[], PangoDateStr50_Q1 = String[], PangoDateStr50_Q2 = String[], PangoDateStr50_Q3 = String[], PangoDateStr50_p90 = String[], PangoDateStr50_max = String[], 
        CollDateIndex_min = Int[], CollDateIndex_p10 = Int[], CollDateIndex_Q1 = Int[], CollDateIndex_Q2 = Int[], CollDateIndex_Q3 = Int[], CollDateIndex_p90 = Int[], CollDateIndex_max = Int[], 
        CollDateString_min = String[], CollDateString_p10 = String[], CollDateString_Q1 = String[], CollDateString_Q2 = String[], CollDateString_Q3 = String[], CollDateString_p90 = String[], CollDateString_max = String[], ALL_mp_Sequences = String[])
    global df_mp_META
    global df_mp_META_sort
    global df_mp_META_seedmut_set_dict
    global df_mp_META_unique
    global df_mp_META_sort_unique
    global df_mp_META_seedmut_set_dict_unique
    df_rand_dict[seed_mut] = Dict{Int, Union{String, Int, Float64}}()
    random_mut_correlated_muts_dict[seed_mut] = Set{String}()
    random_mut_correlated_mutstrings_dict[seed_mut] = Vector{String}()
    starter = [seed_mut]
    mut_pattern_name = seed_mut
############################################################################################################################################################################
############################################################################################################################################################################ 
############################################################################################################################################################################ 
    df_props_dict_pt2__keyRoundCt, cumulative_seq_round_set_pt2_sort_join, avg_mp_pango_date_index_int_50, avg_mp_pango_date_string_50, pango_ct_dict_sort, clade_ct_dict_sort, cladepango_ct_dict_sort = AA_2plus__2025_11_19__UNIVERSAL(min_match_ct, pt1_ct_mltplr, pt2_ct_mltplr, purespikebanned_0__purespikeallowed_1, nonmulti0__multi1, notrandom0__random1, nonspike_seqfactor__0off__1RBM__2RBD__3SpikeNonRBD, plus_minus, mut_pattern_name, ind_or_grp, min_grp_fish, min_log_pv_fish, mp_masked_muts, starter, seq_factor_OFF_HALF_ON__0_1_2)
############################################################################################################################################################################
############################################################################################################################################################################ 
############################################################################################################################################################################
#                                    RegionAnalyzed  Mutation      EPCI_pct, MP_pct, totChrMutCt, MP_Tot_Mut_ct, Adjusted_MP_seq_ct, Chi^2,  pvFish, log10pvFISH, log10pvChiSq        
#     df_props_dict_pt2__keyRoundCt = Dict{Int, Dict{String, Tuple{Float64,Float64,Int,Int,Int,Float64,Float64,Float64,Float64}}}}()
    df_mp_META_seedmut_set_dict[seed_mut] = cumulative_seq_round_set_pt2
#    cumulative_seq_round_set_pt2_sort = sort(collect(cumulative_seq_round_set_pt2), by = x -> (length(x), x))
    all_mp_seqs_sorted = split(cumulative_seq_round_set_pt2_sort_join, ", ")
    total_MP_sequence_ct = length(all_mp_seqs_sorted)
    all_mp_muts_collected_set = Set{String}()
    for region_num in keys(df_props_dict_pt2__keyRoundCt)
        df_props_dict_pt2__keyRoundCt__keyRegion__keyMutsValueProps = df_props_dict_pt2__keyRoundCt[region_num]
        all_mp_muts_collected_set = Set(collect(keys(df_props_dict_pt2__keyRoundCt__keyRegion__keyMutsValueProps)))
    end
#####################################################################################################################################################
#####################################################################################################################################################
    if seed_mut in pt1_seed_mut_set
        df_pango_ct = DataFrame(
            pango_lineage = String[],
            pango_unaliased = String[],
            count = Int[])
        df_clade_ct = DataFrame(
            clade = String[],
            cladepango = String[],
            cladepango_unaliased = String[],
            count = Int[])
        df_cladepango_ct = DataFrame(
            cladepango = String[],
            cladepango_unaliased = String[],
            count = Int[])
        df_megaclade_ct = DataFrame(
            megaclade = String[],
            count = Int[],
            expected_ct = Float64[],
            fold_increase = Float64[])
    #    pango_ct_file_tsv_df_RANDOM = "$(mp_folder_universal)_RANDOM/mp_$(mut_pattern_name)_PANGO_CTS__EPCI_$(EPCI_qc_str)__HQCS_$(HQCS_qc_str)__plusminus$(plus_minus)_minGrpFish$(min_grp_fish)_minFish$(min_log_pv_fish)_seqfac$(seqfac)_$(date).tsv"
    #    pango_dict_ct_sort_string = ""
    #    clade_dict_ct_sort_string = ""
    #    cladepango_dict_ct_sort_string = ""
    ########################################################
        pango_ct_file_csv_df_RANDOM = "$(pango_counts_folder_rand_csv)/mp_$(mut_pattern_name)_PANGO_CTS__EPCI_$(EPCI_qc_str)__HQCS_$(HQCS_qc_str)__plusminus$(plus_minus)_minGrpFish$(min_grp_fish)_minFish$(min_log_pv_fish)_seqfac$(seqfac)_$(date).csv"
        clade_ct_file_csv_df_RANDOM = "$(pango_counts_folder_rand_csv)/mp_$(mut_pattern_name)_CLADE_CTS__EPCI_$(EPCI_qc_str)__HQCS_$(HQCS_qc_str)__plusminus$(plus_minus)_minGrpFish$(min_grp_fish)_minFish$(min_log_pv_fish)_seqfac$(seqfac)_$(date).csv"
        cladepango_ct_file_csv_df_RANDOM = "$(pango_counts_folder_rand_csv)/mp_$(mut_pattern_name)_CLADEPANGO_CTS__EPCI_$(EPCI_qc_str)__HQCS_$(HQCS_qc_str)__plusminus$(plus_minus)_minGrpFish$(min_grp_fish)_minFish$(min_log_pv_fish)_seqfac$(seqfac)_$(date).csv"
        megaclade_ct_file_csv_df_RANDOM = "$(pango_counts_folder_rand_csv)/mp_$(mut_pattern_name)_MEGACLADE_CTS__EPCI_$(EPCI_qc_str)__HQCS_$(HQCS_qc_str)__plusminus$(plus_minus)_minGrpFish$(min_grp_fish)_minFish$(min_log_pv_fish)_seqfac$(seqfac)_$(date).csv"
    ########################################################    
        mega_clade_pangos = ["pre_VOC", "B.1.1.7", "B.1.351", "B.1.1.28.1", "B.1.617", "B.1.1.529.1", "B.1.1.529.2", "B.1.1.529.4", "B.1.1.529.5", "XBB", "B.1.1.529.2.86"]
        mega_clade_dict_keys = ["pre_VOC", "B.1.1.7", "B.1.351", "P.1", "B.1.617", "BA.1", "BA.2", "BA.5", "XBB", "BA.2.86", "Omicron_Unknown", "Minor_Recombinant"]
        megaclade_ct_dict = Dict{String, Int}()
        for megaclade in mega_clade_dict_keys
            megaclade_ct_dict[megaclade] = 0
        end
        for i in 1:length(pango_ct_dict_sort)
            pango__count = pango_ct_dict_sort[i]
            pango = pango__count[1]
            count = pango__count[2]
            unaliased = get(pango_to_pango_unaliased_v2, pango, pango)
            push!(df_pango_ct, (pango, unaliased, count))     
        end
        for i in 1:length(clade_ct_dict_sort)
            clade__count = clade_ct_dict_sort[i]
            clade = clade__count[1]
            count = clade__count[2]
            cladepango = clade_to_pango[clade]
            cladepango_unaliased = get(pango_to_pango_unaliased_v2, cladepango, cladepango)
            push!(df_clade_ct, (clade, cladepango, cladepango_unaliased, count))
        end
        for i in 1:length(cladepango_ct_dict_sort)
            cladepango__count = cladepango_ct_dict_sort[i]
            cladepango = cladepango__count[1]
            count = cladepango__count[2]
            cladepango_unaliased = get(pango_to_pango_unaliased_v2, cladepango, cladepango)
            push!(df_cladepango_ct, (cladepango, cladepango_unaliased, count))
        end
        BA286_Xs = ["XEC", "XEK", "XFG"]
        XBB_Xs = ["XCH"]
        for i in 1:length(pango_ct_dict_sort)
            complete = 0
            pango__count = pango_ct_dict_sort[i]
            pango = pango__count[1]
            count = pango__count[2]
            unaliased = get(pango_to_pango_unaliased_v2, pango, pango)
            if length(unaliased) ≥ 3
                if unaliased[1:3] == "XBB"
                    megaclade_ct_dict["XBB"] += count
                    complete = 1
                elseif unaliased[1:3] == "XEC" || unaliased[1:3] == "XEK" || unaliased[1:3] == "XFG"
                    megaclade_ct_dict["BA.2.86"] += count
                    complete = 1
                elseif unaliased[1:3] == "XCH"
                    megaclade_ct_dict["XBB"] += count
                    complete = 1
                end
                if length(unaliased) ≥ 7
                    if unaliased[1:7] == "B.1.1.7"
                        megaclade_ct_dict["B.1.1.7"] += count
                        complete = 1
                    elseif unaliased[1:7] == "B.1.351"
                        megaclade_ct_dict["B.1.351"] += count
                        complete = 1
                    elseif unaliased[1:7] == "B.1.617"
                        megaclade_ct_dict["B.1.617"] += count
                        complete = 1
                    end
                    if length(unaliased) ≥ 9
                        if unaliased == "B.1.1.529"
                            megaclade_ct_dict["Omicron_Unknown"] += count
                            complete = 1
                        end
                        if length(unaliased) ≥ 10 
                            if unaliased[1:10] == "B.1.1.28.1"
                                megaclade_ct_dict["P.1"] += count
                                complete = 1
                            end
                            if length(unaliased) ≥ 11
                                if unaliased[1:11] == "B.1.1.529.1"
                                    megaclade_ct_dict["BA.1"] += count
                                    complete = 1
                                elseif unaliased[1:11] == "B.1.1.529.2"
                                    megaclade_ct_dict["BA.2"] += count
                                    complete = 1
                                elseif unaliased[1:11] == "B.1.1.529.4"
                                    megaclade_ct_dict["BA.5"] += count
                                    complete = 1
                                elseif unaliased[1:11] == "B.1.1.529.5"
                                    megaclade_ct_dict["BA.5"] += count
                                    complete = 1
                                end
                                if length(unaliased) ≥ 14
                                    if unaliased[1:14] == "B.1.1.529.2.86"
                                        megaclade_ct_dict["BA.2.86"] += count
                                        megaclade_ct_dict["BA.2"] = megaclade_ct_dict["BA.2"] - count
                                        complete = 1
                                    end
                                end
                            end
                        end
                    end
                end
            end
            if complete == 0
                if unaliased[1] == 'A' || unaliased[1] == 'B'
                    megaclade_ct_dict["pre_VOC"] +=count
                elseif unaliased[1] == 'X'
                    megaclade_ct_dict["Minor_Recombinant"] +=count
                else
                    println("********************** WARNING: Uncategorized PANGO **************************")
                end
            end
        end
        megaclade_ct_dict_sort = sort(collect(megaclade_ct_dict), by = x -> x[2], rev=true)
        for i in 1:length(megaclade_ct_dict_sort)
            megaclade___count = megaclade_ct_dict_sort[i]
            megaclade = megaclade___count[1]
            count = megaclade___count[2]
    #######################
            megaclade_expected_ct = megaclade_EPCI_proportion_dict[megaclade]*total_MP_sequence_ct
            EPCI_megaclade_ct = megaclade_EPCI_ct_dict[megaclade]
            non_MP_EPCI_ct = total_EPCI_seq_ct - total_MP_sequence_ct
            non_MP_megaclade_ct = EPCI_megaclade_ct - count
            megaclade_non_MP_prop = non_MP_megaclade_ct/non_MP_EPCI_ct
            megaclade_MP_prop = count/total_MP_sequence_ct
            megaclade_fold_enrichment = megaclade_MP_prop/megaclade_non_MP_prop
            push!(df_megaclade_ct, (megaclade, count, megaclade_expected_ct, megaclade_fold_enrichment))
        end
        CSV.write(pango_ct_file_csv_df_RANDOM, df_pango_ct)    #; CSV.write(pango_ct_file_tsv_df_RANDOM, df_pango_ct, delim='\t')
        CSV.write(clade_ct_file_csv_df_RANDOM, df_clade_ct)
        CSV.write(cladepango_ct_file_csv_df_RANDOM, df_cladepango_ct)
        CSV.write(megaclade_ct_file_csv_df_RANDOM, df_megaclade_ct)
    end
end
#####################################################################################################################################################
#####################################################################################################################################################
#    pango_pad = rpad(pango, 11)
#    count_pad = lpad(count, 3)
#    pango_count_str = ""
#    if i == length(pango_ct_dict_sort)
#        pango_count_str = "$(pango_pad) = $(count_pad)"
#    else
#        pango_count_str = "$(pango_pad) = $(count_pad)\n"
#    end
#    pango_dict_ct_sort_string = pango_dict_ct_sort_string*pango_count_str
#    open(pango_file, "w") do g
#         println(g)
#         println(g, pango_dict_ct_sort_string)
#         println(g)
#    end
#    cumulative_seq_round_set_pt2_sort_join = join(cumulative_seq_round_set_pt2_sort, ", ")
#    "if... seed_mut in all_mp_muts_collected_set" here prevents "off-target" MP's from appearing randomly. E.g. 8fold often appears even when seed mut isn't an 8fold mut
#    final_rand_muts = Set{String}()
#    if df_props_dict_pt2__keyRoundCt ≠ nothing && !isempty(keys(df_props_dict_pt2__keyRoundCt)) && seed_mut in all_mp_muts_collected_set
#        keys_len = length(keys(df_props_dict_pt2__keyRoundCt))
#        if keys_len > 1
#            seedmut_mut_set = Set{String}()
#            for region in keys(df_props_dict_pt2__keyRoundCt)
#                for mut in keys(df_props_dict_pt2__keyRoundCt[region])
#                    push!(seedmut_mut_set, mut)
#                end
#            end
#            if !(seed_mut in seedmut_mut_set)
#                continue
#            end
#            println(seedmut_mut_set)
#            if seedmut_mut_set in seedmut_mut_set_set
#                repeat_ct += 1
#                print("\n"^1)
#                print("Repeat Pattern Count = $(repeat_ct)")
#                print(" | Total mp Count = $(total_mp_ct)")
#                println(" | mp_index = $(mp_index)")
#                seedmut_mut_set_str = join(collect(seedmut_mut_set), ",")
#                println(seedmut_mut_set_str)
#                print("\n"^1)
#                continue
#            end
#            println("length df_props_dict_pt2__keyRoundCt = $(length(df_props_dict_pt2__keyRoundCt))")
#            region_ct_for_df_rand_dict = 0
#            total_mp_ct += 1
#            push!(seedmut_mut_set_set, seedmut_mut_set)
##########################################################################################################################
#            mut_ct_length_vec = collect(length(mut_keys) for (region, mut_keys) in df_props_dict_pt2__keyRoundCt)
#            max_region_mut_ct = maximum(mut_ct_length_vec)
#            for region_num in keys(df_props_dict_pt2__keyRoundCt)
#                total_region_muts = length(keys(df_props_dict_pt2__keyRoundCt[region_num]))
#                blanks_to_fill = max_region_mut_ct - total_region_muts
#                if blanks_to_fill > 0
#                    for i in 1:blanks_to_fill
#                        blank_mut_key = " "^i
#                        df_props_dict_pt2__keyRoundCt[region_num][blank_mut_key] = (0.0,0.0,0,0,0,0.0,0.0,0.0,0.0,0.0)
#                    end
#                end
#            end                           
##########################################################################################################################
#            for region_num in keys(df_props_dict_pt2__keyRoundCt)
#                region_ct_for_df_rand_dict += 1
#                column_grp_mut = (region_ct_for_df_rand_dict-1)*7 + 1
#                mut_grp_vec = Vector{String}()
#                df_props_dict_pt2__keyRoundCt__keyRegion__keyMutsValueProps = df_props_dict_pt2__keyRoundCt[region_num]
#                for mut in keys(df_props_dict_pt2__keyRoundCt__keyRegion__keyMutsValueProps)
#                    if mut[1] ≠ ' '
#                        push!(mut_grp_vec, mut)
#                        push!(final_rand_muts, mut)
#                        push!(random_mut_correlated_muts_dict[seed_mut], mut)
#                    end
#                end
#                mut_grp_vec_sort = sort(mut_grp_vec, by = x -> (aa_gene_comprehensive_dict[x], aa_pos_comprehensive_dict[x]))
#                mut_grp_vec_sort_len = length(mut_grp_vec_sort)
#                for mut in mut_grp_vec_sort
#                    props = df_props_dict_pt2__keyRoundCt__keyRegion__keyMutsValueProps[mut]
#                    EPCI_pct = props[1]
#                    MP_pct = props[2]
#                    EPCI_mut_ct = props[3]
#                    qual_mut_ct = props[4]
#                    qual_seq_ct = props[5]
#                    Chi2 = props[6]
#                    pvFish = props[7]
#                    log10pvFISH = props[8]
#                    chr_all_ratio_pre = get(mp_chr_all_ratio, mut, 0)
#                    chr_all_ratio = round(digits=1, chr_all_ratio_pre)
#######################################
#                    pct_chr_seq_qual = 100*qual_seq_ct/total_EPCI_seq_ct
#                    pct_chr_mut_qual = 100*qual_mut_ct/EPCI_mut_ct
#######################################
#                    df_rand_dict[seed_mut][column_grp_mut] = mut
#                    df_rand_dict[seed_mut][column_grp_mut + 1] = qual_mut_ct
#                    df_rand_dict[seed_mut][column_grp_mut + 2] = EPCI_mut_ct
#                    df_rand_dict[seed_mut][column_grp_mut + 3] = pct_chr_mut_qual
#                    df_rand_dict[seed_mut][column_grp_mut + 4] = pct_chr_seq_qual
#                    df_rand_dict[seed_mut][column_grp_mut + 5] = log10pvFISH
#                    df_rand_dict[seed_mut][column_grp_mut + 6] = chr_all_ratio
#############################################################################################################
#                end
#            end
#            total_regions = length(keys(df_props_dict_pt2__keyRoundCt))
#            emptycolumn1 = total_regions*7 + 1
#            for c in emptycolumn1:7:109
#            for c in emptycolumn1:7:144
#                df_rand_dict[seed_mut][c] = " "
#            end
#            for d in emptycolumn1:140
#                if (d-1) % 7 ≠ 0
#                    df_rand_dict[seed_mut][d] = 0
#                end
#            end
##################################################################################################################################################################################                       
#            push!(df_rand_corr_muts, (total_mp_ct, seed_mut, df_rand_dict[seed_mut][1], df_rand_dict[seed_mut][2], df_rand_dict[seed_mut][3], df_rand_dict[seed_mut][4], df_rand_dict[seed_mut][5], df_rand_dict[seed_mut][6], df_rand_dict[seed_mut][7], df_rand_dict[seed_mut][8], df_rand_dict[seed_mut][9], df_rand_dict[seed_mut][10], df_rand_dict[seed_mut][11], df_rand_dict[seed_mut][12], df_rand_dict[seed_mut][13], df_rand_dict[seed_mut][14], df_rand_dict[seed_mut][15], df_rand_dict[seed_mut][16], df_rand_dict[seed_mut][17], df_rand_dict[seed_mut][18], df_rand_dict[seed_mut][19], df_rand_dict[seed_mut][20], df_rand_dict[seed_mut][21], df_rand_dict[seed_mut][22], df_rand_dict[seed_mut][23], df_rand_dict[seed_mut][24], df_rand_dict[seed_mut][25], df_rand_dict[seed_mut][26], df_rand_dict[seed_mut][27], df_rand_dict[seed_mut][28], df_rand_dict[seed_mut][29], df_rand_dict[seed_mut][30], df_rand_dict[seed_mut][31], df_rand_dict[seed_mut][32], df_rand_dict[seed_mut][33], df_rand_dict[seed_mut][34], df_rand_dict[seed_mut][35], df_rand_dict[seed_mut][36], df_rand_dict[seed_mut][37], df_rand_dict[seed_mut][38], df_rand_dict[seed_mut][39], df_rand_dict[seed_mut][40], df_rand_dict[seed_mut][41], df_rand_dict[seed_mut][42], df_rand_dict[seed_mut][43], df_rand_dict[seed_mut][44], df_rand_dict[seed_mut][45], df_rand_dict[seed_mut][46], df_rand_dict[seed_mut][47], df_rand_dict[seed_mut][48], df_rand_dict[seed_mut][49], df_rand_dict[seed_mut][50], df_rand_dict[seed_mut][51], df_rand_dict[seed_mut][52], df_rand_dict[seed_mut][53], df_rand_dict[seed_mut][54], df_rand_dict[seed_mut][55], df_rand_dict[seed_mut][56], df_rand_dict[seed_mut][57], df_rand_dict[seed_mut][58], df_rand_dict[seed_mut][59], df_rand_dict[seed_mut][60], df_rand_dict[seed_mut][61], df_rand_dict[seed_mut][62], df_rand_dict[seed_mut][63], df_rand_dict[seed_mut][64], df_rand_dict[seed_mut][65], df_rand_dict[seed_mut][66], df_rand_dict[seed_mut][67], df_rand_dict[seed_mut][68], df_rand_dict[seed_mut][69], df_rand_dict[seed_mut][70], df_rand_dict[seed_mut][71], df_rand_dict[seed_mut][72], df_rand_dict[seed_mut][73], df_rand_dict[seed_mut][74], df_rand_dict[seed_mut][75], df_rand_dict[seed_mut][76], df_rand_dict[seed_mut][77], df_rand_dict[seed_mut][78], df_rand_dict[seed_mut][79], df_rand_dict[seed_mut][80], df_rand_dict[seed_mut][81], df_rand_dict[seed_mut][82], df_rand_dict[seed_mut][83], df_rand_dict[seed_mut][84], df_rand_dict[seed_mut][85], df_rand_dict[seed_mut][86], df_rand_dict[seed_mut][87], df_rand_dict[seed_mut][88], df_rand_dict[seed_mut][89], df_rand_dict[seed_mut][90], df_rand_dict[seed_mut][91], df_rand_dict[seed_mut][92], df_rand_dict[seed_mut][93], df_rand_dict[seed_mut][94], df_rand_dict[seed_mut][95], df_rand_dict[seed_mut][96], df_rand_dict[seed_mut][97], df_rand_dict[seed_mut][98], df_rand_dict[seed_mut][99], df_rand_dict[seed_mut][100], df_rand_dict[seed_mut][101], df_rand_dict[seed_mut][102], df_rand_dict[seed_mut][103], df_rand_dict[seed_mut][104], df_rand_dict[seed_mut][105], df_rand_dict[seed_mut][106], df_rand_dict[seed_mut][107], df_rand_dict[seed_mut][108], df_rand_dict[seed_mut][109], df_rand_dict[seed_mut][110], df_rand_dict[seed_mut][111], df_rand_dict[seed_mut][112], df_rand_dict[seed_mut][113], df_rand_dict[seed_mut][114], df_rand_dict[seed_mut][115], df_rand_dict[seed_mut][116], df_rand_dict[seed_mut][117], df_rand_dict[seed_mut][118], df_rand_dict[seed_mut][119], df_rand_dict[seed_mut][120], df_rand_dict[seed_mut][121], df_rand_dict[seed_mut][122], df_rand_dict[seed_mut][123], df_rand_dict[seed_mut][124], df_rand_dict[seed_mut][125], df_rand_dict[seed_mut][126], df_rand_dict[seed_mut][127], df_rand_dict[seed_mut][128], df_rand_dict[seed_mut][129], df_rand_dict[seed_mut][130], df_rand_dict[seed_mut][131], df_rand_dict[seed_mut][132], df_rand_dict[seed_mut][133], df_rand_dict[seed_mut][134], df_rand_dict[seed_mut][135], df_rand_dict[seed_mut][136], df_rand_dict[seed_mut][137], df_rand_dict[seed_mut][138], df_rand_dict[seed_mut][139], df_rand_dict[seed_mut][140], cumulative_seq_round_set_pt2_sort_join))
#            push!(df_rand_corr_muts_META, (total_mp_ct, seed_mut, df_rand_dict[seed_mut][1], df_rand_dict[seed_mut][2], df_rand_dict[seed_mut][3], df_rand_dict[seed_mut][4], df_rand_dict[seed_mut][5], df_rand_dict[seed_mut][6], df_rand_dict[seed_mut][7], df_rand_dict[seed_mut][8], df_rand_dict[seed_mut][9], df_rand_dict[seed_mut][10], df_rand_dict[seed_mut][11], df_rand_dict[seed_mut][12], df_rand_dict[seed_mut][13], df_rand_dict[seed_mut][14], df_rand_dict[seed_mut][15], df_rand_dict[seed_mut][16], df_rand_dict[seed_mut][17], df_rand_dict[seed_mut][18], df_rand_dict[seed_mut][19], df_rand_dict[seed_mut][20], df_rand_dict[seed_mut][21], df_rand_dict[seed_mut][22], df_rand_dict[seed_mut][23], df_rand_dict[seed_mut][24], df_rand_dict[seed_mut][25], df_rand_dict[seed_mut][26], df_rand_dict[seed_mut][27], df_rand_dict[seed_mut][28], df_rand_dict[seed_mut][29], df_rand_dict[seed_mut][30], df_rand_dict[seed_mut][31], df_rand_dict[seed_mut][32], df_rand_dict[seed_mut][33], df_rand_dict[seed_mut][34], df_rand_dict[seed_mut][35], df_rand_dict[seed_mut][36], df_rand_dict[seed_mut][37], df_rand_dict[seed_mut][38], df_rand_dict[seed_mut][39], df_rand_dict[seed_mut][40], df_rand_dict[seed_mut][41], df_rand_dict[seed_mut][42], df_rand_dict[seed_mut][43], df_rand_dict[seed_mut][44], df_rand_dict[seed_mut][45], df_rand_dict[seed_mut][46], df_rand_dict[seed_mut][47], df_rand_dict[seed_mut][48], df_rand_dict[seed_mut][49], df_rand_dict[seed_mut][50], df_rand_dict[seed_mut][51], df_rand_dict[seed_mut][52], df_rand_dict[seed_mut][53], df_rand_dict[seed_mut][54], df_rand_dict[seed_mut][55], df_rand_dict[seed_mut][56], df_rand_dict[seed_mut][57], df_rand_dict[seed_mut][58], df_rand_dict[seed_mut][59], df_rand_dict[seed_mut][60], df_rand_dict[seed_mut][61], df_rand_dict[seed_mut][62], df_rand_dict[seed_mut][63], df_rand_dict[seed_mut][64], df_rand_dict[seed_mut][65], df_rand_dict[seed_mut][66], df_rand_dict[seed_mut][67], df_rand_dict[seed_mut][68], df_rand_dict[seed_mut][69], df_rand_dict[seed_mut][70], df_rand_dict[seed_mut][71], df_rand_dict[seed_mut][72], df_rand_dict[seed_mut][73], df_rand_dict[seed_mut][74], df_rand_dict[seed_mut][75], df_rand_dict[seed_mut][76], df_rand_dict[seed_mut][77], df_rand_dict[seed_mut][78], df_rand_dict[seed_mut][79], df_rand_dict[seed_mut][80], df_rand_dict[seed_mut][81], df_rand_dict[seed_mut][82], df_rand_dict[seed_mut][83], df_rand_dict[seed_mut][84], df_rand_dict[seed_mut][85], df_rand_dict[seed_mut][86], df_rand_dict[seed_mut][87], df_rand_dict[seed_mut][88], df_rand_dict[seed_mut][89], df_rand_dict[seed_mut][90], df_rand_dict[seed_mut][91], df_rand_dict[seed_mut][92], df_rand_dict[seed_mut][93], df_rand_dict[seed_mut][94], df_rand_dict[seed_mut][95], df_rand_dict[seed_mut][96], df_rand_dict[seed_mut][97], df_rand_dict[seed_mut][98], df_rand_dict[seed_mut][99], df_rand_dict[seed_mut][100], df_rand_dict[seed_mut][101], df_rand_dict[seed_mut][102], df_rand_dict[seed_mut][103], df_rand_dict[seed_mut][104], df_rand_dict[seed_mut][105], df_rand_dict[seed_mut][106], df_rand_dict[seed_mut][107], df_rand_dict[seed_mut][108], df_rand_dict[seed_mut][109], df_rand_dict[seed_mut][110], df_rand_dict[seed_mut][111], df_rand_dict[seed_mut][112], df_rand_dict[seed_mut][113], df_rand_dict[seed_mut][114], df_rand_dict[seed_mut][115], df_rand_dict[seed_mut][116], df_rand_dict[seed_mut][117], df_rand_dict[seed_mut][118], df_rand_dict[seed_mut][119], df_rand_dict[seed_mut][120], df_rand_dict[seed_mut][121], df_rand_dict[seed_mut][122], df_rand_dict[seed_mut][123], df_rand_dict[seed_mut][124], df_rand_dict[seed_mut][125], df_rand_dict[seed_mut][126], df_rand_dict[seed_mut][127], df_rand_dict[seed_mut][128], df_rand_dict[seed_mut][129], df_rand_dict[seed_mut][130], df_rand_dict[seed_mut][131], df_rand_dict[seed_mut][132], df_rand_dict[seed_mut][133], df_rand_dict[seed_mut][134], df_rand_dict[seed_mut][135], df_rand_dict[seed_mut][136], df_rand_dict[seed_mut][137], df_rand_dict[seed_mut][138], df_rand_dict[seed_mut][139], df_rand_dict[seed_mut][140], cumulative_seq_round_set_pt2_sort_join))
#################################################################################################################################################################################                                
#            for mut in final_rand_muts
#                push!(all_correlated_rand_muts_set, mut)
#            end
#            random_mut_correlated_muts_sort = sort(collect(random_mut_correlated_muts_dict[seed_mut]), by = x -> mp_AA_gene_sortKey_2_universal(x, sub_0__posonly_1))
#            random_mut_correlated_muts_sort_string = join(random_mut_correlated_muts_sort, ", ")
#            push!(random_mut_correlated_mutstrings_dict[seed_mut], random_mut_correlated_muts_sort_string)
#            open("$(rand_folder)/rand_muts_mp_$(random_sample_size)mutsTested__EPCI_$(EPCI_qc_str)__HQCS_$(HQCS_qc_string)___plusminus$(plus_minus)_minGrpFish$(min_grp_fish)_minFish$(min_log_pv_fish)_seqfac$(seqfac)_$(date_hour).txt", "a") do g
#                if length(random_mut_correlated_muts_dict[seed_mut]) > 1
#                    print("# rand Correlated Muts: $(seed_mut)———")
#                    for mutstring in random_mut_correlated_mutstrings_dict[seed_mut]
#                        print(mutstring, " | ")
#                    end
#                    print("\n"^1)
#                    println(g, "# rand Mut Correlated Muts: $(seed_mut)———")
#                    for mutstring in random_mut_correlated_mutstrings_dict[seed_mut]
#                        print(g, mutstring, " | ")
#                    end
#                    print(g, "\n"^1)
#                end
#            end
#        end
############################################################################################################################################################################
#    end
#end
#all_correlated_rand_muts_set_sort = sort(collect(all_correlated_rand_muts_set), by = x -> mp_AA_gene_sortKey_2_universal(x, sub_0__posonly_1))
#all_correlated_rand_muts_set_sort_string = join(all_correlated_rand_muts_set_sort, ", ")
###########################################################################################################################################################################
###########################################################################################################################################################################
CSV.write("$(rand_folder)/rand_ALL_mp_df_stats_$(random_sample_size)mutsTested__EPCI_$(EPCI_qc_str)__HQCS_$(HQCS_qc_string)___plusminus$(plus_minus)_minGrpFish$(min_grp_fish)_minFish$(min_log_pv_fish)_seqfac$(seqfac)_$(date_hour).csv", df_mp_stats)
#CSV.write("$(rand_folder)/rand_ALL_mp_df_stats_$(random_sample_size)mutsTested__EPCI_$(EPCI_qc_str)__HQCS_$(HQCS_qc_string)___plusminus$(plus_minus)_minGrpFish$(min_grp_fish)_minFish$(min_log_pv_fish)_seqfac$(seqfac)_$(date_hour).tsv", df_mp_stats, delim='\t')
###########################################################################################################################################################################
###########################################################################################################################################################################
#print("\n"^1)
#print("Total mp Count = $(total_mp_ct)  | ")
#print("Total blank_mut Count = $(blank_mut_ct) | ")
#println("Total Repeat Count = $(repeat_ct)"); print("\n"^1)
## Julia's DataFrame sorting syntax: sort(dataframe_to_be_sorted, column_to_sort, by = sort_function)
#df_rand_corr_muts_sort = sort(df_rand_corr_muts, "Seed_Mut", by = x -> mp_AA_gene_sortKey_2_universal(x, sub_0__posonly_1))
df_mp_META_groups_dict_sort = sort(collect(df_mp_META_groups_dict), by = x -> length(x[2]), rev=true)
for seedmut____tuplevec in df_mp_META_groups_dict_sort
    tuplevec = seedmut____tuplevec[2]
    for tuple in tuplevec
        push!(df_mp_META_sort, tuple)
    end
end
df_mp_META_groups_dict_sort_unique = sort(collect(df_mp_META_groups_dict_unique), by = x -> length(x[2]), rev=true)
for seedmut____tuplevec in df_mp_META_groups_dict_sort_unique
    tuplevec = seedmut____tuplevec[2]
    for tuple in tuplevec
        push!(df_mp_META_sort_unique, tuple)
    end
end
############################################################################################################################################################################
#CSV.write("$(rand_folder)/df_rand_correlated_muts_$(random_sample_size)mutsTested__EPCI_$(EPCI_qc_str)__HQCS_$(HQCS_qc_string)___plusminus$(plus_minus)_minGrpFish$(min_grp_fish)_minFish$(min_log_pv_fish)_seqfac$(seqfac)_$(date_hour).csv", df_rand_corr_muts_sort)
#CSV.write("$(rand_folder)/df_rand_correlated_muts_$(random_sample_size)mutsTested__EPCI_$(EPCI_qc_str)__HQCS_$(HQCS_qc_string)___plusminus$(plus_minus)_minGrpFish$(min_grp_fish)_minFish$(min_log_pv_fish)_seqfac$(seqfac)_$(date_hour).tsv", df_rand_corr_muts_sort, delim='\t')
CSV.write("$(rand_folder)/df_rand_MP_META__EPCI_$(EPCI_qc_str)__HQCS_$(HQCS_qc_string)___plusminus$(plus_minus)_minGrpFish$(min_grp_fish)_minFish$(min_log_pv_fish)_seqfac$(seqfac)_$(date_hour).csv", df_mp_META)
#CSV.write("$(rand_folder)/df_rand_MP_META__EPCI_$(EPCI_qc_str)__HQCS_$(HQCS_qc_string)___plusminus$(plus_minus)_minGrpFish$(min_grp_fish)_minFish$(min_log_pv_fish)_seqfac$(seqfac)_$(date_hour).tsv", df_mp_META, delim='\t') 
CSV.write("$(rand_folder)/df_rand_MP_META_sorted__EPCI_$(EPCI_qc_str)__HQCS_$(HQCS_qc_string)___plusminus$(plus_minus)_minGrpFish$(min_grp_fish)_minFish$(min_log_pv_fish)_seqfac$(seqfac)_$(date_hour).csv", df_mp_META_sort)
#CSV.write("$(rand_folder)/df_rand_MP_META_sorted__EPCI_$(EPCI_qc_str)__HQCS_$(HQCS_qc_string)___plusminus$(plus_minus)_minGrpFish$(min_grp_fish)_minFish$(min_log_pv_fish)_seqfac$(seqfac)_$(date_hour).tsv", df_mp_META_sort, delim='\t')
CSV.write("$(rand_folder)/df_rand_MP_META_sorted_UNIQUE_EPCI_$(EPCI_qc_str)__HQCS_$(HQCS_qc_string)___plusminus$(plus_minus)_minGrpFish$(min_grp_fish)_minFish$(min_log_pv_fish)_seqfac$(seqfac)_$(date_hour).csv", df_mp_META_sort_unique)
#CSV.write("$(rand_folder)/df_rand_MP_META_sorted_UNIQUE_EPCI_$(EPCI_qc_str)__HQCS_$(HQCS_qc_string)___plusminus$(plus_minus)_minGrpFish$(min_grp_fish)_minFish$(min_log_pv_fish)_seqfac$(seqfac)_$(date_hour).tsv", df_mp_META_sort_unique, delim='\t') 
############################################################################################################################################################################    
nowtime2 = Dates.format(now(), "I:MM.SS_p")
time2 = time()
println("Finish Time = $(nowtime2)")
rd_finish_time = round(digits=1, time2 - time1)
println("Time to Finish Round = $(rd_finish_time) seconds")
print("\n"^2)
############################################################################################################################################################################
nowtime = Dates.format(now(), "I:MM.SS_p"); println(nowtime)
############################################################################################################################################################################
#CSV.write("$(rand_folder)/FINAL_df_rand_MP_META_META__EPCI_$(EPCI_qc_str)__HQCS_$(HQCS_qc_string)___plusminus$(plus_minus)_minGrpFish$(min_grp_fish)_minFish$(min_log_pv_fish)_seqfac$(seqfac)_$(date_hour).csv", df_mp_META_META)
#CSV.write("$(rand_folder)/FINAL_df_rand_MP_META_META__EPCI_$(EPCI_qc_str)__HQCS_$(HQCS_qc_string)___plusminus$(plus_minus)_minGrpFish$(min_grp_fish)_minFish$(min_log_pv_fish)_seqfac$(seqfac)_$(date_hour).tsv", df_mp_META_META, delim='\t') 
############################################################################################################################################################################
#df_rand_corr_muts_META_sort = sort(df_rand_corr_muts_META, "Seed_Mut", by = x -> mp_AA_gene_sortKey_2_universal(x, sub_0__posonly_1))
#CSV.write("$(rand_folder)/FINAL_META_df_rand_correlated_muts_$(random_sample_size)muts__EPCI_$(EPCI_qc_str)__HQCS_$(HQCS_qc_string)___plusminus$(plus_minus)_minGrpFish$(min_grp_fish)_minFish$(min_log_pv_fish)_seqfac$(seqfac)_$(date_hour).csv", df_rand_corr_muts_META_sort)
#CSV.write("$(rand_folder)/FINAL_META_df_rand_correlated_muts_$(random_sample_size)muts__EPCI_$(EPCI_qc_str)__HQCS_$(HQCS_qc_string)___plusminus$(plus_minus)_minGrpFish$(min_grp_fish)_minFish$(min_log_pv_fish)_seqfac$(seqfac)_$(date_hour).tsv", df_rand_corr_muts_META_sort, delim='\t')
############################################################################################################################################################################
runtime = time() - start
runtime_rd = round(digits=2, runtime)
runtime1, runtime2 = seconds_to_hrs_min_sec(runtime)
open("$(rand_folder)/rand_rand_correlated_muts_runtime_printout_$(date_hour).txt", "w") do g
    println(g, "Runtime v0 = $(runtime) seconds")
    println(g, "Runtime v1 = $(runtime1)")
    println(g, "Runtime v2 = $(runtime2)")
end
println("Runtime v0 = $(runtime) seconds")
#println("Runtime v1 = $(runtime1)")
println("Runtime v2 = $(runtime2)"); print("\n"^1)
### Runtime = 
### Runtime = 
### Runtime = 
### Runtime = 0 hr, 41 min, 27.32 sec, 2026_03_02, pos_only
### Runtime = 0 hr, 53 min, 11.68 sec, 2026_03_02, subs
### Runtime = 0 hr, 40 min, ????? sec  2026_02_24
### Runtime = 0 hr, 39 min, 35.17 sec, 2026_02_22
### Runtime = 1 hr, 03 min, 18.75 sec, 2026_02_21
### Runtime = 1 hr, 20 min, 23.86 sec, 2026_02_20
### Runtime = 0 hr, 36 min, 03.01 sec, 2026_01_08
### Runtime = 0 hr, 32 min, 57.32 sec, 2025-12-31
### Runtime = 0 hr, 28 min, 06.27 sec, 2025-11-18
### Runtime = 0 hr, 26 min, 02.08 sec, 2025-11-02
### Runtime = 0 hr, 27 min, 59.44 sec, 2025-10-17
### Runtime = 0 hr, 27 min, 00.20 sec, 2025-10-17
### Runtime = 0 hr, 30 min, 46.09 sec, 2025-09-29
# Spike-Only Runtime = 
# Spike-Only Runtime = 
# Spike-Only Runtime = 0 hr, 6 min, 48.51 sec, 2026_02_08
#####################################################################################################################################
#####################################################################################################################################

In [103]:
### Converting Excel .xlsx file back into CSV file
Pkg.add("XLSX"); using XLSX
df_mp_subs_CSV_from_Excel_xlsx = DataFrame(XLSX.readtable("mp_subs_2026_02_28_12PM__EPCI_8_10_95__HQCS_5_1_5__minFsh5.0_grpFsh_2.0_RANDOM/df_rand_MP_META_sorted__EPCI_8_10_95__HQCS_5_1_5___plusminus5_minGrpFish2.0_minFish5.0_seqfacON_2026_02_28_12PM___repeats_removed.xlsx", "df_rand_MP_META_sorted__EPCI_8_"))
CSV.write("mp_subs_2026_02_28_12PM__EPCI_8_10_95__HQCS_5_1_5__minFsh5.0_grpFsh_2.0_RANDOM/df_rand_MP_META_sorted__EPCI_8_10_95__HQCS_5_1_5___plusminus5_minGrpFish2.0_minFish5.0_seqfacON_2026_02_28_12PM___repeats_removed.csv", df_mp_subs_CSV_from_Excel_xlsx)
###########################################################################################################################################################################
###########################################################################################################################################################################

   Resolving package versions...
  No Changes to `~/.julia/environments/v1.11/Project.toml`
  No Changes to `~/.julia/environments/v1.11/Manifest.toml`


"mp_subs_2026_02_28_12PM__EPCI_8_10_95__HQCS_5_1_5__minFsh5.0_grpFsh_2.0_RANDOM/df_rand_MP_META_sorted__EPCI_8_10_95__HQCS_5_1_5___plusminus5_minGrpFish2.0_minFish5.0_seqfacON_2026_02_28_12PM___repeats_removed.csv"

In [64]:
### Getting mp stats from df_mp to compare to fake runs, i.e. df_mp_fake
date_now = Dates.format(now(), "yyyy_mm_dd__IMMp"); println(date_now)
nowtime = Dates.format(now(), "I:MM.SS_p"); println(nowtime)
avg_nonRBD_rel_mp_subs_8_10_95__2026_02_28 = [1.15, 1.10, 1.02, 1.27, 1.10, 1.36, 1.20, 1.10, 1.20, 3.04, 1.25, 1.71, 1.30]
median_avg_nonRBD_rel_mp_subs_8_10_95__2026_02_28 = median(avg_nonRBD_rel_mp_subs_8_10_95__2026_02_28)
println(median_avg_nonRBD_rel_mp_subs_8_10_95__2026_02_28)
################################################################################################
date_now = Dates.format(now(), "yyyy_mm_dd__IMMp"); println(date_now)

### Converting Excel .xlsx file back into CSV file
#Pkg.add("XLSX"); using XLSX
#df_mp_subs_CSV_from_Excel_xlsx = DataFrame(XLSX.readtable("mp_subs_2026_02_28_12PM__EPCI_8_10_95__HQCS_5_1_5__minFsh5.0_grpFsh_2.0_RANDOM/df_rand_MP_META_sorted__EPCI_8_10_95__HQCS_5_1_5___plusminus5_minGrpFish2.0_minFish5.0_seqfacON_2026_02_28_12PM___repeats_removed.xlsx", "df_rand_MP_META_sorted__EPCI_8_"))
#CSV.write("df_rand_MP_META_sorted__EPCI_8_10_95__HQCS_5_1_5___plusminus5_minGrpFish2.0_minFish5.0_seqfacON_2026_02_28_12PM___repeats_removed.csv", df_mp_subs_CSV_from_Excel_xlsx)

folder = "mp_subs_2026_03_01_9PM__EPCI_8_10_95__HQCS_5_1_5__minFsh5.0_grpFsh_2.0_RANDOM"
filename = "df_rand_MP_META_sorted__EPCI_8_10_95__HQCS_5_1_5___plusminus5_minGrpFish2.0_minFish5.0_seqfacON_2026_03_01_9PM"
df_mp_subs_csv = CSV.read("$(folder)/$(filename).csv", DataFrame)
headers = names(df_mp_subs_csv)
ncols = ncol(df_mp_subs_csv)
nrows = nrow(df_mp_subs_csv)
################################################################################################
SeedMut_df_vec = df_mp_subs_csv[!, "SeedMut"]
MP_Tot_Mut_ct_df_vec = df_mp_subs_csv[!, "MP_Tot_Mut_ct"]
MP_Tot_Reg_ct_df_vec = df_mp_subs_csv[!, "MP_Tot_Reg_ct"]
Reg_Tot_Mut_ct_df_vec = df_mp_subs_csv[!, "Reg_Tot_Mut_ct"]
MP_Region_Number_df_vec = df_mp_subs_csv[!, "MP_Region_Number"]
Mutation_df_vec = df_mp_subs_csv[!, "Mutation"]
MP_Mut_ct_df_vec = df_mp_subs_csv[!, "MP_Mut_ct"]
EPCI_Mut_ct_df_vec = df_mp_subs_csv[!, "EPCI_Mut_ct"]
Adjusted_MP_seq_ct_df_vec = df_mp_subs_csv[!, "Adjusted_MP_seq_ct"]
EPCI_pct_df_vec = df_mp_subs_csv[!, "EPCI_pct"]
MP_pct_df_vec = df_mp_subs_csv[!, "MP_pct"]
non_MP_pct_df_vec = df_mp_subs_csv[!, "non_MP_pct"]
Fishers_Exact_Test_pval_df_vec = df_mp_subs_csv[!, "Fishers_Exact_Test_pval"]
log10pvFISH_df_vec = df_mp_subs_csv[!, "log10pvFISH"]
Fold_Incr_df_vec = df_mp_subs_csv[!, "Fold_Incr"]
Chi2_df_vec = df_mp_subs_csv[!, "Chi2"]
EPCI_HQCS_Ratio_df_vec = df_mp_subs_csv[!, "EPCI_HQCS_Ratio"]
avg_NonRBD_AAct_df_vec = df_mp_subs_csv[!, "avg_NonRBD_AAct"]
avg_nonRBD_rel_df_vec = df_mp_subs_csv[!, "avg_nonRBD_rel"]
PangoDateIndex50_Avg_df_vec = df_mp_subs_csv[!, "PangoDateIndex50_Avg"]
PangoDateStr50_Avg_df_vec = df_mp_subs_csv[!, "PangoDateStr50_Avg"]
CollDateIndex_Avg_df_vec = df_mp_subs_csv[!, "CollDateIndex_Avg"]
CollDateString_Avg_df_vec = df_mp_subs_csv[!, "CollDateString_Avg"]
################################################################################################
rows = collect(eachrow(df_mp_subs_csv))

mp_mut_total_dict = Dict{Int, Int}()

total_mp_ct = 0
total_MP_Tot_Mut_ct = 0
total_MP_Mut_ct = 0 
total_EPCI_Mut_ct = 0
total_Adjusted_MP_seq_ct = 0
Fishers_Exact_Test_pval_sum = 0
avg_NonRBD_AAct_sum = 0
avg_nonRBD_rel_sum = 0
log10pvFISH_sum = 0
Fold_Incr_sum = 0

MP_Tot_Mut_ct_sum = 0
MP_Tot_Reg_ct_sum = 0
Reg_Tot_Mut_ct_sum = 0

SeedMut_vec = Float64[]
MP_Tot_Mut_ct_vec = Int[]
MP_Tot_Reg_ct_vec = Int[]
Reg_Tot_Mut_ct_vec = Int[]
MP_Region_Number_vec = Int[]
Mutation_vec = String[]
MP_Mut_ct_vec = Int[]
EPCI_Mut_ct_vec = Int[]
Adjusted_MP_seq_ct_vec = Int[]
EPCI_pct_vec = Float64[]
MP_pct_vec = Float64[]
non_MP_pct_vec = Float64[]
Fishers_Exact_Test_pval_vec = Float64[]
log10pvFISH_vec = Float64[]
Fold_Incr_vec = Float64[]
#Chi2_vec = Float64[]
EPCI_HQCS_Ratio_vec = Float64[]
avg_NonRBD_AAct_vec = Float64[]
avg_nonRBD_rel_vec = Float64[]
#PangoDateIndex50_Avg_vec = Int[]
#PangoDateStr50_Avg_vec = String[]
#CollDateIndex_Avg_vec = Int[]
#CollDateString_Avg_vec = String[]
for i in 1:nrows
    row = rows[i]
#################################
    SeedMut = row[1]
    MP_Tot_Mut_ct = row[2]
    MP_Tot_Reg_ct = row[3]
    Reg_Tot_Mut_ct = row[4]    
    MP_Region_Number = row[5]
    Mutation = row[6]
    MP_Mut_ct = row[7]
    EPCI_Mut_ct = row[8]
    Adjusted_MP_seq_ct = row[9]
    EPCI_pct = row[10]
    MP_pct = row[11]
    non_MP_pct = row[12]
    Fishers_Exact_Test_pval = row[13]
    log10pvFISH = row[14]
    Fold_Incr = row[15]
    if !(Fold_Incr isa Float64)
        Fold_Incr_temp = Fold_Incr[2:end]
        Fold_Incr = parse(Float64, Fold_Incr_temp)
        Fold_Incr = Fold_Incr/2
    end
#    Fold_Incr = parse(Float64, Fold_Incr)
    Chi2 = row[16]
    EPCI_HQCS_Ratio = row[17]
    avg_NonRBD_AAct = row[18]
    avg_nonRBD_rel = row[19]
    PangoDateIndex50_Avg = row[20]
    PangoDateStr50_Avg = row[21]
    CollDateIndex_Avg = row[22]
    CollDateString_Avg = row[23]
#################################
    total_MP_Mut_ct += MP_Mut_ct
    total_EPCI_Mut_ct += EPCI_Mut_ct
    total_Adjusted_MP_seq_ct += Adjusted_MP_seq_ct
    Fishers_Exact_Test_pval_sum += Fishers_Exact_Test_pval
    log10pvFISH_sum += log10pvFISH
    Fold_Incr_sum += Fold_Incr
    push!(MP_Mut_ct_vec, MP_Mut_ct)
    push!(EPCI_Mut_ct_vec, EPCI_Mut_ct)
    push!(Adjusted_MP_seq_ct_vec, Adjusted_MP_seq_ct)
    push!(Fishers_Exact_Test_pval_vec, Fishers_Exact_Test_pval)
    push!(log10pvFISH_vec, log10pvFISH)
    push!(Fold_Incr_vec, Fold_Incr)
    push!(EPCI_HQCS_Ratio_vec, EPCI_HQCS_Ratio)
    push!(avg_NonRBD_AAct_vec, avg_NonRBD_AAct)
    push!(avg_nonRBD_rel_vec, avg_nonRBD_rel)
    if i == nrows
        total_mp_ct += 1
        total_MP_Tot_Mut_ct += MP_Tot_Mut_ct
        avg_NonRBD_AAct_sum += avg_NonRBD_AAct
        avg_nonRBD_rel_sum += avg_nonRBD_rel
        push!(MP_Tot_Mut_ct_vec, MP_Tot_Mut_ct)
        push!(MP_Tot_Reg_ct_vec, MP_Tot_Reg_ct)
        push!(Reg_Tot_Mut_ct_vec, Reg_Tot_Mut_ct)
    elseif i ≠ 1
        prev_row = rows[i-1]
#################################
        prev_SeedMut = prev_row[1]
        prev_MP_Tot_Mut_ct = prev_row[2]
        prev_MP_Tot_Reg_ct = prev_row[3]
        prev_Reg_Tot_Mut_ct = prev_row[4]
#        prev_MP_Region_Number = prev_row[5]
#        prev_Mutation = prev_row[6]
#        prev_MP_Mut_ct = prev_row[7]
#        prev_EPCI_Mut_ct = prev_row[8]
#        prev_Adjusted_MP_seq_ct = prev_row[9]
#        prev_EPCI_pct = prev_row[10]
#        prev_MP_pct = prev_row[11]
#        prev_non_MP_pct = prev_row[12]
#        prev_Fishers_Exact_Test_pval = prev_row[13]
#        prev_log10pvFISH = prev_row[14]
#        prev_Fold_Incr = prev_row[15]
#        if prev_Fold_Incr isa String
#            prev_Fold_Incr = parse(Float64, prev_Fold_Incr[2:end])/2
#        end
#        prev_Chi2 = prev_row[16]
#        prev_EPCI_HQCS_Ratio = prev_row[17]
        prev_avg_NonRBD_AAct = prev_row[18]
        prev_avg_nonRBD_rel = prev_row[19]
#       prev_PangoDateIndex50_Avg = prev_row[20]
#       prev_PangoDateStr50_Avg = prev_row[21]
#       prev_CollDateIndex_Avg = prev_row[22]
#       prev_CollDateString_Avg = prev_row[23]
        if prev_SeedMut ≠ SeedMut
            total_mp_ct += 1
            total_MP_Tot_Mut_ct += prev_MP_Tot_Mut_ct
            avg_NonRBD_AAct_sum += prev_avg_NonRBD_AAct
            avg_nonRBD_rel_sum += prev_avg_nonRBD_rel
            push!(MP_Tot_Mut_ct_vec, prev_MP_Tot_Mut_ct)
            push!(MP_Tot_Reg_ct_vec, prev_MP_Tot_Reg_ct)
            push!(Reg_Tot_Mut_ct_vec, prev_Reg_Tot_Mut_ct)
        end
    end
end


average_different_muts_per_mp = total_MP_Tot_Mut_ct/total_mp_ct
average_MP_mut_ct_per_MPmut = total_MP_Mut_ct/nrows
average_EPCI_mut_ct_per_MPmut = total_EPCI_Mut_ct/nrows
average_Adjusted_MP_seq_ct = total_Adjusted_MP_seq_ct/nrows
average_avg_NonRBD_AAct = avg_NonRBD_AAct_sum/total_mp_ct
average_avg_nonRBD_rel = avg_nonRBD_rel_sum/total_mp_ct
average_Fishers_Exact_Test_pval = Fishers_Exact_Test_pval_sum/nrows  ## This is a true "normal" average
average_log10pvFISH = log10pvFISH_sum/nrows                           ## This is a geometric average
average_Fold_Incr = Fold_Incr_sum/nrows

average_different_muts_per_mp_rd = round(digits=2, average_different_muts_per_mp)
average_MP_mut_ct_per_MPmut_rd = round(digits=2, average_MP_mut_ct_per_MPmut)
average_EPCI_mut_ct_per_MPmut_rd = round(digits=2, average_EPCI_mut_ct_per_MPmut)
average_Adjusted_MP_seq_ct_rd = round(digits=2, average_Adjusted_MP_seq_ct)
average_avg_NonRBD_AAct_rd = round(digits=2, average_avg_NonRBD_AAct)
average_avg_nonRBD_rel_rd = round(digits=2, average_avg_nonRBD_rel)
average_Fishers_Exact_Test_pval_rd = round(digits=12, average_Fishers_Exact_Test_pval)
average_log10pvFISH_rd = round(digits=2, average_log10pvFISH)
average_Fold_Incr_rd = round(digits=2, average_Fold_Incr)

median_different_muts_per_mp = median(MP_Tot_Mut_ct_vec)
median_MP_mut_ct_per_MPmut = median(MP_Mut_ct_vec)
median_EPCI_mut_ct_per_MPmut = median(EPCI_Mut_ct_vec)
median_Adjusted_MP_seq_ct = median(Adjusted_MP_seq_ct_vec)
median_avg_NonRBD_AAct = median(avg_NonRBD_AAct_vec)
median_avg_nonRBD_rel = median(avg_nonRBD_rel_vec)
median_Fishers_Exact_Test_pval = median(Fishers_Exact_Test_pval_vec)
median_log10pvFISH = median(log10pvFISH_vec)
median_EPCI_HQCS_Ratio = median(EPCI_HQCS_Ratio_vec)
median_Fold_Incr = median(Fold_Incr_vec)

median_different_muts_per_mp_rd = round(digits=2, median_different_muts_per_mp)
median_MP_mut_ct_per_MPmut_rd = round(digits=2, median_MP_mut_ct_per_MPmut)
median_EPCI_mut_ct_per_MPmut_rd = round(digits=2, median_EPCI_mut_ct_per_MPmut)
median_Adjusted_MP_seq_ct_rd = round(digits=2, median_Adjusted_MP_seq_ct)
median_avg_NonRBD_AAct_rd = round(digits=2, median_avg_NonRBD_AAct)
median_avg_nonRBD_rel_rd = round(digits=2, median_avg_nonRBD_rel)
median_Fishers_Exact_Test_pval_rd = round(digits=12, median_Fishers_Exact_Test_pval)
median_log10pvFISH_rd = round(digits=2, median_log10pvFISH)
median_EPCI_HQCS_Ratio_rd = round(digits=2, median_EPCI_HQCS_Ratio)
median_Fold_Incr_rd = round(digits=2, median_Fold_Incr)

print("\n"^2)
println("median_different_muts_per_mp_rd   = $(median_different_muts_per_mp_rd)")
println("median_MP_mut_ct_per_MPmut_rd     = $(median_MP_mut_ct_per_MPmut_rd)")
println("median_EPCI_mut_ct_per_MPmut_rd   = $(median_EPCI_mut_ct_per_MPmut_rd)")
println("median_Adjusted_MP_seq_ct_rd      = $(median_Adjusted_MP_seq_ct_rd)")
println("median_avg_NonRBD_AAct_rd         = $(median_avg_NonRBD_AAct_rd)")
println("median_avg_nonRBD_rel_rd          = $(median_avg_nonRBD_rel_rd)")
println("median_Fishers_Exact_Test_pval_rd = $(median_Fishers_Exact_Test_pval_rd)")
println("median_log10pvFISH_rd             = $(median_log10pvFISH_rd)")
println("median_EPCI_HQCS_Ratio_rd         = $(median_EPCI_HQCS_Ratio_rd)")
println("median_Fold_Incr_rd               = $(median_Fold_Incr_rd)")
print("\n"^4)
println("average_different_muts_per_mp_rd   = $(average_different_muts_per_mp_rd)")
println("average_MP_mut_ct_per_MPmut_rd     = $(average_MP_mut_ct_per_MPmut_rd)")
println("average_EPCI_mut_ct_per_MPmut_rd   = $(average_EPCI_mut_ct_per_MPmut_rd)")
println("average_Adjusted_MP_seq_ct_rd      = $(average_Adjusted_MP_seq_ct_rd)")
println("average_avg_NonRBD_AAct_rd         = $(average_avg_NonRBD_AAct_rd)")
println("average_avg_nonRBD_rel_rd          = $(average_avg_nonRBD_rel_rd)")
println("average_Fishers_Exact_Test_pval_rd = $(average_Fishers_Exact_Test_pval_rd)")
println("average_log10pvFISH_rd             = $(average_log10pvFISH_rd)")
println("average_Fold_Incr_rd               = $(average_Fold_Incr_rd)")
print("\n"^4)
########################################################################################################################
df_MP_stats_csv = DataFrame(
    Median_or_Mean = String[],
    Different_Muts_per_MP = Float64[],
    MP_Mut_ct_per_MPmut = Float64[],
    EPCI_mut_ct_per_MPmut = Float64[],
    Adjusted_MP_seq_ct = Float64[],
    avg_NonRBD_AAct = Float64[],
    avg_nonRBD_rel = Float64[],
    Fishers_Exact_Test_pval = Float64[],
    log10pvFISH = Float64[],
    EPCI_HQCS_Ratio = Float64[],
    Fold_Incr = Union{Float64, String}[])
push!(df_MP_stats_csv, ("Median", median_different_muts_per_mp, median_MP_mut_ct_per_MPmut, median_EPCI_mut_ct_per_MPmut, median_Adjusted_MP_seq_ct, median_avg_NonRBD_AAct, median_avg_nonRBD_rel, median_Fishers_Exact_Test_pval, median_log10pvFISH, median_EPCI_HQCS_Ratio, median_Fold_Incr))         
push!(df_MP_stats_csv, ("Average", average_different_muts_per_mp, average_MP_mut_ct_per_MPmut, average_EPCI_mut_ct_per_MPmut, average_Adjusted_MP_seq_ct, average_avg_NonRBD_AAct, average_avg_nonRBD_rel, average_Fishers_Exact_Test_pval, average_log10pvFISH, 0.0, average_Fold_Incr))
########################################################################################################################
df_MP_stats_csv_flip = DataFrame(
    Statistic = String[],
    Median = Float64[],
    Average = Float64[])
push!(df_MP_stats_csv_flip, ("Different_Muts_per_MP", median_different_muts_per_mp, average_different_muts_per_mp))
push!(df_MP_stats_csv_flip, ("MP_Mut_ct_per_MPmut", median_MP_mut_ct_per_MPmut, average_MP_mut_ct_per_MPmut))
push!(df_MP_stats_csv_flip, ("EPCI_mut_ct_per_MPmut", median_EPCI_mut_ct_per_MPmut, average_EPCI_mut_ct_per_MPmut))
push!(df_MP_stats_csv_flip, ("Adjusted_MP_seq_ct", median_Adjusted_MP_seq_ct, average_Adjusted_MP_seq_ct))
push!(df_MP_stats_csv_flip, ("avg_NonRBD_AAct", median_avg_NonRBD_AAct, average_avg_NonRBD_AAct))
push!(df_MP_stats_csv_flip, ("avg_nonRBD_rel", median_avg_nonRBD_rel, average_avg_nonRBD_rel))
push!(df_MP_stats_csv_flip, ("Fishers_Exact_Test_pval", median_Fishers_Exact_Test_pval, average_Fishers_Exact_Test_pval))
push!(df_MP_stats_csv_flip, ("log10pvFISH", median_log10pvFISH, average_log10pvFISH))
push!(df_MP_stats_csv_flip, ("EPCI_HQCS_Ratio", median_EPCI_HQCS_Ratio, 0.0))
push!(df_MP_stats_csv_flip, ("Fold_Incr", median_Fold_Incr, average_Fold_Incr))
########################################################################################################################
########################################################################################################################
CSV.write("$(folder)/$(filename)__STATS.csv", df_MP_stats_csv)
CSV.write("$(folder)/$(filename)__STATS_flip.csv", df_MP_stats_csv_flip)
############################################################################################################################################################################
############################################################################################################################################################################

2026_03_01__1106PM
11:06.17_PM
1.2
2026_03_01__1106PM


median_different_muts_per_mp_rd   = 4.0
median_MP_mut_ct_per_MPmut_rd     = 8.0
median_EPCI_mut_ct_per_MPmut_rd   = 24.0
median_Adjusted_MP_seq_ct_rd      = 147.0
median_avg_NonRBD_AAct_rd         = 18.42
median_avg_nonRBD_rel_rd          = 1.2
median_Fishers_Exact_Test_pval_rd = 8.89925e-7
median_log10pvFISH_rd             = 6.05
median_EPCI_HQCS_Ratio_rd         = 12.54
median_Fold_Incr_rd               = 0.92




average_different_muts_per_mp_rd   = 6.97
average_MP_mut_ct_per_MPmut_rd     = 14.83
average_EPCI_mut_ct_per_MPmut_rd   = 52.3
average_Adjusted_MP_seq_ct_rd      = 162.6
average_avg_NonRBD_AAct_rd         = 22.58
average_avg_nonRBD_rel_rd          = 1.47
average_Fishers_Exact_Test_pval_rd = 0.000601886433
average_log10pvFISH_rd             = 8.93
average_Fold_Incr_rd               = Inf






"mp_subs_2026_03_01_9PM__EPCI_8_10_95__HQCS_5_1_5__minFsh5.0_grpFsh_2.0_RANDOM/df_rand_MP_META_sorted__EPCI_8_10_95__HQCS_5_1_5___plusminus5_minGrpFish2.0_minFish5.0_seqfacON_2026_03_01_9PM__STATS_flip.csv"

In [70]:
### Merging Related mp groups #################################
date_now = Dates.format(now(), "yyyy_mm_dd__IMMp"); println(date_now)
date_hour = Dates.format(now(), "yyyy_mm_dd_Ip")
date = Dates.format(today(), "yyyy_mm_dd")
####################################################################################################################
df_final_mp = DataFrame(mpNum = Int[], Region = Int[], MutNum = Int[], Mutation = String[])
#                    mp#    region#     mpmut#    mut
df_final_dict = Dict{Int, Dict{Int, Dict{Int, String}}}()
####################################################################################################################
####### Reminder: df_mp_META_seedmut_set_dict: Keys = seed_muts  Values = Set of correlated muts for that seed mut
df_mp_META_seedmut_set_dict_sort = sort(collect(df_mp_META_seedmut_set_dict), by = x -> length(x[2]))
df_mp_META_set_set_sort = sort(collect(df_mp_META_set_set), by = x -> length(x), rev=true)
###################################################################
################ Version 2 of df_mp_META_set_grp_dict #############
#index__seed_mut_dict = Dict{Int, String}()
#df_mp_META_set_grp_dict = Dict{Int, Set{String}}()
#mp_set_ct = 0
#for seedmut__mpset in df_mp_META_seedmut_set_dict_sort
#    mp_set_ct += 1
#    seed_mut = seedmut__mpset[1]
#    mp_set = seedmut__mpset[2]
#    df_mp_META_set_grp_dict[mp_set_ct] = mp_set
#    index__seed_mut_dict[mp_set_ct] = seed_mut
#end
###################################################################
###################################################################
################ Version 1 of df_mp_META_set_grp_dict #############
df_mp_META_set_grp_dict = Dict{Int, Set{String}}()
mp_set_ct = 0
for mp_set in df_mp_META_set_set_sort
    mp_set_ct += 1
    df_mp_META_set_grp_dict[mp_set_ct] = mp_set
end
###################################################################
###################################################################
#                       #shared_muts     grp1,grp2   proportion of smaller in larger
shared_total_pairs = Dict{Int, Set{Tuple{Int, Int, Float64}}}()
#               proportion_small_in_big  grp1,grp2,#shared_muts  
shared_prop_pairs = Dict{Float64, Set{Tuple{Int, Int, Int}}}()
#                         grp1     grp2  #sharedmuts  prop_shared
mp_set_shared_muts = Dict{Int, Dict{Int, Tuple{Int, Float64}}}()
############################################################################################################################################################################
# Part below documents the number of shared mutations and the proportion of mutations shared, which is (# shared muts)/(muts in smaller of 2 mut grps)
############################################################################################################################################################################
for i in 1:length(df_mp_META_set_grp_dict)
    mp_set_shared_muts[i] = Dict{Int, Int}()
    grp1 = df_mp_META_set_grp_dict[i]
    for j in 1:length(df_mp_META_set_grp_dict)
        grp2 = df_mp_META_set_grp_dict[j]
        len1 = length(grp1)
        len2 = length(grp2)
        big = Set{String}()
        small = Set{String}()
        if len1 ≥ len2
            big = grp1
            small = grp2
        else
            big = grp2
            small = grp1
        end
        small_len = length(small)
        shared_muts = intersect(grp1, grp2)
        overlap = length(shared_muts)
        if overlap ≠ 0
            prop_shared = round(digits=3, overlap/small_len)
            mp_set_shared_muts[i][j] = (overlap, prop_shared)
            get!(shared_total_pairs, overlap, Set{Tuple{Int, Int, Float64}}() )
            push!(shared_total_pairs[overlap], (i, j, prop_shared) )
            get!(shared_prop_pairs, prop_shared, Set{Tuple{Int, Int, Int}}() )
            push!(shared_prop_pairs[prop_shared], (i, j, overlap))
        end
    end
end
#for shared_mut_tot in keys(shared_total_pairs)
#    println(shared_mut_tot)
#end
#for (shared_mut_tot, tuple_set) in shared_total_pairs
#    if shared_mut_tot ≥ 2
#        for tuple in tuple_set
#            if tuple[3] ≥ 0.5
#                println(df_mp_META_set_grp_dict[tuple[1]])
#                println(df_mp_META_set_grp_dict[tuple[2]])
#                println("##################################################################################")
#            end
#        end
#    end
#end
############################################################################################################################################################################
############################################################################################################################################################################
##  After the initial formation of a meta_group, all other grps are checked to see if they belong to the group (this prevents duplication)
function check_against_meta_grps(grp::Set{String}, meta_grp::Set{String}, min_prop::Float64)
    grp_len = length(grp)
    meta_len = length(meta_grp)
    big = Set{String}()
    small = Set{String}()
    if meta_len ≥ grp_len
        big = meta_grp
        small = grp
    else
        big = grp
        small = meta_grp
    end
    big_len = length(big)
    small_len = length(small)
    shared_muts = intersect(grp, meta_grp)
    tot_shared = length(shared_muts)
    prop = tot_shared/small_len
    if prop ≥ min_prop && tot_shared ≥ 2
        return shared_muts, tot_shared, prop
    else
        return nothing, nothing, nothing
    end
end       
############################################################################################################################################################################
############################################################################################################################################################################
min_prop = 0.4
df_mp_META_groups = Dict{Int, Set{String}}()
df_mp_META_group_mut_cts = Dict{Int, Dict{String, Int}}()
used_grp_numbers = Set{Int}()
for (grp1, grpdict) in mp_set_shared_muts
    if !(grp1 in used_grp_numbers)
        tot_grps = length(df_mp_META_groups)
        grp1_mutset = df_mp_META_set_grp_dict[grp1]
        for (grp2, overlap__propshare) in grpdict
            if !(grp2 in used_grp_numbers)
                grp2_mutset = df_mp_META_set_grp_dict[grp2]
                confounders = ["ORF1a:K1795Q", "ORF1a:E102K", "ORF1a:1795", "ORF1a:102", "ORF1a:102"]
                if !("ORF1a:E102K" in grp2_mutset) && !("ORF1a:102" in grp2_mutset) && !("ORF1a:E102K" in grp1_mutset) && !("ORF1a:102" in grp1_mutset)
                    if overlap__propshare[1] ≥ 2 && overlap__propshare[2] ≥ min_prop
                        df_mp_META_groups[tot_grps + 1] = Set{String}()
                        df_mp_META_group_mut_cts[tot_grps + 1] = Dict{String, Int}()
#                        df_mp_META_group_mut_cts[tot_grps + 1] = get(df_mp_META_group_mut_cts, tot_grps + 1, Dict{String, Int}() )
#                        df_mp_META_groups[tot_grps + 1] = get(df_mp_META_groups, tot_grps+1, Set{String}() )
#                        get!(df_mp_META_groups, tot_grps + 1, Set{String}() )
                        for mut in grp1_mutset
                            push!(df_mp_META_groups[tot_grps + 1], mut)
                            df_mp_META_group_mut_cts[tot_grps + 1][mut] = get(df_mp_META_group_mut_cts[tot_grps + 1], mut, 0) + 1
                        end
                        for mut in grp2_mutset
                            push!(df_mp_META_groups[tot_grps + 1], mut)
                            df_mp_META_group_mut_cts[tot_grps + 1][mut] = get(df_mp_META_group_mut_cts[tot_grps + 1], mut, 0) + 1
                        end
                        push!(used_grp_numbers, grp1)
                        push!(used_grp_numbers, grp2)
                        for grp3 in 1:length(df_mp_META_set_grp_dict)
                            grp3_mutset = df_mp_META_set_grp_dict[grp3]
                            if grp3 ≠ grp1 && grp3 ≠ grp2 && !(grp3 in used_grp_numbers) && !("ORF1a:E102K" in grp3_mutset) && !("ORF1a:102" in grp3_mutset)
                                shared_muts, tot_shared, prop = check_against_meta_grps(grp3_mutset, df_mp_META_groups[tot_grps + 1], min_prop)
                                if shared_muts ≠ nothing
                                    if tot_shared ≥ 2 && prop ≥ min_prop
                                        for mut in grp3_mutset
                                            push!(df_mp_META_groups[tot_grps + 1], mut)
                                            df_mp_META_group_mut_cts[tot_grps + 1][mut] = get(df_mp_META_group_mut_cts[tot_grps + 1], mut, 0) + 1
                                        end
                                        push!(used_grp_numbers, grp3)
                                    end
                                end
                            end
                        end
                    end
                end
            end
        end
    end
end
print("\n"^4)
#############################################################################################################################################################################
println("################### All non-merged groups after part 1 ###################")
for i in 1:length(df_mp_META_set_grp_dict)
    if !(i in used_grp_numbers)
        println("nonused grp_number = $(i)")
        nonmerge_grp_mutset = df_mp_META_set_grp_dict[i]
        nonmerge_grp_mutset_sort = sort(collect(nonmerge_grp_mutset), by = x -> mp_AA_gene_sortKey_2_universal(x, sub_0__posonly_1))
        nonmerge_grp_mutset_join = join(nonmerge_grp_mutset_sort, ", ")
        println("Group #$(i) = $(nonmerge_grp_mutset_join)")
    end
end
print("\n"^4)
#############################################################################################################################################################################
println("#################################### Groups After One Run ######################################")
println("Total Groups = $(length(df_mp_META_groups))")
for i in 1:length(df_mp_META_groups)
    mut_set = df_mp_META_groups[i]
    mut_set_sort = sort(collect(mut_set), by = x -> mp_AA_gene_sortKey_2_universal(x, sub_0__posonly_1))
    mut_set_sort_join = join(mut_set_sort, ", ")
    println("Group #$(i) = $(mut_set_sort_join)")
end
print("\n"^4)
#############################################################################################################################################################################
min_prop = 0.5
df_mp_META_groups_2 = Dict{Int, Set{String}}()
used_grps_2 = Set{Int}()
confounders = ["ORF1a:K1795Q", "ORF1a:E102K", "ORF1a:1795", "ORF1a:102", "ORF1a:102"]
for i in 1:length(df_mp_META_groups)
#    if !(i in used_grps_2) && !((length(intersect(confounders, df_mp_META_groups[i]))) ≥ 2)
    if !(i in used_grps_2) && !("ORF1a:E102K" in df_mp_META_groups[i]) && !("ORF1a:102" in df_mp_META_groups[i])
        num_of_grps = length(df_mp_META_groups_2)
        grp1_mutset = df_mp_META_groups[i]
        for j in 1:length(df_mp_META_groups)
            if !(j in used_grps_2) && !(length(intersect(confounders, df_mp_META_groups[j])) ≥ 2)
                grp2_mutset = df_mp_META_groups[j]
                if i ≠ j
                    overlap_muts, overlap, prop = check_against_meta_grps(grp1_mutset, grp2_mutset, min_prop)
                    if overlap_muts ≠ nothing
                        if prop ≥ min_prop && overlap ≥ 2
                            df_mp_META_groups_2[num_of_grps+1] = Set{String}()
                            for mut in grp1_mutset
                                push!(df_mp_META_groups_2[num_of_grps+1], mut)
                            end
                            for mut in grp2_mutset
                                push!(df_mp_META_groups_2[num_of_grps+1], mut)
                            end
                            push!(used_grps_2, i)
                            push!(used_grps_2, j)
                        end
                    end
                end
            end
        end
    end
end
df_mp_META_set_vec_new = Set{Vector{String}}()
for i in 1:length(df_mp_META_groups_2)
    new_grp = df_mp_META_groups_2[i]
    new_grp_sort = sort(collect(new_grp), by = x -> mp_AA_gene_sortKey_2_universal(x, sub_0__posonly_1))
    push!(df_mp_META_set_vec_new, new_grp_sort)
end
for i in 1:length(df_mp_META_groups)
    if !(i in used_grps_2)
        new_grp = df_mp_META_groups[i]
        new_grp_sort = sort(collect(new_grp), by = x -> mp_AA_gene_sortKey_2_universal(x, sub_0__posonly_1))
        push!(df_mp_META_set_vec_new, new_grp_sort)
    end
end
for i in 1:length(df_mp_META_set_grp_dict)
    if !(i in used_grp_numbers) && !(i in used_grps_2)
        if haskey(df_mp_META_groups, i)
            new_grp = df_mp_META_groups[i]
            new_grp_sort = sort(collect(new_grp), by = x -> mp_AA_gene_sortKey_2_universal(x, sub_0__posonly_1))
            push!(df_mp_META_set_vec_new, new_grp_sort)
        end
    end
end
df_mp_META_set_vec_new_sort = sort(collect(df_mp_META_set_vec_new), by = x -> length(x), rev = true)
mp_ct = 0
gene_dict = Dict{Int, Dict{Int, String}}()
pos_dict = Dict{Int, Dict{Int, Int}}()
for i in 1:length(df_mp_META_set_vec_new_sort)
    df_final_dict[i] = Dict{Int, Dict{Int, String}}()
    gene_dict[i] = Dict{Int, String}()
    pos_dict[i] = Dict{Int, Int}()
    mp_ct += 1
    mut_vec = df_mp_META_set_vec_new_sort[i]
    region_ct = 1
#    for j in 1:length(mut_vec)
#        df_final_dict[i][j] = Dict{Int, String}()
#    end
    for j in 1:length(mut_vec)
        mp_mut_ct = j
        mut = mut_vec[j]
        moot_gene = aa_gene_comprehensive_dict[mut]
        gene_dict[i][j] = moot_gene
        mootpos = aa_pos_comprehensive_dict[mut]
        pos_dict[i][j] = mootpos
        if j == 1
            df_final_dict[i][region_ct] = get(df_final_dict[i], region_ct, Dict{Int, String}())
            df_final_dict[i][region_ct][mp_mut_ct] = mut
            push!(df_final_mp, (i, region_ct, mp_mut_ct, mut))
        else
            if moot_gene == gene_dict[i][j-1] && abs(mootpos - pos_dict[i][j-1]) ≤ 5
                df_final_dict[i][region_ct] = get(df_final_dict[i], region_ct, Dict{Int, String}())
                df_final_dict[i][region_ct][mp_mut_ct] = mut
                push!(df_final_mp, (i, region_ct, mp_mut_ct, mut))
            else
                region_ct += 1
                df_final_dict[i][region_ct] = Dict{Int, String}()
                df_final_dict[i][region_ct][mp_mut_ct] = mut
                push!(df_final_mp, (i, region_ct, mp_mut_ct, mut))
            end
        end 
    end
end
 

CSV.write("$(rand_folder)/final_merged_patterns__EPCI_$(EPCI_qc_str)__HQCS_$(HQCS_qc_str)__plsmns$(plus_minus)_minGrpFish$(min_grp_fish)_minFish$(min_log_pv_fish)_$(date_hour).csv", df_final_mp)
#CSV.write("$(rand_folder)/final_merged_patterns__EPCI_$(EPCI_qc_str)__HQCS_$(HQCS_qc_str)__plsmns$(plus_minus)_minGrpFish$(min_grp_fish)_minFish$(min_log_pv_fish)_$(date_hour).tsv", df_final_mp, delim='\t')

#for i in 1:length(df_final_dict)
#    for j in 1:length(df_final_dict[i])
#        for k in 1:length(df_final_dict[i][j])
#            mut = df_final_dict[i][j][k]
#            push!(df_final_mp, (i, j, k, mut))
#        end
#    end
#end

#                    mp#    region#      mut#    mut
#df_final_dict = Dict{Int, Dict{Int, Dict{Int, String}}}()
# df_final_mp = DataFrame(mpNum = Int[], Region = Int[], MutNum = Int[], Mutation = String[])


#############################################################################################################################################################################
println("#################################### Groups After 2 Runs ######################################")
println("Total Groups = $(length(df_mp_META_groups_2))")
for i in 1:length(df_mp_META_groups_2)
    grp_num = i
    mut_set = df_mp_META_groups_2[i]
    mut_set_sort = sort(collect(mut_set), by = x -> mp_AA_gene_sortKey_2_universal(x, sub_0__posonly_1))
    mut_set_sort_join = join(mut_set_sort, ", ")
    println("Group #$(grp_num) = $(mut_set_sort_join)")
end
print("\n"^4)     
#############################################################################################################################################################################
#println("################### All non-merged groups after part 2 ###################")
#for i in 1:length(df_mp_META_set_grp_dict)
#    if !(i in used_grps_2)
#        nonmerge_grp_mutset2 = df_mp_META_set_grp_dict[i]
#        nonmerge_grp_mutset2_sort = sort(collect(nonmerge_grp_mutset2), by = x -> mp_AA_gene_sortKey_2_universal(x, sub_0__posonly_1))
#        nonmerge_grp_mutset2_join = join(nonmerge_grp_mutset2_sort, ", ")
#        println("Group #$(i) = $(nonmerge_grp_mutset2_join)")
#    end
#end
#print("\n"^4)
#############################################################################################################################################################################


2026_03_05__758AM




################### All non-merged groups after part 1 ###################
nonused grp_number = 39
Group #39 = ORF1a:102, ORF1a:1361, ORF1a:1367, ORF1a:1372, ORF1a:1795
nonused grp_number = 43
Group #43 = ORF1a:102, ORF1a:1367, ORF1a:1372, ORF1a:1795




#################################### Groups After One Run ######################################
Total Groups = 29
Group #1 = ORF1a:38, ORF1a:1188, ORF1a:1795, ORF1a:2980, ORF1a:3694, ORF1a:3729, ORF1a:3732, ORF1b:314, ORF1b:870, ORF1b:2311, ORF1b:2313, ORF1b:2314, S:19, S:66, S:215, S:828, S:1153, S:1176, S:1185, S:1189, S:1201, S:1231, S:1266, ORF3a:10, ORF3a:45, ORF3a:182, ORF9b:13
Group #2 = ORF1a:3966, ORF1b:662
Group #3 = S:155, S:157, S:546, S:547, ORF3a:9
Group #4 = ORF1a:1682, ORF1a:4116
Group #5 = S:19, S:25, S:26, S:142, S:144, S:148, S:243, S:655, S:859, M:29, M:82, ORF7a:82, ORF9b:10
Group #6 = ORF1a:3058, ORF1a:3070, ORF1a:3072, ORF1a:3076, S:50, S:653, E:10, E:30, M:125
Group #7 = ORF1a:232, ORF1a:2